In [ ]:
from __future__ import annotations

import base64
import gc
import hashlib
import io
import re
import zlib
from dataclasses import dataclass
from typing import Any, Iterable

import numpy as np
import pandas as pd

EXPECTED_TRAINING_SCRIPT_SHA256 = '28c77eb345fb8fdba207af791841ec1e9f5fe9e6ff32ce01e48d02c9659b1753'
EXPECTED_MANIFEST_SHA256 = 'e86fa1405fb14a1809e2d6e5322565c162c676492598411acd8b7f5b5c816308'
STRUCTURE_ONLY = False
POOL_TABLE = "bigalpha_2026_instruments"
EXPOSURE_TABLE = "bigalpha_2026_exposure"
INDUSTRY_FIELD = "industry_level1_code"
PREVIOUS_TRADING_DAYS = 5
PADDING_PROBE_CALENDAR_DAYS = 90
INFERENCE_BATCH_SIZE = 2048
TARGET_BETA_CLIP = 5.0
SNAPSHOT_RULES = {'prices': 'last_positive',
 'depth': 'last_nonnegative',
 'order_counts': 'last_nonnegative'}
FLOW_AGGREGATION = 'last'
REBUILD_SQL_TEMPLATE = "\n    WITH minute_base AS (\n        SELECT\n            date,\n            instrument,\n            CAST(date AS DATE) AS trading_day,\n            CASE\n                WHEN strftime(date, '%H:%M:%S') >= '09:30:00'\n                 AND strftime(date, '%H:%M:%S') <= '10:00:00' THEN 1\n                WHEN strftime(date, '%H:%M:%S') >  '10:00:00'\n                 AND strftime(date, '%H:%M:%S') <= '10:30:00' THEN 2\n                WHEN strftime(date, '%H:%M:%S') >  '10:30:00'\n                 AND strftime(date, '%H:%M:%S') <= '11:00:00' THEN 3\n                WHEN strftime(date, '%H:%M:%S') >  '11:00:00'\n                 AND strftime(date, '%H:%M:%S') <= '11:30:00' THEN 4\n                WHEN strftime(date, '%H:%M:%S') >= '13:00:00'\n                 AND strftime(date, '%H:%M:%S') <= '13:30:00' THEN 5\n                WHEN strftime(date, '%H:%M:%S') >  '13:30:00'\n                 AND strftime(date, '%H:%M:%S') <= '14:00:00' THEN 6\n                WHEN strftime(date, '%H:%M:%S') >  '14:00:00'\n                 AND strftime(date, '%H:%M:%S') <= '14:30:00' THEN 7\n                WHEN strftime(date, '%H:%M:%S') >  '14:30:00'\n                 AND strftime(date, '%H:%M:%S') <= '15:00:00' THEN 8\n                ELSE NULL\n            END AS bar_index,\n            open, high, low, close, adjust_factor,\n            volume, amount, deal_number,\n            ask_price1, ask_price2, ask_price3,\n            bid_price1, bid_price2, bid_price3,\n            ask_volume1, ask_volume2, ask_volume3,\n            bid_volume1, bid_volume2, bid_volume3,\n            ask_num_orders1, ask_num_orders2, ask_num_orders3,\n            bid_num_orders1, bid_num_orders2, bid_num_orders3\n        FROM __TABLE_NAME__\n    )\n    SELECT\n        trading_day,\n        instrument,\n        bar_index,\n        first(open ORDER BY date) AS open,\n        max(high) AS high,\n        min(low) AS low,\n        last(close ORDER BY date) AS close,\n        last(adjust_factor ORDER BY date) AS adjust_factor,\n        last(volume ORDER BY date) AS volume,\n        last(amount ORDER BY date) AS amount,\n        last(deal_number ORDER BY date) AS deal_number,\n        last(ask_price1 ORDER BY date) FILTER (WHERE ask_price1 > 0) AS ask_price1,\n        last(ask_price2 ORDER BY date) FILTER (WHERE ask_price2 > 0) AS ask_price2,\n        last(ask_price3 ORDER BY date) FILTER (WHERE ask_price3 > 0) AS ask_price3,\n        last(bid_price1 ORDER BY date) FILTER (WHERE bid_price1 > 0) AS bid_price1,\n        last(bid_price2 ORDER BY date) FILTER (WHERE bid_price2 > 0) AS bid_price2,\n        last(bid_price3 ORDER BY date) FILTER (WHERE bid_price3 > 0) AS bid_price3,\n        last(ask_volume1 ORDER BY date) FILTER (WHERE ask_volume1 >= 0) AS ask_volume1,\n        last(ask_volume2 ORDER BY date) FILTER (WHERE ask_volume2 >= 0) AS ask_volume2,\n        last(ask_volume3 ORDER BY date) FILTER (WHERE ask_volume3 >= 0) AS ask_volume3,\n        last(bid_volume1 ORDER BY date) FILTER (WHERE bid_volume1 >= 0) AS bid_volume1,\n        last(bid_volume2 ORDER BY date) FILTER (WHERE bid_volume2 >= 0) AS bid_volume2,\n        last(bid_volume3 ORDER BY date) FILTER (WHERE bid_volume3 >= 0) AS bid_volume3,\n        last(ask_num_orders1 ORDER BY date) FILTER (WHERE ask_num_orders1 >= 0) AS ask_num_orders1,\n        last(ask_num_orders2 ORDER BY date) FILTER (WHERE ask_num_orders2 >= 0) AS ask_num_orders2,\n        last(ask_num_orders3 ORDER BY date) FILTER (WHERE ask_num_orders3 >= 0) AS ask_num_orders3,\n        last(bid_num_orders1 ORDER BY date) FILTER (WHERE bid_num_orders1 >= 0) AS bid_num_orders1,\n        last(bid_num_orders2 ORDER BY date) FILTER (WHERE bid_num_orders2 >= 0) AS bid_num_orders2,\n        last(bid_num_orders3 ORDER BY date) FILTER (WHERE bid_num_orders3 >= 0) AS bid_num_orders3\n    FROM minute_base\n    WHERE bar_index IS NOT NULL\n    GROUP BY trading_day, instrument, bar_index\n    ORDER BY trading_day, bar_index, instrument\n    "
IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
TABLE_IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_.]*$")

N_BARS_PER_DAY = 8
LOOKBACK_BARS = 40
MIN_VALID_BARS = 32
MIN_INDUSTRY_STOCKS = 8
OLD_FACTOR_WEIGHTS = {'book_imbalance_mean': -0.024209602584974288,
 'book_imbalance_last': 0.08380551557986225,
 'relative_spread_mean': -0.013686517644643743,
 'relative_spread_last': 0.024921948930973593,
 'pressure_persistence': 0.0024980568410629257,
 'imbalance_change': -0.03962909197546174}
OLD_FACTOR_INTERCEPT = 0.00048569424366604204
CANONICAL_COLUMNS = ['date',
 'instrument',
 'open',
 'high',
 'low',
 'close',
 'adjust_factor',
 'volume',
 'amount',
 'deal_number',
 'ask_price1',
 'ask_price2',
 'ask_price3',
 'bid_price1',
 'bid_price2',
 'bid_price3',
 'ask_volume1',
 'ask_volume2',
 'ask_volume3',
 'bid_volume1',
 'bid_volume2',
 'bid_volume3',
 'ask_num_orders1',
 'ask_num_orders2',
 'ask_num_orders3',
 'bid_num_orders1',
 'bid_num_orders2',
 'bid_num_orders3']
BASE_FEATURE_NAMES = ['volume_imbalance_l3',
 'order_count_imbalance_l3',
 'average_order_size_gap',
 'l1_depth_concentration_gap',
 'relative_spread_l1',
 'log_total_depth',
 'log_total_order_count',
 'bar_return',
 'bar_range',
 'log_volume',
 'log_amount',
 'log_deal_number']
MODEL_FEATURE_NAMES = ['raw_rank_volume_imbalance_l3',
 'raw_rank_order_count_imbalance_l3',
 'raw_rank_average_order_size_gap',
 'raw_rank_l1_depth_concentration_gap',
 'raw_rank_relative_spread_l1',
 'raw_rank_log_total_depth',
 'raw_rank_log_total_order_count',
 'raw_rank_bar_return',
 'raw_rank_bar_range',
 'raw_rank_log_volume',
 'raw_rank_log_amount',
 'raw_rank_log_deal_number',
 'centered_volume_imbalance_l3',
 'centered_order_count_imbalance_l3',
 'centered_average_order_size_gap',
 'centered_l1_depth_concentration_gap',
 'centered_relative_spread_l1',
 'centered_log_total_depth',
 'centered_log_total_order_count',
 'centered_bar_return',
 'centered_bar_range',
 'centered_log_volume',
 'centered_log_amount',
 'centered_log_deal_number',
 'delta_volume_imbalance_l3',
 'delta_order_count_imbalance_l3',
 'delta_average_order_size_gap',
 'delta_l1_depth_concentration_gap',
 'delta_relative_spread_l1',
 'delta_log_total_depth',
 'delta_log_total_order_count',
 'delta_bar_return',
 'delta_bar_range',
 'delta_log_volume',
 'delta_log_amount',
 'delta_log_deal_number',
 'valid_book',
 'session_boundary',
 'overnight_boundary']
DEFAULT_MODEL_CONFIG = {'n_features': 39,
 'channels': 32,
 'kernel_size': 3,
 'dilations': [1, 2, 4, 8],
 'dropout': 0.1}
BAR_END_MINUTES = {600: 0, 630: 1, 660: 2, 690: 3, 810: 4, 840: 5, 870: 6, 900: 7}

MODEL_PAYLOADS = [
    {
        'seed': 20260728,
        'weight': 0.3333333333333333,
        'checkpoint_name': 'final_tcn_2019_2023_seed_20260728.pt',
        'checkpoint_sha256': 'ca099f1ff89b801be88c554720dd068879b7c6e57f1da0bddd9f0cfd50875727',
        'raw_bytes': 131294,
        'payload_b85': (
            'c-o}72|QHq`~PoCAtWtmvlH2Ob6+QIl%=GUv}iE4Y%_-'
            'T(t;LCBCWEtNJ&y8bFO2Fl=el77HQSKXkULbgWmb{{(QcV|KmRo#yIzNp0C$^UH7@~`<(G~la`T?=+#T&uU8+5z7lL+K'
            'uBmD$1IS|XPZXP3wIu^B;nib6)&}vD`Ds(FFYZJ&*pRd0z(7%T*;+eDMKF>;lXHLR2V0K9~u>DI-'
            'e679Kz>H2k@hWSW~}PerR}%ydRI_9~&AT=*Q<o#zgV_;w<I4GKPjeJ;lT*9y^%B?a^_HymNRIn?I%FpgdRBM^YGL&h6<'
            'VE&Ll09n0<IaZiTNm7DA);il<k;3k>Gl^6XHeNb2%5D*m}-'
            'eDz1eiAQ`!{Y>w6`Gaj_8#uVRpfJ(ytv9f%H2Kthq7b1eL{qWLVARB2wHGeLVCGr^0|G*Itg4=H^~HUzqKJEOn*LifER'
            'b5Pk&*ce|S{Dyckn+(}1YRIE$|Ct964}a@9k6c}Vo;b2U0#Yr07%aR<50b(0pk9^644!sib4;tums{hKN_sM(Ea#ntLC'
            'sNHE$M`Td91Et63>U(hw{x)dYWzeu2%$hsA)1c8G293K6j_9D8@VTa5T(iFoT6P#T@5Z#@T67q+>@;X4GHBg_vf*=Wy|'
            '{MW4Ms-sB7SGkz8lPz>(Bvn?1ZsIFwy}7K9}<1j_mHM119$6)D30F9o2yv-'
            'H93_LXGV}jpK8jy|^xa`}&=@@!eqd+zA~p*G`z52<F}a^WbwQdT}TH?W;3ip50Im+{qm%uTIny5o&4&%A3!f=Ea@<r!R'
            '}{BhKeHOiDC-&ABr=E9(14MQ3(ZbXEs;HlI7ki|hBNLyJF$+}!R^3og6EqJO8w0FlMO4i<;c4f5g!|84Pi>O#8HEV-eb'
            '7Q_CqIIqiMcn3Cu&yDorM*VHEGl|jNp;laOhecke#Tb!Aeg`X-'
            '&yDlq&hKuq+o)U6oo3Bl*g;#=Nn0$UE$N^w<#U&LahG>@_Q$Y`@6NK}uION`>}0JHu~v7m68PLiFK*J`&VDCvO?R3tcW'
            'no4T_<h5h?d+zOW|`<y|`(AJL}BZhVCpoZh8l6V<#&^#LDboZQ^sYytteHbY|Io<ZbCrv*&K@EbF#E%DTO)tUEfe*?jI'
            '!FK*7C7A^lAdAZ%84&1yBi}{@vcZn?S?qKcVbN7023;wqFJ9+!M(Jalmg`F1n|6%b!m&KwE>_I-a*o%AUZ;PE-Ea?ul;'
            '2!R<SlVgvh{)p64pteTTkgd@*4<*akyp{3X30I?L96Vf2}CrygNA%A<HfD&?(C0|SKXau#jWXJ)poK@h*)(Uta?88q!+'
            'j0Z)d-gcd9$hntQr~*4RlqBch${pq=A$&wFt%{OznWXBWG(Y`B*?SeH9lS46C<9jqok_nH^?`k&6Mx{tgY-'
            'D$Smo1JBC{-'
            'dn7y2^UH1AB+hz3ato`O~7+pCj*Hcc>k=wZr25PKysj79V!79`U)4y|_>Qw)i`FZQW`1+^3xupZ#I+d6&f(9oUzA?kg|'
            'u>%T2_X7NpTr~~(HhsAfD7TZM@-*>P+@VOtoxSzUP>^AZ~ccWREbH8-ZzIM{SiD=(DXg~PepI+Qw-JSh2@^})$m8?6?f'
            '+s0DCL;2rgr7Q~JZa%aA(tm3{3}NDdI&%9d9uQHFJ4dK``?bclFIA#mx?7%?zf8kKPn2HD!qTJD2i2-'
            '#45@n71h60#69WrmxdKj<+n!Pe>7A(HTwP5=r7h7Al4Wt(ohth9>QS<3YY)R9;p3Ax8|vL(4F`^jZV5#0&h?!eeiGk5H'
            'WqIm_AHISNMl6cCPss+lHt0o2~s1Tc?w)`<tyNX6uXD2Hl)@jTWBaUvyjEa53FOq8FcM)Rh=Fr$nByNWjTWGLbjpw}gp'
            'U!c;6_*3EwB5HkNuz>a6}JMotPB;KmizV&amjhJmKX4`eMZ{20z{x7;c&*3-'
            'Sv6C+8rkTiNiTIi#_vE(%h!rTY!pLs!tvlR1{l$0Sjq2b_@_D1X_zAo*o$kl}W{(rIoyBYy;d`w3ek&p>kQ44VFElccH'
            '(n$tx(nh5MDiwx{^|WYKsY-Dg$DCng-'
            '4h2+=TBPcWaS;K^!(emdA<Vxr+o1d_?zv0U_+jNKSYR&qLfraVO?+c)~+|F`<h%yosX2(mry+!-'
            '1jUZ1K$$Z<6rfY@VkGEzFOUn~Ym8j`;p9kQWsl70c&M7B;zzJib-'
            '*+7>I$eW$ozBs)Urz)MV+A~rWz*vezi_v5i6=lR7&g~vv4{6ZuA+2QQS0FGa{6>qAje_DS=MTzd_gdL5I<aZDC7KIM`G'
            'n5_25#2p@o+9d#UobnGH%%0%^JieVg<l{inja#{UjQeP&l7i696VhVJm}A09!HdmIF4UTG*4J^;mJJF&fQ0?8!;-'
            '_j~~Tnhj)N^Gei+X|2^W59`Ss;CZpeR{%oF5QkcI;-b_(r-@h8!k-;3^tiMha33Q~CH~U{3*%4yNIsa-'
            'tkRx;!85`lx;qm-L5u#FwawEEx{?}5??JCvpsQ+3jc2}u>hyIsR`FEA-ci{ghRX|s%eh2?csRFxt`w#IyO2z3a)xSjiS'
            '*oC}r2dZkqg26NrRv@&E>*~1C;m~Y(0^_0ELGUQn*Xy@^F$HqqRWvJ&S(E?ox;2N*AenxixkmS^^UmzQlUsul;-'
            'cK|52W(F7q9c|5BUiuJyi?^&f@dcGdSkoByl|PZZSWchDati4nCZcW)8bgWvs>KZ+6icS~m_;{LY!XBp;;nng2j96LNz'
            'So^4`dAtQ(uozBEj3_DoLQ)``w~)86Yp_HKa~vsJp#EuFBy8h|8~xca9KY^U>0<Fzx<ovcipI>pm3wJd+W%{zmx&?<2q'
            'XSWiI<DoHH7W|qqy<Er>4Klx}q!b|5?zLT}k}+Qm*Ps;vYp^EjlLGbxd5ogs!kZ3YYljvCfhu{b}RRVyzLUSW(;(HP4?'
            'NAf8uacxy%LpP`SEXmt&Z^y@68sAXNpf+-'
            '^|CMS@$UiiQ30s|Y#PD;Q$;ibumjtU5g;U$Y=dy0jH2@8!3jSTjS7GCF3fxMKi{#%+`IGC7Q2!G5h%`L4=EG$igJLc3b'
            '!@tktvw6WBJ}*twFjF-'
            '6*rDMIHK!U|XtD#tVq^H6K$DI<X^Q4?P2qYU8W=0Iy@(U2IX{#iq8TchHUfmn*JK9^nK67#eh5c1Ce}Y9lrKD6Gb%h#G'
            'l(sE0?XUbmFXyd@dHxf*hm*|D>jOUn~b=lF)_RhQCtskjur@0k@-'
            ')SV`7A94iY*SjS}7_QG=uCA+D*7(Ugd&D1L~$g`q!(&jxc-3-jTc#+srQLkp9j*zj-'
            '@VV?tqT{a11FVytp2$LQaIh>azV#taz(vgVGU1D85?aI)We})b}AjB_L=z+IYbWB?`c}7J<hjaKG%|A5)ctXD>F&&SBd'
            'D}#>1H~C$7$Yi|UsN<dG(xm1@)M>xnzvn4RT=S!4~pW=XY)iOLnI~MIqeX(dPwvUr!R)n`5>B?ErQyKwqsF|9F}IsFwq'
            'nb6HW1m7ZxHqIy_W((4Qluax{g>V~1;Ym7lj$yd4xjwPy2#69E7BrII5OtcezH#6m(vkJ1<Na>bKFo~T+&dHLd^iOM9_'
            '<LwgfJH=HLb&<DQ1gMk|AK*pJkKyeR4QZ#oeLM2oz4tM^y`pe4NpV{QhaK5@^@;~Xfp`Ph`p|5~lUUo1t49<OBQCjUKG'
            '-J?IQ{Y>OY!4jaSFu&;=wG;wYb*2LUB;pmi?D5M~o1EyC4qgZ0=CnFOGTX{%lX;Hy`mF7cOiTUnxA%*Awg@epkH?h(jk'
            '$Y%tocKd|Hb299tR3yk3viCUbzK80-'
            '&9*kYeJ1ELjm^eRDVV!9S`<;!j&Yp*L_Hxy*v(LwBhIMu+Y5BWbW|zV`yjR|l)|hs1X;_EL7rXtkr9O@e>oDSGH+8O+p'
            'Ho<exZCui-5uh^|A<Tf-{SL1nGwy?yTlh6uJJb>-'
            'YHHy=vmlLrn<xz+?sLf$Nf(6aURFzW^N;0;)g@&4)H_(h|B!n;xjd*<R)o#iAS-&<jy(WDUK#O*Ob*Sc8UAWOpx`t-'
            '6;-'
            '>m$qM9J*HFKJxNZxL%if4@gD!Txa+DI@cr5)&dJ@;Jl(icoY9e2c>F%2OPsZn$$1yoDQ>aP#aaI=tE=ZV=azMdAO1&N_'
            'Wu?iy&z`MgxoH1KXT9`YC@NI);s+>KjON?O|%5zRiit_wcnlaFTSbKDefWn;b@0==|AE<|8H^I=51}2W?kaG`tH5r7I%'
            'tQIp;sQop!fNe4z2J5s@;T;u^O1pAQ~m($(|IkJTdaj#V!vfEOCg_lpT(TiV$2j)-O{C-GCN06RO5zm>I3kiUIUpg-Hv'
            '+>RY&=U`!PZNUk!;5Y=?1aTZVwn0HwmH`}d3y!sYpt)s$gRPB&zlEKR74K-*zT)@#)m^Df*d8Vh(iLq!V)-'
            'Fayik6}?%}_jQvUlC?LSv0Kh6Ttk{Ti0ll;dC$GUoB?hxp2Zsl)j9_V1twg_Oe&8^t}wt;LLJAZbNML>|9bwH3g+utsT'
            'V{K(=X=P;{WMOR+U@z>6gH50%$C_8swIqlq<&L?qo7C|xDYmUJ1wla$HddDY0Rh$l0h}Ofj;)=Yxt&Fzy<LD6+s@X;+|'
            't_K+}fNi)Uyw;U<cX+*xCkKa01PR4ZO;(oninxAcW&5+&af`B1OA3o<O_;)4x|>(Oo|eyPk0fv%roF6mC5tB7|EqrpvH'
            'sy%VnKei7`*&>&9CZ^KnxhB-'
            'F&7PdkD4tBQIfew}q_V#Q$j=4inK!DJVy`2Te!j2ta>tJKyU}a@147Imr2L;;r+u7I!I&du5))u_#u07^I4*o-'
            '{rc2A;Kgd4N+Roh4F38`CV_|O|XdA>ax3#nj4B}YXn~O3Pz_#({SO~K&^kT`eG`DrIw==i561uVq2;kNJUIhQyPDO=>1'
            '}x;A5GSWjoE-6Of_M#x4D8$#_=*1)Z>(c@^`e24+HIFE-puk&ijK*2F4DXP(SM>POt`KH7boGu8XYZKiBE}6Z|K^6bUb'
            '_cXQ4mc)fyXX>gm?ASNWz-i>!sWyP{vu|N0%giiG0t@8JCx@;TzIu=8lSf4+nd7jDwRIU-'
            '^LJWv<ygG9goov0*1b^BJ|%-'
            'ngj<9~do|NrGRXWm)tuRVyGlWfVfhin1In=grps|uDU940G$*HTXvl%P<l9H^iJ_}ImQx*?%QNv|<xs)w%tmyzv6etRP'
            '2k=2O)M>8?PemwPKmm?_~F^k19uEqJXv7i^2gTJ0?0Il&DQh#0otDp6F!*v+t@*<k9ms$s>PFKVFH_vg1`*2*B=1IAl?'
            'gzWsY53iSMF*_k1JBCx@sc-z4Us3v`^);cQJ=!?3moCaXFDpRM>M{^^a@qiD^uQwjG66ctH8*rnXYbL2)UIs{f-'
            '}tdhX6lpA$D>tg||md_E61k^P{*xQOo6FbeNX*or+jKg1E%EZAOei^1kr^l9cPyiDy+Z9Y_r&!045`IJxO>50RbwCybU'
            'Hba>jP@>2*W|}blJvAu9TaWRy>TS5NqJ*XWx&)6IA3+;|1l8zmM>Pe0M)nU`V0o6pfC-~<$5C18a3W3X-'
            'CT?^<NZn4P${a`z5wfj8X;?}9d-P96<(<>!sGLMGgbHWD4z3l%IfJ2TstHM&&g)PRv1gIlTsmiSI<C!cW-'
            '>d83Z0uQsk+fE|g6=hU(h=sUo_7MZar@VMcQy)BQcZX$Zvj%4Rb3nF4x^y@pOsiWK+j8+x-!1-'
            '1^|OPmJHhx3PjVRXVi*eG!i<)-'
            'h(X|eghTWtwL@6({E$Y;EA&clv)X>zGT7KU!sVz!k@Fdwal;qlfYmPY*{NO>cNLCJAgwY(RjW^|ff^L!3m(lDbMAGk4_'
            'COZq#FN|Q6c3V@La^sl5LwT@gjR){tuAn@>KeJ=b0XW%|3i>NILGKJ@DxjbROcpy*H|M-'
            'U&BDP{?x2Y*h1~ZrtzC~vn<z~cynih?`z8#o4bz}prXGce7!yWoq8fdzbS!<;H3c4SOMv0rGBWt`J(@2W2*$NjafgN^m'
            'Rq;7GTzOhc-nbr(7yuv?zO_0vlqaG-70XIt&OSGvl*=)dW>xGb(r-'
            '=9uM&;c+(~UwJH%<l&M0!>2nv$S<UeBC!f0d{s;CeSd6LzwxZqoL)go)2zkk}RE+O4I&g42O57X5FjFqzt0-'
            '5PsBcEySTLA+s-{`l-*6j@ke8vxR)5D9i!6N8v>nH`ri0G$)A(d^8a=W1S5O_;my*|ff?N9LqxLgXYQtv@+-'
            'bIg6*0q+<-B_bG0WD$X7ezxEY_p;lzamVqj-'
            '2qGV$=V5GGS+2eiESgx|9a;rST_@~F9p%v$*b2VC3)!?Pcd6V9Izw_T+tE6v8<>sB-'
            'G^{im*&pLAdvobYKBM&qO(DadIrO<zLZzk?~DG4{UW)8UYU<|1>_-%6^aK3pCZY7=*+|1EHX}%-'
            'qPuKyzSG_AI9z_V96U2lMnNvA_bS{LgREK~^VW{EpmQ1eQ1I<60@i?kM&vlvP!`@6>=TZP!E0^MYuO6@`?i1|dULbysb'
            '#S0If&Sh*j#*SUn^ds^DNlhUv(I5T^PKk$<Ew9B6DdJoegTZ~IDoy)RdHLH2_##~GlzyP!jbhesbzfu9r7MxbKNldp4N'
            'HLFFTDIv$M%Nzrh%~(u~PGrAjS$UPj8435M)SBd5HNg7(5Oc>7WhM*dkn2F%c4?msWX+~LC+-'
            'N<)j{jOl5v`U*wn(Kult2K$rsTflG-AJH0s~O+h3&(o)bA0S2K?Ruj5_M`Ar8cS%gI?5;t9>Vs<qqMr`_y~HCZH7dMYE'
            '_eio4<D2uZ5o+X!q9t;gOcB^mVl3K6JCd8fCLc-15<Ia3GUed7p$$ym}90VYdEF%222;22zwhk6^+<J7E}p^k-'
            'uy*nkTR<*SdsBnc={rU;5=)q{Frv`;4z&MU9gt!a+@MZo7*wogGIbW`fD(gPu&2js1z@{SlnY$&dnyt>1W^RCcnHM3wm'
            'lJc|(tu7~m?tps)Mhe+cG8uS){IVz9b=MQia{g}H`?}q#czyY)<Ij+FSrt0uGhna-M{dgUvFmpj5E-G_*)zkd>a(9-'
            '$KcX&2*ZICY|Oh&1@X5L#24xQG-'
            '&9L8V!^iN>YOjPdO#_1K;{=30&0qtnn=>n_Y6P(fPyWI>{2Fjx#(2zG}wm}55Q$Y|r$#Aym(+yM#3A#w=A`jQV%%;Xv8'
            'adjj-'
            '|1cgL$Aa*tm$+g0Q082PIxfAk5_)ZD#SrCbg8XrWx;yO)?(>?8>{ESd7sr`sU2Du7nCt=48Fz@$0$JAbS2LJt_oJAMdk'
            '-P^<Vl?TGZ~#uyu~|Zh<oDV!99+JMR8B@`=B-'
            'OB2g2b@DSbQH$$V>Cvt6!m!SPYBB(wez>Io9!Oppd(J70fzWS;%8h(ViYJf0)<6bPcv}E2#qym|%L{(;}G7jj=^xLY;d'
            'ffLD9+mPCUKg{8o{l2rc3}|oc8Y{T!zFMkZX)`=$%OBmDjX1Y0#`3jg@7ah-EWN?GxE$0>_2=6Q93Vy!Mb|X+-'
            '*<jz9W9pi>iEJ;K~MgHKvjt<ujT&eFBJOxrU%m+88E?-'
            '2<Km^v9TQS7G^qGniLuhGV0{a93jq7T$Rd8or+~>w!G=^5|k_v*a|!$?Y049u>IB`x#;^!jrlQ$d>d#zi;8t=jcAXtMm'
            '&KpRpN<8R^)#wGTsGt)V9bmqS=p0CQsO0kY#}Ed+;{V6Ww0AWZcVE0(e#2Mo6&&vPTaZ&n&?_HV*`pR-'
            '^(XfdAOWQNm2M^e}Cr{b`dp(NeHntr`Of+1mxpm)3klm3eX#fA;^++_~T1<RgHl$R8a>wN;^ZFA98;Wa%V_PgNtEK5f1'
            '(LJbt=tS)kgwq99A>bO~gDKwAnEp-;xch`Vb9U}e<i{34eqJ4<9jJyUC+e`}-'
            '~$ZtMvOf21y}T2N*Wg~L3O4V#aG%3LnhoJ9}j<n<8x#2&PjlL8*faEIDzN)NHIgTRzvNr-qgq9gJ?Q%1@5MOm?3T~NY_'
            '%LWI3%ke9;FOSN9f^c5V}npR?#PpaFNCa-'
            'b47dPAajABuXGkH#EZwBOu<>h(POYN9N4$aM&vobwAEcdo<s6|`_Km4fq!rQnzBD)LYYv6t^aCgF|_D4f58A=X!D`?4R'
            'fEaD~@Hmgui7&T}sX+p;_KB%933;m<(@Jp*KiM(GA=FbPBoV5dPl`Ep>JKlwUsbQ!Uo`G^6N9iT6T2QL)4VakAF#e~mp'
            '>^8@@_AA<?pE3g&n{at3190f7f3Y0$8k1H*0Qx&c=8k+{MCypw_Qa)onb(gjvs|XFJ6SP0~mZBCnK;Jm<nl>3!F9{#Hz'
            'Wf%areRq&Ndd;M?pl*z<Eanzkl_g={FC@JfSI2mImNi(X`bn=7^EbSNh19ERE<+YtT6Fs<7>$>2yA`i){U44LA=431ew'
            '=}(d+pX1dS&ZuBG&@h*xie*s>V<^g8m-'
            '_s90M#Dq2SIKx;C#tUe6`J<YFKMU9Up7L9Lt(1r~_G?VYmxCBGlo0iy`(;Sq?Yfds7<9eV7!~WiIp}wDEF7e1EZprEpo'
            'B5^U(t%r04q^uPg(UCl^(Mr<A?wX5L{xj0BX7!5WvXF%mdcbw8=JnTPbi+iW5FpDO-'
            'p;i282##ap$o=8q_%TB8WYse?_*w*OcF@GSxS5_gPYX9*=t<3TZ^g9PO|V*bIkcZ0O82juLrmkendW0DWTWOjJo0rBcn'
            '|iZPlqcpS<@%rilYOlQ}H&SueguyJINA0FIht^C==jQiCio(+(8eR&7$1<8d1UAJFs%U6l37r51Gt?RPUl86nFO}l-'
            'Mr^vPZO#c{L0RoMgaGaTmTy--'
            '%Ak7ogenB6MU)LvYdnteusQ1zJC#@o5b_;B2I4C<VZ7wbke|?Jm({ZKCg;I0jBj9ngJ78ELBB2yY#0a6mpom*(`O*gJ2'
            '5V)%Oec{Y|x@Q|cj?>aJ^(*vpNOHX4=i#tw_??FlJ`GQvhWT}~-'
            'HNnDu9Sqg3rrXwMfTz6#H9g=MZai}i`u5r&h&y0~ms1DRi%UFkhwn|4+9nH<W0s)aB?fh0v8YXjA92@??J!MlBH_j*z|'
            '<AJ=!w-'
            'jR2ywTRmC(Q<E;z#zE*<LGI?lb?}Ji}fz+)n_i%{mM!dVhfEnynjU!mo!E?ALzST%1i*`m}{k2N;JYq`C9(o=$PZ8mD>'
            'PJepg)s+(W598CJN<ZnFLZ>hc*3g!5C6!;Tb3*Fns6OG@XQx!;r?QTtOb)2pi&vs+nPxkZVyN0>d`xLBNktfc04$GImJ'
            '497)=bX!@Yi6pefh`ol_(kMM*ojFryB6?tQ3JE8dcLHF;)cOqHWa+c6ku&%%Q>29Wr?H*@#-'
            '9yC2yfxAdD;aBx!UIiS5BYC5#F{3w=!HZ_VBj0<d(JK{Mv!Y;31;RKt0qaZiF_>y?L_gg84HrL|OpeQIQ*{|y%-'
            'BmvZda>PZ^E@HC8u|gr>;i6=PqOB`07$Y3Vo=p<S8b~6oZksBJ*I#J5aD#1ByM@<5wRadV+Zqv|k!UU7n-'
            'M1iIxAb0|bsYa<AZhM-yIQB1$qpDEIcLpfqVm0$2@J}>!7G!)gCUEyX_{8U$Lm^TOFs8IajxesN%-'
            'Le0w9NbmI#=fS{K=yz<R4y{Y-UcNo6E}=H5PBD&#*!H}%91I~(WUM<y#Q6rply?SP!C^T$Ao9nXkK!PBs9-'
            'paw9fF*jpOD(elhV<Nd7FPagp7F_wN?*a~|W_F-00aX4pi1D-l`0KX}6nU-'
            '=TW_zUtUP}K48|EKE%s7SV<ri^0%N?@Ls8Gjxn=%JpD>3hF3c<974V<wa%+|Id`gCDW9Dh6<vwD3ecSa7SDy1VRt+$3$'
            'u-0?9f-_j0tUgqIKNj?S5rD0>zgW-9F2mAg#?04km1ydB2p_%Cp@Qq5!;iui%s%oG9ju<B<Qo%wq-cikl4rtXr@qV{^G'
            'YVExd~)P=R;Whd3bmx2MV@FgG5CiFyHbJ=MVLVxmPTR|BCHUzAhA^Lh?X9VmM6B=i!V<HR}7*)sA8F+?h*FOPT3;(u|i'
            '|sUWFG46eK;fN6c6;<<8VMse&+ye(WiCh6uuY346zoBkE4_A1O>t%k-&ov666Us1zq2-'
            '6y3L>)_rgYaM*Diuwrj5QWGvQ33DSfM5yOBU4137PcnN5u|z)vCa>rvz0sU=H<p`c3fg8$sRSDngpJBK8?I8T>rfVBNw'
            '+7@hnG?@dg_H<^Jf&H`g5+2{c57&`>+DfEZxjBoh7z6BW-HQdVzWEu(`nNmS-'
            'oUOs7w{DtbYpbfs<YdHPKlPr}`uY0ww}2q%YnTqa$37(I=W14;2^!R+0V84M5j}W4QJ>_0%p^*oK-Lux#<zzA<de<^`q'
            'qUOy8WdjGtt6_2{q!=8!`@Io$3KrV2}r6$}fhSKa{Btt_74z#tx9pEQfLZ^Po^Y4!tiZFqO{A@O5iDyq!D`>SIgbVM`C'
            'p+gJ`C!(^yz!CuVvipN>yQF!$8PvSn|3<lYlQm?$0fXw_|uwjP{6_WQAv)u{B&G|`oQ8{oXM2-nuw-'
            'j`m%OGjVLyUu6_%g|evK8zgsltnxucl7Lt~>%c{IgKmoDMg}BnZ@U-r>n><1h{-'
            'sCBzdU|#bHdQe;ezRt+P(hLcxnrKhTVjN*cW+C=}av%GK_G2V|9L9Z?yCCgdI}Qm}BN0p!9TJLQwzVE-tewQjNy=dR;c'
            '&W!(qx(saG3H}Kk0<)@|3;93rFL}O4Pt{rXW$KMm9Hp!x5W@;*GD$#2}zQwQY7D9KJDzdA0v4zPvMr%Dt)u!SCL({Bz`'
            'S+~fop*=WkhR(`}%ZEdRj@p1ZF=3&8^;&ot<+Mjv<K?a6d8!`6VQn8(TmwuYXL+5f?I9eA5C9aahFWZ%|d6R_^Jt*d(b'
            '_0BFZKZphaD(_yl8|a1NW2uHh{R`QGIuN&w?6(xw>>O|#aerz-<nexZ+`^@4;2~h^2NXjZUkBOYv{Lhv!j*g0A`0-'
            'A$m6yVT#*Y*0?*PVEyAOsQvi_^t2sD$qwsJsVEYh`AL%by81YnWUYZ~8@wy`)g1!*Rc}GB2|XEm`UF_L8bD#S9HlkKo='
            'kkAz(f}AfkE@E;0;p=SrYMJf8q#HlwN|xb`p%=^m;tjwhy+N$`eV&DwrQtjD2&<G40eAsGVDkGYiHrTO2nsq9=a?he!;'
            'FSCx<u{r;~fe=^;l{8>1UmXMfUw-Zb=pW(r2J*hs8EvRN2K`#r)#MXsulGU~xa`G}@XJNVUo^>Mn>(0js3+2(mD1~m`W'
            '(S{izTgBSbxduS!u06NAQikBYc6d<{~RL-oL`F*!yD24mjZL<+cbgXhvVenr(x99^nA4Pcm`Rfr$CT+8AbjZTiSla3mw'
            'M)&nK745@Y_uzq#}1k^es)U5a0lIoBb}FoB+6>j`VN3a`C!Wq3AsCsxQQ<K2<Xq+;0^Se<7FrMsR%KaX#;!L_sGhxcas'
            'nG6G~PbY)pgjFQVGL5+T9)`QV*0@nw5e|;ZqP?D*W1QMDJbg<*c1fDTJjF8@ExZSu>B|+oG|Yjr{g#Y=KQ_8$9)y(IAD'
            'C%b2?K}8vVJYjz+8!Pu#7B&z0?>O_pyxrP?UqM&(>l3<Wju-'
            '{R~cexSrfDRl;L6skE#1NqXwZSHvRfAXL}q)BJaNg0Q?k7}$Is@3hVbyX56)m>57Fwdm0Gxm!u|k3_ty)`|h8Qw7>uZD'
            'helPt2cU0BOE0tj4vU1=0>{afxCZE^@z(GrjMV4Nvz2>*G<fa)Sh=kbMnI-WSrFp3fn5Lmt880RlKTq=vk|{TdvfNMYy'
            '(OL*&?#M1wsM*Cbe2X{X=vSl+(7o_e%hdN7qYtTe*|N2qjGcXF3O-'
            '|D#_g~Y48Vd=#z?ifhL2S8f4jxL4kQr_V+ef{Js(ahfwbycD{Ax40y;p``LoY+IPj=;5vsSuhKqkJo>Vax9vW)bB%kX9'
            'NV*2Li3;1$DER21<7-'
            'JVHFpYv%TJDf4G*n!JpA%MK(la(RS;??|nY_ni$JW5cFB9O?(KcGm))8`dH?po}?1E{|3Rq!f2L0*VmBlSSbnsafW~r('
            'Sx@H|gkDZ_ByECWbd+Gr(8TO@8$<~;zGRlJ!F_&O>^9KAm>?&zE_X@{1)Y3u4d+>>)1{PP$hmd(`P_gPb>Q#lqp_Er7@'
            'W6d^mfj9IZNc#US1oQ!m<p$LXeii`j^UmUNv|&|l%36bdSG%f_RFp(^TJ2a%*{rYe`6i(kfZ@xMq5Z6_XV;!wy>Eu1j~'
            'bt;HD#T%+G^8F{An(OquLW)PpY|XTmtrZy7_o<QQV=G6`zqDm~=HR^uGyNW9xR7bUOQqRxa&wA^@>v_*d<)^`uX!AdsN'
            'Wt+i1)8)A8?LHFy>l+l?pF;M?9d!AKcTl8o2p$<-LQmDPxC`$?Mca5ZnY9|H^-^L2@HJ#5tfF6ey@jRglF-'
            '|xiPVk?#>BBV=*pdQVHoVPzo_Yib5z^uKJRVeYPJW?oaKXe0;iC-'
            '+uxJ<Z^B9Cx)PjG8&AJYxJjFF&(Lui1!R=|ONh?1BN=`dp)Iry*E_y|xMRC$1&Pt9caVphOmpx`KR&#B`cCkq!VC7wag'
            'aaf49Xi7(L=T!g?%^}EYuUQU_cs5r|(6bltOGcd5qL6-'
            '6S#(++p^j{uB(Wz(g$>YEDxIt}}Ij(wXZ)x2_J|?d4!YY%qF<Wy3l9Ok7tzfZ~1BfjwWn1t#ZNcy;v*SQlCiwL@pp&tI'
            '53{&JOri-{>PqH-JP&tlN;WFr0ggEhuXQh`==HS`+V6EzY`u=(3DL^CeA;NnS-'
            '*=7o7Pi+FJL#JuS0hj0}7EZ7<xi@oW^h-'
            'LgtQU!?pvgtyp5HV0IJu{93PsVv^=hUTopjj+4pVDM*<OOz_Nh_}6t>~LHHDa_{SxO7KM&7lacRZ4*?1<i0zb@(B!)kS'
            'LDBdHWbv)`%9i&E0=<g{jEYeWX?{__n#6Gdxtpgz+sqMD*VI7!u?nc3l`LrA6OWB@gXr4@jqq01hQ2%O2<wsLO_=?i#+'
            'AC|a9^hyE`L}^f2qAo7HDlGu``QdJkG`Ky8^(cM=h~=Y7CzH4&kC!HfGEzz{sRjxMb=KU*l`x*;9S6H9Aj<;#cFx#?@r'
            '9rzvq=HVs=+n*>)@nqc&mLDV46*F<K^A)+<^3hh6~A2(O$;Ja)K!t>Z%`F*fAe&4{yQR5!tq#dfH=9nkV?i+?<<!prBq'
            'sTk=ENmTcj&=-P16$MlaId2+mZ@-'
            'HWxE4<tB=7Ri<Ud?{rmzd7KB5?+W?rYv<XaitKo`(@!)gK1b)WdcGR=Yp>Kq4!{(;#IC=dGl90UzXC5|(n^qrLwSu*<q'
            '@ff~j-HDLtaEUyY+puVMZs|!=^teB@-'
            'jNR{wiEjIEY!VRwJ4A9ey!)vB=PcxlJ}<Nc>dBJnsu$I_wG!!u3^sOB0@~0H~*w7>V0Apy7)<sB^QyPk9|S9(yUU8f1z'
            'y+AiUIy$7U3-we-QvqQ6LKSCEA!kF`N)R3%j(6twUY*w0}DAot8kA?|e+7E?eBXiMtrW3d=TqN+?cLsw$_rv73U!e7R8'
            '}UEP#(RB^V4fEPl9pd#bk$n;d`gd{+s~CQ5sodr;$pn-QHK*xZoql%6g}vP3b7v=MC#feLCd0}pmJg$e$*?)3Wt;UFs3'
            '(?RayuiM|r~s_Hm+SsY{hO%*CS<Cc~svQ<0jV2yJr?;_^jmke3<^Uyj6qipz4G+s}(0bt(~yFJ<7UA90xI*$zj~T_Img'
            '_u#1Y?=h?~72nMsK?mJ9g!`|qWmP-MfZbCEFiHPNm(_p47t{9%EDxU`bH+`<k{27$VMhioXnKZwsw$DaPZrdtox&sQJR'
            'v4mmXaP~h!gK@6+BltOl$aA;2xK8nCn$32&`L;V`B5*iZEZpM(L5{UQ?ig*NXL4W#op%kxI^_YD{0jVBe1iFsp38<6f>'
            '79X89H*1P|PjFV1A1+!lc6E^2ydb6LvR8NU=i0@DGmnktb#~nn|GJjMo>`foG8eKsS(m1g_60VdTg9WiC3E##HUCZoIL'
            '$DsN;YM8g(FW8Fyy?P&QE+)Y4~mm2$@!tNu<D2=b!=V**?V#X+SeHga;G*D+2hY4IXf2m>nCGXZXN1=xJplax&ddt^`t'
            'j7<ipK&Gc-E*nI(82PaX@(@#y^!oRBIHO=iggkMy-zHgqj2Iv<3?Q$GlP9b5!1HYv2Gs~WCsP=$Br6CmJ339XZ<1r1xy'
            '<LBNbxVxkX`$xqSvvyTn8(l#v#-4&@)84{~RnzEg5i?-'
            'F$sv~CluW#T$xCqZ<VjLAvyQAEUQIq*Tm&86p>%0(J{hB~OnMs_q5P)pP+7SVPqh}|mi1PUE^mN`FZUFT9I};ux%wzAr'
            '&x#%zT-'
            '*Gtt=e+YdeshuJAN0i;NHTA!nuv*Sx*W;5lOi3?I7>6rS#dTVIP|MZ{fb$=|?wQTLMsuF0jnFYJNg`SmDWwunS!6v21F'
            'H#$^fKgNFUgInkJrK+ypBtzo|!%-PGbbP-F3hr-'
            'Y*|m;>#!1&9{o{Cg)SdxM?Ojv2)32Rejrs`BtmBY5XA3iLZN`0v)q&+y180wnp#$gSW5v`OfdT(0R=;zFd!N+7UaAe;>'
            'V{Q*sG~99$W@r&wi(%H2V&{FG*;YuO_IKGHM9)hj1E5xaA4hD$MV_1aC6*xa<w2HZd8QeDYXOe<(C^>X_X}0gC~)-'
            'KcY$Wt+n`NKniG7CxP1NId~=~k@Yp}5hB?`r#F@o8}lA0@VJS(+i&8x*|o&^@^SRh$-'
            'sbq^KkXq&16NiGVvU7Sy0)00}lm!CueR3z}KBSvC2CgmvTlz+t(6OVDf_)Hl{<-'
            'oj{P;@`yZikwZ8$6xT)PL)x#NjG>VcygV}<Pe!ou`_!vwrpCdKZck9sr<Ua#q`}OMPazjG>R5edCS!WV9`aFkCkel|4u'
            '@!7L6h54@%E4XxY=|v3<+M38ObduD7p?wcd|iyUlPc=_6C&?IndYAnYMntkB+;1nJ%||L|Vq5hWzWkU_3YxS|g?q-'
            'z6#da@{lfFg=pQ7B-@OQ3<&E=VDk!30CZEp>L%i_Re^WuA?HMYHA+7zIl#5K6NRKw$z7-x$jt?CSStB;s-'
            'E(s3dA$W<kuXC>XVI2kQ23Ay)Ix;+;u{P-'
            '@^pT(d|EQXCj052TpcQ}@G8yA;gSD<P_i)##jk6_a2JDW1~`jr!>Tf+F%bn~$H*55W(|K7ju=8IW9alit1Z3LfxV1INm'
            'aq3f^3Bv11fk(956i`ruWUyUS4W<&0*F#6}ULfY8G4wamD(+6vdG3MJ^{G_`IMmJ2O9d&}h*n1OP;~XWISShsL*!B4BL'
            'Oy=fafLD02EeNVUEF*-3O1@+lGOL<P+~5?(REVv6tB57&pD2qi(`Pb{0iCrsD=!CwMVe=%@G#o&Ia_J-H4-'
            '{Yv8%=0<8ML$E#T$@F^jjeloI#RK)jU=6<wCm)d^dP%%g_)>(;FYaEBZ+$!8L<_1~6sSuTAgmZoW;dJ@R8kTauU96KY3'
            'k88vz0hM>GN_!>LdQxDe#q)crmxAMZ`xJBDLsF@Tvd&REOlsHHy<tpg=34nH|_H_olf2NhIYHtMmzLe3r%*DU{!4|+Rc'
            'W>DV2Hj-NRDEav?(|XzhUP$A@Xlo8us>=f2}t3zg{JJJ(^+s)^9T+@}YBkzw|HkA}$-'
            '<1q7;GUQEcgfHtjIJiEPmh`hAFQ*wm#-nmPZC8iIk@YypJ{K0~W}}|&De@tGBvd@lrN8xerkl1-'
            'goy{LSOX_+MOOY<92O@HQ<9cq`Lb2mK5ZMe`yB+ylg+p&>p3`{l7shVZaAlEIF8wkj@g5EL1~!}`YI%XvTF+Zr>=zl7c'
            'b#V`E6i%vVrvEWT2~-'
            'KjtK@f?)Ry`k3Nq%t>&?FOlgmZ`EnyrE?aqkcY4_qXOQqc?^S`Zoo7q3?9u>#pb<bsI}V~@}{D2O>-'
            'f7Q~(VA8cKAwUBtJJBk-F=4SMDuL5VfBv~Thada?UXGV#Pxcr}B@$tEnIUcDk|gH-'
            'V0$}}vS6GS^Tej<Inx8Q}$B>ZwmPw=vAD@^%dh+`*iBau0q$pxo)G`rpe`!?m{`T8n&tyYbvO)q1?+|6*@Wj-'
            '#CjRn8aGH`syO7fGn9cO=h0pU&FFw#qf@mDIsfywUl*sbvp%|_UpT!uY|4Tot7RS@UpfbJMW@9H@OzxU8V)sxcjdgTfT'
            'NVJB7JM@M7fX8I<xg2`dIB8I_{6#K>qzOuP^(6O&_Y~Lnm*OkA5<KR171FmCq3suKayV%=P9L&Mz?+{*HaVHF23(ecH7'
            '7JN@Mk(*J<8g#@I)_|7N3KvA9p}?@h1Ex{SNIu9);H~>2NwQ6GIJ-'
            'lE^bFNL5=Jvd-^=2doe}GHD}jby$UAXO*Z(jWjsDeib}h<A#rC)I##fK-~ED8=lWhBNl7l2}YkOV>yqBg|vk-kTuB>6&'
            '_2o;>Y}g=f%lzy(odMQydJ@OBZ9q^%uCv?>gi}_<?#;4tknDhx*3{VbnG?T(B|`PmE6lY_NoELmJZhn$XA59ir!D(!JD'
            'cVcd|__@MDSaSO5{J)ZW4VT>M3nY9RKB(8&9vG1TqLm94I+6ao5F467z`$=L-'
            'B7|5}(>)SS;y@2CG#{9ceuoc0vQ`Y7(d!RCpC{qUl!p#h+7isUxjpGRor^F#!&e{?&>w<+{GgqeMX;isKaiV43ZNl(Fi'
            '5-4##`^_K!Le8WNkkQk-'
            '3^!XulNX18<W2vP#l!U`bz=)<HQd;ht<DAv^jjz(@l#JTPuAR)yXnCqDlq%#)4ufm;<cOqzqmns<QblnTm{SIFM!L$SP'
            '=qK9v{fVY7usGirGyjWjKhV<J2>PFI}pL!EmY*~YkoF-z;04{D(-HCSI_2gW?R48cI#J$NC(Bst;`qqac;+wSww~-'
            'IvzLP?S%xt>wb^v;NEkT!<#qg#u1MuiWnDRUe`m2pY!xJCqvb!ZDPHF?b8=Hl_vUXy{BQrX)PXfLu+XmT-'
            '*&rAE$x&+VdR(iWObUZfVevIfaPZmzw}kfz-wV%>{p#bPPrrp=H^qc}FY*C1siMkPydIR>>|FRFQ-tQd_u<#IA84(!`M'
            '9FV9dDdmK@RHGRi;Y3gpmjPK(@<uQsHWaeR&h%7ORHlXqM9Dixy+VZ5v{5ya;>~%4vsrTgb%iQ|KSJ7o(H=EA)Q+j%Di'
            'l9P1wERgU!c!~?G$U|wbkzEaqNx+zuQcSZ_q_@BUaY!m+2v>zN2C(xDwJCWB|j`NeBIv&_?m+16$037y~c%1M-'
            '&9W9jYu^@}5<WzDjkAvL&$)!VJb#d7ZNhbaYZAHy--dl^hoIrWOt7=t4N@a|q5rr-^1-}-'
            'zNxYiFFXjuZ#T<en1MOYf6zpxs!hW1G0F6#FNPS}b3YbMKTl4noFg2EJk%bU3a8~Ih3kekRxyL|#C~nC(oP4H??;e-'
            'st79j(@>PYx6&ASpq1E;Q*}0D@L`Hbje1WW%x6G(<pDZm?M~=-'
            '#uDdE(53qg{6L@C(?Z|8vIOidr_#Eaxu{|}nzcTl5@Zub;fsm0u-$eFI;*Nd=_5z-;CdK1aCd-i-dXzm*-'
            'TuMb`>7a?a4gLyM!sVi&6T?wF+M5eHakD3xDogh1aGw!3M+L%#XG9kemCOykB(|BNp|pT<(4v^%QLJt3d_~s#L@A;7e5'
            'Wya=r=N6lU{2_iVPp!+L>o-ue4I$f{ClMCEo!Osa~4|5%qo@An5<_7ptD-9vW{&-'
            'qh3Swy~$dl`jho|l&i}I^Mex55`aAg5VsH}zzgLN?cLjh~X>k#nJ_s7PVYDj)2z&Y<HV@qiw%u!E;)lI25diq>=>^2G3'
            'k~@xH!VPfdJA1g}vKzm}+QNI^AZ)WJLATpASnz2!zRfj))}A34SF3{l%Ss$o-=+w%@=D-'
            '+B~5Dz_cpajT=?<O095W)!Q?~6WYqHo^z3)q_`do)J?pq1dGqoNOuK)LjO;lA23#vg9a|1D=A@IO1|{&EWYMD)Xw;x<='
            '&QD+FnZ)}h~uviEOgdG#Vi639`1oRZZp7k(je5hDhqW<nzVJoe0*}R29BQo2}hO>0**{SSbtUtNzN(9hrbq~`keyS*tW'
            'Z{S8f&Fi_^r}4J*;|*mdw3LP5<EOPsDQhgRw|MD%Wj_*GgU)4U!we>#e>GfrTM`a1mPnFiiX5$GjlBDkXV4tUo;Lpw(k'
            'RM+04{o0P;qMwQMr1RTI%)mHWD=`u8a_j`YSMy2!-HTv5?*hy=d;xahv$1CELgE`#jmr9Y_{;MtAy4-Z^Qk*QWt%owf-'
            'k&Tcn-If*TN$|Yn(81HT_gc1?BA?08wU>1But^bs0Z}>-'
            'jo?zmXTrT;_qsS^2DRuNF+YID_T|q~Vdi!?AXXC$^8N$6Wk{SFMXc;?^%(&2>Cvo66$n+lS!OvN4b%=>`YZQbhhNhvpu'
            'j=_R>ypdrVaL|vJUiu>E>KHsn5o8rB=p(Ky&oiYfrXEp;PqX}>J%p!`X^<dqeS$ONz5-'
            '1$?0hcZ%Fn6#%y;%1VS>a(tXH+ROBPXsU+w>#wJ|`K{LaN}gK|ko*W{*{?i{RU`73fsAft2k^L5nXfWHvVsQ-'
            ');__4X>*ocWIaaC8GbAR-'
            '@ZLq`&=1an7&Jrh9Mdkv%>UJK7hYvGRCYl7A(jTki~hc+JRMC%k<!X8g4S}y)RYgsLgcJY=lXX-UXvlryVq~W+IVHO1D'
            '-KRYVZbZ!l9@M4gK-2pxME+5KRGcM;3tVrr^oD&UmoFTnTX)ZdDP?!ac-C%o-dRpxTX}=tS8|2+l(K~q-_e-'
            '<{5m!apy?YER7p@~1{xhvfgb_3cvAl|nS92dOf#N?n%hnY<~*~9ir0CNa3K%e4f|kq&mznS=n3wDtKfn2E(~;81g$y+m'
            'E|>;VZ!=ij2}~j<HOHk!GVjwJ**26k@}#!!4f$RS)h<&=jb!TjE-'
            '9t2QG^#&{{bXw;ZKu^_G!%ca{s@V+Wwr>X{_&SsIRWPr;*G#vtd+b)qEX&Fi%l4HY%;z|A8hPLc<bhbBNkk4ju9wUG80'
            '{87-Sg^SgCT*v%|Bzjlh?daTCj}uM(utsS+UW!mettUk|vS|lBYi1s+DT_;c*_GhFS#DVNEFCYcS%-'
            'aw#*pT{PH4Zjgjnrc0&p?3viLRwe&*L9{PbEpfa+u{>_k<I%XlP(>u~!+IW~QB6A<sS;F{VG73P@XHtuGu-(*0$?-'
            '4j^ycmy@dyl}Vm-pfH@j<{-P=HKJby`QT4$}@TB)v#N<(b3HXe<$fS05HY%JxTOp2|X6cUcW8>zqMVmoGT(z9sQX-i_+'
            'Jba4N?Gmzbu4MV(W7#Xq{_bgL^+k@Ib?Ort<zV0q08&U9N<VDa8djL1}jzMUY6}i(e8kJ^hKm@Yk&i)>#w6z|TclaVF&'
            'lp#fN-'
            '!@!=+j4NGkhAjogS1R#UusT5(lp>IO_RjICOtJY?`$V#+7U11A2#G`s3pmV_yz(OAP2cTaQCiydNpKcLZi^9Rax&TOpC'
            'F3q6$safbI>+GRZrxz@8_j>aum0EvROF-efrdlZDpo6zZ@v!Ji^Dp<PiB+8iE;^uLSpwGH{G)&2GT;z3{_@oq(y*soW8'
            '-rOGSs%)>`?3dZEb`ICC6|~EJq91j$AQHiGf;MjhWXA`bbfRVT-'
            'O{33g^?Hhm|%M7ak#Zv%?)sr&17;wh52rx4?@rZFqWqA_nD^R@y`a!SQ46c!qXCuR}Vl(u6A3%h&-pQ9}mQmP^y>U-'
            'IZI9WQjB5(joGrQo>oDvbB&B{;3L3R*qu;M(a-Y`Sv@?F#ZivhgNu8My%i84W>0@k-'
            'E?(!u3FPSV=xE+jalCmxf2g!>A@P-&wIdaN7{O-'
            'Axi`g|OTG~WqR232ER!F$?!o*XsdjCy6H!x(71f0X7&d6H8v>M(b65@_vzOZLpsN5!AZ$=C6VLBTNF;aKn<&?v2=w-'
            '|ch&U}5;n?DwZ=k;M8MNWrnS`Ik;?mAj^d=~T-Tqf$(6-1rA3Cm-O=vm4Xtaq=)A&z;F6x@a-'
            'q>QLf*#!H?9D@z=$?$E|v*YHM-0+3xK8&!`!doljaj?=8dR@;%bbTtt$UnRe24@}6bxAt-'
            '`AOh__37|bkOVoqS_MsKk7MN3z1VB{DPq23@bP9TLY7bHgFLr6WWn9R#HfEgedcr;YD_zeGoGXh;=hgtQ*BKsS+x`lay'
            ')2L76tLeu|&#z1KQSXCJ)Od!xA4Q)>w-'
            'HaOc`P`f=G~*!<B5>lVw<pUtZ9$hCcBcD_GIUKW7vfYlD38msAPGJN`R_E}uDdL(FvUSb_Q8V@!Dt>BbuA&i+{NcQE1;'
            'SD<#*qv5K^Q!0L+_z3}G1n3uk~gAiubntAt{+y!X)t+~6}YEj3if_64}F3PVV##F%zH8uJ!UCFvXvxly#6Wqm?XS*+{1'
            '95>rldJsmIYShr#@nDzzu5lpg$IBV7AjPtdv*GCp3y9l0C~+E9a4JtNVpWiJkX(gQ9!6k-7F180-'
            '3kiK$mxa<C7NKicimFG`mSZE_{BojgV8B~DI=0Fy2e7nH+y9O$(KTUe16w@asWTX7xRghC(fo^lY(n%~|)O#5O0m(yAH'
            'E1FZv$@G?A5uo{`dUL@ml_=Uax~^&&!vA#0F>>r0{_wnFx<!x%k*bM^qP%y|A#x#GF}CYf}Wtq*nIluqxD38uM@ehpo+'
            'cTFJ~Q`RDlY%)=<1_J<9cXg|&9~=(#Ii(@P(pspNTVA{rCV(MS4b<H-^R5;i`<k$nttWc+MQd-RrGcu1P&Z%-'
            'iYIWzIe{Ra4XRtB}F4uiP+&1A>%bZnTJLMERugq_OIXf65isFh!fW7CZ3owv`Ci22XS>V+fmg1Nb1&4LVQc2~yc1zOnm'
            '`ZU4TYsRSR-Gl=c3__hR<rs9l2956jB*AIN(E55L`tEv2QXVeDf#)yd=-'
            '@Fpd07+j)UCl!D@S2$r~}^Y^O;<TTZ0P4Pw=~nEpnUgW8h#p2phH#^_>!Mw`vPfXrG8)i+pji%TJbuB%4l74#)2e?;SV'
            '2Yh_svOMo4^`tbUADHzDk!2=T(;QaHW1vj;az`1*;1uJT#$?I}EoME4eZ39oB<DR+LQqvpjZu^0@R}y?}engwzxsNd;!'
            '_lH)5?ODbN_>}Rf^=^N&HDYI>#n!ci+4%lFZsSW>G(>BINN}quAPT3b_Mu8G8fFGUl6&;gW%?hG#D{$G2HU>C-'
            '^xBuf<xyyvGcY{Te_rj@%<J9wbAEd;>~-'
            'Hv!|FO=L^M71ml){mPyD3}NM453DZ;z)LQTberNi;<dXTA|&=hX<#b!HGWSXFL*@SgUwjZZT2wuMTx*-'
            'mKo~rupt|0Wu|SxE4-Py9SWlgV3^xgI?+oVePV-Q>kezjQ7bm$<#8*}KwcKMrA)wn-'
            'z@Oh$#VMs;+<rwOg3c2WJ08iK4?s>hwqo|K$nLkasOy+xS<G~d!NaHGYN>%N5LXs9O#&zzy*6-'
            '@r2GRQX19*qfV}XUyfy@m%l1rsm&40{*{SKmn-}~*3QEp>o@xUW)hhVq#-'
            'M%l<fODcO_*+LrG<pkxCIo(y%g<$R61%L^k(z?rahfO^P&xQrgS-`F#I_-'
            '}mwT6VBsY*E!GU<NbJDZ=JwKE0|ZCfitctSoBMQ!6@Wq9p!U}Y3Bfv)0{#zPo|+n#vQsScQrci-'
            '~;=^X1H=n558?Z4L`Q+MyH?>A{FfjZWAq3Ie8^Sl$;_JAJdT4qlqgbi>P4`8{S&fVrl0Ks=@P@u92*Uvt?c6#o8=TkUs'
            '~C_7Sl6F&|Uo>odq&>j;szm*ID=ar$?Z9}jOo3r-'
            'i5LF$kRl*m<*r83??t4`?{imahFI<a8w&w}H(UsK(Q9=xydnJ&Mi46onkkOfiCF`Q`7;&qpC+Z#vnGSP$Nlz%6Yr%XZU'
            'TReWsi@@3-#Qcg-5M9=TgMZo?E7Lw=UqUp*f1QV4{f*FKz#P7uuLp7an|Mx10$dIw^a>A9zh-fkoU04{dO9E4ooZPV4n'
            'EZJR2GQqM8K(uWXP1*jKdgACv6cEZ28$+L_<Mf`W+50;h_$5g<${FW|-'
            'PXG2@LNl}IxI>lMD3+P)qf78Sudm8(Q9#|Hwx$3ksw8=8&>;(bn0R9;+x&SmQ{$R~_E>`!Kxr!K%`yn+i9pO8n7)o^}+'
            'B^cVq;#D1ETqzKSTdqrxfWX@jY1K-'
            'Ni#t))^FCG_eMEWWrHE5oE4URIVe+I4OpPu=qlOSP)!GlP;xS;@DT|e|(G(UeWGiVbV8=ZhA-'
            '7bxFnP}$JQ9BgRW`_AftNT`B%g!z5BU&(Gce`g2(H)GMCm^f)ahp^e)T*|9=q_c%jd|^FZN#0_*fPufBa_LOBTZY;=fR'
            'oc46Lwh+6sb8tl@3iXgZKt0dRsP*Nk>uk^$8<25w>T0Y|$=M!A8d4wtjo8rTy99WEDIGDZ*2dy^XGR}vL#oD)#x6K@WA'
            '6W(tS3;qphoFe`DQMlY5xwCE_5HMxa2VHvesdz&zP7~4xyI01KSoxGq~M`fn;G|Jed1=(Gibb*g)Z&+;HJ)na;#X0=GR'
            '0KJ2P65$Bn!*Js{)19!+202A)IlBzy-ii|c3_)bDzZb!#kW2-AoqlIu%t4Ro=g;}m)Pz=9R79fA=;3$d?~3(r?-'
            'f!nTOdR~Md|6JYA%Hw$q45m2z2noV!{yA`0>mjy}2S9<%akzGtLnpS>0SMP8Mqt)lwwHPc)TP>@ZA~BcBtNF>s&p{h*9'
            'YGwiQsdmH?wY$MMkW-;92$-'
            '*!<dyY8ZFYLnQ<v44JGvnHlZ%Ypmey)n<Gq8aCVCaN<y!i2oiR!UWy{u&h_W%rHlqt&oAdx6eX<Zy<>q<fT@caxnE%n>'
            'bjUz<YPqU^%1!e~UgAOw3_P-dzMubK{ZgP!xLb*O8RY3V4Q|$T2E~-4^Mfc-sa<XW4Z>@x{-'
            'B!La4`7K}IYgNTi>cxC}FF1@mfVW1ZScKdch^~ZlOmaKuM9ktXrJr2#!7lFp&OiT(ALw$9B5Val$S@TGI<dug{q~@_h{'
            '~FEi))k=7;KiOVb8*kzi&Tt%K88d*g(#N{xL;F+_5Yk{eqAL>T>cKt%K{m8+MXymset3vdqBjUldb3*i(GB7IJkNpm>!'
            '!!{+Y`lcH$*|9G{?5`}&E=H(_)x9n%TWRt33~!{DXthCd!!(YcqG(Zr;1q8#ah%|rDJ?;FkJ(56nX|7e4ibJb{Yi5znm'
            '-y<w;pTI!#6|=Tbg~q8%NzUr0#8GK2evv3fk*-G8^RjM`UbvFIB>Erme@(#RV==CDDS@w#<ME-'
            '}Jh=O#0pHwaqUUsI?dC&Ea8Ac#vT?W(J~6goR_s&cPrU?w?xncHzyJo|8gM!0Ai5M2i{)&R*A@=3G@CBk(2CFQMqz4r3'
            'k;GQXd3o{e$}tU7C&WpYW|aII@+Ni_X8IDY&;HL=tS?9X!7A=8ZqZBCVFfw2E!{Kzp4ll0r!kqE>D54pCf?Lv5q;P@c^'
            'OfH*0Nm8uI9yf}i?5>i*&|%X7PSZSavp5bqjHq-7t#h@2kYZfIn9dPw7ygxz>wW)5x?Ez{BVeoQ94a>1-'
            '+Cw9GY0WFnukX@Vs8>Xx9nsIpT=@Tu)$%2dAdboz3?iR=V_08D5pObkwP7}_JKgR=$24FxY7Sp;TVTY3s26o!l=HwtG5'
            '6RK2UkqY3evq~YMdHca%eYtV6S1@04#r*D$o(3r^i@t~kX9UBC>w#2#t3(A@G*=|w}EJ009=|G#@Z<<d>tVJ{-'
            'K;~w{tGIBZdKizaOzW!+p^+=m}i<EJ9cMy28)1T6FG0b+BE&8J|5-'
            '#&L^1jEkkdIOyF;=brt`SjG{IJLdP%4c|_n?~d(cMY=XFRgXtG3w6}n;>b7}T~3lmBk=LK6MVRJ9mSUZpj-C-'
            'z@5RD(6HhLsEM9OkKN6nS9KVaW#88feV1VCFdoo3i@4z1B15ngT1Hr^9L$YN&1lG~Uiyc65`oeXROWaA5*%IVyF>}zzg'
            '5N74Ubu{)&thx`-NgFwt<aX7S){>gVT$PvE$?`=u%pVlJj$cB~=EKIy*3m*B5V!wZVm#KN$}F)AU!ZDY1-'
            '`gs6wJ{mBUh56gwLq23ooixa`}QV2Po{}%jyL=bJkOtPA*kwCZ>WN7WjB-'
            '1A1IuU`}KOLo)x5~m^^|_1)y&7ET5)JPhr19=FA3ERYfHtRvAb;#G+P5U3^AIo4E(f@9brGDje2PIW9JPz;;%U&gJSx6'
            'w8*G>O4ys=l!fN(in*E~*Z3L9aywmmY^Is(4TFQVgH81JVFBdFoKZvc{yI@JZGGpZx|Jt_JCbSE&#Gf`D<j%4-l-'
            '_fJ`Z`(TzdiqGo&J2(8F&Hp_g_J6RxRCP_#G#?c9KYuNjeav3sLKnLG0}v>@`lMyX|l2oUrsmi7si9^~Q#l&`{K@;b-'
            'TBaj`G%7{tvVZK2}YOGd)hTFfnN!3xei@?l>zqy6$px_n^<)T^sO*Lp$x)x4En%gI2ye*q-'
            'rR6QnZZDPo~HXy%c3o6>wkcVUTq@d{&OMXimPVP8O&uhL%>x>3+Z2nsmITg=}vonEP#uBtmA`tt(bfc=LH+EOV;uFh%E'
            'Whh^gpaih0&?UCZS-P9-'
            'Or?Yxy4kzOO*t@OCd^e%i*0~JQ;njOtvUgL4RKmd~o`TGqT0BP@kVYZS$M(MB9MewEz%)brjD<Jk)m0&7$A$F2bq2Xvi'
            '`w!Di9}N^{~!*)@6WKHo^DpYwp3V>WnJi!k@4iqzgt>_#2?az^31?T|EYK29@&@K6<tc=&(9kDQULqaH(GszN|vGY`H|'
            '+6+%E6X04}B*^aJgavJF*e(5!l&@(=;YAu?J2im$efrohbpi7HcH#qpV_5b=g?=eKh#7$uM6>V>Ms9yXZ`|cVGWZ_fu`'
            '+5R<%{sZO<z{VtY<%ak&gP`f^qD;AE@Ixs4udE{<irb7L!JjxV*vV^%bny{)mRYnnRBtJd0|<<uLU13uDSHkT5EoYwI`'
            ';!0<w(_NVvaS=nz`S?3C@{);d<|1~JQk-'
            '$}~%^<A$7u_Pu(0g4GEH;lrapBvjkp3A8J(sh@?kvNf;EI29N?29*%i&LSFZRuEMsewD(E1+mk?k31jC{&yTOCb~@{JM'
            'WiceI^>p5%f`hTpY3tzG7KX<@~m)5Z9CJ*!Wl2KABB83kgT*9KUw`A}@8F8$-'
            'icfyVV9^O**rS+3ax%pTNA7dnP#cGGM=ilTwV4%FW&^Jq;_>tkZ!FmK59hD{u2Zs>AEUS30i#nBtbIbqA!OGQcG&85xN'
            'n^>T#p=~SzeoQyDKlXt(JtDe-'
            '2<U%jAJPM?geco+(j#fh9KA2t!2!;rVnvMr>`O%sWfqU}O@>9d?513y~PO`#v!!dPP+26X7UdFj$=vhSL*Hkk!!#@v3n'
            '!(N_qIY};UYQZ617IEWjsS<)saSMZW%6Cd$v5L|E#)%q%NItFlI{SX=Ik09JRVd&N65B-'
            '9z_#&}{Uf@n88CG*J!Xppwue(9N`4kbwJx$bLTRy(heM;=FkK!8Raa2ERL6*xLhV$pDSlfrPAcpCK^R*Tbz9<nq+rtO#'
            '8N6&4se|Ax6oS)*CFttq57$(fuv=@2o_&)6ZlA_z>Q*k?+de`#P7UCk`LbZXCI!4y2>!U8gzWs?_@p<AEMLjP6toV5gV'
            '2f(Bhry)^#^L8l1c=%v|!t_I;tb}0)7@W;M)lwNX%-3xW<0m{*?{MpFhHoyC7;5Y(k&cwXC3hs-T-NOpYIUK;^^}@Klu'
            'mmT=Foem!U>+gAuN&n9mtQ-X+BW&2P_yAc~Vu7*Ej{$%IQtstng3x|x2NPDm>`n&7HQjK!@rFl7M81{nwZ~-'
            'Xa%z)strKHq!DIAWBfT~`3G;G`q>K-XDKED~1-v+=23p+-~v0QxnYXN-Ix(9ZrD{!UeW)$!|0PV)tQL=Ct|EXQY0~=d$'
            '`b9VJnRwyrkVd>N7y}L_PB@dUi+2-_;4=01w7lDlO1iDU@n4;w*Chj6asY1bY(+ljn{cG27&Rq@pnrK3>r!<IOyB-SeG'
            '?;bN=5}fKCz*Bm5K1{iY798IO&|-E_i!KBrce|!dPT)4{@4@as7uN^jZ;$OJyR6cvTWe)JL%b3Zvm-'
            '?+r9Be}=J1Lcmpi3$9ElVZT5q?zZ=Z4>va<xwnl@^PU4^vA1-'
            'd_8zKxV<qf9a1O1?+bM5FJ6Y`Ik2UGO<dRVq;oF%6)t!O(MYjaD$ksqx>TA**vyO6Dn<3u^Wx9(o4(m<{GXvyvG4|~Vb'
            'Zni(p48X>lRr=ttgPl(|KEc^|9|;|4gUxE0}ruas<4gXaM>=<5O|NS|6a1PJSQ;grz(8j@`^RDb3NWT)PU2T;W*DV8Sm'
            'NhK<*YJoKtoM%2(}yhuz6|__#0(UvWn>wd2Iv>jthJmnOl3`%!!k1I3gp=(nNORP2xj=&HxC!bdMqrPo)X*WDX7ZCvy}'
            '^&gksmHfZSU#R{c)ZY-1hbtwbfLAOKrj48F=W|t9@N59P6aA5yH;T4{h4?l-'
            'nY0`<0KOV^NW2(^={#9b*l`wnd8@!!H4j3jM3}YJQ#eny9qo54Vn_YDhll@az_x{w_^oD`70c%c;(s!sZ&407>2!eTSr'
            '4=}>%_^RJs73xhda0)qBmn2H#mv2-Hh*}RE0KePH?~>?nszv;9@aMlVMwOH<gyppblP4XzAOBPejtmo9k~-'
            'E&B)TdbI(6MyIhFR?cN>AMwXC-'
            'G$h)pMzbvWe}z2cwmTI1lIrAO6!eT=+7JgV<&YSJt7MoLtjvqO2IREMWQdy0`EUd@pJY=c<9&%o49$A%=yC7+qj>+_6w'
            'tt>p9rj1su4BOO!6!a)&HksE)p?KGT`xdARmSA~0hw0KeIHdTI4Jme`?#_*=mb*Xc<>;CKpI*eMTQGF5nYjWKBToyL(s'
            'd)CGhLsZ}vV-JfC!{yt-'
            '5YQ9GU>7FR1($kQ)3+>X>&<2=azd0|tr|jWsVlJI5;rp}<vO`yCBWR%+y(BRhN)CxESdJ4M;StheaozXb4CW-'
            '_155zo_tuYbcLRq#mwGA>!IdCKSO2DBxZ^2!i{fbP$?)J3e*E&V7D6tmifWQ17BGtdpF|9)HkYV*Fm2dxP$wo62`sugx'
            '-1TaCLGyNQdg8+0i=KrCAO8bbD~`k7!~{W{6tQ4HO+81vZU=?XGpya5xFJ_-En^=XT8Gya_{BpTZm+ZOX|-'
            '!C_7+h?~WN<Jm)W|F1jP72%8~nSrp$LKzb^3y~qwh3#337}w>svCQN#JutAJ9N+4KZ+aa_V2~zxwx*4=C-'
            'y?cuQE{7ML5y72^Fmf&CyoHk=e8N@~;tg<jsL^jd8UP68a%8Bbg!l=_Z;?oQIQ%_jDw(<}llSi$mwxcT~T;lV-'
            '(vK+i5GJiMR`mlS1F`IvTGb?yu}sz;*6g)nk8G9H!RwZQx#VQh<Xq^djxu!^G!%nsS&GLtTl9I3@3!(4jid@7oie<b{o'
            '#b78cMDBzg2KjMre8Q+i&!M|`?x+JK&u7Aka!>qFUI%pNB38oX0}#|~0UcNB2pzgfZ>XDKZV5ND@ZEh{9CQ@o4p_p;gP'
            'x$WUjul`ucBRO4O#!q2zx}bpz4qZsa38*5&a14P6@=B79}!d`GtnZT}FSEI&g~Y0W*dL$cXuXh1w%JHn<29o$_=hy=Q2'
            'eRU~7XkPAGPTtoL>jRSi8K5TqrL2^26F~2u~-'
            'i%TqlOJ+X)y@f@dSznF{5Ul6>x5{vYnY@Q0@z>$Hdz+PwRaSn_}f57Hv~Qxc|h5oC*)jKBswN)LD04&c=#X)OiN$U<~N'
            '_An=zmHGbx8mHS~dlu^&$V1%QU8*&SE`&kRJU*M%sOS!xKMm8wvq&IpCM{84t6t=)PUsQ%~<1mCZw)8UykIM5BO{_FX>'
            'pJ-fFj4vw9FlF;nqO;Nf_m;L3wLWfGw_6ij{e_9wgflIj4u;}lcQSR?h@9jJ2I-'
            'k8G+diqBlSRtt>k?bfB#v6`(w<==}9k$b&0{gf8Mz84;z^CHbT|VHe^n<!Zo2aptvl%wkXmZ=VfML1UZ5mrnq5QOdl9X'
            'hNEA9E(VC@LgA@itj<tH?bE-'
            'B*AiZ4<NR(ks%!#Dk1+Ud`E2%H`(gNwKftTp2&*$V;9L6uafJCOX2pUX=Q_wwwI%S^%N%yZ_5t6;^WZQnjP@xHaP{Z`M'
            ')`sX(h@qx@JndMFUwOf?Mx`P{PV|k@uy((qF#7e^A3+Y?+4S>bK$&jEz!F5fySyYLVv$hd>Oio8JAFiJGm$E@z=dNt=='
            'qJ5?@GH6lcLJpCa7alno0)H#2NfKfr-9dAz)5hIKU0gf6^ypRRi$jOVJGa3-aaOs8Ff6-'
            'A2J@?nzY*Dr^)rHAo&sVcD5%}_+}G{G~9_%cEY<W00t;dwZ;=s%|UQybuJiY@CwZZ0Y9@CTzC+(7#x>E^)KH2Xjv9c<i'
            'Bh8+~>?O%Baj}xFfa4+f~u7Lgpf%sKb1WL~Y;cig{*t{`?Zkx2l{Z}%{N3H@$`1^vaHaQK40*%1!R|;#D`+X`_%7$CkV'
            'N{zpk=W1Pxj7R3sGQ6RC;wc4hxRel+Ded$A81AgZZULIddn)4E5O8OL)f$+0tP*uNCSTg%@HeupYN8!rX?$h(3xD&h|I'
            '^=kM|Hwi8wrx=mTeW&)~kKPPms;OZEItK!fQ3P1~RZX-7&}_aZ_-_ZExAv8@;nH1eV5wHmONFGKz9lC`_~{Ajg_5B>8j'
            '6~nn7vSMzEqq^o}Y;LK;;@@eY$iIdZu<yaM`dp%N{h!W)H>vcWVHIPKRx-'
            '*4&7Q&6DLC`=4G64mrML5X$y8q$h~o(yOF4^@`(>$hfE^4hSA<(_i<vJom*SoF?_^cZCXC!!NaIz@@aCNs3@^M(>2Y4X'
            'U&u`ZeK|-V*ErF$P69uPudHk32H+TM3bOmlV06<CdgOXMsj5Fq-OWr<OV%962CHGjYAx>PtA!7WEc&>CiTAWxST7a?)0'
            'IkH^yQNqL``lFIs87CbP4J}>gO_YPu2>vnb+Z5cnyS{6QDXpJ=7+y1423yz@z&#<^}IX6+a&owrRv(n=fQV+(G!lk%02'
            '=AHxF2JNP?en!%~oh|Z0zEX|BO*x6D^+Rhf^Vjc&;lcLC2dlQuTSg3C|%iVptxW>T;HPgCLWanCj{=rnT)+P&5Bww;*2'
            'S;dGwI-xlq!S__NvsYkLfEfWI8gta-'
            '1d}aX&4oul4cC3YtDnPZhMey1>E$$L+9QF9thhb0P~GSv7OPym?;YY*^z7X!p=gL#<3Ofd~OhtohXN8R@u19eAYkqM1r'
            '~ki~Mz&!qSR%xO+k#vWKMUrFL!-;9ZALm$_r*gbRe-'
            '<%J3H=Zs7V3m6RVME8gi{MbK3EB(DO>uz+7|K~*5al8_KtubbFvgN>XtRI95)lgW<11-'
            'nr)8zN@&>h!K7GfN!UmZ;M3cRDCN5V<L-8S%K=hLMA05}oWgG-bIu!WCB_Z`xrN|wF2QppbX^zTB`A9AEI_aj`k;b7Xt'
            'KP5L!3TS|55{AjV#jK3mwdqwF_*5-'
            '`+;`c6fkE@==!q^YX}*v5taZ`kg$<E0v7|+DHL(1eE3GZ6AqFqB5GO;(7P&BnO@{y|R5gH`>kS-'
            'T)da>qOJI)rwQ3Wa6~y+!?0xZmgrUXUP^qzo^xDsZJ2f|P`fvc1e(;X0PP{@rk^^Y&cRvVYCX&TA0U+G&2jiJ4NW!Av@'
            'LDCJ-'
            '?|>sE(O56g%+Uh^_Zp{6vTVSDqw7W2Pm=5;@4A)Nk*$1wZ5;)xX2}e?02)f#$qd+5^Tasj!^6!`~=>Z54_cv(dJ=1u?u'
            'g*r{zz{zK3j_!@m@cyf}?6mW{Y6Bm(sE8X@^WI@m6M0M@Y=N$<t?4A<GY)4oH2b^PT3&aY>}G7cd~f~^P#$H0H31ytXV'
            'L%B=;@R(>eT5&eh_~}fd6@46+iY$YtrGmKoRRugcn~0(3e?o<{IbPELNB-'
            '*hLBF;XUGSkFH~yK#Als9esnSU&<?7MNsEJf1PvIA}Ksxp{36Ayn;_VA@M0oQxmiOKJtP6_@KwjtxIp0+e56uEu($5ny'
            'XV9GWLOqr$Y6gS&+D43*d&Oumj7CS_Fx;kB2*+C<(S|u9(4ymp%8zsK<M~q9$F#!4t&+^<x8{)7%3}1ls^Jq)F6Q=LEi'
            '}CR1D@Z`K^u8_P!|lt@~$|#FgFM#&u}qQwcE%#M=Q{K#K$iDBBZ04;X(3uXfh2fq|i0`I=LNv2Ol-'
            '9Bo{87N5x(O?<cyDchhw&8~=*DHj&gU@j7U?8=~%;Prwy2LY`g9U|iPHVi-'
            'Q0W#`e8c=lX9D<ES6ebwSI2;WnskdI)zGoQSC7DhUc$$;Lo%W!{ZDIVrpS1ZlQqFkXo%)I@!klZH&Mz5c+cJa-'
            '}{&STKGY)fl;(!pmOH*QMpHW7QAq`l#jE}hayru`{pTnt-JkYoIBST8B3Bp3oz?0fwdP-;?icF?Kg7Rg?-'
            'B+%Z=Q%&q!RHo)v?;@K*DU1Z`U+gWC6M~`D&1JD0>;g1<YmoulFZ?aGHf;aOLmCbeEn1_sBnNaDx=7B@A0EiRbt?wX#p'
            '|ro{-YOhdD7nz)z!_RICXB1K($md}<r)`kPGZg74#p6)j-'
            'Dw;9xSoX4V16{tLx3yN3s;HmIRvP?S;FKw74%R+wAKMzG9vw;uRj}_9o1G8MeB8_^uzbA3x8F>9i6HXc)$74s%!OiY-F'
            'krnMJ#u~EG%Jv(>!#ux)*$T6e1?PBfw=w6bCeye1QB^-'
            'Py}Z@E=yT|i}EqH`w%%I=mk0tP6NL7M|XK){MU6CUJjb0`G=h#_$`cKdCyWuBKjZQwj!QUp8XvBUso`~ulT~s-'
            'SyfQXAD@$o24*saXw)Q-GCn3Rg|qBhUYeLlV_|@5Vj3sEX~RVGah-'
            ';pb|$l7*+UkJp(eQZ{oF2#<0sH3VtR86B2L@f`c-sk#ZEME{FoBOG~lN^dTeuN;W;!-'
            'arlRECoU6p<OB)af6+B4YyA+e(O`mmVH5B|D=vaZeBsnrd$|d>(=56zcr|Itq+kEiEVQmv9r$~<FBZr?M+#-'
            'XY;H(Dv7h=C$F=l4a(s}G#mE}{UrtQQ-sxGNj(H8S?Z9=P^gKfpHrkz{(L9AYg-MT!X;=H8HXx+{*tLwGa|G1CSztS98'
            'Rh2r~S=*xTH@PTrSIMkK9@Wd?);fTofNv|4xS8@D1B{KO!YZE<%0ae2gzzO^W~afyRJ1-kNM-'
            '6sp>x_`g1?RozQUA}UbgPb-RV4+k&T-?Z>VE2wqJQ_WSGz%TQ-cB7Otl)W9oebM*m!)|9R?9~T#MQ&`A%fWZ+)v-'
            '$aDw<iC5;2ZwDB1XmgozrXtK?ZscZi`}a{6#(<r$26a2|b_LL~prXISif9a>^7z)Cj(j62KW=x}eXO-cqV`rL?Xa$i$Y'
            'aEF%mkAM#A705OOg8JqjSU!3Mo_{ie5)}sNTW*Pg4qsW;MI!j@r!f0SyF0uO4~OkCF=+NO1^)ith2pkeaJ*;{82q{cHL'
            'E4*9F7*;W|s+f7W+^GGgm5KVTLF73A4v?YFSes$FTAq2k!Be2e)lGtTqu3<a3Z`HCp=NZ$}lpF`mg7kj(}9pG&xmQo}u'
            '`LI1;3R_LooxYVzNY>EHxEv*QWKSv1FUHR#I8*eCLSK-'
            'Q8|Kn}8!EJ<Ki^~eS%zA{(*gXM<Pxl~QD<0=JisGqZe>}Zt0c_v)j^6eWfaO&Hiqm1F>vkXnn-'
            '+obEfYNKyNpDix(&fnuE1F4ORMt=aA52!u@z=AT!=2%oQZ@Yl?bw`v5`p9eK_Ys6n1=APp^h%(4Pko(w=&O$HyCxYb=I'
            'AtIgo9?N7$I;&W}bZw|gZYeoWV`>4*7Tsr=I6vKA%L-'
            'F7czO&K<h4}zC+&joi&*u<rvH+KE7D8EGUfh}%j;6KYu)nMu<d#N4!q0fpwrVd5o2KGL8H#i3Ea1~|IdXMrCVX3aAA;('
            'cc*8k{WX{jRD3T5zR35_h>?9I8_apr?Bm);eWT8zqAHLTXfatPYxHOZSrtE8ijN#Sr;nQ<`#od6~Kk9L6=pNai5`@P*I'
            '?2iPc~CC949{A|!LRT^T+VX`-#xrdt17SJcJFA&PD{n_TQ7j)+g0FtbCPbKAB&rqMtJ$q4U)4Z53CY&ko)6h%#-S-'
            '9T9usQT9G4<&2>51<lYSQi*rXLV$12eY$W_1!guEgXB&HxFz-+<UZx$T!|W(__-'
            '1<_i~aw39<Oe`wFqw_(Vtf)#<uQM^>mzEZozoLe<zzSaJCpuKx3s@^|{t;`Crl-'
            '_%VtMnB`%F(#U_EKtSsHW_@?fg>?I?3UYVAmx!wdcJWoFRY0N4=rD~DAL2~R~RG70petz@dmoF)D&JBT*99FzO1>{b0I'
            'h~hj1OsqGKL8&_iyb*Rf!-_pK%-'
            'S~ZdMf;&2vy5Yot?8J8SYuKt;iI=O>D0j;+6^;&qBc?8J^3*ll{Vfoi4TP!pj*Ya7A%S0hSO8Jl2m0eY%vTfq3^UahqA'
            'e%~?D7?`xoDad(s&9x4rW8WYclG|H$Z!>5H3F-'
            '1v2V)LC*C(+B;8b+vzxfz|RgW)qF$QxoPOte;Gqxao`E#N(esxl{J)Fg3osD!jfPXJtR?$!$*tJ>)L%%pB##;#Y=IC{u'
            'z||aR4Olq+sP=8!EPJ0gU9-v3mY~#=^Fr<X>Y5=DfB8pSj%ZII~x<>f?&q)|fbQGx-MA46K8VxvS~^KX<Utw1}(>(8K('
            'OF6gKWq$L|laebF2gHukI@;xtsh|W)p*8K~pUc)#JE1Oft?HP;{YWf(yItsRVoP~|89CV}L8c-'
            'Hi0#W8GhU*s#s&V!O*=zfToC!G&+2g5L+VllH6k4b)_YDmAo&)?6&9hF-Nq+vx#7&uxF@pO#X&hVv?iy@(zw;1|W{b1S'
            'E`-p_11E@*#YOmb&<^95E=QXS5n#3IKEr$`FZ1}{L(p543}W|wqli>GZn$}qFmG=oc-Iu4^VA{3`4<Tth(eziZ)v&JTz'
            'p&mnHq}vvl13(<As4!AUR==U)DUpNAI{XTqz$vSYN@963&p<CWdcUKf=_S0%ZOZh1F*VaR1kQEY!|sb#=#LH!nBauTY-'
            'tce5DAbGc~nQZ5Ww;=@pC?#7hLoY@`l6#F-'
            'Sz<;`{XXp40412tYJ!koA9eLJiIGE7Pc%vc63@Mt#xXx6v>+wf&X~hLHL%w3T#Y(p4=FQ|%(hK;iza9@hjlq>muHv61P'
            'w0Uw_LT8T3l{g~qq>VQ^i~;BkF7T_A~2ZF{i?~{-'
            'yDZH525T9FIX*k0&^#fNXPXg+W#;f>nBYh+kQPR$c~53IYCJ0^RVCeB6d7Vh8yKgcyKj^IlsopUXi!6&KC!l#6oea0)W'
            '{Pb6D~(1{SSaqB(nj$a=L~&>LQWyxU?x@pLfGT@+6$0<S{jNFHRr8YUyhcrfGSe&C%_B5$2dAgeD5{d9K{C-x)S|2-'
            '0ii~r~({|Uh6?FK~Z%^lc1lTVe$BCxc0)@{{d!2I`S`m*O8&K186X@Wi=bZrSRI{C1P`w+<;Pr|iFi$SO490s0_B@awm'
            '8R^G^>HN}ucwo$?=O4U=tvOjNyPS)d=P8V*E?uDJ5(SVgF%R#qOGJ))5h#;v4x-'
            'BbL}cD0ns(bBU$lK^h@CKn>NCEWp;wGi`}VU!R9W!fH%GmDH}J1dE!b*w)#SHN64eM62L3dLr46ZIeL$S-'
            'ejtS_e`I6GuQ1%c=>!&7nxS1%BFm#;fVLGVq5tAc@b<rkHN(E-onZ$a*cAawj3<zL`D-'
            '}ksf3xl<*@sDC`?>Ff>(35z~jPLXnq||oY!y?1y>!AS+NLO{Tji+O$oSP+JNElgN(^sE_Q>vI-JcmA!>7e;>NBy?5rzw'
            'SfcnDVr)d|A)z9W9}gkg1{bK);WT8JJAxVCb>P2#mF`ZcAdgsC=w)R_+G6=|Bs&eYr_xB`6G>FPoCW9Qy6D`6XCX){9u'
            '|a;GSb&a;&+{N^pI1<v`=Mp_c>l>Q{p`M6A}p1wjW?>Z6}C+Z=}gGyNOM9C2cxc32J#Akli|$DdTU2Yjy|2&9|LY@ZB$'
            '3_`0#i^UG!!Y%8Zf!})P_+6eyDE5k^ADJWlCM(Y?C!L0BJgx|kKH)|vI@<gE6rV7|N>Or-'
            '+(_norAM?l+0PC3=9Atb(2h~0Jc<c%)%ROc}$z-EMY!$?X=Yrfs!vEw>_Qn^}gq=y?Z_*4OioLOle4s-'
            'qb7<eu4%|9b1ixk~>4W_vv~Q;bICqQVZig>KI;a`#nW0#`d<DL$D}%$)!&G8=ga|!jATuf!ogdiH%UzX>X@NF~;vIyir'
            'dhbAHUZKfSiqhY-'
            'Dtfx3<|?`*7)evL*P3Du>90a9NF^luW=vhX4PWk>65tA@HAek9fJG0dK4ObV6nC(*tw?S(~V1UV}caYrVc#sQ-dQXBuL'
            '+VGuC}~5qM&|g0_3h>u^VI$HTD+prZN?p9s|w%~hL#scTFc%bkh(uq5r}tb`L1=HT9wfCIw%XfE4A2MzZTlevqbS*Dke'
            '9t|S;!3PhP*`URb2N?9B8U$lkp#02UsGYFHN|gopZJj8(xZPoK9uXl2Kf4oyb?2yQVHMphCx#qT@vtg3gznV52XFoJ(J'
            'P@A`aWK!=H1UR%Ys1#xEA8yS4s4J^%~eYe>>DH5rKx459qh33C7vyQ(!X@2|g+F317GfxX*K<(K|)RlG?Mt-'
            '(L%cCPP3)_?di*`e5K?>MSqvVfGLcb!3BB5i5P*-<~$`<kthuqsI6-'
            'Z9Ucx>EkL}Uex~5%;@na&@t@;^~yZVZOiIttLQLwaB0Vq-YV8nx(BunXhVDADNwxp1ZzgiFs4bI_2*v+CVlS31(m0$uJ'
            '$hQ-}aY&d{u~j%C!)<LX|jmuE8ry6L7FH23&T!!1u3zDBUnYZ;ATis~2Bjv@Z;-xem~84e|7Sz6|BeDb;y-'
            'LJ!uKj>6l1A@JBZ4}SC2U`zgcYU3w~p6A=)w{r?T4!-Dq`We3L^wi-pc?&oD!|=z847eE{hnJTff^*L8AUls21=NqiF`'
            'jBHQcK6PvP!svW0oT>=i*{%SBzi)_^l44b`6VhaGMl_?_Ug0{3GZE?^F20T@)0{zT)}rdJ_M}3=G?q=r&bfI2E`6@;ww'
            '`YND7V%YPysYFr?G$`~WXc_7VG02XV!K*xoum}r!Y<)-5lKg6Sy;22(BvL0LN1>xBD0(|tu2=v!X68}BlS-Be|NjSd}s'
            'AR|!r?{Q?VJ4EL<F^okI*Y;ai!Gv)B0OW>gt&c&!TwtbZnoNtPkW_MFQy!hl-'
            '`2hj>R}1(pXvdUcgDI2EuDPht6%tf=unVtp44z{ix?fxza|Q61xlrHzn!G7do)JwGkdzO^{XkJ9J#M6*0}OhaP>Ph@U?'
            'Y^x7Im7)OO*o$&^ew^1KXZu22O3^lOs*JBzm>)$Dhm58DZ3x$_lgcUmg57>N%wfRRd@(V)J_eA)yi4%r?8lp|P77Qevp'
            'wGsO;E!w=x$vot;j%CfwbGYh^OrnWG802xZ|cCi5e1m~>W@dXqVb;j3Dm1I0Y;St${$W+JU4OzhpYTJCS<EU6EKe+HSm'
            'Ut`#;Fgrxg5UEdpMgD#%e*M@|TwfiVn&xW&FY;_m%qh3Or}MRz8g*=B%&pC-'
            'xaWJ8!rTLJ6T1kuI$Jmh|<#D@Kn^aDp1nq0q*x-Wi`FRmBh5r;M&Q3--vDPFeOKt9GbpMW49Y4|Z$kkn*F;7s%g<v(<p'
            'G$nk6T&+Xsdt8M4IIo430Ua<pCc`LQjacA%5I251fi5#jw8T4*b>P-sdiLEWvgvaujMUsksk=&4*_n+aLY$B-'
            'cndCF)rF`uBP`aJB}VajVEyz4oabsnaSuB<kfVUM8LQ#nj%}3F?I+b4dVsxb0rFIPEAHBm2cZo=!Es9xB#9Tp-ct>*?s'
            '+->E$GCY`$70#FcH}S+aN>n4mR%jNEdkK;hz5Qbjxm2G&>+kdtx~$>*p!-'
            'WW~YE<{q3nn}&}UC&8*oekgsEjiT3C)aZaBU2km+>EXUG|1>9FTeuPq?fgq#b^V|crVdc7RK#N4`$QDH<I&<(I<EV64f'
            'j_Mv0~i=V8X;6vQ7$vZM7*Ds`w+hxf0@b3xaxM1~wfSrk#66VBukZn7kB5*pq)5Vq8gJx-'
            '*mH>NMi{e}Aya%oB3#WN=2JfR!sY2o?JIP|W`brJC+wr13>~V(<;dxW2-*Rqk-'
            '1BLut+zR|ElCtzFRcXCKC0k?;~1J^f2^tgc^%2~<Kr>D1J$iED#`?3_5>)*oa%TJO0avWt7=CMmt=CfTelJbXYGu)Em;'
            '4p;a))Hx0g(G-L_F;8!DF@D+7lB<Cb5Xz11udg}ARvbWs3-^Yo-'
            '>0dgBz%W+cPTs;VhbXFGQ9#FIy{5i*3?B0S2iBF#W3pZY|h>VkgTnUMCJ$4BW#Q#sDjGk^?U;Yz5WV1t1=h&AMTJ7ibI'
            'zJF0InawUzERa4P$^^FSt9TUN}Q5EcKUO?VYor5(CIv_#m8Oh}Dp@~t~LBWB6{aV>j^71r{h*V>R<qPN-Tmu3l%1o#4k'
            '4gCY!$jd=0EPxHWIhe|Wwb><N6$Gcn4%{J=+X;awWZGwF~$Q9qKlt7N~>hymux_@Pl5QGLjbJRO;Bvz67Y6i55B>|#Br'
            '>a$a80-v)m3?yJj8QeE$U!<@ZtYR}<=A%LC(C_UHMM1i$vMkeFs;+Db(C3`5A37XryYIS_S94;BmFg}Jt0@uP-'
            '5_*_}dI%b{*4l-Nt@@p2!J^2tM!eT(1GlBS5`a}Jz0Q%bNJ4?5v6h6KcWM|kGke|-Q;N!Ru?#6b&yE#L!Qq-Gp3a7EcH'
            'ay1T)iHEE4+BPq0@2`Z1YFBq1G{T7;psg)*1wEDq;gFKZr9=_2RX%Xc+?FtdAz7<&220VkVA>px#*|*0b33{#0a-'
            'w_|o)_20N}ott-87r7a8EqZeY-iw;)9oJcw$E<v(dPvF9dUT7*0LF3(5K*U;`DqiHkB!e(aSsR3>YwEz}R5c21Q=)&BU'
            'V+8TIDFwm_>R}$)}J|G`aT!mf4z?40yEfn{V%?8vZfq!{GffbcDCmUu=v>`H0D_fo98OhH|>VNDg2JH_i-'
            '*BE0ia3)`xI=lMK~ga|JS;2X)3$M`(5PD1E8zNxACEh%omNP%@B$_k8=n$GQ*g&o+@)Tsm;PG#4J73t%)qiv(+}EO2?^'
            'PxQUp&~bJ)oDqq~9WU2{W_KJm`ou!u??6(dR{)yKP$(GCpxlO(B{13z!z0JR<BkumJtU5KcIAvu0TdFQZP4*=B3^oOm<'
            'R}Gk>>tmn15Xc|9ECW-xnRc!sCvPS2v(QRx$0}djThpCgH8;rLd-'
            '(g_%Q1FgY%Tp_LV6n6zMZ+6JiI<c{VWOW>}c1aQXXL$ksV2Iv%kYhx#@vJAuX9I=d!r^#TMvv1bps>sp3M=>`pg)9t@f'
            'gSS?f>D?e9vy#)um5UbcxfKp+ariUiZ3utDiQ|0g2CVE42F!A={!`8A_B+#U~;0R*54$I(XnYgC@n35L%*bnz!o8pj?B'
            'V?vtFDaX#rhZ;^1tF2)gV_MiYfQnD6O}Ywl8Nx9b?bcb~^r?%seau1(@8&oFwD<YPl!Atu?+(0}piMB6wAj6Out?&!VL'
            'PGc$AcX+0jQ{@Da+xP-DW&Y4<f8;|hlb7Vm{(3BK?L*xk2MOm9Hwc{E4GIrKae1pVz8sihjjbp~uTK(i%l<t*;Qo!A>9'
            'oNuJM-Y(?{pBWJA~HuUa<4WK^W!?!P_tE$#&V;Sw=OXGpr*Z-'
            'S&daQ8I)_Yol@fH#wY}(@R`Z6=2ms8u*7Rf>7NR`e2>_4E}hiedG5cIDgz9oX_QA-EB+kY6-)`YQC_xK9MG@|H^Xx%|-'
            '>gVicLVMVGyeKo2+sj{FS}c|4d1`pc2rK>$#{2tU)G1Eb*^<*Y8iD1l5mecF<)@UMYv7Xj?{vLlyY_+fde66HF$3%0ao'
            'pn}&h<_>M6!%ZLIiRUBmHOd3I8NkHRSJVko;M4w#K&>gBe-llVw#E?2wXSI8WkuJjxk3FiK4!vYBRqGkpBi~wVJvDtji'
            'p+CaIgFenD=ObOUWHPp*W9O{(TIVxmAGinlZ*V0XG;fFej7dek8P|9zRIz!(Wa;m^|+*>#7Emq1HGBl-Cr+lQyFm=N7`'
            'ToDzkj#$aoIl&mXUj|R#NP)t{Vj&E#?{gDd}2ZzBiND*J!gu{<lFKO%NEShSi1JjxVgs+H)srs@84eoQI<8OCRHvB_$_'
            '6RfGeHP;pv3{D}t%4SYk`Obp82!7%@l%@}d>UR#cf`o!m!dozcs>F;>)L3>#|~;-'
            'b^(UgHNtTd5BxL0M7^>FX!u2(tQH<%@C%C4Eqoog%YG&4GzMJqs1|k#{9$ApB+_-'
            '&4X{~85Vg{_W7pY0Xz|~PY|Suy`0@dI*1FRr9IJ3V>@~GGqDZ8+MbO1xv*8<WJ3wC!<i-'
            'lY@5oe|Bm9w6ct#LGX<JO_{z2ZVdC_&(Ihc*JcTjPjDAtW#C*kg@u=Vgku)IA+^*c6@@VC!sw}3O=+mHy4Y~SOP$`!2Q'
            '<16so?RDT>P=}IwbIIt&L@k$9evA<bX{alD&v@}Unz2?m6y*{ryb?EtuJz(@>QM~kKhLJl%cSU=(CfJRW-'
            '~q&a>Ju0Nn~JtB9{7v=`4PKnb=F;#3gU@$()1YSXprZZri>iYJVTWnMonY=N7{B+gD(YRW3}c%0aEwO3M4IkhG>a5s<z'
            '}kQIQd4vv7=`5cI8*M%>6xnMK}U?S}g*mVtEH|>VQ3zndIdn<l8>HvFFr^w|18QkTy0z$HX6E(XaWCabPOS=&jk7<R6Y'
            'PIlru`j)JJs)%baxtaGUQn^+Z5Z<kY5%fz+)-~xmfm!OMQ<;WLyo#|9rZADZw6F-i6Nh2*sR<PUDmN-Svnzk7Y{EOfmO'
            '3J)P8doh)!zKRL*C3o3`T0G;bJuQ;Pfcia>;1A2vO%C5eJ!s3epR<-'
            'ZcqT{0IR+cuFmhgEQ!S}U&Jw~Y9e3c(Be2@>D+hP9z33{L;9#P3&|@m|pw8XsWMsypMf+gpz$6$Ju+s~<*eID&sp>Vmh'
            '6GV4)2FLBX21h3SdVx)RA26abb7Jn_d&6PwJx~7sfce~*Y_YYFFxrMj}y5Oqcy43H?W3phRl$G3=2LYd%@SE|8*!(>Wj'
            'WigieN5qQk|HEjJ|JE^`^bgU3UDny6tr~&*>cu7<lws9^z>XSlBxQOW%_qM9pw)oE8a*Doh{2@ahMPKn)ATYamiX~FFB'
            'ksBY48~I=0_i!YtW5Ra=*rik=Pq^zKmua9kFG38Q~&n-6L-'
            'uhjX%T&roqX={Kgw{Ow!Vh!k4$Pelp`JvN3oE#RQkW2TYGF=Q32W0U%R|S|}*uwbzIUl8zLgCqVGq@L?PC~Lq=uOehn0'
            ';OvqujZ$fcGYGJ5mdrh381a6hBs`E}?e^T)=9M9oUF00w=wFuuu9TtbT#DU4OoyM%-'
            'a2{jLt<e;cStnkU@Z>Iz&XXW#^5Cxn{E1COddK1>^<dly|t)ueqe?p8?Lo@@lhgdDL4UZ#2QRczQ&sN=JHl)SV#h{4x}'
            'FqV~$(z!=a@Noh0NTE6x7Kss)>UR9`Bm%#-*yCH%$FP2B3r+nQg0Es@iR7o(ctE$4-ij~6N3$GRzw<n<;aLR9JNn>-'
            'Lm?=zN?6iSm!Rvn0^}HM!kb*v#8meS_>6|&E-'
            'Ob|&U=vhJsPJxMw`(#{RD)0B8Fcv!>y^2sAiLm4dOfTS@}GC{*({WccfwGT6-'
            'F(;6;wg`jb&hQL^1u2V$KZ;m?s?RO;!ZI~$wGyXPtBAa?+q#Yai=wro(7x(eC?!8CS95NheC62_PTs6CsD>r@Wmp0H2~'
            'b$-~)CrWa5DM3#un;zIVO2omJaL8t2#FA@-'
            '^W_XYc{uBXj2aAjDh)S`9+Dp$w(8uP8%k=Yj}rZu3uJP0x=!GiU&QTPCMf+%uN_()O?wT(;onxa&If@ANYD{Q-'
            '8ss5;btA(5v79K=Z}K_yUp;mRu7V`b#T?fLfW0#1%+YTLEvy96zOo%YfCM#Lg^)5{h5gO3;Z!?+cR{~jK|@+Q<ys<4E$'
            's%w5m2SuI!y<+Bi4d%Lj~jU0+nWxDP4{i%5+^0vV<(oTr`wyEpN(-'
            '@biLC8CdF=S^>jZE3}Ivfetjj&%(4p&c~1G6|$BKLHwvLxtxGCa7!UeW^GUHrjydbR1SKuLjlH`Ha32Ugp&73^Lk1O)r'
            '(bM8);4IPBMhM~+s^+D|9wE#?O`=UVdLzi)Od2!>a$xnQWHlmy?|g-'
            'eyWsFju>9unrFS27=y6TDhrlw$;h<2Jg_9|60)lQbtt5Q;;hG2L<!S7$y&qg+KWHa~zlGTVrx0U$ds3sN6!hppU)7}J8'
            'wk(p#i3|Fs&bwzaq*CG1NYeBP#0P-'
            'Mt4jRnXS^n)h9=GhqNZU{3f=3ht*eA{IP*ZBLRE>;WY=x$HVZ{1obn1_!`rda~AESg|<ys@qb!FqAGZ&qVze_vSZ<7k8'
            'ay-a^q+PU(hH3PY@I5<V@SQU#m1V=Eh%{&^ti<y^r5Io-h%XqL$oDx2<n%Vsa5gWDk}}}p^#fUrMeuop18}&#p-'
            'xrKg!4BOFPgq!@aOX|M-'
            '|zOgN!VE{IDMUvTmd10wGXsy9J*G_rXnuI857#<5W~16{x*RFQ1!$(Zhu(`SUcn_|}s(?$!a7PCEEk{TfajUcl79D2F@'
            'dXyN#`FwDNZhPdt4!uavG)b!CH7(Pivz4}m?XPJ)Wj};(WVT_?_@R!O+G}0d;!q6SCjfy1&!lC7>h_3BCP|+8~OV6$m^'
            '0AHVJnc`Nd!E8CYa6U86vtQX$7;i$U%?$V_0%m<8y&n|vAjSPe_2Hl!Oj9GxfF@AmLEuK(hs^SfRo+ln?_Tac3AS{IDT'
            'C$1`Af_k<_JWD0(c4@Molfz?F6IC;tH1@#Q4_cK#mg=KXYP!eo;&Ck;$%<AT8r--'
            '(uP5*R*o0PBExcucZ~R4z_}wF{Mr`O_A(pL-Pl9m*tpf5ez>d%shG^hurT-50^7a0Eh~ZSm-'
            ';Bq&rUVjT7fBYF?618ca6{Fgu2p1m^lf0Mu1_<xW;s1_Dv=W(P!mF#&Pug2AQo?8trtS&+KiDpb2--Gv0@-'
            'ib&<lv8(0{nC1C@c__sdds#!=&xq_#-(Kk{!w*+#?_J?mmFW8v;Rb`WT#(>Z0w8_3*v>B-'
            'SXZ(Itl5Xt@0=QJ)2e4U^hvExQhT9*WZk_TTA$^?M|Jfd5VYLhb*cegO`jP2)?hZo0kBcT;kc&!+#!+M7mW8GY~LDl#Q'
            '1i9)7`=9KW9?Fp$=C`E<_O-'
            'f3IOvy~9%wy)6%wu@Yehj6Il`@4?hBQcN(43#o=iUF+f345@zPexB_geS4_C9A{*R{^t=O~=;XO0AVGuvCenHvJGGYj9'
            'lF%w!{n9pweGSd~VGk4weWX_p$o$2!1m1(olg{eQ^k?HxvlX<Aag=r@2&P+Sw&TQv#XR>lUm}BMFne~G%Og<TJX1JRN^'
            'P8kAQ)j0;le=y9UD!3I)jT)mfABNrz0dq#@NfM;z^{YiWa())hICvg;o^MFdi}hJgnn9p3hCY?tnvx<PnRPLZtP;L2-'
            'c^v+7jgBu1EC9x={M|S28PcML0u@$CH@euOuP%@$_qz2ARisnkp3?Aj&J}5qDJ+`mfB3w6dPE_PPa;*tqqqqRa^Dw(mM'
            '!k?LJDy)~HeAN+!wETjGx{3`zk`1^xBnG7{A<{@)WW`vt3bCZ!Xv&!Ha^Sg{EQ&ilOS>NHqyiw=EoNw&J{3m#wS)$~`y'
            'cU0f8S&GP>1XA}Wc>4FDsS*)mY(-uR-SZZN*%ezOksL5Jq+ELC63pb689aM36DLQm;bmj*J|4^7p%F)bWFBo$_qI#H`u'
            '!WH~-=M7tj4K__zHZ;6F0lMWG>=Rj?+RT>1R4Ch5&$RONob;;-'
            '_=>>cN*$#M&@H+7}cSvxSgLIN+WH>DW~EXwN1!|d(Bj9nM%NT_obbzRg?e(BB7krM*w6e^1Hzf<VE9h>Qvb>(!0b0o=2'
            '>Sbl#HKWq3EUK$4hi374H0`9tf7ictcK43|1;6V50sfI^HLxLX3Gg&FQJtF0=y56-'
            '=(!(6w0J&p>rm2m&7VfG6EJ}6rmbHqVeE1&9Ne4%n;vQ4<h&;0W4V~Zz%|sG^NE(d+KDL^5zuJR$U3%sA%s?EqSVdBsL'
            '@%?m@4chK3jj2{?>dv`~5PU5itSF+H@3GlZGR5-DoRPg_H5d_%NlNdU6#)q+c7bmV{BSHBVq0`!*}{h6g&F-'
            'Vc9R4;eR&gpk)v2>Ux#aKTqUFn`lXuN*i5I+ib}yQD4VPb=W7q&W0X>%-P@GxD`-'
            '0dxs_L)2aySXh`3p1Eucu(DzOJF}g2z-'
            'Sq@3uR!%>3FPC5I_l724Z9OVCvI>OWTZ5ctH<ivAaC1I!qw!+bsw)zJp&9{ZXIqD$2hJ#1nJv$%64H>`cxlRlH4@tzb@'
            ')>u$h#Gc%}iyn}l0r>Ju0S(Kf35*nv{Q0R{bNL_gYCQC+Y-o1UtvP*U+9(V7;Ox8i1-'
            'eZWGbHo^n7$Q`@Xg^qA{!E>}UqG!doHX9{8_5tohig~!LyNr@>@D7h8wR3bSmOtJia%sbJWR*c+$ppk^ug)bS~i1Q5T?'
            '$1kT-kfa9;;M1g**kv(KyH)zb`e;Y%((b4Q&0H!_3qTVEPNI2VGNRwT?h>5Q%0v|(OuKWzAT6P35b(bYzqFiide{op!6'
            'VOuJ^wi|()c^23=<;CFdw}RI^8nkWi5b;Qe1V;Z><VdOn{+BMu%y<YM9ak9P$M!RxrtE<}%VPX+?kc^VyM?iyyA2Pv?u'
            'O*pXe|Ho9)0igkhQM2;c@^!6{=XMvGmp`y<wUIVbvx$Pp^qit&3s(xF*23*R}_?|3Z3pVHkPh)L8RDT8eeD?iNO$iNtE'
            'Zw+y*Q#VBU`7~EGY)e7rAgCi!P80Mb@eO-'
            '5H<QZW!wz@%6VyEE`azORQT#z<Wf_}v@@RjnyZ%bC7Z~I0(VrHP)##jj@;&M39*$9GuFQC0P9&Ygix`nXFibDy=G3^L%'
            '9oyk^uDC{K$3iH(Tm;hV`sj+J2;Bd0ii%&{j~PAd896;);P-'
            'q#y8L4*m?#RO$VqYByDb33rcU5|kp(RNf^`VG!VK+Q#W>vN4q}~E44=Ozi2PhnER^7fuQM(fBK;E{$sEDAn|Yxz(3^&b'
            'R-ymj9FQ&-r^ew8q;lO69O9pkhwhd^{_`O)dHn?WHhMzrQ9s!E!XCbz3cwA%70~JHj8W6oq-'
            '|~)Bo^&}gpY&tNPIV3s!xHUrnwrtnFu;j8?kZ%Sc0nrsj^fpUJnW*ud?R?IaGu`zkU(faA9^%fD&G?dO^i++h9}Qb8^!'
            '&7#fzwV$vr`WF6y&-!1Vts-TTK3}Zn(s1ccA>rhA-'
            ';osjY$dLcVFjUQj1r9ZI&%kzkw8970d5^M&ZKBA#&3_?xD<v0dufw+R47k7<r*8iJ7-'
            'X)k!9@83=ubRADI^*VKG?$#(=<qXZUm9n8gZiCf%<Kcr)>*eXzS`<#3V5e&W$Ia$@3Ms=$iuGw6Oxd%AYV|&jTtQekiQ'
            'xit?8%P^aiDj<)|o(G!W(m;Wuz{t^va22>y=X&Y|N7KWRm8&T!zTGmCK4OHy-'
            'A>fcxhRX&|sSp}tY~}2jEaE^uVJ($1@&%P88K`^bD0cnHp@OFmk<rLyv;NK%T-CEs{$vYE2w1`Cv0{`HeMkrH3xKTaI-'
            'EFu9QP*sU{l9=2r5p5162hmu=W}2=*$>#?&-'
            '#<((CHnRhID2`ZfvO12{f^l5#H}g@gy4WW?by<DfyJx@cD!^^Ht~GrB2Y7}N&**(2nXU^@z2ib8ouVVsIn#iPbqc=BNd'
            '%2{{Q3q1+g7<n6(T@8bMt197n+EVpxyq`%$@?v~eXbzh~<EinHeNeR68Xhc61|jLDTD#GW$hY=2THCTwVv-'
            '3lPVr#Tb_j0F6@?de6{zIy1#{-ruxcIpaQd$wxoF#mN7}!U)OTy?J<e1Ni%x{M;vul`w-'
            'tV6P2is2jpWKTdBWWA3NkGMfNR_dNh}Ag`8-aRYW34C4-'
            'c`9UVVgHd&O7_xBQ`#W8zS7B9Ul0d*Y)F*3bl*kW>AfB~SN)PDLCp${fW-7O`-9ehFO~wUmTC-HZ=?TS&-'
            'sKTX{qft*gI@buabGM+w$$FP##(>pZF;TJLbQ8Knl7cmS}XLJ4b3#ol`3s!LC(KSml=pCi=IH<x0mab#S;z+_#sRN`<Y'
            '#)4Gd5mtJJc(YXf-'
            '$ev7eAmAvU?K11&kQ;pIAff(`WE^zaq|87=+%7lPE2Bj<A0?z~dTasJSf;aTC7qWYJ1|dtizj4soe{cs(91BWmdKreyp'
            '&^N}=~#4uhAu7p&hSXzGTD5LyS3$}coq&9oMu-dJ2AizeCeqMeXyJEsHp|KQ~mOnzSe=lgd##$(nt|#mN8o|Z}74Rs&O'
            '4i9pY8=_?&$<zK4^{sLpum^KVCq+gmDfD*-'
            '54)SD5b(C{cc#smCI7lNK~Je%mGs;Ul`)fgu9;a=^=hD)~T(F;F&`<_+OTV&_}J<y#FO$;cEf!uw1fAUI!<R0!E=88lM'
            '-T*om}m%R<x-{e!yE5h$gxnwD35CsOXa@LB*5YVcgd3O8@;43&X(qh09ntdBS^PNAYG2{raMj5$-'
            'Uu)x=YSZbNVzU?gfVCW29<&UR_<QA~rzidUp!p&4HWE>R3T9CiXf#?m!K=9_1=*Jxho0C}J(YO<&?RCLWj|ZK%bYZUbK'
            '^)&cMIFk^!HCxz6DGp&^1?anAoTzwo?jtB@GQPF9cDckiKglKRmfQW7JL3GfXU5bl%4iQWBmcbdpCfd-'
            '^9d>Mm4<Wmkmb-'
            '`=Jk4Vq5SvD3Rx59Tw@uYqu%!=vc&ZznYJ!0o{ybyZ|2@Q=nGx8kS|GLITqilPZ(2);ATdJj%!Nkxo=qt0k#=N6?=8CB'
            'Cj}#k5%;*?;^TDBo&_wTqWPmTd&+6@<}HqZVAaiWg6Yex(;0yRkyv6%W^FAU^U%`&TMZ8X1X0W;`HfsRG<94bYO^4!1|'
            'o*G3r*QD;#F){$R1(DD8jb{d3$ztR<W6qdl)r^QXVl72D5x`j|Y%aS!`M-'
            '_T2S`z(}^+ZCe8;rxd$bdy7XpUT>ZGktSgkOMlH8ceBhttW03e#H7kW*NKJCMQS1H0NDKxb?qY?EokZ2tkWRazD<d);O'
            '{`dx@d7KrJm_kk2Y#T!oxQO~6vmPa_D+5=;lX&Hv|S^Oa5-'
            '3RONE1XW+52spFQMx7q?mvrwZx2hMT;?H^oG^#};R<@mXA{(}RADTHa7a1kg!6LDp;r7Dm<``0nscLwe^(=EY5K*G+3b'
            'W5#M5AhM<QsfHX=XP)nF9afoC+H5b?mLL@h=JN8J?J=Cb2dB#ns=WjL{Ww$^%t3PSDmRcPMxhxNVd97$DeL-'
            '!}|7=lAz36F#~;hCyu-H#1od~}Nd1+Mq-^B<>1^cg9*x?w4J#F;>g@;%t_rx$L#5+!A^I{5639voIHqf6f+wGx-'
            'd&AoFeoz4etsWLLSF^#;~d>h3(@-eGF2%H5dJTBOW-;WK$w{QtGo8=9z%fqOx@&dn~{epuofz-'
            'u!fPAqCLB7L!jFP_Hcw}QO==Uc<Z-5?Y`)x(U-'
            'rR%wrzK=pb|#~)#(}u~3?dft^Wn=tDcFo8L*|$+MoOoUKP%JdOsxl~oGyplr`s@;FBF8D!!b$D01{G4asBBql5^gR+^r'
            'MEvO`zFmNC2b1p!A|Ph+}B9Z0<hBbzy#@C@@YC_22LE3He3$-Q*ayy`nO9Toz|nQ@x$ybnGbU501B-'
            'mv6Fs%Y~4zjW}#dDxt?2KesA<L=A|TyRzav|ch%Y4J&XE;fTDu{_XmVGEX7e<0Z^NviDP5d7Y=4VKw;z^~sE3`1Le=w3'
            '2}tDfG%zCQ+NuCM{yo}}T{@Y|5|Hh`{>cv^dThd5d5b^sc@USdYR7<9bM!#<TX`dpxc=$jExjg1G@<>n}|CxyCfc}!!9'
            't>J9=Fd5+PB|84kfYU37t{#@bv*H=l;>$<ef29L!rk6mx_7qiF8Uqjgm5I{xeaIFJSD%v1rwJqV7$lN`+jr!l_l6v@#?'
            '+B6v)6!6rCrFU<^@u{I`DE$A^aTqMfQlj!kpDZRECR#tvx3T^>VZ@v_=bpKB>Z1&s4m)Mjs>TLe!Xu!8aWHG5`1uEV~|'
            '!`+xY*O>#&@mF!?+L@vWmP>*K)$cL^)3^<iggd_G+5dMWr!?z`tCG=we`D`|#^-'
            'T{jz4j91&UxW8S0$8KBTg$Fn^BLZpkBNV9Bmt=Yij2}>ydi0uCa<fZ&AkbXT^}z;)AOtd+4H@CrS5`RV3k#9GxtFMP)C'
            'EK$+|(&{6iqf)#!A{NGHR+vN@W`h<Y4iet!LF@RGW!lBpx0*P)^#MMKwkd-Dwf5rvSsXqx&dqoRtOH|<1wg9|4){RfI4'
            'am?<NyeSSE;LzW03Ir3!_b9(cq}zc@^wb(GEO6iSUQQ<b{3#$);*Yzl>n18U9el#4F@$#U|-sO5;d}e{&{tiX7yA-'
            '{VE_`^QIt=Hy<=2jo?>v2Z^lM&pLe79BXn)$^MiElzCx@jn|NxW=1nEoxKVH9}CES2@cd=+k_W?57X)<Ju;pX3d?=I5|'
            'Pqz@+8y@Opjz^Y?vuLnIDOYkJnMPL_s(?xSR4Vybap~7oynNyYP31541npfJG){$Z=NzE%sVt%9}%Yb*2oC4tmp#UJLN'
            '!yYu+{xH0-BM#1smCxp?Q2o<9n$YV8Yyjk-'
            'E3yz4xfa+!7`j!sa*LqkR>M~g6nVF=y_#++tTnsr!cB6&r0*1poKD;owADjH&krwp>P;~qZ8eF`}vOSch?p73u`Cl|KY'
            'tbE0a6XSab$iGspHlR{=>oz%^{B$1MNe6u$IsXAg3-'
            'fdY~Q;ZRxN3zV@J1>?|+|x_L4ia(!>WH;sSBsP#0R|HGsgZo_Fc$As^Idec<FyM!&xg{9W$|c7|*4m-'
            'AV0iR;0(&O)Rb_C&p>8l^k4sHt=Su58SKFAtZ~5WzCmqS1L!c)^0f`>YDMzjlxyWjnAtdlMCSx1;UpWV{vQ1TzyAm~ih'
            'SmHpHSm%Qh(+Z21L&HPgAXvxIn+7)z$TL{AIi}A#?1MU=M<EgH-'
            'a8m0iezRGHUt4ToT)iIW%ufa7KwgcyE%n4M_ddz6s)7Z=&)_Ck2x#7q!FwknVDi>`s+gXHzqOsoi8<4BwQDeT@0P&@XH'
            'LU5{qy8iyMC=xO&;vfSVwi%sSvBBykPHw45z2fs2!6EeD>YYyu2FUwRz*e`#138juo(H{Zpc3{gL&%Ivn0t&2rDNR@%X'
            '#Neqt6MH$8iQv9E@k?9h2*nAg1FVBI=^;KlslZ}`se3vY(%BCZRc4U9TepDSxf=!7|sM-'
            '7&RrY(Jw4)5({Obq0G2+m?V;5YMZGjaB&XHizHj*Cx8Z?jZsZF$wg;A|$Xyz`%OG0H}GA9}whl8Qk@Ney_wn>!SUx0o4'
            '>!DLdpXH#s7-p7Tq4!=RsTxXw@th2}f3|{_ZiprJfpfw5CIQ)@T8QsbBu#l-'
            'YF*rC(8(qX%s=^p!8=nZqC&*8afIwtw}!tRcgd<UGmuF&!=m#taJS_bV>l)jEbS<2^!c-'
            'H{@n|kHthuAobPmVjy6b6MbUG+-+`XaZDd_C0lkDS2>25PydGR^c>x8ML|`2qim$+kVKImw?^J)XC=Op5Dxh775V-'
            'r_!a>M|UM&fDDsvWcIScXmwQM59FGO^!p5pdR!PIl}dYJz^7Sq|yu;`sDZZGM96HCJ>OHUOFzS+^yfINuavXw{~twy7E'
            '2KeZ4X<W0m!s<RFR2=ZZGll*5RWSocH|Imo(<1QR#3XIy<!GZ@3(xi-Y`&q5mdm<mgy0LJ%zGHc1QwHHM-'
            's@Kzn2+;sTydk;R^LJMl8$9DU$T^K5fwVN3NqLVB58vs1+2!Qp*Z*bA>i^R#AK=qXxtJ!SM2|558P=0Q`craHHjYLYD{'
            'Q_u1IwOpRiHN(Hr>X@i}YL*VGwD$tCoql#Ceq2Qe{<kqR<XxAm2=uidQfpqNg6o!X#>A)UerRuq(12_k>kjt%-'
            '&S|WpFOO!`#2xj83-`~FO9%O&DOM8lH;b^-XJUZy`#bK_?k4f-&td&r7IFT(5iG-ZLO*vNE;eW5P&Plx|BQn7Nrj}Z{}'
            'p9s9EGi~53#ILYvAUWG0?WS#1K$C3T2$-'
            '#BPoYp4_DiDhqm9c9Y6PXGR1+xEmsGJQH`#c;f|n35RwRF{X>nae*3=j=^W3!qKFbcHury)vLg`ag~U@OTfWH>8NL0LI'
            'W3V09!2<<#ZOt3~6qVs49ZT{ufdF&J`3jl|+YI{h%fnfa?rI;bHniEUDnc&HODevhOOamr22o`VOS1&Wj2OJ3xl{esGv'
            ';PMDSobdgOpx;)4rZ#}u_kLKC$4;;Wx%hgcm;6K!4rh>(57iiE*BzmJo5I(D|PF{VEGP!KL&utGx=`iq~P{-'
            'iTdpHtx2;SQTVu@KUEbfUX<2{~m)e^BMSV|+rJPBHgmaxO;ID_`#5LA2f8q6o_$U}KYta`o~O>`aMP+}#N8K_~*YCUj^'
            'Z6U9%rr-mIAbwOn0iWWL*e1!tC+#F`%hRCR-'
            'O`LV)@smY@&Y~moWRR{17qHvJT%pq14s8ug4vNydgbCBGzqhSC%dzW%X|)CmeyfncmP<w6veL+m6*cL0M~}IxLKOAT+d'
            '9Ch|dPtuyYMr`0@h%5crjr99$18&c)$k>5r_%l;VnA3*q(CdvJO94h_}#f&Y$VK%Yb*gp5?fa{dEknUOe5GA^Tc?R)h!'
            'p)eF^Wl@`50`UB}3qB3bsnwN9ASYFp!d>l9P!;7ScDmmwa_g{a_*8L)$V+gwcPH^>F?8$r9RNlUV>oXY-bu?u&N(?~dQ'
            '6)d$y+mA-p|mgEfMgy;s&1Xo`k^MZP00kaOXuED2!&|Z@q6Mc~BF+N^ooRM?R-'
            'hj#VJjz=_Hpq3AmJjNad>1a?<+SPKmP0Q-'
            '6edPrTs<Xwvx*ZE4&z+x_5x?%!hHXfL_dk6B({6r513skbCU@*gjrf$81`)3Rwv9p0XOg<rC+yNt+V<5I99qR|JN&VU?'
            '+G8IG=EMjtL?48A+S{P?P%;jiw}6Ff9rj)m#S)cbTofEmZZ#Q`Z3{w>8F!yZaO6V&p(F6=Ljm5u$;GnSl!@NrNubnI3b'
            '83CAhn@^IG+h5M^OpYMpN;M{!j8+Rk~)M!#R|f>!cYaN;qw81&T&%VPk3@O-'
            'lBqUH+ZKij~3$Qau4OE{CwEdJThIkHDibrP!&HilYIccszlJ?An+G@4C$K@$eip%9n(nu8Y`}-pQbnl#I0jGFY-'
            'e3XIAENyJfUIC%O2N<_~G#TCQQVOI!0!lIye*(7=PHVw89#{*-'
            'uig8vaoUHsg7gn9*$0r=>#ErWY!n%&&_5@3sm{N`jp0$vW+C<gAC_#aN7D-'
            'G!&T4()3Eplh<dlmrJ1llSG8A6p9M)!b;CD~#oF{{&_RQK#Z)G4qIRry@pMnMTG35A(>-a9b2qH2J;aAxl_Ei-'
            'D<Vj5<X95|uo4b!eN7SqqUBZtW_vX{5{AFOR?SW0Qr*YYGFM4ZLIbr}0`BsyK4g!T(#VG@6V=PR`izA-'
            'eg)sUq8s;z21iyxr8ezw{HF)l&!LZ|Oo&q({-|!@)&6TDHj}Bwp15Wm-lLi>GHW7p$uf)6Wq}j(WJtpU>Bfz7wfJPK-L'
            'n|W@D5(BLaz;f+POdPGP0T_0c{eb7JQhFZYk^E;A1u_%0GY?xV9fJ~Y?M5Ok2i9mu-'
            '9V7x)q^tLiaFC4;jwR9SbxYFvR&B9<X{;mCEVcp+MCedS|CUDyg0#wPxJl!f^ru_{G5Dhal{#97L!1T2$x_h6jNqI5^q'
            'B*e5y<w(Y4x>xo5JsvE_ss5ih}+x(!9osNuC0hs(ChZu0)fHxsoU|_l$lTEl7)zdfO+{91vbutX8ZYDnYXonSw`dFA*L'
            'ncno$1io=$clRbP7+&dmyk3p-E^B~&T_N-'
            'h5N|0YZkliS3LCISzW2AALiOb!dd|~bfl``PRjw3TU$$H69wsblRk~+MYx_;1HMPz&_&W6<hhh5V}alh?(R;+ZOtzUzR'
            '1LZl4Ed9KpWMchvW40Ts-4&90$^K@Q-'
            'XEX!ALvpj;l2yW|TN*Un<gy>tk6^~b4{5Z2sjUbxT~f<B4)YB|!87&okplDBt&W3Lfzcb;OjA5MpEUUT4=>cTvYEX>??'
            '8tfX&7)j^nXk2}H3=*Q3LTu6~7*^H5YfcgM)mdk-VOD>6FML7^|M9@jlWy?wNe_duq#v|?hvHsqO$=SY14l)fFdc6KH9'
            'I=tpY~tIromO<cx(U!&X-'
            'qj``QTgHx{wxkz&ZqO@WW|Y*A}48;6e`gpEWI{Nt@rgPta1+wKsK{k@Q6l>qx3a>(0Lu4u!R0Q|{<?BKj8vSw0;3eWlu'
            '!>4)}b7v`>>Q|!s0`ovJ=P*sp_y?=SFX8ty#+1$Shi!{{)#Frtk@ld6cq0tRij)3eeky|ADLqKXq?K^l<MpU(bOz@|OM'
            'r{bCoGcigF7K}u*fVLw5#h#1Rz$XSCC8l!qiMI@YT3l^kdPZ0a~D$2k{m45MIE;{%|%M;(nUoFrxx=L>>|2DTE|mMV9$'
            't7nrt`p%>4qVt>>ScHN1jTKm|b<v&sTBJ3_hd&Ljxk*SKijJfHRi3EJor~$<bU(k;24d}(pg7nxESQnOnyH&+70;Qqot'
            '_{4I+lDP;ad_uT9UkRg#Q0N_Lx&l5P$qd01E)_Q*L@S}o)|=W1k7<q_H%NbrxX*6wuAQZztGR=hUd=4;*`ZGLr&iv`Xc'
            'h_fjS=i@?{?e@SVX|^5bawaT1j(?g0;dfoslHlE1|%IC-cJH_FT4yVZs8Ya$hFrq;v7Sv}IYUl51eOR4B!4vo-'
            '$jOiu6P&4s1Dd*#0FJJ!~eG6r=tbZpSeOrxbC5te%F&W+^H6h!n67zpAB4*#K7>+A~VeTejOb`u2(XMn@x!IQvTLTG>O'
            '~bXq@wlaSKHD<qH{RLa&iZW4jh7B=p|8!k;hMr#kg(kk!iS|n@sc-MSw`a3L~h_P5QM(#_hA2cGqHFvLf^SACXSmX$n('
            '`-hzsz7^kFUZ{ri@jU&@a~wrb?xA4Nvv{Ay@zN`Ui+RzgeT7W6$G525M&Y@Z91-'
            'YvYs_*Z!x%a1Li8G4yG^1F#Dy?M_t9XSco(lr<%Re~lqny6SVi&G=LETvI?#;TPZ82VL}t++>(n%Z%*-'
            'TbCm6DC~PGv^0+_05+IZJDCJU9os)&lB2ED$0JWwv!y=Z-P~H4m-'
            '2S7loen;{44!QR;F7w(}eZJGV<TTd0fhOmk@b(5h!NYlcAQm;dZ7hT!^_saU<U3>iOK$WUD;njW?W-'
            '&ex;V_6gG&aq$&2+88f$U3s^Vl1`x{zK2p8pESlUErV%m^#k^bLRbK+~`O|y+#7O@6ACV;Wi{c>Zp!eDGB|$2{`clI2;'
            'c8gj&B+sF}ia?RZ2m7JIBk)glv8?f99va-4+TP(|RnkcV4D-'
            'QmF9cKY(o0gz|z$7gjF(CZ#Ryn5!aqikN##cFI!d7nr9CvxFUv<$kp8-fDI7+(9fone+cAC)-'
            'o&<7v9$v=m8$Q!&J9)EgH-o9z3s@a3MK{pw%8qQFz!C>IHeg)nJw$oX&1<NuELE)GV=9`J(j<vTyK;=8_X=207Cw-Xt$'
            'jP>w?x*G2MQD9t8LTgR2OrZ$$ZzG{(6A*N`kt53Uh`4bty|6bblV`-'
            'o$7(RK_Tc=b{o@ZEZ99M!Tp~+VCy$I{B!sYyznYShca#&Yu^aM#r}-'
            '=p4FH(wg)yhd!vMyC`vRq!%Y1a>Sg<w%1QWw@ZJkxH}#w4t|$$bem=DPfj&NOONELR&mgO*5;7J35D|Z4nrVKEyt$r^8'
            'TFAY*NEG|_%;BxDpyHnV=hZ?Q5`%8O2mPmlhoVV47V>cK%w>><eHut9O|h@`2(Av@zD&pb%hXNl{Qwkay5=LNy2eHU#J'
            '+DLZ{gn*6yvvpH(NoyJ#7`e65(Cev^a0!gXL{?<abQJB>bZx5L}P;b_yvN9-'
            'D&Qk}vXYQ&pETKVEJ!?FP6G*v+KND}C34$(bJZ<6riVIXxb5gS{I@TPAlTnS9X<eU#+tfNGnhfd;h(IK+g|0vS+N%)~5'
            '1wNI$q92WUU_y7C4j(#!inXmU=kN=9BF6y(9k$`humUXp!Gn^*k6_I`W6WDTPFKF<fvZNPc#B;Q*JuCFy=MW)wI9L8RV'
            'H|(B@Q>gPD6*RTPQXj!P*wZLRV)Mu-53plB18QpK>u881>>Q0ST<44+*@;!tS3pP<yE-'
            '_!pj}o7neo`s#d?NKl3dl}z9$`AKZw6~aJe0eb&xgMFv&Vbj%6xcAEyRu6w8k3G+Tl4L7v&@@Jc<w|6~vlrRZ^9dyPRp'
            '5dgKE^^>UXYs9g>aW3c$wFQ6Mqg8>u4P`pRWLJ&x*+TuutUsHw|+2(-L^O(HnhyME-lP`tAD>$h6@go#MxcmyZ@GeX)h'
            ';+zEJP9*?^ALU{G35oY>62D7&p=*Ar@p?u08o$|JWS=lr6ny`Y<;$=9x+@BVI>&5}Y5{Qdl1X{8kSP~Ni2M<q>(8Zx>>'
            'vsfSZI!_xb|5D7>tfTsU|g#vK>xEpl-6BBWXWDUe?FJk*kxg=kpU<ir~(Ct2;@N*Y!12zddKT=<2@sKl-'
            '~=!oWDQ}P5Q9s^Fn;P;Vt1U_M=*d)bL!vPE4N<*q^?G7DZct)`%Z$@{t5TfwffgyFW$7J^20b1U%Q2p7jA@5Le*?^Q_Y'
            '7B^Y5?L<FJlx)!khu^xKAZlSt&KH|XiLV)W};Ju<MOSk7Q&6Oym-'
            'U(B+Yhzbotzj>D8&VG*#Rav?w!J~Yx&!cKvjM#ARmJ7uY_i}@EZk|C!BZoBIOMb)epw{~*RlXsym3E$@oOVaPc4HH>ks'
            't1OA}<h+(5hshRL_)GawoofGfX^fr*MgZ8Y5n7H`YQ_@Q8!Xyu?HR^HIK&zX=Y7UAENkHW1Rp?R-'
            '2m2BySv#VY}!=AHvKCF;dIbNp%Z4t8?w;MJ&8_~1TAMmT=T<quiLf-'
            '^Nlfw^EP=SA%96zfHTD6%_Y`BJ+?%0H;iN;ubWEthoFb4+fCbn+nNABF4FyNm=hTQ^DM7{+B9N*CA7hJ&U(pT6&qJ$j>'
            'OKS^UJ@Db3NJz76#+0%>@F;WvL~a?S9|VnIYN(4&HmoOR)BO0hq!W2d?~>LlYzQfQLj0N~Q2NPJ2s?5a(*3z0a?%SFdw'
            '=1zmurE4lNjYF9RbPZ4{PfVJR-'
            '>*t}r3mhmgg@ne_pfYq}7Z!x_xeEQIfQttj*33kB&baNRi!m%Z1+V^f~!dwz;cIVDj3@AE+BS1I(lzNfM442aIeJ+lAf'
            'PLQ{|g{9}l$nUwwP-'
            '}%6S{&?$2hs+3&u1V0yHP}bx@^Xohyuo=9TB)f?iGdy^h1wL1=`N)V2#xcz#f!AfxrB)!cC6kSMSG(fc>C(OAiZ|a)U~'
            'VIo^%VQ(rVJP9-$-'
            'F#p~zxOR3sfyi^<oMVGNURFd|DiX)tKhZILW9aX+h5mXE{3)*nHa?TIBS{nETiJ|~#`Cyr&nq&aw*f`3$3m6PM&y|+#C'
            'rGur&V6iv*$Zd^IR|Mt)vvLJeGp<uR76NCy&i?Um4Wb?nc&;0ra}xg=eZ_pxA1RVZ~*QTMrhapD2(3j~wKVe2Y6d5bsa'
            'F!H#f0sL3{EZ27bX@-'
            'o}OBJDB5)B85+cd9|PNd`RsRzu~TqUd_wUVO)~9Q|`Vaag~SXqmAg(lQfwHLk_e@~K4Wgfi|lh{N6w?T}Qu7xC*OsCiL'
            '~UI`u$J!c!(R0yE|oY#!amsXO6KItH|LlKy^V%RYgi^oS^6WvTc*jcARw96wIIW0$MY7-kBmc$U%kZ9~|Y-'
            'czn9e|8N9w>f(0_xZnDD<%l_MHuaWu6VV#Q!QPh6|D7bsx#R+ZE8KXbEM?00y%Ap+RH?P9Js0KYnM_Hav&`orWBIKWb0'
            's`xRr1Kq4M|a1-@4R^ZVFd;F8ugkNP`aQ8?OHCW7nCsHhNd3z}u+TRAPEgv8S0%@Z6ad;}q3ys1Vs9j)>QU`945IbQs-'
            'EfdbNigA5PdO2AtRnsWZ=j|u5hYXmuz&C~?RL3^_X^@jm+eJJ&|3(P9Akh_A`4lK))2gKDZ2Q-'
            'Qop6=2Tcuaa7(bA!TKgmI<H8P<FoNrvRME_S7o5SBaZM7j8aYY31WO>F(%4>$EoDwRMzwuW3gr+EETQ6g-'
            'bJF@_rpm)DO|^w2;i(u8IK{x!H-h29~v)f>^&~(Dva0;WjqCrBqKAHXbGK^4svzs$4W)IZRFqT!uuyc{HVa2(-'
            'rRao*GI`1GkE1p0R|ev59#1G$fI#q}h3Ssag+N~I9wa{{HV>;TC*^^n6>fS#6PSo`V*TBTnC+x><(7JUKz74(VQHAi3`'
            'vLW-Qi&(!V;;^*h0m$+1LKlhEbp3Zdl)WW`U56F`TP&KMw)l-'
            '=a<3@MwiLAEbKv4gDzWRf!k(aq48;%CwF{G`X|<mUdR0$ge%&z2ZC!<`7siSHd3jv!bOAQFKf;`M)1*K48RTEOgZ}g=i'
            'fw&>C)eM{(&XE-TB41fzo>(C9*^kD^lVt$y?|B~r{lw>oLMc_4z43c@Z8CD){i{FcmDI?n{z6@Xx|GD_WWRIc6_9pUK|'
            'J4^C!UYgerusXe1+w_mI1$8GP*mF+3|9&QIy!Nm(;A|I~uEL$9f<+<vI6lf_={eGma#pqT429dLR@C*)mmYu;BVV{>R6'
            'u?0v>=)odGE;v~CkZ$G31);UdIK$on)e`L(8J7oVd4|ZJ(q_zAC6B`2x1wVP6UNWfVeSkQq)RX3(MAyzR12of?;1ePEe'
            '>X+qOnKuEmiuy0l&|rVCrU3XleFC*S$Vya^48g&ITWDWsoWPT3ius4kujXfP7R!Cocmq=(__^gH}|3#*vgX%}{CX9kup'
            'W-O#Ef!$QGoFg*U6G=6nQ_j^c1j^BWU%qHYePs7GT%dz6j6%-o{$DLp57{NWiko&JUyev5k^_pi1=k}*)^+ygG-'
            'g9C`_XW^<6hhhSf@$jU5s>aTBC!YUuqZwis!pwks_});VG<6Usx4SkAB=frHN^gTA=GYcq<WFwFtN;!sP#RhA|r+1pvi'
            ')}Zitqv3vq*8JFdTJ1)Czq(P`0d!h11@xV9BwXxc8w<xHhI6_O}8QG-'
            '9aydmu@kR|!OQ0VXl29CYP!BR&$H|GFaRN6wvsw|jrF{K47&GG$sBsuq>486=&(H?~|6l&c^_lTN9dpQr$OwPlv@8WRF'
            'uT5}!dMiBGlt#KEMyX217kuB`h^{q}_~~a0h#7@|Plz?j#uZTuWkEPvunlJ{xWF`IC8@vtm-dPofRF7K*rV`?Jd@i<{9'
            '@kM>JVOtxs-<~PqN8@d(!x_E*Et-6~JT7wU}08h3mg<qF*vQS!X-'
            '1gR+7I#2n*eE5_V_ZAG_9q2CEuu_zTT7scU@sRqo<5+&YTF}Qco3&tI{k(C-'
            'g5XfClnjWWuvV8~DQaC{cbSm(BDN=2dHK<u4hN&S{bl#BA?DGv7;aUtM>Ri}+dJ!=BtKdjb3@psQ3PEqe@zOI(ST-jW6'
            'ubwK==I{h^lP}4=Q0c`ze4_QQ#cy88=c=2!l2E<SwH248(3M`AeKNoMLqChtu(w9ONS-'
            'O9LUu50vUf^jf0a>(3*V=SC4S9<l{ZTxW|}w8$ATEd_L%F?F7z?=kc<08R@P_qkZ%G2#56pR#*H%{J6dY^GZ9x$!-'
            'tEp4Onny5AVVycHlAK#9)r%`mimAEqt!2b}uB60x2p9dmhDH`X3T4Ht^X{Y+5jz#YsP)&j5IO?csaC)B9~Vd(dhSls`V'
            'I37|#&So}Uzc-)CzEuN0<_TDS{U)YuDkj^VvZz(bIk>2?3%AKWtF1hoh}IWvq1hlFcIL0f*3aMQ*M&i-'
            'dZmFrcL)dPlNGh^PMm^B7Z%9x@c`T3)eI>INlfp{LLsfWL~Q2=%9)x8zIh?2U>`(zZYW^C!5dazog2vNL}K0PDGc@uq{'
            '}#OfmwAhZhRF8u3wI_dQAC1Z2lxv9eYjs2JPXJVLNpW5{H?XB%&F;hE|sr(%o8~uqjp*#-'
            '_f5&f04<|7{+=H(Z78Ta|J0d@7Et$$>Q=pTIdmADa45n$FQ*g(?9Bcw#gOr)vG_zm`s_*<JwZTK6G2s0t=uhr_U@57=E'
            'Z!MV)qpkfwK%aIz0nY*0u-%243uBW}&u}T-'
            'V%P3(vmloK=H2IV5g^b)*bo)}w=sxwE^6~D3`le>6DG{UgFZxNFl@GmNHG-)%5p->+A_m-'
            'z0bcVdYOEVVIG%2231zq;Q$-Ago)3arRw^qn=q97pO+Z5@(F#v5wnN#<G(0WYfP?<Ra95-'
            'XKD2+Nn<qPnYvBe`t?$Qb^$f!Fn(gqugoTgg{m73|OHepnMENe>#g&q2@Z`8CZoOfFYX1MI$@)YjBMHzKatoK6d9m6~4'
            'nfD6CG00$##oTp#0XJ|$Ewe-7%K!-FmLx5DUvLPS9R*-'
            '*{~R{;F;w>8i^O}@3Zy@WTC#^3=Lo{0o|ksS`f#9ev)I9YvK*|Y*`5Z?Z4Q=!~4JVU#a~c^dFpN1d}0?Zfc?0OHLj5MT'
            'VE%pY_1zbU}4DM*c{^`$adw*D(#P+~SDEF&FUOXM+8qZTP`;wif?5h1(7QY#j|{koj@+^xatUGAal!{w^YZoI63IMw|?'
            'byeG^4y(PQGUBSJzjkwOvr8{@xXoKi<ZJ7JK|K5M&E9ay4|KESHkc0F8i+}S@Ee;N)E55Mu^>z^bv6J2yD8S8kQW>jtO'
            'XzW$MCwPL!ZphVL=|;BQke*xSuH61aRDPk`2rc#UJggSav(dk5~S}{Vdbnol1cQZ*CsT8t1l8)+`j>bHv5Chj>Du*VKu'
            'e?;6-MO4$Z&bODhB)lCJ7j+V2}i2la)B$esv{*{cG-QnP4DzZAFz@oF5Xjl)aYvwCv}8yT0M!UdaMbft(Q$h<g*UVEnL'
            'F?J-~dLRHC=5uQ-'
            'K9~<Rzxvc4I~kyJ&Q|>AstX3riqI|i687A^h8<7%@B}|MRaT8b>xEzG3&nEv<S*^e<F89Xf>uG2#v*v^Uj`M+FM;e(16'
            'Ix#U>BOK!px`dNky^<Lrx?Y?yQ-ho^$4~OUYFb^XDPTZEqoa$Q4!#57UxU?$lUc94=meO6M>WXv)1ytQ<*uP~j<otlHa'
            'HtM!OR3mN0TN)B)~JOzA>$8k(E8*g|_;QO>?bT2y+KM&QCJ>B^rcQ*{uK8+CFrDf>g;Y0>$9PoE7AhD+5@PnQw&!aD53'
            'g;*&tZ&CP%PXO}5wSWq0Avlz$=_~1*riiIm>gsDdOa`u`_Fl74f{>__(L_d{xd?mhvLci@@`<C*FfLyNW5XW024a*Lsw'
            '}Ue)6kDeR(fZ@L@j)o2#O*$U;mX_s7BK>u_CS1u=<?Rqr!&C&E6yvo*~LW7F#y%SAt<X3`-'
            'd;L%H;+S=p6okff?@o+}bwSTCf4)k%sZhHBP9PpS0L$!ttO6@y=bj3w%<fVAZ^d4AwO2EO+`*4Qu0GUevN5#@SfbY2;a'
            'G)e;G6|e}a)#Ww6hWJo-+|S;LgDw{RMbW-'
            'n49_=<kh`sY=RYT;vL6tH%1v<Y12gC9!SxT25M=(uIAC!KsZ0vgaTVL3E!eP3<$aoSB#!gUxC|LeR(B%Us9o!TtVReWC'
            'B$4P7#w&(a>etgK3U8;K%nxxSf=P-Onf#k&FXHU0;k2%Yv1@3)#c{PUx8uLmjv9vz<FXGfp*HQ#V-'
            '|95&1Xk#mY9VrDOFSd@-'
            'G7pvmU2P+}dNe<49X935$Qarop8A_Vo1&tLfqJG*FHBMJj;^hk`3bJT>1PA+$cnFnOvWM*cIJEjQLP8A-'
            'a4D>({Q|+j+?ojPSroSA@?b&PJPpS)t?(gzCu^5*ox0fa7znwbg@()=5Zk>F4MwElPC*o9C@R8}P+gXicoP0<>BQQW7F'
            '@4r2GWKubac3oO08S~2d!IRsZK6fbmtO-'
            '#?`E8_9(4ClmO54ccIAR2HYC888os=@vGlDsAsaU?n)CxRSMw7cZBNPc4IicnZ~dEmhf!Qe0B`+FyDUx%DimHMG@ujKB'
            'SsF;Wz^qzZF7Z`f6C;@r914tpkmH9~rkgT2M;Jj3y}c)2I<^cv8**dKbN6GPo5=qurtZXD37MQwLgKPs0cM)`CNX9o9-'
            '8p)8MO!1a$%jw^HEa#A|z#}$)c!3SV79gKUvKB4!6Y#Gb*T!6zaj&(am20hfov6=eeDOW>?Z}$c+(=k}L@)M@8LLk3k4'
            'WvkU!^7=+;7ZFW;vO1=XWuRd_S+X!e{l@%k7uITeGN3Bj<Y^xoHRNg!)XaN96l+G*<4<Xz@=B<LsmHCdu@T0a}3eVxD4'
            '99Eu&Yr#t?qF8PZS?g|}^<qu1LG+Mv-xl|L?qcXsbdA$tzX>+S$t)srEThnms#5eo(m){;xUgX+W5UDT*v7n)uu!|<6g'
            '<gkpvO#U*|mXv~5gO%vDZ%p;rDSyT~5mzkOYEGl~*uq60Y2dw5#Nr4WW~Giq;IPMiGW7iiN^h+~y*4xWai)Y}`KSl0c%'
            'P%akuTlOr-Heoe3(?&fpI&1N&CqZa9>}GVz#ZQ?@)>@{3$h@=f)U!dC!AWVg)9Zzk##wr5Rq=eW)K-'
            'I7lyRVW}0VW3c2CR1A_p@wtd-'
            '6bhuu7xTtfLH|<9s7@DweGW<BZ61RExDwDb3doZ(PWWd37(3OK>DISt<l#R<Qg~PvCL?gxb2-'
            'DWxqrx(icMI3k&po8{960jxo*>23=QH>$wBvn@XxaX9{mYmnKEC2`N(CE>UlsSy^3&2gaO0<N-'
            'l=$|G|no7trso0#2^br|K`VVPZ}{V0$3ktyRTeev0J5!~(YJE=N*1Zb;X>yN$UG+{oCWhAC4wAuc)*H5KIPF~7(7r9J@'
            'rE6+ju?xXPb$9|mN;|jUEqTpvM(uTn?eB+>oOC=ssoy&5x%+{37Yhi(q_!x}Z%E6dtBN284JhFFIHzr7eXp$5v&*akuy'
            'Ml4af_9d|WDy)Vm5on%ACT8|<=B<86>N^L!~CIGYIV~MZf*_7ah2E5{;`(cI?oBA?w+*TT}ES`7%zN3b^`>jeu4)LY$P'
            '1qnE$y7Ir<{uuUtPS-<ZVTf<RyY+5uNrm80XIcKW%toJy>UAp1({SPs&ykg&!Y^W)#q;($r@fK{AC)@d&We#^)0H~-'
            'Xb(X&Us)K!qzYKAipej#5}0~r-f#O!A?*du3&K5heGaL*h3Yq>!6$t6@ix`<s;oldJJAJRL@PgtI-'
            '&(H(1i`edxg1}fks{{8g#}|^E#H{27XeXsW`SDL6d6$DNbK@B+C9(%fbTg5QJ;eC;Oq_k?`F5-'
            'p{|!s|PC`e@b+FZ{f#*TDpl;<GGH<&sX3vyk`5{mEaP}u`KX?UN&K`rkD*|9-cp64FMZna`7}$`(!%jWR0dxO$P$z-'
            ')=oYpW4z-2i&VX)=7LFl8{5-'
            'fUnHvqttsu^M6TahjCVD)#@zi@UymvSlDtBI@xhJ&Ysv{dzg}Kq=kp!{YngseozcIZz9oOxg2kdeQvcR#0F>x`NC|@5&'
            '?(t+ibAv~t*&q%wbY7B^M|Q$o@|BcVdyy;$Z?I5I!2xX%)(?d@w7c*h>$l)Wbz(Hk*fWt$47;nrDI^GK_Y^DN>N{oa-U'
            'L%uDpBI^G+8qDCAQyNhh}e@$ljI0u;7gsuq*1o>B|u?y-'
            '@~U`#kY&<1%&%|2Iaeh6(NbF_)G#Zp06F%fK>)o4tAIN}!+ifS(p8l%FZ0BS(edwJsNGp47s`WENhT`T=h34b)rFk>0z'
            'TiJXn$I3~)=nk?did(t~WZpRu-vlK^_<S0Dk!-M;cZqCM>7oOc{gP!}{_|Q`bzv)&&-'
            'P#&>V6%z*tv`)!0f@RTYf$~vDG)mng91r0m=~o0ZddhCHI0R#zi&ZuM>F=8RI{vmFOlX}Z>qz`hN#z(5N2J$l3P{<mwj'
            'G9g#TyiVG{?IMkP=a@&+<3jj-'
            '<W1uVVA1HR%LVCODn*eRCB@Yxtj^y7Nyc1>lhRNe_up*kROQvnYz`Am&hgp=`Q&loqa@1@^_R>MZs`3TKpbnQzk2-'
            'f(8k|D1cnv)Aqp|Jp@UuS|@_91-'
            'iv=ZGGHUd}LG)|11M~M5#GBPfOmmc1X{x6rwnz3sz=)%u37O#hOn@r(t|6J^~8lqJqbKr#b7V19Gg-'
            '=h0kk#sxl)+w1qE2A{;Vun-`HgYqd=edqXE9a`ZpWNNf7C2B!Y|dz7%JfgZ-ZZAYkDFzIQEZpD97WY=G!>1O9*nlv&il'
            'A%S2w#2X$A@VLLDJfQ@(K>5dK#STeZ{U(Gv+t5>nWd0Y&_|N4Q5_C1s}QzkE(L$IaN2}dp+#_*$;A@q3xZWFP_&6ab)p'
            '}vN7zxDx%vYYkS4Wlf5v0mi3F2W|co%BRh4X!pXM*XUj@cm0C{`tUwKDS~}7mLKlwsGLNmjn9s<e<#xA&gb@fWnbrQto'
            'I1@rJ2HI?9)1%HM+lVNQ($eR-'
            '&0{0QR?XQGmHEY8!ukC#g?5S@SwxYnEi>GAV44(GfevY$1<d4mvGhxp*+^>a9QLyd0oNGE23VQ}f;JdFt5)95pAAqY$L'
            'f!!WG<WA<*_+qgR6s1Op#Hn+{;`=BGF9-'
            'v%4}+jFrUk3Ts%Y~eRpiVL!gBf3vs~s6Pp>;+z~R%V{=yPfHU@)O?kGc+q~qC)0&?KbIsE%P5kqz6g3iVU{5W-'
            '*xYg>SVH6L1HmIhLZeK=yeIX5Ttr(zRd%*Dk56+zthSfanaA?tYI&euGb}#TI(QRRPdck_C{4^iQq6rd9Zo#CRAL;e~N'
            'IX`D13NGYEY=K?=9Unw7@7<DK0jIMfsgT$k12-8#^NT{2O3eHhv^?rV#-'
            '|~5Uh}fj?N6&viJok?y1JbOJ`WGyvreCYAb8AX)6glx1L12{DJ&Z%dtS3pZZ$uU>#9op_Pj(7-'
            '@d|f2^J9Urt}t#v`Q?si+W21Ck_(y7#&%#V<n=N<vgBLkb~da~jY*&$H%vy6?SimF7`{Qc{^ysECU2^m+b-'
            '=kvTfZ_fFwv(H-B^}Sx~vq{y%Rt>MXQ?O+9Y@e>I0SWn~<m{IwT(Ub32eu3l-%WS1)nhsSnigaV4`$E{hX;^1&J|-'
            'LPT&<I0os*z0yg^^LD!m46g$5kOijbsb9ZafH1}M%^>RD>NXlhjxv>M3oNj<-'
            '(`lNL6b@7=hbR<q;BQtiUXKuB+z(%fb^S9C)EEOIMH|5SCyN|DRtV>!ZsN!8YtZL?9Y(?mk^lApzFR(9SL!ekc^iUf^?'
            'UK>w;ZBrT7a2_Z)uwHQIwAjf#~vA^xwrk*x#;<2bL({*Apx<=iUi$m~Lcz*{_YFT(X!e`)xKOA7URENQAp-'
            'E}$5ak7GRtao_S*(sr{37oRJk!@&iR(;Q60hE_o3`9yrp6^Q@+*iNl(NKrStEx=f&Ma(C>A>ddgxqrX{8YI3Wf9)&O8E'
            'z!}w-&*IXc;`PbPYVW6auF4N#Gj&hd0CzQ9Fz6u-'
            '^SNDZDkG8T#Y}Np;=>E|DG(8J&S)Oc6}z3xcrEqqu2BIazL0LzkDk!SKs!$bB*Eb@!a0=FT;;`P=|m?q`B>K3gcA7KcI'
            '|K{Uy!rr~Gyp!G5jT)k#LjJ>x(hXa1_X>S;OG7UmOo@iRQYdLc=s|~eZ`yo?F3`PI-'
            '5reiM>_6HBPZPA@*=`T?i}eHkj$yjhy%@rR7IGFe50PGhKvdy0<D$hD;L{O8+_(=z$M`>LEpq^MP2Q0*3rTnuGs)`Uw#'
            'Lj8;c$g#1G2@U;lZ&QY?`wcn{2`vXP2oF@!e*4;7cOwzlunx^UuV&vE67czYlMoQv<)zqj+<}e{l9iA^g1D38F2RXq<3'
            '3&MAyw2b@rbPvaACc;_ncXf%LL52G=5eJ?vK_7+VI@xcT04C&3RLDoIj9$d5f1V}HRB=<Jn!6e^)46IItg-'
            '6#xO>QB)ws6G_56n>d-'
            '#9#UPlEYwb#N(5fHEW<K}{fu5F;MgEpZWiN;}xM3@(5}T_62e=>jUBUZL8yRctNQWng^NgEA6IFzeO~Zk^?XyNsh`^~5'
            'Cx)0TytG7-X(_ohdrl<?yCRs8YmFy*~mNJ_^)gJsAxwX5f0s-'
            'FEvzR#V@aeDe2^l$N@@t!<#MM)4I@!BJMz5}f&bHi$*Qj9oh1=0;WVW+MboGDdeZGN?wb5e8>R{RM8ot?MAr+lmW^u<>'
            'YOuNC`zMK~G@WB0ZyK(%>5~{RM06HF}qokDtI?l4y9_7anxcCd4TyYqDWoN)KO%mDMiL|Il45VFB86$$?U|5-'
            'izWk4&a{Dl)Pp!aCgOBuG&A@Yiv(cz~99^Dw!`#pNpmXmW{FEC;bPhfx>yFFg&mRcg-WvG4XeT`5S41154!}rWX7;fpF'
            'dbP%R!0t_lZ6<t2JS)Nb59f>Y=DA`AHgKWfW~cVM(sx()LOQQL~B35@rEpX<o=SL-'
            '}H_=%8x{)fOz~VP={m1!&tJNi`xJ7g;NW?Fl%=$_@%hR`O|0U*?2b^D76hM500^iA9@j)BzJP}^A%`x;$lYca>1dE57B'
            '*IJI0t~<4Ln%(6yVu=<lJhy=WWCubkELcTRZHW-o9yR=|Yl84Q#>j@f^5z{LD7d{~wTJ7WW>-'
            'eD%)M^+KOa3;PB41;;OG34WierUDrBZ&|BnY&-'
            'H;99sTI4X28jy+oso(`S(MD#hm_}da*9*xCky^bVDL7bAvD8`vthWWe37U_|A{Id5WIbG<(uwTnS&RIL^X!r-'
            'pT9^={@52^3{05#FrDL1-2Zqw%98U9TS#aVH#`ZI|c-'
            'SD9sv4b#;N^2U>y|Hv6F*95rR4|qlG(YxY$6NAs~f1?hdLs4UY>q+IYO_D3!rnj3AUZ_z^K+!^xAB`$l5=_A{ygxy)Y5'
            'h-}Z!i`mq?4|ASmv*2j7=rix!ibHT-R3oJE%gtvo=VYhfNe2b5VXPkT-I-'
            '7_^vpjb3LNexSY^F=c?Wj^hF^;e(ZMJS8Q^f=L)>jEnDJ-FDbe7Qz8h45KSx+!5-$0$t8(>-'
            'EFATpUg5(Gv{HNegzO4|&1NI3dU-={??)#5CyCnrSU(ca-+h4kSw<3sS`r*t`SJ)A-6vN{-'
            ';Me05RA9>u;wE5>8p|)E#lU&I?%;+MQRiWe<N?CH*O4r<YsS)j|A^Hh3v_BYjLN|tFjHv_Z$D^}k()CxWGRJ?`c^p98%'
            'MRf^zrF_b+*2R6Qkw)3k}W##5H{SVB?w(KVHVtOV`!mr>q=|R9+$_(;-A;HWQPSH25cbhZgr$fTG!4blhT&2&KrV-'
            'UhlRfw+2Q4BkD@CZguwq1wO-ZVx38$1@|~Vi*pWABcliS0bF@&PMZ^X)4C{BzA(UA>$7h)|JTOtHWD?Z!-'
            '&ip8P^C3?<T@&B~x?aRM#>Xrl0z#TfR=jm|j!ht=MtbZ^--'
            'I?JC1L;nHVueuu*uEmo}{eI}j+zL<5{KDuh#n`HpPg6qJC|&1)J{D`yK3W@~$Cbt<$Z?GA)Nvqo6`maZh@K9Qp!QTR$p'
            '4zlJZGtg*NY4BUbGply7Cx4NQR;EpU*gd)mQ8?U(DIOU@JK1UchS00{A^NA7ad%sb^6Jh#Ovk)Ag6Ak9QL_OYUV%x2B?'
            '9%LD56>nc%?<YtI5rHOj|BG_)b34gyRr{|%X_47(B#0fiq<!centFwY+UHAZ|ndhQHf|$nVy-'
            'Q$2crn(wM1f#B2fUliQPVgFzpZ#gcS&Spw$*DG3ng?>nFC0(!}0bn6Y}@H1Y9m&50=t;?4ZS)h>w>hdRg$zhP6N%oz#I'
            'rg=au~*8pP{bKqd~64)qLL^L4<BNr>c_hD)Hb~pk>Ow#Co>D?L@KfA~R-'
            'z{+NUl1CFNI<xkD)8<3N}DFu!z!P8EY&!|dM;)NOR`g-spuO?j!J<3MopZu_)PYPFv-'
            'P^N3>FL5QdVKq4bh7yx07Uzvu{gbMp+5^lm4{ylY^Qo-Y2InFBi(+X2Sv;N1_7XyP181*~6D+kK-'
            'b`9q5Et$GN9D=Mku-'
            '%)naPA^a$(M9L!y(nX14ijVHxMlc1l)!RY!gC0ZFI<XGxZ+UvgCfB%qO`5(8r0f*g1=S;__^MM=4M9_b%;lW!@anBm>1'
            '3rYE%CcorK-'
            '&jbB4laBY7t2~;%!ULj3<E>w&6Rn>4)LpG`A6w}PwQCRg=2V}FCFgjQ<<or7ykPP|6=u?UTr}hWz(zH}mC>|s0Gg7ExS'
            'qiMWx(DCw%*ItHhZkSu!Mue&kk)(+*DdP6EK^bZn$e8K-'
            '5YUe?EzetdmZ*1Qp5!g6gd29V3qZR97b7)SaJl7rHb+Mz+1Le%MsKmm4yWoHS|cGGc-To#lXcYknu2&5!81UA9bCknf@'
            'HMTevq~#{!}sr3(>dFK8;e35UaCX?LSP{0w<dRYsmrLyvas8+Ai9vYZrodE&M+Iaqu+fvC@sW`^bdfyCBP8WbVP49er-'
            '%$;<GX}MR}uH;XS*Bv5PRN84yVHAD!JOQ`bh2V*T6wn?phrjw_oM6Sbv`efM;(z(!ur&qS9x-'
            'Nq@Iyw99|PZ>E}N~lA64X6qqNK=<bPg?V|w*)?xz4O?4)oe>>)g7r{kyP-'
            'e`Z&1?%EH;Q74~c&%WCCqh5d{PHfKmJ|jC{owZRcQAECpF|y!C3auAm~EC5guA()?GXEn=$zLls(<95S9XxCbg>)kko~'
            '{^)othZ!W#D*_$%}e7Ci(yTEPvzPu6pW-'
            'i3jPPd#V`tALr?X)3kC6Zb2=pkb~u@cgy~sk>nat$*j>+UzZGzjzr`7P!KVIkp;E^>JvVUyCvSVo-'
            'Fx2bkhY2pk%rCfe0RbI%)4p4`efTLT#MV1T?7bti&uWoV<wpnpQtseg45j%75!+?UBjM97t`)8Ruzw9^>XH^M-t-xrry'
            'Jq1?%cNi`m#bt*!;hEdPAkn)F4vAkRm$p5CZt;is=IvYX*L;BUT14Q+)O`9wTLrJ@uZ3N=d6>tyOhMDai&Vk-'
            '8@(qei+zVYK;VNATwMJV<#m_f#vTLIGjsygFbUYJA<W6ua;7N`?l@2SCVWeXpaS~-'
            'blYGYSnnSr=5Ll_^MZc%Bi(3}pBD;j-'
            'CxARripFey^l&nGHLT*twsjd3XI6{g+z`T{CpS<>^c@9J1vO3nJ^`e3z)`|C7}LhcAdH1(Jsq|arMC!#)jLC*ndlvZWA'
            ';?zw{$;DoY=3M}*;fl}UQGUYByczld(PTG3TL6}eX^fTr;w_>zWHMaCD0{8t0J<010K)L?o4eSDzjt3JJ?h><lNhpS9I'
            'Vf{UCP*3Y;Xj?1+MwT1(iT1|A!YB+pXi9H~@iWi;dj~Aed^}`Oh~6Ou@U-e1{pEKB94j|r()V`I?N%mDQ!ALK-'
            '8yKmiyh2=auBCC>tc<0CEI}S4UK)hrHuJ@847F1z!9q;*x@b!-nQ1r6~e-'
            '&HKxeYs=$|pH!)257b$Z$!^gXL$yb3G7<lv$Uj4TK_Ht*#7iA;Logn~SWi=$ZZIZ-%-'
            'N1Ge_Cem0xmcZXhNkO$BW?F;VMo+nn7CX)pMF?QJ6UJ3%&G^fy?iP3-'
            'A1kMa<+ED3dHEcaAavBX5KmoGjfIWNJTQe*1rRIhFYohfFkUu^(GQ`ZsS^U6F8)-'
            'fQCM%_)ex6fA}WCb9ELvExV0IOV*LW5j{}pRfCK#Z1#&RgreF?s9M{Fzo!%^SM(6W>T?eq{HhAHKpa$4vZ=_DWHi3$j<'
            '%x1SbL@g40^wV`eRqR?!PS5-'
            'VmrUaDF+*U_1kQ>y+_CMH5*YI*xCl21r*ZIH>316uzR4XN=%Ln=9FUYcIVs?g+Lm{GhYl9Gp}7z*;w$N;D;tEYk+$cVC'
            '0vb_dw+Px#<)-wp^0bA#RyE_Df`VVa$w3hOm`&|W+i-'
            'lcm0C!&KcXjlU0>yw~Qjtj;120&BL4OH$O#0lMO*eUTCGi!rDAhj2ccq*{QH*@3s&)hI5P=iK1o+SOxcgnuk3jcn@)A~'
            'EbWPX7s`bLkjujE(Y(_tREWDO7JfJ`I^N(AEY$Phl1oY44`7)|DzA{}||3jeJ+1{G2XaA8E6*<+eXem+ZOEM6avZLQ~t'
            'GoKDTTI~%Yyho5Lo0o0(sfDU2WuWBthah{WocgUyfa>Dkpp~kCGg^zFG&2U{^mssEWdL#_SY+GshgjCSjp>@-'
            '3C_<(Y1|SE>~*T9I~6C<ZT2~li*J)euCG*8AqQF|{lKqe7br~S!qb<6426Bmm<JDtGIvO_)CHWR>EQbn9QT&(<X2uG6&'
            '0DljS~B)!+;$W=C`rE*Ds<`G?l3Ql_Rprk{DKgANj8tL)fp!#A8SmOrN*VmA}X7piBlHTItHR-`NVE{1<bg-'
            't0q%*YaTWFok?KRYiq7U(9-'
            'pz*l&HtSNa!Gz0?R?4>pEBKkRaKgh<%{u>y5BMXT@i4&&f?I%>23C}YvU_w|2hutEn{$M%n>*%06)*eOk{BE?#%p@X9&'
            'tkuhCRkXDP_cs)x>r|0MPe+7cxq$h<qo*@Hw2$<ZonS%$245>EXX@mlR2rkX!4ic_&F;86egGA-KakFtqa1-'
            'qIqyX$O2rOFQG(n8r;ln#cf5oIQ=sh^o}M`$u}XyJ|qptMTNjGrjuk836MP}o$<o^{b-'
            'u)j;7)b$SP`Ktz4bVp3HfLkMA_2`w<G_<_F<>Sp)R*8iMg=V~~;6qT{9+8Wt~BF`qQ-!*#Js(Dh$9{$5-'
            '`3o13~EJ{Pr3NL7%N=7*+K8*e|1u@*q@&2hhaJ|JGwmw}zf-'
            ')4K*@lN$YqU|<kuhAbEQLX{rcu@UIZT(gf=Lt;++>&G)n(Eg?`RHO>D&*BY5q`fJcY5TU_SHdad#3qDu6e<*P=oEJdm*'
            '*ne`e`278kwG--aP%llPHw^R_At<nd%YfL<fQS{&<a~u!jgRYZFVA7F;N-'
            'M_*<*LU+{V`Cnp#)178ROtC4Qg>S8mEh!(SUmm*xXk`ZN4wa^8ktdgJ(FJ?M2U<d!yKBIm2n68#sOR!`EtgaG80VFwL!'
            'SVb>_V+ZN0S(bR_*=dVC&_BNb|I!Qd<odoa0ckz$XO>p?Phs|!N!8oripmOOk=<W6a-'
            '`#=Wet02lTHj4pm2g9ah6{9l<;7zmV`SvWKQ>)s3iT29;d%5Vq3b)?EiW^P?A~U4+4Y%Z?79!dW{=^@w+g7{*2ZR&k9b'
            'mP9;RsRhAmY?B&JQ6F4}E_wP~_A(b@`{kHbMld=V_-t46h?4798cfzf$tWUw`oMv7Kag^RTiQ`?NP$J}77ii=Y---K$_'
            'tJ1t9Uce~ggQ)`%(5U$VHmOI!7R9-'
            's7gB|v*wv(Y>s;V%s)YF17<BZQ%`Yd?q4ByDTkNzaOrP;Xn_xfkUbm3#?bHLDaX;Fb(?>YVhUjE~2gpw*Lv5Qk<JI(0('
            '0%)bI_(Tan?@H9s2jvKMOQSf2*dIoIpQ8y1rvYM$RXtz_SD@;Ozg0Q!ON|f-'
            ');+J^;;;ImrJB{5>UM9D;*D4q=GZu7+V(s+_U*jCcz1+$wb-KytUM+K@`;XZN$EKA9`}3A_+}k(vyB+c!ev3t}SmR#dk'
            'yTiRpTj(C#Hl0a`>iJPnvXj>DnN#%Q>v8kbrZV%XX^_ISep8DF{tmh#^uL8<0wiGeuRGaUk*Inbv44m)`p=v9j%wsGkm'
            'QdGDI9vK$Wh{P#;f3=(1S=xZWJwFIobp?GC3WzRbfk<LCZmkW%Z7+6Xb&m>~uOFk{iod{;JBh60pJk`2M|iNx6%BtU;`'
            'Pl#>>yq#cB9-QP`lj<H_Tjd)XD>j{nTkfBOe4!j*-'
            'AZCP?UMW7^>;_!u8fMRnY%)7(O;W>*eJf0=;7Y7^|)v7IHwn@N4#Gsw9?Q@D4y29%iBu+TFB<#;mCEQdw7k0H2kuZOIX'
            'pBf8<WHCJH5PkgYEqS-Q7)Dax5*=P6JbRZPKj-FS-'
            '+nj7)jC0#l@^W5Z|gv$PLHHy&*g{&w&3kAT+EmSpTMwxBMLY(=_1^R%L6;9Z2e;}GtHwHrk}timq}{Za2EgLQv(0A09v'
            '{FEwPDY;t#ze@aEzh%>IxI_EHseU5ya%rKN+n`cAmJz8e2+TL%%UpBTAG;TXPbg4S7mVb5!-'
            'hh5%07`cWGkMmaJ+Nf>RI4d3uO^eXuKW>gx;zG#!*pF)jhbTo!dbQ&|9$fXFj*9GND4%X6YjktzLDMKYvd#;pOx9qf+I'
            '_0F=Kvh6uAqzOmEu}IefU^pfV|QhVXC|ak|e)^+}}Gee#?%1uCN%S<Dx-BI-aV^PGb<2f#M1q9Mbh8$|p4Nc=GeIaN~v'
            'GZ_W)~o2#(*u^M)c7?G((*U<GiH;4DhM>x>J&3qNNk$Fe@9RAGN40BFI<Em>&4AUK`n12m0{~8ggwPj%ImBrrXS%hcB9'
            '2h5r<=BnwPqEnXG8}Ii!-'
            'v|zG}X8rHDB(;$uGaCzvo=|xndDm9B@Ok71z<;ybuqaD5L7Fjp$l3i0<DXv*%1xoGM>LgfpJvKkY%}EAIhbNiCdXECK8'
            '9*1=M0PrF`6L;Cbwrc;UpxqLW?;JWQ>Np1(Ye6xhkO}<<v!g)lj#wW3Mt0Sh4JtEB$vs`DniNhYb42y;f7{25D==?|uv'
            'L9^3=g-'
            'Ti;_w#2ifh5V^>ZO=ZW2b?%Aoo^J@~nL5w!M7ljDDyVDQXd(*AHMXSg*Hk1tw>Wy^ZN<H{!HCfRa?=Ys4XeOqXngE%=N'
            '5{tk8yN=sGZ@_xzI;fx3#=r_LY!}nO$GwZ0I&czK{OyKQ+Z3Vj-zP$(RKV)_Dy9reiyhK^0hTp1!nJyD<b7sCSt}Lr(7'
            'X(s=y*yCw|hd$=@%g0pGyUv=VJHKhoBLjg?ZI4P-'
            'J}o?9%@Z!nKh!RKyG~8tg}vkQ`k3_Zeu!YC=krENZ$)!^JNYO#5rVyyPZb8~uZ=?rw@U9DcyM`xq&l23Kohi1W5%xa@Z'
            'oac?ZePN%nI`TO;V%U;a(TL^@v4`Qp^Ht@IpiQhehITHc*$(+HLEbHh~;GAQIs}JXrrsu<O?A%>=(Az|XiiDXbEVI;)-'
            'P=XNHoiiqkiR6|{v=-6oCg2hxKB;{#K_U4SWJF1f<*ZyJ`|5eca}ahsD-'
            'faXl2q3q84B~Jwv>BRxs84^YL<j4R##n0h6Mqc==Kg-'
            'MhMr$mB)R<J)tnQO<w(@su}62y5Xfxp5*B-vwYsq54>6vAdcP-2T#ugH==592|{P=^Ie%ur-nW?=DCVNucX(KeWhxioR'
            'AH;0U(jXN~O;vwsX`+$l7ys(|36qYx(LiUGAYxb;gf{P?59=rX&7*|#@?v!N*#aJwSAtO7Unv$5GM9zH3j!3CWmAa|E@'
            'B(+y@KDJ7ddkM21A}&S#Ep&h|i!&rI<Tbf{ZV7Y44=eJrvW4W!9%LxgRpFqvIOqJjXAs~r2pt~cP|p4ZtoKcj=K7p^f6'
            '%}PnLDs(YYi;s^?}J}hhgED-'
            '}vO0Ax6eX;y!yWq~DRm6_*oPiRH{A_f)VTt&gOc^<bOeQ3%}Xy}hDL4Rd~_BGK@~Yh`tKvwJakTpFj_4oHIX+<4eNC<N'
            'WH?)2_!M^u+xLPKW^@TBSjx?}tag1jglxbA|Ho0h{hpA<4?ln#5k_F!D9D;^xG#HzDlc*j1J7KuH=jJzqh&@Moyg#dyV'
            'my`QvFOjofIB<B)FS<p=7;?FJNO$7^I_`7CfA&`N<WEh^%9g`4YflJDbON!Hf7pC`4uM7b9$+a;qW2@hJdtlpxuoi0we'
            'dU>G4ccirw%|*rVwdgxgV2fLfGyPX6U(eP2?lH;Hm{T?B;XEp!5df-'
            '0Vn$qpRT90VW3dc)`TpQ~2e#H)aHIp~uoPYMieOJ6;QLe5`ed?k71sX<&eB%9}~Zb3>G3$<q>3b12$ui>>Mpq5Hr?<hy'
            'E&O9s+O>$h;c{Y?z@o=u=cf-qR#E5KWlZph>GlDIh&yyK(D^xkoe#P(if@Mf&#yeN}~xWn4)qLn;UsK$@oHk+N2FZiR!'
            'cXN35J%hZO^}V-'
            'o4p5*Qg;|S}k^Qj&c3*mg9#XC3xm6Kv{p~{ggVu9)+<XivS7l+qT$I_ht&~_x5UiP*hwFy&$Q%tHSUHu4A<?0*b9V+Ta'
            'J~+!6qRs(XdLwjGXhp}JrwLl()Cvt#X|4n$)gqMS#=O2D<8sO+bZVyZ+7ruT?x(GeHC(kRl%p)yQm?zk;#(y3!izI<f6'
            'e%l9sjt-'
            '#m$6ACmb2To$|#cESzMx%j~u&k}sJoeMU_?_v#aTLHtex1r=#5ANI+fgRh%*&byhusS;#6t2!{Ol~pWdK8G)za_&HQGd'
            'GRt0!KyA=KdQL-cyzOE+=&;H7OIitcsS(8gt86?zil66N5#S_Zu7-VCwZ)yT-IbYd{|A5Qwsht|_k*zTvp2=j<V{`sl6'
            '<Jc?62_0a^URI^)7BBG|zbI!#;cVm=xQ;LW<fGWcYq(sK2&E>nc>UxlP#J2*`1M_QCAxzQZVbSv1R+j5$wZ&tU2r?SkG'
            'AS&k*^hrxGwB8tT&0qoka`b%4Y$lf8=kn>Kq$XMAl)STM>yd*hx<p`VjLRbMOvwrt?{*<P?`Dyi-'
            '3*!_=2^grzUS&G9b$Bf0?;Uh;7ke^CM9?iu>lMhfQ!l;Vj*HT5{VG8Fk+!RXX0r;GgL0Jo?x-'
            '!J08l7mk`L_!hPpM66`XM1IX)*E=E9F9fCJtV>=0ChY=>73&?L1u>&ZQvJR-'
            'qc6~VcsJ8O0kZ<{V+v&n>GWx_A?C;2qr2|nY1o86>_>tK@9fb-pUYgyO$4)rf)Rjb0At4&!r0%4-'
            '*CdJFst#0P}pLA)dQC#LnHf3wGA`z{)lo_%phd*h@Ua6rVXzm_14KYMIzKMe)w*YW7h#P38@wY-'
            'l{P0}hG};JX=jJTq8CV_RIXe&i)XXa8sL4qnQs$m8bpZW)1jZO`D^>9ezI(+yH)vYfnttH@m+i%xtwbmPz2HT#nbaZCS'
            '0l{J+xH*6iKq}0N6bUMC&VvcRBe0GLsBj_`(Veal2s(3IP1g$K|#`){8=EPF0ez^iOD-'
            'L0$oGT;@h|+KFb2zc<=5pTnvLJG>n`#zBQYSHYkZCf)yKgu+-'
            'Vg`#r*iP;tq{yE^MzESm&h3|!??_oR79vAkMwOMEBxM((1KnNuboG%1=?r5*c$YNor%!dKpM)n!k=-'
            '57^^74<WnnymFuJM4qZ;5e}=Std_YRX&B>OJ@+jGC3co(RWp}xWLBhfgB7cy9^{&2P*0}@DL<Z4YYJ9j<M3{NC;XK&>_'
            'Co3JO!ySt0FS%e7{7n}Qf<_L$Od=v<o5$`7rzg!mD1#^MjVV^mVur5neaHc6}i(@NJu#gWuA)B8_^y3Zq+31ZPLM?*R0'
            '80y&m>MyDrR)cEfqDyx9KV4u;ka7EYWm2dyXkz~s4s&BA{fMt8m8CUdsN)M9WeyhCEYxnui*8Jes17_~nJF+Q!FLskoR'
            'py^gV%rmlsOG|n%qxA&@X1^vOxn=)*eodB)$>9HC|AdP2|HJ-?(euG<+iGi&UFZi=B{$g1Tzyb4nj8M`eS?s)$H-'
            ';T1@q+RF#FrBpnljE+GBj-B9Ab1*au-}ts^q^5PB~O!K1xNxFKj62$ZYgRd0VdSqXIIU^Fy{=}_-'
            'zX>=4_0ZOge_~2h2+C0q0&yS4pV}KGyz12Y}{|-'
            'DMah*=+`aqvz8wmavil;_5!tdS@V%xcm;cR9OGe+7duf+#X&$hu=@WNid4DxCHJ$$Jz&XL&ejAkC+%HBA(QtA2<c(MF3'
            '6h-fcs}q6Pe*ZDyEsX+UYZusS`4c|P^1+ef`?$nlDIFSngu7&qpvxT}I;U_B{`n<@-'
            'Es?w@$3IE;Ohl)Wqu>%1m)5Nhf|4ymImy*!po?hWv)u&9uRwRkFNc_f?PNABSJ#+=!VUbcr{fD=5$Ho9%CN-eD)Q+pM8'
            'Y=yk$A6DMgd*@-09W+Q8(XGMy6A!FRTj9E%$t7&hfBI38KZ&ge>mlZDbKu|E?fmdfM51PwBZo?v~c9-'
            'lngNm;M9k?Y=t(EBnP@{_JX?2|}_vpXSIe)2FcW(nghZC`MCb`rg~cT-KRSr!b+qB@rrqtBH>3}nYc&zIM*mNx?akzB@'
            'p_b=E*D)5>1C3d@0DAv~N6NNk3w0<t)yuZ&`RrM5$FE~&g2C$z5E8_i)7uXv1!jSV}0X7`IO;Uw}Kx6k7+O+pEQ4Ewvm'
            'FwKt`LLR9Zc;=Wtu&l8&!Xp^meYn<KWY@vg4f?Z!5zLbgtJkY&B|oqy=Q_@x{d|k<XqrKuMoUHYYniv1~m0*u-'
            '8HeGzWY^Z}W06rz1oqZITwM@k1SY<A!tRVPu;v%J1%hI>$U%5aI##JSTC_#5v#^<1o5t5DYHWV{h7_N(<T;w8Jq1=#y5'
            's85~E8!}ehQt=DLl-wumjC}7OFGjQAP5a~2Ighsd3K(i|ZC4V}BxafKeUUL9@UoXX@ep^7HJ{t}^@IhpY!C*`$ZTorz1'
            'f63*aDEQ#Dt|+@PQ0f^hbmcNjcH)gGC_4Nw4t<|5Y>*EVEEiWg)3KY!t<U!<o@76IC6Ulv1<Jff>y`l$H6JK^xK!@REh'
            ')aWv9{)=e$Aj79Tu(8il_M7lLV91ITVXj=7U>AbVvwC~=+;(DcB&Tx>9@>}4e0`b11D5LByzL0fYKA}aqe(sBhL{bvpg'
            'berHWZDnloDy7XyBKY!$8NTmdMy^NSft1hx;oc5!5-'
            '*Ss1@FeGzdVa>W8BB1yj!8o*$V^y`H{d6)?iY#oKD4Ufcf#RAlD-SP5w0~?cfeC1v{biw-'
            '}l``eEdbv#`Ua0%Web)8C{7dDoU;O|3HCh_0YnsoMC==PV9n{HCv|1KioP0TcCWL0Yu|iVeMRJohoKI<yd!RHMnv5kWj'
            ')=?pRIf6zbt2(*=k;GZ`RP{f~KCNsPdWjDP8!{=_08F&=~jyf@(7}}x1VJEy883&RAk|eYykgod?NOp(UWBS}syz*9%T'
            '83VRFGoGlJU~=Kbt)V!yPUyjegfj*a1zFUj_$pg508j08pdZsT&M|qUVk>b+_Mzc>G<Q&B^ExgGJy24yBbUOtb%OW9N4'
            'yc5&n7Bj2eEIz-P-Q;*@Cslks*?RuIQt&g%v0l0{TIzMXu{i-'
            'Mm9!lcA;wqMJ(5@nuB_RWY86yF?!n;oCxi{hOiAb1`t!5h9vo<OG|EqvWph_E~rjvw`c%gnG@FIkLbO+x51*-'
            'j*8weGF$aU9zkukr28F?7_A2bJV<_)y`2m-'
            'zUwyqu5D8_uH{ACh2f#tiGeo`MF!2rTzG1gGs>Fz?4vJksw#g^3gd+&P2iX6A7m-'
            'wD9^j$z_B8381}5?PtaxMT7zS>*4B;(gcg!*h4yBJG2!XNpiGQ4$(ghv9^N0&erogC~AZA*+8Jaz4sp_PcN@D18|gY;D'
            'G<#vII$H~>FCji4tzMI;z0n9Q3<-tA6+Pkh}Z``tN)n9nCFzGMtEIwkPNp&%mjGzV{)dO-Afaa1kHBz`}?;QnjzB;#^E'
            'y<6{3mwgCeEK2)CFWS^$ylf)Qq*0U{aHMYb*05j80q4}MqJGhG(C3y9F8t9r););pt|p+FrVC<4EzT@1K#5U4OfchP-N'
            '_Hf)MaWKV>e>ZzUU(R^N)P8!1xkmpQIqPnvGWTq={GdJGNW2JIl>;4z`^(#o&I#mgwccm0Zf`y*WwvG=G3poC7v!MxnQ'
            'G8MbK!!JEn(a9WrLzI=|r#1<wlNK%J`ALDR^!Vc)`IYz#CzQ%8@3vuCTIaqno19^l*8O2I9;8|*eV7wVWWX6C+RX8;FC'
            'F2}!Bl1n-FfBFNi4(nrz|~NUf0LA9ifIoS>H8qL={bt5-AK<Z{zYSs--4m{4X{;Ck1=u0j7ricWZt<-JUw&-xHes*j-'
            'SktS6i5{Z13ZN@1E3C<vBixeudNX_)zKZFtyz0M^<eR#-'
            '68H5UHI=nUiNgN+$$&ewbue%XG7wN4L@!8u2(3;t7F5IbhES!+Vbr&VRoIBIiEgE^&l&7b8GUJp<Psn2RAh;y`1O9JQ>'
            'yivDR2$f()AGGiYO71TRO7V3E8hmj)E_$7*_zj=c0>QB(XGY#8+xqwTFI1%sn!CFN=Ow)3PZUs-'
            '8WII43U#!H=>}+tHKx!hKiCy})Q8!8g!cG-oog6<nw3OiK@kFST62v`RWms!mjv=xi=)42FU~=^fbka<M=Y!iIFIyP<P'
            'uJn8p)4BlX9o6V_=D2LHlW4x@E)fTF0q{G=Mxvug2}{<6X~ey?t<?R?<6a~O*5vF&tpZP9p215ih1u5Sidr%a%=%K9nF'
            'H_K~G4z7zv6y0>OTK87?i@iQhbrp|^1wEZsa!SyxwMe(mggK77LN|8fitde=hll?!mz(~>6I&aeljynw4f4=${zh0d8('
            'D9S%Ywy__ggois^Ton!iUMaYuE&&@4_%rTXti)X*$#mO`8&tXKGX1cTNz0hGiT7kJWX-'
            'Z=_7{q}c6L~h`yA)?uZPW6SBYWNLo8AbM8`#8W!vTk0VJ=*7w${olKxAW{x1zC?0*ow%Pm;@y$)sfw_}@B9P0e(q|wS{'
            '=<c%&^sfsMlZ+O;P?n9d{f$u67K$dDuTsb5*>so>G(O5ll8vTpd~SOMV|f?j{qNDp8|j6MMR=jy|1>ab55cfO2ADt1!&'
            'bL780cWk7|d70Kbjsy{Ln6H<hmLK?zxd~!lLjZ$Prw7{b2r*2sHa>#)wGXOx1tqp_8;Tb?3PPQA^T5!r&Fi)=yCJJtkm'
            'zJsh^0wxVRteFoogKL}TMLxaivp!dN9{_4cg1ry%Ly_gA<XCG`hqz_+hS5v+@Q^1O_fS$QIu+=%5CT!2Ae`oKV76~Tq;'
            'n8R)wu~C8K0vdy5#;w5#X#v0Fz8MoDwnUr@k^FC6p;d{9xhbLOOp&rK0qa*8(?;52N<pal>fm5mzf8UA-Dme%Gh8KSxC'
            '7wl`&;k8fLTSKugO9YI}4u?6dR6Z3biXYp*A5Gv-'
            '3w5*>y>fGMHAOcZ>Tf;=M0Si+IOmBR<{Yuhne^raNjey+oqR1c7tjEB7xh*ZW#T<8~tkIlScVAjjIjvL~=M?+-'
            'WStq!KJdhc-'
            '4Q{;>L(d;quwys{fBX!@eO#KbFP%g76IX1%vjBsGondi)Fg9Je2;o~lu$Rl8q3YT(xX*JQo>vG5x%$0~1;UTPWgj1NP^'
            '=GhYjx<$*Uuqr**n@CWQb11yi{Cklw?f{;i>9rbUA0owybjnF7^)6e>93^DQ*r@wk%j-'
            'CdW)U5=uV#M1z`h7`6X52CW7qa4~Z(_%L0mlU)(R?3feo*k6U=>D}a6C_kpI4+ixMFQ}cLsz#ruKK&{3l~^8{WL%HEjU'
            'SiVpm%{69UpnjFx#?(>8sTWoBzv&PfRJes=k!zmJtij7A4}vnND(OS|5Z0xtYIh=QAT}kKv|%eh$uXQ1ZP${<4<GQJwc'
            '_rubFE=uJP|f80xhBgSZ^3=^tix3Gmy-'
            'G!{7)u^|v7Moj_;a+h;W>r!=gg?EE!=aL7^pyk3+6m&d*Sxem>OCZSWZ|<D{BS#S8245v;zWHKp3O|9r?S7&B|S^&zm_'
            '3Xf*zu*ycmxzs-?k7gVgY~2vK3OS=x0c@bgN45Re_gLsrM|m|H4xAF*J(+I5z0I-'
            'Q7$i?g9UXb(<(b^?Kja?rf|7h12DqLI^a+*<DgJYH3JF`Ysz(FU;k#IEPMj4qc_AkKLs+xC+jDLKyvy|dr&?m##^sgI('
            '?%6=em@F|2nVbG<%K}2zh25ykig(aNxz_wd~wo)~?{YUC-'
            '&bvzz9=Q<(@5iA0>M?XrL_+f6P9k_|IjS~<gE)Oax*9KlKuZd6%U{L#?`d#*(;(Ccs3T)-'
            '3NDk+rfEtJ&>wC~sw}hU#&{DHbI;#yX`n*Nns(utxv^-hT!PgzuVB90YmKW`itw-'
            'V7!2iRqj}L<B5S`89S($|y<#5vMR+snEhNy-!UVm`w!+4t)4=-s78>51hl$e#8s9he!)RP7&g1mqSlCMR{bGPWXJo-Pa'
            'v#1uK0%GT|Ila~Dcp0Z6t0f?Gi<E4(vH{+c(W)R#sxenzuylGYRbVC9g%ovVt|fGE`*);ZE@i&KN#90o{@ium&?W}<3k'
            '%|pXh=i=PvadtR7mK7fRG#t*3s1Y}i!OK;~>rBjQg^;<27QeCr|rXRk-'
            '1&tZh5h$IvW?I?+y^_I*&KQ!q{2DQ<jw8UyTsJ}i6>Dk=0r1}mRE@&kRx*o9ktUK%wTn&n?7om1U68_dCk<e=mWW~Ert'
            'YZ2CA4+hubX?zI?95ZN@{h!e`R~Y7)DklFKn3Q?bio^q0NB#=mTn3wfz0d<)XJWQ^z+Rc|Bmg#bv=H#2jfA}*$;d-'
            '=WVw!a)GIwWtciI52LMEm~~JQU8K9|uf*ewg>Cg@Wy@RiY_-'
            'B0>p3V8@Q}VaR0t6)4g76Y40rb4qpIIGlJ*ZV47KkoFljo7x^yiEr|lu|((@sjE{emM4o`5s=MAD~x@fbgItCSn!NYt{'
            '=)0GMuMhts>9G+YX&Z%9?+EN$w;zsA8DY!z0emqtL#>ZrBCbNZROp32FjsM-1<z}Epb-VxBYjYj>y4-'
            '1dtk!bUEnt%2AWfk@u^HKL}`WKNkd)+TFcUAiCmnN<E3J;sqjc+IkFWEsq%FX8h)n*;w56?)BXh<tElb7)v^i$?(0K_-'
            'C|I?8%)2B^JCpIcgWsm3UBn*!G$-?RDgF8EGyH(U7H_*>iM_Sa{4a#Zk?pPQCv*kUo9kf-D(&r=%O1y6^}kXfVJ-SkoN'
            'sLCN52&#Y3|2*W((l%Rhsk!_Hu%2V!;D9UPmZhYM#cAab1^+xGeaXo+45?rXU@gSjv1jB*Hd{p$hNxgAv6M+Myq${BOt'
            '41u1uG+K)72k-G*?A#IzKd1(^oo?4SlDijd^WC77!+}3zs<V0?h%u)dX!Yl3#CxI;e;RaS5|0mftd&HL-B-'
            'w(>ji%mI`OWBJj!uoU}sGVt{x1>8-'
            ';r0+}tDJA#VcWALNM3?t38CmP=(57f~O1UcAlQMNK~3L55^7l`1lVJKNLQUylWmuzElE>ARoNB2t0U?N7iVCzEl<;y>J'
            '_?!hL>et@seLGU^@jBd@H)$I3V=X4}nEZmn!d=Dd`ucXjq$PadxZp3-'
            '#l~D0z6>cerqH}LH5*C*W7&SW4#lK4!bG~^)tW7ui{ECKqvd00=i=f|A5!`<EHkAEvfp~>@xV5nfeBDCfv-o$AI_-oj1'
            'Hy3f!F~+x4nxgiehAFl!YG!%PhYOtkN0*sgN#@-'
            'xmKQq>;KKC!7sXT>RBXV&2M0@P3i#ifKb*!?!Ck`TM{Kd5ZvEvjc(6a#G_Ia#}Y)LmM@Un8eSkd%F6+cmV)Nz!{9#f4u'
            'oTF64i274UH3lY{|<UU?^@Qn=S@{$)+f{b3lhi_tw%To<{iT*hl?TlCWPu6B6hogWo!gu1#G4bb~H+RNas58@V}`gP)?'
            'fg$u2Wdri;zC1GveW$c-'
            'JK<#$jgOmkA%*Oh8OjgAcI2`Q{9nWgmW}}<Sw903w?4GqS*sh6s?eY*EeU?3brU^Hmy@hJ~O>mc4H{rk)a97}miP`zrj'
            '4DErby5(2y#U%5<>TA7TpX(5qWz-&z={pVXnAciFdc(GZtla@OR>~BiiMpdkVJ=?!IvEkc-'
            'i?Ak&xBJZ#&{KLcR)HtqJf2)xr6aSX3Gc#MYkEXp(jeIK&U$S-'
            'u1nKYrNo@gjA1u)vU2GpM*$8mj}|f{g7H*?e;gzLhh8lKgAfYH$n|QezBIaAwcrk3q-'
            'Jr*LxA0o(=8U|63B?pz&&N}Hz1^F>OqTgwFwXC;<ZRs5xQ)o<dJJAsh#U_JZVXahK{m%x@?hhRs0FK&llWWzs2(EXze5'
            '|6&p{%KuQeUOZqd;tHQV`G=YYdR3Gg@R*wq%`s!#P2A_dqz2I+d1{n?*XvD-'
            'V0MrHsL)Rc^s(WBk}4^DAefzf4O_9bIh!Fg*IaM>)+&-'
            '&m74A?LmXbR3P;TB@ycaN#gcWNMHU6V^`jU$>1(%EiHi|$r$?iO&T)3+GE#xP52$A4|IVV*z^rE1U9H+RZ=L_C~(lDSO'
            '8alO=B0mFUA1#42(bH0Ljk|!L`IH7|ywbV|?ByG8Rm0^e)4S`ffUYlL4;B(=nMx5)WzL0DJG%;Jk|k)?Wb4Zx*XJc_d-'
            'bAAZ8u=toWyVC?aT#KUI0;GBRSHTxt8GX1)^Nv;6XJnT`Vy9<~f5@2yp0sNYC4~_c5;Dk>qoc3x2XD%_^T~vcAn#1gpx'
            '}{K66$3l!4`M~`5aY(91gLb^hm);a=<v5>@{J~<N4eiDkJv!(b6)0KW(4J#JD;K3Kg+Ym-'
            '@tRwo}sC!LytWwg4&JSs8DSimY#BlKWCSd6s=Hpjr|qs`(iViY>5E>@!Mz^)=OV3e2B?|Z;3@6hj<o-'
            'AouE0Tw@l4mHG0x@Jl=WW}Hi7&xe8hMPIZ}(jY_JZuoG%6BhZ)<K0dcs5UWSo>&B|Ul|De``z)(f;g(1pMVVwuB5EC60'
            'WY7#4|HVkRTh*t_MkAySu}qEq4(7mqOOHVBpHSir>B^!I!PGTyQ!T9R*A<A|Qyd`%4u2-'
            'PIU;;+>A^zX=MQcr`oIw~?{T1XrKdh1}0A_@u~?y`?4__Pm}XE4~!s>qa${*((Zo_vI`%bz{Q9ar&TREjXyC!>!&taz('
            '@!FU1P8UuAq^-P#@iACx|k7FS)ASm26XnXl=}3rlIstx{Z?_X-71Ghj^q3XI<O#{EkaF=Mq9d~-'
            'l()Ki1d)0$;6p<D4%dIJ3W$bhZ>m*7ON2~Ff%3=Q@5uyRuX+ZUP0EwBo~mqiqV5=mb~6hz!(AmiX|Osf>aBByDZQ+FE1'
            '=BLi;NiF6m&B2auiFnGn5q^(+1+Vf^80J^Tn$3j}-'
            '`z=19j$`yW+AwFiUqf3uN!RVG7R!R(fJGH@QKkUG#mVfTi;c|^UrQ{gQPBMJyn5Ya~{zDF6gt~7i<9|(>?6u`H>(Yua7'
            '_ZSP*-'
            '!2X5|v2PN$U?yFb9b=5wGshkWBuRDge*B;@K>_zmsT{~zL717E7Va(vA*coeycX#q|Tpt)xeVbI`Rb&lUJBFdD`32aoD'
            '<?}nv#73}JhcAwgbpnylK6XqHvMBm%?UePze)^!HC|#%DTm})b)tc05FY#94KEW0U|7un8h+Y>;<fFlK&MbYPz-'
            '!iu9Cy$HptsAzzI1pf!X_w=(=ShvmPn}#mj6^>S{2FAJ!!A%Ur<h)*Jfa`ZtndlMg;I_fhF?AxJCbAWt_xSh+n0<GX@j'
            'a%3Bu53hmqu}plab{#?#M)7g@C%D+kO*THb#x6JeO^#G0<41#6^n&9pSnJh~ejOYLE7GIM-'
            'E7w9*CXuOUyo2NryUP|Pr<o)26$><C-z==!Pxp-'
            '3FGR9+15#!?4I{sXjH0!j$^mTy}w&wgRTHxAVHMsvdG?`2{1R7hNm0l@m^jYJL*Ih{z`YHLg~`r*_J`4lL~S3-'
            'i5H`^<75&!7<eFkHNSrw_%RDC7AAN!$fsMy8f~xMih)Pa+sws+FXGqdu@QjEGNkq@1U|sKJ+WNfq+RoS{%#A4=<GAr7?'
            'lA<q@bY6bfU?8szoXm-'
            'J8}1$)UgAXxa8C?DO3#W%Rf*ct`M_VS_g9fHvK<{Iev7!4mg{}6d~2F5r<(nF`lsOt4!^!fWe<mmZ$=(Gt1zlI!)SQ`P'
            '8Ldn?q(_6!H+6E0vf?$ES3wXxQuuZD-z@^?EyB%DK`;C<#5gg43h^i&d`F)t`RR$l9DPs88V^q*sgfYY6$a~HUMmL;-'
            'sTZzL-YCe_)0oSAKT^gx^t~81(hy=3y#%&tJE7RVdNA){(T9PWI9d@-'
            'u6yMXhew4N|Hp^D_lh_8ycdBUj}`D@OD*g+{K05cmc|o*=Mk0Z`Ea$x2@aodhNGMKQPN@vjxU{r_WDP_|Iq`SbBEZ2%r'
            'tz{Vn%~xGJ*Xf3N8I#LqYp%Tsabi5$m1VjO6XG^X_4A<r%_$u3T7Rf0thOIRe(}rGZ8oVYq<^F7}qh@VY7TO~49on{wE'
            'a&X!=&{0Hn;XP`-852mR(z}N2rD4aUTmebUP_Buh*dqafn*{4S-'
            '2&0uw9PXXxOD6CGX8IT~o*WbdSDiS>&$<9PN1~_(pD!b`Q3CY~J`&LiFTCeb20=O}=~kmQ`o(KIs9cx9+iE&6>bZa}-'
            '*g0&QZ``U)+lVzSPGMk{t$G!41Qlo1gf?aE*+D>11E04g2*a%=->?L)&GF<y-DzHvXlH-lu5bAUc<pvp={gFC!zS(Q~b'
            'W*5UQ?ZK!QgGc9t7qKwk<t_*}(E9zFaj+ebv>_2Kx}eK3>kivgc%VWi_e+~T}L$-'
            'FEo^JhD`RrHd^zHFr(y`N~6V=xVO=EWM@H2m6{PM39t<0_3H&?=3_=?BFq`L3TV&YaCprx)PY2L+&f;x#<2UI2j?T{L<'
            'nN2!_SJG!;!7hSVI6HfE*!^d11kXUpP=Dmr*_(TO1v5dr-xm&P*SOBZew_-)<efYJ*6q`qdAfewHw6;~^d-'
            'Y+O=WRnfQmg5gHNI%A@{95E<Q&NP*+5uJMc}xnF$_<=ARD=>@OWtj)!&;7^?`1T?L~>i-'
            '*Gdj{uhol!wYawUlY=dba9?k7RGPQf{eUaTxxU%uf@2LcJDsczEwU*ez{VcIyRlS;KQg@z0TH>%%bhKfgrwr5Snb=%Kn'
            'G7H;v{h?Ebi=P$3zjL@7}yNg{sN_9LaFB1&ayE|nn~B$?-'
            'Vp2y7dOgPszCX`g7D50d0G%Agc`~SRrUOa2v>wb6EI%}PEeb2Se{(in|oonr<GJ_ymQ$lF@N+`+k#@qX&;dej)d=+X(q'
            '2<bqmFpC)Qk+YcY?6V`?j>NCb{aKyXJfNPD2f(*VeOvTb30y3+ztUC5%CN;it<_GvU1c+I*L~5ccWY+Cs{1A7vJ)v;qZ'
            '$HYUoh~Ykq8k4^;`EHe!TBj@LmjayORmj(||^KX~bI9GG}j;wiuLFl;YG`c0~c(|Ru4DI$mSdgkCVr!d^Avl%-'
            'MWD(x&hoI+SFdj&&z<IJMVE0j<?($m);T%Gcui^*)WPJ$JycfbQ1(J8xEllmEQ+TUsfc$YBM4ls-jP~ylP*UgDycc^67'
            'A{hPS)A64UE3B$`}{f9#dlTkSa%Q?)sJDwo9(dhz$X0l*MolSc|&#y_rcGfRan?2N#+|yV6e+|T-'
            'O%@ibv)`oZk+dqdG*3t!;2PQ3XfYAs~B{56X=ragO)`{FoJltA`uO#yib$SUMlu?z|y*HI|8s>Sp;kB~#g>k`Upi18<L'
            'xlNGO2z`|-56-ZLX_lc#n)1QLqeLdjw&jV|#c9eSGkJY-'
            'u(6Qqlp7PoPRU%qQM4C|i?R@N+GzGgFU9w8M1%1<e(7HJYw+b1+>20F$PG$?PInAO+KnC8Zv9LyDAH4J0#u$wGzys-'
            'Q{Md5{9~ra}>sdF5Zom!PG?Rz2e;z`$giRn|y8(Fn5+Jkq0lnWDiZn_FzP%7YS*|ePUbUAwo0p36MRjn~rVJ!I2hl4}5'
            '@&C^L4NJ_0l&B~kll9@whCwwrK|ht=>w-RNNjff%Uh{<<B~D-'
            '^Y)Q<I?ZH)cQ(eXiiPJx#V8XPirh_?asEqVV6U>GMTxECcuE{ud&i5sNbN+0^MlYW-'
            'v%dUa^%U{e7xts5)F^MCl9ph$zJb5Tpr+t7Mo_CA2N^H4I1EF^+wc=&xVg$X*dxb0s|(~G@{E_BQRhlgY109y3Z28UZ;'
            'G}`pHk??2^!~*$XPGB9XNxj5v38Vt-'
            'T<T~g@}5#bc%E+pae@*t?6GeHh`m*KC$OW?KE3tihw!Q|%^DBtz~Q{0*;U3nKgcuKHUqZN1eS5u2f9ZdfoO$u+9)3^J>'
            '=<5<KyeWB?^o*=Wry)HE-1`+)wQFLu;c7T!q=D_dsu-BQ0ylkghiN7ivh;IFi~A#bi`yEH+LpkN$&;A9XMtvoStnNR>4'
            'lPglSI;<i<sG#!Ut_HR@!Ml47i{V)jj)hAMavNS#${n8h^2D&uzwxYfJFK&p`ToG#LZG3X|_u2nPjg!1>SvMkS{L|9Tj'
            '}+^bJ%Rpm|m_+FyHH0?4x+xV1q`kOd;ea02%w!MN^t1p90t~pLv%ts%)!)UIX%*q{9MSII&n3b4;AF@<Y?fEF^`6?ox='
            '{oiBG$UepWda7d!!XF+6US^qK*`Amo>dpZsnwe3DU`u%+hKv9?0?aFk##6kTTd!eL{QWx2og%aK=0a896xaZow$Ax6T4'
            '(g{F4eci;|&JwU(k}Gbl|uLq1mu++#hW>LdvcI;z3AXDImWkHbILiqY6&74oXp;kC9<m>VpC;avexS8xEYjV985u@_W*'
            'FqH(#ULiLu-oTBk$3gxHC#<ung4)Dq5YK-G%)~{|-'
            '0eCj#csqEr;MN@!49;0va!x%X1%rL@L^pQ<~8<WQ*kW49kK^|rb2K(S1z#`e~tT}?8Ly6Kj}DkEcxmAgZ`6`qEe@m@KR'
            '|xG+bjejQv8eQ_Tg=toTCLKd}a39Y>U$HA(&b&*Et#Bj(5N1rTz(gFG-Vf~PvGU?r;npRX9eV$)yLxkUi_Yuuo1!Doif'
            '&V+d5&nz(>ZCdyqapU}Ls5Rpi(c%`=Zk`6|J*0+D_vqoEbUJ+c`iq)>L(pgrq%LLL=#=r2C0VXbQwo=W_1-'
            '$XP%j7r_rh@Bw%PbsubQ~?RzV(rBl)M8M-#Lga6mYP89q`(+kXFMvPzEQ7l$xdF1H7!8+uSKB^1Uc6X0<qC!QSN0-'
            '<LL$cG{}%ZfA;1GXpXhE+4Kgd1Sx6bF4Zv=WM<r}5LlIBHcS3^6i$p!3EQ;Syy~=~q_0L~Rt|TsDZ{=84(y%ON4+7YVA'
            '5heKI=;d!G!PUQR}3aRE~qAe1RH;s`~@6vFw`fIYzaTZ#Xy2I1=8}JLLpu?tSnEgH+j(V2hlTKdn+3SlTawA}}u@#)Wx'
            '5LvrFG$i7f*DmY$hkM0n2g8b!R1@9wqqL@Ti>qN|M!6`Q80(xzs9&f*$}_?xWPQ7AY`SzhtYFMtVy}Ma6~s2c-|ex-'
            ';q%OR=n`z?ISpS-<=ex9l<%A&G;w8A6K_?LBMYToH*A{l;`(i>TP$pD6*Pte7hPTOb*_^kHMh$ROCKbj(M`-'
            '=u=k!am(8=^}|8j-I<Ot%?hM7%m&R>uA}VtDsbe!i$Qt>5;#-'
            'fN%%SHFt!=^2eQbHPYYo|rUzYqN{;Rs&IaNsN1}5d!OH6*V0C$n5p?#%(KjtX)N*mRbUjKRnFDerLUe1%Jow#Pfx^GM@'
            'ln!GqRjCCT@5Nws&Jmh?EG3(ennVcU3&3pi69M&c|gA+2Uy*##MO^@Vay^1TWmzhnJEc!vbr0WTo6Fr_AId2VGWnJmXp'
            'gv(ZE?NfoAfaxNmDat*Zf;tTYF{hz&Hgy%@_v^5E-'
            'z9}P{3Mi`qLS8vnngpE?iFyeDPZJ8_}d4?B2#jXjIr`}TI&)Tf4FL7iZ-xaho3uE>7is7`-'
            'G7xyT4>nG`B043!_{AjzRL4!=M1C}t=NW?VfqcAqZ3S5^&<5vMHA9D90%OD@0{g2>!MFbaBqWZ3J%1ixlmflIB!OActB'
            '9M&uA^P}G>qyiVf=6!bWLjD(i3G6WWJTW5xI*`=P$(g32!_!Ck5r6hmfK3a~jkg%E_g7erWf*92Yye0KZxm?hvcRzCGE'
            'H<iv?^vlPm#H(=@)Tl{;X0ncU+uuN{n;Lyyq)aLN|0=I3LxUT{{lOy2o^978=79W(GU5X_-'
            'H$XAqDo%}@fTZo4;k@>9vgA<%_VL~)<&J?Qjo%X-'
            'R(j+9uxOb3;~9PC_?uiFa{#n^z?|mDMyu*Fc(EXuF8Y&+kN1UR{RJ87W^@JeN`&a>&v4*f&W9@fM#vXw3ZYjxAba)`G_'
            '+EKv72h(cX=am(TZigIvs?Gva=f=`rjnV2dh}(-'
            'u>ixyc3Mo*u%$V2k6exU8tPi3M+*JG4x^{$(++d1XnE4)F8H0H#HW5^cyg2{t6JZYyzb(IaGA`FsyGMM9qdQP_+|6H^X'
            'cw5Kh8JRn`!oys17iZz<(rFM>9XIyiIh5!N4m!!k13Nnm6?)p+5G=cIbkYEdP%eeoN`)-'
            'O~?r~_%40Qk{+5`%8{Vfq&Xdi#*OMsP+W`i?)r4QbNw<uR{DjDiiGbMppIkpO(<6^d6kM_`@0AL?^PVX}A?!?!7eE|`i'
            'yzh@q#w5kBd?)IVapbYwr`@p@`bCA`#1hudIV-AWrqP(mv^are9t=wLU_D>vP)wg@BlU<|GBDE6+r{3d=y?wOSD;HYT9'
            'pKdyMcDG~7rAu69S@((BHq&RkoqkeVqSC*<@kN%qS7P$xJLmCR*EB+-vH$dtpNV1E0CLA4Z4+1;FdFp3l|-'
            'L$fxG?F3*P<u53jHTamM@1yWwU0GCD%>@McV5}!hxb48dOUw9LTdE3$HK`Hbp$Ka_urN}uI31>>eu;OqPId5~5WNN0uJ'
            ');m%lUR+JH3R6+%L}<Hj-qI#AKvHn#m24QK~rR=Uah%dJS_^9bW9+2x)QI*{Kk_H0r&Y!VS?Iu(2Bi|>tzvLRBFMsP7|'
            '&S1`rYTTlm}356FugT)q50Hg}z8`3h}<C3fMgzC3fvv=EFfF^A2sJ8=JWH<QEhjefqpfsD3J(ydeTz}`-'
            'pb*!HYw{eKf)bTwsuq6%b8lpg`f*YdrCUC?*o^D*(feMDY@aM`}99Y?mx3L~uFR_5*jR%ob4aS?!9~qm#{cvc$F1j+;U'
            '|{42J#@N=Y8yU)#xH~P$d~}@>o!OD==z6xTOFsa3iBGU%8?{53&-%2v9Rat5n4NDfoW@GS$5YFp+(4toNR8x>y!-'
            'z&$H;i*1dEy_Miw?3y87#vBiHU3JLFKmOmY43CEWq*WEtKN`6T6#JaJfu>`tB9xxeyyn(-'
            'cl*wh;;J1_n2)*^2l_ud$VxLB0Lzx+gk-dVJdB<>{jW$fo^w!l$p}0&Z1zIZQKuOXI_-'
            'j0I{cV5dW64g~vivf>eb7g)Ub;)Rt2M(yt)-'
            '|hZH6Had+Azs7v@>*!~T^|A+n(bM|5Yw_UopsH$`({u}LcQ<Zr<t@eFW#E(6{Xm*5nKK17*R;{jD)v=Lqkw$EiS<eD;w'
            'yk|2$FU&DqbOm}O2EmOPpIAEKj?dhJu<ma+Gd+i3NuMcb2}GlOX&#jJugAW3!BEG`4<=STxb?jlM7M_F&}0+Sls-'
            'Ycm)-_}>lvtwrEqd*B6vm2hIzOj#*aS46y<2}ah``Irv`~)-w;%Pi9>_FbI=pL3mHXj7--'
            'rJ>1(^dp>Zj*XK4kwBjkvN6I%H3ggIon-DZC9MWJY8A{O#>lGIuCP!w*C_v5=5$HfQewZlH7VrwYmiDp7hQVrhPkcB_f'
            'OIblL_tI^3E0}y45pbF>PM-'
            '3!sDqOi80U47Gj{%DuG>X2>0Xa@D#~Q^!Vl_sJ{x&Dq~YuuZ`xLR2NzA&B1>m46^hS=ybb%oTfYUQ+N>Zh`5>WTNmz2-'
            'jFR}Z;4MAFw9O^;OFx#8-'
            'fIOY@u&>nHlD=43)d6z+5%#kK0&qmnsC%j8227MKx<_XBr+yx)vqO>A;u6#oeI{hfMvh~f%L_$c;ZbG;Az=P7||@m3x|'
            'hD$F?AH(@hO?L${-'
            '0njp#b>cV5%;b6NdA407z!BPc(`oVuWBriw0{TLr`Ox=g=&HL#)w<5CUNV0lYU^sr^`Hgyco$%pRGJO8&i|;P`(FgZzK'
            ';&*L+zs-'
            'g+c<TBcUC@qetjJl^BLoSeklFfmyLtf3V3j{2Q7FNg7mZur0%}~?;3NU$^I=oi0*|?Pt%cV2D92X=YeFzYuY9ijYk*kr'
            '5=tmHCi3O9DZ;YZ-f<Li<Bx6b1{VARiQ+6l?pA(%%bBPtYEY^2Tlhsf^!@XVdnsw>eTT-'
            '$GffQT2#b3+DM^hpcytE^2e?EXG!p!V$6%w0siC!yeFVUe&^Z1vQr7*SYLrn&jjHs(+D!Mw;^u7C97m%FMVxr2djA;aZ'
            'K?ZV;8XjR%^#VX`c@KICB8<e&n(eqf*HHt`z9cwuN<}d}R51Eym3-'
            'jFr>mk1|&Sf%?0{Ke&%vCe7$_$O=yFzrzeSI+6ulOL0d~3Kibd17%mNNq!||xbkeN!1@~OzqSGn-pQoCCSstc84Q8DK0'
            '<tZ2pl-$0-m<laqUDN)e-$p{MYCF-'
            '~0v<v*OMF&G~~3|AX@fi^j|9RXnY5&FcmzDk~=|hc4q~k?ojjT8vvg^3dzzEX_XINPI8&4tKEMlHq^DXlgU7Ve(i$OCo'
            '+FaGmWSq1KDxj^7XJD!&!&_66b8#mnT|tb^EjdWM%P)8S`8KWj%}F+|IWA{YN6G_ZMy{<W9rnoq0$Z+=7PshB<g|MMIE'
            'zxtK`2l}<ny0AHyxUsF4d$Q{<UuC<qJ=j~)J=i0^+}Mwcz1X*YUSX%_y0Q<sI<qgwy0i5jy0AMJyRbJN^<cm6@nBco_h'
            'i@Qxv_1pday4Bda}>$_F?D9yRai1JlPi{ud+LLxU<*Dd9dqxT-nRqz1dC$&g=>)ceeD9J3Cd}o&6vE=jI;p`ET@Z{2%D'
            'wtsq6GGC9f4KN`gHM+qx4bSE8owSn%RFH5<WE0A2*dX2sjD~&B9S?c1>S}cR+70k4G5FJrcpot-'
            'E#71j`Ol`EGbCzEqH#SP4&RYc>-'
            'krplhqp3nhmDX!wt$smFG+c(MHvOZMJRFe679|B#cE{%a`Lp`|DC^amHT|^ztOMqKhPg5?#RAZbB*0TdX3%XbeX+E#gR'
            'Q0;>G@Q=MvlWp9{NvP9S^378iEP-'
            ')n5G1@7##D$Z=fH6H8|ZC7@Ty$c&lJ=yt1ZtQ!XJ=w{}o!HAmo!ABPPV5GS%WTuPSJ~UYII$=ET-'
            'm)&Zfwy?H+F=hGyCceC${e~NA^3_|9Ag;t~EaYZ}e~aALy@nXhygHkfvX_4v>INo%H<i7396%Cyl=6p;YJ859X3%C`<('
            'w)6hH#l$QI)Qd#nr^5H2wZeU0czmq2uLh7(+wHoEo7lyUJ-mom+3E^wu*ED_bE33mZmu|92rV`H8#NlfkiAfW})^ubH0'
            '!*lE=WXU&YYF|2ez}r#zyC(R>i<B0nvfw3xX;lX_He=F8Wq?ba1lPNi>KRjrO6B1RC>HA8S>q{X+x(E=2iZ}PCKCn|BR'
            '<}uzIm3ZJMHHEk986ixu!};b>5=j|P*{bWj#*WG#8%fr@jtK)K;C9J7u^uB<iqM)fu;eq#d4AC17hdLQtFtw?<fDT6s&'
            'Qska~J9g}Dfo4)p4n`kh$uC`o#9SM+@8x6N=Ut@ijvtPeHR0H;-B6k95B=L?kd<$Qa%Wq>H1{0ln0ld}?$U-'
            'g?ib*s(rit~OD!<&6b(zqQ}OIb9H<4{f>MXIC|1x3kJly>G42tfAs$8E>6wwCsO#id{{>tc(G3!XH<15_1+3w)z<&xHn'
            'nG)usd;u1J{P--o>lysp?^Oy&$$v{h@F7rN{O&e${P-'
            'Bm4wg5W(^t_8(=_En6i@gVRVN#P+mP;{dxhOU+>8%m(OhgofDXCBZlhQr|H(YYBWpH1KZRSAtK&`%Ky#)b)7uOcwz{Fe'
            'W{4jb)>s<4=N0~!)v8Njkl+F!M0EP(f89;_-'
            '3w;9YvQhr_Krm9na(Jk4I6QHHC`<G@x<$792DjV}&}_BU8Q!q`c(mm9=w`;fkxz?a;++rDU3*cMWd7dr#YP1YwPBEV_T'
            'uBn7>pARQD(Dq}a`BdsiW`?wfBu%fZG#0QG4eb8;5AU;Xnf%(n4Xl1hk%VfQwbIUxU${UK=C;F-'
            '8CvP}d8cNsSmcW*xXtb4niA$}QV&<-O_~F1oDtEsTSVHM!|D`(oQ1uZH8A#Io=C*h`ONUIW$6@mRC-'
            '`N)0PqYIg0F@en#W&)7cKL_@@f*%f3%+5dcF&988W!^oC7fD^DwwQ9hdAd0nvHE_)7f*j7D@5<$tqaOn+v*qIb!s$wpW'
            'dd<NdmI*ad&m(u;aI&dIjC5=6Efb7xpf_n2<$YspbOmyZm!b>vfKyoLH4l-'
            '0|YDV9HDIVaK!hZ7%CR=?Ntm>+<e@y}!>jlyu^Ryr>+XS}Dy5oTK5E<a-'
            'g6zZcFydo^xmRDn^RYtw<#B+_mgR$^+#D!!|1sI?m<Fc%y1)#AKz3RX()EPEZ($a;HO%C`tp%vE4<Ph(89tuL@+sL1u_'
            'wF@oj+8=U4vCn`jLmeEXakg57lU0ofk?!DaGpbp4ecYf`Knv$g&%)c%ZU`T$hNUO%C-'
            ';uGUsqaJaCpur`vOR8YlxCxYm^18Eo&*a+nYs_^ik2yQ6M15-'
            '&O7*Fa0eW41_c)A>~SV*#R=5L@9i;u&hA`V#fiyOZ5R-'
            '(h_1i1C}Aug{Afb`AUsOZp0&J^T<_m&h`ET{;F%s7}z*YWz}y98)d!#~!(8`UtT*u;9btB~fdp*WQ2jkAsHVPk0qz8(<'
            '80Of_SHnkP4zG;I|lRpe>d%@b%B9EH4a=@VeF4OIN7~j0=#6_}6P__9g+}D%CP2~zqq84K4m@M$j+Ydr~5;U|RnIs#hf'
            '&FPU;OOeZ*dvLM{=kJUmKMkBIw}w@nhx!Av>+!g16T7aGl!qmpy%Ii;CvW~fht9CT)GsZ6Otjyt$;?3B;&{XQDD02H*v'
            'r110V9MA!gog;xVfNUfoN?z0Zc|o5Vt@KFf!e2LwZegfO_7?8YMpbHFmG1SeF(aPh`EeB8U7-'
            'X0jDO0#m%`C}{S`jn&0x~IhElPh$-'
            'Oa&~k!|Cv=D8OF`rRB1;{^)Jg)+|SD<DH~+?mxOvEEPuPMY68GzJ<d_b%BntK<2ImlRtSEz818ggW4Ie<2(fy7YdV)!X'
            'ov8k28t?k8ursNkP^UCtbSZbOJWN18_Lez|xuH59?_I+50OF@=Gsa+2CWMHK!A&$`e4oyp<diSdA-'
            '%b775~EsjgxqyA@oLEa%1;#LEg?sfs)w=y_&H=X%=VG5F;4$=Na2Ut=Wx^Qr`ixy}~YIfY73yNh)U^&|t9!9Q*Kex1@='
            'HO2>Kd`7l`K<{Ay^|zdubW|?iai;w-i8kZbAau%l6jpt08^@O$&)>iIA-'
            '^gP8Aiv7r$xH6BomBH7?Dm1KVIhwI;sQ6o>K~2{8En8M(4!7DSx(MB9rCasP{CxHF%j%ZCfmKRkj)zLS9xPG`8P@|Y$R'
            '8)KPa7t{=OQ;Apu%(>M@+W+=|-IiKdBohL>Bmb!pEtD*sPh8(8V$90zv~K(!-qH96ecC_i1<4O}7w<e{S(?f<<UVS`-'
            '*$d<>4SDqOR&Nxf+Aqb%EIpn6=eNnHK_a)r&pxhz;CG)zU7O+Bv*hnXQE-=R0Hne;z197ZZIyn2nILUP^?-F--DE3Ksg'
            '4Q)Hv~~!yGc=El6D0ejxob1Vf@kz+<T*xZgZUM~!cS^REmz`cV^Db6ZKovKNf`l5|-'
            '9MIDF2W2mPF4;<m$PSg_K;MzV9vdT^wdLI8JVaxci(`SsyTCf>gY&N3&@fv!dzZ!$i+=g!6IFOCV0GU!7@JUgiUhG(`;'
            '>y8X*;I@@)rL+E?IhA`JLK#u1QYWD*yUGGeA4XD-Rc%Ky0s1EHrxUMQ!z}btb)*4EYu&cz-FgMjM9QOvV;3({k^G58ZN'
            'tyd`P&%Qpi!m!}>+!)!qdBd?gmYD{RJ}(!p?hTQ&F=K7*@qL69dh8|z2fq1@>e>9g=9!Y3PGe#~3?C-'
            'o#zVQ(bQT{^+{JeOvykUE<8mVrx6C4F?<m5kkCm|@izEY)gfW<BjA{|t?&<=8LgdgG7HH+|v(j9lTouRZKrk&efTWU0E'
            'e4$3mo;I5d0&$D7Q=1x?SLz&6-'
            'isml#(y9o^&Z`D1XFgcX>5qF&gkXWSFa9YKf%ECU7#E+5mRbR@vMU{0SXHoigyQp<a8R`0%o2$60ylMM_)=a+)|3SigD'
            '^?tZFz(F!B%*^BftJudMt*ujZrD5H7qha8NIUj=-'
            '>1X)}7;5u|bRzy;X<F<=_*j#8JtNelCLmjcTItW+m=0w#WNM`5FNc8__N8BTZVOi|bt^VG~mX<0GEr)1VprU3iw&e6^K'
            ')Zm%Vm{_}5cbS3cehhs@iK5%A@GbYY?_`tMDWA^$y{0VEJ?@0(<vGoz2QjKIrp%TbRE$GO0#p0f`<eR!Ewruc&6LQ~a`'
            'N4dYx7dKae-lvJ?KR#Hdxu*Zd_kh-'
            '7>21^kVgvPtmV@~M0?~W+Dl)i<rmFRz~7g=Yx}BE`L`ce8~CAq{5lM~X@diMpOVq^H1Ih#<99KM5VYbkk!|OL)OrcXtm'
            '=dfzoIcMeJxzwn~CaX^KsMMaKia#HxYcl57(`<gnbuZ(pDQ@RQU4<eNJ*hG7lRY#yMEe8~bRE;Ur_FGnec*QwPtk$HPu'
            '%bF81oLSD-qczsk_^Oun!H9T9%(m9-'
            's22lqPy0=hn_d&GR)dswMgN9!SN7;x;{2p`#a>ai@Pud(D6XzsbrCP~!Z7Z$bnm`rv?6G)N89K~a1EuyyAnH#$4yOv&6'
            '}FjRw7e6jeyt?smR?ZTKewTFO%#e8ZUm9KKr-vB94ym`1fiZl%3E;`J+7t0@2(+Q^+y*LpB+c3tvAWC#TTI>Ta!_I7zp'
            '1VMZ){iCCHiQPvZ}VL1N?anSS9<)M`30<2aja{(21>H#b95UIlIRvt)@~i@*<!57BV?J|4}^L_3})<XajB4@1JSI#w9p'
            'pYq4++(95R5s$4N!BD7|hu-fl;G@1Gd|k`Is!ZC03x}?fl))~RkG(Wln|tBz(mlY$_2Hj}7_#ZS7hM-'
            '2j(558uyy__SamcSb(U^nS&s2Ej1>0cE8|)`mUsut-NZoRa0B@gq6+gaRgrny)NxV%ZC3M;1H>$uBqe@b`1(Q&8QEolM'
            '-@V6IOZIPwS?k$=#TnOvIk(v!gfrvo=*kTYr#6rjP~-h;En^)u+FcD#&7pyahpXWzw-*@(3XXvCV#wq>IM-ydYzh_GKP'
            'h1)?}B;SM<5a*>J062iY3?79D3e)L|@zTy>OyoqH9@L)VkU&t#MoYo3Q6uz}ICILKO8aT++L9-'
            '+jVWhj$9$=ZM43%!fJ($^6I@N0uD*}$KI{brBwgl`8rr5ck!gG?X>{`g+jjyBLLA~?7M;?m<8H~A=R*{_0}nmKSLOCSI'
            '2OT*y?J}5800CW1d8V08OAed7Y7mc`}_{1Piq&!1$^>*y!NW~5D$VeF!(#fiPdRTuDj<p3sLQX#8DH{w{;|18YKZ{K7i'
            'KNoP&+wN?Df-X&&cB%&Uv>8$n%}Nsi98G-jhYs8P_P=bv+PJg(-qipGLtT@^?}PZhoI-'
            'r74l_}qanQC7MAYofg+0~wE0U5=2U7z=o5SR89l@NlEJXv+!1D1D<gds!?bSKK{Hl8CVM4d>bVj;%;AbVSLI=UvLx{ts'
            'KlZ5FPYCx$u$1Y8djn54*bXAgIiw2;lUa$=8eLA;?&FA;OS@pBZC#F>?H`?mrm2gYnu^VpW%I<VmRJ3A2WYuVXC+%`sd'
            'iAz^U=t<%UObcE~U>wG4xQRw>XQ&LC(C;Bz@`6j@RQB}=M^u`$OC(_53zf(Ia{Pl3MdjKrP=>!Hds4(E6+h7<D`ka`}4'
            'N&>m~HRw6rdE+yY`xHqp2WJuYdukZwcbg6#lcgou$Kc(nf7sLfiSo^9q_0#GXyx=nn9h|4H<xL8_{?Qc8Mw=OdM^q}89'
            'I}Nivc}5;aqPk`Sv1^s#yM^@6^0OxLyeBI@dC-'
            'Yg=I?og3}8uSI6}I!tK31KI)$@l1I$oY>V);YkauyB|lK)J~wc$~ruJZ4ne73M1oJqhK578{%uRPow(JJlsEM$EsGHB-'
            'fAo0(V*#Y#Dt{j2uES;I<@wFsdRw8!y8ZuOuclk&mui`h+}~vl9jfg0Xq^GI+oK4mr#-'
            '1hzTPfm<vO{1S?gGdc+#+g?PTr3LiE>WvuF!Uu;QB!c&qyR;@{98&d?P;ci<ZkE%>0=-'
            'zeA}<N7KJ0;gT9!DAH4B^^CgAYNlO#qsLF3AKKStQX7Ipcfq4s(bb-ywK+MDm-'
            '?%Sr&J?IIAriGZV<gLNBD8Qg5G01!H5-'
            'vs>5yibWSfg_ki;NT?v}_o}b0omtycen?Z;^un^$=^g2l_`FK)OB#rSiJKa5xGd#)`wIct4Fl)yH67Q4tn=>&BJm_kqm'
            'uH>^7Q%TO7)2r85f$+=K5P;a{fDN-g-b}$GW_WR<gba7M@yMcqt5^#E@n1=G-'
            '8Rpv8%lKFF!$Y1%vTA+`=8CU@WgT|(mHh^F-'
            'k?fa8EuQR6{S(@;(Hc75=NI!R+3HT#pE1w9uD3<jGyfWa8mdJWb6rMZ4~mS|4tXPg4!P8MkU0RX*GD<C<nb)#beFnX`-'
            'uZ!rZZrfNz(|ph(>ZJFFJK(#K_R>E0sHXqUk{M-S%D#dVl=F&En1!a;&n3=+2GxN)%`;R;P8Yv<d8jIa)7aM-'
            '|)CES|assnWV>1LF7egdgC_%&S;SioeC(&OS9z;&~Uq%P6{38!E1(c}<3&RPJsl+DQ6q#&wXHN{e6-'
            'K3rDJ=meR5AHNutDCNWh(?z^p(Q63T3s_>`0++s6*C0w3%sDn#g~?6mO+Pi1L`L3MRnnu&~_Bj)VvbJ4%eXQMM1O^SOb'
            'p1;;{HgIr5C&Aw$PCkt^a){p%&+s4yW&w$<fPA(y@M?@A}w@_~)+B72$flfy)qicx7Fc?@hn4++1{Qn7G#TrC@gJ#X4c'
            'gjXDD=pa+D$Ot%f6QT8J42-_NhCjpD;Jht{@ZSzVjj$_N(*2C~e%%MALCFy6kcy8a$LiDy-'
            ';+tfToiOJhrt$i5_<U*{7&3Sv>jqmwD}Eg{!olZmv6>rvM-pUE)NLrv@?{YL{WQV9x|~a04|Mc!QyN!R-'
            'u|O%qy+IzJK!6C^Hd!wLEd}%5>Zznh&-DN_cU5AS}M83QC^p@GZBA*!qSL%dBE_QZ1!7jYDw0n-'
            '=nS`Z4Deldz>D9<&QX7~kJrp!4S@x*e#)xrP?FYkdP)@?1mJ*InS75P=D9LHLTp0Y<!Y@s)HLHFT0<UW;Ub)!j<CoZU#'
            'WzHVb_7|y4Ty$Zl{^*ytdRe?JLa*4%A8@^Rk$43jY@sINe+LgMq1T%a<Eb=y;>otlZSCqjqb~QxLX~QG-'
            'y?D}ghSmQmL;7+fRQr*MRb4kBL>!6Ajf=ED>JQx)c?RZK*%CDoNjNpN6OZQ|!xuSGz`08uU2^~)ML#DylXBFPTl<Njvk'
            '}<5?#8cfaiG6zEBxK9j0X9g__eYHiudb){7@VH)b}0*d1s;ENqZ3eF@}@6@let_<N4}V==#2%UgBAdyP{0MvPXb4<t?V'
            'OzFuLDAK;*&AN~;@_ayM%+(H%YTWDiK5N&y5#2R|hiqdD^;{CR(pws-'
            '6c`H_r<wt5*Tf9`^<B1kBpG;9r3322qbH|Z0aj@yqIuO@SC4U<nurT})^$RIry@}%`qo%oVA-'
            'o9qGb?GqpHh&NR)*AL`zhDy+m!!L1}@<qqFT0lp*o}r%s<zG^w%T^J^F(x46B2M;74lrBthf090Nagg+St*FC?^88s67'
            'xqk_CX8gdrE+N-'
            'r#JQ2)tymz1MQLV#o2JT@0ZWSD4PC?$TP<YsU0`f<Pu_x>$Dm%SoJ=ir#hUDiGr@tIC>(+ppU`ODesMOdma)Hc#dJl^%'
            'qrh=X7I2-N1>>1B-'
            '`UYyq<LbD`J(%g${&wH%dQ&SH_3o~LkKLZtOOg;Sh9=X12>2(6S3nrv8bqrs=8K!=GMnhG39|@0_LMmY!LZW$_;Seopo'
            'wyH7L!>A{Tm1$o6l1&^jyx$G+{L?)ReWhX+H!(fBR|Qi77V{NU==qZsgm7YjJ|g2G#0C}`h`B6m*VW?LCF${7H0%`qmU'
            '_9pJ<>6+oPA^29bhP<lN28A-'
            'fhkPl>wXzioa@w%u{Uh92Sb<}DFF`$`9p=2PhOZWFFy<bHi);(Pjnf?7m$y>AsC$%eBpV77Q}Ecyc_c9K8PQCPMb)`=D'
            '6}LM7GK;3e$u`qKRF)#9iP!|oix;1eFAsIiIPQm0r>pQ1Y;*3i23hrX*EX{IQDPAbK@#7oSA^04n-'
            'jCKZaiy<RUmmq6$qUr);Fi$mNgqXAb!hTJoLv7M4(7s}Q8CMc|P>CtlHOCr@_-'
            'K#oEK(OKF86L~T4wVMr>_>?r_*DeJ6!2vYm4um~b_i>4LCVrpIkGqDVa9P^|^2d!IS-jR7gHseVoP6N^l@a<n&5^aF&>'
            '41GoFVQTyjeHKC+N}TGx_MlXL5Ftczt`D7{=OvBr3AmaMml36~NnH|Dty(UTnRJQ?AYsn|+^No9T<So5Jvy!f5TI-'
            'e>r@!3XSg%kbeWe{$%j9@um!!7SIg<P19-_)a9D=TuU?7jqstgtuaHTO24z43e+co5;Mu>#VFVCU{us1T5((0-'
            'g7H;KAKThS&ck863XgWL66W=bB)nlK{OPYYK;h((#H%G%mK;LAe$fLGqE0I5?{r>vs6TT<-'
            'ap9+!w+N}M=)ZVW{)$HGh5HDq|@0PVK4g38V`2<%Tp&+3D~RlNrHTq;C+vpe`r<{gS&*np-'
            'a8p<<^Nla@b{#(w4yys)k&`G30@qs>`S&)zC>u;lc#5Z#5{aTQ7j)Po-'
            '3bf*NKm$?v2C<F>z`OW2rgqEY<h!eQq^|~*7;C(Bwi4|QRsmOj6P@mffR!5)sXNCbBFpx}&}Z`qFV`LP&l@KO4;?YP@G'
            'P}+y@1l;1=J(@6$YHlp_??b$euGmznJ?ofodu!Y37YSiXOD}j6S55`=g|d6S$Y~H*~Mqj=y5!nBEv($WD^fP?;TwYdZ#'
            'LNX{dgoZ^PQyj{3=Q4pFG#e(mrH#F3@6BFM~VbETFF#8uxdG~*Usy8?2)pcutU*I~t3+yA2^V;d^<qPn9);FLrvoy<Eb'
            '%?I!Z&<dh9AtAp(KRM%(B)AGTjoT<fOQIF>;3>0K2z3>xUcZ-'
            '?+Y?8aR3&6ZD;vbZbM(;6);k(OH=>xXgai*Qcq_kI4a?W%LJa0q!->;yxti2+aqV9@gN#2-'
            '69_%PLbKki^#ew+?p>Z0`YWZ37K=N5f)rmg8qd%G-$_HaFC40b$KI9hg1@TJqSgu>u2a`-vfB$%OO;g-'
            '2su@PW0lOdXyZyN3`z+k!IPopv4hS?w7kl<Mj@fr*0gLkT?RICF}55-x$0au?G0YL;vhHf`{YV$-'
            '|L1SUZzvw^XQOeSH+9-'
            'x)*0;6U_TbRNp8CNxw&d)1FFjfD?V52&k9AiewPHxYQ8fn5#tBt&=^^6ux(c*HEt7jGKL?@evYHI-BxJbDf@&Rm49KaC'
            'KQXa*BY=HO;bBp(-'
            'W!_dDY<j03(><#};7H~Gtyi0euWPT4HYoy~BkpcR#$_gHhxguX(I;Q6U9*(_BO_UFz*OWJWSB{}$3*xEObPD;?Kaa-hZ'
            'NSE!lVG&<7VcjB5$-'
            'PehaaV0<59&RR*hx>(1u@h2g6Ngzgh{q!oo>zx(hshFdr{Ho{gqrKS^1fKln(fqUv&92;+*yU3nc?wZ9C;1H1A0Oi#@*'
            'm?q8V+)>jZhK%;jhf*CMI(>U3p8Qe(TaT8I+1yX4xET-HZAc|D+md0^0t-'
            '5(&W$44b+FK8f?i69rsMj@q2O347!Pqm^Jpuy&bK20(}Oc@N)=<+*&6Z<)*x3uNe=G}CW9M0K=qs;Z0URrGKSjNt#usc'
            '`ijB&057zjHX{A9>8SQP4+Ki?L7+`1;dFgQ8?HCt-OukaC-%L@4*q0~9T@?5D|s%KY>kJ+Z>A7-'
            'GYvD#BjIvT8DpF30?j5biS%t@)}_^h$jYA&owxFd+}Yi5sMv_k)w}^&YgW<K$M&!~*K^RXQ8Rf|B850DFoj=yWze=Q5T'
            'a-'
            'J=6Llj4FUgTtdr9xPQ_E?on$rciT_F_=2C1p9LI1(&D7E6d64?0j5aNBfU*!d{MX+I+m?C2?s@q%`+G3#Js|=$F+6B?-'
            'I#?gVo>gP1)P;P<MLiptp3;nP#}WtPqvfqu6OWhWdU$}Pk@PcnN*Sfp)1P*@PN-!h`%`-'
            'c@s8(yG<BgTzeSyS(ai`0xyUw3t>hbg)?)vf|37Rn2vFWn?K92{$&E5<JtkAJoHK3`aJlsHVdR}mO|jFP}nWTfR2f9!;'
            ';|wJlhpWfBmU|z*9cR{i6s2$WL0Yn?Sy%J%b>w>vY1w4<eJVqx8^r5W6n{;Yu_8{%bX=Jj=oAGqLpEGk>V<@xzf}B-'
            '^e8Aiw2$k{+RmysP?%*rlt$XHp4Ax9x(r+&8dd-WjlJe?a&un#n(7PjdA^5q{mY85MWw!3MKTuzU87B*kc<xMMaHXb!>'
            'Qk{I-vNJGV9C-'
            '_jJj89BTiF^Gsy!>$~#+#pjkM2Pv<q4POzeY`n?XM%GKN<pyec+`24Z8ADL2di60{)%JcYpr@7O&_er=>=SYEc!2OjO}'
            'I_g2Ujv;{sFam}P_^KtWWLkRs`GUK=E`20)-zA3s3`isgSC6J-NTmzwe&sO~6JILA{aTo8sx((u@*U(QT6s1BMpbIr|-'
            'RW$6Am{>3GQJRcc_T(uOQHPMLVT6ngX-'
            '55;M%p55LcCpdEI>&x2p=56mvDKFk|712i;`uPy<>#jsdQoP(<B#XnZ9X)JihwgMJ1gYz)A8%7TbJJcE{ncVMqwAI$#c'
            'L*@OBz#+*1RA~yMOeGI|y5IuKR^*}T5(f5$G9X%Z8Y;I~pqH8)BvuG&_Pkd{*^6~dwsR^vE4%<j*`p|ud>P{A4*~QFKo'
            'O5I*bY0xnR$^I@H8FlNjghc?>erIb%Yb#UrA;nC&}+;Py^@Vh*di_JKP~2fyyY=p@KG6;b^?E7n<`1LAKNYif%*zYsn^'
            'XJYK<g$OhBE&wFvcr5x*y+7v9zi2yHyb7TQm7##Zil5U-0g4ZYh(Y;wA_*-u|#(e6c%9k|IcsdJCZLp!X`>m*=ya(-'
            '_tia8~Pk@gjl|*`9f$8VMsHK~ZL3<XktabNbRih$R_4P)%Fn1_>#zCgr{}2gnAGj=^f(u*S={Mz6^x!=Px9n}<6r&5>-'
            'P{d}BGNE8#09dwBEgqi9h7Sn@urbF93%xqzFLB)U?6P!a1}BaJb}xLYT)=oUc6<!3)egc2*`UyjMiU7PDxvA3tNWIPw)'
            'aQ$;PRuSoq`FP2M?$<LtA(c&NDy{&KYAhO)~r6uK1+UM>L_%PqLI=MG3bdPOp8l)zuRAJ#ccl4a+1LHt)4@H!m<B~hYK'
            'c&HOrJ@dpBu7WK8jnmA%GxN|RB#~?hZebjR#z<d;IgTaA!jsQyky~ad{uYy?shxkRjqM@)DsYuP-'
            '(>|FUFYFeJ%ttfilC?2NuwfVoCF(h#R+9fS}nXVW6v@0e;WhOOg_ShDnHsx>*H3zMDlCX3RrW$6IvpD86NW(@Xfx7A`6'
            'P)S3XyRnMXbAW66A6>i7tnEcYSbmTbE4&JHZnPsOLKt<d-'
            'R8OqF#gXaSC8Z!SDlAx2d7;7|%pG(%_k2OBX;jj>vbBHzM3UcD(6kkx@aR6V1D`C`;0wz-EHRa<Xuqa{|=%4EVDXB-'
            'mld_7w%dv-xqt_VFT?PRKvM}WToEe?-'
            '6fR5$lPixmP<@?NYRJE!K{d|@_eDJ=nkBZxGA0N%g}BqMo!X#ib_doSaHr9WYv|C~ha@wmiGJ`0lzMFp#vXjQ?%)wPKk'
            'kDsN*=)JYn2d~Z;6GvUGSCb81#>yA{94AL5yUPGuH(1_<{uNb9oHAbnb(l8dBYt3DB_gH+h#D1w8HbG_1oIdyf4i6_!?'
            '%*%(L<iyT2_OI7{L;IA-'
            'c=@0GMY$hSY11@uBGt)(D=?_s&5bfNE$Grrg!^{ihTCYG9a}b|dro+AiW%S1KVq&erfi`*p47=w#2~&*$?=wb_(;W*RU'
            '!{`xDiwHZ)IsOIseuB!F07x}4o6#xA^g)6PPKlm|G2pr)VJJX#?SkpzKjU`FuzYVpY))T$Qs!2_Xn}=s)a~f9ZZ~#0Kv'
            'QnNFM9K&e)wZjDjGvYZcDMHdue75|+0-'
            ')X=i#(S%zoU|qX4ZJ6AE>ko*5vm)Y2H*2!KgJ{&S>cNTeBZ+rx@z}OZSR!EzORsF8LI$(JX?+j5<5xpp7{@cxUCt;kFh'
            'Fmv{llE?J_@%r_rk?Rf2qcwkM!)f#dz%+8w7^mf>ARkNS}x#rAG%y=o{{alE51L5HAmZ`If@!#AA??*FcZ^e#WPNHNbW'
            'F4dW@904AHZ!@5s3pe>#ZF~85FRLpXmx@XH&@M++O^~$WXe{K?w7&g6F(Sh7sc4Nm1Pm<wrAJ%cmlDFhOK77Fgu8nCZZ'
            'hQnSSp#s=c9gj^wiv!yh{CMK2l#fr7qmO=LD$Z7v~E9y3k!~-`D6rs>D0rGi8Zk3)j|3+)r<7Z;iA_R6G7;-'
            '095qEp`cqI{{1HlWqeaO@p&iN^QWd>vXh4dZaG8_|7#_~<415`vmdp8<qR3Q-'
            'sF``Kb>0|jQbv6hc$+qAvdcXUTO5O^0mUia6vlRd($2Fmi#5316C11$(7W$?*jG)Pf*ja9!OnWhK)Bk-'
            '~r!T!p>ENXS+RVifjbZbG7ih^Au%&=z$1ZZ)z)GO=e#!AQx`U$A&N0K`-'
            'A21pR$crTYM;9C3m<v09j2Cq;%!o)H7{X5t&OlghytJU^*{7DoiZsVM_&4|ssyV~Q^hsnhNc#p*rUBgD+%7IF)pqtydz'
            '>B((BnL?ffdgzWMSyTEF&Aw&A=#NRvl#W4%pR!o$o{A)WlIEW1rx_i!tj~UnA$C<Tq`5fbi-'
            'G;<F?1Up1R}t)MV2%k^nvqFx^S+91@ky23RB#T!Sc5$UXUuKj#{1=D{er~k8HxHf%y>CXAUb(FM`JdO{5zFkZ1oA>?-'
            'S|-(Rxv)ZPh3I_DI5nikH;a$Lv7ayQAk2Oq$^kkb8J&*5T)AsVZcP`}5b*qJN>mDe)R?&=d#b4Ukz?04Zgy@TK`!-'
            'E?{Z)4fR5|ql+z?6bY$m0HmDSaN8x$q+mXDtB^`6!UBx<{227ZZ6`PF#2@9(8PYf!Kp9Sk`3$>_Tm5kU9)@|K79SEM&u'
            '+COP;RLGjC@PO@xUG=8o$f%b}OJRhA)KUgN<P{sz7;A_N%8J^g_Ylc~kMR82`G(EC)l-'
            '1d8gochexYNagZhzna4|gGP{&$$Zj`f7bxjvY7G8_fpOi{i)-'
            'HgMwTj(&;D_<ELfq$LbF{|7S9>y+44M}r&>*5D*rf2+lPALX04}s}tHgwhZFnnVefDxl{=wIK01CfvMS?&pP@UjZ7wRF'
            'O@r*c3>FB2>_wva@nbgJ`Xg0!40#NG?$=oD{CL?`cH(V=jB+AR!EBe>wK)**T-'
            '@I4(Dnu`lK66p6bDUc3H$Adv>kZT<QV$ZDch*>2y>ym(3FYhn{SMD=yI?Ks4$%I`IPNY+)jCu9e6Xbb*<7=6v@H*=?9('
            '-DhpCw|M%0g|tAf166=EKkyDWkcNyinb;><!45uyMj!6Ja724b47LKJ0}iKP%ki-HqY@naAMu7I#j&!G#OsAXWK@oOEA'
            '<vxep|YhNeB1Ie@WX7nernAw7Y-'
            ';zk^^ip(wSw?4<?m<<hgJ3>)9;tZa1u`K@P@H@LPj=>EPQe~rC((t5qn)%SAqX#Vt)agJ)2YXcJUkz+0OpgPpeiYeeob'
            'e9!+axMU~rXa<on=j(L8)6m_sfsECmZ!PbeR3rt@#krStygP*S%8_+_5Kt@Ibzo)klxF8*cpk49ltU>^uazoUn(?val<'
            '{Kzj-g!YHGz~tEk^jNk9_L-Q09>)WsxH149jd&q%2M;{C`ILF6odvIji-'
            '>QzKK@RzfRQB!iG`mZxOa#`#U=vFOiGAP^EdiUD;F-7ghKYIAq)(tA(F9UAfo=0Sl-'
            '}AwPSktDcA?}97=&L?+5fi6p_E2fpYs02d-PAtWh(w;+`iiZi}Yp^Vg#4a4p8DEyG_$b~syk9m--bE~`hH^U#X5Ph@t('
            'kKO(_<kgMo!)#i5J|8_KM_Ea+zL@i<5!;1R(X=}Wk7UMRwBTd>tL%b|UKFhGUxe8!qZw@pK9mhfCmnXY4Th59P-'
            '&_|L=pt>b)g6y)Jp})?fkUtdM0$63n5PvqEB%@#E@_}_m2|gx7`$KuL3K%iHJDQ<eLpo=+!svB>wIe7!l~fW`-'
            'APsv|Un|9a!b*aUL((?wkNOq1sFUPFD(?P%(DkNhqZ2A5m@jHRA1_HPs<jwyWbNvVif8I;n{-|C>!aGxe*--'
            'gUZD;pHAdcc#VQ^b418^WdAKq2M}`P=G_B4x+%s-ZMmH%`EfMTQux*-'
            'O`v6J+5g1GGQmjvoZ|arV3upf`MnF?d~#WUW8G{vHI0b!w<&EQgzK%mv@&K_Dio2ug7`p!sMtF85JDgZc4r<;7dpboCE'
            '%?e97Iy^fFEG;o3C<-cgVdl>B0j)kMD5jYt+OVhPP3(R9*k!`9Rnw)Q*)8CgwN!2F}a<QZZFPyoCFTVRiz-'
            'TaVO<tpBZ$Ho6%Od_R2Z3F%2^8mXK=N8?d?l&?$7}dN*kBR3b-e+Ou-Tfu!KX+`(sA%F{X+V!{~y-'
            'gG%Cli3ma}!h@{a_2t~;djrY0l+f<s&LI@3#q==N}d7kHJ)?AwB``mY>S%y$)P${#>EFr$$=l%8l`quZX=ijxiKi7Heb'
            'FaORV_oZ5$6he*N~JgNnBl^QGI*KC7w;uh%y9BihKzU@oZo!{E)RaDo$u~zBss)@`^#sH+wYFz8@(0mPOds|$<V>@wZZ'
            'tZOcAI4q!25klQgohnXJ#bPt-~f4hRR~dCw|Hik}Bo3vbeH@wX&e`6>PW<}-'
            '2@h;pPv%CMu~kNB+T1&2~U^pVa6QKMOq6d8!V@h2e6R|M-~A~Aj6btrF~3rbg6q^Vhk4n3HU;&<-'
            'QscJDQZf623_B{l}59S~=UJYHjf9S?5n?dw*0MxDFrwjSwpygj0ID|0Kacw#*yqy5tY9~OS?+&IcQ^$Y%#z>W3447%G#'
            'eyFl>@X)na$`#9UQPwdH0R>i3o)=NWH)>d96*^%2QhU^I1zQXgVb#Z_Rk~G>9!x?Ef~S-kzsnZTM+llc!7-'
            'C9?Ujc57T{r8D%S*sG4&;F^w(NIKDxgGRp4YrUU}ZzJ)_ehda#A2}Y?*U9kD{n!U~BC#F|aF(#uoqg0q6thXOvT%FOCe'
            'rE}l%Lxbdeak>OLIEN^wxQaYnfbN71J^9*z_v?QajYVkF4&a<B(IYmzsP|k#xKo=KZof@-+Z>~BT;a=-'
            '9^G4e<pBp1XWbe;=AZPTJxd}Sc76@Y!fAGy#paacLg;`i(;?nY(-n~1ms?_3MCvL)BFQ2=xFds(;_+)?Nr>T<-'
            'TgtrnC&NSNp(=Bo|n^R{?Dv{?gd(YKrG~ib3Q^3;IOe0fFXtytT(4dE=TPMm`4lyvk5KBMD7xEMbJBN=)`%r5#^$D1$9'
            'Xw*;0kEZ4pQ#baloPo)$%o>!slU@$}B$xf^~n2!om1pfKQl6-'
            'q1MuDR!NO7gmD*Ql>=r^$Ab^WQ;*ln~FYa~^h#>iI3$2AADSHM1--'
            '(<aQDDG^nLYXUi5UkRQZ(J2%#|2LuimhX>EQ~<esz2;o);ZvqV~A&kZ5gFgw^7YT1qH<yF}zzWu;$Pblu>ZQyk8}_`gR'
            'y<oOuJ4O52c~l}XFz2f~6cC2Y>AKwP~f4sbFDj$XM=cWaC?240Df<Ci{?;XS&@Tw?~s-'
            '^R(Mv1lky4TcZB9rVmg3+iqk42{=k;~Bjus#B;*Gz*L|P<aI~M59r!<~iMfH8`qv8kpRIpjKRtN`{+2?*&jk{~6u;ydR'
            'cHZ3cI*W@wnrkMhFFsQn-VKY!_@^K=H#Kl>)5MT!p&AN=?q|H;vPwE4gBU#a~M{09pcSfhWYgr=#%L-'
            '_Ve7N*6A$g&;L)T?ofY6s`zEv0F!crp*NtA}9V(_#=&i^o`*clf$(gfV<Tx^Bm%F>E<!fDgVp;-;ddxVt6_*B*C3-'
            'gZ71f_jjj@@5Q734=i6A6jC)oA!J2&?{d0c<}vBT%BL}AN;9*qz3*Q|AqSh0KeMwV#xcLgH~D>(4v)(+*@~4Q^{)rGVQ'
            'bQ%&`=FDiaL8t42xhcrv`<D~IFH)&U$VWo$8TMfMIZG#=MLEiDx|mQV!~dn<^vpd~%<a{=v{%0jnX8#4H91cXJ6A^1-'
            'Ogvi^W{g@W`1O~9({odg+`8({o_I&)eke~Bpy#Zu>%^-K?x#LnLpmL76GkZcEo>cn-pNkD?a`Z#5EeZ6ns5I?3oq>yc^'
            '4Y%y=aaXUEp#G93$H7fLB~OH^7&H*Thu3&Ey9rG6#K-Ze*7zX;9>`H|Cj+U)-;flfr&8e9SX-4UEpP%I-'
            'Mtx$f(KgrbWkw!Fjj=zbH4N`%oVF9BzXWN<x_a_nW5W)LDEsCS1E~)*H~v?}Of`9bm9(16T|EA|abhz;}Le9WfW<^xN~'
            '}+TIq>tO@|56@1Jwg%Nx@aRCMA&VglD_rP<lSbY9-lBP}@p^IWLBPp~PRrnR5{OSW}U<JZcVKw-'
            '}+KN2frc`_2E%3Hpqyk1UH21_UG!EMY!~KmQ;laT<CEU1pWImgftBew3u9&8GnwFiL3mQ>jbj#!(vViQyFpe!f$QFgab'
            '0n#rWh3L^DmSd;w}fwQ*3f%cMf29{(~uL8g;z?8kv%g@wUZWN<vefj2`&KxwQM-P{S<c9RG^DN4rEBUk{w&`z&-'
            'wX47W=GSTp2|$971;meSi~@6A2H@(qI}W3Qo9^D>6FO2f~b2*~$41108%A<49Y2$LI(-'
            'Onzfl2H^}q_F|KzVfn<YN(K%9=Cv<X$6jdd9b)`m|?PM3E^hOf`Eh%YWO4|!-'
            'orN(G<s2Qy}y2b~3@QhKW%R38&c!#U?LnCjTgfBJ+O6T6l(c&#VRq{XfK_KAZF~bHHzYI%92X6s_s<q{d4WVA<Sgkm+~'
            'VTwXd4UU6MP-I5_jhFTHYh)SYggA_3s*$ZBKGf*v00LuB-gUJI8`f_Iod>C5}t?v?W+lT|uYl$$lPytqPM?tq?8|=;*B'
            '1u`LXj8JDaZEP>yUxa8?a4cE<n9p|GvAA%S?RF!>wGlj8p5~xcyZ47S90T>3>M|>hu25TL4VIdSm0m}!k<3_4`%@gxmJ'
            'PleLebh=T&(4V3MAq84Rgk-'
            '6&gs488PEYOd%iM4p+s&NEa<`(;vamp(V%d2j|ICwIf?<A2#G@d848OJUE|yI`~ZC#*TKj{UVc3*XI+VZX0khS6zjQIm'
            '5WOm(|4v0DjhZw|l%X?G&^F&k3m^8!zt266Z>3&rbhf~4sK7@drQmMI@h$y-'
            'OZNXS6*YDxI|fuGJYm<9eiA!NE@4fHL@B8Im<f@QEQEc})UNBH-'
            '{)?L~V`)Z0*&(t(nYA3n#D2oP*aUe2Z7)^rGskWK`Iq+yBseF11FMo^x7by<@vHXJVHuI3KILc1y)qrOu33YZC?4W)-'
            'MctLRkoE64H6JoTJ<}`_b*voj$5sK?g5B7^{yN<+>4mSmi@`8H5aNoE4sZWWV^>YmSMJfc>~=Efmlse*vK#F-'
            '&H$~C+T?v@Gq8it!|C}q(7B!umS3#LA*~N2&mjujECSh#Ii<K!*^{csFM`E~sv*2S5SHxHz@Aho<m@e@+cz!4&H+6f<Y'
            '(f~rH1%bzMN`tg@T1{4YZG|*WPIkhieD=@Ji(f{dazlZprX~2OCQ?bB-'
            ')xyItLf`~}<beEuA=`)wt?v??Dif7AoZV}(%r(}yAWsh#{=dxNp7$_y?|1wr=E1ei*PLUr6EV@+`oiuapf=mRq%Z}*R;'
            '&T)g~sZYq#fL*A{Qv?39^)XW|6y=*Ufakk2hUt2u{VW}L{!)#~dhW-'
            'poq%NCbtvk4hMQBfp;zt>Xcrff!#|wi!M%R`<bDK`*2vY#y{<z6=?JviAc87N$MD?!QZ$Qs4EH|y04HS-'
            'zpRx<$<f8gH~S7H^O|t`eh&H$c+y^JPdt=ghW5)NVV}MfmdFjkgXInI-YW=GRzGHY^AE%6jb~9&Xc7buSfOuWFY?++Lh'
            's)sI8}HO3vL=g(DmE&mxv|x{@Y0+w<fc>6qjg*)lSgz)^w1*Qi#`@y}=^qC3x?<4_~I#LFAA#<QzFjzpuXy_x{x29|a+'
            '_e-RDJ@{ZUdCIm-sJtKeSD1r9qS*-'
            '85i(^|<U~qR86ut<?Tg=liIqHcbQxcr8qmjh0Baoe={RxZtN=V|W0a{{mmEHt@VB8Rc$;Bc#_i`b;DoQ5@4>}WxWs;yC'
            '*AB<CMQO~vrF4PRN4y_;6`a#`fe~j4krVgeo5g(i^6n7{>YIggA6o;@Us;^+sfG<LPFQx=l<g?E08?gW>F+-'
            'W$oVvWX49rIxIGm?f?GCV!Z#PNc|<U^F94>NJRzgD`lPMu4E<=IMqel$!3b|}$Zo2@bnST%JBx{zYAZnTuM(*3Dx^B!0'
            '?Cgi2LLN?9KYm-'
            'i?&<S!#}MdfwPCGU5Y{9f(*EOKLN*jTJV#QKfwCK<l5A6IxLt7ai<$_jY&QU`{j>1{O6D%?*YfI5ppf#8S2g6LOmaIaC'
            'WK*ng4Y)yfHrj_k02vYb=e>QB(&z-'
            'zwJ~96km1T2&B3){)PzBQ^EjPqFpmvS@2&9UAV~1fs$|aIU_E<j<Z>rkAm>&Y>JH-'
            '`I(3B}|CGqhd_^`<YaIHG*#KcCu8a3DqaH@yV?-'
            'G=FU`{i3D?i(l)(7M;~_Uicql$lx&?Fb%@jg6SZe9EpcNPT<y+xxjy3gM=!dCZ7N7H2-'
            '<ZqwO~joI5IuAB<}uGDjD8>OaKyG1ky_elhJ=i2zm0ncero2p1R%!EcfX9j>#%{~te*;?-'
            'o|DELVI3q>Hf>OSH^Ex54E1f9~xh+3*2RL-hKzNSh%=r%}gRa8K4r3Lw7y$yzYx=8dDKaic{4@c9C8J+*4z_!T-'
            'YNO9$%GcSXxIYM=w$CHePMv7;X)9cJjir&CD%vsqjeI)%l$cmv#Jy=;!0<eet3+~X$3!%ZcpX9%KN<o&o{M}QLhw{R6w'
            '_bMXvn1=_+kAU2zii$T?%oyZb3g?-'
            '6Rg0J5NFNoLwNaJByt7UPfN3_Muq>FJ1Rl9o0YIgF?T9aO&JuJUz&d<wg7`o8W_f(Ocm%zXs;c5yaE052LZ(AI9dp0vN'
            'YWANm7lLv6SpF7x7ME^%Z-i^VB86@QqR+Lb_Bd;_wVU%`ZXM<Dg91-'
            '?$c4vg5hbm(LjVcxdHpi||<d!hhzOLIWR>lV8_aY%E2V>}LOhGOf?IbxQ0!b8h4)VlP9jKfiQbjuqCW2QBJCiswLy+zb'
            '4I0Wi~{2>33H?UHiq0LDIVh@ZHA+41-OJ56n>dir3@i5-'
            '+aH3N}3TR$%5rf!TFmBdQE__L0DA#ln`^zejxA_ZsVch|f%XeYTtuwG&fC&};UNh(JLxDUGD!|=_cQ*u4Z=-'
            'G$Gbjc5yMeI!+9}YyXO5X`bI`CMmi%J*A&<zz+GXC0@#codu;5l7HO&!XU%lL?X&yC347sl(T{!?jZd*aesgb^#;Yc3Q'
            'GEi;ejO|(f$n}?=@W#~^#!9@{Q8^b;9QffxKma;Po<Oa`uZaEQ0?^pZpxgTVAj-Z0=e*Yde)V8jT-'
            '!|>R#swgvlOk@V37*u4p6I)#?va4{XwRRD2tiF3g-yiol{Chn^kdcAmFS`4m8oE0!GfqgF$X3zWf@3#u-'
            '+GPkRTber$q~_ieQB!X4b#O|a+jUE;=W0Jjt2bdO~_-Zzw@ygx(W*fU-nuX;?>ul=SzQzf{34lh$LLW<e<(FuNOh0~F('
            'Bj9pmFH9Kx0fW?=;F_@-KTamoR^27oC$@r28qDH&`YgnFv6~DPo4bsc%Oy0J-'
            'k<S1qa7}nxPX~&IQX+`N%SzmwGrm<_swJCX;X;r$Am#5Aq$oIK4P({8?G%c1v+mttWt=BJ-lA<WZf*zR4%1nQcmp2*F2'
            'mztSIz(>qWU5e?aV$mCWP!U(xiMN%jqg)3{JyiupXlmdvW~gXb%^z!r@dIMMA1KQ8@%WIkIsb-'
            'tSDw044;+A27dF2SrD6l6MmU(8$`C&W~GRsix(!^kN6ug3R!9(s9}0LNSZ8WyZv0u{>zIr=yFIL&qP_;-m1td-'
            'jas?Pcl^6V4wH{|B{{eA)VK7V0r<p6kP^5DM4Fq-Uq9dl0HWwd`k2p5f_@xyc{r1-'
            'x>I<HzYN8}0|IQJ0Vew(Jhx{IKrz6vrcebMrTE_!9=K|;!HkP=d%TzcKm^8|1>Uj&FlCMX}sAkCltkt~lRaOFt^b-'
            'Un&H|qxgFDQd~;*7>;`r_j}Q(`Xb0k3b$poo1t`rFtcjIG2MOO5E}qtED4{zzCcu?aoSSmO2WwKVgbJA<fhCKbQD7>;r'
            'ojETcW_~srDvsUc@j1I4ZZ6}g&#gG~Bwe1I{qbvj;T!b~I@8Rc^IUWv4r-'
            'o67P*mn6+WuA|4aR9CdwChv6MD`#v3C>v+q;N_ye=VkdUwE^ygrcKroomdSOLp72a_O;Bq%l(pgWJd0^gq;vS?i+OeD<'
            'cfX+J5GCP8g<X^F0nTF61FC)-?mx(zM-0a_(FR;1(C6$Y;hN-s_G)rg~UI{vhnqv!KXGAC171qIgpWD<@J|8Zq$-'
            '~kkugUYjO+?c88`-'
            '=z3jaP%g6Z~5kbc@oPEXe`mdKaG`@#?S6IDRPbRpc3$zg0;k_&6MN5Q|g^N989a7|VM92+Rc<a6Sj$b?W*7QF<v&Cdtf'
            'y;kI$Ll~(vNFgD19$21}#ZFmhKoT~WlHJPp;j8sZEK{1%+*<qD+im|+x7l5Iw9<i|4weH|n>2E4+#P*CgyDu=XQ<eBPw'
            ';lW4rxn9K>OhhIOpjOnQzl^=yML3pY_7T#*fr&wib9B*MR*KQ!KntMCT5z#B*jX_&BHmei-kDt<Q&uq+k{nN!-'
            'VmaY<OR_a<~cutu|`Hqcr38s9_@FfO_fj1S?0l()$+ey^99$6bT?&E0GfM`8FJW<Zl~RA6;%2zc!hfcRu28jHgyXOIsH'
            'Ew7`}SS_5>ID{dgyf}Dh2-'
            'RhuLfEW1OzF{dI3L9eKYTvmWM3e5H|#`pdn=TS3`Y4w)nuf<7aI3U;i@a;AQriZ);#h6{i~I5=86smzV(Bj3j#4lnip+'
            'v4B|7cBy_)=kB1cUVAWz<Jkb&kG0x@q@2(BIVZ|pBzI_Wy#{8iMd~0!@bO4-'
            'QDu*2zXJE?H47d8#f}#8Xm7183Z2tlDU(ikM!uc4>ySz0=2@@x)lTq%oC=PeuAWkMRAoBMPiaO-NUu(pjCvve__!6wlY'
            '$O&u6r>y2c;s$7-fW%+Nujp@3~gZN{eQ4k#1Oatd`Z-'
            'G+pvp$ond>=5iOm^NyD!K@bZzzpA8h6<wViQUKp8y{On$d1YqA&VDo&mzz;o3(fK+dMR&t6ci$@-9K4mZdiA1@mM6-'
            'b@W-U_DBzjWhiU_P^{<_c<h5A~*pIKo_p1K1MUk7dO8#XGg}v8Q?z@V2_r4=7D)X7s_1SP6>fzsj8a3J~Ow_xBX|i-'
            'NaL46<sYeSr$|xsCZu_BxP6Aq)@GzafZ-'
            'Y7|4U~R=1K0S)!&e`~HM)lh!~Qq<v9uVXUJH`tj#=<pHG^>r@}L#mAwcL44RW@?8}l}SmXRk)zWfHBUE=stZ#9U1NyV2'
            'Ek?=1!hIW^?g2kM0@`yVf@Bd7K+7r{zCDucF$N!L{iR$3hYX#543~{Ax4?NYpg3qo-LcD7qVM-gIf0+xsbE<}IYCPm#+'
            'sq7eD#j@5IcT&w829S-fs7OnZ4~mSwmgRLuCD-'
            '9v>&r2Qd5z2azA`cc}xUSvT)R94$8aN!)^a4lCvb2#LacXq>)kRZaNKyU)1o3;A}MiQ%B=_oB$uLAXAaT>bw2~K;s%OV'
            '0Ia!e6JW<v?#;p*@>v_8V=EGUeR@jyeJ&@28n9@869yKJbun5tJd#_4Hkh^@lyw^trH-'
            'n<5{@rh8kqK`NGdFH^E}A5ClAKVP_5J;qdQ99DREOQWN!&4tButfhzpfx(d1<l|w{kBdk1Dh5G3ncAkek2<=~l7i~?j#'
            '?1lkT!Y~-'
            '@yA)uxH!iO%%Q8#4pYS5;M{2*nAKSZhUX~S`i(Gtvp1puBL&n#l|bWn4ZB)rxX$?U0;;USz#VI*z%OYN&OPmojQK7kb$'
            '32G|IWqP=dY8-r9to}DFc4JHv!p?uj#UbqjZR`6r?Wrph=iFeze#DHxqJkaP}BDY*)b$l@8#R+zrpH`-'
            'n^WJ9_i9Jwr@Bluj$QB3r={KS?cwmvf3pQo;z?H0LY(X`loCc#?%b-'
            'WF4lsxHhn`@{Gvr3v{H$Kg!<6EMD>fbPfmQKc&%3qBo!6Wiy)lWT`SXfzyr9y}*0ue$JC*FyL^JqX6zSK;dW#dNdGHsa'
            '7Ki9;hr_+UvW^!%EG-LCym{mF-X_jt|@g?*5;?l|05jmFVMgJk}`Y+&*9v$t6s2WE>f*2jFA*$q_;_5Ry<Y5QKf_|i9a'
            '^^gbZ*bFcPmlZ+bHUX%}jD>$qMf6mbAEfN6h3Sc>q&oH${d5r^=$0jIR4hZ6Q;k@$=OD^Gy#%s%KVZYEB#iUc0@i2_)p'
            '4*UOS7bK)3-'
            'aAKlKXEQ%_uVe1Hb<tKuD7b)veFi}P{EYm%@u5~b$F5m}#JnowauI~pYE5Ve9A7tWKjGNy!!(@c%lW|P9reE8c<2TmQi'
            'M>joAgL@(+D5l#>C;v2&&a<28BF8XVaax8d1TN9cI9ZC=Y=RHdbFe%0HyJtCfsgi$z|ym~L2i{W{q?s7u9r?Ta@SXb?A'
            'r{CUGkdo(>fOGHXkJFxh<d&)eH0fJ%#1X@gyeg7HI0afcxj?wKaaRki0FKp1-'
            'Y&Q87)(Rx2V!DqqN|hr6MTe1+hoOvYRbN&GFHf_X~YL2K)GlDeuL`jQAyn)`*#`jd=fN0zY}M%U@dC0xv~G*jj*OMd2R'
            'jD@wQzSBjQJ;1!h8X6+=X-q~w{^<$?CHa5UtmGSM-NMUp7Tu2V-'
            'K#+`vWZN^9l+pmAsV(Jlsr^4fKhoNkYfyjpO779nuUQ+L@R_X*hT);jlz%M6x`Ii32u$^p};@`@UOQ*Q{E1+J9ie$9Z!'
            'N{KpK|LaLqT7olw>$M>E&>g6oF^P}g4t(%)WT>azq0e54F!_4V}b=L~EfUrT}~Gcb2dAF|UmQRwq)jC#r;AHrH;>S6)h'
            'd>eqGrw)@gp^uoU_8WUl@8E|+9;l~LM+OQWLa);tsK1<rrV1TIQ8WboAqd5^qCjJpA?~brL^H&aKt@pkzx^H~$M*Z6Ru'
            'C66<%cG`_H+T}`5V|jFAGj-'
            '#F4k0F3S4R0nL%K$>FH8=+2gbV~#awXe+?!9n@y1W)(1kPKA@p3w?3lEGLLPmJ5rVy&%Fl5?}ooCR>*nlX4Gt@NH?w@F'
            'T`Z=nY(1mkCeu>hM8oIlFmqruVvnutzBpBwIRZkk26!W|u~zDd5)`oe;Hn9J+t*q91SB!CD1vFuEy%<^~bis5f8p!!tr'
            '$CL7T4attvlqp<b<DL58miE|gX5+}aPaA(0VbxnzdTCde0tv-YU)_xe(|AY=a+)l6CC1d`{EoiSe#z@_{4?DNp1L0PN('
            'C+KF+0>67IU`K9QbzGt=td}VZ=}_`H{<DNn^5<wB(U|`Aj~Qcykd%pL&JQ)LtL=>NjQ6PT{27;hr#gSKG3c-'
            '2lzXS9eZXQoLtvLcU5RmGVTY_rZ35Ep)RP^eN60>RzV_n99wVoCft}T4A(1zY5DJuXxYL?`i1Y(f~7lQNJ1B#+|pt0?b'
            '~ct;BjOM%tGlC>mbi(7dSn6$ZqDZf;ShFVe4oy`%#nyG!^&Pr97U|S>d<v_^)dGF;xtGwux{qas#Oibb+F$SD~~!m73>'
            '9Va$&%SUSps?aE9zHF+KT<8I@go+RA<w2js;UI9_Ucc5c5oowwn4X+##+4EM{S#0bF)e!<+kMiN%?kuQZ7zKItv+=UVJ'
            'f?xw9=hkvOB8(-2dBnk@Y!e*$*+jOqXQiH?DmT^6Mt&5i3#(&d|;2HB95!|fsTF-zA)6lE02YNV?B>?re|6+cOn#f1dD'
            '0n?=n~=m&@+1t;0h+Hjs96HeTS#gx|N+a2eN9$jkUe<?u5#-'
            '_eZ2z5CJ0z>)f1F2hv^8|yTxgYc8U2ef%r2TD~Ol8|eJ*+-Uw=B^Sf3Gk%*wKH(h@ip*OEd{*4)-'
            'zm$lHgTkKd3M60k1u|^q9FH?$+t0&yS{S%BtsRTG{_3<1Y62?a^JjCN&Nh&A!2YpxZ`{sh85{qh6qLCl9x<CkU4a0M}m'
            '+Olj_g^>;E+-r^M2yJ_LZh!OJSofD21<&)&9X~-'
            'siI1wBHmzZ+2BZ7?w9+XlY@hXTeIzbKuIKc_48q(<2LRVf>!0FT>^3*dHZwnZJ;y(>|tfPZ!PoCrcJvBs%Q$qJ%;o$Qn'
            '3fOQsj_8FsL%zB+75ng#4Ax2D%FitD*ntqUWdY~>O=ZyNjs{U9F;3x?M8@i0wHPI{8(laOnCY7fb;ly<J;6&b(0U1XG8'
            'aH-yCxoEgwoZe`dArS2^}-wG!L3W<Q9**mE+-v{}xf3;F<cFIuhm40f>xP2V-'
            'fP$i2IsVJ~x;@_&$kdh!dHrni7={e8^JiiHQeMbI~695#h6rjP3OVWylqO!Mu5TeEEOMc@_q*Zdey<9BxC#d#c&va`@<'
            'I1c4cU10XCLEMx#iRnipam~m8{a|6Nk@yHfc|j>*9}U7w7w#~|<9+GB7wtsRFb{Mh@8NTi6tb<#2L?~X!z!a6jN6S-'
            '@OhgP88S2k<x`L8{NHgHrtU|+x+UTobtaat%SV5Q00y7NTnLSL2*+#FP*{VB(W*Bv_nZVf{tz#!&YvLn^^;-'
            'Zv;n^GiKTN?I9T5L9=^1;vYqo@z*OA?Do*wiMccdZP5K4so%&Y$H1-'
            'oYzPp|*{%MV06l1VM)E~8t8*xaim}bTd!QbuqGd%MgFh!m*#J;G3&+|NLM%KeeDGd;q@Y1}c7KgblV(2k17}XuF;^P8-'
            'ro<y(Hh-xsFs=oIR=F>o`^X&?pK$6PjU>R)>@=L+?T8sdcR}Z*HCArqhpaD8$@k+SIN`Mpz7(h7nWV$yJ+~p2jNGN-lG'
            'dQTyA;(fR*^F6g}Bu`03H>GkjAJxAd_v0>2vnrsVk4^+J#BDzBd)^HQ!)H=uWnY?#y0ksvs6DFI-'
            '=hMQwSTQA;MOZhEi?6RuTIa^IX$2knq-tp*-lSK$5ppEPw;8pm9V;lyM#x%_=LU5R=y-'
            'gp<a6y!Bsv**Gpix&LE&!%N}qoHyJYtMWuGF&7IZi*RD2MXwTrV>|bH^Mq+Zf0xHd^Apq!<t3~(xM?uYy@3lIFbd<PTQ'
            'c^G7B#k`oS*`bt)rz9lEQ#VMFC>ViD$nPp+5F^z3$G2=UNq8wAp__T=O1XQWTgmh2-kRHA*1ZCX-'
            'HJr*F{G`t9&RnO<h8S~>b&jNh#r3Kdf_y?;Oo=3ACndo-'
            'ihArxrk7Ol3Crv~L)z==?Y&j_nX99L%{YE>GrBk@oQwB^eufg_hbAY$c8unXxlgNt)aeHMDgDW1XjqEsNcU=Tx-pPKva'
            '|iW`dIzPel0d}92R@ZWL%#c<+9CrLT(YhJ6^@rfkzEAV2FwS|-^<YIk1dWSrGct_7CpyUj>`Emkd-S&(-'
            '(cGvP$dldGS@)Fp_|hHFv2|8;6oT&q&&F1I*y$VE;1-'
            'oaNwywPID6Tf~J)^ThGdr3dU*MJ`Uq!Bl*&#6{U}UqEf<O1z+N$5=7@2YGqtKG^xJC+eveQFm<^9DXB!yMLJDpSQ-'
            'eLOK>?9>yWZP!89$=K&q!CLt-AcvpU$(X&XB<Qxt`X|FKoRnWx7hbQ3FMLrN-E&#GaK@>&hu)-'
            '$<zYoucCtFHj!2wt5W9vpFR{P^_yLzlrzeNUR_hM>A5)253VCOm$STBA9pMQ>rT{kYmsM`TJIS>YKH)Oya?P*-'
            'hBY`eW8}Qr+g0Jxc6ucY_TpvSmZzGG!n$1B8!DJBr5{1X2I#K?%5EP}<LFaihIKmV}ZqH-'
            'D<5UPu`_96R!DMRoQ=aX0F&ZwOcn^wR=W*Q713sx7fD@qw;NN;4bxtgW7~2yx<Ws)p`mkWK{OCpq{4$PYhd6UEq?`IFZ'
            'o>F;wP+j9M(Hg)%wr1MvB6_RgU|2`zFuoZKgnC++W`+S3ChAL+u8KU=OD`3B#B?o&P6H1B6i}95o`|gr|s?gF^{Ja5B4'
            'r4#+o^BC8!l&=~B9SM+_`E^@E+T@ed;+;{jefu?$j#J@K9XZ`vtdgpTp9ka}JR!)m+%i=Qys&X(Z&>c{X3{IEH$9(UAR'
            '!x67ypg-E#FYQvm=5`x)N(RxMwPG~n+C4gVJ&PWgy31y)JO@WjB`{Gy9=SFplQUwQ@mO>#n%UJ7lQwU(+Ln!t9-'
            'Y{J%9RL9d?Qz~vuUu-d#sKPqos?kv$t(bpub)Kte@vbDo%L=`*u03g|a$_7h(*V<C{R~M+4s13kL4-'
            'eB8R+2RWj4q=7pMc1dVsU~3%kl~m%5f@>t$mWy=Z5C$f^!*E|wI9{cPA{mQNIlKzm|GKd+y=ub5HBH!UOM&+V7v7>KXm'
            'M{XikY@C=EZJ=N|h<9vZ4U^eFO2YP9QG*?T4HY6<js!N2x<AeEF9MG%pK#9W7zsnq&w{^}&q2C$ZMXyza8(7QBM-'
            '5UFwjW`!jY*F$Xj)<_DYv=Z>f`BKd7@_-JzA~ewrB1AG7f~{_V(}5tUkG+G>6NiXJ-XC<IXn^4Ghos0$5ga!Ppnte36^'
            'QOYR!a(%vkFCNwgp<56v4})U*uqWJQm8uLEqAT>XPk&HCfB>)BX_Dayo>DEoQ`GX(kR|7=a%z-'
            '(l5}eQ0^04Q>Azq5B~(#--y)An<t-8(vrA$gNQ#b}j-Vbj-+;g43Ye_zu&O;-'
            'T!k7bv%`!zgiMd?lR0_Ww}?vYWKXuWLUT%U7qs!MUDrU+*j`vucS-^lFH1e@o+x<*?PK0{@C-'
            'LjZRY7V(ASzCDewX226uzQ1S3KJCLx2AU8loQ0A1;_2n7c)F=y5bVyzV4TJQ!t-rAelluC?u9K_l$Q=4-'
            'yMa85pED*_Jk@fY{yfrAL;-3-'
            '&=S#98F$yL1f`JEPmO`Haiy!tyN}F{h$tKPh_H;{WCfXedvqmU|dlrz^u03fm0Td_~x8182p(9+;jYK_AFgEcqsvv+z&'
            'y;G;<iZdKKGbKEo~HLuB5La9q@2g31pgDYq5}OOyOzFKZ=j^IU}?%q+;c#>;%YU?E0I&T#OBtMQmzFP=Fh0MgkW@I*Tq'
            'vIL_c;cPrDlUt9<yGt-KFBV2)14w(p)XeiXnloCAbw97+d|_wY@bDsNs!mg;SqXhrd=od{PNn?!Tk(P07TEpt3uEtBWz'
            'CfL0hI1h#GvVEaIgx6HxJU#ikp`vr`%&pl?}j&AEHceu3eaAs|2gpZ-mkzDIAw9hSp^gP&io*|8(b)_FtlS&Y>1|+%cw'
            '?rV_B!wvr4)KOtOOlEK7+;M@DRkiYjNZ41!G=l5^Y0`UY8wHjyi+s?(j`R8EyzZ9A!8G^Q^%gEa|Z(&&FC?4GWj@HKV('
            'z%j5srp532zk&3H7DZfsG<QrUn|YNE@Fk!w_alVyS?yxZwu)B<)@i58g<c@OY|A*E4iNRjQ;*R@To~9W}Ykr8{SB;5Ve'
            'KHUt&mb?H$OKjD)=rE2+fW?I3xz8+MtVL9d8Y^mJ@8;R_5v-x;p-T|x~-'
            'to5ri??sb$8I3TzxgMVA7E+eSdbr{=kM@19fbXsS)Jt(WyWv<exXXq?is}HqQK&}6R8C!0hYzaA8>3Cs2$8c7fuuS?I2'
            '7*(CMHMVv|<f@yA%k+((BP{@-'
            'X@v1c5`EDjKEq5u?uKaOqM3e2UXT^>vB3X();oK9hqr_W8(K*A6#$J#b%$0p6Q3hSSk==<&}diP=6&NP01Xi^hPY-'
            '{oR{2suDZ)@6f}oGfUbIE8Umoh0a;Ek+4O!5WDySoXyV{0;XJZrcjH|MD?>UE)J#2UjAi<2vqq>p+bK452IG5Jd3!qu$'
            'BQsO%Yx%hv6N-'
            '{&vU38or09y|;`6%(OtemUur&#Ws6Nu$=38y;D6fO+H^Zjouiw;RIPioPwhu_YcpXzSy(u3Y@^<s@TO^G-'
            'bcWe$8c{H?*a%@?}9l|V5mAltL&V)xK_ESR=~_Kils|H2!#JX=P{kL5sAT~PdN9OgV&f@YUz_+daMEnA&StnSV1(s_fp'
            'qFNiK7d$2HJL=$GQxX0C%M-'
            '%4ZNXcw1#rhpJq*2bi#Qtz(mk7YqAss5u5NOKH+(mUUFBOE>FP!jSERy1aRa<#bp@y9tOM~4cc8yt1B&#l=;43Wu=x8m'
            'w)KiIx?cAi{T-'
            'f!kG1Z?#27()Awx)sjKsgI%<%xH0`_{vpl3ZLew!|U+oC0Ke3mD?bI`)^&kWj;&;!rrW`o4;TD*Do4tQMR#nX*PVX{gH'
            'RprxZVx0$lu%`%L4F6<k+%CZ2#q(fG+C94F`z^38SE7Hy^=XcbF9U2jgk0GMzEMq(@Wz{QKF1fT4sIa!AA@k~>@{GR&W'
            'nqJW+A`t%=bSU!g!q?xb5tM)agvRD?Ec(IoBemDV=QFpG}VK_)1so^??iHU3A;@K@<qx0*Q+ziK=}*+#T2q)$t0H%Quf'
            '6()7cF8p7CPT7ic9JL=AcQM`6D9X21($5#tq(oGLDph71F)J`a{+hjk%pv@bqtuYTxoIFVQIU(5aTod<;=uxwfM#iS^z'
            'SuKdfjUcq;qlM=)O6Qj#zE5t@EhcVbL@@qxz&%lmKx#`|6lacqCBt}C<5E2Rq%pG4FuQBW(y=XG2{nZ$*KA3P`~sjO${'
            '+1QOY8)zU>j+$r=O4`!_Vjm#~3t*+5bXTp;pE3#|4p#ie(3z_=ox6s}4DJGUn&EW8)}%2T1*xCZ1_-'
            'lU_iim6I&8i_w}2ztI8g_VMSAk^~)T+b%q7O^UjJ^u(kDTgx@WAB3Nft_f0F$?CtKZ?<pgXs5=qxk(>H$K;UOT-KnVEF'
            'lW^0z4m7x%eCU10;d1igW+Xbp+EA0cm55)N-'
            '=z_^w(?0S+6j$;O7O2dRWbqV3pT_*4po5)#R4r*+iMFuQ0!QWe%ac@=>#x;9kU)eDEp43Ctvo&DPx)}7!?8K3C>w!PX6'
            'upc#f!(ueG&N1aKhB$p-0M;hmu<v*2SdT%b^)G^x~JKcbb@UZ(TEA1<)Co*AKfQ24|3B2pilfMRT62y8k-'
            '1w=`zYz{ZzvkTaksex|TE|Z6CBmvr(Zg2g2BKSoO#i6$MYAruju!ue6&neA1YF4fFw16@7X*t{t{SO|cg#RWLG4uF~1>'
            'WI<<PIqHsw!t$Vn5aOZ=u1i&cE254}=Pn{KpM!D74+U5?v#WO0%;K1S_{@HD)DhSC1c1Xh6Q4XrG}tH!gTW1?U$zK_gQ'
            'Z}@Ar&g#Q^vUd>Hdt;-++xVL)fTpO?E|8BOEATAJ`ay)1G<oRP`xz;(8da4<+X+-'
            '5@Pr6F=@fL*}3Mh8yEiaM@9kb~s#xeOyJ<Vn+~s7+i!6;-#Qfp~DEfnh4TnZ|Pa~O88!oi66~B;-'
            ';}NQtYV@$x=dSQLPUJyGF_B0yf%Y4&%D(lC-'
            '2K2$<m|_#pK%TzoN$`pRpAYtCM}{nC5XlV67)=JJB#{CJWVycX43caqK9yFr4P1a`&|sMgR6-'
            '}cu7ySWto@1DU4<C7?2>j4=X4x!I*EU-jfX|qTnwYQf5B@1p8I<k#)?mdBj_q!mgCj(Y!PqT&3>_JgpFAOfe2|FwL>5W'
            '<5_$kPY5t$l{C0i`Pz%T?4`3Dd_`+1PnH^k7M(MDpYYUr%_Nf5%W$IRW*;ADhEU{PgV@<Sa^?hJ%ws;9}T97!;luD~t4'
            'k{B<&1WxbkfFNciFnn@gz_c7*@$LrYZOzpD&RU2ltHABGEo8{g65h^<1+@*TWIYX`bsPVYRY9dtw4{?R+`JhJ!@uE)m!'
            'tSzbsl_D<b!~*Mv&g-2l_Ymp#0lI=)S)KgG-'
            'c9aO4Fc?<3hKXL+D^n=#n!zKNg4Y$>m5F>On#Cfi=b0avmTn6GamQm_zg7?#L$stP?cpVE>I=W$ts3pju;?%Hkv&Li{a'
            'rt#bG=c*{^dzUdDM<_zAg%ao;{z<PbK8tcsgCX(hD=HVZ2ks6p#_tNd(Eg(YJ-'
            '6;7BhuiPre@Cy;`MZf%RLFEq9?o{)c+XtXiGt;BZq82ZkYe@9_?79M>vJwKxg(7BH-qOn!P`0#bHYb+WQ3OTAzkJ3VzV'
            'VbA_!Mn@Vg>R^#&{O^i>XUMMzQ2YVcvz~hED&arevD<{Bn=L$fhHWQ-89zg<#lbq=w#)w=sE^tf-i-BM|ylcJYWcLQz9'
            '4?QmUu+`>ex%XC^cpbQv6S!^RA91L8MgXo;US4)`0ITf-'
            '~PS;?m}}JhnGdM1Kxa~#U+kxVZ9VmJrIsovkzj&oIYIZ)qz5D1<1<vFEzvhZsOS661tBk8>(y`Lczo<8u&9CJ?cfM?dm'
            'c-'
            '#@hsA#b$VpUx)a&6foQae4yiL89v&51b70?*~7zG^h(T5a%F!u91669QN<`&AybW9Vt(q{zu%H_|6!DMOGb~Hb?|bi2c'
            'wRZlbansNd4<mP!v)M8<phY^TV=Q#h4x<wl0qBQ7_iOspn{5{{YfD#qj#x{g56~hu^(}X_9KifAU{e7+yd5-'
            '}o;y{s;bpC%$ReJHuxNRmGSVe<*1?ERP?Y_QL6Y9xUDEPakC#0^?*7T@_=F&lXisxsh&6SUQeJrPtthjvoBoR|co}0he'
            'rTfp!%|ax}vN9hG0;!OPrm>eO6NN>rk)0S&OHZz)cFiGxK$R_yy_TjB8@7wi;y#rRMD$$<qr|KI+L`CNSeZ~U9Jw{vm1'
            'xh{w9t{h0QnFnoq;?eG58@QO?#%$M{$i6tA$lQv9ZEya9i=rb~JxZcHG2%GcEe^(08pt`Tjq@~*u;<6TB3WCD8G^-'
            '6NdMAr7#P$D%pwlxBsznceKEZuxecT5te_$lNl?A)HcGuR$I$vNoHw`hKlw$PmdyPB!auTk<Nx6PhCBmUD}TANeEr>69'
            'y~5Ak7eF0FC7mSSHsLZKPT4HSQplVXiwJG?Jlg-dt6vAWjt7U(T=SCz3wda1b3FSMj&hAf;X#!-'
            '<`Fh%bDd<?8=g9a%0)ac(HV@c(8U7cb0~NGb`23p4Fl2#Hum(VkM4wus-i}VJYnTzxgeJ#3t##!T-'
            'O=Z`yiPN7LM2i!IZyOP1W@V^qt=kUtM{$gW35G<9VX-R`*&zm^HmHqKh2ZXV5WICqG$7pu^JqGOE0oAK1h#1f-'
            'Jdr0W_G&+!Skui~%R<~u*aIKI;F55#sgmA1`Y%6zvDy-'
            'v7?XrW2c5Dc(yii#uxA_fKNN!>OAN<ytR*L@({{KaO)8ceb*2EnbR)DPsD_F&u#rxQURc!6fO26sE`mO5BDmCzDeO%$r'
            'dP_W5zt>)4O~zhh?NV@L9V~ER4SM>p)bc%880^kk(&@pf&~Rq4zPquO3%jtKQXN^xEnQj1mV2-+2|BX|i`-'
            'bT_b;&;^E_Djd!1P_+1FTq=l|dPH^?op`ET(5FY=q32bJ+uj}gs{=p#B@A&dxJSycJtO$+Xg(2@&kG;Q-'
            'O(7KyUN(>KSWz$|-'
            'pM939Fy%4xq7F@sYG61Ft%LJVI%)KVec+B6ninh*=v2EPdi9%ViZ9#8aN6*ZKJF~3+X8ls9k2Z8`{u<AH!W>kedIN@m0'
            '0<o`yYwX7yNJV|4;IpSRSmHaW~d}eh*fvy&KD-'
            '(vCI%=|xujW(U?}y90}7%N17OCl?kg!Hs2l>l!O%^a@M9+m*$8{xU27q$?|r>nbaEof|9X_+{3I5Ia_t_7#>viwEmA!-'
            '0hYS6GRf_N;InJC^t<XBM}+6YIe*JC<IuE33)pDr?rlYb@(1Z&sPjRo2~?R;+7IS6IC%t}L-'
            '>OO{Wb7YpuRWc~bN&MMTlW0l8UVSUTBX9@hg#_IUv$&xd6Vx3!T$6Cy9%PMT}V?9;Az^Zq3VcDm7vU1kAu#E0{vW{1Ju'
            'uhjavF<OhW%bU?vbi(!D52GpWhU*#ax8UWT^hXnpZ*(tCi0z&d#1$P|NrX$zsQej%Izdx&R-'
            'dKPj98``E_VjPb^7ybEiqVU8G7bloZ%Y(+j2|<i6Qu#;*;tX;@i9T_(RKb^eh{{=VdgjxHH`?RYU6d-'
            'sj4b?hBck_{kD5p%Fqsg``cdX*Tn{t^a%1d*#&g>FlG0wr0bG+?r3Nd(n+V`f2C`g~^G#z9i_!--'
            'CQ6Q^0?gS7FJhNfZ39BMjxlr$vj&@CD78O%H9$qV&|b*E!~vJJ!%*zLVB^yGA$W*jGn-QL~Ep3IJ*A5vBly=z+J+*l?1'
            '2d^yImvEAtdGn8b$ms+bWhyZE_RORG)s!^rFQh&Zc4T1hyZ`tnb!!f({5SRgU*uO6THl583sDef6b}a*SE8V>Bh(Hr!<'
            'LJ#jF9iicwy~UG!62Dy-'
            '$_lg60a2;eK(>wBHfR#k+#I>}z0j5Kr`(a}iuVn!uaqp=f+83m+_bg`8spsCaiHF>@)S6?U&ccXcQ3YEFQK^)dMJ$Z=>'
            '(ZUu5X0WN6`!Jfos>KWw)Ib-uUd{6F^*T0`)q?;ZDUC1W9w|wx2<QH7FT9D(MMd>l6EZlhEAo97ZV*VX&X0yOMkZ!vTy'
            'R|LwdY~*!hfWj0A9K)cl|ByhuVBhe{>Ab73z)m`4xFzzMhiBiFvLGsVW|3Ts2qrd{Gumt`{{j<8jP6HaoWT#uMLX-'
            '6u_^TGmvHy1f~ZkKyr>$T~}Wu>FFGWbvB~($?Z(MMe@;p>jU=KbQN9pV*o0|J~1Yg3*mTCIiBvUz}tT9#P)b32^{0&tl'
            'PR3HmmE?YbPS<V1F`=54!;uRDI#)$x0AB`~`kTb1~JY3-FcZBV5pZ56VrN;q+n$e0n(?wBoAa<Xjot;L-'
            '@~M*N_#vjVwid7--1W2#ZV1P*L{%+7wf5#rBS)ALRU6UVKIYMl-Q|7C+p?H34OMB@?K1-'
            '@&I;GK~&Jxe^m=7K88(w(ONiX8FBjzCP3nu2RCfH&J6pw;abhREiCqWpC<czuzq(3Rq7Ur;0NKhj`lS1n{*zl7g3{($+'
            'jK=jCF!N^D(-W;6GtV2o8&xhI=KY5nEP~v7LNr*86jwE4xdLFr+<BJA-MMRe@gS+$BgIa|pTj)^`&DyR+zTd6Evfd_0|'
            'Js00oLC?tC(QI-l8z_d@G(Qjlu-'
            'H2PJF8qh|hYWV03Q<T)KX2MtiPgw(HeETzd*8zLDW9E8NV~`L+%F6E>o3M;V+esAfby<mL!nzXI_mI^kKzJly<hDKuAy'
            'fu6Sv&dQG_3tQVDx&AEXcAfyn;V<-'
            '>O9;;XHHr~OGa%{qQ*=B~1Am4V0ViRc(aP6_CbA*m>J<Y9@fAeT`2o8%e;9RJvv752COrExfNT0PL2JST`wlbl)vUizs'
            'Yp@(jw<TibHy9KxS*kN1xkt}LAYQPNaf68Dui!`Q%zD#5q@zBAHqS#nE}ZT(u|~uA7CIhn_Yjc0nF;9utkeOvsc=}iv3'
            '&Au5mlVi%A#@B6~;-'
            'ZzDM@U5qtbtAH`_2I^PwV1=kZc{FkgXN4+&=9dt>88m>?Ri}XyvJa<2+o5@Z8^#<sgqw4bBzA11>z99`U#@s#aPLA0*m'
            'ECT`;%y%-T>KrF9mBvkCB8RNr-TjgJFR{<V`3g7j*-'
            'ndSf%TEv$eo7tX`l)Gzqq96$6K_~Va=Bu3c%SWNm=K^t>h;gM+)p1NNIpY9KU)UVyxyi5axvKL{buqIfK-'
            'G;vY6&!Ls7o()?G4IF@TIt!UDXFYPw%tfVE#F%7jFtv_uO5c-6?+t)f1f-'
            '){vK+cWiXcA`~Z87)o8wIe*~6&9rRX05pC_2Aa42<aPQh`MqjQ7{xh8f(U2O<dXxq9Q5p>2X$MugS`b@u2fFJ`fO};d5'
            '#2UUoe!me_>~3BJaIATADF|j{*r(ZogvV4f`R>6PT;Av2B!)IsOBsIj$&;XIriuc9CmvNRcVhXPuw4{pSX>(THmRrhcH'
            'uawFtJ}Z^G#dNemU)&2agU8@QU!A;O$U*d^TzLu&?TKfM6ooWH}BN*U(F<qC4%R*2CAyiEToTjF~;wJx=~0nHq~!0De+'
            '5M&mC9=0yTn!grbh|lmX^AYgi4Mu)-K`3^!1IL&oNQ>=)+Q}sxlf;cESLaEya(~hu-p_Q&+aAb?;$ePOFhjxPCAc6vA5'
            '=GpGJh7{fShW1=FalZaOC13oKIASM~q8wH!}e~bERSU-'
            '_?v8vu81>Mizb!z6uGC{(;%?&lp!Yg34Bk%*P$tWHdJxjoXLe!P(0iIlH&fO(rK%a>AIb6X)g_PJW~H%XuJOc{cN69T$'
            '`Z+y|v++*rn1j@x?GFe!Zzv!io1Gf#Imc{(8ilb6JpvY$3VcB}xbd(F>TF^369#y-L-'
            '2ZV2K@=SxB_h9OCBCag#Bb6ib{~y-'
            'AG_2<6?YdEEq9_qbl88haboPD7tVAM|21OJ_p$w6vd7cNFM`_l4&c07dr8G!I8A6mMq(mw7_&?v?Pw(|U@AF}Q-'
            'uK%3{;l6y*R`&D6Y)B4Oej}m%6{C$WJ~+u;;HMXCD2GG@-'
            ')dso=`lL)JW#P@<P#yKon+VQibdMxaHz5su|Y^6`g#{)qDQHqpAkdbU76Y9yQ>z-'
            'xO2ce}o2uCsfnM6VgPwf%n)pAUnfhb_a|6<?Kec*kQD0yF>cdFYsnG4Z7bfhO;fF7_DIgjHV;6XxLs6GVrnhM|I93acY'
            'AP21&@dH4i@Jf2K*s$B0^*6q?;o!^;QaV8>HIWC}e2^JBbB0Yghz^(_ne>w92i?^$><kA?epw86TczHqsCDO?sz!St3}'
            'V6yKg257cp(7-'
            'OV9@oW?h}ZD`fG=G$QivZ?&%vE17wJj6Cj5S&4Nu2ihOS+9P<mGfo=?g{LC6?<7gK^ib!)(Lz!wI7dyx9lAaob}h^fvA'
            'SlN6GOiqpijK<@H_X#+}EzF7DWkiQNo8VXL6Y_d}DfLoF1x$I0G9N0zDWa6oE_(rj#vei0c`j!Es|Pft;Uo0B2tnlcZa'
            'jVL8Qk3EMe8RlU~QlRogVr@Tb{p#hy*d*Ra*m>Z+^j<7cqEepcqDmVvuF)1K-u7K|yE^E-'
            'j9ye~i{4j|2n9_>}N+<yNrmCYbeN4_Ww(l0C}f?26M7M6vJ+Y&&xVbGDbG-'
            'O~Ul&)tFjM33#7&x6B%zQ8=GR(R4r3)Giy1-01~5FX!8xb|(piPJ4OxlxcaqxBo)Q+PRZ4jhNQyyx-hoA>m(w<iwg@xg'
            'q%RM1P0!h^X+AawF3aq9^})kGmQa5#tTSVg+ha|uf3a4}VeR6u0$PgdB(7cBN;U{MqYx%GKT!q{taM(_*LQa)y`cL+L+'
            '@1gQ01;|(Wf@E)8gEuwW**BF_Kx5hwbWQak^+OE(A>NIuVVS5g=Mud9ji~6u#~d793GrfA@b39AAa1*Xu~{A)Yeks>TN'
            'rS-ZZH1l9jeH&*RY;@BX&O?M1IS=z<jR`L+|5oVqFy(Kc1kac)JZ<|7oaD<CpNIJ`kS=hcPBp_LJK8x#U{&Q);du$RT!'
            '5AaC;m=Hg5th(6m6p_zqH{wo(t=iDY1M{^-Rv>XrL3&n{}JDmTUK|2C3!9Kg?xS{Ahl(=WXrb1O*<`94#G6f(M6aga-'
            'v#6h>FuF9#W5lvC?7BV<DYHFc-nV4BKJDNp>CY#K+RaD6!+#q@_9c@wXJ4b23qirvg6vQ4o}%poNxWHHKzuU|!O=JnM>'
            '?*cd7L)tk8C2Y3iWt-V-'
            '9=7uN(*SrI5cxl;iEVi<rfV<5b2ZHin%B>7+SiZh8pny7bcN1F5K=KhCIoI!VRNiy=bM7`Hh*CZ&Du@btwNIC`?3?EAE'
            'c?DM$=J9lhi>Fm!Y+k_qHnZiwQ@j)gGy&Wbk4;(Q!riv`iiNsY-'
            'S@=q=7mGeLQ*EKSOz%?wYV+oTuYVoY?#d_D(Ko<6;W5lf&;YB}zi?&IBn}?$q{oNTFpQqTI+a!&Gm0gbS*G}7Srux`N+'
            '&8U6_{_?j%NROc|4edJJxp4m+8Y$dQ}L_R`-D4VLpb`^&onXeGk0_OyJ3I0c@MI7~HPPGZgw&agsj^)4%j-'
            '^PhAD)7oj`xKkM4%&J7eWC!A6;t!#gp2#ILO>!U_vp%>`#g^~vQx~<+QT{c&J~2j&$FvyY7DD9rl~)*1FTlA!B1f#}@@'
            'Pk>ZY4MTum0m3Lpau@M!72LU|pgg693cicHTHL71+>_YrvSbF%UFDAEQThAFGU|jjqq^L8CI4oZ~HK$6x9O<+64{zEwf'
            '+vP4|^EDu*Hd`8dgZ+N*n9EBqi@q=X+Ww)P#*N*jMuIM8?eba%Q4A6z8!3tnF%GUn4NE;u9wZTA;58S)ljo}s?QtHqPK'
            'c3XVcJ(M+k+2kB1nEE|gNY9Zg22eVn2P)u!=YehNN@6j^)7BG`&I{Z)*Pffo{4ZpQW`R?8gWr;J;Ve&fS~-'
            '9#CdHz!<q94jaC*xN~{$gTPMlbSeFK?q{m@&AQz%rQsBdt34F$%MBWFyC1P&1)IG5Q<UQ|zeQ`I5URjIl>(kME=nYhiD'
            'uQ{e7nWOX!O_z}@I!Y5dX&5|&H6gb{qg{Q2o~bcQ#w$+egcE0jlu7i1IjFEfq9mK%<cQT$f>6x@XfFXGK%d;Zl4)M>~_'
            'S(eY;TC@D4<4RKk&uYIyz7Ck#C02nzFF;M3L=T;q}p%Ti;B^-MZ0Dzt^#wji?UxHneE&cf3X{{Q%kQmrK>&=wI2-'
            '7np+##NN5$4bH&+Ys`32M_acfiVX7^OFiaJGzMGv0uHKAbY1q;6iQ@$iE1HZ1Y9}$8@pl<4r2;@gCmI&cf|S&oJaJJ*1'
            '6aK49}9K2CIWA)3`hQmr5<Z1fyM@eKyp;b2W-'
            '2cA;pWEY&OUJM&AE93cduSrze3_WVsjNS!8@T|<XVwXfW#yF}0YlMYU^AJdQC5*fi1Fw}5)VFLNjx4>(QXlA|MCvK{Mr'
            'GpeTia1#^M0~wUmba#okaMz#>4yk5@@?=i3bLn@swZ#J+0z|&O0*DNH30*wwI%Qdlmh(zZpXYGN9#Q4!+Eo%RKz$Hyvw'
            'f#4_8x7#r~kcWS*v%`9PzKXng9zbAqAlMwPpP!vp~WDpciLsoVHY`822Zx64<wr3+CJM;nF!dG#|L+*harv_x}!Xc(=E'
            'ljtWLUUyv$_H*{H|$Y?r1O=e>gymH{^`Pfm!eT&n-QSuUy^eEEu9%{#0ZWQsO1=giuH1I;R^=#L^df??jS=4nt-zk@ea'
            'F?mRoT#GoAdAUa(-'
            'Pe&0d5c>CbVuUOclDZuG?tB>rl2GBpw4KBepn45bGeYx7`mh5?)+dnTcUab`X@39aNZ;ZzcOWwd;kveP%=fdw@Z_)4@K'
            'OQmwAX~?%^ZPD*wYL{Tc4gqoTjwBd;Y08`atzIsf6?>JyqsH!Gaz`Y2ZruGgu$u+$}#Rhn-'
            'e|6&SN$ekSZnD{Ey*ppKq8Si<q;!9iNt!!sY=Urk1cea!sV5@u~snaF$^2`znoIlLxW2Yzz{{V^GUR0=FzuB8#}cVx6)'
            'N^cCoX2uQ(b5<fHGgYc$HdrGiB#1$0mV!(8HK4kqhV4%nspvyC;uet}!t5FBH!3<3K>joCg)$C(RY@$<q4@%5TanIv#_'
            '`(%np+Nz>o~%US_xHi?+XgtuD~nTZl<xh9=W$pCC>=+3f@~gn(J#Ys44zG!T~9)hwK?6j#tJ+B{=vsXt(aBv3mO{Y@Rg'
            'GSlyrKcdHxu+xa)@RBKv9h2@z(n?ov)<h7(r(ErJ$}ERwnPFu03*LF#|<JpX!q-'
            'bw}J?ucQJ>B!+{Ndq*u{sws>q3}zf7-mNVz#NM@tkLMl^#<8Ey!|;i9tZ^Mn@RY5%TKg_I~P-S`GC4yAc@E^z!;-'
            'a92^RS$3r_%jnRtJaqGa{q6g1LJqMXoLFUERgdS`2f!aU`=9y6=Y&@9{yqZH0Zl#MAuZ|F#F;Vcl7lE?;Q;gqI4al@##'
            '9Va$8^%A$tGM5>0-'
            'ntb!?K)yh}$*_2?kSm`MfxY99YDh{p3BI+n59e7K@pOPgh{LCO>CC*AmXQYGHh^R+ZD}unr<hgqYSI3$gOH4^f(Y4Yp_'
            'Gn8hBGcsM5$d-~FdK!g*nIHJy+SvHrkHYygU?wx`SuBn(;Z;!j1m|&uQ9R}^YFhOq(^r(MktiF(l^P07BvM>*8-'
            '7DaY@)erm)CW!}OPT5Ois53&Y|c5Ag;17NsJ*q!9BX@m*cr=(nD?LDMQZR1ZTnY%^!h4PKJ%1VYZb$nyZlVGfBi8bQvz'
            '|nMVZE8q2$x<7kKBUC^_c75WZN~KrdSgj-5LJ?_P-D*_cHzwYm<AlUK0s_t=BstwB-'
            '~@CG7I%L9?)<tTFPf!nOR4F029*vcry%A=K_n8ATC-'
            '9mVCxC54+;9>H$=b%J_A*?Waj~P2kNOy4v&Zz&yj~55PSoIoST+xW#`K!nREkj0Tb|lJ$cwny*g^D%lz)U09CtMDH@B9'
            'U~qi;}s$^pf$cwx2oVf-AT0QpBlNQlK9P~{q=etWY>yl6EeR#+eMO-it4nFA&kRm0h~Bly)mjq-'
            '^KGTZO`^HA%gnN!P#vE?Q|4j%r5)+3+bnnx2H{8@tgT0LM!{wvN;m4(CzXPAm!O<N8*L)eJXKOW*{{@i~HPxjuyZo_Br'
            '0bY{1=bEvAKOYtrzQ?<_4M<h-'
            '2eM>d3wUl`MeOB1K=)q_Fp%zGR|=@W;fO>U?0g?tzc=6q{ddGX^Bqz28DLxPd<HWs=5wYJBZ(38P%qvoSbXOzJ>e38=Y'
            'N}m*|{>%Y<fxUo+Bh1jl$b<Zf5?UC1AKQk)G}MK|SBK_$=22#kA%^uTwU?q^`w0P+f@n-E%qBY7-'
            'D~U=e42vK(lg9S47tP57&93#J$eag?(6L9dqxF5jsRnI?_kARUP2>aRgWHW+)HAEMUZe)y3o@{dcq=!@@8MB0E0at~a>'
            'pNm3JZCwQ3$_phQq};%|Gz)D+@6n5P_E5Qw(u!_>JXux%`*vr8pnD;VR)1nh)cL{m2{x_c9ffSMt1#apn|hmA;wje*NY'
            'ApuGy8Z!`e6`eKsGGR6GxTnC*d*o1=w6B1VVpW@xHS##+BD0IBY{5k3V>!_C1XiLI(Se3u+i~7|gpX$%W*)iqSV8K*vr'
            'MO274x4>3cG_FfC>v2_p59vuL^L(l0}PlD~S?@3GPmVf@J7{}bdW0Z$K^0VU6HT43Wog#_9`Og0H81L}ui$}28>4DbIg'
            'TC6{ar4om?+)E?RSM$QrIV6mC0eO2jSCYxvG`3M^>SWKD%CZJa!5H|*ia59BLp}*BY)v+aRv>!dy~%l+l9M^?}3MMKAd'
            '`5kIvCI8RPGU;i(o6N2x@NQ|YgduTt0IjRsSCIaQtBFMS8@2X4Vaul-'
            '=Nx0d!cZO0kW`)F+b9u~I8fZj@FM$F22@KIEhUNkYr_r*WZe%t`JOS)jowki~FEyHVmGnh0}&9J&R3;Esc$*mRKkXy16'
            'tvt-'
            '}=c`&wyzcZ*f6wNGX8poH$Avi?aVv0VrxLkXdyG__gMxcfQ0T!Pa{TK)IQ@D87`>KMz=@k_7aoQWY!8tbWh0Pce4%e&j'
            '^Kuz6qqVu!S|^(pt$=cSY#MMO@A}CztCcDo$Uq&`R*|9+&HVVm4hZR>iAbw3OeX7s`_FMr<mam(KaItQ;|*_&wdISMlU'
            'eErU$MrUk!)G7ozh=1dG|9;QUDumX1UeI%?VDl`<A}Yt8}r%y6jLvk*pqO|#`9>@jX}G+ImwG8=Mq82dkd!m1oE>N?{F'
            'OQK!T)_#yoX8mF7jtqgAuqG~BfplU&6D`&z!2G8*aL@iJ-'
            'N31*9|Cv73XOPVST2Evx;MBoE($9O(y%c89?@KK5soIlqU321j1BQJ^SwUM)oUnSwp1VIHq6J5l3B?0P6t@L@<d+U7nF'
            'VY(2R!<XFq<1d*=*O`$0vrvXqaLp*xp(^n*8WO4XS`apI8L@EB&kZ^Q5FEl8e4Jm_nsAxE1FCFJvAkwqXXjC-QWnrO<N'
            ';{)eKgm7;NFX1mT!)}d_;MyohcGO&h0D6eDFStzW*Dc~)<B?~+xZ6&15)nU~<;RCH_OMaO8O}InF?`lrz!j@$h*(??wl'
            '20bGqMeImL|eT_7wD5uf_1ZTy|5UDSmr;0<{l!(82S$=y_icNA>zKfNwDftiQ;<JEsu5CnV8c`Yn3hNx^T%hBT*dH8$V'
            ')4R6$hm?}&Ap{aKo!>Iu^uMNk*KR3wkqCgCmPs6ElHyCDa#mU`ixK95Lrd^!R;g4#El+7EooeMjm>RK8)9}0pV_lFqqs'
            'Rr`DH<SK%#t^<*6~?b*!MTPR2(zz%`H3==bKIBhGnxi(zdc6>rE6@}ELkXMu!Eh6&QM|g5sQabVegPML(=aCrsrzHU13'
            'CfhA&P|-'
            '@*9x@6r0CEyOtI!CsFI5G>n={3oMu&7=zCUH?Vj2>arV%>YV0UZ~y2)>h3KLJfO2G@<ih@`w=DZoQA?ZQLC5tv?{4zXG'
            'o072unR+qlYE7p}g&LFbxnLO-uaTzcgd4b5`EUxJVE=bTHpKgk{17s;c=pHCpMzZI6u`NMAFlZW>PVc4ZsLhhuk#uGsT'
            'Of~CXTvRX%D+4`I;bj#0kn|39n|*LFv>*KngV}FvK7xVjEl`W-'
            'g5cDp@WrSRlrHDMn@z(Q(Af`q{k;%<cox_ko@7TTT>^%&3_6_FWNIw$p`!a(<ldzmvgc<qWB)q}oLP%Fw?~aY>+MVYxR'
            '{^WnB>P8muUuWMl?i<iKAq4Ci3ccf!Ved_;_m*SwGJZ-xd5M4uK}9{`@Q$bjaYz7cC%A{|UF7bb`vN2u56a5jb6)jWHd'
            'O@Vq>oSf2DH%ehBE*y}Z#^0>j9U5n||CoVF~rGj>01rTtqkNo+tjGAWrq~>PH(9x9#XFR<~V7oFj9XL)^_SJ#FkL%d}^'
            '#>TR$D!kSJp9mS;O{{tI;ZRwsa)xb)>3KUp>~OiUUO$u=Dxwb=}(}@$p&@*Ud8!KcH_;leXt<$Gm#yuqD_}r7!;mMgG$'
            'nnUj9L6^yg5Wvz@qS2OHGwdthQ+1Y>DJ9S)ERlEzy_*BKPyqn;|9KfD$Hl4qz?%SRa#QE+hl87kZK!Jq0|5Wl$|1bFhG'
            'XTb)zcYvGY{BZ{6WZa`;i`(#{t}g`a4WRz(LNR`uKfe6K13ce;U?=6r`B#20RtBD>)~xUNXkr9T1ZBb<H8~utK8H82@-'
            'R~aA7M|Y4b)sY#+FP|!Hcrl<Zs(bTEa=f)cBL=oG^eMrxW3nF%$1b&4Jj0XSjtmiW8AxxVwRiS!`5?KG&kaOraWb&$|*'
            'gVK>O+c1H<OUhGW!gv+>PXo03BERCCuTlt=mqp$Zs>Z8x_`8l$utP5fCLIu1iw?W}k55UrSHJsXd8J$+hQp2hQh|P6?<'
            '4JY&?~_z=_JS^LC`qPA3hrXy_e#vlV$(9sS+L8%k!bKgN29th6k&#gu|hJ)X1+zc`F1F=eh@78RiQcDhXK{s;Im2#!v7'
            'lKtStkuba5lT7@*J+%wqei?!~M}Y}A@BkFrmUaAYI{;$yU+=T8VE59p!FqZn}Y55%|11L$bE1(xXB1CLG`L@W@-'
            'd7I84_tL|VP_`H!bQ~w`RV6UkV1QNbt|*^0hgo;$GS#cg!rGdA;%F@aEvlz*zKje$JS4(#b-6?IB#$v-'
            'xxMLxBtKJ3p@_Z{jRfiRU1+l36HhN@qoBDrtv36D`TfK2xKtM|#3qp|x-Zb8GY+MG32-'
            'ii^D&Q=Ud1c>OCa675emb%W9F4oV);;$dWJ5*n)CH=oSBH?wRUiS${Tu?tYuzZ{uVy(Tf_<9*aZ(;c$kT;_ORynQ*DjT'
            'MNBKn9Af`Sm?Qh?4Qz`Q!RmZh<a-bSN2ML%$=npUU(m#OD(8VFZtFl|c`Vo-ug5iRLP#(g4X^fr4r3LWoij?4$N!zvhi'
            'd44xf?HgiE*3@xIw2Z5YDa$gRI`&^vfS5y!Pce@SI+aXXZtLW?d~iSMo0z&bW>7jrWPxxqJvumnXTGx#$leA!d~NEztA'
            '73-w%n<ifwZ^4Z)#7a!@v<EMCVxkelIKP`ZD3j;u@_Y+v2m&W(^GU4$fPh`!%jo*hxQR~bt42bqcC!Jc@!%$>uzbr-'
            'Z2TK`IIW@o|HiArCL8L;YU`bdo9O-I7CtEJ8W$MD5`32B2(@9xH-r$v{h$*KS&;ru9_|gP4DM=!0M+bi6_k@QZ-qAGb0'
            '!3{fVg7*~|DC_<jz4vjOOA`{KkNS`fAGICzj}$SF2hsk4qA?<;o1f(+QQ1Fx=}gUzh^0D)SwK5(i({6MMaS6lA-'
            'P0!qi+<2_GsOp)qZvakh@Qt%I9$<ns#7rI0WPu~&k&%GdPC&M;_g=|V%Fzc}!y2<24T@Pmmy9OQcr$(-'
            'LH*)E0mu9={ljn04PN00VI{QaNgkN!92SFew+rJ{4TA-'
            '}+JbUSb!E<O{3;!1b2W0er+mDWQ>)I3wv3+9G72_@|E_YBb4@s^x_a~m0tXT#UWj~J3oHf)nSU&-'
            'H19vHdS2v#;MU@wtBf!G~JR$l2L3w;++_A@4&{dNlP=f5IzXM2<7C+z9t>3&xC*gX0l{2zln3;rkg|2Oipx3B&A?;51_'
            'gnR#O%*FSA$*^wuKWo^yb1V7(Dj}G09hQHtN71-WhIctXxxa)-'
            'UM3fz$%!Id^HLO}Cd9CQ`AtwdAdGiPn(;_w4VK7PprJ-'
            '6ZhFqed~vA;9HjbS=>7}X(k6(ExQmcCPZ<YB5}+u&8m<n0Cvr~$$*Bl#j(FH!96gi5#ubL}O0$tsJNG9^)wIO^Z#k&KV'
            'S}f)6IgBRMXS|i#8p*@({dyQ&l@W<)+)Xt`99X<56=Romr6WL9T_FFg7$%}$OC+rG0OIaNix4loM|JNhxyY6C~*@(?#^'
            'ylo^}Jz%_>F1VJ7@kea%QXR0%JKw$c7nTeM(`68@SbeEU;_!`H0^EYtb$_uLWqoRorYo3_L52z_cA{{#fw`N3+zGSDwi'
            '#8A?RGV5EQEVvGe_OWS_`$jldF#69sKcEt0nZ%9f9c&ZiXWp@%gw%*UAgOh@;R+Aa@xV3s#>ES7dxA;Y+cI#NBa27xE7'
            'B~NZ#2c84_M*JVEybcX%UiyeLvbr{GAG#_;oG@uG)&hx<UBl?{_?6=t6DY27^n6AH8`ln4H~z2?OPu$u}EW&U5y2XkAn'
            'X_YJ?0)+jxqPEE+7$d_1zFF^k#4=zw#j2YU=FthI-'
            '0X=T0Go8hDSha;_D?g%jl2=*#nNf^^AG4qx+R!pt1%{?mVEole5KX8DQW#4FuKB>gmlF86B@tsziO>uDgV^0G3s3sC)9'
            'F`s+ULZsKmd;vlh1WGcI^>@jbaZ;jPQ5z($ftUc4yH%wJT)6+Z8{i>VjPWB|h2>RGupm^Wzn8hV)R`Xl`&@WeNg$T%4&'
            '2Px|0_3_e<Y8jtV230I$2W9>EzJSMlEV}0a4x<@5rXfPkAbRe2k1(st!cPg5#)j}@DYb@SnKtHj>I7Yo;*f^gbJ2{W2)'
            'u&!;;)w@=#`6$!JroqIkXH3H(X=jaJnL3Xe{;(bgFi<2T*DW%GDOhiwj0hH-Hv6FZZNHLlAh4f#Ycm~=<t-'
            'Ev*D*H_Oa`5%h5)3kl#Ud_L-'
            'm^>lq37u1uYi9f05LHYr@2535`bz~ky?km90;zt5c1PRUD0SN{aE@YDy)QMiTQehEXHi#W5%mLJ}7uf&zZy?Avv29JL!'
            'gQAZ~pgXz`M6}b$tgfZ7bTkkO67tFNisP8DG6iL8V)4kBBiu|MA#yVhLA<RW{0>NC`*;=_TG&&M`UGwN{!8p-'
            '`rdT2RxOFnYlr5{91zmWgfbmdqBnX2&RMp=?C4p{LXFv+vmSL|zit=4I$H$|<w~GAWreo|qQHd&V!Ej%)SXJFB_@5qWp'
            '50u+bXzMfe*E2VyVEPM23Ek1yIjzxXH90ctZ!lfXaZV%y}Bt=|e64BICyM1bBSP6t0<u!gV!sRPp@|5yOj^CpM{}ZM_J'
            'dkMqP)rYn@r`btwaFd;K%A^c_?#qVz;A!^n!$VuaaE%U3PN$(Z-UrK@0SaI-IK1CGr?!b~~dqKrDkY=3hL!)XdGKMR_<'
            'I=zWOjM?P{wIi9RtufJ@QBsmHiWz<Tv(RpzS5}GyWs3uWuhzM0m*Zsp?J~;-'
            '7gnXmUuMWPdY`ytKz`5As^OSTZ6^*0d3(URqSU|5uo2yiX}?{Y*+ch<UMz~v1*2)WDvx#@p}ZW4}4Mkm;^`j>RWtu@Sn'
            'aHk3n&paq9ce@4RVe)2%zgF-@%*ukmlEWg*Tepl64v(F-'
            '`1e7X2+Ng$TQ%mTxnCepLWg0Z7Y8~5fZQ*EIXXmpQ*Lz~^PDaHj`B^BY1$6~Nj?W2ZQv$0k42+=p(M))o(YMXxgi!0YJ'
            'gzDRVC?qJvDUUcr-ULpg>_c9TNoEK}To&XkGntQ!<3eciod;)cT+TUTy%g+RS7Yz94Pdxw5yU+ZU}))eFbXf;h5?&hpx'
            'L(<0(TvxTQml6j&vKu?2CoF(>Z9X>Iu4Ut8lT<3Fyp@A_~(b`1Q|b;v~2X9O-fV#w(0-'
            'CKluFHTGCw#{y;UXk@M404|F|vAEzNn#cOWkcclFU3eDOPnI%lhRgAw#zy>TypnyoF_R2^QN}cl1nN=63%xHbu(R$Iz7'
            'XC;)$TVFeqMf5zV;k{tbUAKCSTF?Hxfm~Z{*UaYhb@K2!vl|L$&%2xWLlJ{N4N<-'
            'koe>=rKv+O}yZ3M<4zfwIOpmu7izn3OSmz6?_M$=-FclctX<}*2WG2*X3)(Q>=$xc+w1-'
            'xkIdpgGZnsB8iHB`p9UT6$0&D+i=$U2smUM2)1QQq5T>+w&pRxaegU%^x`Gs{^oR0mU5@wVUl1`KEsxXXos`r3!$Xi0)'
            '9RUWGvynL}iw}g!5cm*vn$OiC{<~++DmGax%goR#gR$E_TH63q53MM;eX>9-'
            '^BVlkiSB8#>tg;jKnEE))nu9rdTgNJoI$o7BOIR~hikpP#HXXu#WZSoHbU9vs&Sho26&F=}TmeiYvcDzjW6?Y<r{ju}E'
            'PsYAHhvH+Y8X5s8Tf7us|M_?+@A2UA2qf4$X`c1FEvAj0AVa;s3VEm-wE8CKx<R5@8V)^*^jV?-'
            'kdkn0yVCdWy2yPiGptD?_QML3nIT_jjHCrFzV?PaewPQW_<ozOz?o*_+Tm*_DBXC-'
            '37VUXE%w8#S1%D0N(aLYPVE0{J3|;DpW-bO0G?Gr%O@|<Nz*1D`7s3ehXgIQZ6`JX81=`7B)Nx(HWSL!{q&y4z)~2y5M'
            '2avf^8xy9`wex+$7$C*HsOoR!9l4x(BRpRw?$lFD>E6tEa@h<-P7qsCO1%%ej@NfmUFqe2-'
            '>Gx&?E369k;Z{Gs@vGy73o@35bFhqYvR^l^&W@as!9A6EAFSA?FjU*k!e2BvHEoE_^P4ru7z#C!Jf-'
            '*RqSgl97OWW?mSabpk!=xH(Rqr{RpQD|OlT5tG!OlGY{RM8TbdjOyj^>UJpTY?VMfX@gtmHlXUdNE|RsLWR>_Aa8nsCa'
            'zz~SpYt8r!5(39eQzb@dbv}dnJy1t2aywc%t|DHB9!4O8V}|7;*OEMGTU}NAe<IxOJKon5{>LB60S++DbgUZ5^3gUyR<'
            'fQ}E(4Nlx?`9**6jcl5I2PRt@)9NPo8(Xpk2q<7VlpDh21<XL*~;rM&XHCq}nXOwMbxd!GN4TC61f`a%YZ48m%(9e@3C'
            '^H_j)+oRY;XUXk<ADoqmXqOz8no~9!9zDA@X3$c*e&0PtY_}n@4XxL?`$N>Ix@(8Ob2W9ohiS`P3-'
            ';hjBLKn!W~>4c=5m)I3D*6um6~UqdltNeCjNY6j#HBT|8KA+CoK|x@o9r0BO_JV8|A{rdzuxyVZ9oT5U+ePw#iqQ~Tah'
            '(Nb<CYkz{WbRqkCZVJYlcw<}APVjx>37WH#A!_mxhNS3WnKJ{Pgge1Ak8}7qB>>+?8=^<MCK3$<v9*t3(;5kSg!p29CO'
            '2GXo(J0psZg?GEmoDu!(`ifxGRUaa2W@aJ8FrL8ep~Rbrdl-LG$!PkXKZ|sR;(iJ3mG5GDnaQT8M({nt}0fGiA;f#LhD'
            '-$R&wjSVP+3CSN?bua_k!tUuE=Ra~&CFdK6$uTcNDBhalM3R`BK!0A+fU~$~(x!=-'
            'kq1qg{b^HvbZBa$8)HeEbT#3r~q`;F)iNKc-'
            '$Y?Emjm^d9fM?4+_<RB(;)gVxX57a6R=E(s+J~v$8VpDI1B?qr*RVaUiXqLPiuyU7WS-q>IJUbUDXSWyW#aJL-'
            'Vt)md^dXBh=n=#{isIQH2og74o?*ELSsb-HGIbl%a%)UuGl0KjGoJJ@M$6LfjTJZvkbi@q|of2ZqoPfA%2=pq%n#Yh8y'
            ';R^UD=*ZoWBWb`H?HV@hnZwZZsUi=X)}zXjBc6|pyb3$CqP1KW;A;m;BiP`R=dT=zFZvB5Kpb~?x~vU6u0HqmApYoy`V'
            'bpd$nXE?E5&ZHu`-'
            '^g_RV$RHgHdymjmog7lZHhP?4f$JyXv$zZ25ai0j9Ui!9%q8W+bnEZuo{(mS?r}|p}?4Krc0_D@xjU>wk_8-'
            '_8&t(_6s_Py~jQQhCG(BL{hn!Z!CjBi1U{uFEVC#XS`#?HFl$^av59R%Nr`oZqd>?LKsqU51I1jXg!ny9lBktm5<-'
            'U_YOU98*~TX@FpthwuNd6kD-CV1UcHu2iGn-lH|!8dW}1kN-'
            'o{V^tAp$KYUInJA3sAt0<NjoUzA%+7~c3;0}+c1Tb>HKR&dp#wu1OD(iZ~ZWlz|tTY&ElcO3VUkG1VHdtH<L}|%L$S)c'
            '~dp=%HjOr|ocylnQsaC<)7x!^hmL{&Sx1&;r?_#HL7FL}ZU<YQ0fZMDpFeCL4Cb|joT0+QGxvgaDwmI<3QXY4`D*;oh1'
            '?XP(3OVhoAd|fjE>`A3_#X@MfO8KXe2l@qe|j~{pMjzE1no;cQ`K!vv`bK)u{<LeYYc-'
            'h?XoQApm8#4#jU}^%BjR%;|m#_+z3fd4=_hvi0P8J3nmgY7&-'
            '}OQE9U(qml?=*ONi=z41P5SXGT;%QVUKqjB*1o(9tQo2iq61NyJp2b*T)kVv~O81-'
            'ApS$j|pAKY<8*=2IbcAkfk8iOb|-v~0UyuoGtTTtFL6elm);NgU7v{NxB75eL-&&{1|5cmp5^zuM4ekB-'
            'a9tPd>!uaRyHmY|Z7Qejo0;bOcsNC@sn|Bz2y8}PwZoY<OT^g9hu7<>lwT!!!`?U*~`jX}e6_DH=fvwST<ht`~5}0X^m'
            'lld~y!dZp;qiy?urUz)&6eO^p9g4qG9GQZ%JJs9V&E#TWiRFp0gkN=C<|XFMx|j?p}3i38Fs_I@;h{UQ#m_<y$r&39HS'
            'd&<pP{hh5X}YjPl>rtS<xQWKGar4E`>HeVabPwuWvDY;{8UV-'
            '|4zt0H`U_!|5!@I#lkBFOR1!Lm)fs9Lm!y|EyL<csV8)2d=ne-c42SapNI^_Ac;sSRznWYFc93zSs{!5--_SZch6TrP~'
            'nx7=6p%BEsqSv|m{!Q*J95KEm8%b~~~E@t<dix{-111`SdCC7fVs8fFs;lA>U?Y%P(V%Gm**-'
            'k`Jsd`JiX;}uwKWpKTwF5c0H4UC_1$bF0&rFY9$m~yDg)6M;ahF{YBzm*xS4Tg{%@rW}jF(WYxf=|>Gw>w%;z?Ez&P)7'
            'AW-%{9)=_(4igm$<9f24U9|;$H(@3mXAbAp_feB02vbs0&6CG-Yd4t@@@b!aRjrq9zwkRDRzCzsMgUQb#9{RQA7j3mmg'
            '%@|EvBNcn28r4s&t(Zbw%!GE25x}jql>WMfG`!mv<=oRv&V$Sba<${k|yM_agUiV{N*ym3c1BF(kcszS+T$?>r;Mrxg4'
            'ok@`V~#TVc6ZAl_7UVuiMx#(RnB7!hoSlUZ3{owxzteyPIEZh7RCb|<~~XFYX}z78P0AC5Wd!%S&3e4n=m9cBRp6!^ls'
            'LT+g8V*>9LC;ax+4VN!c#)Dk#)UrzqKJ%DDRZtzW`gF)Cl^m?4Qn(|q19z@{NUk1yL)LRY(8lAL@Cxrj(=%UqX=4q$uN'
            '|Xr@|sXm<Pz>4<zaGh?WQG){$vw(F6eH&iRwI`$yS*kRPtpHnKl0z?!O+6ae|{5>Cr_O2>Ro)hNrm0%pY>xc9A&ATJVE'
            '6xN$%XMgnV4YNQIZ!m}Z!b}uk~F;VfYCxzhS<m>QX#!YmDMe^V2(_4?Rd&~=WT@DADuo~^wH$p;wCaJ%00nGV2@mseph'
            'C3&t;&?m#9Jq&dMIo4Y9SOzc7&~zCX~jFJgr5VY!Q@RF)HozV!bAjY-'
            'E<w3!nIH@MTm3h4lmj08jtsymeE6Xg3Ml~AiRi5Co?IzG_yYvPjGd>$(=0PHFbuzhPl(l5izKDCJ$sve$r3-v(U-'
            'oI_>KH@{bqU&_1>w4#>Gd{QC8fGSCeYmGdyzO$RQ%zN)QW^9=*U44I3t2!sk9Xd_k<<EyP?xqpxLCz<UP^K1f8z#@V!$'
            'o>QqXWGI3_epql{2E*5K>+OM-$08N9EWe>zUcJv0f@_XlI@S~Lhagnc=d%5>c}p@-'
            '(0i6Pv#!1^Ers`dBixsKUmVj%Tq*fjV73`y9)^)^5|pHX>ec5hQq?85ZUmaq+Kxwm!smafh7$oJ7b{0Lxo0~zCinB1@M'
            'qDM)i+6(+yVlkhA|2Id!$09Tu8Nl@DBij{6eK8C0T6YURM~1e?UKHb?yr;pEijuW&Uo5RPZL<6ptQc=!PyI!d^}yf+be'
            ';_y7?i|;LHXmta6Z`q@*L=*Y&QXID_JffK#J$%IB<-'
            '{Fi;;Mv7ka2B67PmhreA`H;$_z=_=uaZOWiD{5nBtV=5PP|LJvKe~j32LiqpEBX$(Z#NV$a3E6~io&6=#GqGAnWN$Xu#'
            'oWDUpWj*t$E(@<aF10$|U@a^X=#ya^q9M(}EeEvp>O!d2??!{U>lc<RIgJ*MsT8)r%$c=Pae!+o-'
            'r=a0k2DMQe#HRNLvC_GM!L^v+b*w~F!C*+-eh%}-'
            'O7X<tF5uqNODDQYs7!4*vQ}J#mUH1C=6@O0|E3`~&rvYWI0!~oE~ImeVC{|tctBE+Ii)in1;m=MtaBQNvn!}^QZPO_dj'
            '-??L}Np~5=w4)M7AXSgd{r;JmnOECswjRg{z-Ru4pC@AWA}-'
            'Zjdyu<K*SFPjsn`59W4j;E(n`OgO%uSf{e_=9nhzI^>IU4vNBWh0n00qk>)<>_>-'
            'j19&($1hV!`68;)3SpRjJ+)#Q~p*eVsog){6H48QH9rr~Hc>kFC^;qMsT~S~e7=pi0kmH{d0QC!%fbnC79Ch`92pu&n`'
            'aFhVGmB6&qnpui@E1hxPR7>00BSVpjVEh<QRyRn^oNZh&?j3_eY-!#rd+`B6~6HMy&dXsL^+m@xeOa8F-'
            'Xjk0@?0MIH;0}qo+3m@m9y8#gfnfDpZE&7TG^O1KMuJxNdI)Y?Q3T^ZAu9+r=M`E>8!M*TbX{A49F!IN59D2enf{ux2H'
            '|-Ksb6#b1CESh$0z>Z?Ms*&yxwasWNvJx9%66Gpk94r;f2hvWaekI$MA92}AcmdHs=*~_A6YX!dB%3+N-'
            'LjR&q?AP_)IO`racx0vG=L1jaiMP?HyMPOtQzBvH`U9F<GZ#froE}#(gM{g6#=Wm$xYth{7ySK2cn&&|r#qX`tvK*s&m'
            'X0K=XBzDr5^0B9EX$6>WqRz)7pi<v&f{96`nh_8^qHh8GnCWCQQ+K<Z}7~v;Tg@C;eV9(qRG}#aeJL!<j9VvH>>Vkiq$'
            '5z2tUOKRkY94@Yh9uq{K((eQ>CGSHRuFFgX+a{z;yL_l5gGFc)~gv(=OVc(~@a3C-'
            'MW%r(Eyq@_9jWQGPBKr@)xkm8iLMHqyc?DsrQgq2G7Q3uo348YJ0PVwVkX0kXF=**Q`|?z%yy%OOx7;A&W-'
            'H0C$f7qTY@qSZDlm|%g@C)|(5bx?d=0k)H{X6#I&_JK@+QHh=3<n~7^6vSN3`7L4$9{o*p(J~@ZRVnc4`TLpSBm$=mr!'
            'WTnPt$2tm`83*aEVoEbJM%Cw_e?DW-'
            '+B*bBelsvMA)pItp@&^BIQVpmArLR0pFKKURmz82(un<9kH4@BSN7rEG?JdZ5@L||n_z=~waP$c*#+{ZYaZ~48x}h(KJ'
            'Xjr%4JI+{tP{8BVb>aZ>D^qEkNeLX_B!-Jiztk52#1oB#*AvdVQ6{HBIgdf!|fRfl-NE74|_xL8p8-'
            '(R2eY;bv!I8ZGz2{aYS3k2fvC=Y^q{};F=AwXc!?2pHy>T$)AOAUq}EF(tV*wXM`#mn&RWLt!Oo|5dv@Aho!6L;qeVKP'
            ';K~{ocX~G7i}E>X|DwK-TqMcef%PMwlNd*;-'
            'A8IMQz&B7YEGyG4u#CoNCr)!`}TRxZXw|2SqwCTArKJHu9Qubhe_bm>T*4H^!&t!)~>^kUoN#9j^g*d))Ad_zdWq&t^('
            '@HlX<aGU`tXh{3%@@G?A+95pSbny+&4nua(>;FT}-'
            'y!(ldEH>f8re*l}nhbOE=O{2)A&Oq2yBWE^i!gZc9{SXM0}h@vLXXXfxbCAY44ZAoWMwO?Jg0|$XKh5MTV{~`<`rzvFo'
            '$7dA#fZjheCfljJX!bu9JDl2#+2DQBON)uX@i`^H)Wu1@}q+wo-PC;~rQxOBXb!3&`N-'
            '4$!W3g;&kjz<Xmcx!Zo0Z2p@B&vW@X7Y`Fu-'
            'N28xLl?u|hD2%udr<Y90SctqqO@%gj#rcdXITRr5YE7N2egUL9u@FjGJ!|yGl?xL5TA$LA}Ll+fZMB|9Lh03d$JM)daX'
            'dwe;((7j}YgLusar1#9}~=KXOUgf?Q}c$Vr{YIHe>wZLtUI57(oQ^#EiXRs+$QFLck~9(+@>4Oxs*^8RErl{{DlTd#Kz'
            'cm88AauDHfd>n*^uE6RG`{0&eI_hlSPT6bM(WcW&AV1k2p4QEWo&Rv@r}3apR1n0q+`_<5hlpUfEil6Tu&|*8yv|nQxX'
            'ud9Y}y2@Ez>mI{4a6*UIV<>9U;}5gI0Pw=+X5DKw3ruiB%wE?Gy*&4QUmES_$Z!l7Ztq$@n}vjHqRNM%NBSw7A(wCLVC'
            'JvwPpNtPZx(=oWX}cs-MF`j+59)FN?RN08st51oxa;MHFYxRfddTr>rv>P6U*RReHhw+&|71i|ReZh9-'
            'w79QrQuswrc5+j`!wp+6Y+4uDs2DNBlZRl25|7RukALD}`wx0AgIKhDgFSg))Mb6Z(V~{+-'
            '&xt!>hj;R{;eA3pBfho>n(bQ9*Xb|~_@`ei4@IMD;4vEc<|}>ltCnoN;RlD}6Ty1RQM{-'
            'e36i32ct>R^G?sgy+&hY;qK-su#Q~fbeh7qe!XR651FovSg8Oqf;~M9;&^Sj6Ox41{uC12Q_9h%$D`X+|p9dXUnhV|Te'
            'E9ZQD?U5HK@B%Yv^uo}W2!x=?%-n_E(^l1<&ES)!X0!nbz@uV?qFzeYEfNW4KiBgVOfP0^j|-UqaJyrTJ{g9i-'
            'm*IB`Mg?DZmv%Y3zy7g{a@O2zSm@gY(A_@KJXm3fF7FS-'
            '+Lc82Dh%a34)jEXOr`PT2M)4!_)bNSdS$g4Lm1Qf&H>yw~I7EOA$Ytvt8ryVIA@C9xlaD@5SDZzu>0dqU^jICwje3h|G'
            'h&{+@oz+Nf{jrL!I`wS1<yKN!TYF_3~0~Xky3We87vY>gbl@aFXhnwcLz+YB4jE)?J+d`pqMb3J#9lp=Vz9$Eg0}61b_'
            'X?wISpnTs7{xZT%)+k&QXI{aNEnxE$MTvo+U$Z1gVv{{%I-'
            'GWT+*QR#zojw^oK+%q|j2G0itMQiv#X^!DI9g<~%zK%Kf8sb(uG;IQ1K(EIrY2u?Bsz?I3=f^du|!6Y%3A7RKJVgIo#a'
            'IMgkGeRXT$>n3Lqa=eXXwF3!1d=;O}D6+T;eR1e>r}pk#9?m{?GHh>|4XX{#vCTQYFc4itrkQ!<vFr&tnC_3hx7={0$2'
            'X#?9fohrb})}_iowl-'
            '0jN4yLp^uQ!k62x;vr`b`ZeY+e*VymW(W2FdwCr=rX^vZ&q9h{WI!i_hlG6f2VQRh%%Uqu<&qTo$^R+MxgAeBCa=TMT}'
            '{Y58Vj@4-'
            'SAhw7Rp2mp~+$~=6tIt<Tv)mode(DazQ01@64v^=Nj?eQ5ODgsDmOdD`x6ff}GFN93?d|>a8V%rBB`PG^ldEEGq$Z#aP'
            'gLFG*IdX2F}$VRlSi5DDmKLjK($I^7qGTFv(v^Be?#;Ty+J+SvzbCD&+_PYr~b?Z?G2o#0lH2pTRyc=ftC=bA4I#m>93'
            'J;#d~BK^;><4Fu$E@Z%u*>#My6Epa9SeV1}<KG=9Q}&6>B>41mgbH43gB@vXlsV)KB`xRKrkP8j)^CO|FIRzq+63d=;{'
            'tTdDFj#60_rfc4bJ@d4B3AusD7S?k<rVLo}5n#`lg7UsT;m`nIiK4nUkh+n>cTu4~zdq;w$sxXz<G#Wo0ChJ+T@qwr{{'
            'EuJy!yP8EJE73RDe`-MBr%<zfe2wAY|72H*h0adY5_~0`SH-'
            '8qylEiSx`=F0SiLWsG=OFv)vUF4{pF!Jj0t<D2<HxVHFw5KlT6#C(&xNn?mGC7T;t$4Ob=DYAmGsY_?7`>xCD@Yx2ru6'
            '5!Iz_VwLXQH(0)fkSDC#d+4FD0pm{6%hOq#O9r~xin)Z{acU9DuZiMr#FX{XZ6Xd`*#21%Ei0G#ajEV37Dz_sU=Z41o`'
            '!}tI$@CXwOu~w$My1i0Q=eeLMl*VL`$zcrtdS%f356PMCwSOa1A@LwK&uAH@6Zffc;qu9CGQFOR9}II4X0?5T^u@fA`K'
            'VZj^_TI#9d-JhP0<($;EXra559`edr|LI}-'
            '@II+%!L9)?ZKF5s2T2aC!S96Ka|zPo(jkTW;<tBW$#hyH!LCb2Y2FA#+aFM-EGe&`Z3rR3af!q1R^jKE34=+;8Mt3fbj'
            'A_4D3f5DLSFjjGGqF&!TFk{FZo2m((y&j80Cf*Qt%9m_7bcEh=6=rCfN#gq6G-'
            '#+U0444vc!W=oDfmqbd#n9W>fuh3E#-yQe<bMRAU<aB;h#`1F2i*2<72MgTMK7@Ho-'
            '3C6?9UaP4^i@z(O$sp)*PF!=wn87}(O;LI-i$Kb=Y!*O1uFcgfcKD>2u3l-'
            '|6!odmw%!4rECZLkAF*6l^!dD&!2>=;PpazUK03Wgu6fD5<Xv_Bgyg)>zbU}dr+y6%gm_o*ksXJ<AMPQh&ZM0~`z1?nQ'
            '=iD=k*;F$d(zERRJrm&Im=Ufb4N)Nyvc8Op=832Y~?5Id=ELl70%1)oAm}?jZXN;sFzGyF`@F{bWKKtP9ZR<hs-'
            '#I6&ybbdc8X)D?7BaBt6>M9;%d{{sMYYBH@b#MlGb>3J`}0^(r4o#1efvPSx(?!O)p1LE0xX)lhRGM612;?_kZPB4#OH'
            '&!r@s*+H*5rrN`gDDXOJ7(#yEWN0$v$Ugz5#p)M;Xz<$3Hh9-dwdk*<Seo9AQj-'
            'z37k<1vppYpMX?F%R=<OflKGJ{8X1`G=(?911m*7$@IKGW&OgK#rdRGxT>5+coL|jbL`+k0it|zvOVO`c-'
            'V^b;91K4^Y$TFbaO@#lG}t_E*VD2#Tb5TPX_ul<mdvh9Vqj94BK_iSTLF0GQ41!{E=|Xn*_-'
            '{jkm+THCV8rQry)i(JbZJaLyH_dbc9zvhFFa&)Omehmi9UQeQzh~Z3@6uw##4C&vlL#Ry_TYBOcz8b#>YKL3UAzTtq7+'
            'i&hsyX;*>Mz?~!VkJ{#lY_X7EG-FMZb20p+4_DNZbCL=yr!e)Z=yF7O)RPjW%MfK`?os_kvXx9srLL#5uD$vas^yV$$|'
            'qo+))No(dHk!0{z_u=<leH7j}vFa27Gk3GfXhZI5WVk!g{4ASO)J>)anLuI@QXzmRm4BHToUIEoubE=88?Vt~M^{2q%)'
            'Iz-3&cMed3edUPpB~NnLG(7<#>-Dn;A$NWTG%f`<!q;^;Q@DWG0(;3zjq)xE{Bv_9mEU+0dV{w&-'
            'g8O6v|m^uw}tzJlmBD2Yq^|$gM6`cn1f?mIULlkT<N`c@Orx6rlm%4B?eX23C<UM>bB4*ql5Da`SxQ+w+T9siFol?Mp%'
            'Bjwn-W)ndG#^pI@-'
            'uhz~3D9Wqr;|fv~5tJsNf+$D{(h=Ev_K8M~0ShV`QEW6ts(^G5q@yB;^ePBeutAjFdsc{|SU@yEtbiKdU`r4SSO_nWFZ'
            'tGI^?T>dd^@xA%<SxM&i^_0|C~K{cJ96UslZVzL33vZvTSk(6uc|9o?}}As~+)im9GUp>-Wad2iKCTq2U-'
            'Ae;Zph%V53iN0QK!fK%pdCH!4Yr>zsx=!ifWD3v^fgJ&Aq`T|T(UIab*#=u+XfOgzdR53~dq(T>f`{rFlQE4Px6hB5Z?'
            'i|2P-`|8pnWf-'
            '$B?Q~!`JgNrOUJH>fJbV_@cxcnpy@Ua`>%A7MG<MVOyUT9+m!>E8&+d&_*jnXyfb{aaSgO>{#tTaHV9rR9-'
            ';|$6f}<=0JQC;Ix&xMzL-'
            '2D?0id;>#Tq)E{#6UG4#d}G0slAT^wy=isRqOLPhCLD({)YySL~7%DHsWmaslxMvZ{ftyR$ZCJ905AvF%H!tF&(*!1KU'
            'xU4^gx)KGr{@~XG-d848dGT2N=qikU8-'
            '~Y2_JG@*b+|*<j_xUnL7T*0qMX3cPN$0`SnC!!{D(R`t2svZw@6Yo_l@L$O%MGxZ5wKf)lgOGI(!lLE!DUg4SPMM;MB='
            'zvT8yw-'
            'f6vwFI6SEl?BBRbhHIkXEmXN);!eQI9{;)<zblhyEo|mkO>ZHT<B8}hXV(H<kzZHVJoK)SIWoZD@i*HGGL(k);Wv|iRU'
            '-IK7=Z6ihPaJZK$ai49`PWlm6lZL~r3*{87!4uVQ<Kl8c+bQKcBuef|a0&Z%Ok`4+4{YDws84Safb6u7Q!r?&a!@ZE6+'
            'PWpD@VRcFLEDNJI3V(uWw@%Y|{l`?XK8~cF55v`|65Nk&J=ptbIVfcN;nh4JP!G8cGsFhwW#vk+?s*|fOgV!My9VYMnq'
            'C;4V}?%o>*-'
            '|W8VG4u#r5}h(x{)Tt&OhLkqhJO>9Vobm~cN2I{HRf=gwEayp%D5q+#c&+V9ac^~F2uaF@HVI8hB|>9mq!wPcJ;I}C<-'
            'A-LjN2gmdBOC-z8aIDN}D2vD-BQO`UZ&wlZsndvzhcz7Bm<JzI({alUN2<5-'
            'I^@;Y(iAH#?B$$+2`YX3hr5GtU1}NV>s=rvbw|hwkLzeY;8DjP4Ja8gmvr@>B}*3kO12uvaz5URAehw(VoBDtooM1spE'
            '+Q|&BrkFUof&b4!w;{IsCwZIrZjJ<Yw1kQ)x6tJe-'
            '8K=5AP!@|=uIRlv@Pg>;FoykKK=I(d4l63Po+L!(VK5s!92^VM!pEAa|m-'
            '*BS7aEPNYY%|DS^uy`C6Zr3*S6Qpx?xjwZG8nt?0XlqrpGsN;!OHx4+AZgR1;0rPJT~Ogi}@w!WtWbQW8?)B1AsQJ14^'
            'w{bMk{T@W_V}S~&RMZiXCKtw~3<zBb^7e*?8rfv7ga2IfbGlXKT5lFr5Zc--'
            'SzsHvHZ#my7III@t6Z~Ga#Z!CuQqbum%Z+_)}Q)rAgMjJxQy(iRM`3~@BE1;3j4QO$ehmdOxr1aWhY~@MA?nG-mzAOr_'
            '7scWx#gkwkW<zd&I~6?@r{fN({g75KjZ%fhkfdIXX`z+y#%CTHj!L11eVN#7uSEQ%A|N1i27d3q4;)t>0&a*bp4^j(@q'
            'hTk)HmnJq)YMSc7G%2JWt`Y$_zur4=REKS7lMMB^yUN8emztG+tf10oq!{xpv;u1drPyA#uJbH)2H@UzavxO!o_pSW*K'
            'Ps@tP~MgqP#*H7Qye@}fMoPv2`@wjW!ZmKe&mTJk>f%aH!5FalGckXzAzx{1m?P(9=C8i0MHYmf*E$Luw2V{c-'
            'g?F<}xc8m)(Uxz32cQJT_T<1auQ^aJse%QIB0*JKAG7M+@xHGQ20qQFTOZt{`SbsUS(_izCbuLyW!)tl^I|tT{dyU<b#'
            'p=6eFaFY>W0dhjZiY@I$FA_qEYLw#KrVIcEm{w46D8)58a}`sf&jP?A6E_$Gx~|>un;_o{0g~BcVI@88*s$(hKc|VC1v'
            '|Q$nthjp8wQ+4>H~DRiM+zyvC-{T13@8z-17DGf=BlknL<Z~62>5mZT>f`|)EWK@q5RH~%F{+F-'
            '7e`yw)t?R)<;*)T6Xanvxb?0lq2{P?o5OMENLBEX`A!LsMKInXfzpQcLjouW9Z%ep1x7`IKZ0mTMod$xlP6~oCUW;)3D'
            'GR>Xkq+3<^c#ooKMGGO$q0y#K7@};CbA|+VS#Tpr?Bq~x@w3CbhZNsVjiPfVky=*9Kq#})#=?9eVA>Ng4c7e;4hX6f`0'
            'W9TJy6H?%CP^bK|E99%YXj=;tNyO1}+e>{H-(g^6RS#Z|~nn}x%k<d6i-'
            'gJ|kG@cb)@bLO3BA?se+L2+y<{OjJff#1n{WcW@`yq#i=@ez}7+E)$u`a1>C`XGyXQ!kOW!28fY=`QRxXanif6JY18%T'
            '%`a9B#>KCQr2W@Ud@q&Gn0qq4in<@5o3FDxY+~3?FUmnoxo=o3l|b=`dU>pMhz%o^;NgD$tEC1uexBv@v5nY8QnPnPvs'
            '3DqjWU>0~hakWSWUSKw-'
            's{g_#{3v(wGW6`V4xR?ygzXv?#{GkI*+7uFxY$bRxH<hR1cnqXwE=Ql7c+A>S4SJW`fVb|Rb*bt~K&x5YV1X3(?Ti@GY'
            'WppDdc2z^Zi=TV=k=griz8mQ9D_%DCqZePGo&0f$DqlNp{3LhhIJ*ui@GTMd4v)7>IWHWTzUZpCO%}=+Tq;Sa>}%+(Gs'
            'uTZ2_4jc@X%rhO=hHZ}9s93GSFe1#XSJ8+?qI3%#q1xQl|k;6g_M+8#Xw2bVk`>)Rwz*Lyv-'
            '@NOXQ1P}LaGlVx`4J62*ixkD@qffE~O_K?P_xmbvMbu7ueP1~6C!U5{@nW(uvjDAr>wrtYchd=_Ixty616}j`$O-'
            '3Vu(PfTzflhXecA0K@%gW(S1s)!vCZNzZhblCPObnuJ5QP-cLt>H6mVL<?nJbc7FcAAM0*(q>6;bkP+dwYoI)XqSB51w'
            'PvaWzHu%7mhEe{KWUS^D+B$p-'
            'WKLD&{AQ{SJVOnx$~U+8s^%S3)1(DPD`ZnCy@&Kt>lCPb+d+FTgz|pw%iuj77l#{uze#I&N>ui85@%RzA%9;M7ko69La'
            '=!$KK$TC?K`+QbFwi6mU&b6?=Rul_x5;qvY4ReoF^VF*1=5i1k6!vK^^l1x-'
            'GwlXf@Q+%hgr5ORI>IhfDB+hX7+0jqt=4Z#16oMP1tez_=SJkg;+a+OJT@`}<eoK9#AQer4mK^)IKrH2(O{`-'
            'h9{pKUR?f40|)N@B4~2RiLsVD3Y4xcValmDS5}hu3`aYR@P<X7334BZ}y5<y;J2&`u@&#zR+{4qd#g0=grONY{z|(0?N'
            'ppWj8$&Dlc&YWu0|tyeTSOckQ))sW}s0FMWLUOv?2qP=qxlxrnHeOWo`9bi!Zm&T#}Lq|_k3;zfEe_IafuNWV|1Z0FWY'
            'fc0+)+2W?(di+K*{vAHBXB!&_n9YiW@G{basiCP<1of1I+%&9i)7M`d>M=MWTw*Fi+O%3f!Wj)#Vpm2V(ctq8OM$!rZG'
            'E)$x91lf-'
            'SZ(FT|pm`%4m;%NDWBquKFHPhui7enA4$s-MJMpY1!e|7?@{`>ubWf5xExg0xJ0^6moN`Q2KyACrS;&!wQw-Jgi%#ly5'
            '}PBqH*A0<bh@bGAbGmd}bi~3{Y$&q+}EOX4jgvb=sGN_oD%a5QgFT82z&&p&)Y#iO%{59^ll8oE;w9|cuDy-'
            '|3e&BCEl}A%%@%R!;3pj04syT|U<;ixZ?4k8Tk^yBZ|3JUhp#EUPXy%8+?Tors3}c)g#XQ;_#cZA)#+>wvVm1xm#vDuy'
            'WhAFWFu}<|%y-UyjMbGmhB*_)9Jdc+b|`OS<}8S2l2f-'
            'aJ$aE#^O8tr<HZ<8X<rm$v@e1&*uRZgF1dx#ogKy8%?e|b76vd|KWt}mN5?Sd+rpWeJ%K~{pX{zR>mTT!IjBF~U7BY3`'
            'ca!rSIFa_M*i2vLDco)bn5Z2kaldW;GF8bL`E>j`QtTu=#&vtXzjHk8ltw2dK^ATl`bb+_f1>EDfd~(xmy=WoU^7<W%='
            'uTEnXSz)wiZ^Yj@XV%P8O?MFY;k_(xQ?B!S-'
            'AkV2PVxkokgSJ2+u=Z5x=)=|ir^xv&t3Uu}JclYuBe^|d{Hfm78^}>zI#Rg)C{htaZ4Kw{)!StaN`n&qMZ}4;v-'
            '4HOKW}UmsyxG(LOF<^4=2pftOs!1KXBwMXSk9bjYGGnxF~it=hM9%g3@a;>nU+?jCdOuiDy;(F5ZwVCx<e|54~qR;=MN'
            'b@q2?OHh7G>uMU)GV`*dFqPc|I~k5nZGu7mHfpOCQ{|4TcC1i1S6xw!hVag~X~l%o^|22f$pX2P$V8lMI?GG?=v+dKRI'
            '8vIY$W-}IR!e*z9{^87Dg<tLyHr1TXiUi&~Ve#L)_J0b?yRu=C$n(myiasA%-i-'
            '~Lvf;qIXYl#RshMopj16<dpY8twoW+LC*|406+ukq0*=*Q?4X4R?ZTbS7!-g%{ut#~zoG-'
            'w+Y<Px9WE<&`Ul`{+Hf$y0?WwT~zc9{M+3-'
            'v@Trp*J$6t{@b*FqI3l<q>#nly7pI2?y$7V&`dC4tf(dSh+8xCi|BJT8%{yy#t@N_mTB0EyA+wBXm5gQh9=Z)~Q5552!'
            'vtbc;>Qxped;vCL!y@inX-Ch0KHJTvY*@sdMIXoC`FvzMXErS2PT}KgSg;El7ICNW*)J^Fd)csvJB1HkVb#tI7A!JS!i'
            'Scyix8U?ai{Q!AM7H;hDF>dd|U>*2(e)icM6|1!7f5<Smd6C4@zJcAvP@HPT^htS#>0z4U4!_c!@qMase9_ai{S5c2;B'
            'o8y0b=@ZxY5e3lK1xKnslHw&(1!y@h!o|b*;&b@3{B-z4KB39jf$znxvEj%4zJr!(LB-O%Gflt}gZWb$A2ZY-'
            '2Pg%QHELJ4Z!YwfCeqXa#kvt1GudEUx#Qsmq%!Y+~%1NKVOWzFt58bYvt)%2vfAz-'
            'VhJTY^v97VB{=D|S8UANFY$c@zui-<IhaRYb&m<)OWWE~u{-59dH|?wL6#'
        ),
    },
    {
        'seed': 20260729,
        'weight': 0.3333333333333333,
        'checkpoint_name': 'final_tcn_2019_2023_seed_20260729.pt',
        'checkpoint_sha256': '1193033758baa4e901b6dda634d3097db0b058ccd412332c126102cf17d90391',
        'raw_bytes': 131294,
        'payload_b85': (
            'c-o}72|QHq`~PnbEea8B(ngYfopYU(c9B%lY8^$k8G{xTA`v1&gb<=qw8`ApDV4R-zN@67eJ_=k-'
            '^`$QKD|Gm@8kdY&%=7|>pWkt`?~JAmoxS@l2T$~va(`-'
            'z50tOin(&#S9%4r^*mg8t`h=Q`AnPKPi%3ISD3_lj+mB{yx@jFo-2>-'
            '?BV6k<A|^4NN70?5L^u4`g^n8d0zg06GGTtD^~J2lI}cz0d|6O5YNjeP~Mr#b_??I@o?s`{Q~{D&cTNA94Rd=r(PnWKi'
            '72yn<L$Mi~KYne^=hT&WrLK87FZ;j6SEAlceCcdq5CJ*0x!S$B~<BBW9yvGuB2tnj<g#CH$jzjJvzPk58wSK>0ab4>p('
            'WF-2fjp3_I$fzy}A>F2;va8l^$+0Dx}kkfyqz|czRm7RhHoB=CkZ8UfsMUhSvN6AJ!ilZE}QivJI<ES`rRGkJ20^NN4-'
            'B$%p(4XM$?-y*)-Tgs5V1^vEm9n;CeRv%8PS+YXlF^*OHZC@jLf1n&sY7|3VGf+(PD+1MMFur`FpW4PIt^-'
            'e85}7zII0shn#URAz#03uLBno?T0LOK9PKWH<Nh$H(`|5kCsminnc%?D``e&lr$PN5OcRblr$NImgGNGw#+@h=9>>&yW'
            '7gB4pFh|4cLvRSz)U$7oiNKT7)uDFI$^-$&<>o5J$-'
            'e;M82$gpv*XvI#H9mP}V}!lup!C9%q^ZXZqj1ekX25512V;W+!Y`7tBToo81Ys<#FsBICK8?)s-*%9w-'
            'aW+)k837iyjmHNO+JfX8uk;4J*pmqE`F=kyyUAsoK?oJCy~UHnHymvmQjX(x6WkF(r?<NT*XgFlCyOHZf)$F<X<TbD(5'
            'p+%2Q7MsWMbl|M`+v4xkt?WrN<al*i^!~%*s&0!uomgKU$IpS||F^}iBnI?^8gV$C7P(y(1BDiOova`pC)j}#($iv(QM'
            'bA$&6u;MleV^twoXV3?WC>eaW*(`HuiM($FK|Q$ui-Dcd{b7Set~b$WB%ikF(i<6aBZd-^tt3lV-'
            '|^>7>PW(Y6X{ah<ey9w)(pv+ZwZT{%na$ui?4b+Wd1v66+Xlup(T9w*g-'
            'llG@G!=58Ay(i6_v$LzL8Gn>Dv%9RjI<Z+iPPPLl=TD1<e~!G}J)sty+)j&mT^93&7WZ_r3V57C2Tsx77JnyiZx5QGKB'
            'u_LV#yyC_jOy`--'
            '$iI<CHpZ4*qSiD~pGELJc@&ofgZxEFKnGJkrTJ%Hte!;2iI1vB$`(=t(o=oam%gcG37k8tJ4VkHa`{PWE*6$H+U?lV!x'
            'I>SUenVx1AP&UUh@d7K&tPVL{$ekZT4C(W2s-$`reqMZ}c&Uex-@HmYQoQr=u>&n@so-'
            '7m2<xbX>F4k2c>slx4I*)V1fphauXGT3o-'
            'mRWAQ_k(KvflZltarQ1+SG}?$K%|0;5_)#qS2otuem4GjPtP5VoR6BM?#B_J6TV7oK^=;+us&{C+}%bnmOlLm&NCQSbW'
            'iK@nt9W6_4}Uf%E2Xi(OfK+Y@TRdDm(2eV4@#LW>_eS)X{E&kmgSo)&wIye~ayM*5tuowRRVv<@NddnfG&kMq-'
            'k^Q)(`e?}fxOt6ym#2IkKh1Y~cu7u#PPAFGW@TY*wl@k0Gp}Eq6zj$03!A}QnFTu~h9d{>{EBlv<Ay@9Viu^w+y}MNU{'
            '8s5JQt2mBQ4p#q{jDPEN&mkzjJN}SYbgGsq12_J{99w7NJB-Wp(@nqE4Y0n+tov`{CD+W&|h?8u39JEipN#&qFY6A2Y1'
            'nj{H6~T(T9oX!-e$T|IkIwHU46oa7X-RYyQI?*~K39n>|{@9wTCp?cuz8v~aclqMLHHMRZ#+Ssr&>cVcX;Hgk1^0#-'
            'KSo4MnEOX!LuCWs{Tdf4w8Li&FRm~jn$C*JU%#2a<lH~!5w5wT50Y_lHrjl1oe|3x?FTKuM4cG1OcG&XZtLcWI3J@s1w'
            'L<+P>VPX&W#+~l1{^DD3Cw210dECj}{3xz<m-'
            '{Kd*;7UAX(IM?!OtMkdCS+|gYDzI%FEA#J3}ZaJO%OG{kSuQ|MdACAebFIy;g8%39hc^+6aDjp4R-'
            'FJ=w0jATB$QJ6k9?)=78<bYJP}=g0O5<l2h5DC)#2Hdk=TInZk@n`<Y$Ea@aCxa{HO<0?8#apwpwF5}t@(SrO)*htyPv'
            'PI`N53YZJe-Mv5SI{(l;)$KYS4NO1_g&)7ey+X(2M!{_JdwE}f>y3;h%?vKZ<TYfzfX`a+u6(4&DF=%&z<e;W5k^=?B9'
            'q#qx^-'
            'ZIYCE*{CGV>7YIX#{~78U%od&=yKWKo$$5oq0M}6%IP%Xx9|LC(b^vdsFn{iBKOR@qSyAvpVesHTgSl*BE`r(4fdO1W$'
            'ptrag*$hrK|P56E1Y@$JXfDiFn5tKV%Wb&{Lv%s;_k`lcbuCmS0E|KpC5OLuu<`^MpwTTZ0^#(ZWIc1rj)zvUmIP0MUu'
            '<^)w~B=;LI<`*Nx5PItwF&r4r^wc$EIvQn_@O>UY$CEtP9`seXt4mr}WPm+E)m|0tDvcd32{|4XSnx_kQ%@jpt%?k?58'
            'MEqGQ&+erDj{BojE4oY7vr$y4m4Ds%N2$F2wXv&I-'
            'v4U;&r+=tMyLtjj%**E>%Z2?r@MchA^){VzTH*tjQcMY@)JgB{Eqq`<?-(}-'
            'x>KYwF&56@4Hz4Q5a5legCuh&#G{RLH&OR{ZW!YVT(e~7EwKTJ#YD=7(suxbX6kwZ>xWnAw<|LoOy#?eY^y<_xE4LUEK'
            '`}WCsQclj0^IdAM@daMyGX7JosG{e%nDKW%FTZER7an`<E3x#v{6PBfK<il$QGnEAJIukTL#e=YO|VT6hx;=h!5qp)3F'
            '(EdM)8}@r@`n#;*-HHFtf<|;F@!w0isXK{(6fshGO|JWzsC-'
            'e~VSf~E^PktcN*4X6jX#UEMU>*cq89&EZm#a4c{Pw5BV7Nqocak@S1&*3u2KqHVmlX1DN!-m9^9>h|7WckYa-'
            'r7abG2PYqA6U-'
            'B$*3<AkxjM8blEdHH$yt#A$yywCkTxbfZnH`F(<(A75(eCZkL8ye{v80rdk%n99wf8WP*<*s1!xZ8vcON4{Z)yrp%#(W'
            '(E4Ob8Epg<nmL$@<e8p3&8L$Ka^c?1b;uVs5^gn02*YIq5!4R=BEHC$H+n1MVE-'
            'b%JcV33=y7f*1vhQE)8hNr9W2`o3UJJbGdq6egcv5_R&R%{myHz`p^0|U9q!Z>MBj#dj&k@8QL0|NzV_7pf5juP$;VS}'
            'acA?^f|ar1or{dp^A8)&()d9I*8!9ZVILq|i{qGh1#8RX-mE9kR_pv$@*u4^>x*@C3|`)PAig$x;CMmiIb)-'
            'BfE)9wtV|1)%W?kk;x1Rl6Mh1WENlc&FLfDfC;*7#Gyoh$IG8`$|Mn42MtRTX7;O`xz`&i(;BFJIxV$XSr)0B)wRs#2m'
            '6@9EDCapej}hEPhhbJ`_nwH50xN?#zm>p?U(O9(X+ZpZxn*es3CVWJ@#CK{p<FDOJnfRC5pq8nR4WornM=jx-'
            ';U4Cx1Xger+YVFDuOaQ#!Z<QROU{!!<Bev2@_$Yl1ceiMA$Q4#=JvUEOG+~)Udfa@`zEf01VHdf3gn&vZ(FLx5NFcXBI'
            'Hau<6+83Wv-'
            'g4ALSeX`xTwvS?dsR{?iCG)BGCr$;q#gP#*a64zI%ibfufQN=YzeXfCl;O{%0K>L@5*nhz7GD*P>c;i$y_40}942E*d5'
            'JbU_r<)!eC6B8qu6_r((J-oBza&PUKJdZ%!OA5XY?^1AD_PZT=Gu6A7J7}d_t8`y$b%p;JyU)W-'
            '0YY>nkxEQpadq9{eZ&7|Eyt~rk{W}}pT|M{i>gDF<s@u<Rd3ScHehYtac)yF@o!%?+y@zHT{ovi{^5wqm<ox*y-'
            'knBl%;vioIa_&mi4R*tb&8k%BQE)Wi~Ft@pOk9WEx!8j)7S><E^(rkRv9{3vRgd#f~-ZHLYMed+Y@q2GN>-'
            '`*)HD$I>itEBQEuSi!Xok*{kknx48dIo4F;?-QsVO$K5;dtXtfzZ^oJR`@6(p-'
            'TDtVBCWf74tZ9c;)nhbm;S%SXXl#q56kNocTbc|KlQOo9DnpV-gNAIx46}EucDD(y2K6kPM<dB8>_qL&s%17ikJN(F7t'
            'njTYVWZM`K5~xU0!{gMmA`#F_G^){@VDb&G4~weNE_=@Qp`f5xr!wtAQNTyo8}Q@s2i@m~M8xS4~utHY*l@x>|cpH}L1'
            'iJ!ERq<J?!b&IS1N_)8FK$p0>Y0HZt*1Fx|Q$vLFP*CTp7wFFQ3g9^hu5>jtG36c>&QeyQr&R7{W^6YjV-rs|b59R9S3'
            '`X>S5Gqw19M{owz~n_!qdc)ZNWD6^fWSbXX_iVjm<su4c#qFO)T6D%uI~9N4obFzt^vxN=F6l-'
            'lCvU!p%nzZ>2xii`Th(_%F8{`+bY%pDU9yd$n*$^%d+%{^N$@-'
            'M!Jb@Nm;Nax>KTurPNuaCddpH*$3|^>8&Yb941HaQ8GbcK6hGbu;s18yguK8W|aT8W@|nn+tkkVd7!PHs)4zFA1VaxpO'
            'Y;A$6i#%GFem0#8p16C*=6cXwlVcebZ7+tkcV-^{?n+|1p`)y&jH-_Y1x-'
            '&o&Opl9xG;Ob%GZffdb!1mA=G;k}scZ%+=?km~Of~|8f+fTS#<MKr-'
            'F!{X#3s3!A?0&`}$bze%hhXdB>nqrjG2MoR>z!atclLGl^YUZ|{x*EF+c4Y2+`!b+&BDyo*u%om!ra`|jID3s>FzFYV{'
            'T@^HZXH_H?=S^urM+*6@;1_yLx(<xS5%ld04OwT#XI5r@Hr;|2X&$t*UM<H#bjn4`VZZLo-'
            'h|BesFLzK5wNTi?{s%)^syWUepFl)I~m8{0sTb%7T{wxPbMg}Ir&xskw?k-'
            'Izh^zTLRpY4>tkC*!z?io>X&We&FIwpwL06&keO@XuMf6>M|kXtPrNC`c5>7vanw?=qPs%w$v)(Za<E@6UoMX)#t7S@0'
            'O;YwU5yuG%2_tE+6<)4MVzPmLjXo9^>FWHJuQp=46$6evK{eS%oet=lt-=D#|t>LjnTj6Pw<^K5)-'
            'bb)W^I;2#QSeAjxDOJ3|94_PF}la6`g;1)CR_f;XZrtNUbaglaniX{aBgD}SejSE0Ec`^Y=I=bHK+mC|GEvq2Zu0tsuz'
            'q{FU4qVn})roD$&vJ@58dPc-'
            'R+r9v<FH0|~Pjcs_R*e#>Zt2KPRU`ZX0|7q|(%Qrf8#%foR;;$GZZm55iTN;3Z1YV^7l(#)w66UI+xC)F^s2zORb!GNl'
            '}_++CVv;Wr#FtAdzyuYA@inQB-HXqVa>5(M;JYd_21DVaBrD+cfRnFq390_<fdI!Xfy$>x-'
            'Es$Na6Z$H>gtb%{^2;}q)t?RN={jxrAY>1Op0TFGi)--'
            'yo?2{<&;>8o$#fid5`A*PYFNC6PYTNW01Qrn(e~BU)66V*e!+xx323Hn-gSa`xn@*7M1h%P2vIC9$AH%aTO$;isWNLpd'
            'uwlcs%9;*npKH$9Van&?RiqhlZT`;R&d2{5x;(@B4fWpiOxANh?br(2eP-wVPENDJXg|z_x<`qQ|Nq*-'
            '#(evs0!mN3u2eP8cJuBH>0-M5IC-'
            'L5)(AcD6bc1s2!8)U@)zQZYL{PKd0ZOZc{74@bpc55oCcrNfcvTwUmBmF^s7_9m(7?@?q|8ZRWd$eZ_B-42WNO14(_-'
            'VbS%G%>4tWA#aZ@9V;!vln!d3LLMB0(Fz4bz3&4mRc|buf4rYujT^+2rx#MwcUbV<4(j2k6B}r?s6kArcQWXA41`maOK'
            '9tjPw;{L8I)NYg=Z)=daJn<ZJ1X9a`bV;gEYOS+yt8n55lrJqv*T7Q=w%R;p;dCqVe)`xJ4%bH@t3#oPkpy^6ECr@r~c'
            '{to1oqSM<h$qi@c<YGO!i-%@hyRs{}yoP*cO>an%>AZSaDpyeNH)9ZX5V`Sz-xRjWK7Y@V<av(vgwx{FU_^Gt#G-'
            'XCwUY$8DCd=4;qNvZQTgZk)gs&qiz~~ttUpc*?W{<y$j<s>PkG~Cf6$hjAk%!QtT8mF!|DcA?Ql#zIq{G0))tHjJ3KnH'
            'fhhxU$Fjf5|WR2v|uWMQKano8fbeTYV%}`^u%)dqjE6*kRUI*c*%@0&KG8Jr(=fc=NT$Ft%PCt<wK~KAL34hef<G_d$5'
            'dZl-M2(Natk6u%Df&(Y2dmP7t~sQ&Op{LkRs`7tRk6Q?8%WQRBTKcS;HA?oyu2!y*|J)W+1lz%7e9Loa>Xt1Yp@-'
            '>9BG55IW6#h%zIqc96=l_HZl*Fui+Prd<TBhKak-2r_lR(B6Vn2FIew?0a}i#!_2<dp;qu-'
            '`ng1n&MMSk3g{syW8I5x|MnerG!LOQ&sS69PDqhHrJ96uDi!8<i!<}JOrRh)99P^tf(sRNU~Iubi}7K8v|ef}go^LR4<'
            'jBx{?s^pm!AL&3;HmH6<LrX7fDI-'
            'C29SYiV$Vjh!XypxJ9B6XR3KYp}a9Y?tna9vn3v0B>JN>Gmjdr*a1N^`cSSL6TxlTB09IJ3>6&rk(|DgWMV`SRrIx%T#'
            '4U?<1SP|g&3bGu3rclKAI#heHVVT>&;AgJsQTo6{qc$2GeQW-BgG5XKKhb33^#hKe}8;4$be!^P?)<sD42|K{HI9@yQi'
            '~cs~X3&6`Mf7{{XXP(fcJ#F%Mrd*E}-3DVXcLxo?Lqh~G8U>zT~mt`1d#>l0tAr-ogbZ?uf%(rRw%pMDQ=Jm;X$b2GCJ'
            'ay~f^`|hZMr;I_6*j?Oy;+rwN^kJjBsF^6#BDItX$ut;)SHe7ehAG8VvMQ1BwcS^P0h2AK$$Hv)Iu{;Q2aQUVa{qmyPi'
            'H-vGO#Ly<krNoI%rTZ-'
            '0RWlP2<AcgzOAU5N8;Bw(z~agd&_jqQhLGg0bQtdCB&(PYCa`p#%s{1F)qQ`YXl0*?={sN^uYS<{A(U!1TMi`fgwO7iI'
            'Oa{_Xj9$;+ieCnBMG0XUHALhcwMwBl+k0%b~pnGEgZBFUY{{4o~<M9#Jh3CVocL8uS;X6NkoCEZ}@B$1T7K7hXMU44;1'
            '%{7Nq*rbD1o0t*=~T5ZDBUj4JpGx6r|sIHMS4Dz&as9hm1&fZVgb~~Cs09B5AZZM%u?+X2cjaB7$2E7Dnmn>`PBOg98f'
            'xkDo#H@vwkt1Ap0Jgd<wA7ivDz-eg!yL#^aA~flNl&Am;j<Vf4JCgXxf+;*8$iTHJd<7mA-J5bqDFbg$c2G2-'
            'i0(0V8fR$RclZb|6!Z5O{bz6LK|p21g2%ZDLHmNB=KuY<#Z5s*0aK6*4-Gxwf(Q@b1&p-'
            ';;ylq{Qu10+vUM<;c_{k7Yn1l{n_85!!@>1bm8U=w9=c{d|t`vCctX)w=Y7;O_}K}-ibfZJCS#-'
            'jNuELeU6x83N=%&=9bHk_4bSo0$&^-'
            'otJRPH4?wPhu}xBLfKEbv0<ks8d5+%qV>_$e#k=vFv5E(}i3qmezKKNDTh1e*<a!p{O(x^?7MI5IGwy1FeAJsxnt?70K'
            'vtFEO~zoo-rPi5xU8GU?Gt4i-'
            '1e~T(MUx4f%9VjiUkDr?;R_ps`l%&opyjLi}oKrdu%a^L*1@nUt{bdEKROUQTpCswtN9N;e1t)ro{w#X+)YHTxB@LcAs'
            'Wak(7C?&SPuTO_m}TWSh2(~>VrE-SCmk#G>FOz2*j&|@F|yO8eHKN-'
            'neQreSkqAmSl^#f+>?RxJ7k!KDVHe!1>52A4+=iJ$<c-{HZtG$Cc^B8+o}1xyl`P<I@na6qiQG3X8P*R!NDK-tWU-LFy'
            '{0}OfcR{DjVwY)47e<61#!n`>Z17Nm}$S`5_GZ{U~Pn_AD5#sRUx*wnNQ=e)L|>FDiEBTMCWkm>Z|EafjJ0Y}FaafAm6'
            ';ne@ID$F=9f61^G-8$XsmcUd@DC;N#+z7k_v8-'
            '7t<?+@T%u_+9*hePGKe}<uIPK?v=didn_l0~N2(YLm|q7tRW$w$t3I%AhSP8|Fd9#76htG!qGF9v8cD&L#I?bUVau4O-'
            'F=(#bptMg;jd$5A#%#~n%73$%sFEzwZZ4BMiJBxll$`kf&8Uw~1i|PD}hvDb#9D4c3G&F8L39Cki;|C8J`c7Is$_{O0H'
            '6&~Xhrzmx*n<s>Py0PcSNa4w*X~erZ8K3UHVz?8pHBLf0i*Qq!L>)l=-'
            '=MPda$4org*=FaqlJRCo+M|$pQn0|L8f|e7;8T+DSZeLxLG(Rf%J=%P<(lnY=#UD3x-'
            'Ruxd(R`>Q#OYGpniT_?w&M*?Om_J<G$apv~HILu4`03C)hsC-G9*)HLS(rht|`T;04bptf6-'
            'Ad?5HI%)L9iz%WgRee0!QB^wnS9qI{-8?-F(T+boE<hAKi%%d49H2u)G<GC_R;<DD{K+DFgp-'
            'shSwu+d^1_7GMHIrF#{89#G&v0K4{qCgWuNNwcJ-Um}#rCqZ^tHSsOejF{%oEnNOqhG3o9g*4w~cn5gz0K5Ud^0;jd(;'
            'n{OgE+ZKvU%%kb8Q7OjD=I;!xzR9uUpgEs$w8Gq^3>8+H%fxnh+jY6hkFzCv2O4!sM!4u2Ha6&Mt`(}eZjG`qtZTDy&{'
            'z=Y}Fz;8@KZ}s3LG{2QXKtOenf|9=^O@3n`I_p#Ld}JYub*(~`NQr0^l!P3uFiw{wJbQmyFb^`5K=lw}^Zk7df<e}UWe'
            'z3EREw!pmj6wF&h({>uq!LQ;ec`H4iHa#v&A9$q0>>7~_HRbxuw0FsHPu~z)yn54)I|=|X4{y#iC93-vR9Lc|_-'
            'Rjvx2f_lc#$GA%BGsSEW3j>>Z3fJW<8WXrImsOmJ0ONov*1+Xh``8*89^H^0a2L9Q{=KBfec^iG?53nXEE?rA)*$a620'
            'YtFjj1xFTJ~A!;6T@#1l?a9Ix$evRnV|0wRy9s{f0D7tkCK`!qe$X$O4Wj9^Xao%WHeKH1jztW^bi&emI=Lej4Talit*'
            'BfUtI(!X>5}Xl|Ld6}`W`^hwq@{m^!<AV{ShRC8w(Xh0_!SMNUn|7Yje6NQY+xOA^4?~+E*(SEbYz(J&}yo`(?x!78xQ'
            'bO8&7Y#nubpLG&9S^iusv$6yyzBQN^k^xL&$PEjaH;PAF>AWdCJ+68N2OAmN3<j{TT7^9teT{0<yi5<&9kKB5%w7vPQ3'
            '7TEiUfQilxd?o%1PRKn0%TZ6MkWgi`@j=vDz8$5M2{dPF(UEDJFn`V!*29aDRHcI+UD<yM?#e$;o{y^siw!2Q>{wrBZI'
            'K~X{)lF&f*1z={7Os|+A(quAb;Ht^q3<FzK2UNDock>+a3fJ{N8l4Z6HZMl@6RO*Wqbx9%wAC!dLIt(=x0wkj#mLOzj^'
            ';I#Z5WR6c<0Q@;puFUJxk|5M;o_6V&L#hCbs2K0|xM)ba&{g}$p9@LH0mteR38%P|tg8|zZs2Kee+b3Lw`LDO)<QJMay'
            '?Fo~>Nk=~cyC1aJ7SHdZBMauhd0h`=ToT-cfiu+0!F(iqHMrJEdMG^FBi;hN5+WL=To-'
            'e<i(pILH`!!KjxyBZU;`By^LzAjDulwj9F8Yb|9~~8R}$@k(n6}flbUX?3xJ<pRkP4`c{lirei^RStEw^+XW5t*5HTbE'
            'pXEDE~-dO;ZM>;(CsZwk6Bhl#cBV<*TIvR#w9n&UYios24~!=Fps*}b`(99SmV2doy4-'
            'CFLUKiAq@05NJX3t2P3U=m@d1;@?B~Mj?Z1pY<>0|n{gFRx!H@p{6dW`<uyV<;xriV=uE|=e+B(zQgmv_P1w5m0Du3uS'
            '<Dh;L)<KuL`}3CEI1d*)8Cdwz%Sb?RFir%b)`xHUTdtzwmbT8WWX?{^k)pVd%NO}wG1Icm6<tmhmp5$3wkCFqFH7mn2K'
            'IWgk@t7$KzTdI`9th955f+W*)`HcX`;X?u$7!12HY^5>9eG3Kbl2NOr9Px!xD?jDHM#+C7#r+pSLjWFLUZ`IgY=GLwnP'
            '+Y1#|X*k+%G|YL{jAFHi@x$UMWX4XmJeRD_Pg#H@Z_P+7-'
            'lET>t<>cooaG3wvNIs~h7_Y!afVek?hd|mS7J7O@rEY)K3ocyrmG#7V)~YRh#0j1LJmiwxKlYq&*#B{+2fd!kL4JgYev'
            'r;a+0sBqXRKx%t77hJb;b?nY+mY27SEG*Xc!|mj8*ctQiuYsYY*d^hDX!8Q}4q!7o=wQqOiLfWo8+82xzz=G5&ay6Y71'
            'D$a$)DK^ypapH7-'
            'tT!F7{2Mh?*P7Y3cmoVt@fd2~%g`<r?$mXgzRcUcW=zw`256df6%<>)LZezE)xLiqK2nhe!|5X!<C$+@79~e3ZC*%{=H'
            '^wXSD7=Kf_>KQin};3<2|kkbAs1(dFX62m1bAoB$h#S;CH5sIymMks;P%zX+tG8tp0?yGS{%yI19$lHfOE{41uyHF}!7'
            'B2fH_gqtqP^u8NwBJLQU~HZuiO|D_AVChx@^(?X$tei})xQDej-&*7kw$xPYE-'
            'n2ZCrPr@jVcHD$lKp<e!8uZlPR~Y6)Y72epdO>;YR{C;iiBeS2u6E-'
            'Exu442~DSI#_rq$%6oSTZqJh@nwsS>b&>=#LrRh<NIr={XYFwCrCoS2dpp(gq%S?*ayi4Bl};rmN-'
            '?u4+Hgy>Cheb((Aw0Bv2W@qO>P5L9C<?xl$N89`nu3AF5h5dkp`1@-'
            '2p6P@9^CsQ?a0C8QS)52P?r`Fe&c{JbHQsClwgeV@4>_Z|`U@iswI)t(&6ByR$}gp^G!7RV%`QO(URkxIGqblA(2+Q*e'
            '^yC%C<GEikgmxMh+qGtlt|dbjI?+T?@OHHF1Y;O;b{?;^&ulxMO`pQS)O?+!);DA9htpYV=jE`CCJ=3>!xd~cYKwtWUt'
            '7j28s{Ivq>t#~nU{k4QlSguZFW#ySs*;yDIy@nr`FqHO6SEmc6SqRQCeds8qo2>8QVK|g~i7F}e#5F#*s5h7GDHS-'
            '3;hb%dKFg3)uiwGyZD$2Lf=8ofr81M6u8)V>+F|0$aN0uQ7KnS#0qKuLl!T1~99P%|>!g8tb}5Aw@S`_9DRMmZ^1u!3x'
            'aSY=d!45IWC4<YMDXYNyh1m29nlF%#f^6(FurOtN!Ew~2fse_qn{HXy4;=Ru;(7Gn)4cR)Gm+}EK@K(a~Z_RMHKH-ho='
            '|m;q9rz83|n<JXF0BCcH1joa#rwm-~rr*N-6OI1+_V{(Pj@>W3+bi3z{|*ONb~o=^S^rcD+TQ@-'
            '~QwRc*9>Z}X6_U;}0^?4<$=)zh4^Uw3q=88Du|5=<~oP7ZT^9Nwe)H<B>O%?M74rDscTEI@RdbssGnoL~19UHkCjQY=)'
            '=&R9=_hhc)kC$y!+=lfaeQyaE#}rUMXAJ?lx01A}wJLpWU?E&0z35AKe~A2tiobjkCU6+`Kc8GGh*|%KfBk8bC;tC<bS'
            'Zkh^b3O;3uS!j{*~G*S%ZhNgQ<s$HK1(sbySob#^OaZKy^cJtkShbvC223q<jII9xBFWRxeBp{m$B#@(S-'
            'p8R4Y$G4SxlKsa>tB=ub*5u(HU!9&;AB<I*T^beMUQR`}{cdyFP-gFo)HQf%S$tqODhGINhlm%;*%BgWvS|Kj-'
            'EAkJW0G8eexTL=sHh5=1S<4|%T{wg&#}<M=r;&Q$aEMg-Cgbd3oAGDLdumKv87hv;A}7xDW3mT%gN~X3rBI**n_C*7<G'
            '^kh8!?Q2eBcN;RyC2a`ES8)0GG%WrogXjQaGCRh*T&BLy)o<vsG1+zFOZKzBS#37PW^|-'
            '=U3QH?<zmOjw9zHd~0^ZU<a^GZE4!6(Q@{VematO=Q>qtUMpM34S&_q&9kfAbO=I(BV=B(OPzpJo9~x*OO|{2X_*i4PU'
            '9G6MbR#h|%~i#)BHJ-^Sus_)(*k<jCa-'
            '*?hgRM$n?<fhSTUah>B&3?iDW(hZL2Vu9fQQ3a>pON3?8PI%d%p1M`Fhulmk#y1Klpx@6}NC+4Udp@6pvA-J7-'
            '`tlRKf4wN*66^CvISrpqQz1?Q3bF>4N~810`0UiwBfxc@vX<G_uf{NUsG?e2ww?idDp0Z@moo_<Uut1c@!P4=YVzeeH_'
            'x?9j}&KfXCbmR7P|ie!TmRI`;7be(QLMgTG6G@`_&YT`3YfveI!}zuS~V%~tC8L2cYz<4EzNB5>xdddgHao><bi@NLyJ'
            '+-$NPd{>VKU9ba};Z@M@_(}9@Iu365fxjel4`{_T!ldXROyrkRyu=|`e|RWZ`36u!ti-`OrWAsB$0^fA>Ck5UjjH-'
            ')1Gc+aQ1re7mw!A1>vkT%Cb&h-'
            'uUmyb6_v2vxDdovN3gV>C87Pz1u*mpo9vUUga!Uq=&=hazu>(PwT}b0>i6TJljm^g_*7CC-'
            '9!xyb3~81yWm3qXH<OCe5{En!ey7@sOcwbV3mUn_$Tgw*{rjb2L<Ot+xif=LT3v0Ku-A8s|tcry<m=HHI)J-'
            '$Xk*Fj?Ej$oJ;F)^N5&A$AbfL!LDYi=%GAr9<UV`#qiKw;WM@*EJ1~`ClH_K1<5z}LyNi`%32-)9hW3B!q5Oddp1B!-'
            '~Es;UJuQ?hM~LBP`pt#l)f}lmOeN+q~gtq(=2OQ5Ae%!z(vbr@apri%%N9nD8&*kjGZFERGO+_!oz$@PK=MrNm01^)FE'
            'j2@ey=4hmpf)qDW4y61_)v4F75Ca-'
            'c&BV9LHFsQ$DXZ%DRN<BMgOOw)x>cX${+&6Fo6j}M2{mKTZI^!eBkeE_K4CYXh3C~^D>)Q{DLI`ga09F}OgWArJ!HQEp'
            '+B@L#|hg3lQ>s*xlJO}8*N1!0!7q!i^gJ|zviZW-Cv32?ra{QnHmHKioYAPPY;S0usRazlwjopt&lAgjjg*5aFm4G?hd'
            '*hm6i!pq(1<dzMB8DA&=)Zp~F|E_Z;g4i2&ZNu&Ba`za*#a;%u7ncf8Gw7mVRNsf>y;jF%-'
            '~(rN$OzrP`Get2^A1}3OC=F2dZhCAn4;@REa;0D)v&8t&#?|v;#T}iv@#~lQ3RP1&$;|!BY)eC~N0IWSL-oKi3LQA0y$'
            'IXEx>gU?Q~Gxv{3rIz-'
            'L$x&U(v5~<+F`%cui^d;x?Cg9ss%PH@ruW@<V1J?GpC&*@%qwws@0H`op1zCGN;OfdHFy>qk$S9QY*I5llrS<O6oU4q_'
            '`mM$p{cK>wy7{b~<}Q|{N`>gV@C>zV)^TVXR0HS6WkL4y5csnB4BzL_Aaeb{POQsQLyl!OCPl@<xTwRhZ_a7RTAfQZJe'
            'mQS1wfvks^@DKA7q^z{u0JkoF%*5qRH3FT-@oWgn2bAII{K&QN6GQ)9t3gA(}-?vA&b0J-=AZV;-'
            'Eax6G&XW{k!9D?w2CbPzq{=6#YbEyaY791ZKHkA|&b58?jAL`>{h18;wPCB;`ZlE>Ckcr8?(=H?Oz(EEV>LUM7%otvQ1'
            '{(%~1>O_|3>>=ZN)sU^MZP4+(08T5Ff~9Ob(Nn(!Dz7$CDwkDow&`1H{JeQkEUk$tjnnaK_*VX@g+s}?-AXvop#)acj-'
            'su~oB17*cTjz~1Y<GaAUYjLMtzI3$SI5hv$x^U`p6T^vzCKg@kdg*@+CgUZLp`0Jv@<o3JU$J$?{=`;GpUr%Qv(tt~nO'
            'LcODsr{x>D~jk>;+Zm-'
            '_7&Lmf8lPrW5Jrg>Tn}p*&ou{(fZNTqwD*sd{kJ5M8g6G%9;E<bEaBQL&OTl3|9{Rc)&1VYs$kL9e6|x(>uD8JuyIj20'
            'q5%u$rm_4!WFd&zlh+ov@X?T2D8BC+HkEr~K<FiMopM5n=~<|L=l~pfaS(o9sKX_uKgndb40v7f3Px|JsJyW2HLO0|2L'
            '@d6gK*Onh|0Z#AJiyJc5K7?5zBBxMiM1<cor2ZUWmmfN5PJ7ZwS_W1-AAr;3yv_zG^v`{`e*2+^9ygK5lrtk1|$;Hj;-'
            '6w?Kag!!?1~_-'
            'yVPXh@E)l+db#qRGuz(B_D*9d4r2EJuvdOr!#>FTu9kE$GG3z|x3jSW0xE`R+KVjmpLMV+X;@=?h`Ykef&~%A#XT1eM%'
            'mP3h)`Ku$t3%+(T{r(C@G9j)<X`wA7PER=xFiV>8exie1Xug49&7K86_Pn`cU1@;ZnN3Y4_VR^q@RMWfZ_<o5zP#aE=S'
            ';KSCyM>R=kIa!f^b!^KYd^WSEDI#=_rdY^1m`rtUM8|}9BzNO1D0m6sXIsXAa<r2pNu+>x+)7$y<UP0oV^*a@hUZ77#j'
            'nwN0FxL2}C|s1wDssBFB?!ShEZD;cR9d?Ag=E|8+f_n2$p2pLdfy|15_^menv{t^j#;Zy{b~8%}&vh#Tu3K_FuX*~*;C'
            'f*-HQm~8_INR*-n-2jt$p17@dB$jM@LRH*a2fgA0p}xi$dn-S~GfNVneZmNsJn=ErTPzRfdCow)na@}g-'
            '`oS6guZZMa|{f+wVEu|Sq6vq{h(eyZlP+AvoP=MDH#5+KUSAc<<CEA3x_8>#B<-'
            'SVAYxVU}o`^#VWZ)eXO5`pN*Da+l?zEJc5S@T@K*r)4{NLy*Sx^X*z4tW@U&gb|D5f+7SOr0mav?fTL9hsRc7jQJU_}I'
            '2l`0fgu+thxtXQQ<DwZ7HZH>NdoUX_@VNoaa3DL0{F7E$uf!aQ1<2}DLHD3vt=u=BfSV##|_5k_wPZ@qH)wkyIbUCPzH'
            'vTCgU*oN~*x<5Lne0;)w10adn0&j%z)D4^+eP!_<kWl6svSofrpOZxAZ)##NTHegm8z5QopM-'
            'olDGwpemc0+zOwLDhU|JQ-gAel0rS5mA8ORweV@*Yt;1Q~N`PVG>AeDF^yN1J!2`!tA%7@zO(cAcDQ=to_TWHrKP{!2P'
            'p0(@Bo0EUyHPUJ)2<p8_2#)FHa0f$Tiyi{~0FSTW5%iTLaq+)D+aSiLpI=TC=5J4Hrq=31yVS%GJV$7B5YQxq%p7aw!v'
            ';ok9yV7=urscspLMVD*gMe{)B&gEkCdZCRK%w_7rhaW7NUKhZv-jdi9Mx)+KK7MvJf$F9rJU9LY>|u+6%G3`SYO@i5a>'
            'j-w1DW6h${1br75vW4!soNs!qek3u{EYYGil>_96H4aZq&Gd+K3ubXH*7n`$(bR*jX6pkdMlN%E*Ksrk<-'
            'r;F(J|@!0HEd~VPVIn$!4$UEN9`tllU_s10I^L{8?-?@+Ct@*<DK6b(~?Bo(u8$sZs-Bp5Ba^y`xI;aM<ScXoQV*2FCG'
            'm5QVz;B;`%4T)YzBv+Y+b)pEh~wxjn5#x?3WfNPe4M)62p4I~Vt>sn94UT;)V$PYc0CkBa&IikaI~>LEe9SSJxyIv(14'
            'DhX52gF3AH9-'
            '2P7`KOG&xipeFmSCo)>S7!_{`Mo+Q=^`o;`YZY%`>K7i^PGIvNYNq3R`MUsGHssLRCGfy{Hzdpqr|wzJK-'
            '`&#Rc|a&{)99mzULdh9XgOInYJ6<`ya>c>!nzQnyQSd#|0c8pvg>@RbW`3?_=8IM0}fY4^>Z%Vs22^aqz3{(55|^Ub^f'
            'GnJa$=`$Pq>>ZaI``s!)?3D#Gr%NL7L^4&WUlC%aVx4GfI0~-'
            '7b<#BLb_ak32CY=AwVFg(c*ogKSfiOGWf@!LafW)9CO5@mS%>72F8K)%4wwJ3=a^zMV`*<t=VVD~=Hv%zq_bJHF=aGB^'
            'L!1`Ao}$%ip{!#874s#6e7e30RatX!#H^w4@+1crxQv4v4q32p#w@gvN&vZ}U1%7QkF_VlvEaG`B_p>B#Qm?}$;XEvi6'
            'mjewEg7W$tE&=@IriHZHS=@-dSFI^_>#m)(59r3hE#056cRG|2TgW+7+dv`MX#u-'
            '*yNNKf4egemMraj|M?iK`F~Cwg~l~7=q-'
            '|yQnd5F4pdH!)cDK;ITN5Tvc5R+4;vYxM3qp=FNL*La&iHTP7Q62C!lOdt$aJ9)b%^h)%?4V9CXk_{n_g-kV%72$P`OJ'
            '`co4rE|!p`J*5V7URNs9dL7<9vBX(N9&)vQF(L()GYOeZLif~saqs2Xcz!R8JFRBOAbC&UV&LP!Q_Qz9>koMq<^-'
            'I2byYy1v_OKaw!I4Mhpbq#yNQ4;Lj7Ml+~%H)laEumOJ6F!Xl!-'
            'x*pfqr@(1)1x8FvXGxWdF^2a~0(Y$;O3(a2lG+7xOYR<ca_0n2tuBXn>ASF_-'
            'X342pQBWnr^F}28&V8@RN86Wr(PB55asi}*!Rs`P^pgv&C`7tUUDs_yzwWgH4&iV;snjJ4^tE77(z&k7=G1`!adbxq`>'
            'qub^YQDxZK7k<_GO?(y*H-k(h!Bi{)U#_k57Ci-d}2Z^_=+574|>4(!eB!Ku~-'
            '<d4)szjf`bfI&1W72LqOL&^Bmra#DXZ;-'
            '9hOX2O(E4X^lPMkO=6juAqCMyyi!iW1yfn0FKgLb1x?k5>I`7?lM@16&WmG7yy+p4iK!vth?RKZX`7qZs^6i%^#497sQ'
            'x*dz#%TmZ>-'
            'Zc1<R!iLqO(b`>O#u3`Ayw)A04{r&K<)C2)cto}Xk>F1c~4G5`#udgxMnO_*K!q1_iPfJ0S~j%RqQeI`x(4hp#`5tX<@'
            'T<D!JKz9^7;JM!62}O}1^lh!2`(;Nf}}EUjk4w`6l%oLNt$@C_*SD#W{~VStkp!KAsSvc7B}W=^UitUJXxK4A>DeiiIH'
            '4F`eU(@*^P>k;rl1E~?C9Z)7Yny)a(1xg!tQb8rTL~lqNRovU}#NIV=to-OSTrTyKs?f=yr1bYwO!-}s95x-'
            'BI~1Vl)(i{`DI(1}g7tA+Bvzf;kBL!-'
            'QBNfsPM^!dsB!*qJas#irboeoCp>)7FrEy3W`j%m%Fr89I#}X4d$9Jj0`wbJhf1sreB7c8(*8s6XXqLHP?<yNj<^5_;|'
            'SHag+$TO<m`Z>(Ca>f>&Cbc5B1T|(EBH~{-zG@oqHUvE3Svn{&OIBPy#HVJa9UeW8*k&xc=1|+f7UGh-Dg1TsRSTe?3A'
            'upC|@0wh8wiK1rFF3-'
            ')dO67lMxILiC!O3<%ffs4nChEody@kj7P2z_8pmKGd?`RC)H_iIIR(r7CT&Zr?B{3@8%do6sdy+@5t9g3DM)lfU?8Q%4'
            '&uq-'
            'N2gB9Y+5Y{J*_zW$_DhI)w(UJmmSPV9<*#|D$%RpzO4anGEp~AiQ<DHR4(2`YOxngbxSY)50BF`3(8Ns@^k`s<{5lOH`'
            '|Gk;YH3!`OY9qd%WlL>*;*I8X7ONv%8CJDLlk<uX05Z!gC+?L%<@!2!>p2p$m!@HawF=}^ZlfM0?}xCNda&N&IUYXshU'
            'F9)NA0h=hs_^&ctR^2_I)2GIF%Q{d9Qd>7}fv-CP{<-+gY&PyB>l{!$Eq(5%9Y#2_x%%QQzz|nHjxI$k3mu7-#zi-'
            'L9M_6WU{tv#$&m4IPK{%2jZ8g)wCq@g6p9E`e(My^#CC5{Bfaf!n57D06edJMEhE@DUerh;x6IEl5yT3M%+E9T^~|c8='
            'K8p904hd1ORT1Bpm=M41{XJj+@S%X^>4Z{8W$`;s_+(ay2Z_k9X$+oEU)H4nsj(rjWqwt`$PEC8Lili<TlV>qIGinJXl'
            '#?>C-FuY+085!t^HG5sK^4dDQvf(HWnIlQ5#D~H7(P8}D&4-'
            '~hpAUMG$B3*#0{XS=K`$l|szM}@n>ZG=i*hN&>4R~<^a?cio{Opza`EPenS!(AOiT@l!eMKYaZ*SjZdw?OHNMf63l_wK'
            '_%9mUS6dPHW!os}kgL>%$}HAqqKpdGo1p*30{+OlNOXNT3fOccOqg3lm4|M`Ys3Ml&ARa5dn;*lcf!*jDu{7`5j-'
            'fp&ieYG6eIN_Eh8TshOM?wDWyF_$Vcu@kQjCX7hgDl);f!+JUI#KZfre1DfYod9x1q%tVcUiNR3r6pe9SCLC6sT{Ws)6'
            '-I+r`ds#w~dM5T=nF-Ma(I{D)f&<<@;_v@;0d;$?!AHTv(4u)g>T5@0>)Q$NJ=2L?c6vt=btdo`vrK#*UygjwdeAf~L-'
            '{yGq;>kxe8pRQzvKE8TPFxE%^3rawmhV+ziuXn-EDF4aATB9OC<dH_i$6LIOJk~uzfg{)oN%1?&0UigfcJcsZJU^UpWP'
            ')hCRmk{g$Y=ZVya5y&n>`)<V8UDmJLr5x0n$)Ed1aGPlJKlCz88YW7($8ybOWdPaOnCpo-vJqzrI=);YULpYIFjj{&oL'
            '1knFfAi5iV*X|)wRmSRj`2!DwwM`q-'
            '0TZ34RO?KBXdyR7>VQco8g4F4P3u?5I?p3Adag`sO*~oxN(3g^(4Xp#wJLUX5M{xdu9(TWyG0pdkXN;urOGfU&op-wE$'
            'yZKY~$8<<y6-'
            'o8a4;=j4>qF>3LPWjJxsA*^4$53HB4aQ6Xms_KRz3Dy2Z%m*HZYcewQ;k#?8)c!p9dHMzJP(oB483zh`$53C}wxPnW5X'
            'iL*LDl;^u<XD!;5)^_ohv!4TNQW6gt-Kkw-'
            'd@@#Y|lErHt4PugAo3JMi+ZM%WTN5tj5>0+q(M(fRpNa%gWcs$Y&mPQ_Q~^Why|W!^{_bDbttlS85ZSw1M!&dA<Ug}Ey'
            'Rqe<8!$kxjw*Nom#`IC8YrXH}j*;<%;IS<7Wi(%6_F?^MAl1254Bu1N(VC|-'
            'ZaL7>wU*0H3jz%%+2CPPdsr6(y>mW)8?8D0&#zPv*64FObhNS&%tdE0FK!j90P(kq|GtrkiGWHIfv64ee)&Y1}-'
            'iTlBe4xfHIgSQ%lJIQ75UjeL1G`ky$W8UP_{{k-'
            '7O(xt%8BeC^V^DX<Q+aLElFh!v#5eyNjd1nuO`cK%3(#wZd53>!^vXN{Or-'
            '$IAyrtJfQrVRau`3vVq6(iI*lAsg%M}_I=cz5r!{UzarLCWAT~s8L~6C9H)ET#S1gfLG<hU_{nMwIhq%PM<<>{^TqoGd'
            '-fuVDl5a<byqBlo(Hiy)*&R@??gpO7Kt>|#>%ZV<ff)1hO0$luh9|2;oN$>y*HK^>PW*|H!ez=aH$8cO>oJ+Cb(z3jvS'
            '>P6Q>!4cyH%ozIyy*YGp|RDLb8v{x6bA&GN0pKU)Db>=)tvbw7#X&BMsayn&^cHAtn(0rWIGhtoIj#!%1c=-'
            '#vr9li?YP?=HCYq13JI&_m&vQ&oHA6iWPFj+x8OB17`>bdxQ?|dAl%%}Fw;K0)3F8tvyXHg4Q?4qV~-'
            'jSELwonCvbF7wjEEys`i)^ddz_&Y?0=*SKkj)0ez;OFfyv>AQTKNwu)hq_@hNhCCEO{KTNgV37+~@cGwVS%w*8+9c%!N'
            'D8_EIi0))U+5(Xem=8!dtdprXq&BulF+FHY6MHv6%tzW6cj{GfuFOJte;XWrx3i%t;RJcc~IrVduGLtx8;Js_LI=I3VW'
            ';@lakD7|PD$tm5B+t(f;#&wP0@!FQK56_1~jXx-'
            'd+*8nanSoJ`YRr&(h2XQ+9T()Wv8;Uw&RaZ2u)mx{#uoeIrB)NN_;FwQ`JH08V^INdGf%^$Ps1^-#13soze5G-23-'
            '7fBE+_8!GbFmqzF#qMv0@8R{Js7N%w}v!C9#C>JU1+DB_G0uJEIe6n*KeAK87oKb89GE);Km#y9z}0_=8M!-'
            'zR**gxw%ReCp!dYA1<K84R>MRCN@Q2sck&tC*P<P&hvs(2W5k-@1-'
            '=_Da76rLv<P?gK_Kq_V;25xbHt6MwB(^H|;>gRFz@^vFzKN^Bx{o=vORTkUZs<0*L1SI$R%pY033sq%4ktt(#qGm%e*}'
            'P&Vcn0V3mAuSR_1G>nnw1JKg2ZuRVg;rb_F_WkiGlQcDPp*lCjKt{V3l(nv8=4cdvjisL1$)@IeD36(b?V9PS!A-'
            'Rdxu%b`|68W6o&FGem7B4_};b2AKdWEElZeDH-'
            '=Dlir9wW1{djtp<D2^FZ_Uc`Q@WLC?xZl?P@c?jKZx=DXaXa{3V5&}s>*{55dZBOS0i_yjK1o`8Yp*R!+>w1~L-'
            'QHs9x2sZ~)tfk7+;Q6*SzzHaU{bNVqqn2}IYSd-'
            'o6}yETs67O5EC==Mx%j?18#g{&4f_Q9rB&y1D@=d+!vckP%uI44bpK4Sjo1KRjh~aOW);4q$uy+0EHIFc1r>X17<G>i3'
            'ok39`kC`Eb%WrXoiG#=#HU03q-'
            'q$(|40@e%7T?aF|h5}R7z>}WJtVNNOUrdF)e!;OqPBOlPtGV>kq0UGbspPByGgX+BRU{FQV*2e4+8#0V=%V0_d=GFwaa'
            'CcgZs-R@?_$RO4{LS9zRarvS|-x5DC@CcGb%i3|Bzcvij{jZ!~TlNN@P>&xSyI3ph-'
            '7c`TT)vYAHVh*gemB)(>hVZF00oHw-0)9JgQgZXdAZ^-6d}AjES}D`;OlvU)*y@6<W(j&E*1`tWN^;EY4B+s-'
            'DApbXYswlSt6Gviq}>mbci(`FtwYFc@)GvGdkj|vW5i&NH(`5ZTP{mmjz%s=v1$KkaEn|8oS>0Vmh~K_jvEatZCIeDHX'
            'gOtx!^nFz2IH)2{ve@63(GSWVL;U;0H*a#5h4=Uk(*vwF4!GcHoHq>dZ;|66&MvMm&3dD`=gogI$?}@X^5#Jmjqar83p'
            'flF|fhpH`xk#YZo>g;a*=M1Ew(N$}Wq5Ne8t;O4p=thZP5s6Ka7@enf*itf^Q8MlIqxEWp@?m%?=uEseRN`T|%1PW63S'
            'emy}AkN_@g<F(hV*PE%xb=k<vyhFEcMn0@_XIpHQ_XrcF9r75nSfbhGI`@6g_OZlva3-'
            '6!v{X0o_B=elpAKSG}#AK>;MdSN63dnODYUhano>JYI?~a%4_>l<Z<p1wd5FBt=1otrx?O`&Mwxquj_Dc^ijwx-'
            'G&QxoTWDHl*cKDGx6x<!&sXrPEEV7Lg&tigB^|O(3EEY@3pnTBY~mR`2edFu8;>U_NctEFLJ*GqCQQ-'
            '6PqfMd+#z7R2oudbRviizCt<FZ-Jtr0K+D{fjX}=-'
            '0?I3$6Kht)tm}EtVN@4VFnz?%fvfxT(H<<AX>g^qcY9&F~Y7D&iGb9Wb08-3s0uh-'
            'j0WhXYRuk&Akwsco|}66vBv7!9IAv9hm%L78aI^(Ybl@%&L(|Bq(h(bUZjsHk6Aq363Gy>~RdHYukYIVtG{d-'
            'A=YHO2_>>4S*fI5X#cI79TWULy65*kh4ERK}bA#o16i#CI)lHFQwkhXohEBb5YDzmaft&g-'
            '@{s<k!UnWF8+PCkl>Jm#b7^{_QO+nP-Wxr|1Cx@ap+EI4+S&C|CwBGJ~k&rSoyi{UUrd#SCLpE>iZbS8!ca9iB6O#oyC'
            'F!0bl>7O3Y!&WR{oeC-'
            'I#af!vN@v}%jqwRP?`UsV{?gl0PcnKW&b(%V6mV}80O_aLHan{pYwV=H(8<)@gNEzQt#HWonsrZFb(0A`RIHGuof7Ndn'
            'R&PB?RTY<0Y7T>8)}$~T)?@=FW=GI)(M^bVnE?h$+o7+)4)_@hD81YjzrEbf;?329xvRIs`6C68?HE9YAIrjQMSJMKRt'
            '?_P?<2b#*W<Yd;jnY#O)NR^l<#Qo4^pA&R7ZO~HDK>aC|$fAa#t2%jFKcP#4H7rdqoTO@fxty`#2R_VotuL9w1+1{~s&'
            'w{m=Cm|NmQ|LQyhOq7)Sg6`tpA6j6~VQ7KgDl{V5MvbXF_NcPBnKF|FrJJ}f(DzibQ($M~RfB%E;`||z^9_N0}?VQ`~`'
            'orz9fc`%2iQ&e!SXI(M?R0|B;8Qdd4P_I#z)os*mdRQ@K28(2%fc(sOo;dMAS3tku(&@T1orW>_BgD^S~q`6*ELbG1$y'
            'M+aZa$1H-WXDo5_!=I?@(&6!pB%qIOIynkuLei6!^2WGR<c<X2_z(Q3lK*Tg}ka|Q1AdkZo?9z^GII82S-'
            '051D64YTe37;BM%;UzEOXxkZF`85cwxANokjg@n6b`<sn@1h%`kK--x%fMZ<o7Mi8pBUSTGZZ39Nw}OpLuCF^T)eoAS$'
            'X9Z@w#*!CLRv5zFVEa01j_B`>qE6E(oKWExTzk*JpBGF&D!O)e-'
            '*Al3eF9EZZE7xhHC|(xizD7qXy^*Mq@j+XS~SrD4(2L<oAn4K0#Sz&3qutwnCtFwiTE8l{}*A&?3Fy_Mi>I)nvLEP6C+'
            'H*Pelp37iOsL15j`n^w$Y*18y@b*-gd|*#zz3-'
            '#Po#l{R=|e2Gs>65c2`4I?V4r>=zA_SGJ8n?HwCm}bj~x49_Rdj=d^oo@?OfET<iLh!{5WfRk~~qJhvTaCIA~sjilbrV'
            'Zh0a|SY5>YsX^i=T1QKCg-NN#AqW?kq6=%<AojxpI{)iIkh^pXtp9SuwIj|DG@{Lto9_nt!<x|2AImcSa|Rb%JOksMdk'
            'I6uAGRJkNk4VWhb;xkK&zw3MV-'
            '&&@ULU=i&=_;hwkIbnm%gzx`mEE$RP9OMd_!EpY%ZDRj5jq!awgUKv$eiVm2p0its}cSfxRi7A(OB``sWs<{sEKc!BBg'
            'B+K;ccC?uZ!v%sh^!@s+sH~Vu9o9W2{JuHlM$diHlO#xccwf<1X$x38UnIb}Ybsc$b`dn(+=xOKAE=x8V(G<1^c+qhEa'
            'h=D%{hS)r?v1IzZ8}E){m<uZxhFToK)(w33YHfOFr%^hnyg9{BYtZEMY!}O10fgo%LFz_nr%i?s<;tky@~ZwH$5#JArF'
            'yB3Nuq2C-y)uy>b3Q~5`5tm!D-6>Es2c2986;CA#J*oSHZ2}C2~AUzuF0VA<)M1tQ1g!>D?eYqpv+jtr0-'
            'S9^Lg)8v%3o&9S+CXhri)*P$kFkdMH&XvU2)q7<QB&+CLaSpSs?3zp`m2jPd{&Q>H@%6G+5njKilL-'
            'm1`3t)Af4xex4&`#2#mnJ(SAtX;RX-XqZt3feh`hSBq&t~M9Fn~h<y*gmVrww<T^ZJG+kM!C7YB3tuo45J${z-'
            'vzxZoU-2uDA>sg9z9ASZw}{=-n#|CMkYoRJ9A#YodYGxg&8NlnCx8glt-xocPsynZ9aKxI8&(Hfq1qy0Epe$rDnIWu@Z'
            'KuMvZwceqrriog#kuCTZJ>%+R0v*JM{R%ZqjI+59J&tbhNh~>v|k<(j*FouYCq3;d*@K=g*L`x=Ri&@dwNFP>5HZ!VXV'
            'vHs8S{8X(U@JO6FO)t(`!cqJI;+yRa~i-1|<qj>zy0lLUxmJy{@2s<D2Q2h;yv{q?m!19|fkju*u-'
            ')!Sxg<W+3vz%}&nC*joJ8q&^7#nn~%*ng9Oh{TY3vU$9!8~IQ3NZ>8&BcjFBr}O|%{B~CzK`|VAMo3=Ga$Y@74`*v!au'
            'j-@N`@n3RpYCmKbF`>e_-?6L*PWfG*?=rGe9A3)y+?76f&5L-0yLc5YlGgsgu@TW`L@h9{>$e{C{eeEI=0?L-+T*Duf-'
            'laMBh)?5XZkI$j&rz$RZT7svvd+|eP5?VwdywMiGY0W<Tu-FNW4ul~8!aJ008%A#ihrw+=6)e^<hcNmY61C)r9%nhjaK'
            'H<%-nk4GM|-'
            'g`kQ47qnIrK%gm*hL8B}Bmltr0<*u@OiHMy^}OX)p$ZmNUoJJU32b0~H=q)>tMSIl6Y1&mh7w~WvZCNlCKfsYA`?)h7a'
            'Hg62k_Qf^UaWz%4Y`Y`s9Jv5*WDd~<*S5jw*L~!3P7IdW4-vPE_u=`{a13Zs#DB{Yp)-'
            'PsLRz1}PHzVc2V92p`GX|>cN3mE$;~#@$-~jZuQB$K397w(h+Zo{V)arbOv|=`KgBB8aPcyJSY8hDqD-'
            '<nIu)btCxb+q1SXd(#JmG{kj1D1VS{}jEBc67JuJpPlX*1AEe6=a^RPhd4TLSWg58VC$!D8REYfpleOdF2$$L{4#8yYa'
            't{ch_TDzGc`P&R)7w#Y&Yua#p&Nsb{MPcu;+vIcK70ka_h8*`h(R*DLOMI^<aGl!&nkV~6I}qd!@`nL?K70_|%Gx^N2x'
            'oSufsrC7><i@9I=<{TjVyG=bw54f)Z23Gi7X?z+1KI5y(KW}+f1bSf<Z`OgeFKVM1THS;t(N1y<Z=OpWzi`Rk8-'
            'mKQ=`5c4?81@w{x&(qCwkeF#s?g*TOoZD?Hl3xdnNp<-JEFmKh8fQM`-'
            's?7yz!oV$E=c(@zJ5WDbhCDNI@J(gOT=uKSo0j>Euv0!D{$U?>I6C8VwL)YKFM@R^ByeiuAF{l;5?2+3p*?>sRd?^h)<'
            '_Og_ZZN}R1axUDLU&ef>#?wSb|BllxyM>-'
            'C3H2aZXEFhv#+Di99x_YJ1?;`vq97bsa5)8pxsW?W9U|6dui3(V0iXxVqproVqxLi}$O7=xsAnda(*NEorB{SKVPlbqV'
            '1$_<*`epNZoNdtesCQwzf)JU%#I%hYodb{BHf2RvD@QR5!gt2|@GW(DKx@fiHzwnrn)bedeV3c~Yl1$bY%4`efn@Yrff'
            'CqhJ7QTwckM354sy!=J<FOJeW%QjZ^4l#JL$`8v|NMq+jJRX=;0Nvgo>b_V8uSvYeSIiQo+4Y+^5q23E89^v{&>P<ucE'
            'gV8C1}>=P2F#I!F+Zgdd3=)iPe`uM6nl_FD%7NyjPjkwXztU+JHPBrTEjchj{gl(l}a-'
            'JmOraFxyO)&oSV#;&sHC6rowwHCo!y2b-S^V#19S6t<fn$=$&q!uOKCa<qn@$t_g>-Etx{lL?Y)m3Sk=pGZV=px-bX_r'
            '41!dtxu5N$e-q$I9)Xn|BN^?~H`L4<VZAX=`C(-'
            'z{|bb`lOej%JkWxzp2sVvxtN3#ix);y}OSze{YIe~L$ok#QQ6%K}M$5R3KFah!B|RDk3AUaTmqW^KCN&RV!z5c4?7sdd'
            '{B<+IX-b^{4i{;-'
            's|=}kkZP%S=uDvf=%PE7Svmm%r&Aylw3rMBnK;BbsL7>)VE&y2NLa3}!>{bwliemtFOQX@UJMR;E!6aUPyPWm}{^iHV4'
            '*orHx&~a7JlzvZ^Ic|pj?$WtDKR{mwaDe`oF8VOI0mJW`puNFHD5+>eH7S2s?sE&XZ}Q`k+G2QlI|NuyLovkc1-'
            '94JV3^J$WF1o>w!L?uc)%L>Px(M5Qxay3%OU7o0;#*3jC=ddKuvBb3>6h2=e3vURLaKbp9^T#w##IwWizpT5~^7xqKm%'
            'l1zPH-'
            '<8b)!Al~({$3F%WXp@UL6aN^0SbKn+(kn8UT?X^~i{aMBK1}txg;V$H;AbBPd>s4_;#MVNnt3ycdvX%IwJhLNj2eQ=7v'
            '@2s6kPk&l9W08q=xC*7~5!sOGoNyd|(NY7S}@X;=pqp;f#~ZnP_~{7T+D;gV|SQz*wZ0^*zTC#x+(!#g)xeuBQW)HE%$'
            '17>luzy8&HlXE3464nH{U03Z6EY>4h4o6j9aZ?P<pQap#+9_#SIYbGt0_QMl<KO&ib6#8tpQrU=poOm7zUMJS#NB10n$'
            '+WpWNoVcbTZ+nKTKM7B3)aei-'
            '1OqBFI4E{oS#h|f`#5^fcH%p=7{ZtjIsbc@;Vh_s_#>Ws#y4<a|(y?InhopAvT{*<FDp%=F$VVU{Zlo>qo>T*7Z(zIJ<'
            'C;dE%;IYIh6m)o{W}a06FYKN##SLtB}jWY1(hNmuk|PKD^fN0vTZ-Mt<b2;PI&zn_rHd*v`z@(Y1Y(^$Ya3Ev;cVg~0M'
            'b(_5^c*xNad|dVu*N$M67f!<3?aN@Yq>W}ozs4)sy|^VbiOB8Zfyb#v<X#vH3^rN9-wlUhHK#H>t*k=LSu-epn8!#s69'
            'oa@nM8Gy9=UK{8gpzm;P&kY(Eg4Q{!^_Xdg}FL^lv)KbsU5ZHHoa<zLPZ8<{?d;yI04>W$@+L3Md<ON6lNG$X){({OM4'
            'G`q#H$^8#K_(=Z05m$zZ?u`X<xnnK0>GGJFxk89T~BGTz`AZg+ZZ`OC=w%P`Ir6LgBn8JYTRIzucfFb@Ui|&!&AUCgsu'
            '<qomLb>1~SoYs~+H&3%T3qwd%dVdB^}8>%_2Xqrf2!1Uf4T*F|J=v-'
            'd#V^VqCSw^auS{E%5W_AFV4>T;=ZqTaA4vC>y=m;c)SzDE8N?0dwmJXUY!R!7d22-'
            'I~~|qw;f}B`L&v+#jqtU4A^DPSlwd}%S8{PdHpc;bOz><v|q$Z*Bd5B?Qrmx6?n9j;mgJXEDYdhJbxts*WM)pXI>b_eu'
            '%>6oq=F<^Bh_G&x7udOu{9Jv2erm2YJ&kg8X)#g!_gq#D*Yoq3+P${1|R<zoQxr{&??QIvljrL~EB^bdpa4ja5F-'
            'a5)gP9gZO1SxOAmvtVdg0n5{uXw78%f}O)O?pzuG`V9*qLNN=M_yvPs$O-()Aqjoy%~<k6AMC0}am^B5oM*R`-'
            '1dHn_I8ixxo1PP>S{bZ8cAf`lH7x$Uj||6hHq3j$ek)kghE>s6My~r#q6+pulYay1HMo>Ne<=z`61B%zy87I|AYQP@0K'
            'q#S#lGUNF1TT&HG`Aq&u3-'
            '`I1$XK6&7s4m)!E$i>b)_*?Lje6ZOKf`5%byPy?!pLqzM=XpbYYY2Ayo29XackpIH5iLyd!q5I9u-'
            'T&?l$<w_17`b3c6Tp+mcN3pTf<m>B6k^o`41BTlPPj+NE!ZT{=Y&8=KpW{7b^b;^ShM>!Th`NprLSpH8!FOPZzzRa<+l'
            'k<Q)##d0fyv!wKfxf_Ur^8|JNAs&#(b0h;KT$mqmwOv*k;qF7b<^?V4P?r#9^FRM{wIS2fiFhjG-REAibIlP@wf*jvye'
            '05KU2KisWg=zu}St)nes#OUp&%B`brYrEt-'
            'A2nF;}~Ff5ZXt!G2VRhfCUcRS~=4KY?Xa|^xLdC#7~Iha&|uYsjs1K+!`=uvzpD@^#IK*e$s8vB`}Pkg9TIb*q48YVqT'
            'sw8V+^S9w#Ao>F6|Gb-F-AWjVEu>b`)bzm4Hk!8)z%Lrb+DA36@Vze+%`;01d5-B*}o^ujoQ5AYv<0-'
            'pV$)b+qZoIWiJ#rJ~9xu<JEC}o-'
            '`ITxa?xd*7FaAU$#18`K!hOvfG5cHS9&vo^XBv6lrd3zzc_8#&X+=k_UT9Em87smD50ehbd?&V5BtFun{XLg1L+w6n!l'
            'hv3|q7F1?JAA}tczTXkg}c^c+SU$SzvMLLHB($yI3JkHxfy#7SkgP4r{G%G9oFWgOX%@)EgU!|g+b$WME`yQ?zo?b@+W'
            '!NvRA_J@<%`P7H`G;UmL(C!x1hh94421)+2vP3Me$OF|+F&{Q5nNF$HI-o?$UL_U$o=Y;k7A9J-'
            '5d`fcI;FBjUs;~wjJ-YtmYD*};1f3R=6i<P%37>|1HFtQk~@H{Gz*p%q8yp@yir7;^)^dn$`^3u!4|G?m6I8_>W2ECJq'
            'aW*L&W$jg9ZQxdLGY^7a-{WYvp)Xp7Br|$F{NYi17-'
            'Oivjn4C~#;n}kuxKwIt1e^<cApN%^im1z8XrfgEjg^VnAM2WBh=j5ANKXxK%;&(t{V%1U&>Z6e)$2^{x*Wdf*5$W?mC8'
            '!zeNALN#u(|C@dPwguk`!aG*>BOhyd||H=?teNGqxw)<0E$BlF>#S4_bsnRdE(r9d8Aa3X}rCz=w5IB|(n|&=A)-'
            'T5ColyyzHWUlyx7!I>yBLc9mcetKDYE`p3LLuXLEH{W<KNYr@N*a!+naAQeh}P`?E{Ta%<0GcJtR&J%7){GrBTRg5~CV'
            'tI6-'
            'I;6L%IaLxzSnkrIdph0**uPKYF~B3xK{s2s~%DB<Cb0nvmU%!;`S4=gUh;{A#E_el*_mImO>BZ*|7Tb_uGB|>F8!ZE{i'
            'V3?|a-'
            'tyn1oZAmBs(az%%4qcc5W&h?;0LcGo<dhr2_yZ<ePVe$5iXl=2m2XYS}7AtRSj3cf9F<GBe8tyYqbWFwVL7H$8cP={~<'
            'P6^uSxAe7u^Z3c^2}vCnQdN$d?|<jTilR+Sm!`B5fhdl_gLA4x#9j|<_`8YwW=IfS~&1w=o5A;~{<3yb1AaZ6AxdGsL~'
            '1)fLXd5P~-'
            'Z)Gl=Yqx`qqg7~Sn*pM_`SgZD6gd28gl)|VP(E@2TrWK)^5;TeN;C)ak7dF7P3fq+@iWcNEh9bd0wkA57(`UVKyJ7TW+'
            'rxmzGW<tbl(h4{mbb?B_lfHA_5Ci>k(t?SRZ!`;MRH(n40=QpP6-OY`p6PuNmgFTHF;Mxc{SL4zKCG^f|5$@T3l-'
            'TY=AaGoDgN0aKYEhCxs)Xm?JNtQBE+{$deBX=Df5g9+HGPmn_&t*CKm9W=)9z;Zq>EH_yX8<-LJd-NzPVDtl41XsZ6ip'
            'TUo)f^+9T)_I!|Az*x4J7@0c!~M=Ps+7Bfo!_%MZ{F4==BZ)Vbd$%QnDsIThC8zZBAmru0Gh(`URX7C<wH^g0_-'
            'iVo+Vf_#Es5Ma9Q4Kt>I|ILSkk+#XPputOVJUJSeOil&BI(I1DJpg>oGZnbG;SjS9~2I+s)P00^G|9r;^8_$F_$OE4&2'
            '8el=3I3HwSa$UWq<U>5{_lO!=j9l_z37GBwOuGUkO7AkIxu}Q7)}uvU<f3F+s;H7EDyyj`9_F;N~vVsAH1xz7bVKmQD%'
            'D_!|7oZwEg!PKVNpk-69J>>liPry1yKpYAdn1W)Xfhjz<AwQ(S-23e5NvSVpJ6F*(<tBc;m8Fw-ed`C7}U!!B3yHeU(8'
            '6%+F6$^lG0@RGX8y+pZ2cM{rl4dg<$!8Ct5DUAEdxcu=2n7%Edn(odp_Dl|6#MWcj&KRhiu>dEHSh~Eb5GDE2;j=&;++'
            'G($lxv=leve7A*fSjBE?6<2bVcIQ`6f&+ElwP}vji7pS7E`eKoZJ0MNirsfic}7930+^=XGx3+G>P<fl+hmDgs=CzEHi'
            'dNEa@=4f>N+=n@(ZKIxs1C2$=ds)a!9rb_r)nFtR;Eg46ZnAlNThUd=B$INmWB9vcGgWkB&_?=qtaPTA^=*nju8@9owK'
            'rVLkxAWvjb25pV?8Vs=UZ5DxM7ihVOtURja3Wt9WyNlx3r{*^zAhvj21Sr2%#DUWTA{P84$psJ!-'
            'kk*j4l;shqm2?Sizrjx(LD{PJO7m`J3E65ePH?`C!Ea6Au@f;U);h-@E%+5%adf#Pkf+D-T0Ym*3d2>=-'
            'r#pC_$3kKp!656rAuMOFoUL=LIvSk)^4O)^YaX<r66hGH<TLl!nPhJ%7;7F}+446iRrh2=Soj2DE^LR4hk^3KMR|0r7b'
            'oPfN`^Vx}hWtik%h0@Y8*gdDQ8lxmkx+crsO``G8esMVY&>YwK{$$0)CIjd2F%W6iCo{r6Q00<}t5nue=MCm$i?$JYaI'
            'BXxwILT<mL<bu+q>AW9Z3Y&mp}yT5tI-OGV7&-'
            '`Q^TN$3zk)+mcD@qV3q8FH09?_(5&CBplv1pY6~z#5mS)86Qa*fP8%d@Ey5`YsF2WZz&5D3m@VV2~+&E?-'
            'DujZkjfnY=c*@6$=*?vsS&bq9IEB82c#>j7?1Gm)D$Hrs;F}w(lOjEY^kJBKpXiB0;$RNDdWqR^e*1bhMCM3S0u0!Sbd'
            '*CYTI?O`RVj=w%8l?pZ}kjIZI5&NDEflSgj8NykIVN2%1-'
            'dpPR%omghM)2(`WaK?EX#&829=4V1<Vjo(Z7l4ZXVpK2AKt1Qz<h*VRjA(e0CC{X^+U<XWdsQ>@)R7vjZ4(9UND*L9M^'
            'T%^Tv#MB08;u$gV>9(WSJRyVF?{mccpPsVKA<BntA(P9qU|H54yg83&RTa)c<G~oo5=)SQc>}{ubNNv!NmMw4VsObif7'
            'r4w}J~I1_2qe(V_<fu}>6;9=1R{L^Ld_9dlNB6qRWc&V0`V4>!Ux?Q+TYA?QC5(Fu}1SZE5QOh_2Z;UG7o6fbAKRglFe'
            'M<lcNXFKkFJXgEKGt!SP^G0~^eV>!GWV{dIEM`loIV0hLl5E9w?TYjQViE0=TPJ8A}~42geB5`pr6NwrnUoQAbAC5ESM'
            '(pQJ>M<A{PeBYH&?p1hn4$NL;wt;GebwephAzw@oztSu;qsoQi{2B8lL_s3m@vB2mKD11?@UggYG$QgNY1Y%@223v%7`'
            'bZ$I#&5lMiWy4{wi)4n^hkkJ00h0HWSUT2bAd*+Yh<aItv*}-'
            'mD)%yYf3zL<rEg&ftc)aWOOw$qR|%rM&7ka#HTKDcfo0@!91*j@FIr|WTjq{#MlAS!^DH{H9>q00AE9CO6%=Z^3_|OK;'
            'YpuA6f7%*AMY}eX8Mtzb+5py{4%u9bH-'
            '*@N%(HYgH1a}SWdg<lN7TkII_YC4j3x{BU2Pbbw3eJDQC$4<cvZR!(giVk_KOFC#;cHI@EI-BP@Ne_1!zx!iOU`QRM^`'
            '_rBAfT|>C^m<n#mTt<HitKo|A#pph>SS!QgJdBp`qArBc6v-g`eY%12HI)-'
            'd!#i}rYXiK$B?vmstfwz#O<0AW)!2a_>KRhgX)u_59{Cd_U=4>S?2j|S#b0j2XZlX_WJ(QgZHYyfskf+9&L$-'
            'svA{h~jr!_8gR>F*81o~P+}hxcN)C<ClEDdw+E<WOU0hm9tFGe$gLLQ;U4}IsGoUVHOv>YWz_KU`6W=k===K<foEf6V9'
            '!^kf--`1Lb-?4XHF*4W1EU5({9$ntubQ~hWgeX5jp=(-)lSEdE>BREd4dLa9-'
            ';A?Vpd@3e^Bb90z<28(Ti6Rdjs5X@NYL1DI1g1f)szmr(o+3C$KOX&~z_oLSxN5Sj*XlWA$8gB0-gv;4wk$ZB;OJ-'
            'W+3I;KvQzaxliT9D{Vm$Uv74c6RwgvWp%(uDA@Vm{TCTBo)%;GGByO8mu|9htb9xMB;Tfz|W)cWT*5^40##>Jy~h+;lx'
            '3ZvrDI~rxwH53>#YRcf$_5)7a#sMW3FFWy-y;WW9@T#_eJXY|j1ls2CE8Yv*gBy^=NjiCIhA#JcI^U?UVs-'
            '@*Ngh1gj5lx5D9!<@(*q-O`iVf64#R`J?On$A;!FE6yff8u56KPp3#|MNpd?+^6+a0qMnSu<F6b2UC#_mFTMtY*EQ`U-'
            'Dt8yJ<Rbl|xlV&?`??3?Gvtl#SeTTBhe2mRZivbh$22bVD79;LG6H}exRnvGNbJh*A!YO=c_f)tPTL1~Er^=;bEm><##'
            '!Mu0CHz@$}(!5}HHyd2Et|Rx?lZ;-'
            '&a`H_&f%$&%Jrr1@OjJvKXsdV(4LfcLYU|#y9_d7pWlKE4F3gVgh%=1n?^QuMZvom{yupj<7I;|o5pLRVjX?u`;K;QXZ'
            '>&5@<%}h9<iHhhh<OP2JPFh&m#`c*jZoXqLMXyvO*o(QkX*qAaE<@YsulCX?toHY3g3jqLR@%$uOBWn4#tc(K2+Dh2tK'
            'yGrY#9_xaW{0bzLTi^?UuGm9Ya?T3h0j#vk~p<pZlN{ZagN6FqPEhLo1w0v~5{+`C;6SL#0^JJQ0z@WvRZs;mIJXQ#pS'
            'fj`I}Ifx5pn#lZ3wy<0JAS0Q%o^Ct$p0z3T6&{rNuDOMuMHb~wK;w6Y#=WUJNG<jvrRn9YxTl)HWqSiGZjZ5c2Zxah$2'
            '^#Oe(WSuW-6G~TS!-'
            '>7(lsU1#vyP0`{~lz$<nKpoq~AvsQmJxol!#Glx21<h6r;d>AY0?pb`2`HuYe={|mU{SE!sTCl__8h^#f(p%=GMDgGTV'
            '4iot-(iL{=A{Z=?NcM_vdQ46><a@D1*jqtfwc)e(5JQmH+o)$J<sAXm8S{U_J1bTzN<h{WiMo}SO-HB{P4ikn_SgKMua'
            '25*q|IH(|J1zF0%xF#{)QCc7$f;brHKK;e@L~klb7qfKEHIK`*t6u%i=DnE8@?Fdql2zPT*i6-'
            'kwp#NhJyXJTOfk8Bf`p;wLg*eVX0xH7$j^;}~=oO3>dPkn9Sc7hu$)$@STYe|?J$BokKn_xt=igvJ0VN7l*{C#jATmR;'
            '3Hari8gKcYA`^|ZY*GEyfUG54$FR9Xx`RfSJuOce8rWw-$t-'
            ')3!88e>m(^%R6owff2iz<|d;GT<bpug)S<%+zD!8BVl=WY}}uOJYlCIVHr6Hrom4eQYn1svWGfIIgvF^q`eDlZ?#jE^0'
            'hRGtquhD!8AG9Mg#$%JQ)oDgZP2f{p?VSmmPV>b3Z<p^X#;E53AdXs~)tPNP-'
            '>J4wdE`aa83>0s=ff^;gz!`f4bt?TZvZf!R#Z~Z=h!Ep~oECh4-GosBc^F^oi-tBp`#t_?M95{Z;_54KufQHutdGZr$q'
            '?LpK?}cq86ZL@*MhP8DKJlwLy|2A3Wi)-'
            'Vb8X~`r+%yGEAn!?YVewqcn=GO9TtKK0GC(1v|x}=*(6_s4v=us}G!Egy$%M*^BF7madGIa-TuaKpdpamqNj=ZiJjp)O'
            'uBqTqoslMx&Zc*$rUH!5|{kp9}dxuV8okbvU{*m-'
            'sg<B+ofY$*Rviu>Fe=nP>PGBP|5MP{Iv<z81y5HnqSK&j+*jF&M0+L>3pdVeAh9^uJOC*T$=9SJN=2n!H7GaZhMz{aVe'
            'bEe%829hjgb3o~a<lT<H7C|M71Dme!P`@6`4jZ8=@=E57&ZIF514|SAw<M%bWcudHLn0)sD1Km5ASzy9yur8u^&y>)ue'
            '_|Lm%{w8_Pmq0fy&P#jc@$4?|AxJb3Lt4}EzNp+l?H`tfwa{!T*cl(3L9LoB~c0Tm-)f_#CcG%I1fdg0$Jy4ov`3yHm-'
            '2J$#6elMa@)m>4CmtFnirY6Xdj!=eIfNHs1srb2XH4y+CA*h1nVFLeVB{JC>hor+VA;F+R%yoBTrO_JFcHxxUcEC+dLZ'
            'A>b<$0lddb>GAk4G&Jr3jC~Ph(!Ykd;|M=#6RCj9oq8zuOq5;_It|Ch&yr*=Z-'
            '{n^Vr(*d1LhqGpzL^`{uoYxvK)DISup|+Uxb3XR6H&wQg~J40Wt0BA+x`esgkw`i17%b(8;~{bX^zx?x_dyxvY`lvyhy'
            '-V*(BVvADt9mQj=a4Y=~QW8Bx}#8bfn9iFD4#(2ma_c+45;Zanc9|@^EYhZI$KbbF70s52)b>9LY?mZ`28=DDxxEw%gx'
            'Sf7fIYIPZEJ4<MdEE4~g~%*-hqwY3m~}6Pv&(-'
            'lE`7|Tp}yP5OoS=i`Ou9KjoaY*Xe#`!d&t;@cfd0yNmKAYL9$e?9E{%NqToa|Ir5W-'
            '*mx{Py44fb$=@UxZV3~nrr><#1LOTTo6&3b48T&ls!Oj3eC=jwoy`jB=c|e-'
            '9PQXx;DwP_GvK~!2YxVCLUr{dbo2d8ckmk0w0Tb;ZB-'
            '&vM?Xi^>y3Ec_9!rg+CVZz5Ek`ZqRn!zA*)3j)BW7&isx+bV@tsCqV=F{Tu*s;1lYj?JJ8bgAiOb%!#5E*ME1rLjZLc6'
            'IAC9c^6FJk;IIY1?FuE`=}XD4Wm&LWei&T#hXU&-Cu-'
            ';zkx%0nQD5Q+dC@dOx_SFig&reI_@v=fV`lZa**1p8$3Sf0)WY(0F`(@70LQ}DXpXC^(*Sh}+QAug_^=7J9pQjJBSkdY'
            '?g>5KZ>epS6yxHdr_}n%ZHO0nL&h!Rk!d%I)*)>CDf$n!-'
            '<?JA>k*pS*VS0V8q+vGS`)mQe^G<!927K?g`g{%*sEm+4<8z0_lXE#ynF;4Pkf>6p)1IQX(8Wu3j7zH0((aHp;VhQcs?'
            '8>CJW=SYP|ty#PhNn|M0TKZ}Y<Y-CS&Ov-'
            '^0?`6!0l&x5C@F5|!JoLYU}!6apxpIuZWL{?gVL6;U@E&JVBtbwrGH0X~4G0;*#`!p}Svdfvs2>Fw!@xKi7E$g*DGI_M'
            '>Y?M*SX}K1(^J+Dp9t8OpH=+4W3=VNzM%(#`bkX=Hv|SNNJ=g!kLokkRWwPx1qIKAF;{^s;{Y2ZFPw|(C2P_GfN40NRC'
            '_X<PY71g;c{Cd(SDJtyN8j8x{}BcCYIHC(N7rgCyfjmX2LevwWSfAxkJwLY>DSKKwQ~t9DfPo8PZi*8r~^)n8v`Svp83'
            'n)05?Tm#&o_Ckj)H0E5WC9BYT*UZ~FodFA#^1@efGQf?jx*d<o>{{NV7a4740MiVMDVkbpOzh~wwYv}F?yyINI;ilpXZ'
            'e|Z%8&C>*4=q5G8QNZ65K<{tYpu=f)&{e*L7Iq&7jT|LBy`>r##CDTF&nmQO|4K52GNCt8fG#MCr2IBl;c?e1;uWoo2M'
            'rqO+5HvBbK?f)7r&!fMORSpg)poymcX>j3vlh~ay-~{8D;MIf{tGyeCOtc+E4{LbjhE(K6Sy<^Pa&EJwkn-8-'
            'Ve?G%RI|;%C1&`f64LKWoPm$IJidjpH56uIsX}tj-APYi7|`)dVG!QqlcN4LTiGA>ztL5HZBYbirlldgcma*Uc>izpa9'
            'OF4B;G@Gl*GGk3>6{Uhg%rGVb%W{VydV6VB+jH|UT0z3-C&jT#vJh1|Fc-%qg$#oREp$-'
            ';1%s~46E;=&ajT6nWaEio%XLA(zH7#Ylo72a&>W%Q#IEDWCtAT%(b0N>}`?x~j8fhF`OpP>i;b`Rr9Gwwl+}c`7ezi}N'
            'GADjAwtyctDxRlx?_9yjW-%QVdxd<b0KUa$U^d@PEZy%#ETUXd+~+FpOB!SZaYjSx+k<pJFB0`@yTCy2BjH-'
            'S8FgbTV2fQlsb0+hrfd_sZ3$xB=>7t)Ei6H-^%Z#`-'
            ';DfqK1AtXH|F+uL7o>6^ky4@V4^dT|51*wn^N(#{RPdo?Z+_Xv^9M^^c<&G?8DmQwN&byK5Q-'
            '+rguB$&OX#1rZlgC(tpPw!L|Ud?|1-FmXAOxfWl06H)PCQ$K?gbU}RS_%dsF1CtgmFUh^8r5>x_-*duUf-EJbqvkI^3S'
            'Hs<oEb_x(1e}gn!se^r$T@x~IO^AhmKI_-k+d2tD)wOB-y)PxypD3OIMC*AB<#%hBJ<<C@$pf8@buk)0spLF)X%5-'
            'zc^i7(mzEPX8$13QhV_Du@*eEOOalaPoRyFD{*C13Un^;!a(dKzt+DZqdLnVqW3n+-'
            '%Nn?9pYL+;~}8Snq<7OH3CI}4Uq2BOjF)&*1WD}fSzl-slTK!{A>*;cJr>02ftL<*DUx^^66`m`IMr7xF4*PzJm4}HsP'
            '9*e{^f0E_8Ifrm36ugT0m>EAOTcV|;}Z<4D|Etj*PehQK;lO>d)|RD`CNL>jd6&E=djZHB>HLm0~q!Q=kg(0tnm5_J}0'
            '*3m7f{GTMsA2vbXEoX`KU0$kqz?RDM4bxu?1I(P*28`$b(GyvBppLs0`MI~S8rNH6;jAM(-'
            '82Ml*CjxvJRSOu<$}hh)hwTt9?<7_g(?)VF?;A2&3Nwvmxc;3s$-'
            'IQ=dCDQc=(r!3_DYwelIjk+Ktm&IpD<{Wb$131lsN~_%4;3{vJ{W`R6rg{iguG+UA1I!WFouahiPfKMSvw4`B4mEX++G'
            'APw^pC{Gvz8y`2I`>*?WJE<1C__si8?<|$)ct&;}Z9rx~E_N<zLDrUO+WqJL9J4t<>SA6te`6_<a0$9KD+bRMxnR?>H='
            '1hg){O1zbJ0EQFX&7Z(wM>Bgw+&*oYpK5Jo}hd20x(Z&gdg6I~wvsyzzp776fczQSU`MkRqQ<&CW#N@dYKIv1cE+^_nx'
            'nm#@QrvS-Pj@2!ksNjA(hRlv2C9I$478=7sXMYmyItzx|)VA?GN>9r?7Z%GVw)|Ww<P!2rz*#^7A2C*6a@#zIY6j$O0d'
            '%jwhPv}DUkQ4=){`DxoB@MK8-6KwA!z7Gv2izOcNB!v#*s!1wHG3b?rwbF{-'
            'uLfx<83B39*ltx>*v<ftq0snS=gOc0oz~9X;mm5SqwXR>Pa8H`OzQhZ~lbz&JNt8V~+C1yx1b02EXQw;Cf+AbXn_3UR>'
            '~_2ydvdBO6`k<zd>h<CuPDBMH?D#L(b&SigZA?}m5)=`O&aeStI)Z^7DiGN|Ggg6&(?QP-'
            'l5zQ=Y5mosOm*_}k5Z3rUPTjBX(D~ywUh6`>LVN<3Ik^7+sb&D<G^*1{r<+vZhy5ynEfRKhUB@Ewk9%iOv(R*MYxC`Wi'
            'jPp<Oqx~oZq<^N*jwIlIMkuCc*r51#N#I?RO*+ksA<WzfM@LqnaWaxqlVV`q=}n%$Qi5`|5pqxJE*w)khS@wnK*Ebfw{'
            '`7?fh>3Y9@#;<R35_p!C1J+6^1n{Re<Zg2jICBnEZMe3rG6MkWLYmpB{rsg9aGzD<qk|x9HXTZ6u?20PJjP;N0=YsJp!'
            'm7qzb<W2vF^>y5j3FW(i_SFmB}L_IvV%3w{~HPfupYTTp{0Q+xzCY)pG^knELIrX~$qCLz&yI-0q($)vJT-dlI{UdUpv'
            '4c`+e|Wie9R_qbfd1rT(D2+z53ZQ=OSy+ovAB~q#~i~)N!s+`v$go-'
            '><b9wUjW;c@8F6^7R~DKhZ}0{plani$A2Z@f4v!&xaotXN+6W4af02N-$*Z>!YMyBqC|=zD&CWD-'
            '0EX=53h!?a9_O7Jw&DM=TYCu`KZI#hTB{wNsHMryxdX`xk}qHh-WW&oIV5_ms;SCzbo(-qYMl-'
            't%YwQUl{y%F2XW}3g}2(!PvEg4{t}-6YUHp3ARyW+`a!1>=O>aa)l@4>T4O;`n~}6<-bFR=l3+f*>>aKd7FVbb_#S~4-'
            'jpEyCg?G1#(tM<MEY^aFWLjB7;Y%=~y65|A`Qq<_?0DQ*^h>esXP&t=`|`AwzdYu-'
            '9%KE6TG7xpRfsk{2Jq@cj3rNG%VnD{Fu!>;$Pf>`$>ig7A!XYq~z{W^$gI#^g+1cCWu8{yG#67pA*4FX#M$C(2d$_P#E'
            ')$`NA@CD%a5)+ci_wh^^Oevo`|kPbzaY3^x8rp`)Uw6msEuqp&hS4rWgt%aa_q79g(BY5TP12Wheh<)m!pcMQaPo{6dl'
            '-Tk)ok_vZg|~3s;Vwwn;7k_#i!&VGZv~-?wcr!hOjn*u!QuB(s4*P}Q}W@k*}fNIV;7N=-'
            ';N@0Z#1OsQbomvR`9JGAir2lw1cJW<X1;fB!L@i>LgL9(1&i0J`7Jisx)&GmqSbwFI;g?M$;M{kQ5e0NfsAFK&}y_bPk'
            'dKd~U%j6AC;3*@B%L8}nNm=oX^~Fg56fe~K)T%bEwGRd`qj?*_rCXJKTA*<vs)Yeae90vLajf>wF~U`@_IXiy+|Dsql7'
            'ty_crexcZ?Vur6CQ;?{1gVe9*@qG7nn)+{$w0#BGGZX~IMjT)#wF7hST42wt7`E9a0$rqyu2DvsHuIwKT&oSTx)PzxE*'
            'g}!slb?lJp6lj2W4HqQOjNYXwm<IwRp)++$iIYtJN!UaV7&bzr~Uey=<J~@yAa+3qj1a5WJuK!1CTmydIg20)tI7XXz_'
            'y7w-VK+*e{NcL+=W*dgk1G8E1w<>CS4fpuFd;dh)5Ihb9I^UwHz(mE0R^*x9@TMd{T!%4aCk1-'
            'wc*|=i)KDhaH5*6Gz>E4_WoR}P8ImkGnXXh=Fy6h&DIQz5qYurVZrv<1JcO1j^l%rHYDA~J86F(hY%+TdzLfXb!;N#(f'
            'E#aMz-KdPJZ)@<eq7{84;Ez=(uldDJA6LkRki;E;?A7j=7IpyR$C@xry8-'
            'nk<zW4iHnd$YPK~Ocz&rbDBAi|WJKyHf<*j^LJ*UmldEz!O9w~sYqb(R3tfq-)y-;1LlOA_W0F&Wr_`0|oTCRk`=aK7-'
            'mM`L<IHr#QSOxYka#%i-J7IocA!c0&L#x0|Xi-'
            '~>mR47R?d1hwx&BmmEDkCI^J$iT8##OZEg5k^9NU;qwQRr6c_=S&*pkW!6+Mr64}L(6o*iA+Dn<UTo`q6{3jF@m4>+IX'
            'l1rEVkY`KE$#`THO$pG2LE%1_sy;#Er{2=QSBoIgQ4N`2Z;+Ok38?*X5j`;QgNE#s!B%D$*?1?4<tK3lxZEjh2(QQAE@'
            'Nogn}?ozMR4)IV3-MA3GcWoFeAT=4E|MwIQxgpm*+e&u-'
            'OwTS=*7vxE_AnAA#*WyI`qf9yL3ghAjhE87}`mkWZPxaO&g;%~M(i`I%;LbixnbH-_T+>p?K^Jr#Qj{IJBFpFP#ih2LH'
            'rXI!&9K;N%Tga4ZQu@H9xp8=;<+P@I`cB?cPNfnTr++|qt_BJwBl)$f3$yi)cjz@-Wqxv;HSof+QU8-'
            '8}>y0G1U?l?AdQ<U<^8tntpBEOqdW4tkIg#)9L)P;EP3X*Qrqb_P>50G)RKKv1q&N&QxS*OovPwa}3I--'
            '$YDNz0RHD#xg7$2S#l!@2oCsvm+y+$`sI;bTFG5M*vqU({C!)3cQ7k$QzD2=KPAa_@;EJ;!owquiR7S6&0n1Y1+Q2J%C'
            '-5$o*B6k#(<9XMdlLh<w%{gDcb3{=Ww>}ziQIh1$?gn|LKRvC#YV*-R=NP(r;gFBI>k`-'
            'CKrdYE$Jcq3!wP1oS;?#O33}hC+hmR>6|3)H+hJsrK0hYZzW4ge~zi-TjtJj6EWpZN3Gg&9C^f|elwBu<brcxbN3N;m1'
            'UrrcPJhgm|%S|GlM<<snKIjbL{3{hzEi)K-'
            'WP5NoFDbIdh#^q^tsZcVij0*2k#$VpTAy$;I#eXX)asB(!q>0qW`7aqo#skg?qXWJ<%}!6sp<Vwj4N1Kg-ymWTDGJ;Vj'
            'igWhW&j5s+&pKM72sqcl*qF_oLtQ+9Hw;f(uqJ}3mb#U)mX}Bxw0F{SgX}E$b@~xfoiCz2XB54)ms8B^=X9f(al!CcPA'
            'pK8M864Ivgs~ra_<Br+-VybM;~L>q=k+BxCiE1l47g$YWl^yGuL76<w-'
            '<LN%!lrZQ|Kb=0Y&>=X}zEaOx!Z2yT1E?P5C=|vUNRZ&6@?8<~vwXnulw-V_@&HNECjVj>hATr2DZSUi-QdGGmrtg?=*'
            '7vmy{4{(|guQKJXd*5d2+UNHMffT%idf(3s{XfXc?GTXHgF1T(+Y5x(pAu#7jo2!ApdIMhn_>J+zHyyjIs^FsR2k85}o'
            'YAwR1|B5^;LuKjN#Q%Gk5nkS_{X4$<Y(+vd__OzB|-EDC%kUa4IHmLA$hG6=KRP9)y9<!x0G7)MYIB<?IWSR_b-'
            '|4>?8^*B_!1J3^jhD2u-2pAic8?9@@Vp)t2ex`E)kOD(!$>FIq8uzzbPZoydRuI*DjrjqB{Y>7H&cPzjd7O-'
            'lvf<U=J``>dMv(!v@449v0HzRmPM_)V>Pr^xWiK>D6<1$%1KBHWl%impFL(Me@e1N9VuvquWQ*iX~%LK5&uJr2ivN2<J'
            '}H=%E~Fg5pFiskE*h%ze<2Pgw8Z#<##kNk0Gjt?FcWrOn4PS}+4kr-Kcg7Us`Vzqx1teTF%)`V85UYd$r9~JS6X9t=n9'
            'Y@K}b`Z9BAD)dfg_;}o_^+rK^2PtLP8c@A3w0Sh;bIDbv*%gsHs3>zf^t0i%Yw{4s{mnLZN{w3H==9T2xeV6=(yXHIWO'
            'S_3rEAS`ym@917_*)m?hMIIt<oI9{6GF0H(e^i0cP8;_`^C5V_MAUfc?S_NViyB(n~5_MV}A%kD#seH3<Ok5K;odANN#'
            '2v_h%<IMI}GI>4^wC`79lHUa|^peHWsX3$?FoCtpJ@8}J3an@XIHJ&mViP$ybJ!mqZwtl%Z&SP=UxJ5^rJ|2)A+#Ov2B'
            'VmM2seFBnes7&)>M#_dWp#M+W`ZLH<IJ4ykMu+GYoG|f;{0OEZm(BA7k&}fkG#&R6Yiq+9jcAkvC+#-'
            'v<sdc{u!RE7dYi1ufk|{1K^*+tsJ3jLS0cQHjH*q(NB2io`pHQ!L+Ap-^WKfcj0IuyCt0T>It%s{du-bSE#B_npM2r$5'
            'n{=W%GbFM(>T%%zoqcfdbX2>i5<km%+@%p30^VMo8??%VUo*U<#%>Wn1^cW;5#{&naQ_Yfx@BD(jyXIxp(hhbtTao39)'
            '=($;nr|o2sQ?VBU>hI%Bcn<Ee_QP-Q%W&Zy0>$-PQLs`QAInNZ`0`+IZP*MmKAen~ubse9w}Yl?^-'
            '{3}3ou_41B&y5Aaao;)hSPbokt=_xrPeH3xuMxqCJLlMv&MQ-1M-4B-'
            'Jg8L}lYHc$>=x{hO4W9?gOqPTIKn)>i!2GC_rv=fhNNBWa%$(mL+vq3Q7UBFO1SQO=wp+%@WlwOLWnBBusI4=zF1GfBu'
            '!mZPd3*;F_0G>l0qvyu(t(LJgVtt|4$4c8bVEtUn*3eCtPnuN1&jBstQ3!G0W#4CYbaML3Rez??vr}b8JkNyv<zy7Jrk'
            '8+yR^be?daRDk!_v3#yQQ#td7eDDJ6Gw$}Sg<{SJn0xle;Fwh3kb#C3zH#W{(BVhw1wW+6j+DuF()^*GNyh#0GE+_aD3'
            '(^JX-w#8c`FZu60A`@nE8Pxt6t3D1bQe!~*!+;`p(xppa*QlIs|-Hu@u3V7v>wu6i@7I5Kd{n{$-m--Itu=)%P-'
            'QH=LV4Oka%2p)>N;JDRO@bKbAi7)SAfH8(@(V4LI({2=2ionVxF%W9%iOg#(c&*omdFAF%vAqUEUU<_JJa_QohzYpjJ#'
            'gWeA>D)@EhAsy>TjRO5LY2Fnc#zyI$Z4k{yn4LY6T%Es{_*J@oO!Ym0({G%_3iY_ku%n43@<9Lw8XEJP2Zv=FJCCmDK@'
            'foZ3LD{v9dM6eE8|6tPuU8>AN;#ZVIito_u9XP1{kZ_X@od^d!`VGUek>5O|<+Avq>gu&-'
            'ko_IJu0!3EIVy%QRt*DZOL5t5IaQ_{2z7T-'
            '>*apE5Ji&Z61m1V8Mp36ux>2PCJnc29;64p9bJqwT@85?8FTOzhK|gH!?FO#yd&rt>FQU<uiGC*r>GLlZjO~BqV4vPM2'
            'vK{-xSAacV~Q5^to0|&$<IC*dq9G1apX64)JhT6KT=GK#p>Xckp^EE5vsCq3AU@>Mc?!HnZK4~ka5Q|ureZ--U@3(p6<'
            ')Q6{1RJOv9;St`j4MnM^m1Ek#r7P`INqg3hnQQR+t{Ranu1;RjwYSe93*c)?pLu=*uPHD&-'
            'KHI&5@WsbYj*3v$`M)>0Kjxe+1&{Tc{uI9apBBlJ;Z4`(_Mz`^jUk)i(-'
            '3!u<t>o8LNf6&WORsVj(7T8HvF`Z=TJkE6tjg;HP)KEr_Rj-z>t~wT&(4sUk6EbdmIb1VC=Qi>Br>9M5UbG-epkzg?9*'
            'Einlc0(9}Yu{{CCE<kugr+YGQfy`_cCq6DZ2@dF~87(8Spt?&VLRaYh)l7<)o(Uk4d6JA-'
            'Q#M5vI#7Rdcqf?Y@I(K?(1@7`YsIp6-#ySxkxPVFN<&c*_--dV_!RK`0m)9H@(5O}gu3iVH=A*XaJjNCD&aqko{N;H@y'
            ';!%oU@fKQF^^%p@eCh@ErnIZ`8b~<pLMSwarX#}tU;lv5U$6In)4x#tKj<H5E}iqWZei3&y@TvW`$6uz663*o2{bU)!8'
            'L9<_<qnCB0jl*nAIv=Woe4C{?~wG(ROg2u!gDIv#d3|Uhv<=KagEf0?iZ7pgyR9_A2vWzaGKJx*gErS3(od6oB1I7q~E'
            'VfUaNMfGeC+fcN7M7H5na`Jeec)wTcqZ~7N%{|EEGY`&>&r+iKOY?{~qW8F=oathn`|7avdh@z4vMVb`VeVyGXA*3QoA'
            'xfc8Xdr1$gG%!}&$CJz?(4iYXhafGqQMX;MaUF>J>Pf#SO2x1^{oAB@4eRE*RjuQ9p~p*YoAQzt?o?yEI;PDj%!S%G#{'
            'qBULZ3&$A|g)x*K!+m?!h(UsopE)|2^8$(z~k;Kk&>e3dDt;m_1Kf0-#5>%+`<@n&w3^<@rEx-tiiy_f^MSD4?G{h5i|'
            'E;E@q9!y1NcV?BHH&gA?Rp!E&%S>m9i_HJ%4`1Xk@!#mz_#fymyLyJzI_W{9keira4Pxm^^syHFmcTWcvScy$78=f0qL'
            'zxeG^{(mZcX)aWXGzJS8ulCwT2?bf@i;J`J>esHy%NAKTc{d?3*I089zzTZUNd5bc~o@;-'
            'art77;%cYbq4Ifp)sa(U~F^y=)Rd<+iOrPmUh?AN|*U8RY*r`ZxX$^ncd(V>-'
            'uPWJ(CTGK2NbGegxbGK&q}m=ROn%;N)Bm{*T^GsQ1nVX}{3W*XVJGnf8wXTF&0%3LvVl__80!OS+h!u-'
            '1HDl=yEB9pV$gDK$X#sqC2X7RJx+FM^{E_Zch{?hPbmi)THJZXD{x&8Z9W^SD$^KzpP^V?h3|EvGd^0L+cjs8vl1O3|+'
            'PlCvYetI^@nEE7V)9$1x>MWMP3JrB7<FEXwslN^t+Ps36_rGHW<`&Rx55gFu>Tjtz+nM&9jbf>4aN#zoA`%w=oV8OpjB'
            'c4SqmP|UX-2_3YP9GT4V$Q;Q7wDv*0t{R=uAFcnJY;DU0R7RZ4Q$gzU%+5{@agSk^FD;YyJ=P3#?@0VAf66Jm-'
            '2;H%Wo;`-'
            'vd`<#pY!Gn?Sn^lTrgR>Al)*TLjX6#P4qgvaf3Sb0Yy$*rnabs;ssY2S%Snr3Q@+^&%{*x7;DI+%jX_7vjbehC_yt`Y5'
            'DH&`j>BJr1+A6jVT<I}XySg_>^^7ORRjG_k2_+|wcOYY%8b`3hcLbQ8)2`qO~{CAHHaMTj_JPSY-Q4@w-@LZgViy`-'
            'gW*FRu2jF6j1l%l+0!Hsi?2`LNk2PzubZ`&y2fV~*s~?er>8BxXS3h!|%%P7!9c+URqC!y>Y`vxm-'
            'j219@uZHvGK#=6nMLsaw<wGa9!ABp1#tP36&{y1#F0}P;CihV;@>n7=7A#Iw}ydS-DL>EY}O>*!dTNgf<Jrj(c{~fG1S'
            'vTAnfH*>c4Iy%5xW^a(OF+EOlU2PaOf~J}WfiO=oR6@*59u^0C1%fc|)BfhB(u!LD>S9P{vprsfJ*<Mo&{{5gn`r%Zu8'
            ')CYoG4j9HOL5?i6#&+L@@TBxMH1cz?d3H<TYn4daKe-D(nOWmQ-'
            '!MEO%f%L0t`7Q1pQ*xY0^!VJG?RY~`C>^BVey<6eU=BSb1mrh{Vps0bShTqdy}(UqcAuOSt=)u@TJ!rcBx}1`tP`hUrc'
            '<kI#U~EJoE5<t_&*Kt%RBH0_?gU3gmtIK9VEk3jF=i_^Q|pw3CJ5&Db`qp65oC4>zzXucg8Y<uKIGki_GSX8>5WApSQG'
            '>+T=KV&kL04tPO)*M!0T_<ZVSsY{}k$>P<}0CZZH3fYPNb=jYPvsAk#P%2#tUcX+39zP}FLTLgFuW2Rq6}-'
            'r)ZwAemN*Nz`Ch+Xur>Mx9q2m7>&~Nz|u0PpGZfzV#+cUnvckd>?HnYNU&(93?wIej=)kYAT86|<oOVMtr4C=2qi;vAc'
            'Kz@loo=$5<zuA2>H}eQ<t7I(wWp$dBZsiN?;1(F(T8k_89if3$vampy2|8b<>7VTt@S9%^7IdvAsWnTHvnmACJ_a*N9J'
            'XTUej{*FnM<E61=I^mgnzQ)@M~oVepK5Dmjqwnu7x#_W6YrP6YFs#r4aW#R)yP2R*>3g0hP9htaX=2ev1H@|13famET0'
            '?vNzN?$AeW!8tciE3Of2a0h5_>$a8ZiSU1E&zFG=d{5hD$pR0v^6I*EFxm42e=RK))wt=syhl%6P1eCF;A$xwvF)ne<G'
            'Vi`n2>WVBY8ZtOY3G1mi;6&?!4P}p_Ci{{1tY4>i{5-bN-in0iH+<JmQBtb$bOOqbAI~6jmO;>(qD(=d3WK`=?L1g_#v'
            'bIkRg0u7!45-ybyC3N!GbR+BsSbCjuSl?=^RTb^bfPV=ScAruHO&F+t>>EJuY$EHpnWf|}_jWO_IVs-sunKwct5ts7>@'
            'Za2cJwOpuwJ^%-'
            'X_Q4&&bEx@rE#A<$j5{WZk&!HqR<^RJtTzv)rv6ff_;WB{HV4kEu7hnKqgc)dJ3!>mLhUS$r;wXCN+#12foEbHuBh}zK'
            'MOX*OHTvOky}*t{_M3Mxuntg3!^;aD6}4r2N8BA73tdwmQ$a|jQ1hXZQ~;I&dY+?$vBuho&nNhU07GDji!IPz^PvnGgK'
            '8}J>w31dJ+ikEO#`P_(3*z`r}|-I3(URBUg-`z}2lk>FJqrm|MIK%(EJyc%Byd-uaL&*vpN}e9B?GmN42k0qCz*!}RHF'
            '`tZ(tR9H0!x7n=0s$<fiuq%i>-'
            'Ln|Bp6r0$iw*d1oD;s6+LCbDb##e%9g(w+Lop3ISW;q+JmT48d@L8prBKk@c@6k`3_y?F2!fIa(ZF#VSasIG#usTYx+R'
            'Z{*f*f(l7|?r^A;XFY$F%nhY-WxTZmU?K3E4-'
            'gP6usGPyc{bbMjLtNn#wpzKT91C3BAG#1aMX28;y$8g8{qpYCRjVQ~Ti9cQ$Fci|QaHcX4B(GJI#nuwo^v9EYy=@6kJc'
            'LP~w<QKd?1b~Rfw<SK1Vq38A<Cu>G~m|_yrClsBx)n0V66^`id&9>hOAld)PRjo8{xWUI+~vKhLg=ABz{XN#GWpvGF1}'
            'f#ql;|u>)wu%UM3>@yE<c1$?8AMDLXvyx24V1B?aGv0{ey9M!{>A4*85UKy-qwu5Z%8pyhr0ZLB$V8Pi=`ZqI>cwCd#+'
            'B+pqCWJh&W)6bxw|!JlDueD2NCEZaKA5M%jhSVQRK2Yc)O(Au=iFoZJmEN46uc&i(!q>-'
            '6>PAw6o!4D8o=gj5*Qyz0|V(1G!_>@@%~`!{k5^~(D!733^rLYEQ1w#407?pYtYSDO=XMUfxE&os(CS#4qh%p*SqK7EI'
            'B~#XfUaZaW(Shg+cGb&3HqY!ltY1F|m*n9u1~KNfQ@p$!9?h8OB=@Iyk?w0*|Xzf!Un_@@$(83dt2R@)b_dkEUCoeyEc'
            'Ee#3+QHG=Si8B2qD{?ScS6kTnyu$*<1UTQI+0-'
            'UKNESiN4CXpD<YNJnYyMq2c0@3SOSWr_<u16%o2xk}0Ym&fCmd!N#UnCA%jj|rjts+Wf8GP_)Bj*-5g3>b<`kc~49`-'
            'W47M%xL$2;*CM={F03dTq_KhcP_Wt5*&XX!7ABb#D+a85=T+|zRbS@qdWIIkMoPBBno*KL;a7Z!GNS;Nn_ujs(TEG%NA'
            'QH|9O;GofjrvnD)gX}JH%AlHZ@J`_;qgAB#5}-'
            'a$A9{0}GTc0$;GxcVdP08!cZ*hHbIL7ns{2hAuPMX4PiElMnE*%Q^kBg;EpVDN#93>=BUVb_5tffG>+VpU3u*NCq!$+a'
            'SWJsHGLS9fgKAF8Nuh@LEc;YLrukLs{G<e({Hx&@!x*`|{?a;|W|}C{0Z$+4z}iML*1<w;7O#{UzI_&pCwaKwfS4%Sm*'
            '&&L@_q2_z*7)V2RyQ53yQ}U!oIJbEE#7R7(R0o&u+emHZjrg`^|B}5o<xX7NmnkbQl>|C?{4YO@L{Vi)*u&&<$m|^ke4'
            '-)@1D{-02Sh+k@TYfnhalREU85!Ly`N@&p<$v<F4aNLmp6g(ezxq2k{Xe1GgJSybwd9--'
            '+_Zx=&!^BvK|{xa0REkLcOXTT;R4il#m=_-W+RC`o}5{z0*STqRB)w-'
            '!y*D6?PGsD{TCji#p(Zjvosn{XA6THPAVc+S~_%7av$lr_tX1xKlo_UPJan)$F^ez3nVF|;k(wjQ0o<_r(3*dK%;>PGD'
            '5E870uE$!5mz5AqC4FY$=G!QpXNbbb%OJi-'
            '5YFt3!&Ms_2+w+d*#FT0KHe3g0;Oq;q0T0deQ^|bqy>TeH4YfsREG^ZZ}A(N<Ef1o8Jl*jfOk_qSgcw|8>jCwv^#mo)v'
            '*sOx2usXW=%dhbx;bv{kQ^$y}uCmh!^<xR5EsLe1g*a*D#^*6aB1Z37QL(aHH#d(AqOr+j!;$Q99aBJ2NfOK0*kF%vNH'
            '!LN31hbC1@lu;4-32o-'
            '5|B|Jxi;J`9Z+;Hp!R$F9}7a}Y0s2k48YC`Un0t5$NVCK+c!k@7U99H=<v_ij<T<HW{>wX%X4zRIUj|ZRq+l{va(=f!H'
            '1(hE~D33Q6kho*uI#B~A(>{3j=r2;gn3JX^tt1kHMkMS@6jWbIB)PV?(ZqBwp36yq@>ju-'
            'x4Z!29af_3l4Drh>W7zRhUvAR>bUCuZRFY|j4~I*&{}RDRbP+`Fg8VBp56oV6;<FuRXJ%}ABPVfSA*`^I1rBSWRw}sVL'
            'z<T$4C9QVY^xnyyrXw4rv|uFCY?q?)|8{f5wRQrFSJ-'
            'a~q<C?oHZI8i8&CO3*bi3JQ+r$g_(#fa3$Nj)hGo_WwdOh%dp+mB%UH(GEH|X^f}Ln(>$F3`QLYgWTK;z$;S2`V-'
            'Mg?G3~6(>8mCj*J)ze-wu|iJEYOD+Om9*>HN`EUfjdftlOZ;G}exdOhf&l3T)%&s&C}r1BIJyRPHXoge6nT?bI{NGdkU'
            'o*`06k`U6Cg8McNP?yj+a!q5LA@6mE6*ksGw+s8?)eoY$^LRc4-'
            'bg1u&aH)#TN%)}g~H36B>3r_fS(raqZd0=!1>)~!1zGYQ??$9_y5FJ<y+`=D*<gnmoe`6X2P4lhTya!2iJ#5LveL4ex0'
            'sI{d*fowt+X?3FsxZZYLmimahU|$TP+s$6|#<6}2goAi+Am=r8O=%k1=^{!uXA*}aTBl1Ty+ks>s+1<-'
            'EDMaR7t>()jM;Pi(~%r<bryJ<+qn-hrFg>LeAT?rl=(ZON;Z<yRVjGKca;q@~Kw9r$9TZgW~vzJF9?)fb9_XlA`Rvc;%'
            '_OQgNlwi-2D^O_6#U4u;#5X?okXv=WPMAhA$S*KKW%DR_(KAW|{q3<>^9`6h*bU3eT}VBTIQ=kg0NtC9GF-'
            '#o5}C)hAzf4$;+a7pE;9^kTPH}NY#@{$+YQ|XSLy<P`Jh$BQ5yGi9?g(opx=w@D4F>a*-'
            'vLE*LW}9t=C{2buFcHU2|c0#t}m%W66Z2Jy=#}qh&=ZWT_-'
            'liQIz_#a{tV9BFuNG#rhW)#LibofzlC!kfpJLhKPC{7ftH{?R6sRSl=?nXNG4%fo*AL5`&}N1ZP0n*@D}8cf-eN*A=<L'
            'm{nT+-80qFK@Gl<h_-s;hu|0L&tG)dM%v)M^P?e97Q!`K`1c<#PUY4qr-'
            '<7`Ay^1he|AZlue6MlEFTUV)?fsG&Zz@fS8pq=z9vRx*Ev#Bk~~rPY=f5oQGu=f64i!73lioCFcFI!TypKB2m48jLi1?'
            'gkN7_|Ao7hW_hDEp91|_+6`v+W<YG>3|^oY;9oYy&P;!hmAj8aH5@w3lQzf`UJI6L5%j*BKS<rFXGy3tU}TEmxuhTDO+'
            'qnz?s0-'
            '_;Q<UOSOqV16>!JceD*Dk7vOP#6FBvz>59AQu<dL=x!<V?r$`_iY%3!5F<ZfRdn346RpP|8C|Kg)j+%?CA+thL+di=pc'
            ';^I@n8mN^rps$U)in|JB}Y<)+4q0<-'
            'Ukb3?=LYpLuF*k(fNEC8il@s;*Xk)2*F!0K2MO%*z+EWCGGIE;2_L*?uCDkOyKdQKCH54fvog492QBy3HEc=Lf<c>%a#'
            'z^oi!vq+z;Icbm*&~%dkzihI~8I3KnX)@Zz~8!*^(=?y{RFl?W6he(!m4KKB;rm!E*odcwpzyA6fZV^JjSI<&Mz;L^WG'
            'DW514?1o>`$yO$GTLfXO#R!=&;l$taP585`0v~S*r#8uo<Ys#StuJk17@VzvxjItxAgKiZTi573<;#@w9f5ZZwvc7A5j'
            'k5$uzK?rN<UtQeRG|FO7>7ew_Lm!zXE%@@-eN0fe~?eDD!eFzPwlpcgpji!ajl_od1Xnf49S#HNE&n`!xQ3Eef1-'
            'ec<HEY}%e!3t?*MU~pgpb4oS?Z$&L5WwADhy(@(^3$BA{>TUFqJ46lZazJ!)IXYdF!o3bf<mH_rEx)x%c<SEoIw8+s{3'
            '3UdVZEr5Y)lD(>r@{H%JUgKD|*pnV-'
            '2eMNaJ|(22|&cA;p$~xW)J^=~D#gIPQnfn|F{rmqE(M_mV2z7+?gJM?u3%5s1FL4nu|*<jn0%Xgiq?m!s0)@YX62d3=F'
            '#9zO%8G%mmj{uea7Q6E;+j}fzwOc=cvg@3OnL#S#iHCP`CD-'
            '^OYEq*!1zpjTi(L&ah(OP_++KV>%*;uY~4d|O3hM;L1wtw75#Zqg@5T`2`iA!ROQzDexsN&z~w+!Z4A#keZz&!)jpj-'
            'G5%nLkecb+N{Yn`Iu!TEKk_H>iR`>A;Bg*wZ!{}R|QJOZ~D*WqRVlc4v`3v=UFKt=5l)-'
            'KUluxT@cV>oN!hm%l#HmFyhOhW&GB3c%c2gL?Wj2SgoWC;m@pxs=M^*ToiJcCH9R1ucSp2PVL++eoe1Zz^G;7v#dOk@{'
            'A(cW$N^bx^-#&*OjC7RZsdrUsbxWQkh5OjC!1O1j_*zFCV=hOzKK_WzkEGJohQ7GjQi8HP@prOG9gty-SFAODxVO+3RX'
            '(3+mDWvLKMae_Q)7US404D`EfYrj;b!mKpMe!E!nWGa#Ib$$QDjIxSv%z&#AB&aRaPD|CUKUToQ<tV`N477RKaHokN>L'
            'cu&j-Pu@5B4(L`K)QL0sbb6fshO?Qld653vc1@6v?P2FuzLzUv8>?Gx(g=n41zt<kh78w0Go$b&5rXjbeDNeb=sc-'
            '<Fb!@3O@W7px7*cV*0O__Wt(7=T4ad?XD4;5EtSikI+;Y+D_EQxD_W7Z*<cEOmQo!JQ@sX{2#ri0E^HgGh$1hyAcLSx@'
            '|67-c5vqx-T;M-D=w^D<|tz#tYYye~1<Q-D->;~2idy(ka2wK#iiQ<cgXroLCtPK%@{-'
            '7SxwEhXM=nsVC@0&2i;3@i!N5MZ>hU`0;WR0{U^7zW(8yjJG_2LY9y15tUE{cTvx6NS0I~r5VqZoZohB#iY2R<rw@M<>'
            'KL|X8INJj#-'
            'S=9i4?rO6>?{T0_op$)S^g8Meq=Ms`9ds2(8eQS`ovhIxCc6ah!nw0TI!AjB!PXU5X@ICJiEKx(J`x78#g(kfT3DBFx&'
            'tkeCP3lvDQI#F$FJ()D3*N*+_h3@==ouI|5+MjFEj(xB|+@A5HiIoLDt2W^#0BgGNqw`Sz?B>z3U&@b0Pv>Z?8h3$3Ll'
            '0#2=PD&e3tYdx8<sXoKz6Y&dbc9GY`O@#B?SP?a8pEx%2u%ve1vWIe@yHxkhLr4RlXl>l|;e%R+R4j~zNm@^rNoX3mcM'
            'Nu{8uD(YM{we}n+Z}H4gv@>iBTRcegf7JgK&$;QJ>)Wl6K%73Dpd{9VFJQZi_uK3o3+u^7ftiM@RDr;8oZqk-}$~mTE-'
            ')^1Ad6@`$#!h9YCbYu=epZHD!IGdOq)H&QQQ?mL#w)=Nee^so`+U9@<}a9}1Rn;ij=bs*0lYy+S;_HW>{ol<$F!#C<eb'
            'a|v%b3=+<HNgymghd{{){kJKzuFszv`JalB5NT&xTF3$Ch8VO`hy!RA(zm*v5W%p;M|(w~{%Z<7=sisIe0*{Dk|JoD+e'
            '%8CBjHZRX4I1Eqtef|;1-QD=v;FZO259s*W1%r{5A8i+jK5`FD=Brm40xCBN7{no`b7U6*k}9U-yUIiQP&8crLhyyzu-'
            '?yEL_-ONEcTNx4Rf<>rwv?h^PO`49`_vPjFua!lCa2%KjR(0t9?uq^c*uA0LS9;>9WqVO0=*L+8&Z)R$(`H@O?os!46m'
            '6zzh#j`mP(%|Kq)A-=B2J!6TMMcHM5MS<5C&Ju~5~lW$B#tmBuSmEbMnFhlA+BuBBYJf$XkaF-'
            '^Zt7{Y#BX>=a$StT}=Yc{R>z+j>UC)pI>12@(Z}dO^<MMvZ2qqpK+%z4!Q!D(0N}}@EN}?sSj5K&%8&ZHh3>Qap}NwcY'
            '|^C;TBxKwV7B8#GuZ#P}FSSi{)$P(~~nsL_%jJ&<6*}jT_-'
            'GrzQppey+m8f>k<Q?|u_=y9)fb#R11=y>wvL7EqPh$vUTQ0Zs=GLdd6Ve6{3P?byFG<jURv|KZa8V!Cii|1NEP9||F--'
            '0*q!5U%)@g`6IaL@%tEyxzVS21Ay!#GP*7?rL64?wX_1Mt0G2Dx$D<>M=1saRpK<(jcX{AK&-'
            'h$1CZ_LH@K53SBP24PQfHa=sr3NLXQ)bsPLT5Q?STx6v=N7FKd~YjeDlWTE(eVmi@_W+VW6zFi<ooucu8+6S^kH6Hyw<'
            '&(QE0`#GPB&-l4@bgG0HgYl{*wqM_4{u>=Tn^ZtF~PiC9-'
            'YH8d+_0s3HpI+8b<{rY5KZE$W{ITl}dT&{{t9_PLu`|`H)oJi}132mh+C?qdccV(cbYL!z0j=M0Bsg7_%&VIooT$p7tm'
            'D>1=2VtzjIqxdy5F3(?-'
            '|2I;?Fgonb|#5O|?4ajUy=^3XF*0w^Sz*XeC#|w8f!%?;^4m4r{F!ITD5>a)NZrzYaruF<G&c_m_-'
            'zt*zU1c=WhDEG|H=_MmA)NgDP+J>($g`gTn7pbSGK~Ys(yblz{pCR1E?9yda!(np-wpv=w;7DDd?$QHUunQx1C)vSOR9'
            'X&5`hm_$luf47&hevq`QPFeU(G&kH^8S$`C9#f<Tq)5q$f*5!yVj!0s#g&>J$F*+e>7K0HXHos@~s&!sG(mKJEH0kGRD'
            '8M$#U7LN+l2_+ncG}U4{Hz0;YWV2yOg$G1!u7~<<`*GRYEKuTWhl%1fC?xurwf5g4G*Sp)g{@JCgL|w<>9b;Z(DDqbuD'
            '%3=heu$IiWP|Vxj+Z-!P-'
            'GySRZ;2WGZ}c>GEPU9AuJD0le%c{WkpcVHJ2abYPgCI9#uff~`KTxG>%yqqsL>tC9ulxb$}7msWsVArg6g{P25(BP15q'
            ';fhK_JXDc}bMMHL>7mS7ugiz2m}AJN;R}KMcd_i>d>WWJ%vvoz&U(_h5+8dB;|uk0+Pq{buAk~;DXLH6A>~@|6E3aGAN'
            'E6E5(#<!`*5vrFJ24U0BKnSEM0apj&8k;LAq_=seKnF_NKw7NCJ7YKFs&}BQ8Jsi1IaVBv&n!;1<VLObRQ7p_c{p6;Bv'
            'EdT$1EcBg<3UpieO=7W518$bdVpzW$!d|{`Fk=DLMbU(%2N!9dO+A8GzeG)VV3s@g-&m-'
            'XuHxYV%(@jC=QA;uhV~<_Lm{WRi^JoS7s8_?y9$%V%KLIKi8&UgD2Jk5;2Lt64>5=tf*cf#aE?5-c9*H-'
            'A+U77E;sPJWxyVPZ5D-cc1hv#`sBWr*Ghdct-H0tc6l;ay(nXANo6S(wnTc|mzIdRu3-Wx}SfrnYe`lB=@6m;M-z*U-'
            '=c93_0mdGcWT@U~pfbMC!C$8l*7XiRn%xi$ZO(zAi!G#jT`=lh-v_^+odf=bW~l2~3WAy~kR`%T<ODW@NpB%KhV5f){h'
            'bCQW{yzt#u-;_yGIMEJ4wi+DyZa7$If(TG<f&|QWj*AZy)8!K&dyx<VB-'
            'H+!OfmVgoKS+`1|EKoZei$O*4M9Hd<mJ+p371Lro%z~r9C7~&oRpSl)7<M%>*=cxyWYR4GY!%JbF_Ek)j4g`S_1zK$T5'
            '$9+#Nugp9oGrWuy4Sr?ZILoK$qtf_`(4Q4-'
            'yXoll>!c%KhPU|QFKAT2z|8fB5FEk;M#XvU`~4$9vzH>EB=e{gVTI4n+_zr1s>pL*@oFCyI~~W7*}|uK*)VVa<cFM{im'
            '4;^=vy>J04H`o@}6(LSkvd-'
            '@Qa7_c5;8@RjaYenzevQ_xzThwYXraI~YJxXJ`ESfUw_(PIr(>209b6f^7J2v0V}VCX$B@Xrhf&IP<Wa{?7Gh${-'
            'GJ~`u=d>vdap9+WO<iejq1q}5LL)^z^rCBDxYrZfr|KShCBi$$-'
            'kVemhDuc~O3a`X+&}M2WYQCw#G55`AU$q^*_4c8|FJJiP5d!bJQc34P2uYAP#9!U<*f{u-'
            '@XraOCR2A|<1;z>TkAQ5bgMzjvq)0BFb4LpD#?lCig5C@E!fx00ngknV0?<hFHOSO6P=2rei!_mvmX^XKhR$pw;5f*&A'
            '{1Shs{}%<Z|ggR<Z0&2)plvdYv|ui(?g}e2FG!eY-'
            '$nE*sZ28>5t)8EL9sf?SiuFtDcv?>W~{juIoH7RyD4P2W<Nt+&8%@D$i8E6}5^-'
            '54kPjdUh+lX>(SefYcz?<zdPmZj>D+Q@}#944u*{SEZ|D@8MmI-'
            'z&tEf9|_!O%mzxa@Na>AOCRhN}&!N;8YvN|&Qw$8D(N%7@~icM$c(5Z<ihXKPPq<Dgp~Rs?;)s5eHaHN2k6=<dUqdyVj'
            '2A|DQs2->2!p4g?_hj>OhjtI)Y1`{R@4c$U3iSI;%?-4b2OrYJ{dx)zki!x`#QP{!{?QhQq{c9#*-t-'
            'l1r$rg*AFn}{$01N>AEbYyH{voAHVB0Y<5%IgQ2D`%8VnWU*bYzJv&D}lz0HA~*v}+>TRYa3*JJ$8Nfd6j!T6k&Bm<9N'
            '#?xDPNM;)edd!cP9q!WLywkLsUjjQ!zTx3j52&HU7@AEup#S<DWO6Fu&5|V6oxW?h^7~2}yH^Dk=cd5IhEu5d?;x1$ZK'
            'ck6B@iSMOAVgKVWjtZoFWqN@LN7SSynb%!|&wgf^A@Hx)SV|dEkD3mU|!O;5l0r21lG5IS_ps`Q#UXh*K%qRC)m%k7g1'
            '3ihK}hxkBZ&%3(6X0@54jV%Yw*kk*q0Ilm8+;+H`C#FJnoFOZzLl}fk8W<!!+J~B3FlE33W>DZ}b7{+;$u~c>;YW26!@'
            'Q7|`k2gmnqd&B+U|idPoriSJZsMz%MP4NzBq2$gFky}=%PRIN&}BX7u_2d!GI~cFZMMSr>jHf9eF1CoLMwb}XhTWS&A9'
            '%;AT@t+28B2C(!#MC*rjF4a{ZzSIr>Uanx{l>$a4a3G^_5PQvy6;O;FiZcc`;ahgV!apr;iNkJvdlU~mk!oHR%G9TQ}H'
            'SsfnE90g}BDYD7_J}O@<re-I4$zSagFy;NJw(`af5bB^9ub6`e_C3G@-'
            'Dff5{cGGNwG*}bt`VCtCs5Lpf{b;Y;KDqN;z=(^ajpoF&Q&2tcHSW{tO@J)q`>&_VgSc{nvkD{2OjB>u8uP}mHkiqhhq'
            'h<<MPIS)_HvQr5%g6pMfKjWh~olKJb7d{Gk-im|R_gE1%S0&6+eAoErwpiK+OxL`7T4r2&5rDkAT0Pndh?CLJFCPOoOU'
            ';OY7SG?yB~f4`$aJ$9Y8ioHLIc!t755=~z`4S|krY2?6=Dn5`+#FIZ)!>3PyII}tp=4|Mt+YfV-Cfl`8vPg)!-'
            'f5*LHnn0w6&H$gM5C&kEv6i)z<Ira_}R#xatLf9fq!=5Q<r8&ZM`;#@wQWA?h&eXa}8bJkcM81Jkif+AFc>t(+k4&c>Z'
            't(Ivu-CG<*t4(7|f_{hN&`hF<uf;4?1a=ZEJj@}OdqAbTP41;4Xg#6e&_uA#|{wX1@m$@vLmVe$~YYp;tF3u?)|M<yWr'
            'FAt@|ldw%~4i>uEk@inW5<JbpsQem9H%<aR9~Qj-'
            'z7s8`K0@;PORz8DF)Hdb!6mh|(0WZCUjLQG1qY(&kg_=_8~vmE_BDdx+IpzvkkFwK0jU0N0fyaPjP$AsikdCM50^slZ1'
            'Wcqdq)NB*TzA=s|}19-lmx;OR*5<>+~zF2J=%9^xL9A_;u70d-'
            'lwys}$NWC$=7N$7UR>yMcH3mJoZD)%g5$E`GCVfg2fWu+4gam8pLW47tvNcYGP>Z=jHTC=;}#6``C|lddvf=$G=v4Ill'
            'n=M$&Sf%ot5Ys(yZ;z9`?D-)%Q-2c!e)@RVi)&pD32Vl|Nb}+~VI2sy?d^OoH67~$4&JyrDk<blw2chA|63`vKj-BsxV'
            'YjX>Y&o(4l|Cn;u5udGS}|bz5^hG#B0tzv*9dOsrP*R}t}O8j^6Zv=Ln>f&kMi~w)g4l(!q<U{<cQrjRxLZ5QD!ua#=6'
            '`f^0b=hHX$@7p2Fd;2^cNc4DUxxkvVmnQRgoVYXWTGrpg1*P;V!S18i+xwS7=G`WVGjUGcc_12SC9reR0wSXaDi@tC3?'
            'NDb^DeF_&)NB=S=^It`&cLWCQ#34hxm%bls!`FSOv*(e7E6)zyePd7}Yz^kgcHy47p4o<7ib7tzI*qq<fjim{+81Qd`X'
            'Ubu5nhJ36jqZJPpa_bXbW+UnghqV**F;Mj6FXBiFsKlJg{UDC3O{$zT^w)HZjoFEd{0Xm*R_mPl#|$7paWAPC7I%gLr>'
            'F3hZ5gUQHWNr+^d1`hsxJoS%eBU!~d4%3=McY1WMPdH}&#JbZsAR%#T1vULF+D5=J=-'
            'lfRg`wD!EJ#ZxOJSsdbB2(QOXi*mnH&-'
            'mfFOEzyy2KVnLyrQp*&J3M`GKeQ1TwbX*g&rt?*qQRJ8;A<2KM#E!{sg@jH7Gt?u8<Rb(Qo5y9tK{ETG9Tgf(5aS|@5}'
            '7+vp?M_qca!WqjrIK?Rh_txJc>DR|`M7W)<Y^lQoj@yYs|5Nk{xCLuwIN;dgmw0Q*eR!#tkCE-e<o@3rSZDf-RxdKfv8'
            'rg`Ik1stD+q!7uDK}F)&=`pQt{f8Kypf95BY03kNi<h1v{;cc#=Di{5U5BV{WZb!>dS@?D?>Hu>kw$YAtxu@s8f*TM9c'
            'y2{u{o(fO=7MV+HJ5<}73aK~gDY!35*wJKgH_1*_~=D9#He1MibRrcNBK-#7hG0TFk5c4AyCzHGwy>^KtJYE^U<Rdtg-'
            'ypc!9RpfK!QCQ+o=oz`DS3YM-K>h+uS!61ZZ@oY&cQZ4sfgA0s!)aF417G1PZcCXK~;GT9IE(A-'
            'IX7*Moy@)bCwh6vzNxsJ5{hQ;T4t-Ez&88WaH0O`#}F@C*0cLPnNmI;TysEm^x`(H`L0Bk|*vFjm{HbWB--f{mr2=t1{'
            '_H{|hv2P8s$U+#$W9{CJ_?9qZ3OCc9s0LZzY#vcGe&f2F#>d9`l5peu%sYaif~!<l&FnGY%X%Zt6I1L6H<et7u27Xoi`'
            '>8L#!#deq1c=GZpc-'
            '^2#4E83Gw#!I%rIx_G?ZPmxVv=mm$VF!<HaXH14&TpR!PlpMkg{tbnBi4TorcHoL)Tl9!5qQA7a!3tD^AhE+pB3xbv^z'
            '0ppwr0U;;Bz%E12O#h_)g8Gf4)elWiVRu`HXMb*(DG8BrwZ4`|~KT|UA5pEcF#o(Vjc-Z_TeK;zM&#Miov&|!Ir)5#-'
            'KNEz>-<N<v8bGzgYxJ;;q$9(g5K!EKZ?}0;A;);SP-'
            '+5qTvow_4{FKspeyJS#E&Komxz3@Fn)K}*WNPQ=dG7&km*$o_{=2>cK7PzKtdOqibTRr?X}o$w+f=i_9J<%jLBsu(V)o'
            '~FS<6fb}joucsp)kngcH^c3w*Tudv~akq~-'
            'Weq|)9Wn<2XN!mA^L^o*J(CUw6SaVI1UY#p}KgB$8yY^xFJm4mDMY1sSQXn4N(+F#QnxV+N6i(fHjw_bDq2(p>=#w)nT'
            'y3<4wlZ6&MS(AinXws%$7a`gsv4JZ<>Bl37Sy6{2><yP;LmG$tniN<^r-'
            's>#=oAw7~ijpo7r5@GLMOe9C}EXQvgUtNPyVw5p9t$Np!xMMau=a@f(K%9=s_Dsk^Qch?~aC$${|e^awVai^9f(=P{f5'
            'v(nu1$mwDx<73QxFkk$gEFtT#mSYdj2wC9sw1c#dcLJ}p5v(2yhkurdEX(9ejLoxa+0@s8caHR+yw?#pX*En9dPLEw9|'
            '?3PCqI@>@xsS%iRi`pLw~vMfS=*-aDGw{=*$}+w*?bGH{=?Yi^X7Osv>f!-ot0E-'
            ')N`sOK9$|!yWclaL>;LuzZHaQYj3?Y{pF(%G(L61w=tYGy&JMCg}#bOw2p88yp|9p}*lS2<_-'
            '*6_0tt_4NI4$Snk&TI5kL=mJ<pvZ1_s3?zA-(U12m_0-'
            'Tu{lGNx9%qP`;wa8@+ejB~5FrUC%W+kr49pR@1xY*Ip!dx(6!lMlT}gWY(pR$j+A86OS1~H-'
            '4TF?O1bhg{qA#brh|Kv|xae~QCFOOI!+o3{*l?I!;<tkXP6xql$2Zo^-'
            '3_$qc_gH*{!C)rFTw#yIUJo=gfz$(jaEE{#_?Ik9Lxbw%SG;(PsHa@A}F4U1dGcl(7M+JW|EI%>+<F3l<x^IbbgRX>su'
            ')Km%xWF9pLGc3XAr*p}mv_sc0)hxu_E`_Dc&7Yq|mV(@FBk%bupbYy$nkRUmNFn4u@W6P>0%kfCi-'
            '_<+w2rNVAdk=}O5wTlDZBhsiI>3}a5akCTUUV&y-JW0>a0(VIToyMfSC}+dV_Kgn(*2C-'
            '26kG#0q?*b5s5fLlxfny1aS}JfRdk{0Q^FtW3RF5A_9!O60)7sit*!OQ{E~uR={az&^9nprimw}<jz-DZW89R7{e20yP'
            'ReB^6j|X9`k!5J!bcE<>;C{)-^V{QgW&aUmQ(c`QTX{v+!q@F#rK2IJ1-'
            'wToXCQoL$moZe~k1dG@(3yc}>YSa~KSp$7+;yK-&Wh^7okpe)(8N3IjfnjYV?g6-'
            'O)zPMyQ6t`C7T%nO#8ML@@hBaEg*&<vSTP+2!$=hM;*Xv&Ggv3Zt|a+nvMa>Y<zT8`&x-'
            'q6z)uE;kz19zQm;jL#2_1?_J!}@EX{4*PcE4477@D=O2$^v5UeI0y~l-'
            'Tk6>tV}RMRxn)daCTMkJCwQB&GQZT*~8ycg^*TS36bUsf89X3v5D-^~<nLsgvY=sUhj@0?6mG1se>lz-'
            '1@F$L}l9A|i)8dc*}L>nRPN7KKdi5(Ynu6PGGYGd6qq!QyAduvYvc@JmPI=)HL``dS1xg<rrMF5|Rv^?G=JkdMLr@I4$'
            'Tv0;S$YNLl+Tj?{7M?~GR2$Rni!GffI_&C{tiRH(j>S`9spZkO_Q!+qouNX?3mE*$F4WPI<1RU)%&_30k#@c+OkD^*x!'
            'XLMzd0-'
            '00&OF5DE)QwLua)He^EO(RR7{=Aa$&IKB5Q&vh3j6bl7ZE$NwfVnvUhtts+G=Rmrb&e`J#Yv|A7MtX4v5+MLFoc<U!3>'
            'Euw)B^s#<6Yn(6{XT{1_!N*r0>F^CT{NA<<zQ3G{yU$d!P75_)hOj>_+jAXUHN#O<dkO8#NX9GhhKBt)NoChaV&;eMw8'
            'N|jPgFIdVz2<Iej0|BS6-'
            'p3mK}~Mwo;kXF0B8|A8gbfYx!^HueAOL^9RONIjl|>Ym||2Ly?ZX@VzA&yR^(mcXeJ(phhz`+}nyepUqf3p6M9AKO2i*'
            '4#VoW0N{TRL;`OIqU>BTP?s!%_qt!`%4N41Bc4|9t7tjo)YO2RO(lx|{>>uKxv+m%I4G&gVb1J+GY+W+th|W71fKuj{M'
            'Fhe*=PR$=Pv{~xc<NT)%3S=aBSz%#_ZJP*!k!y{>a^dLt-DWvP(gGMXw^hJ+}?n9S7*~`!+C~R*nxtpJC@@I=)?`1qDe'
            'FxbW=@?2HJ-w<&Eb@unCQJMa+Ow6Bv_Eo?~sXO5pX@L+yrEL`6i4i5c^*e9bzcl_vN&BUx?yIkUf-'
            '#hcbKD?TUo7K>Bc9U?>7HQ<WCwTS@uv|6v<LJqicskb`y{}9`BI^rT*Kvz(e5Xm>J?_%s^KE$ej}y4BX+_@+^FXI#Ij!'
            'S4gHIhoAl0#%bZW$)hlByGi**O)ziP_&{R|mbjKnqGU(oxPHPTQ+@J~JhLjjp|CfFRUl^rnQg(Nm!c!x?E>!9SlD~8#M'
            'VO`Ku@ThzMhXyy`*<@Y%tNaUDq!&e)6RN=3nF=}AD`s<7JtOECoAKi59N1zILEcQdk&RC)NX);lWXkI)@EeVznovESXv'
            '{}f=?M}rnM$7g^&)t66>i`Bk7OMQKvQ{l!jtoptoD_G^}<<DD{T$g+P7)3qz=wy^P#}u!w8-'
            '0uyeqV{PMp7YZgX9qhcmnbgTyEZCk3JJx%>n55l~+x;no(6Bz;?+-'
            '$CkccIh!F!}{cLVLzOczVCI&h<(&UG!`*e11HpEo(cQb>7{_*0*k`zji5@j|ZVgy&o+TbHL4l@tBzMi?QdzJUr`Di}&t'
            'l(F0dINubYl3`{(W9F7IV-'
            '@P0bUYliz(`jTt%?O<2O3AM$Az1Lj2@49u(X?R$+*x)WT<(m*i}kmFy+8{rSFfQ5+%0JGzw|n~WdYciErI|YUmOr?hFi'
            'j+xZ+SLynV149S`}!<5V|1&$Apq$GOo^pDa4CaW}c5H$+l)q?0nALQMQJm(IJu#;#Eo`bFkq;0G@_|8xMo*M)<G{9QU<'
            'JqhlvDWT84tfTXF)bN4ub%uMKBXlr3psqlT)G$WS;Y1YirT<{@d_Em2@_;VACk21L=;6t^0@ygM0FL(_BlFNbh)og1uS'
            'UKg&}gC!=VQQVi3{q^vPxg_a-'
            '9dKL*NGIe4Xn(O@MWgIRCRQT8zoV=8xXM%iRyg)3@k~)A=y>LMvE?F9&tO9q`H18Z;)jAzSJ`om;2_93?wYeQOkzYf5J'
            'QdE*UFYoh6G&&4_mr*6ZD-HAG>Ex8c%_#CPTz9j4BETO)FCFn52$1a30NS*M*iIP{y`D!`!Dr^SP;t7a&RRs!jWuW&Su'
            'g<d!Zam@n9-'
            'ePjLH#8IL}jcNoOTQn9<@Qr^`n{I90<gv`}V@$mU#Fo#)Ea4B1G=fZN|cr4%o5nFxp?ajk&&pD7&x#jtZOsNzn`{>MsZ'
            '%6&_&M^g%c%8i4QmI;fDy4DqvehgU~`LRHEfo%S*|eCTd~i)Wv~JS`J2{oPBfyl&DDwPiTI^cf_o??yKnZK@K#4oTH|?'
            '6c>AhS4Q-s5K9`3hqO2c@?BF2dK&DC0sZch~e%~;8$}%)n7&sJj>%HBQ2o%L53U@eGlty2Z7t(AW-_z3oo)W0n2W{?nm'
            '8_-mXmct$Bp?=`q-'
            'Mr2=nP{G_Cq7ccDa#f$@q=r`8`c<i>|)SnPkm<q;IjJK?F>fN+n@h}SUj?kyACFEX}CM>QvkJ26k<YOBz`(Cj~t-'
            '|vPeDHc4t~D<s+ZrP4l!JxvNi7qqa%`b%;5C~6R)FH{XYdg`&@1vP8hP!<0@-'
            'NBXGS23B^TnVrtdVT*cVMpev!z&2;egJqaQTqAj{?sxURbnBfoM{LoXEz4@NTP&vB$nUR9x0P5{(C*C(H%vq<;Y8Qj+7'
            '3t7?TU@jRA%ZB?f%)XaQo7xb`^Zw*h=Ob!(DHwltdqahsFK}wHaH~KgS)9@dyvL<bY>O}gxsJkr6Ob)iNM$-'
            '&SR)FGpwB-|b00CubU_VGjBB9%rRBKS=_T-Xucn{B@xxU;OR{6P5PsfaK(_Hj!nvDsbZi9lpm$#^mE)5~38fHXd$yD2n'
            'BF0;vYyh$6&|!^V*(gh{RBx(3Zb!u$k6PAOEq@5!p8x^zFlPS1UDc_4W&^}RI$>?uC8QJHY~|n%y!A<BF0~D;PXwd(S4'
            'OMd=M^!Dr1ToRkZ;B;xKoX)f&4h@lK~7l*uV*C9TbYulM<IyLT4EhWw&)mEQtiSR*M+RmDfQMCrz!6X1PXjVLp1iCAYN'
            'eHQ8q>z*A)AF0czDxCuHt0Nh3;sDk>jergFE<uc(G%R?*iw2sNkm)H8`!l3aH7Ax4#F2tkB84y$qz8TT;-'
            'KXz8}FLj#wh7BWKKSZVIxso{38^%IsatzIGB^gL1M)5^>5IxF#*1zPVl-'
            'u7vXRTF_P86T2nI|`PEJo{|3>3gZdC7NMR)Y2>JNqD-'
            '|+&gK7omU`828<>V;tGaJOYZ=CUTaVV&m3gY+A>u_sI4B>rQ09rTO(M2T$Cv?x?jm-tHmJtspKMla@Cy&V2Jb8$;;b)K'
            'WC~FF;@aW7F<p4vM8}#&msLtcqr4Y6{5C_`sqtU80I$w0TVC^xY6YCxTjqL%j_o6A|V%9^3fg1;VDNB^S9C=vFL^i{T='
            'XPXIg^JD<4_!=^Qim<U2UtqJX|OYL7(E`Qk?I3}U^kaS=Ee*Z88aqd6ZTQFo%xu(ZNARK2Sc-'
            '*+C)CrpTINnVf2QcckP`@BPvuPOQt?E*nj&UGuU%{(9l^$XId_nt~bd-u?jo-?W8PDl3TR;y%lXBr*QkkR<M?tOYc5-'
            'K*d|QaMi6y^sYS(GVhLKzRzdW+xmd$)k)yL(}rlD#?R&>wkRv`7DL?C!Od}yQD7|!!K-q~=(@S!d-'
            'W)!{&c~#cpKmesfGGw3OH?14YtAWadny)@NYQ?&rf<oa(4zAd56M>6n`w^X`*`Fc{r_k3-vC&1tHx?%$m(Y!CAv_L-'
            '`NsdVB!zLI*4m7sn#$Xoj`I48#BSZuIu{1;=IbQ2)sfq&pvCbjDsBPQQt9i~sYFNgGOD{!KZ)vQfxWn0>-y4cm754f-'
            'l|(7K#&V0Et$d~-'
            '*jc26;GzZ*h#Rw_cJM*!Z)J%~KqrC1<pfm`?rao%7Csug8vXEQtTXH*c`p^%I!QHhv5{Y*>Ol8e#SbPGEQR6%*wYwcB8'
            'T&VT?C>q|drpHTWpIe+tm(TCj>K01DFL(GLM`Sgz7mtV3ofh=sicI>*`YmfTjGL~uP=Y1pj%c9A1VJMvmJA8vsbfLN)5'
            'ER9F|1EkPZZQ03+$oqUw2~V8+p34rU<!LR|EN2jDOFU!KGjN$fFjA%v-'
            '*w)b{{Gc)784NEHX17h>!=ck=1NF4(`>n}&D#z)+h3_QiC-'
            'x12?gsG@@AcdUrg!PN){xpYcuYv94GXVL*dwxlUV!J$9&#%VLslRrUqJLeJg)u#CT;bzQ<@`k<A>rke+fLwjTkB;N*bc'
            'ojv+W&e(>S1Y;AafZi-%Zj(5&Wo9guorXi2M%Ugh#W_LS*MVydxBVGr+^v(66RVw>J?jB`%$Jf6dU~WH-jR+(a4ETc~-'
            'DLY3AtV(>!|PFp?%v*r~#*<aeJfXQJRCRhxsbtkAIE05efn~itkj^R%y6AZFj&e+qW2!~SrAaLddF}N=aJsy@&_-'
            'qKYw9a6lawHyni?p4k$tql=i}RN~qgTH9g5uK8<lgfIpfx6f)kj-'
            '#rBV{jJ}=H7xgBKrASe6t{5Fs~u^FS+O@nxACegkU!FVOP5Lw$wP_ftqI0f6WZhahXigH40)lRbO!v_c+kH9Q#Rm@P#C'
            'j-'
            'S{s4g{1xp;3f)V}p0yyT%PyHwGbs|`D)e8}!F)~vTHvX#1J*kd6s7|EA{BAY_VT8?n=EWF5&oAhJZ{;me$lU^X#RgU#7'
            'UU*Tdn|2&4CYpoF=-'
            '2v<^51PG0y_oSJ2~pG=*cJ=?W_T9`4MWla4pO9P98>C_o0Pt77jG>Vn)aYjH?ZVor+h1gFd0wtMZ_^-'
            'WilPAEC!AkE3VpV*GnBoG^8xSr;GHgLRo1gzL}x=h*zZH^zLh=0YZ{d|`!BwtQ^s^}n%XYCb%AS%spdbrjAAGk&wLgG0'
            '^~@p&YVf%n@<V|NxDneHdc+B}KkEjGC3>?6-sxZ$-'
            'DTh#GX#n(Ijq2Rj&;BYm9Fz(m%!OHn;n_4lj6v~J4lpaVvx(mmrO7U3_3-f-'
            'S!SKikocziKBPmss<4+@&y?lXR`%}R#br%+gbAwHN6rMhjjGzCcf#M7|tkK_3+WLaQ5@-'
            'A4m6dR<v6LF`FoSQWFJP7xK!Q^(1WS6M4xcv)w11_Z)t|At@f+jG?sEKptefXQ7Vi81?a?4986{FEN~IFld0a|mYZ<*8'
            'T0}-miiqsJLu6%UW@KOIag~|OGE-Se3oU7A`1<?{zsK*^dG~mn$9TP-'
            'H(t{Tsx)ECeZsrFg1oc(4@X53spIBPG*R{ms=r!50xvv3Gqps_wG9Df2|ZM{e^1pfoq$U&fvC;F!oHW^k;PF&SFU(MWs'
            'S99aaaP~EV=-'
            'voRH+HE+xmOmca283g;9{amSf{`0YB*V7+Um9{w?~I`K6u>RAIHO!jD0#`!>qOCgw?$$&1m0X)Sa#(J?j8PyeJVTD2te'
            'Uz#KtgdXLsAGqB;x>Y>N;t4)iWu8Yjl!|mdir-cg8%Ji*o)l^vzdS7kML>oPG)Y`63)R}lOl|rae;M)FF<}q5OT`|*mV'
            'xBsHQhcln3v!I=z!g|3nVFZeNZ3E3@H*W)HRAS&cOzagc%$aLQpNEG!UZ!?twv>#BgL#Lo=VqI)P9tWeF%7$KXIL%`zq'
            'KA5^7kC7LSV)L^=*b%J-'
            '>=p4EA~`yQwXOw@o;^&GUr7;9I}vyvv=?=B?&7a~a+r5+B^ZndWA4huZ2a0lWRB=z^V<serPxQbl|+~~a~pA@=@CZkIS'
            'CE=s`N15M~(Dza;$cpKv;6?4%iP7kaAMQyr^F=n<IiN-'
            's+MD&r?V%^WgH6Q}m4dGg8RvK<*E!B+9ZK%kx_yIgFrV0XMts>_wKu%bn0XuNb!-Ys7D*0}%gsJrpy;X#Deq%rB-'
            '3NZh7d_#;#tA~*NK!O2pVrqf=S<#xu^-'
            '$jtx<l)g>uW3`QKH<*DB~zdIU=8PD{3XhX2GR4`EXzcY3YLTsXFu$@B*I?Ks)3Zz>(Hn4n#lf_N0mPxM9un8XuSB41|@'
            'OA2W>BMZbT5I=_c^M{GA+#vqRTHZ46n_i0Apsuxv7w`fb%fcE@+}y0`;o4i<uTsXE@7<B_eb%TUF89LuESFw}SfwckC*'
            '4x{HW=t4bgc{K@#Yx3Zb%>d~SsYAd073jKC61VS7LER@Y_%Tl$OU=w#B|a+1+`9nAV^?8oj2IYRe?@8;Ww2$;kFGmzha'
            'az8K(mQ_FfYGCCf+Hbd&fVx@#G}j8<i#P;z1z$Y%};;T*ea?i$F{M9Co(#RGVzjVgyJFGH=z4vF7nK;FuRL{PNs|%M;F'
            'HM<a`onHoq0nW9W5KYrX0;0$~}j^HxYd!Q&bLc8U8nIl=IWV2^0)aahWuPc93-g%R-'
            'wL=G5Lmr~7ekDG;{|$u~d0<gj0*q}hq!VVR@WFN!2=88q-v%!t_gzsiSrd;1Yr?=GQ-'
            '&T88z8;?n{dzjGidZdRQ&|!2<~ydgCEBIP~*-'
            'vB6QsYehWH)wz>zT$ab=H4Q`TytG^M6rYKfgfC<jGUJb|C`lvK3hbw+;VzKX-'
            '<EV`qN|?uBmCFbEY^n}!ta*)3vNLe&Cvni-#s>d0LLlVWhU~^EG%DT;1}!eI#{4uqA74uNigv-'
            'krH#N<5l0WDp2T<O4wE};L@;kdIBbm1g1rv>tlulHL5uJ^)EVm`xBISROjQ%@{9mr7;s*8(`~aE#5jfr-faa@ig0@luv'
            'EvejxZz|vrDB7NKVQY8KI3Gm#V)#}?ie^N{em4-'
            '9)b#Z(XYD(;_v%GYZU`Inld4km|&5E1RdL%0?sy}@K59k73BE|yB4vr{roH|;mr?*n$;0-'
            '(tA$i|6D@7=a*Rb&rgyc_hiVLlv<kB`-'
            'EmTM&rK&X?W`3FdcLHj?TYsAUoa+KNPf(e*Lvrc&!hK>0L;VNX9jyU*PMcKAzr{3iW?F=Kky-sZvY@&eC9FEVKqzn%P5'
            '5`BT_9lYl8oJ5WZJA5#($`VO_g6>$>)KWRw%qlV&IaZna1i3_$jlOdiGR2FPNWAz3W`)v+9c2NbH%f1jV+5>(om%$6w&'
            'G=CECnhZUgo77GFl}QoZST1b-`O50v~4kb`QCwjB5PntL=2VTjmL?aP_$^U#<$$pusXDu)h)UdmtzrJ5LLsXUjyW%5-'
            ';q(>Vdj5kLV&{F6K!EcX-'
            'Y>LVqq)Vv0`MgN(EVJQn{9<+6xEaSc>ON)dkew7~YS%NeKFds5fOhjGib19;^18&>#C9xHaKD)9S?F_#K3FtFh>jI`&`'
            'JWEejrdJD;l-wp06>O~Y&cn~&Tj>VV5k`yGKgzv87S9|pCYS!KhWj^-'
            'aU(OF9y_!iMYp@cCbwK7UFSz8tAA0Y5eapPL?2w#-2xYdk<qep6yGdMf+v~-'
            'RONs#MxXV=F1uvbQ(IA5x9l=eXp3R+WFLZM;;T_+(F1DpJr2vQPXojJ73otbhl;bwnDHSMwoDzzf^RG6I*u0@aMlc+Zn'
            'dx$Mt-EGIiKjUxmi|@A0V=Jb#&457kFYIg=QCPK-'
            ')1FsHo=zU9~t^{G$~@yo14};~>~SJ5CO!UI0z+jd*iiCwAB8;$J~;GQQ9m=E;l0t64pC_Zx$dRwmsgY>XZmN%S4B2MC!'
            'O!KxNv;PlvuC0W5F>G?xO>d+J1(a(WrPXxfC%5bb6=Rk*Dk1^T*Fl~%z!?wu`jJG?5)22$a=$HWezFQ)B5pxt@J-'
            'dTv+RCXrm4*vCR;ar>o=RnU(msJ!bYwojB`f!X+yenrI6V%xFD_x;7dZrb4TeDJwJI1~JO;0euENjmO6b*`L_NzmwEsK'
            '?i*IY=%hC`myub#ZA2y)Yc9Xt|_(!@<S^~c)4}CAL2Rvu@z^7(ETxK2zYIDrLPAi{vQ`8a-'
            'dK+j>T`<hAO#!~QCt+wWCp`5#k4t#YVg9ORc<ZJ(#?NV-'
            '@cBhVWa0vK^c%%qYXFy;62{AAb=a8o4F4(}!aAbOKG$#oO(Jr!%4`dsl>bPL!girne>|Ryk_KfSNvwFs4NZgo0FQlegT'
            'yy@nQ8(@TMa>c<RA(OtV2C9ag_MU#xnPVXxd;$osF~5U+*Q>NO7}EE1zIboivt&Fwt#sFf7P^3m4pP;Vw-'
            '@){_hU__9_4*B=jphE=z4#j^{9BR>-'
            '}u0NvAx7twqqZj>gWR`Xcm=d8qycp{9nBiO>fJe##h~Ms?bcoSUBMR5Dv~E^l>)seFjZs9#)lb;>je)DyTalxtKWTEjA'
            '1n<XqlYgog67LWb}WBOY<YsQiR%GPoe77j_O&3tWsKZ<{vOAS0wH@skm-8%5XI~qn0|c^Sc_J(Ev^-'
            'S#g<BHSe1rj;SBuNA%XSxvhnMKXv(S_!7uZD$-BSF3=08E^sCznZcUSnX64<eV=G3BTD`IHa0apKIZ8X+#Yph;dRi-'
            'f0=U<uQqzZFD8};-qaO#s<;B;);yx#vxz-'
            'r3_nV^p&2p&H`NL=g2^>7~6BET^P^@b{hP~;6#aovFW8r^jS>=dTk%uv5f&-'
            'rQC853l4CBz^Hu|coi*C5nMwA_cD5(;|%AR_B^U#dE>e9gje>T$Erwj4qWi3`gNGTke@ka2J#9N_BjMz7Ss6X>DF75Y0'
            'g9q!#Yezexr~8vJv(O8k_WWio%4&qRq7jxx-'
            ')z<HQ*C5;vl@JA=)mU|Oq^Abz|fZ&=z|*>{BPChc5Yn^UEu&%r*`7Ryq}CMr$bP)I}A>L_QpeUsibB9BQnEvhgJl1Q#X'
            'fn?3j#$R)O0f*&K>x>AdvOvww7BdlNP__Mp_r5GZ{AM>U;9psYKSjy;OO@=~Aya+%aT{tQhOt|aTz`*5uSWwjpGBO9!w'
            'p~$ENHf(H!f13<J=$|0-*{K$oz3%|GeVaj3mYYqpFOiz3#_+zuiY7Vz1U%*qbuX4OuRgm-'
            '6n049R7N=HUH{0cQRZjrb4P;zf--tyju)zAMUdn63?uvYXJCvckS8|@h@3~f7*v9{Irf4^4>x<ptpbD__d%asAY3?Q1*'
            'w1JQ8D`rx~{81Kh-I$=n!PbED=Q8-'
            'e4+fybgCNe528JjVNmJ6sFnppuc_rmb>v`oh}o6Kjy=G6@RMuOOHW2ZSl%LHrCi&qn(-'
            'mq1(PpS`$uK5vjUpWLrzFKK6lQ|4x9*n_4VVOrnDitHAJWJpHsW2IIVhF)MIB@O-er-'
            'rXU1_f!VH;U8s;Z@tU<A#4J*(Pt=o`&%&1e@^!VoFG@s4KdK@HZJ^~2on-JA#S7qb%jQ-%|scKFIa-'
            '`#|u@d^?|s5OAdX|vzog7(gxd|L8NcZQ>xqQ39g3btj0^RB;K_Y7aK~0+6N6(@~U7|ay`egmGR_s)hms9gEM&7V1k}Ks'
            ')c_QuE4h42^iBoMUJRgF(y7oL#SOSoGZz}sofbce0>XQiEpnOsyhhVOGIJmnJ#+LL6D3UTcFmW8&EZI7Qf2y!#z9kIc|'
            'spt+YpAZ1)p3sS2=vws)dD+nN=9<RD1--o$Fn8j!4SLiuB<u;XD7&P2Utn2DaJ0RxZKQ#Xu)L0U6@w_}3gN^ew*`%8*j'
            '(m^*roW*!j4NK~mL(}I5{CiOwPbf6uxJ)dVdG_J{;Ma6@MF?ZryqM}G7it&>(u9G3jtMii$DxA*4=CVMT66vrtqXh3$}'
            'hVB>E8)N#hZ|t%KwmgaWzPv`beGL)DyYMUc#di3-'
            'j+3Lg<qkOe(uWD*O}~RwcCz^~!8mlII87&RHaGo&+)X`AEDLRboKF9CtND(a%z!;Y6Gxv^B**-7-'
            'F=gI_+ljS&!+PlPOWVJHe&fSVTYL(kA2Je$-'
            'Eo=aYUmO>7*!assxWT=E69?7wfnWf<gwSE$QP7Sg~SK^fw!Ej{ZKTz4RkWsWd6L;fdD6siKa@8a8!nV_J_n-uv*s-'
            '6aTNL4iiOckX`FfP_Zvvk^n<3#77c*79j2IQNvFGv|x=CA_-'
            'O3In>y)2i<cHrF)a?&E3KdW#^d9RM@#2bO{?y~!B$(`orY29Ksf5)SSgpJOy34v4Q?mKEJ;Mi2e(hr@OL?HPtq6wdI>7'
            '|D4aSa$lGAnvu=mRx8(N0pz|K=B=dR8cQ7Wg*ca{vV<dtle?|#hOSO*E)G?8_gizaW5$0_9$Xb|%vg{_KUrQ3trAqAxW'
            '#eA?VYQP;l&iEt<L3M38(3gC~C1?p0Ke0iv$LU~ioDW~${UGm?lvyIS1GIZ49oxHHaZ>v!9&vlCVazZ@tHLbQVGPm(EK'
            'Ry=c#!^T=EZTRGz_|0gjy#9;j8r}v<cSGcz;BP{Ygy&dG_svQa>#eU*iBt4|8EbqB7t^WR#`sW^H4#p=fUbthFD+8y(B'
            'AHg!Gdn6IKk8f($SHj%NpCx~1ah$Kgy%5ZHz7q*QA0sqw{c;i|Re%D>acJgr|RtbtooON*Xd|qPbzYkS6>;#_kvfvubg'
            '}-*XLd406YKN3u?03+>KSG6&KHEvWvL(@HGMe?~y)_yZo&%dHY53foMy4gCAT*^7&J_M85AXC-'
            'g)u$+{q+#UJRc;=!miX|$`Q^l2GX_K8e(6ZqxHiYXqdE=*jl{<`$vdwoMo_-PYB#YiXi$p2g;>0u-<>&+@9<}DqId-'
            ')eaaM)PUBt!B8a7iA>i6H2Ib$JU@L1UU9Wy^z9b3(e?zRAxSj!;9{ByrvQHz6YeiDf#q)qHpoAQ{4-'
            'N<Hxn7dqBrQ`7f#f3&m(B*X2MwEPwE`2gg=d>;8M*4s$s%Q#yVH9rsMp{fVKy+r%#Z+s!)c7%uaAhet^86Ps1j&GgN(C'
            'nVo%(m+jsk1uydT;ebafP5-B;;rOHxl!D(8yA!wZ!14|7-VAB~_cU@bHX9=PT+nzy8fCSF0Jaumi02;sHm-?-'
            'Rr|=3L;1K=aW!0PP9!DoW|11-'
            '##A3w`1imbz9|l&$M<k}7`On_R1~4|vM}RoCKIaWd&9!K1pFN`QMDlR5c*tsOT+ii$LZ>9JUm(ozuu{Xl+^=>-'
            '+qKHQ<h{G{D@=?_lZ(b`xBVM`~_#{bK(L%Cz9W2P7ZhEf>n<VRdP>*aIrsBJ?184?;L<V5pj?n8H~%Myy^N&fzWDcjv_'
            '{xps}|J{uBjiB;WI5y^o0pF5dvS8<ASQ>gO`#-}jJF)0V;r*Y3i0fo|GddK=SsZiPU<49t1@80qUGmd%-'
            'Pc&nF5HoX_YUF!ExUC;}Q{8zzDlN6D<X-'
            'fGUw4vj|5X<pRG%h`T1AecHW;v=HgXD<!D5T>{M4h>CyjlVZ`V43``GRUMtZ<KEEV;K(nTQ&ckg~g8DD+~3#**JcV0U('
            'g99P(Y3$725z7K|gUeBv#Gv+bV=PzU`FUx~?i!8XL8U`-HzlcJ}EiAOG0p1)L6u&5qyZ#-4m79yvaxxsM7O-'
            ')@<yLqVr3xXJ)RD)%gy`@mz~EY6#+JqXbe3x!T%Ys>0cT-'
            '2(Rv(~TjfEu<{?P_qKpGVitwuY5B_ISiw{(8!d~u6(3ir;KCb(lY?SZ7`L7PsW4CfJUtATX4KlG*Qw0B*Jc8Jfeh}Svm'
            '+B~&)7ofmxawGp&fN`g_fjI-D#cTe<NUx%mO|AkPWCNlZdhL+Np<=BXz?B)kaT&5e-'
            '5XkNR0x0DAkN7S%JvBm4F+CkAuke|KM2UFP6T<L(n$b1DU5SS#Kr-'
            'Ay{%bUW-@?Im-^>=78t$;71UyT4jPVLlIEsmxNiiff%*?0hm7R1--gmEXm9F;h;Mo_Gv8wBgtTLDajYtKM#c)Kh3~l=_'
            'Y7As!TKNtKi;Ich;H7WYjASq^S}muq&-l!{Y7`H858oi;9Ctwy6&8Z_P&+*IKmD-hhfd6GW-'
            '#1TKaLxaKVvyfVlErQ!%&lJ<@8il$>gHW%~SG$(WCl_p54UPHW2w=v$!+<^0syRlYG7>zatprhSG8Y?5Bk@i*}9w-'
            ')quF`LoOp*m|Gugrt+jNs^sVc&TUu9JH@gl4)KLiTPU(;Oa-'
            '6&~&2i{MepdYP!h(KO}hD44N@Wm&Sd7<+`^OFMW!RI&R{p)XVR;m!z?}-63jKkB(>p=0D5155qKwjPu{L--vTA5=g8+{'
            '8bIWFQtRu<$=E@Kbw$w8B`VB~j}q^<l6<l1`?%K2hpS3m%!&VPma3zYCsOfj)6uYy^d9VE25iuM_W!J_Le_^C%3s;cra'
            'Gb#{GGq2+L;t#Y^=sQ$h@&Wb{LFlgIgI(Gq#9vC0%+rs>jD@#Y|IA$I=6m;8!Cr%CE;EG#rL(N$=p1+;*F)#6C}(+gHs'
            'H#zL6Fgx$AsQQ+$0=<vzpE1?D`Fuq}BjN1+Mg+&jom@iLkx;5W`ZS4tJj23Jn!r`0vkJxc)mE%04h5KJF0NyDkcz)uzK'
            '|aW<?kN+9osreI>yind?9Pv8DcVo8rbg(tn2F)DZoHVV8&Dc1yqqeB=N=L3Axxs3QSGh*veN%=#c!1P--'
            '23=k-$M&o6!;{;<Q<8~lYyQHD9#7=bJPfie<%DCh91@13P(OJWuGAW&YXmzmb_oOemgLZP@lWA_M-hI}-'
            'Ua)A+(jO>L`;*okDfmhap3P0M(4>I7@p$-'
            '@BE7>z55J&&d$RDdk2l#+79|H{vd4{lfowx2C%%s0IyAILf5KDa_(F^NOVLH-'
            'X~RX|AQ;4T=s)?{Nr@+juLe2`9OYaA0-bLe4*bJ>e1^(H2yIQ$MJM)nDr|sx+hX-5}ThWb=}2|Rrb~6IV-'
            '`WLk9}pPi_Cu^Md*}3PT(>gRUO4fzs+jxN74d%kH6Ab)%F&ym^;I{yhH8Fx6*+ud_Fmo$<mT;}-'
            'lf?ug4x*W<^<Be<z831!V=h}(5J;9q?n8h<6C@zuSs)X)(!>Qj(w%$>$s730&>nzZ?HG{o`RQk!|c(6ZYHx14c;XE&Na'
            '-LMm{jlMzG1RcEE7Yp0@xY$d>%s_Phal9|(2e-Ca!PH0<eEa*6DCk_(&{B#bitohWnw%&6ko-uPs!Q<>dlh=?%wz9f7m'
            'jsUgNjKS$kYtOLPG=O>#xFu%SPnW5jNHs^RbH}PJzQOV<^A#oSfYgj$`S2&?uyn81A}DW@eM%wCWbTRAPfUp4Ax4x=BX'
            '^WI?}-jggua#PV<y{jjftkrtK+8|K#%g=SB@w<Zxn-i5%O$Cq(yNj$c$SD=5N3Nr7GynuTuY^srhtgp_YNcU#pK+F=-'
            'C|`h*Uw&ZLYc)9jUVu5X+nHQeU5}sV%b|B}B`sgpPh|ARaHUZGoVFU!px6W~`xOtG8rF39&};ZND2q{pH>*0>O3dfwS-'
            '2{j4<y81<7fUY^hUuRDA5o=R~{~Gm)6H(qjWNBBLy8hN^u?Rr!vk<!TetrU1h(BZTO!aM*1d`ms?xVwsa{hQP&1e&m{a'
            'aoKM^CKSsSxLhKf<Gjx9>C-'
            'MDt24~ip!^dqcq)06S_`51lthEZ}2{a<lumTQ68sWF;czW$!C*%G~N3b*MhI<#(QFirNEaSZgZ(lqjlab!66G?+e%@0t'
            '`k|74JN{4OUhBW(18R!J(kVD`8v4#>(6Qvn-'
            '$hd>V?4lQZD6b*os(x^z$OA`}?p7K9GsB=g(q#SCS=Jl%P%!x#hs@l?SYYvm>YQJMJpSpd5bUDsGUci2mln!hXbPQnxh'
            'VhmHc`LJ!w$*K2LCTx@aSwA<G3jwCNOUUzh5=xxjojfIJE^5%OoKqXd$(Em5s*ojnKR)5BXKN*(G<r)2uh#j8j=tcwKK'
            'h`DR}YvF8F%`HB|WUJrn;3oc@CO$HQtdsEGTpVd2fJv6rPR}zo^`az|IgH_p=NMAJUq~CYt(fU*as`$qNo_E<%!?I!wi'
            '>}7wDvLB$N`%9QOdq(fAi+MWs)Zp}t`ni(1g6`q(O<z9y4yYQQSCW+z~W@zamxaR`(2oRb`_6>Il+Ow*D%7fN@HKj5a|'
            'xtftNK;kV?KGu#c&Q@X8i!Ilu(@W&~y3_i)<M2xg_asIzJ%PUy%&_L+PdGW`iPsvf~ZCkI?Ur>hI9?`!CJ-NQBN|I4$^'
            'Vuf`*URO+k%tBrAO79TxO>TzKhEcjRI|xP(EMe1rUu<>f$D57m_+z~aeHm;9I~7;6%W|49y5J)&UF!$K3KJ~<k|y<OX9'
            'YZ&wSoEFssU;Smty4F1?<#>v-'
            'oVTrl2V$yyn1(wqo1yPp>0wwf+eQb9$lv@+|)ST8PHcUxByh8f?Nh$eX^N%p|RWOD^Hy)wc_uo$bP-PVuzQgadB{XM+2'
            'oZg_t+5X*LK2hJPYC`a`=xHvJ7IbxXy5gr_9!G1{=f7$}8l?B)zkc&O@X9=wR-'
            'h!Nke^|2#O{@~8AX{VlD7??!1|bSr8mcC@A$w&lz4UM`b8*{#!oJr+vb=>@Zyp)pYzd3jAL51kT`w3MpWEo=p)DX9tB<'
            '|ICva<-3)*E%;Vy$o%9U~mYQEgY`-'
            '9F<d8CV_IaUuJF3K@w_lq#4w)CUH>sBn||Ag1?E8#znRLma;!DErnK{3mVTooMC_&4~DvHh_+#D4X~mz6(R)=Tq&H)#k'
            '84Y=6%rCE57>af>@aj?ZU7~#VC)hs>J3{aE0gNoPAu+A(jfg09DjCxZ-yG5T9arXnnBC`*NY6w8~B<cq<@ZL-'
            'S?AH21d!t+Ff_i=Ij*bDhxIs|*FPPj*xK6YWp9GI`5m@z5ki8^n9z64VLH?cZrqQz*<nf+j_!HGc*FU!enXey6)^<VAQ'
            '{ZQ}XQZRv{CW(wt-ynMw;`o-f@oyQz#irWs1DX4w<JCg!N-'
            '4y#p{0LTyzudr262m782Xh`Ji&}GF*{sXJsm<GakBi)7G_D(b)GZJov(i0Ry{XM{6y3re<OtxeL9nkKo3eJLFSE6u^=-'
            ';FvF8eQ)Cf@T)6;nx?(1uML%SVYebUj;O$bU_MkoFoRs1?eTW(aTqKRCR)sW=)gx%L8Jwm%LXu;*B|F;jL=FRMMx2Rgn'
            'vI9L-y<pW6{qdJoMBab;Ro#m)slS;V2*MR{BgmZ#=@Dl8xZJe;ryK<pFp6#TfPfPWmkF0`IjoD3a_C8CF-'
            'x8|lUDKu1dqJM$76S3HG=nQb86*@atM=ZyY89&D4^hxa>jae=@_xOm_?E?Rya#F~m(Z%+)7-'
            '(8~Y`$0Q^Z~GFISNn_WD&okJflj)OGX{Fy&EQrlP?O_^kl^+bhV&l6Wuq8y?X$)x^>7d(+nJ9RDC2R#4`4HU2_}E6$JD'
            '1OjFbie#@_KqRD4w|eEgSy5<|hz9mxyISI#niNV{W(yAn9g?b2GO21bu)7}?Vojh*(><Yvn*G`%Yc-FCO|j8GOzOcg?R'
            'vn2d73d9BT&%>^qIowqKMO-#;vPokvL@fw}-'
            'j#DR?$b@m%hK?0N&|hk^&6V}ny$9Uv?iy7Lx|<>B8{yH>u`=pQQ^fh)}%)Ye7>xWd3cjPlY0Um&e@Rsh}Wz;_g9mb2Is'
            'IpOAo5H@Wbi)YeaO}X1wvdgyssxz`?+H*uPVakTOMb{KRvto70H+W0^Q`<PhzhY62*^Lw_AC2c-'
            'pExZ7n7mZSvG`(o47Ku-'
            '%2Rr=9YZH(ln<)hj215g_IlU9H0LWBI9a69e_z3g!o|7oYgtwDZ}6FG>Y<A15nqHg+VVG54r$YIfGJ+jWS0H2&6qwT8Z'
            '_|>Zry!q$R0`tT0<a8XIIqd<{j4-;tYd_S>W@ARW7Y>*7QZ;KXxF-}#ch%~mnSv?(6c-'
            'ME1Qh9(6_t2DpoChj=s=n2`B2Q@WJYonz{?|xp=iT5{9k@R_BC?-'
            'KjbGiZ~Xs|pLmRd>{o4CDDyXwT=|!bN92C9_U>8^s#P5@w6qlWyWT<hg%Pm)3Kw&w_-+_Dd<?qEMVJA@^O-'
            'v*m$N^L`Qgv^&v8n*2ftsMA}72*VU)-(%vbkeD6VRt<^F4-'
            '^g=zXyLF3f8dN1K(_(4HCm;OJPX^?bKSN>qS(vPzV4bch#F0Beto#oxpy=HWPE+%kWZ6~V4lbpY>kr^btx9~mG!12v-'
            'RUBiP}b>*RLrSm;`!6Zs2h7B-5{tBSGFvoICmd{-'
            '^M|C`zXtMVG%j;x(rDvFB2GVU_W~b9@Rgm7uJ`6;+Y71a_A50%Pl9@uliAg%>v8;_6|_Lwpe56XbIN8=VvbyE2TDEYr!'
            'Rn4cw8fjMlIy2$<l7Jw0;F0FyLyw%}kdz9z*k6kZQ!XWFQ}jt^!yw&FND0SC7VvnS_sFeg-Wv9EJAG{8qNpYs^S&I!=#'
            '^&6|sOJo0cad!V(Eyz62j~$M+U@dKis|-'
            '`&!p0i9&A<UIUTgw&yK5wCWiA!26$jSNI@+%!u5oO1j4}n(@b38(a1hV}mr7;I-'
            '<twaSJiQ}_#~L7v&km4SahrIBRaCFcuu7dyZ1H13lyaX_QsOB^}pfK?oIgPl{`MZpG#$<)$!A>QH}L?IvBB%i$S{~T7!'
            'cwLCK9iSjreCYlb#}tBN9_`*u_reRH4t^QFKcxdFCGHxf9cKwoBC!m8N~aI^Xtp7Y(06PGsQmfALO_6dZf5TWX$`*QHp'
            'lq2qAE`!jp9WZg>A_yn1K!fyPsQ*y_a)x@Co0Q0!8!3kRSyOn?R0_ZDnGx9+_t4>}H`<y##Wfi{B-'
            'gqKVlDUK>|PI4*|Y>>j5pxO?nf+6_jp{eryRVb>)^QcRbbyXLh;cU{JSIp6+^2q?2<eFm5v61TM=+Br--Oo9HNDVX5g`'
            'A8J4(e(o>r>=X|3#It?y|d!OP-'
            '($rsi<Jlu<9?8TdFTKIeFBR^{v*=(*El8|8g_UzGbxJCdFs~b+W4k4~uwJsXFXZETuI-STQN#$_P)c@4*<zdG68OAgEq'
            'dO&jlF6)&@WboQKz24qxEmd`L;Pft8ot$Ci`jr=~Xa~D+;|-'
            'qo_s8C$fBz2c+tGK*&E~<az0d?>6S5M`$hKmMEkS5&AIdr~xlr<3Z&I3$(hTG{TrUptkcM%v%=3dikM-SZ+w8f+d^a5>'
            'G7TD@I~0(-Hpknb9ngg0~jEg&WbK=y!HbyS{!RVGUZa@mM78y3+!Q28U4mN)rqmNFkD^*l1_>0{;s-'
            'grhd$@N*@Cm9`{FP4gh<Z_LsT{&0|veZ?sDZG_Gfb>#kmyU^LyiD#^}iQ3*soHsYmZI#dHZ<ST}x%&ckvxL|WdH2!Thy'
            'eIoU5>7jj^ujVeG*+m@sFh!4t)$D&LZjPYnXx~@A9yH<pXSOO~ZZTJ3+=G6rWXX#RqCWcv>e5N6k2JW!Nj|km!Z!lNtE'
            '()JGD$mj|LxpMWp!-|_J9Ef_y<h)VZ&Xe6cX!pQVbBw%2i<{O^CTf-'
            '$}GP45q=xbn;Pc0Vr6p}5fl5jgC0g}yJi5=C&G@EwhSt<u5uF>TCwJPWje*%yB|Izggui)T0fP2OBnQ^ooRaFSYcHG2A'
            '%Z{UtV>M`IF+m7JF<ZhEY5rd{%}&85uTO(hO(cykJquk)UeK292$fINP)NOss9ZaQotv-'
            'Y?oKy2BSMk#WD=80qt#YktFcjB9b9ItFn{MevT9!>_^B@^5hrzlsS`mjC+9)HnMCk97y#DB-'
            '1NMHAWWoQhUi!O@#T{J*f`n^igG+ya3%@l_pgEz4nd&zyaD-bTZyq_5DGt!C5z(Kh|T9Cgj<6_R0JR5!<cJSCO!w2*qD'
            '-M1~;HPRfh32+7ryOs_9j|8?f!154A3orP3+qfg_|Hy|-'
            'GRpM44V>v7RTrHaVj<bzSEW_VSXg*m4pfO~Tj#s$b@zd0A2F5*W8gG?|rb|S6Jx76iDDinD1VU=?$=~d`qRLHf0#%~6w'
            '->63tb5Elxf9?i->aIR9ae{0-'
            'Du6R)qV#je3*2vE$4ar9rWIeGlPOn%E;r`h;4=JLtcsgo?ga5qP9#_ADy<@;m=x=U5uf;&oEQn2G#hhgbm7IKD)j5H1^'
            '@3c7^AX`D6V=-'
            'KE`rmQE3e0W;GXva&{rVMh)fpy9Q`O4Tc9S2N(PKY!!*)@D+I=blH3qIL3kfA1Ww&^&~CcTSuSkP7ppTEnIW!1k76+3?'
            'DVDNYB@C8kbRww#vz5Nw6KBH*6udhu2`+#C<q$?Gurlw-#^f*J9_P-'
            'I#aT99Jp6A|5%v7_!Iap%#5yUD_pprX}1svM>M)-t3?yb9K471wj6dDHJ>YjmV0yP{!4UI9B!m|7bboQ+-'
            'V@EsBSLt3&XcC4l!YJ|@pJipZr|DRg>KKm`wOz{f#+IF?<CRU+##%B39sbGr$irnl)5sU*B%SObsl`{MMM5g27P;Ig0-'
            'w71~|2v3jTHvS#(;=MbH$aS(}=V!vgi@prGVrSU?xC3@42fzX)O4Z{(<Mm6G$T_<PMsAnV9<0WH%gxE4ktM485wN)Qj`'
            '3`ckIdOO2#?-'
            '1kozwh7rqQZVbi|q%S<6y!}${UjBdg3OM3>7@fCFH*#)|_T~#j{duW(N23~v>NX|?LP~U~O7~lW6V*SYiYP&@m4|p5|K'
            'dS&-e%zgWciDk|YOi6IS05C&aWj8QRMT+R#bCK}A)anu%Jy2`g<{fSw91l)>W^%NW%cVpWjKO-IA4yOvf-p8>K(aV-'
            '+)hh+t9`B3ij{RVpO_!!a$C##^u#z@Gim+l-'
            'Y%_B2NbOTespE`#)T$?#bZr&_)r#>u4%gM814X1rqq1I&4bChLU1rO~sOJzaqh~_blo4-'
            '2}S@o5^v}C|s+Wj*A^OVGWl#z1F@A*Zi%g+FLTv;9M=unv5hRlg;po*Akn$mx9*nW!S4ZjM*<LL4LIqt<~XS3Wpf68Vb'
            'yy^8He3b>0y6ayp{fne7n1BOK=-'
            '4sK)~CF&uCz$35{S9F%)sg8K!xy2Hu)%QS`bq2mkFhLHRX9NSYV0V;1Tsu0#;x2Wo_VM$<2cwRdV=l>37mudeI#1BjV+'
            '|%W`=jZ%HPEVm4+Z$1ku|<i)xB;bpuNu%E&sNH3+E^+=6*Ec4p3qU&uM*3y3HI@FmRz%Abc{HgYdmqU_7W0mGosWC+Z?'
            'hit*x$=%Be$4Iy(N8NV6ZQ~Q%E(CdRIZK$Y$E}jnZH}M-$?hyyIZ&fHU5Kc^j7gUwc-lav|o;Y~Y32T-'
            '|f$wcwydPhLmtUVFGU@ryyv2r0myM&#OeWlv5W=&Ud?8A1DgJT$j%ONJ*j6Nk0bQ>c7tWf~-'
            '$oqB@GgL$aw)tZFid%ud%)g_c-'
            'Z5<72a<iptSi043=lW@{ZdmU15&uAFdGp+UpQBM6k+7g8o@sLmK%g*e&5E|H6)=Y~6Y6UgJcCno{B1`ZCOR?j^tFy>RV'
            '}Bx}{TV9-'
            'y~!!3znaL_DH!yE$XaXuz#pLs#;uG&D}?E^4ro(I>9vRS1WNIJsvsG#K{R%Mha%GrmJ?1*PLW`<yUI34NF{gAsk9g-'
            'Z|sp760BD+hrdS0S391{+uO&k`K&q54SWG`Ufoc~&@5k;&Xsl$MS9hOCWK)23z66@uUlOICy{4-'
            'yWA0MuIE5^YLkof?%8>QjfhGFc>nW0wJ33w@GA6zmA#{8%}JSiCm7eb9OFt-'
            'S1gujpuFCCozm`ZkYX`w^C3q*?T#U(WW9I4-d-'
            '^PHh%f3xQt0stbIyc@+lEjJJNTU2}8?KT2&MI4>hDsX}amU8xu;ukv>bd+Dy_PFP3pacx1^@jdL*0~?kEh|<mlj|pd>O'
            '9mdD9iui_zvTP^G8}Fx{7g*}g&`!fAoLeFALZ=qPFwl8ndSK7sn+Wwczl47*F5kXMtN=<f;xxNZa%oF0UWhYv6Ac}LT}'
            'a>0L2RV2S{g37y(LZQGf+U~9ZuS|z&wYMD%FIPYj0cB|XQchH?C&9ddgCxBFLpD1v#fmE$xXyNx(Y@n5*p$R#%gtgMDX'
            '{^TpEiT(JP(i;W5KZ<@5$BdG$@tOsX80pMz1(%p}wyUlxEdvbbJwFhb>8<7geMI@5@rfofA;HtCv{4?g7!=+|0Zc6)4}'
            '>jCr$sNFNx0*Vab3IhLxCyLA(b*TIEl;OPth87_v`84hF~cM%47H-'
            'k*%W+*fjrmB4`jrx<;xcOKX(na%NmE?2!)hB`+7Al0mI&b7&$4|SrdBMY1YpW$U@!+kNdQx~X8!S`8@pGvOe*5hNaz)7'
            '?5_tuNmH=!&vX^1~@EOB{8q>Qb1yDIX2}3vTV!xX^h#V-PgPWv)d#D$;@8ClXTR%)JUIb146q|k>q2U}w_+YCgc0P-'
            'O_idx_dE0)ng}a1N>#`mbT|7Wsuazv_6$GkW6@Y9W{OGs?uj}4L!{aycv9&3f?URFjX#w=wkHZj~bDMSdAqR5umH?l~Y'
            'ie<&lxQF2rWYUrc8Z0mn?~KhE$gO;TYm$Dy<G^s%KKrPbqwy8w<P)}^5J2(GQAY`gNQQ%@Q6|y?3*_NACC{xZR5O9-'
            'P#Vz21m(kffY!{Nn`8TAJpidB{*0`LZ$zHG(58qwnHVWIWY>SIX;o9ubb#P|7ILA$;Z4qH$m=l1w2@o4BHYPfnVKvEU0'
            'X!H%^tJu2Lghc{d8%K8C{E+0~$@jlg?w106me1$X#EVS&|mtma+;<=LCD&8e2;emO|bFBfDEyDq~buQS-'
            's+re^;ev8_*^~mes#afhp5Tv|b5_@xXh<KBa^4uKolZS&nqA-'
            '9yUJMkNP{8rwIiAzGhmI`<^m;}xevE$$cO`B@;QCYe;!7~BvaciSzi494T?uyG<vMj|hh#jU)&-'
            'WeEhsuV$~wD}4_p0G>ExY<_?S6Fq^?n*iZzTPk8fnpcrmaHf~d~?aO7G&jed^;;Ol8S@=wc!47{2FbKl#zaJCCfIv#+-'
            'xF}3~1(-'
            'A(q4~@f99jF8th<~G7uRe>p*P{U+DsX4hOdPEC#Rr%^J;R^=OC)iTB6qdZ5Y<4hBaOKSjT6Lp1(a%JZ6fje4x-'
            '9@Ep%2s$xQ>1TOYZgZR?1YICO;Mx2TrUJ$v#I(zsgaC@GhN#5>gAy!D8qV{86auT|Z=|WJj1-'
            '7n;rLog*QF+({KKty2QS0YqyTdj-w6g@AuFGPTOB$Xk^+hA;O)#}w5>?OV!5~Qh-'
            '5MKMT=S1)>Bghh0$VuRx(cjjU!nRilb-'
            'UCKvyGEI7-vu%ftt&FJuS}LPDgoQ<8jfjm5<E8rVNGO!l5)pvwXU+Pvfd6<cr&ivyoi<KL-pdu<?Dap?e@EM1B5|J@_{'
            '##iyf<iqN%NwJW;HkIb#2ej9?hgas=qt)dV5dHTJnP02|s~UEpB<BNqeB(w`EsR3u9OuL*Yr+=y4D2_#f)!DH<g0iXBz'
            'T%(-2P{HL;MV`e?W0<tp`m6Evyr3LyqAn)aI21xjC;Bq$G{@3Qy_rk70DEp^xmXdk0O9h4?k-'
            '8Sv03uvpOzVQM|};qnNge3BnO9+>OB@K5@fRR)d~p0vZh7n@Yt83D#EMDAM<!#?INF13}x0R9rZ>hYFX+%^Ha`V`!8*?'
            '_J7ZfK+74U!wPaGqidXq}Fv2CG}C&wsi2JD8(d=fQnADYzC>O#`9TH4N^oA>jVL6G+o1U@OMsy%}FP*wYL?h0o#k^E6V'
            '?F@dea+;~WMm<q>ffV9t1@<Ol%w*1Uy`MVvfvf6Bedn0Fv^Fc3iC?Jg#{?`epW=`+vnW2F3$vHkw1EqPXWY0q{<XqVg`'
            '?g1sU8V29@CO47_uiy#yT<66SC3)8g%6aExPge&7%|aHpu2abSG%>P!M?lR7}Ah|ZXQ?hgvv^c2>OceUi^nQLa&hsbt|'
            '%UgEmaCq9Ol$AS?Ur2<(<B2c9k=GP)%T4+N~k$^vPu4SQ4_<Nypc@dqU2gc;u0`<2SwTMnJz$a;Pz5Zo;+;IXR-'
            'X{qpm`2GZ}e^!EivQ}uR6o#jc<`BoUA^P5V0orly!bb}=fUnCMrh3Y7&~!Dr``yE%6IzVAa34_oKA$~%d=hn^%EFHm%d'
            'uNy0OR|E$pM7}NZl7hcd`UiN#-XxYJC-=OSgl4+c?_FAAvEK0^D{v8K=H-GcUEC1&b{$B)3NZ2j3`QlO-E{vmzi>Ly-O'
            'BdOqGbu%6uJiNg2e+IXb%9kCDlhrGpk@Oi!~bk*J_>Xj2<P`D3M+OyGB)fJxFy{3z2y6F3NztQ8T8l70#N{v@-'
            '!t(JhDphqD>vnIz#=oEFWJM%IXzimrW^Y5H<rL8~*oQ^dKOo^)Fcz1n;9{x0_}B9@*cj~vBk2v$Z8rs`EgZDT?=SGWc;'
            'WZncC1N`1fg?hK#9Yawcz1LvWTk~800uwXgoy!cP;o}ZyEjQx*gn2PGPlNGAtWeS-'
            'nd1242&Wr6ZrsG3lHF&DXX?tHBfy-Vng}EB=S<i)bL0-'
            '3cJ=n2psI5*Uzs4T4SrovpjW*f#VUy<JNwhu00zIwFnX8wWI&T_4d<+1L(`|7Ah0j5q#y?SV}m{*18f7AW(1fjinB!B*'
            'W+a&(P3rj5o`PZ%-D^y2+=lDnH+<adUDxi|3WeK*wnUBOa|vc|0*jv(+~5-'
            'R1@;;di*F?*_l{#Af_p)0`hdN$r)w;7sRQ>o|cVAy7;1D}1*!?xUf<TJH`x_mb{-'
            '1!GRpK_w)_f7b!{V936wi37WM8T<@KS-I05s3>Cq>kT)iNA&-Uc42AUS)YSRym8vt4G7apjFtuA)B09Dh5&y-'
            'N2)s;_CF9cnc1~Q%@#*dY%IZT(V%TnIF!EIzhN(665EcG8$vn!EoA|2rdSPp~Np04etk1fxB~lZsb035@Mm)<ULHy(8t'
            '9d0oZPkNK*%`p}IE^nokemi{=`%%gQ1u^`&5Lr-'
            'l=4QRGZZ0X90V#sOt*h&}jfJCEKuyp?l=Dh|ujwBG|{TSPndUyY=4nd`CmY%AzYR)Ou*Sy;hcM3dTHF#3aR$cE3Fu#>l'
            'dRBS2+EyblY5-agyekIPPxuRsrW8!HPK-swu$gb!@h;Qy9LbY3IeB52^*(gGu)b1ln?RSye;V1Q)I1CY*t6)%eBXo8Hm'
            '~dv&ZrMB>6SRlcnt0N~6VB*arV9cBDfGee9!wn$!`>@T;d;O$=>EGOpMo}V)vm_iGsmGsUj{zM-oh08Lg$4?f{{l!vW`'
            '=nKj#hIzWd>U92byK9;a`>6Vt{nqtB0WRNrO~Z1qKWMYM{R+}(himXDG39~E$rJ0HgcLSggy4e({IkM=uk(Y)3JgHN1<'
            '!|7&JO{J7@{>p=!_7lWCO^!_dD`BkD55m$1O<3&OK@|=a;~|3rTsV+Vd%YcDp3yg~_&koAoN6F6(+P{l9>OV;B#ck`&9'
            'c<$glX#p@Yt=1;{l1}^ZUi<^lJwDxIVFt9BQTK#BGp6uoEl~tV5yXQNp-'
            '7O!@|_KybY$T;2YF`pGP0WmRor9ltlul3Jk%tu|X&D|_!^0{jPs-gg=9CWql%%{$o9d<%l#ilX<JHSS$!h_QXD7;xr1T'
            'stU7a+h+@$9*jj)|dv{j13s6BmW_nL<Js+I*nH^lt7}|Dpc!EhYjEU5{_g4;Nyr5{46acEgz>zK({D+-'
            'wiobQS$<O79ZnRa0Db9B!GnX5T5GgWjCDtjz8E1#6(~T(?L-'
            'm{=NxDIynoTN5UAcBRnu&=%Jyus1qWl6qzys!N`&5hq{p)=;>N66n`@xKQYrG#T&u?<N~I=STb3*#R+t!dr<NBWhhTAA'
            '=jTipl^Z)$h!Tv$&_h0UVN!ZW|zBySYIpz|K3ino(_Sx77tnKp^ebGT$c6Im__Afz45EnoPRK!q$_O&fwdKgZ?O^d81~'
            '|)3$;{{v7QuixYM`*Au#Uwj=#lSNN@KvEL|H!1<%I7ywXevp1YIomPRn7v=B4tO*n9BJyxs?2iq@=&}|%m4}a~&$Cm;%'
            'WVQI&9f6rxZkfjFE|tJOoT^T{W`@Ey#Xxr33m9%`2AdF7=+zsijLV)x*fS6J>K(v0>cb4TM_r7ShOc0BHXckU57Vo~7j'
            'nu!V)?=Vx;?7^Gw(cw6<g+mysZNscK;8zs25Y|<bCvEQ3$-'
            'xQUSpl3;3W^0M0`<sN+gyd@^ec1v2I&vs@Vp+otH>um>1Dl!GlCuNZ<t#(3W-2B$w=B4-'
            ')8@NvPMe*g5LJ4FIuH~&py(8Q)v8^54jj6d9YkPOTDOQ2cW1GnUfW1^!Kx_r%qtLa6o9SzZVr+6{`=J5s2cYY8a_mY%7'
            'vW1Z*C)}l`2_FKH?i6@T4NSraGq9D8whE&wr!*^aACoA1`+@=}!m-'
            'B{`0h$Cv6i#JwK+}DsWJsBg)EkJ=>q~q+hM$O1x#GHf?6tBFdZX;JU-'
            '@d@@XA)<aES(xn+#~%D3>{nn&<vstu1!|6$DV#lqR`Qq<K(05Ug*K<}Y3`hB#RJi5M;Wd6#AP=+^rne_odzKg)s{gEXj'
            'qJqbr*OGI`SHpkuf9OHyJlJZi2f}V?Aae!C-'
            'E(d@`$7fnw(DcJ+;)<u+0Mv2Ac=Lq@`w+QJoIV!0Edk)3=}Ei$a{jd&f^Tpg+k=FWC%Pj`w!2`On{AkHoAR~L#L}*jKi'
            '(P@KN_QsBdney|zDaWX2Y&e-'
            ')#>uo|vwS`DW@CnKleJDRAUM62p;QC4V_7)^@9?A3QL$n*w&pHW<;&<Ocs@%Tncnh||a3KsS0<NQk%__FgF44Q^w|H8M'
            'Jv2!Ip>Zm5pKc|3me=hhO7C>4ypU5<8k*G^v@H~x|>9Y7srQV-utl@u$ZoV00BCQr(?leQ?YdMscW8p}eGZyqMfK`VK@'
            'Y(e+I-'
            'ZfsO6?7S>?_uo@#7}6b0)&mi~0Cl#~Henvnc1&9#r5`B?`8s(C9)i*PNevu`(#PNebQeMi%z%e#ts#e;fAvX`&U=H&7;'
            '}5bW)QVd-'
            '=d$ab>un%Xgqh>go={MI5&yUq(DW`Tgmgz&qJIc$wNNB1uk#*S$qjP17|D^85+981F0g}ku4Lml1?Mk5WMN4skz(Q`>N'
            'g}6;%@A!%YPsGEIre)A|w3IX)34vS7qR@Yp1GzSR7!~jNkRQ8~L9pBjC)ao|jGrxqB+u)x`%fv0BSs%Brgt&&bC#mYEF'
            'VZ*tiZq4{;=QGgP}QG1UiMoj3cKtsc8M@IsG1>e<Drb^O5D4d8`mCZPq~ifd=?L*4{gu>+t&@r&1bLLsBTwCK{60eY_+'
            '}Mj<IwNM%K77?C}*_uhMtGGF(xBO#<^mXu0~7TRC$&%eJve%JNB-hVv*yRYYUKh8Oi^Ei(`&UOEj?F8?EHRNni5ZpFN!'
            '-<KN_*{UW{1u!g0)_tctjsKO7Pvvt#WLvGy^RWnc2a%oa`?>KOB0n?D91U;syyM3&U~hjn&iZaR2sn}_!1Rs3L!==2z^'
            'wGU_e}s<lUZTc{?(oxrxBoKnff^v=$XTgXL0$(&<T?AewTT;I{aD2wCL?^YW!3WA|Nr)h|~goX&#X37;7qWBqv1szGjj'
            'RSJ2GW-xr%9eN(DMkUxy6Q3MFIgVR&cEv)-4=V!he-'
            '%`Xu=W=X1%gOI1a2Mihm2=Z(6x%6dZtQ|;%FXBwTK|<zeXU#_Bc8;DN<hZl~^K>3+;hBF_23P5(0e*fA)PCa6){yW(RB'
            'A!#Aj}^o>m4FJO5Wx#8Wb`Q-'
            'QLDC}~I#xBphpx1vC3)4T6ofp3oaVo_+aB1%UXO^JZNFIalXem@LVMi66MA&<9kSxBf2If9i<VV_N95<W-b@OO2N>-'
            ')Z{lXw1afCinoME&Nt;7L`NVI`g-1>J3IcxGzPIb?GY}N^+5eqyK+Vb(4z-'
            '@3#HDq+fc;k7ABsi<NiPcva2AXp<`J%Fs5wvkHY)P3UkG?yClxH*Ss~UuL=bljTNFxPX6Ht0ZD*f$|20KPMq0@?&Y`yg'
            'WJI%der+EQM(k=t9sW5sfc7Ty&*Nrw${6JJBg&ezolnDK6hF6I;kWBo^sCN(8xw7b|n{0Sw#29Y-'
            'nZRpKH@sFkk2?6>M%M0C<bp>nrtY67KlSc1Dr$=0eEuSO<f<2N3yFc$!rvra<P%8OZ>C`xitzKvPuAXmO1ehiJvF>{8L'
            '5R3#(GxK)NW6-'
            '5s&~9FU~l0_cpxB(88?ke3Z>#h+f^iAHIhDC5^`Kh;gnqRG#62V@+w*KQD%p!bv&2ImC}Q<Kj@YUJieYN8*YlDIjp^8r'
            'fHNhYH;@g2KzG^k26eUGdfzzU^~^_}7gL_KrQE_`?KBc9?;lvpPzgOoi!GEtpL5fEyX=&<VMyVSX9bxV$Hm&IUNYs|6m'
            'E7SF|BmAsHJhVz!T<c3)iTDh^~_?>*bVps|eL2gvW$_kw}i-'
            'C%qBgidEf%e)+f~Nz>zbSi|nD2~NN}RFr*=>;SamQfIaJc?^32vN!6<6jR#l6GV$jkeEm@4oX^%q7$j#MEk7llFE!PjK'
            'd&U`%cBNn*Tvhjw2D=hk#L~Xwg!4aKc_;+9pUQptL;wCQ`JSB{+&lR!gTP*BZQ;vKpT=-'
            'YW4PU6w)zBYp^ci|f#L|*bAu*fq9!7?OeGFtsD8LwR2)WLKpzK~kl&>d}(AjX(C`hrhts5SAmSRv@KkkqVrcq+IQL+M|'
            'A+;2CHD!~oJxy3-'
            'qX9YVtZ7a_60VWqf?GL!P#*gj`z5_V>x~Xd9E*lHlSpd6O$g?<s$s&@gV=oa7_k2AgYvtvkS&=D*4zE?!jB-'
            '*!F`Cqy0;qH4(Q^ay_V2-n-`AQlu*;54EXai4<tvdQMcL-SG*O(eX_gJ?4<%2{N~5=HMMyEaVai+U5t-'
            '(&NEKNbraUL6nq*GiF|P!sMFXE)4S%*Hp^pDLIK%ZT1B|)ddcfcX;kvba{Bm2FwV1gNA{Ehm^h=0OJtVQ^>$D3eI^C5;'
            '0O}gDvs8+b};WeKWLergM-k^vK{(}5#M=gZ5GDEhgH*L8CMqFkf#V%L#{;X2V1Sj;w*UaY+;SWoZk-'
            'H#0G5hc4G0*R3g<_fi1tHh-'
            '3s8>@RxB+Nmc`elL4VhB6aT|91+8Oj4Bl&`ES|g%Q(>o+$QZJzTN94RVl358t?frpJwO#&$J)8WW*s-Ud@Mp-'
            '2e6oe#MKWf<b<il%FQiIH0t9Q{{KC)4&oUgrwXuUJQaI$p)v7zRToDFY*4>BC<c#2y7?MeZqujx&zXtC|RHxhKH=lN+8'
            'k<iQ_mE!em<97nihVLZT%jHW+;hBKQXRO<+`5*$HeRSQJ&RltW3SL_kKBbO=slV$BOh&{E<#8U4^O+Zius<8>eJ&_<-'
            '5zWHriYl-'
            'v8=!&1?yzeC2Us}Rvn*sYNJB_G(9#%K{H_qXPah}#HZJh?R6W{TvO|;BCu(MP9Y5@<BZuw{5HCe5_@Z<letoZjrB+Arm'
            'iPv0>)43x1HTwzYWt}07C|)I;{iYS%@B2~ZD4guA0^mNAlL3>_~A@&`7omT@zZ!iGn%nyJd+x3u_cy+n<3V^lib(V0?n'
            ')qaLbb)#J*@_+;%%kHXo$7H{JzPo<uZGJPQ$<V(|Rbdm^5<0oS}1A|XQrqB1wZ>yne?#I-'
            'QU6!gZK$CJcAwt)5Mt{WY`pN9IQuVCnbKSuF3L${*}tQ36>!TsTw64npZIV_0xIZEu^ZZa%<#DJZH8*{Gb<15Ex;0nBp'
            '3LUP{FBgTfuUQa#N`QQqc~5?BI*HtNIW&p8ktmk|tllDt3kK)SY1%5%qrzT0{wEZp=Vm>5-'
            'F#T~)D_SAcZ0@!Hjv~)Jh#dls#NFWJ4IgXUK>l+_^y`|Nm&GERNCRz;zih?TMCH|WzeR#3me)gT`SxLf37V6jd(!eP##'
            'z!+liYFyQ0mX-(;1YGzeI4XUw~)1?-FRSmC!XVj0h83cVfB8b5-'
            'ekuzk0izgcGwE+0&44i?7aoTz@jm>Nz!WDqK=K78(O9@y0iiT*z3lN+VMk_WX)1e2!a8Wy+6uPR&J$+?MilPg_`f>p40'
            'q<(8<eKBT_80uZy%Il{ctY^fyQr%Y15TAmxInCd?mQ+AeG3y{Z>}X-'
            'csvTn3WtCvF$0nXO3{sn2RyQaK>Y_lbbXM=<X)ue@&)J=kpL00H&G<L2(4}@;@SOx^Ex?5nqM+R_;+AsbtndF#6o0X9q'
            'cgdLa8^1Tfg^`pO<G*?5s8fNPVSI&wr8)SDhImcHCfR^b?97Q7kEFfB}PQC|GzMz7}6*<#XS{5BjoL+};8ftskiF5dpl'
            'xosALe{Nehu*L3&1mn;Rs1O3-'
            '(Ab7I_N~vCgTM8Q}+uQB9PNW{wHC94jWD(Yw6hNv>FMUv@LshIYAW54Ni%WOl^o;}}#xX?Brm3Ubt3ePm&;<pJK^Rh8B'
            'p-Ib12=l>li>X!czSjRDz7{U9Y&RKMl%gkZb=h?Y2n&<PIh@ML4Q<T!9T~bInUj*0r9E>UR}9^@%0&h%^6lJBl5E)gbw'
            '=Rt522S;8+RUrYkU)Jr9TZ3A~!9!_^5M7_<2<{#{cI8rx0bjms+h9PpUb>821RtytJODownt3#e_j5b$prfz!u}QB2DY'
            '*}l5Kw<~S5D7yfr&sKuu<U4e1tw%xK$8^`HV4VM07*Yy9QKh>-Xz_+rWKJJ|>&lzprgc4>=T^tSd-?Ftq#M{Ab5Yq-'
            '8|BWe!KF^m38qf%AF8Os;?-'
            ')T;>|_i8}i0`52NsWMgr@wLq3>W#gnt%X_$OC96dVrqnB?vt~k653YTq#h5Rwt%Kw|@ZP<ZU(+FDeTk+QM0o*-'
            's6(axh4k4cmGR+%ddNc#__ExaERv(4wrox)#u}O0GR$HJ(u{^|gxS)+xJU%HZCNAF?C|ws03{DdwYHEr%JH@GU%TY$O8'
            '6UmceF|<5c)}5lCBUtF1J)bY14K7L6txAZGyP;`NFswFv;|Fb;!tz^CeBp&;~S@NB5AA*KjKuNTbd1i*g1jOMmIRzWDW'
            'h@59yJ+cZgbbF7oY{!Y!TWVDpU#qIqc}@v6yU+_n~_w&7v$cSA2q8;jym;eU)=p<ujvE18k;b`h?~3c~H510Y#v6V6Lp'
            '2YQWr@iA8c_Bn*m12YwnW#5HgUR;KO?~m~9W?4A3&j>GmXa#%T1UjzP0ndf|@CJhc`i}ze;+G28Rgpp354E$3@`G{3=U'
            'zy^dK!NGO~jLps<6DEgu$mC2HCZVAi=pGi!CzfTg&4Za3K#=Q&mw#-HBXPx`4sVB-'
            'p)DpQhY!$HC5Ox_Z10(;_dxHLD)-'
            'zM&mH&gqsh^9r@<_P`RS`{3!VL!XMhC(;iqG4;t2*xoiwM>5vHk`L0T@vaHS7R154Hv;fdeItwq9>YAVIC$pHgMsTW;Y'
            'sy8S~=EM$z;rcK6g13HHz+UYp_KK633N#&qy@~!ICAZFf!o@mAn1Ho9@PBEpL=9y2E&B7slG97=gZ<1E`_P3cS{xfggK'
            'aiEZx#8vR6yB`;6GPU<Fv{>*~s+HEL0$X>fHDG-uARin<SqlDq)0V{qjsO4Rdf)!_DK;*;^hU}0zj40cn#R@^(<s%P13'
            'Ado)=4#+q+0HN@DVwWBDNNaW4~N;)=y&_~@ULZna26ee!0%D4)=XFAoqkOjZW*BZv<K7$%8?<I0V&ri3A+aqmQM2Hm_#'
            'gQ`c}dE14$_SaEzf8x0571^?*2L86J(u#_r8zPHTNIqgxpQbFCnVNiasbp0PY<f~IUNB*WF!kg`(?XN*4J&ox<Ks`ZAv'
            'zvYZllh>hJUJ;eI7$Aq{Va!PLh4^)`pu4*eHB2wS&o^l`GgtOOVPY9MbxD(Q-'
            'H8VWMiC6na>;wY`b!_1EF>{jV;CP^f{s%9q@cD0IBsMhZ?h(eQ>&-x-'
            'm#FhPXaY1_oL@L3Q8LMWF~P4G8h7QE~^{XUL8d~-y{^+H$va-DkVeh_M|HCBWX6bhxfttgpuaX;=kMr2XqQpBA33C-'
            'V__qF5<*%;fv(mGjpi1vmf~Wss?ZWa#XC+L-'
            'oNfko@}@x|QmQ)3eE%k}eIFn^7S6`gx&b*e{d}xQoNr^6}z{G`ti103$;7;QfEnaQ@UkMx9_T=>A}rPyClpzZtjV?Zj6'
            'oZ|jL8iigQ?n+5XL^}v;!T`c1ZN}y)f4dPF~ldk*Lm|ECJr4MM)$lfUUP|^#}xdv#N`3q!=TTpAqXADX>MqS2a@%GRN{'
            '601YNvVSJ4?`_+FDr&7UH4@)7AB)jJzFjN$N;K5FUMB1X#6(rMpk<REUmf+*R<S$%Dy8?mC1~Jfg~z;Qy6O&<xtlb!m!'
            '|N87iHu!8=D;IPI+u>GxRhVBHt8&F~IB3{^%O*W-'
            '9SL=dHH`Wd!6m>}R%Rb#Woluiw1qTZS;T*f{_Dk7G_3Y|=R>OaVO^mRE{OSO;$4mq-AQVl*-'
            'FQf_H0ocwRMPB|E#(<*><k_!y!{GTBXv$^BvN^&I+$*ZlXQYxmQ+r3B&HUOw$@GS(Kv($p#E~57=f`)c9C-'
            '5a0O}=tL!bJ&UU%*W*|*{%E^=>&T)}Fr8TZDdCT}9rX$-knUFrP;hw#hz63{=s2n(drppg-eBOf=^H2uva#tA~ecE|@S'
            '8dV5eMJ`G9|3`KHYU1vzUbyp26XlwWfrX4X<YLCdtDfU<;5%RK+BzT7uFZuKTN7X){V|5O6%%Ulic~H-'
            'j53E0v3@77MULi=ELr|Wc<d4d1}l0Pr|LAY;_p#fAX`SC-gQTj@Bq4{TMv9+??jeD6i#pY$~sUN$@tIy0soAX<$tq(Ap'
            'JksKd=dE09p2}m|2ws#kz8+a(o>M`xM~?ePEO|D#C{&^|%yUahr?woCjNnqapL<zvmRA;hshKY3pjZFTNC7)zWCaRXEP'
            'nf5irEK1@Bj0s{Nifk-<CoRVn9QcZ8PfA6)wG)WZi)rbTAT@7M0-{9d%^Z$Flp&~Qp%K!iS4gX*KyZ#6G-'
            '!*tL_ZQt_T2wnSf39<3Zo2HsBvH;xhL{`k=xS%?eRN{VN;@-'
            'aeLb0e*Ik*?_wAY1#Gc8}cW1_kxH8#G?3n>}&P>;NF3eRDu1q&^S7v#ZCo?bW7E>?&7E@-*m8mxA$V?w`XD-'
            'R}U_NbhXQrsQGOt#9Fw2TPnE%0F-d(Nr-'
            '{9Z<KfvFe=0l$}d(lX4ZTdGalgyrbK;CmUQ}VKxwKZ3q{(bX+_0{e)y;UAa-'
            'Y<`*F;$6V!N(jrU!j~e?^h>_v*I>QIlx17?kK>AxG+YCue)4<WDdQQna!{_*^VbZU8cKg${9B#tZDsU1>BP6jP={B8Rd'
            'J*$l!9G|J%PwcV}?^H~9Db5AeIL@nJrCWY2sj<;=V^Ys0kAaAGEhx-#FKbYfl*y2Ui|c4mfn+B0`+x-%E@x-'
            'uVZbY*_9oXelx6{afQVhSd>Fc12<F{{5{Vg_w-Voq;#WHzsGVv5n5%;O!7Ov@&F=HNd^=93MM%<ivFOd{{XoG0bLypiw'
            '8{LHZXzx?YgvY7pE@bCQ};O9xZ4KXvHX`I%S+<<iw-6dT?8?%n%(%kEeku~x(V`x2U6-'
            'Cj4bGPUXE@!&#{dF=Fp;41Lkw@i(BIupE+Zg9RP1ZWrFvO3IQKs!3)?JZ9tO|)hM#2;qt=V&d<$E`Qww!6CzVL^R-'
            ')rB;Ze>olIB~+I3yt(Y_-'
            '}5+oc{*D%>MwtR9PW16mLTCGFKYp^B4v0UnM_uyotl2BuH?2iD6>N7%;AbI@gD(oXAbEe!+s^9U-u-'
            '>?`HXz6<x7+ZiR-LRd3r^P%pZCHkhS!E{;`;m=a1gSC>_=Gl%p!|BMOI8Q#;OaTqGt>8n-'
            '6+FE5AiAI6M`ycStonWyl^lX_e^v;3I|YOJwXZd1)q?QFqX5!==@CgkG5Dr^91>zY$nz2wjz;;y7p|85g6+?UtLX%Z$*'
            'iFdP6U#@W?}e(-'
            '3z|VoCd>{*XV+X1~Q`Qg#q8@!NZ@saZkW{aQy2|4Tj55<0mD;)o;);=oKz5cfg7M2K@E#JF#3vKuC@Y3N=>X*{fx^YET'
            '6JStvvG=VN$AE(OX4<1w9?3Mqc+xaCnDWBKZMSSO~10jmBa#DG<^8jjOp@lc=^_Ox{8XX;~j1Jqk>VI5o6oUXh;mnSu7'
            'Bc6}b3+JI(K{$T(;svF?A*_fz2`%dq;H2OpTrGPD>o}@N#;QNK?&n?d-MoyXzDt8|0}H|9W&kKPjgnB&i{#i?8pCG6b-'
            'c1|Js$9?$D|fp8lik2Lq=|6Xh=5TOGnTn7Z{hUp2It96L>WuLl!tHLsFp^zTOmpocw|y{k9$!(oT9iGYdYA8i0C&49>1'
            '-qOvF7;AWLN+-mEMC&W@&Em_kn)AF}i8>x#n`}^VE@(!3h8V(<&6Y-*x5OC@~00nVnuvK1)zfA-'
            '1+9bjGdL68ZsT^?o>kGT`p3!z-VP3}$ytlKP97%nI8>f9}jps_3AJ2;~k87c~@g%(yl0`JX-'
            'J^Q)e%N$3o^|SdBlRgbh`Y}mL!nIpc+9(xx@zWNyo3el%pSuH^=jCU38-}^0qlY^C?@M*LY*{}-'
            'f$=TtM`F<s4>#xO2h<bP&~SqCXb#+#R5mT6m5@u=E-'
            '0e<Uw@aY@@}&d6@F12xaw?Sdw|NaQ>Yb7^yrX%bzcT+_rI6u~PsmNcJeyeeom9HpS!cX?K|UlS2)spX2ijcOgXA606is'
            ')EM){$er4;7@Vb~=*v~=Xt6H>hfnm9iMTd+ts27k^z0^FuuX+SQ-EHax@a`X#;|dzp$F<xaI*Lbj?YlW`c($-'
            's5_lr&iY5MY|uf)4NGCn`yLwUPLTxmP<SjG$Lf>qfTIa9xWDZ>1b$JdSs397OK%>;Me<3=8*~gLGMS_+G7isMyr$9hXE'
            '6OqF4j%ugQ<QNo|@DIOS$_bS=b$3JN;zM&N6`ebQN0cJp@YXYVcF&8oKaiKvaJz`fZPf=}n5r|3eCYt&#&9vxOM4h6S-'
            'x0>I@Q2cJ@>sl<F+Fu8OWOh1c2aHSiBzplsi8rjh0MoF6Oc4(w7j0#m#Q0}e4g42nZI~75Bx8DbasZw%7J08WIe=wxi9'
            ')(7>DB^VbG>Qh=!IzUR;HT?LUYw6YKapIFhzOv|l;<%l1m2*rY#?i_+*^*EtAOG44j|nnJ~yvklIyP?0@=4-'
            'C>XN{#;m$){Ne>s<eev8W~72%wk&YzbU^xVF4n>+8|v~^2`2}(<HjC+TKLfbI(Kct5JnTjKtX;kw|C)c@KP9V9wS`2zs'
            'R?|4AfL*!ICSZ)k>zt=&W!X-'
            'i~|Wu1`hawfi^<oPLK}*`B~Q<CS<yXa)Fc`a{HpK`^g~gXc!ckg&ZOGLp;j#+xuAe4+x^KfX*=e=o<A11YG+9Ke*(A^K'
            'jDO<soYGf1>u2HWgaC~TZh*Cb7`j3yrM|6UnR>4W<)`0@^^joJkEuX>;(j|blBs)F30F`e1@1RAB};Q8_xU=ytXHQh4E'
            'EjbBNe*5XV^KDSdAxnxzCs<E+OfamrdSmz!8QhmG1A^-'
            't@%mi}IAfxPE9YyY$IeR7PV}J_QIc|*o0@QWwjd_O9|L(@LNZR*V=_k%Sk(W4z~m`lyeNRcr49I6LjZ>kheJ0nfy*1cX'
            '|I4HzPNaiuC{wc4DZ~6qLtyWuI&;YPhADGE|xejbspzB74?c)Nh&;pVY_7w+3(CtQgd?Wyyq*@v`3WAFI*~rVycba+VB'
            'LI<&W1`d`d*SoCTm9cMmo$<HlGm2lygVLqyKGk@~N^5ZdjEhi-'
            '+T^Z|b`>sCgovZe9`^Dj}K&H#qKS1I}OB>@Uj4xq!kjpVbN3|!!9hsxLKAo@-cW#qSl(|s=x-'
            'cgCC7dheziyAs5ItrNv^W;lZ%E0d%C#y6n5(B5UqQ}8Bkbl1y$mkS<9H8_v!xb|3*Fnz%J^Cx74xJYt#tDnhbh=!fM)!'
            ')Kvls)EyRC`hb^+|n8-'
            'Sv(^X1P8RY8z75BNJT#Y>*f(6h&#IC80j)nFky|MP+hA7${8SHUw~O}H(%2M<QK5S#t~sC~;IEPPu4eD4bIs+JEP%dG%'
            'Co<JN@UWlCwb6ML5?jVCN7uVaiz|T-68ewV*lMN-'
            '{uw01jI$%XPBgSFXa1zY=WYK8TDn!d<@IT)}+K#^@qsz|XJ;_8kv+)J)XE%gB?6b6AWR^zl@`T)Ze$;5m3tZP+h@90ka'
            '>Crmn0z?L$Ld2cP%KX;1d3pQGnes*UV$wo)ivMG-)EFnC8F-'
            'vRuuc*LoFJ`G1X@t>gvwj`^r1=)g_G<TqaP*CQIc_9Fb3n9X<16P<qcc&{9!Ew7O3^gWB+#a1k)nIEd9#6%6%pCC|)#p'
            'yTZhY<r~&4zEK%ZBZ}lt?6SNQW3`a9vQeJT#*%R^OP*T%}Hn4*TLx}!q9iMA9tR+i$;OcBqMD;yjpK9#~e<>{9kFHcy$'
            'N7?p1|>zFO?!tHr|Gb<i)b4(~LDK*qkEUig~~(N6w&KCP6js+WO()9U2*xq4c`y9GSiN=QoLYKEZr3mRoR4E66jpk}}k'
            '*JPdopWS=W<}M%TKHSUl@hij-FF!c5wHK`GbMf=jr&RaBQk0)*1kr(;^wDrO{FQ4&!%}6~)VmXXe~;5;7p39u9d~^H%?'
            'C`<<sd=d1-'
            'SX5f$g^ew(y?D&Uy#<oYO^wL*u~Po(E!0^dYZB1vei)1kGB%$c)HKylCwUXO}Hw1?w;fciB2}pt=ROzT^Nu^oEFM_Jq-'
            'tgw+voBvSkjBg)SN#GRhN#;u~DSE~qB!s4*yk^w4rod>?NfvlkQwa}fc4=;@=99(pRq)7ge>)E+PK1rzq_Go+|y4xbq('
            '=Q)eJtEP~%?~UWoP<t4f82i2lRm=p@Opb5-a2c5B3sjGpw3|^ThcA(8zzP24?OWyjT;S7xP>QvF9U<d-'
            'Sp?5Z{(VP2wka{iCh!4sOE7M7S8GtH>ayGJ=X(t9-l^=m1jwn?;cno?g#-'
            'xJ|G=ff&6}HkUpV;E{ym1x5ofQx4M$2lBvj(3$%X01^j8W4c4c%!{x{KNDyZ<8azT=dGi#O76+kjkr_TNz6<xo?t;zo1'
            '+0Q>b9zm45ZAZHLr$hUcC|Y~)kAN>_jNrgzm<o&y8D=RtB8n&UZL(EBN=S`7I^Sm36VVKjba`asQF-'
            ')B+1v}NxxA1AiIO*b-om~cBkR9=})AnJ{*#(9FTE65_*p<Lm{tE$YWdye>L@R>P|T{hepsE)dF<S7RQ6T)_~-'
            '0VRDb(gLObY8w_6Uhu#`{s$#GjH77E`UUV;=I<yV$8B^R*#lr9ZwBX4008Du8j{es*$=P&c(z78L!$dRaC|?d7S>B0<@'
            '*U79WCp)FC823lKVJUsgo{RQq06O5z#+H^hI;Msl4uM0C^btD{uQNR_x9mp)5UnoGaA<G@5kfo67g8jiyF3#ER+>5gpR'
            '*lG{G_m?~8s!+vQD+HPeNN9_wLTw-vsgsfLDqML4m}5L!4VSWQw{G~!PPJll2_-M*Sp<rgM!Uo9R|Y<o$7t`p-'
            '1xdis?dT{9yrDRDMh}#-c$Lw2h^uir93Gkvv-'
            '1^WcdowOMkOdk3!l1&v7Jn_eOFyK&pkG#QfHTkt{7?p4OD@CNIi8W0^Mt$H6n@!8LR-yl)a^+^vu_!|GKpj?Ul<6LL3#'
            'ME><avBjv<{T@u;Jn2D&Z#afyH>wtJSaB4!)#nAmy3bI1YN_i5v&lm76C6$E9MpD^S?2rRq8gbkz7uyG;^bcguCMF1hc'
            'J`>cg8KQ)DH5vpf<DE}ZP=2Qy5<cg$wiG5|-2QARWZ0k)$4THCwL{x0-mp<ASx)5RAZcb!VWv<mhTF-'
            '5JWC$fqRSb&vdSoOVjDgCvKHj~*&#r@9GeI4VzG4ty+8Jf1l;xqX)+%^9&&@O6ONSgY!hnh4bgqx+E_d{Q)sSyN+yS1k'
            'o;VGkT5-t7M4Eb+Uq0`c1(cQ6C2SdJ07ymhvG2jBf6?!Ee<3mvCIm#(VZJlW0|!poEYc8CcTF^8~u`Tx$hyK-WSB;>gT'
            '3YpSzG>&kw8(j9^{DHuCD<QS8`x2fJcwh~LKZ_?oMnu~p9#Oanwv+9wR2t{5lpo*UrIVR<Oi%7kmHzOgio>&WY+1bL%;'
            'QGBBbt~={RsBjD`RIC=$?e7u~n_!fQKL{Uo`@<hYXIvlc!diEq56_jWg7bpY(De8i3D(*TinncuQhEhtnT8>A-'
            'yO6!h=o^6e8Hty8VdwZ;MUXxSkw85O50t4L&DQ2Yy2AP!V{1d-'
            '^LlOyU=No3*+71V86GTHVA$pivl;JfyIs*F&(ElJ&PbxZI;lt%^RXCPT=9Ak8l@%9ISgq@xz!pvRI?=+o6+=PMpUlYB_'
            'W`k^_f0_Ck_gJIn@MM_K!B^j~-'
            's@{E}HKKK$WnCSuDUHue4vDY5++6gUxnn6_P3U=og<B4ksnw!HRL}1Pb^(Vphz#EXWBp(AFdC?t{O|+@@2zqX`#SsfGA'
            'o+FhP0Ipjdzw&-s}0Y6b^|T7KH#*gCq19HfiB-Rs(5*dE)(m3acv2V;=2QPQVv0tC>K;kI73B61k7Hlq&r0;X=`#QMoB'
            '6_V!&FQ_#J_zmi%BkmH^HiCoo7<2fI4A(_`x9Fq+#+Ih}i9%a>fzs@I9iw`a(3;0!)CR)AGo7f`|1`i$%GdmvQK1`}@j'
            'V34#SE|S=d58xC!i^sG4`&_}V(+h9Di6-KO1$fUZgxIXk#M7pB49`E|Ae!I-'
            '$v$UMQDh;em)*wRj%xts_u;EEH@@yp1tI%%n2}fsoX>Mn><TBWy2~Q<6UC6w9zumA^T9CD4nFAb29K|S=q=L^{aIH?^p'
            'h%J&wmWcKSimY_8&;k_J&NxH^TkrKA40U0`Jd-ctCrGuvLD7Q>yJSB7=-Kx|+}tSb!foOdwfZ8&_Z^@oU-'
            '%rky^xea{`h^9^u+y$s1&yc7qIR-@!23OAGP!*^d-Fd3>P@%BBi;dc-'
            '$+hT)5NeLjWeFgr^yNkU76|B>)bts~CgXO_jfqEUgfcg9`m}jek+m~0+xZN6u&BI~xsV5W%w=(?eI#BXrIUGs;N4UStk'
            '|zhspr~XQ@pTTMq$CYSUQdI?=07kV{+i61MM8P&XJTy@fRP8Yz{1)TMF(eyC;LMv4o!oK1+Rc(b2?qMj~`4tyTBq}1Q&'
            'Aq(=%czBq#qdZQ{`*@%k1}^Rf{3xTeA{Ee-0-'
            ')&c<?0gU2YSB5I@N0@cJ$B^DxPT#Q_sZm)S?rl+E>7*pUhqDjq13pUS`x#`zolN+{bs2V!-'
            'lu6Q`LJe*5FB{zA!n^SK!g51fRLQ~$j)m7D-'
            'Wf>%#aS2&NGMFh~S#XFO!KzLR(Gqkr053h44#;9s6GwP=9_G6o0}C(Nibzm*^^@TjfjJ{T(m_s)2u7H&jbn(wn9gw0)O'
            '9wHr!?1sp3;WMeA4b<U=r)||lP{iQhW5sL>_zQgMidtl&lGgg0Bz{GARl=)XrUo7w@0$+}zqg){fwlqS)^-'
            '6d#(}978)fm3&5oXzD!!5pYP?=F>Y?`#i;l_OAakfU?136?g>ohFmDg%>5CPoilBNh#Yc-b!=xNG-QlcZQM*{g|b6xHB'
            '{oG<K6?!>Qs60~S&1Sjl+P{W=FeHH~^X!K!lR#r!u-XFBW;}O0%;zKyIxZ%gS2%1!*g`015GniA>4C!NT7{u-'
            'iTg4lQ;d_64?jVg3tzp0^a0fXLh2Tk^K73~G3(~%~Nz3jcR-IiA%u{RvA@9|&*1rS3$(Y0LBEsn3?}rC|PSPMX9&~VjN'
            'cfzxaQl`hDiIlh>My#8th)+MTPCtR{yMYVe@&5->f&Ic3Q+27j_)MzV<)>G-C$%0=U;M=-'
            '0zK;?9L+Z%?25DYxl4eM^ouWK_*Ji&x1nQgCytKHQ*VM$Iuf=sNd3yfn%P?^MnT;3-1Ecm60_5ZY-'
            'n5GzlMSJK)NdnaGz)@cGIZEPkE>Uy~nW(JBp4f9rq`>vw^Zvp0(J#H0OPYZSV~4%=4bu}(b_!$fulzT+|iW8+B7WvoLf'
            'vmN-wpcu_11>uQDB`j%P4n{4#RPI(T^l`J-'
            'ddJzK15}Y4i;uu$;BJt4t5q{_t``OrBJqvYJyIi3LS<eYhu@3G=$q&~sF98(Z(Bal-'
            '}Zs5P2#cilw>eV`^+(5JKzbuiP?Dd?0P)*fCHW0Mv|ghL)iDZmNlal1ChGlS+QnoaY^YHViv=L%CWK7y;~jEOx&jXEdY'
            'dT%HYwz1T;>5Pbzs;AnKeI8Xdh2nr}u(v3w3OnQu&FDrLYl*#vsG4$AU&hrmglRxs}mh0#M6ko_SIB;Ro3<&$Pa<}cRV'
            'A0DFo?Y#`80d?H+R}f;X72)2;btwOez(Dm+I$QmXEZCtzjEOU5CS|~dF>CTi;V;n~+YK;w6N;4Es0I%keAh0Z2hRk+k1'
            'g$ZLLiQA5_Nz+Sub4sU<F#O4P#Up>;S&XCfMMaP9|5X!=3PSvWrI$_xy^$hogQ}f1(lwPq|_ecQiy^(?;&+7w}ui3`1n'
            '_5uAQliUU;-;O71kI5pV=(j{)NjP->MR2!n>*a(q6CW;Tz`=O6Z1J>o7gGcLa$@Q*gjPeV{c~AZE;9D=K{-'
            'h00#yzpovl@>;3bFir9En{Yy+Wf=B!v$`|8PTo>rL$2U;xuyTXAuA9u$o$fSl<Q=)Y)+uhc?W)93W);Phiw{HsWeHd;Y'
            'eJ`JYB?GLH^L@8>jTT*jYACx3TlJIbf8=pDD39%@$`c5<oeUHaXw|KHNP6{qEEXmu*XqG^l7fz&T!P}u}xw~=a@tmFox'
            'Ec2|whJ~hs)km9DPt|PY<9yZ!MVU|D~Ow63#yM)f=k6geA&uYJGlNL9A4A|^<#k`+FTB^vYQ!0PVNlb+pCc4<{0hPOMr'
            'o&D?nm*8&x_LC?_qJO^kGWU@-m)x&$vp0r6zqw%LG6WnQ753*3?6#e-'
            'kob70{2#i&#kM6!MLfM;SH*PE||AcIu!==21)l4B4r+l=cIeDU?f6{=xTMbmwG@U4?2T&+o^g)Pq^dZ!J{-'
            'khP$FWZrO)f&3A;UE6};R;UY4xps9DR}7y;KK4KJOr7z+Uq?AcO{|Jk2N5bV-'
            '2t27_zmdVClgTk}+0;CbgCL@WwAz)hfhu+jCI2`6SA#ri15UH}KkV59NI8$lrAp=ov+E&xRHnJ)eu5B152RoQFN16KV2'
            '>`;3nlZiCT*b9lC-0Ia=akb^6U)$oH0Ga_A(!=nz%+2>>7Sx<PRq6yKNHDEKZ92?w&(0kb-'
            'P*JaAoVwTmhqR2S{e@UqHr)UzIc0FJs1YvPd7+YbG2tB?X0`6|gvHEeu)A)HN}k&IVu?C}Zy3DwD8?t^e$d)eN4_p*!3'
            's-'
            'P;Gb=mb27M&rvy51<<DyLEfNNo)I;F=ITp*yGV%8C7V?R|9aJv_0lQ%vUKjgJcU_zWp8by*XP)N4%NuME^@bNB+C?Gw)'
            '7(t-Is$jdMB_pIR`3tm1u=4YFtcbIa6kJ^p9Js110UL8O|w3RY(I|cqI{r)Oi?|Lbn1CH0P-'
            'ijQ9xoAZuXQy)zDqE=Zm2)F35rNX?1eEDTdLIaFDg+u`k(AzA-eaZiBUMKICj30~5h2=m;7hnJcrA=U^30IRBZIJ!^^s'
            'K2PYo7ZTW{?g%}GkI0715?uVP5yp$zk;CUD#>KI)c5OSA+7t-'
            'OI5nU(`4IKV)B;idLionE9XG9fkDgrn;NMOU^t~U6Xl;NEmwe#i_XJYkG>*}pVaQgw0KKj@(e=R@<m%#Lm`>>;pUnD*s'
            'Ka9%H9CV`W`|+ziy};XoyIsP5XEv?QGl|Np5(~$20UX$;OY7Z^!9%X)7(X*&@>5r_{~9WpazCMFP2MGQ30N8OVF0)Bvy'
            'Vm@QMTnoVpc_FV^OwHQ5LKLylD2;Sn+WJAs?j0?9z!URKNROmOOI19G8`LfuQ^mF9?dJ~$GA=K0XL_aH84<YavGh{LJB'
            'uh9Q-G%QQp3qe^maCt}r(iT2rtUsUzLMOIh@purO_<MlPY8Wt_s%62Ufki_u-+-'
            '&je^p1CwP9|`3vBsB>8NQa1b_2HyFXo2DJ2)UvtqGH_$wZ9%ERa`3mEf%N&7FIrfo`r3~t%qr0ZV+PCG=x0(nVtZKjwk'
            'Sj!EnOQiAU#%6eL#wIVA@}0DH_d(Pdbv)TF$%=F1pyxj5g4lQ#c294CMUkiB(}i$!Rjs1kQq6eqqXBVGLNekW3Ym!%p8'
            'O2MZ@h&t;Gs*|_S8fEl?*Vw)qpRLrh|R$Rfvrd!@dhiAd#@7rske6ZfbrA;>n(naoLM-ypAP>_cze|54@nUkfL41GBP_'
            'a1&^J}hKmRN;Kt{4)ZWufmK|<{&<FW&&RGDR{%B+7sw)^hQ;r|2gm9U)6;_XCkf`F*5cEb7ewnX@<sy0bruiP${+eO!_'
            'p2iHO2Nd<DTtLFJ?Dw9$s@aB1)fq(fx>&d<kixx@{Eis44C+ZKR+_?#Ya;VeC~>;gY{8YBNSgXKgMNy_26woKCE1N5~i'
            '-bfL(j9;OV6l+cW0XEDj3-@wigx%vPh`jj3q;)eaV#`@zq9Zcw_V36~sQhI?z0VDI9^xHeE5A8n4rGxglq>J~-'
            '0N}J$w8k20U_>6r*lC;4|2?Tu1K(09iY}{^QkU=FeU;2?XaXbjVEnZUFUaf(=H!kBoFBMe3^Auas^sr<#HyF2lB2*=VV'
            'cB;X)^GBJ0psKFlA{j9W53YfpFh#n=5}!1To&5YqR8~|8C2sa!=@vabl=}#s^8U0f+pM1d?W_i0}nyeJwp`L-'
            'U0O?ad>!6#}mir;+|NAQaf_Mt>QD?^2r?yT1UaysFK=U*b18SqLAn87@2=P5_9aE;PhcXs1CV{%!XFn?6M1<obD!vvs+'
            '=AdK^eS4V3#dJXE7xnE;9Lp7?015{k93!I%&S9$~Y=P219_Wl{n1ZOX)?>C?c-'
            'W#TOB8^jvBlm7817&5#PY$l~)YF8cp$qT2;I`?4Lj1$DE7viDB*CZ)1i_|^20q2dDLO@6%z8?X2XP}MqmF=j_`yyygyG'
            'iCREQRzrU$gDYcgow|1Ytr`<h^Y&gD><6dY{R`3x^``!jh}>*p)V-'
            '9PEmltybe(kq+$TDkVen(!tRv3&RV~;kCdix_(m*PA=}FX7W`;;m!y#EjR{LubCQ;-Glw-jx&zCtcE+g7lVK@qGQ!XGJ'
            'B#C$7>tVrO#6~VE;TCa`FIuevO~H4E<o_U0+G~*Nzb@<xRMDk1nk=)P`?LkuW=yiYb%_UIcw6hIezIOmmL^VLZ4qL>xG'
            'BLcxFYFyrH^2e3V%5>}Rkf`e)pmJ2R|Z|Vo|-$EH!QJD<YUa!ao|8iiP79dKMp?GVi0ChL{(tR8`Bz&L}5BgftJumzq-'
            '0%hE>fZs!!nEYnWRbMVd7{V8*DRUK9IW5>4uGDD8CVUyCrhIjLjT%_IKmkQd$n%Sy0Q+~c+QIK3?Cu$w#Z?`cMB>p>k7'
            'NhyFx-uEnE=t0`2dkG*>s32uBt}%%P=dFKRUBUEA@(VjVmt&<{OU+^|I?8jp8hz*LW8@Que0q-'
            'Xtbe%S^XwU5B;9j~YgcQL+PtAq+y#)!@3F0!qnoN(;eN=+_ag9R7QqW`UG$ghvY5dBa%`(6frRo$SuDmK_~j}MK1=R@+'
            'G|K|^_0Li>!P=D`^yMD>huE~0E7%+tOYA1=tjb!ZPPJ$hhlEi*EiiSG;rq;?)kp9CRWkrH%cI^<FZ%oGQh+Me5eKRN<X'
            'TeSF1_(UkNzbzNu~_w6F+tIT{97Kz$gG;<e&sXzhE;+_J2uhxzCY-goDYo6^nlGdJ&1`3Ksl*y++FX5CO^7K;7AK>ZrD'
            'ej<@vy~eF<oHEC!4wD@ov$EI2eJi1%xP@cYj+c)j!pUHxi1czHiY>+&Ty+mT0oIY+@rI0Wnt6v5UDx*+;S4%U6TLi)u_'
            'kz-jLvU%~s{NVX$d951MoD8vk!vmtgABeZAJ3&#P3;kn)Nd6l`y6Q~=<Gp(_l;*iWi+u+EO}T@kHhV$!<x?oV!@!@ZUh'
            'pJw80)tkuHp1J4gda_qM=R)$~p0o+~?8ow=fnOOvgbyMwJ8yZzqf+_lR|BHGSt2K|}T)0ojAsplCRZHo2KVw?!a4^H~Z'
            'WiJUa!OdTDp-wIDZ#i8C`Ls;ebgr)>M$K}C^K=?XP<#9SAxGoI8RM}(YzAz|VTm?)2Szvg`SrilQAVHBI0Dd>apKt8=g'
            '5e5Af%)KKxC0*Sn~T@51*RsRhaGQEz{eYEs5ZX}E3+N3Re`tm&6+c?p|=gcJsOqk&KM;j_w*?`E<<0jE^HUO1?!J{0^0'
            '>=6mGqXe7-'
            'Ft!R{w3NX{IVtlC3`7{BSjne)K)^9$0pCR+Y&K`o<kA81U8qwrH}{LmdwKBj#jhRr%u?c)__J2FMhzXxI}do!bC%@*LO'
            'X~L1r<eIFn%aF(V9RAGv1-'
            'F)*KxmmKFY}Ha<D#>1zS|P`WPKPkbu(z(<9wpR*$3yP6X}*PRYu|a6xf}<0`lZJF^Dnu&GtgjLxh{J@<`vdHdtsQ17zI'
            '@XbvRRY>SbFEVD)Q<x>YTUEB^#&iN2B9!3wx%45_MAv(mb2&s)h&?S`z?E8iAld~=^mUX4QFbM4TxncbA5hz^b1dpd;q'
            '4nl`+`Q)neo-'
            '+&!^52@uB%4RZ~93Xt*c?zQwtQUl%lDAN|>NhhisFlQBw0fYU@qF<5_Rmxz?I!3MRvth8MWXr=wJEDiPNU#r?gP;8(mf'
            '@iq-cB@9I!Xa?h7t|X{d4|;e4!QUYrn`bYh^{)tM3rN5a<p}ujteNVM*V3U)yD+v~0eg>mV07^kbj+xMBGUwvOs}Upz8'
            'it($ak_%XAi{QTmu~eX~Zp|3>w%1u+J5tJ5(2wmhFUag|)E#O9h%8@?})s>c`8M5ko=~(IM+CizWSx`ktyIn|3PVHw8W'
            'niwz`He*^yBv>JZ8e3J1=Y{2S&*WuvxCdi3%#E;@RkWnCl=95Lpl$A$I`(#+~XEWM$y(4cg@X5&*U&iM{V^p@c61nz;!'
            '_5^g_;J}ojP_H9EOt+9xM7MzpJK^wJ%8}po=GJp+8KONtvGP_57tfmp~?aE5MEUbg+|HfFgpgdd)r}0Ul4hg7z-'
            'Y?^)U1{V(xCkEX7|bFkYR3%IEAbYhf6JVIc(q+U)qP@D5fpwxE!q2=WOQLG4RDL_bd)zI_C4WvZg%*>Efln2!csF(C0T'
            '6OxyEuxea@Ok9p630d+WcaNKdJ(57?Y7cx8uZJ*l8dCH7sOL~H%CXOBc$g(CiNhP&PAr6h(gfIbC>3+ZCgfg>1+ZM7?8'
            'f6K67W9HDp31s1NW}$!S37HaCd1cHY>!!XZ`2k+%N*O9#N>T5eDBaBMBCjkUwZkr_ztY`8AxV=;}c&xn(hRAe6-'
            '2U<aAyO5n0E2DU^iP&3s<;P9ypY)Y3xYyKg8k|ah>cWy@+2{n?#+X}QW9b%5K@N<nYPF8c!FBd9d^>hh56TJj0S)XW*{'
            'Aw(|&<YvxMzAS<K0Ir91|eIOsQYd+oK##-'
            'lu8R&J)YZXFT)@IeB1+z3?HEg@WGnKOsH?WjXlnMlsTIX976f{s5=U>jfTjX2Q}C#uo0ei27`sJCx{;8LP7o(YE|2UI`'
            '#_i`}+VCz3heSUe37PTMY#^<ieeYmH0c(jXqMC0^`tVmYFUeDE7PKd;J`;LLvf=OKroJA6C%j?S-'
            '~uDVVnWB(7eahj9a3ETt4dqW`H8a$6KoYvn<*y2KHZd|Y5N%LN^;B!k~bJuV+T%&5z5Vccx^Mr6AT=>yhLd~l3KmX>>B'
            'yHy<oN5+vm+4}HE_86YpScADAW*GB1jq#>P05LIfg8mo{T$Hc~ODBEt+E_hUD%*oyg$nQ=T#xXz8Y;L8NWYB<PL1q`C8'
            'N%;g5?h#N?y3dkPm{-'
            '5bPTd$KL@;X#U5rre{Ywl|N8~!u|DNTCfI7A}w*Vb{w{;CgBo+0m5@o7msLMgOpevkdfa9@5fW|EmIhxs;`pvQy+05#}'
            'M6L(k?d^QU$8gb@+0tGPV?+0-L;ju<ULl-Z1+^-u)1TW1AMqYpx2w@xs+Cp3yJ#h-'
            'xX?iPb>(=4||s>;l5OLs=4QL$Tm<2c9u|fOW0s=;QIXRK@NkYj9+Ze4hDH^0w?B9c}v!Id5V>$D|ibR;%FN-'
            '=%cRPPW?VGIgrCd6XP+k3!Rn!+84GFKYK>Epk<O(Tuleh>+Y?<Q)8opLpi_o?kWc>c65#Gj_wd<srCYfj(;q!w6R>slY'
            '~S1&DaHj^#2O1!aBw@}hz<Xs&64CJ*0XyY)R{u(yG+Yeg>BADBd~k}P18P$HElU*cEYJ2Zqtig8~@7XFnqqS`$lTq6<*'
            '(;7pl|2dv8&$;1f_Z3_uzZhC@1Lpp|1vTbB7{UHuXlGv(#_3xEju+FIubc6FvMVYtZO5L&*3{i32+hPhQMIg)7X2J0Ef'
            '@0Pm%$W$?OjN@vWr1ki39eur(%OyE_TunES5k%^_1HKVzPZCg~u1g%>^N^^a41wUn1XPF5xzoEhr3g!#CjxvNxhg&fr@'
            ')#!4i?)cG1*zwZ!H+_M`d2emLgqY)~Pnd2~v3#?h;F!tIPz7+VvkDEJSP3sDDnR-Xo^s3XD6Z%A@FAKL7TA<!xHrV&HR'
            '?haV2go`PkZtWjL~6JUk5%STwJCk%Z-'
            '2`2;HX8%5jA+TTL)6ST1Zm$DHaz|hwT#$BumMPZ0zj7!0mrXJ{*CRw}(MZ(iQ~Nw_}AwIW@?4BW`~m(Tj6)j;ntq@_s<'
            'P8CL`Aml@%y*D+{jmxJqjTao8_4>~6r(hr4(AUUg#kupFt2Ty^aLKCWGD&UoW&ctieVlo=mf-'
            'T~o>6wWUT;JJ)nim)1Eky}%=@X?tCT&nTD+tUk9zdmFC;V#tjD9(ki5wp1S^B-'
            'RwDIE~n!Z#9#tg&Z<%1%+HZqz#D=5T0-r0m>a}N65p7Y%wj-$?|KK$|$N#$=|tpCvqvi6He>U&Mpoa!LQ^p>JQ-'
            'W8nM+Jd>JT4?iA0a#ChF>l^ll>Dm)>G^%|Ps9Z|UY@{^>|VzIW9>cTa*o^Z;kKe^ScRgr(;%(udv=jkM3NmUBZ-'
            'C*Y44@9ciI!}-Ss_-'
            'c4<*bMv0J&kZk(*zn@pntLO9FpZmq{{qOjm>o{H<pTjcfE6L=m#m3G$hP>@LxNpycp*^ec=K2lr&AAevl%7F}pr<5BZU'
            'qte^qBnoRnK5jb#hP895T2RS@6c3PM-'
            '9~M9IathrVUKQkh>1?CEHre*sq&W#a*^ESS+NA=jf%qo=h$Wvgrh>%RkVzc&I5dQ|Ytv=xn27G~9Il%eonJ`nl6iYRNQ'
            'LUXD$?C!h`JZ}9YP~jA998H8FjuGe$Oo9Cp?^x27IdFYXDXM0i1?PPhD4OgG*BC<#=bv0ior9rp$qmM;H6y6iDb2XBBL'
            'T!8N0W)s+l)cUI{5rIkpBGYjO?-wWKc*AOhs>^ddvq{du;^N<@?}b@-*F6WdjW-'
            '66m`{FBps1c0zfZC+g&R!@R#9xU3`-'
            'RafzW{9<iJT>Jx6_jJWyq5*JgRD*;}`Ctlr6&zSzrsJ*gPG=q;4+;qeK~r@VL*2Csm$HWGv~v%!cTmQA6Pb+c%6__lV='
            '3(Vu7imdHjrwN1=6!^^wWn5Sim+PoSK&6A*)vO?uo>-'
            '=XxNOYXkYF^}vx!(Ema@n4PP|gbNkupd1E|v=rdArULF)EFwcuqvXPibe-'
            '2}Z5T5Vi)p5<pr;Z6FOIQ+`olPK<a;KWt`(*B0rqh8ls{SyccXxF0hn`?!S)9?P;Xx$L`thd{g5w62$iASOd+enw-'
            '}Tj@Zvl2nRZG!f>W;$p*TfNq<w+MbPv6lIa9Y#cNbY>oq#r0*CFlTb^6*&q3*r3F{<p+!}|(_a9>LT=F2u9XH70^_Fy_'
            'REjf?rxtWZ*O(TpaJt2&7o6Yph-<=>Us|-'
            '`RUy0E<4wUVFM4W|Rfz!D;KJ&s4M8~FS5B~|c&NWEnLp0&wjDPL(5p@z_I}MB9Zh?jCqo5#344Rx7n7gPC)XbM-'
            '%f@H4Qg%14jCe|?6WZa5i!c5e6QfPGQE*PUgbtnaVQehZ13!sBWF5aO{BC$guB&Ckn^seJ?)#DTk6jc~zShFL?N>3|tq'
            '5YzyrI_hOVB(t5bS3Sq3{6*$q|1?c{db8Ggmy8OJ~EY9x)Qg9S!c{64<cD0CMHlL5;Qq-WQ3bE`d_8fA}};a1kO;dJ4h'
            '%SUGy_?L|?KGD!8lgHoHXLio#aYP)I=wmGKg$o(_M8~i(QlZHQ8e!-O`VJ?mn&a%MyFCXp8bMb7{V{p-'
            'Rhtl9c)P1)LHPd#0+SXvO``AQ-4<m41>q8-'
            'hDmZhainvdyqoM9<^yQr;0ZQWVr)o*v)^qh3|Mu2@{6{N}zX|?t{8yU)1OLI5TuUfpbl2I=&PVs9TR`lxG`)7|5=>kxg'
            'C8faz%m|h44>|!`?$+-vtB;=v&;bXT|{vGhFfrT+6oi9%E-'
            'tEe>iUPL5H=p1qZM9VxUnuR*$z6wrEYLHTa3=L;}h2ts%(nq(cfjQcy%a6yDhfqS^Dc|MA~k`_(@Gzwuva{SWXvb~d3-'
            'z1N(6T8lpg`09@>6{?S9c#(8@Q%ssD0ja=ipk2ZX*Sz{cRc)TGxPKG%mmG%F$ouf{i7fq;F-'
            '}Daim>PONzfT*!Ge{6SgrgCHzbdd!8C#TzkV)APN<^&x-'
            'QUvb|1yq<yo6|@WYA&uOW~*L#KD~>h^qUfSHrr=+99Ke?DGE{jnZA&{qjUOZTvp-bE6-0rq-FNmU4clh4SycNa>_IKX-'
            '~8w9u4g4)`Cc+eh2-kohAd!Mx6!6lI(k#U((QmF;@DqOnxw$)(Ea~s-*>+#3tU~<`mff@5ONRF8-R;coWNSzpom~6v-'
            '`&5WIegF>zh+@afa@eXar|Z9sLw9fAHHI{YA(5K4Bi|Jkvz}$H)b%psgOxW#bn{K*fnQmU;ZP`2uka=vt-'
            'A%VB*B6V3$1}jzGd~C=J60*ZAtDNNnp6^ttA>e>@fYcG)rVl0u(y$!fY93*gHG`o<o!tR;R+Mx>S5%{sIotX0U(fg>DN'
            'nz&Jz<KF8msMukd{EgwWT*&PMNKi*VC=^%!^iK|o4x5e!%a-'
            'p<A1AnwHpfZjjwCUgnFfYABe{KvQ<kApaTs%fCr#C~(E;URFvVw`cM5viDBY4agjO0>)pC<xy^r}#ws~FjKHi4*HD5xL'
            'M#iqwQ(1tGn_Zv#$VAxWu-PulE9i?IBZ5If{y(B7k(h<%NqluR!%C0m3=6E?Y>{LUgdOlnuE&+FC&!gtrA-'
            'd#LJVxzVg)6z2qrI&Z)Zfv^;Cli@eOU+cr7Gfq<$^4q-`;e-'
            '2|{9HEuNY%f%n@2sQ7CFnTw3DZykahH#^1z`q2r;K6sh-iRHc{oQ69rgXa%o;igX%Em@uk;>3VFw0Z-'
            'Yb(cWjCJf!aNurKV37l%Uf!ngGaMHIBE*Ep@-iRrqvMPx<ZVJSA#|>PS7=xcU-'
            'H4i|DcyXP6MlC&!r}*aLF<qgRS#mvDr-qdjID<1g>UGzj#$0LCud@FQVsoH!~*w@gOD_qgfbI6U~oAdwUT7%mXy;lH69'
            '8^g3r+Q?tT2L^M>f|h|>PGToa|E&$5b?x``Si85x%!*O)K1!S?|la4aYk+9#sX|NBDxV^f3X_x(}AQIb{P>PzA$p244v'
            'eYDzM0z&f|;as&MvKa`$*p~w^rv3mgZ(IW+n(K5VhJA3#>^Pb)V22f3@^OKr6H#Q<;Ho2v#6p>y!PB%D+hU)SKLQSfF{'
            '_O8R3or>WEF0SFoa55A2RxpgTXG)M1?bA$#>Tk3;}*!8X?7}tJ~c|b57)g#)LP1d!`NzPkKn4i6stK7qD)B4k0UJuG3m'
            'Y7OuB<#c-!l;N0l}Uu3f()8!DB$c5nW9Ot~184f|-'
            'TmXZia5bPD7W4sR2&4l4wQ*|Im;mqcS|I23D6~WyvmU(<N0V3%vXkLR0;l@w6b~i9vGvVFVTl@K_Ksrm<hr?@u_2pJti'
            '<HzL|7FfLFCtZ5t|8dynP}A9G0+vZ^2a<+&%&Ynx}x8W@FKcaMW$`#Wj~+z<uS{EF0!P?V@uZvCHN!HBc?Wu@6DGL#-'
            '3a61&Jm3qRJs)golJ1j;SLQN&G~1_+76vCjuEK%oeu|8jzcK|EAtoM5DO<lwe%DY$+5ATji1!?3R`Tr1}Rxf635tRWYM'
            'Zu!!}r@t7F>%~cee+8^Bc}}GUyx@$}5Mz8}3&YU*0bF$t!P-(KoMt^@N%_Qs&CyoOG-'
            '{(tTlAnMq?s&v_ZKY0GGV26B*u>hfJyfjTC=1WtzQ9tD%pekLMPzijShHlIGM2LUnX{aZkYG>8kS0VL4uJMPJJ>ZXH5Q'
            '4OZQ~r!F~hIo#G*D4rt=41@Wl1JRTTJl+box4l-'
            '9cV&$`VczJUr!_)LS{&P3R=VIEB)+AQ%FxE*zxmJ;54maS?DPa)KxQE&=oRKl@#d<1KjKWz2|0=A7R$onAajXs{UpGSU'
            'Cvn)?+K;{vOgy`9<3%lTa*1U^{JyS$1gB!`*ep@sMD<Wt=QciE28;qZ7V?`L;-XPsT(s>Xx@Zl-'
            't#g8SF<B5#+*^sRQI<G<ZZ8DbJ|g=!WIzNx2a0pN|HKzjbUw5n3i(-'
            '>Wj!B03=7~UPj7HE4#N(UGE{u+31!<<K&CJkS1aED7egm}>TnI;WNPB1*CJfBI}aO=wBiq!H^gi|A?s{MNtDbuc^@|)>'
            'M<1thB<XD&!)3Z&YECMM+IIj-'
            '2%lT!7PQVhtTWme6U~Ugx8;Qg72p{bSTRL1qYg;KW{xGwZ!0+jxn@9u7%CVet_HRP{wL%L@!-'
            'P!L*e|aKuQNa7})JxMc#m><UbLct%WDO?7~p?Yu<8!(G8t`aY>y<A*y%wjgJn7${7)us-xJs8@WTh#wva)LZKNf!2Z~X'
            'jDH>l75wvZRrYRs;iZLF%{G;*<=m-'
            'j8~(?3LPl_n}J;}X&9ee0kV@q(B|Gu_I|$rGNo#unqUk{d`9%6&0m}Y4Q`?AaOATNJceg9^xt8q4_>T$>tO)RkpGPLx{'
            '7qN);vYcesc^n>tVdTI6yu=7O%JSHX{zqd|boMjZQc0ao2o7w6olZR?E9+^>-'
            'C;vMGQiSA3}Tb~{wzbtda)+hOC0^%!aH$0{8P!O6)SkfUMvNbnEx+x*140q4=y2=U8drTR>{MXcO)lc4WDM)_90Wjtx*'
            't#{0qttX-)^?f$7knA{(DZJv~E78jmyve5P9V4S#6W$E36k4h4X-'
            'O1X+e8)$B3}8r4fb+yVgyqK)jA7FjBgVTTJ6E&h}|$Lc?m{su3_iGLueh}fw5ZZux1&He9KC&n^dpHTHQFb;qV}@M2>='
            '~Wgw=$`$Dz!Gx6LpKTKRQ!Z@beggZb3H>_=gz+25=HlYHe*$U8Le*!&E&fT?_f%tY_Kk%pC#==^EJb%yx{^n=EgJuH^d'
            '$5fJ%{0UNU7kcKZWv;@#qj8)Jl-'
            '%d$Hz{k`1*>kj$GpyHj5;~pFBAdd4h{}XI&=gXDPMa=}SJ{9m4}@S?HTsNfy^slHdQvAn?Rxyt@84F{!jcb-IEva$zxS'
            'WG}$%y98WXE3r%^j-'
            '{PxtgY6S2fnG{q|KuV_;n&^+Xq#!Ue|>o!UwR`qKq6*m1Z>yu7YE6GRW%^i9bf~!LRL;j9=Rkd+$_Yzu05ORBb6nX;i^'
            'G{bg`qb0oM_xWHb`>-c1j(=qmQz`UF-'
            'r1<AWSZg#4{J#0{u$mtpy%az_*gzbjv%uD>45EM6kR=$3p^6MRRhr1y6DGo7{BA~rQYFxwtss8yHG$*!={fy*i0<}zgd'
            'dnYX!Ni*is)Cd%=f;gjCYm5m8OY%HrFr)YLjq-Z5`F#*9NC}?D5!>U_5-'
            '$mv~#&B3G^laQ)i?Te$eC>`WkI9lJ3u+&IQs@a-O!tIrY|<3nnv3gDqs9L#jIVq--veO!2zj)oZ1qs}8(o2Q0f_O2ogc'
            'hup0T^&~3dO^HiMB;(wmt^5fK|J5E9NyeNOTRqI1}pPUtQr1Eq>e;^-'
            's)DYU*koKP@KV8zX=UFHIZc<gw2~w(Vh1pIh@*!Vx<?cb7F)@LNdnh7+@{z+(`KN*AU}Vw#W$1fSjHd^6-'
            '8UT)AKajKT@b6%T+=nK^yx=1UYgeQ|-F1a$OxGtT-'
            '*qMeaE)d&p3O!tSZer6Cjjz{3<h7Z7A+d<x0=R#5PjLwW5KfJnd59`lTc<o#QK?@OjE%b<kw-Xsivw_TI4@oz78C|`lf'
            'H7OUfN}r0H*85MAs_upSxuM2af#^!ElFmFr_t`H<`_-'
            'nG}oY@c?x~}^CV&Wdj#;|CC2G_AE8sY1lR_|F!{<_me*`QWl1OE{+ah=tU3t&)Eudxbuxx@#^C#xo;V!56#j9^uxOw(>'
            '@p}L+hz9QhMEG1v$;=t&x)bUm!pJ}7{T+9NC-'
            ')FM1eI|A;LKiY{WK_7wvPs%fW>1s4ejRr!kzqKSdVQCZmN$EI5Sepun<ypexY??~A5rT#r5~-'
            'oJ`hs_u}TgIbUi6OGTq63}t~Hn{Vk3Ve<h1Ba?9W)wA{)xkZWe=i&s_Hn?2@A07C^Ne~j`tVGY1w1%@1{1Xjn*KUYidO'
            '{dFuvr#3-'
            '^ntIk*Q76sV(=W(Qt>%fwNUrDWYLKbDL(f%bLhuzJ;QsH%yBpN&z_uc!na+i&3;xq3WuoDFko)1hi(E15O&!LUh$J@J8'
            'X<|M@>XLHf4R{>hrXQ9v4B%0ZNlOZ@F2_{_}tnR|K@XmY;<=1(^`~{(`dESkv)4Kv3XUa+1C4VrS7-'
            'WUN?xRu<tdU>n8M3XeVFkVo0$$NUnzPLSu3$AL@5#X(>2mbg(@j_(5rTgji;4D`{kXoZ7tH!hk(F<a^Pd-'
            'i!s8~ea?8g(VOC`0gebIkxuN{tXVgNo9qOcVV4t%N7>H+M?|Mg+|F{Kt{t1EE7hPKM_a~N&CPD70>$E>BhqWYi1-'
            'KjUgfTk}vbuYCPIq*WyVI#)-'
            'LL?YJj=0Q?zVqj8ah{lU9gAk3q1bWj7}HTkXz#d7%$7BN*7x&>s$t&3Y>u>K9A_)J`?EcGzYuqF;MWX3hFGx$blzIpxV'
            'C{JHD&Z>{v;vF)tPVa@P{QBni}QXoDjWThUo47K-%S$e_V_n8=91eM(KxcB7o}A&&y?h}L<P;|TVgE)Z)$;67-'
            '<3v(aBw=|VCdEXSv9Tll$LIdg88i1euD)EwXvCfNE$)vJ&E5f@+u=IE*URur$Lr%-'
            '!or?y3+U14kl<vYnUOTd?7J|R>F%WDOq#d6gL(NVl%IAC+&Z!@TalbGqtK$PFp3{))mWKBPi@>p>6Lp%*$iB-'
            '}w0y(@^5QyCS}PE_eErdmR^fGp1@(u770FQT0s!4>#PYcb_O6~J{(Qx#{p%%p5)qI0-'
            '|~TSRz%&*PB|=CoediY45|F}W^(WJIl^;?9oBGEplFL7W-'
            'z4j_Nfdsl!_+%J=3r!?i89WQ=}n=y~O@7Clr|Q0NV`?;3<Dow@B?J910hl>-'
            '!wWiBW%ao15dozFP2Sk0lN7zOuM>3F`C;3g9#K6tc%8n9-6Mg@<deL-'
            'v3wav60oIQLk<;+Z<|Z`s0{e<cC99@r9P4h32go(h({yU4G>U>G?vNgl7`C#|f*q{s0qQNNi)e|9I5P4k@Kr1C?0G{=p'
            'W#^n>X=w`J4l7myNeh}R#1H}W=+Giez(^oyIAn2+Nt4kll{%;DHxFQUX6$_HQNhi3+jKRQvGT1z}n4D|+&hl~b2QwEAc'
            ')YJ1UJP}Bx_SzF>TzPXMP2Qt-8b-UVhfzPtPZt0DKP1q0bc9N;C|gD{Gwb7Zar4GZ~k7C?0iYuy^3-'
            '2<PXvv9XVIeAr}9?IPB4jr@wd91Ds7j_tLA_C)bZZpXI<^2@m+T-T}Ve+y>1-QBWv-gW57|V1qFiKAu>D&(0=+qlp~DE'
            'a(qfAJ+%Z^}p%&c^hEw&Rh6|{T$iSl#a~K9L9lhK^R!v0b{;-XrOI~b)whtVU;GDB^-'
            'vQ8c#7we;XY);G~<^kKzu~GjQQvDSG1yqJAe9<vWEzwmlifrmvwu{bl^kGYqZU=99wL4{5`<1iD`!5lYMbVYTcYAk7-'
            '&7qb~w>gOPvjx72pM`29SVeR%8=}_?FEC&1wCkBBG!u808E|j-'
            'J<0(%VaBYD4xmn!hahnRO)*;6Vq&9vrRQc^|I`wEV9my*L5jexxA?t!&x@VAcRUN%9!@@K(7R<9<1>%cBFzQr3%p{GGW'
            'eW!|kuwa<?0x9(T^BH^eM0-'
            '=v?=5!b>o38*Kq7y4r<U5I>YCu^CjPcb?nJkvQLzQUQLvM1hzYLIR7B=5wXNmyZm7FwKg*Ss({L^zW}@Qk#P4kLeBO=*'
            'i)rNQv*)Z%nxforRfkRow?1}D<XsUSJgtqr4ELq|0=M%<A7^9N}+r630S3IPczOxBYm@3@bgLyYD!e&!bP3Hu`3on7sr'
            'D6ymn|?I7*9ORwC=!M|dOVL^6M5z-'
            '~`RaE=SdzslM0f$Sre*KcD*bPftRNYkeALa07>2QNN)prdg70yY=~;gd;4d||o{CNDdIQLGjm)!Ty+MoZD-_-'
            'C55+z)U4RK|ueKWgoiLM<NLA@RKTKxw5eqzF`j$sRt?{j`A;f6j&r>cezd(_PdLe2UkrOYp{`x3t^K0jkvxk{H*kbPan'
            'Dz|-'
            'B>PHf;}Ks7NLF`>{=uJiM}I^=pZ(S`>?<noX!W}Hsbk>H5JsHA$9!2BCEC*NMe_G3vTZ~PnU;lpbfyoZGj^YqE4${}p)'
            'izmZUu_((|ilbBsl>~3%vW-J9GuA-ra;ibFPl9Y&*M#x4X{h^861EQ>2XNj)OYidGW+OlFD`>;czYG-TsRv97$KakuGC'
            'jEnhn#BR?7kevtxwm{`mHBaZd{9Jmt~_vq9fjM^MxnjZJ={43b!SzgLY&;uAiBr<{UoANkgchvH|9P@nWp~b{<Z(M$ki'
            'ZJ>7fm29zF-!L;)Uc-<|E-'
            'm)>FdLNz<@k~odNtc@IMHAxt?lUniPlw~1l|V=MHL(gQWvnXNh<8hll0#;i!1hK1|DD~3n)9|oOz&Ts=X#IY%NXMZ-zs'
            '?gynyUJoe1|DE<(viVRBw~C4F%4GAw^}9_mw)&`2~4`<1;>cBB&%UWcQP!#=PSy+J?ZG4Y$858P$>gUE&gxb$a`CWI|Q'
            'O$VTFR?o-'
            'c^PTuU_86EoRN*SSZP@?m2$?=r0aNb`;c40yv>uIt3xi5{H!>gE&Cg)i7bVbB3&m*WA`t#*1&1`U@ppI|=^gV19aTQ`a'
            'o-Noj192RHxt5?Qt`B`4f5XoM!xu0qjEwMl>Epg4TXC#rRxm5ox4$mrygUHwghUgcf~t1UBpu+2$p-'
            '!($%i=Sl!n|e>fkW)2-UjQ=bX0cgo=Fl4RPDod8AECy-}z7X+Bx0qLbq7-'
            'hWy3ZFj)9j+o&wwK03y7_e7g1hLtK@c`S&<4pZeyGTK60GB*@J(DaU8ZvrIM|M$@3~gE`mLIDs(Z8aVIGw_xQod1y{84'
            'cim}mI9S-;>gNK<PUhlpQk=IS|)0TyBX?j1b)-'
            'Qq&{fMKf+hK=8%^bfBhOX`fSXLB6%5uLF!+Raj%$O!CkLKYy`*pN%U4?dei#sk_>I+Q;g`iV<4fVr#u-'
            'KmoN6#^Eh4n?)?cfEc_)8#n;THz~8TmO*v=Bu&c;JX(Gd$YFj#Upl@i%E9TYPte*X46y_FEtK=Hx&-'
            '_a0QfGJ)Tk|I)^Pw@`I`Cxjf7!)mcdH0Awm)YAJ_EBmGxziui+=}jrPN9!zb{L}<Nu~9HwaS?eV9KmVIl?*3$VZ)}Y0I'
            'e3#v!nw)^vKeKB8pgc@fNWEi(q^_dY<JZ9f5DcGBEs*KZ$y_1wY@)#{*^Wq2*pq-I-ta2@lP~;e%-'
            '?Z?2E4N5=4t$OUMJ)c_AhXan%nvn`Rtv)MUxi+>R2kJh2$I~G=WanpAdYmwba5YqFWgKhW`{OD0le{6a~;)LU1+fXFv|'
            '8c>l9*V7D;TTcI4Oe(lQP3<DhkjaPP_Q>xO9w;8I}?mGy#dlDC(t7=3h%^6vo=*%;(}uXXrg!kE;LIazqu_rcqyI1YZX'
            'DvlLJux#{*2h6@V$NBP4%$EIvum#?%yNG;}^q_+dZTU;d1Lhr8g$4tbR9SPa_S;dpiJ0A9-'
            'pV|@A(f@9eOMAC2twoazOWoJt=b@T$<D!2e*PgVf$H*+*!V}?a{W8qI$gZ8p~XFSFggd`&os+D%*o@+~W_1-'
            'VV+gSqeVjv3Yc2q!v#3J~6+LL9MK25T>me9R*7Rddjm1umI!*c%+NIAn+PrZ9kOw|%Ux(DIfMpc$1?^CR}d>SKSAL7?S'
            'MQGe>hjrnfLC}$4PFfiZjt`JShl8Qddw{-H$pqWxOb7^iL^GJ_tOUE0c)Y6=v~4qi^H?D&uQY|7<HgX-qX+h}JaGS{Bu'
            'w#1BmI5?y$%9)Hv~e(ZWip}2t<vIYEbg+IGPonMk&`1D5s=}+b;Q_@mL*B8K1<0xBqI#JX#^|mKOM>c#|yG;~;({p8VJ'
            'lg$dQWp=mr8_8vP4bufl6FEjA@QA;Rj>SH{VF9Ye+8YtMK2@{_hF{C6JUy3SIMfn|2@u&uJGLOQJl_6mI`xc`${ux;=>'
            'kkS&VaWD&3NmbCARsRbO~lSJUXeg_l8gkG*x%%6t_2RO-6RLMkAvyn4Y*pUl0xz;{2TCssx{ukYy%6-'
            '9QuVeZMUgSZY34E8b?0g359zqA><no!GF05u=P#^MqY8j%A;N=*LR-'
            '?*wuoe(*VdPUZL)n)5#|@6;u|nCmUmakvNZ~tY(iC@IfDP6V+%xn>jw+>PAn?_tf2|3HWoR0W2w$LCz1C=(g@SnBBF3b'
            '_XZY^pHEaZ)F~+Jg-'
            '4F4}aSKb`RPdo}?BBoZ)fka%j*=MuFmK+SjHFTOEpFBq0@zN4r4s=nIy$Nf%ie6NwYQ<ACo^AdCdKL8Foa_RVp{@<M)?'
            '{K;RRH<W<cCDoXr5JMS`daMNtt%&QfrLew43f>Ograuc;qu6P0F!<PqDiItsb}XG$nY4&8o5UgzD2z3lwU8f~3-'
            ')`4Nkc4?oLKsUOsvVm1gV9vqT7ch$G#L){$_)-#Wf-'
            '~912Mt_R#A;f<FW1VeGyolwp4s<M~$LT15fW`T85@71W{FrV1ny(ZrFJLDh8vaMDQyI!AKwBx8b>bBF+Z*oIk~G(k=22'
            'E4p^0=D-'
            'g5nd~KIC*drCXz!CKDM5`FKD0#@jBcTE5h;MXt@7Z2#?vM(lTZhuKuA#?jB2pnA}mSk(`5)hqA!S{RU`Brs2g@U6_2RS'
            '@&~G7%gbEpr^j9)UGwM0iJ0i7*s0K$+(-'
            '1>36uu?<pQKZS0SnuPG56eT><OJkUGQf<D4sxJo#bc#tS4(3Hfbw+2y6DxJu-#A0r{GOjyu93-'
            '8xK<K~=Sn2tW{JQ3gqkFwsC0xm{IPDExsSt+?l3Q5p-'
            'u=jQze%j#<l|ByJKQ%S4oRPWlSNAcP~13%Mz9}1`!|kc?P>!?#k4aX@S2;N5<#3Ee1eHD`>^<OKGt+H>TU+yqcRrfVJQ'
            '3(OCZ9Dx-'
            '@fP@%uegF4}}9`qbgF>z{F+UnY^xWkb9A0I>9_#!#AFC&1wak3}y5UfKtdAIi};ouW}mCLCV(kqEE44q7ei@Zaw+EXp^'
            '6b^V&?<$oPt8KsjGLwqo?QU~%Xrr@8R8LTm7hfTA^IBooqG&;&)mC}tmV-Y`cCp(lRW&XjMV-JD%b}H@uaT@J?V&Lb88'
            '7ei!3vgYEc)il29~<28<3nFeKW_+nFWgas%LAt#@{mKW=WyPn0{BmQf!3}FP`y+~%OYOmwNph{a^^OqE-fZ&PR{Y?G*7'
            '%>szj=td~y7GBshxC60Xb~%;(hr<-'
            'cLLUuZYvq)DL5rVDsw)d(X`w1kK%9zu}}X?XKf5S?4^;7lBE{S03t;cH4~h<}(y-'
            'Ryj7(cq3AmHVObrT~1f7MR;FZ|GWmVL0X;53jEZp-|^1<Q(U&7k(UsZkxTScvm=F3Cg98z8mSAdj$UZwc?Kn-'
            'g?$_Jn9a7N39d5!EKEw9-'
            'Gro*VBe6k>f=>j}aQ)<%*M0y~KG#C30B5XPMiS<5OM+dA&Y^L_O)Hk@rr*EopCXw0*>Sbo~lcFmlN2Yr?>HC=#tK>~O&'
            '?LC9J-'
            'OGIS0VnvE1dZng=mJwUMxK=o@@6f1EXiTD5wF3pYvuH418m?cy6^<Vgfz|CHV6A0}uLdJPID+8D3x4RGB|v|&J%-'
            'g07f_S83Vwc^U%w>eJREvX;9SL0tXb_rGc@a2R*!g5V7VlOclyB-xoA3j)f@`;eT4L~dR#gif^T{waK7?>wDQqHjp#cd'
            '%g<Fmt)GUngOQ}HX+GWO`3v4Y`b<4#;&Ey7SDoL>Uf?O`EP82RkX$UkNA<a@VdIM^ZLN*ra7e-x9%swIdBvmf+-'
            '#5t{%XNp4ix=2=40pJ6c&~J!Tq`E_#{36%A6ahT}>Lw(HJUMJc8<EH6-iJCysRq*tRyGWqRrw-'
            '1GH=(b>hUFOJLc$Q?HF#ZCo7o^&%_R4pb`#Tn2)(hcv_4`Nu}X|k&~0OUFa;r^u!FrJr)z3bblhciDIcL%{sJr=x|^2f'
            '}5zHnYw2%j=eV!EI?u72?Z+#iI3h&3nP@iC<*+q{tHz#T>g90$#R%C!!#l$6a_z}U(Y(0$z>)|7_<DVG6v=RB%7p$<1y'
            '<}{D%01jz1lYQTHA=>hVcFW2}__v@DWOI%~MsYL*XL@6$WjY=^bQ2djPZPhphhWLSB9!moL>}AJI$POte0F0Jlhv|d>9'
            'ymaf5aW)7IFdmUr!j<ki(aAdslkC1j<_6CyZrl(8V?fijO^@?qc0EdO90}J}<`aY_2F&+l-'
            '6iyfIN=JGefhQ1CI7s;BLTzuq^{>B|%HC@d1E7u<&#;U#FXAQZST2sKr*p=s9*IMn9^O9lqWw#!?<$bO7Aev(Ep4i1va'
            'yBmy!3P7v24#IkdVVXMv2l_gZyW|PnT)_@yc~9up<maSikca9lTE^g(UW>z9IY4QDJ8XVkM4or@!73jU43?+{zv~&G$t'
            'MOEv(sVrjy^5?&O)QFry;X)6?VG+qAJsOF=#jf*YMRsC2u4u>B_-'
            'H%W7y`od>11fEwOsAfQ?f*W66P*QKEt?sbX237cjmz7d5dr<Z`<s)N{6nhC*mv*c6uJ@P&)6oP;8;}>&1EH&PVHpg#+h'
            '8BYFxk^U)okO_W>n-Iy<WF}e7vUM@H0)420)1VP==*7c5o^}PYJ15=oq~2T#OkX^sRlPlZuE!41-'
            'ZaodKX)V{9xN=KR6$E4_gv_sC*TIzSn$ITX+_nr}jYTlrF3l2!O2IBz#<GL~?Y5>z#b+aci$P9<KBw&0d=^ByT-'
            '6(0E1a3a0TuzdwALs3zVD?PyqBto@)o9GsG?u=z~~+!c?&KU;D@=&3D8fAj<VEJM^E3E;#H4lqoThO{0l?0grC9v@a<('
            'B}F0@@Ff$C8~mjM-p^AjRw6-F?i*sH9j#Cf^6YlB5fG}ce4|r{K{8cL0zFDq6-'
            ')^1}G%#fX~xkvNYy2*zMoCcp|bC;}reyh;<6NZ4>~?qkCa;(KfuJn2Su29L#dR2~h{SF)y?iHj8?J?($er&Rh=54w>UW'
            '=O=XkkxuHw6oj+3bKLZ+Hh3Sc#Z|MG_@|_lQ9N!4R=X2f&j(}SqaGXC*Z2rD%*)_r-'
            '+rv@H$?NeGU^n~ff<JqV7>Nz`qWnyG?z9YZ`WQDdqEV!lKSg7O&xIS+hKY)KZUh16|q-'
            'f2Rc7!0@XM}KwnYnG?4}Niyx4^i@D%g1`yt~dr<k82f|J^;2t$UNU7cj=c0cxBu*qE@8VfTz;-'
            '>z%&sA`2mSEUu~6)bUPM@&87QSyjc?vLVR%jjO|SOCAig+AO-dz!$6Ju@79jRD+;B{fiEmH$<9GQ5<kuJvNckQ`novbM'
            'zTJagcVEDvxK`jZ3c$N1hvABxmQGdS4ceY%2LVsJ;Y0fkOgkP7W*7>}0*e_%*EZ{zOlsi@ksowr*J(_(RY#wnoGAa0i^'
            '%t<;yZa2l-z!R{_+|^zF0k2%v?w}IE#VU*%4wK_gve}VlBMOr}*`eFA8#eAkpaokpJcp-'
            'D%Z@Q$yVJ>AUSPaPcHsb*CZQ`7Go}&qJA2cD4OBqR7|w1>aV6Ag|b69Q`_sX>lWXBJ&1r&RIkIy4#3@X7t?LWTWCK_WH'
            't^OxV|8fB_b4y8Fz6VBNi9RA`(e6uZc3pFbqgsgf4O%VBt35Dt3=!-QKmeZa7U7n@GP?jwRYG;kWcj_sr#dQ-T6XA-'
            '^f6B%Jq5{%A)S`-&r2d^S0h)UcERI1!Y#5si_l>0W09qY!rXFu?0OE-'
            'O3;Y*L`+{f2hw(vEauU?<=kJ#?4!1r;MaN&6l>|D-Q*R<6eReGK9Ig17A>gkN)^=We)qX2dX$5S~GS#<ro0yH-'
            '`;L|Tn_(@y~cDDCpUZ*#%vrM9cw|&t;*a}WLwW9lp8rYb|f)k0yaAN)i;y8K%22a(HfbK+W2yI23o)$Q<)}IKZ8sM-'
            'm4?fMGn~z;SpuRH#c~!^ZP+ALK%#VU7zOCTG_7NrR%AxBhJF&P_1*f)f&?8sH>V-oL;Sk3?>fGo}^=_9ls%4r`<nd;-'
            'UHlAxPA&nJQ11Gff*E38Jg4i=USz$o3I<ij39=--5#v;Sz=~-'
            'PT0=pM{TG%pPTCm2vEv7cNlp&D8|1|yzF;_c{Wf$QV$x;vB2g`!4-'
            'QJ*KnoH`XDzovX6YMR!Ry9&_udp#IZ7C>bd;bcMg&tHgc9DU96Zba6yJ#!gTg~a>{lG6)di;b*8L?l-'
            'r#{>Kb`@Y=)na^%_K)sk?M9D(ko-'
            '!ga~f~NtJ2fH2ACi==2c`U{vEh(;%W?oCUY<50G)y63kraO)`BS5UcPMO#YDyt_GH<UZez}nrC2Vjy92E`Qhw~8JsOzM'
            '}rnuvaaUL$EVwO(cN6_P|+FADE!wA-HrBhJGqF}wDt&Ak6nb>nOm^sTN8#)TA}udw-'
            '}?p4Lf$er<t@CWyc#~oR3RaS!p-'
            'RMR_YuOuoT~<0Y{3%2jaw4k$RHfL+fQ;ckKZbpN?n@Lik+Z+7V8GAux5Ts3B~?FGFf{Gjno2x)`|9{H>YVUppn(D6LTu'
            't|fmf(TyislpFIMqu(TkceKkf_?f%sDCDsJo_*)*Z=%@d`O*DJF$s=xAj9U!Oe8ft9AIA(*q^PZBgn)F7}EEG1zW)!Na'
            'N#^lamEhT)@G_<dUdU|kClX1fMYJ{q8k$|KZxTSoVLl_ED?0r$lA;Ib3<;MBLLxapq~K<93Zd%BVYvdKW#uVmb&a|&)r'
            'sny|P8<^f$248fF>EX8)=-'
            'Pb<uQnl_pSls?#A@KvPo!?oSa8F&9#%zd$Bot+&|Jp>HnH_6t1G~&IX>5C#?g>ov>tWBgz$&pMc9ydg1QW?N733MJl&~'
            'JZw_Wb?BN@Db$>H@=Uc!R6?+gpdmDPMwSxBtijmxl;m4Is$YXCuZrw~+J7`T$v~iO1jtt0eS`UH~G4S0i2)%YzlIMn@t'
            'c#PjI4Ivo8$ND>eRlp(^KOJi>x*G@qyp=DTj*WGW%#tw3)v&&pggw^cCQS^%-'
            '~Y6d=UwWL=HB!tU;DsI%bR|W6tiw@Wx{kbgGzx$Ii9r*>ei@MLO_VU_BYENx@Ca<xoCy6h_{ck*6uGIAs?Dk@t^)b7M4'
            '-D^`U7(Hi{cBn*3Aalmd*eGs=9q!WS?NFT7^<d#ynX15jW63@Zu&+%l*f@NT1c^;N?KEf;RrbJSj8x=L|$+>BO3a$09Y'
            '|(r8`bq|yl7jHrrlX`r>ny(3w}aWBT=3LQq{*f$@#t0;=(IM1VF@-'
            '`)RhF5Gbu!)%pZA<uZ7~_Xru#{@NZ<CXik~I9=ju~a33c)%}mB83+kwv0Aaa)jm4X4S!mNb1lQ*k(y&E+j6c7h;cCl7q'
            'RL}~^~~FF!F&rNtNAD_%CW<9-'
            '9z|hYbjb3DbW?|_2_*?9I}(8NQRg_2$OU?n;Hc?Y2K)lT8;l|?!d>x7WjAbX`FxGn9S6>L*tx2bu~(6jh7741dR}U5VZ'
            'n!|16>n5C76*T*jcHDGS`*UU27ZJcw^ugQ8D*p-'
            'j7j)@ob9GcI+M(iNkgUq^|ORUlroltGPUp}3&4lYHbc!}mH#Aj2x5OQYpT=us!qJjp?(JYUk(T@tjW{V0rZ-'
            'G&1)o8j=0o#1x170g4Du}k+E?pD_a5}*k+CCQ9`doyvpQUH8B`W6&dhrqLxX;|XthF_O`g)|#pT+P{yBKkTkzgb^;a^+'
            'GYss04k&)r-mza_3wSPOQy9dVPEB#b}e2Gvat7_rq0I_6UpG-yV<^&>Q+!xAqgID^wgcUU5j2Bk{<(8O7Y&mzld>3A|G'
            'NJis#2}NkrTL|yf)8UNa&N&Wj4V;5~=+_X6>}$K>k;<<+lM4XvN*3up+t7{Vi2$r>s>G-'
            'N*E=yA4DS0(Nbjk6j8}iMkuFbxoU0k&6EA^PLDgi#@Gn}z%}ZxEroeMo5^i37l=`}KgHmY++3{ioa!gquGf|GQ`LA#~*'
            '`E=#wG3{t&4T`y2k<F<zV2|hxb8r+3%NaLk0Jtw^v8S-OjZ6uUfo_)|8V_&>^&9-tL^)M`?-'
            '8Ax2ioI@l++QBkGLQ_j!yXR+pja3_n(EwZd=Z_sM08w{)MW6@1<KmL~20MYlBGhLdXx;8>#@HLG)@izQ^xa#0awtbIu3'
            'B{-2Qn@N{E=AgxaMc`mm$S@ITqRY;-'
            '5ZKU*Efwu}_r4)&5+ziBa}z?Yt|g^b+!&B}65G5qA@6M%bss6i8I1^Bw&)5GHJIZ@G6rP(R2w#|`N5K%JOUzoA<*%`07'
            'B>X?gd^S)_ap%a3s7Q;|(|ow^%X>w);mqoHxL?%3S;`^_6y#TsY$UhsCuj5~J!;Va4&q^w^g`bY7ULEyH~PYqpC*YuLQ'
            '{6JB)~ci=ID9-2p|^={$8Ef%2L@`ZMH-NjXFfc|+lh`(2h&<g?YXx>aD>{d6yp;Z~=b-'
            '^0+<BNtzyZ4ex14E>N{rIX<7Wyt9K+l&5HoyEpeY6X|dcP!MXNSm){7vA>&t!d*4+l$=?U1h=g%kUqv$Ae9z(ZG_x!tR'
            'Xvm-'
            'D6lmB24mlE{f_%F2o2mXWIVXC^%&4zH<<x*<?I}EJNbEsAcC(FlSF}~@}rqM#vB%<#m+%p{|vd4yTYQH8Zw8!K8!QJGv'
            '*48>v&GiuCmjh}&65zGl2zkxyaDG!MvS|e)@8hpTExrSOI@Xf_&u6Iev;nt9)#7qp0UU{wCPm$c|AT+=A_2SqZ~w(YHm'
            '?6S{tesp*w`Mk@^LbF1a#D_;I8ORm^~s*w;ZoUxzH(SyWWqp?uKaBlZpF3?`G^=cpN6#D&PRi8w7-'
            'Ev1Qv0T&S%~t~mHXnTr)TpD{tRvIJ0B8xPQ9i_>3x@W~?wh_9?99R*8p+WRS<U2p_89!$Z7n(JuKf>`*!|Nr}~uF!wsA'
            'K9S#Kd7J1D-Y&6WpAdUl{-`7v?p^&(T`c9>c%vf=gVy7^J22ux-l#ByqKX{Zp;~ZAEs}bC-ZcaH`9ULlesk5iP;n5&Ge'
            'Z2j_Z3e(~G>A1xH<(NyojIXMTDyxz_kGbyfYC8sgr}hr4~4YfOEaOl5aw#Z4FH%V`&;;}y^U<hL+Gc5wYS`2QFAO)Bqt'
            '=v!k=Gzk}_dv%_WqIJt4O8*8896_=$#Db>WV<)yogVbp#k1n5ALT>Ejq*w2I(bqA2Fz@g!md;{vlDYYJb<#F9BK~wA%`'
            'DwOrOLv{yxYo*uCwkSMkUa&ERC2gc})T@Z>ReO`S97s(z^X|e#G<UC!PO`|Mq-'
            '#_WuU||02Ih^l1oF^qd28^DcMhhU50k+Fc$@5uaeDT%kLYv|nWkrdTnHPq{E<Kf5sh^?EQ5gn2VB%(ycDR9$7RpLSz@X'
            '|`qFSn16iQ+Hub?r>vjTKO@T4+St8ajs1NSMJQ4kuJ=AZkLz^63)!C-d;@V;lX6zb%|*j=g0JCI{c^pUMgzv-'
            '{AjW<TnYp7Qy35PL%XdV7=O}gB8nO(M@NAQ1i?dR!|x{PIoBbXucQqvh}8CRFBXP+tLXCmqpZd(-'
            '&5Pp%WeOF+`iY(%^p3i}liFVcjm$PdEH4piesgu~sT1($h-'
            'AI*A+HXh}jXjS}WT?jM7!yn>souX&1ab9d%{_Sd12e((PV|NkPt$$!ZiCReF9Gg{S&*%|4_+_}$_`Bln=`K#5P`KZyAd'
            'FZ_dbLP$^=IK*TOyl|P%pK#7OwU73%yWzEm<K!Em|=Xr%qI^oGsgp5nC*)Ln5s@!nDJU(%;}*k%*r-bX5reaOutWVOr-'
            ')B=7F=$OnYrtrhS<gb9CH+xqaA&>E3aXnYhM_d92%-ncwcsRJZYA3V-%zs$HG?pW9cNe7pRZ`y$+!xAuE8ncLi$-'
            'P;|Q(J9W%!4nQl?v^Xe(-z*$RrQw4F_tG2XPudOA1*TWq%Sc~{<US!f&=r4o&!@S#hoc@e}&m`-'
            ';VjF%!zr*$%T2L*OJ+B>;K7*dbFrAfQ^0biP``Ez5o9rKgx#x8ZKSOMFdWk6aL3Jbqf5;fi$%0IC^;Kc&yANI`gH8nYR'
            'RGdoK#xlNHD}xkhR)SVG2vuhHjCFBnb_?$UV%wydkW?~%LS6-4<h&=2b8Sh-'
            '2@G<wxS+IzUYu1)JY8Od4BS~1qaihG+*w(mZ}SkRqH+fO{HGZxQaI315*Ref{N{*ZCEZt#V<j*QC|Dj3Q`M1S;a;n5a4'
            'FG8rYYvlmzf?qNnZA~NFyPWC!JBNvjaEQ*C_B7V19cNgw{NfDWw$Ehqv=TMvv0*KV)Te)ghjp4$!&qxaI_hErSF>g<J6'
            'I>AG->^KI2o1PgdYlX{^K8&&+7U6-@O0-'
            'BERb5vK1_re*lban@3rvT2X!A97sQFLyizFxc82~zD+d(UTqb?GgpVP%%>A;GTXs(k3YE>&7*rH<UYo2U5$h13-'
            'NqP5{Sp0pbtH#V4_Tth{rBL<Ah19R=)#6<uOpZqzlf7m*Kh5R_u;Thv}{sv}$~eV;}ojf4{vU`{uFhF6`&koxQCJzkgQ'
            'Z{IB_76xWPeGkdUt{WA8{PQplfGueCV0nM+jz$rt7&81;z+@T1E**SELy)v<ac?SExy@sWSBf-'
            '4jD1O2n_$_TeJY67KuNYWD9}nNexC=M2s{1TeUucRRZ+{U^ZA9}w=`gNZ0fOZ}Aue<!_(Tfnu2}6wp2icjRoF_dGz;kd'
            'yci1k#!0w1<`1r!DQCT4q=3wx5_lyqiKQ=IpvSdh^7Tj}j@@sDarWD+@hTOvoDz^Om8Mw^A>gz38T1+$!sS&wklEY^0X'
            '!2Jd1oog?)`y5a2poC71kZ)ZGa!G`53UH7uo{{u+r~2tWh<A7X3B41{zyQ=1ER6RV!G(bc+*<_ZlKsL=M~(><8<rS4cM'
            'm5K%d0EPXr!Z9M)gsZ+lZbO-UsR#CK;ny-'
            '7|o*)EB^ujI~UGi7Pmo>`vmP8&@$I5`etjy1q5VVOOo{c|)f5&rJ!W|EZq3v-'
            'vuw9;I_|%aYtuZ2TpMzm4I2ymX@k2$eE6wG(2?<%{=r+!+8>+k==4YS5E4Nt;&)gX__`C+&w#Be&>wF*}tcaxT+ytsB+'
            'i?6x0-'
            'k@|!;*Qq2=uuh;)3k^q(n3bl1HvW{@r5KdQycuWw(<A_GK7&CJg^v_QnB$A#9&|fiEsL<I~|tR1S=1eBZ=NlxvQ$qMjC'
            '_XPrJk-A}sZgfVKB<$}@X0CG{<0IN#2B75NnSSh6iohKd>clSQJ>uCTHjAPQKvmEr)IyRKACZs-'
            '4iM7Br0y?Ej@O{H}Y`!sry}J&;W;G|AjtPb+F?D<-'
            'asWkQgUMo<B(yW*2K(H2ShPPFq;BQI0b?h4T$2D5u1+}OF@d!U1JQ?1NcRNu8zlV;fyoI6DBd`W{ZDpc;i5QtN0CCgUl'
            '&wtoFbL%h8S>71<JDp;FVWAZmyQrRhD3qzue!Uxb_}|>|;TIk$~>OI~EXXxEFL>5_IGqXF&T>8yKtaAX~SF)vDADuoU0'
            'lBMwip;nRLG%+PL!+5izaZN<O~D^ua~?dv3Tc5~gBuqX_%EQx+`8+-'
            'zOXi_*%#2>5#Tzw0KJYt|)?I4O9%IkJ5S)uFnokhe_e_}_!FLBtJi)J>_Bqc5kOp@#9(}WtL{i6YSgtoyRDQT!3;?RZW'
            'MqC>DmG&pBf*5OCEYXeuiQak`dKE_3EK0{6;fHCqdpAnY>G>alZ0Jxfz<Q7@KrOyg@YxYawmkd}1J5!^&_Z8e-+-'
            'XR)sLb(<8a4TKYXk4g|@9t!S@PIXgj%#7(2w`V>NU1=c{CFxh+9?+(dO>{nbSNfxC2r$`M@VhJ?=ZqDu3+uy<uP%3ZG`'
            't4s^&j+Z`E!ak$!Cfi}yZnzfz@`^)3vn_eKy9~tkYt!XMP59QUpH@u-'
            'lk?gd_%8Axm^f~L!m)bj9I2y6E4*=V+bYyC?t$J<U*N-'
            'R2nY@G>W;Sj1IrH$NTUMq&KFrwxJya%62z)j9hhD*3b&vTCMKSNLzgkcPW9rN%28ZZ*GJ2&LeOyId}1mWf{R`ZBd)K-'
            '(zixfRL&yrG=)g1+(PV0V8`^CHZ;Adhx4(HrFU)^YQK6@53&>wtjh;Dmqk}}24ZJL4LXISgYdTJ<mfy-'
            '5E8cmzVb4VFnNL&#v9<;vLY0J(1%iw`mr|08AnvEg5q99a-n4-'
            '$ZZm)dEVM|SIA<tl01*5A2L8q*bYNf`Rk2DcrffafusvBSS{i*AmVudqmPfF`o;U?$x9P3TY42gdiuks!@Bt5r#7BF@)'
            'h>2;6@Yue`J~F0(4H$!zI~2sc2~wyc|p>`wOMuh_5g_TGWOgpd8+7?u5h6|A&?LjO+Og|G!H+Ee&Z&Bq1cD^f`}DL&{7'
            'Fh3boxtWXpx4HfOJNn2A>sLy$P8XBUaC6c1j5``!c|9<~_*RAVu{jdLx_wDyNkK_4#JYJ9U5`HNe#x+5Kc*0ST4cGPY*'
            'ODv1vosE+f3<?r-bb+S?;GS=8j4R2d4O(}5`=SKLnG&VXv{Z6mGrlg=R&nCo79c8LA(NYun&Th@<BLoCJ)-'
            '}23cV{Cdq&DA&~nv1%v0i!kX$Q(6-qH6IaF1`R-rHt)xqsGL-'
            '`X6%pm?2Pw^}+zT~)z9en`2rVrR#8CqW7;6Xwhwg>&aIrLf7_Uf8J&$7R&pue7tbihVsdV1LMv{L*n7#SFJ-'
            'S8qf>vuemf<5t=5;OYDFp$1|EHCV6pd&veaXol)v99r4GzFOhbx+*%3m;CG6ygApF)|_ZP3`8M*P&3*cIFn^k3FmbhK*'
            '6(=E-|@wADSHhPg*K?OG3Jr$1MiUlX{X88QV9N!AwLm77ujP?8tHk&tq-l;?gIQkrXOybex_cE~U*T-'
            'yu@92Ft5lVRbD90{i#$=io7`3iv5XCW;g;X8%nwXI0+}FtE`~}pdQ!zHz7pw-'
            'o!L~XWM2%K}&O3KHxQ7W1zh7Z<l@OaHu$-'
            '<k%m6*(&(vhK4jSEVfHvQ+z?R)l+;)wi&hy837WWYDyUFEYna^?HMga}~8w&9D14uvFOzOIp;LJ9a@+}9SQ^hhp+W7Yn'
            '9JiE$H^Q;FPtAtLE?LXGtmKYgQj<WC?+YBcpbjQ`|AG-'
            'a4_7V~g7bRauybA%4xWjGgDth>)VFE+Dxet#c4p$z!59qr`5uPGt?1XvX%tw#iv(Zl$Gf4jFcHjvnQ<1VGA9^nhr)?Sb'
            '~b8VU5PqRlgh<cdePa|P1s0$p|t-'
            '4lxXs^R~=5KHCs+%<fTV2?6naU_nj`c$#?^S`>%j)=}Gc(UNgxw`Jnyt(HX+UY{YYG1evPaB<bylaa<o60EhPRvZX>l<'
            'A6mPcsU%!KN1^QLw?_&-'
            '(Uy)3ZeKW{}D)w&DXy2;tBXE$iu75Z<uf)6@BFQ0}YplO_zP~X2DIg;$YG}dgAy+Vji>SP6Rfc+(A@N$bm$AFs6RAAx<'
            'f6)OmITq_ZxN_oNIrKDvk<@)1O)<S`ALyo-BRoh4sg?t$QGA@(!T2RLe(O&Uvm;fv%&3|Z!j(!%#4FgBC82#vrS&uY-s'
            'HNj%TP569w3FeD`Aw9{upb_>R>%L{+7v3PeULFoX^UJk0jM(U&zl3}amj!Kh7p9i~qkHS)Kr-'
            '5%!F#=o$}a7Ng&q~OVOchYEGfmy$B%=_ifg!;b%5n#AdVp~wh{N<<1lFbQ~SJD2`=%k2DPMmJk}~tgD$n;X}L07zPc8_'
            'G}hxX*M5-m;h`@#`qDC)K+I7{LFF4c(DU;q#{YYPTW=a6*M%r}Tw98^iB-'
            '7B=jWD(d%uC_Kq*K%yv4W>AB?%EguPqU$#6s>F7`c)#O@_*o#dp7;g)2NOCic}x8m(paxi7rgu_=`A*lB#4s8&j-'
            'e*hkn}#PEZn;Kx*_nXv<VI92<s(ii-'
            'K5@Z9m)hbVOvZ(OTNAo3)2!G+qRFw(CaKzUj7FAFV#WYEqxfYa7KA4aa3K_1kqP1o@XyX_??Z(`}Ao%`zl_MQ-mvL{K+'
            'rP54gbp4NK8K9JDmjAX5GU>u^^!gkI@{rMC9C^mSKRw{{5%9x)@_^Xy^e`(zZ%s;46k-'
            '(lA81@fLsL*tf9c=7!ajAa)wtdm)wqcDtzd3?d8<2~^iFJSFln+#ij=s}WBH56}jhtwtF%-'
            '~;gRD2aD^TE(8`SC1>a;S^2@6MQ&Tf0awdv8agqWT}G)V@Wtr0Pk{JsI>%d`--'
            'CqG0lS2l55}qNzp3keq9PC25~APGK3&)a20N9iA{he~RuaOauSwS)AC@08&fF5btck&3v<1HB~@tYzG<gX^YUE_X2*-'
            'u7pMdPLL=z#pZRv_|oVtY>&ChXsKhO;$DiJ{%z#uB2z++ve9(?URWvp6k--7lO)4JoC-'
            '<D?R6<|FYGbYO)A5$>{}$yJ`W8yor8AkYM>Lmcue>pTKQ(d_d!4En$QOq-'
            'a5j}y)bApo5XzE8k*C%1J<tZ1mWAtM1He8ns2g0%lsY~NXW<68@Hki?<agWeS<}J7h?IFe-LI<sC{&vAGqu($F7X8==1'
            'jyayL$Z*SjjPdAS$N&5xsG%SPP7szTxQt?;0?1Rkr*%wc&wDcLE7LO<MbHQ!N)Nk2!fF5-ii4Srx^dKw-'
            'N1winXaYp2mF5ud&OdVZoVYv$nG;=+$WPLs<xaa~lN4$xbZ3}YST%%gcb5MFJP+Ly21s!MaqDV&u__udMTk~Fg-'
            ';ss$YNF`PN^dx!T7kj#-^fbEX4>#5n?`vCLT^M0{F^sOWj-ZAq<9oq9b1N`mW?za{R2L-'
            'P9=}pQsJzQ0ogvl2fw~lLw}(p@^sIZgT_;&ODH_e8YhXb`XM)*A6CxiXNvp`#Nrjr=+|UGqOY!H<p%`P74J8|Be68Z<r'
            'Ty?Ga57#ze4xi+PdBK2Gec9w5vZFtWH;fVwx>Jlo4cl2HeJshz_WC^uz~SU(!wTIrvk-'
            '42+#W({rkOG4<_XFr9O@OI!$FBEsPo34$~22dMJGcw)kpiaeS@v@>0eVW5)$+%J5<fBp%oU3VR>CuU=P!z<XoFbWUoI6'
            '+*h7{ekWo4RIv#MmPo7`W&IzWymeU!|R<mtBrCqHVw6ss;LBVH?T_oyQGZu2qy*i`lSp`U}cMjdW3AJQ{5iQgPSD6nKL'
            '<XkhUlyjyY(E7G%JIQbIwUMUH0U)$l?a2YnUp_b8Am5uGS0=TgECRDbhp+STzWJ!xNc2=CiT>&p)(QpjTNeB9WIZs0rQ'
            'i#Tn09f+g4Sw6$!hFl4Ah9?A`&S6yMf*4ys2#-^(O^8lZ-)%*f*m)<K%&fm;XhChwm+O{$-2fl?%-'
            '$hcm%<X+g&K%+zLi|9>_Jr50{rkLF#vIyy8B`t(U^FNJ^ZHDx5~!sbwUfyOoyTFvgYqPq5~4Dv91}jOzXM_(??p1yvhh'
            '>8u^d4E56GBJ<ei3d^ueUza4v^`W_>3G&GwApuWsLeEGbhF_P!Z{;^p{YwsNcq*dnR4(DpiNOo)FGy|odtx=>h0{ak^j'
            'CN^9+R^Kr`Q_&;{2UfcMT!Wp=9matS_i~BMmuf0!gRXE_|LS#hyQJniQ{MVutWf<net5&dU$r{e~jcEA+?4YANa`$^l0'
            '|S7J}MC>>4vjXfnSP>Za>XU<7jbEOot2h(8m&^<iYb{`*~{sJ+5K6qk}3j0w}HTiN*mie*zHZk*PAZaTEm})g9bcg6)s'
            'JBXl9i=TmwnxL@qy#Xgg7N5sUPw8ULB)*Z7!kFy%*P||$!}*ny!Y)Hs%Wpn=xcJ!_6|OB-'
            '{dLsnS0^GSUFr=B}v+XpF_fE8qAywfd#hWETaDu&TvWNAEq5>nfL)N5@Rwg13^~tAxLjXK`*yr_#;Fh`{p)sy{`_&jol'
            '$^Cl@aAx4~G$GFH{DY=*E%KAFy&L#uO9j82tTR8TwwU7omMDhDTOUB8Q5<88QlaRF`{7e<piQq0>@{LJW=Mp#fMgEG^j'
            'u;f<+)GuR`Lr(9&KYAg4PW*;fb3*{C_mrpiB;r%)KDd)bz<9kVz8jTd6+h91;q}jGZD<RM&eh6^2{)?dE`Uc>KjWJZeR'
            'w#-XRgLeGSyr|&}tnoJ9u3^{L@%Ko}4Yf?@Tk46#q$f$=89d#4cEUtOH9$PtfG-'
            '@9@YW3X?t$0|(a+JTxE<l3|fBG@FB-qVml3hGD4SJPSK=+O>XM$V4@MDQ2Uw89H4IB*}bn;4*g?4<|mN3jUW^0f!dBFZ'
            'DPQn^6JB!mJ6${&<E_P$F1f*a2m>jxf8|1KO+}1Bf35<ANrvlm3T%D_-OJa!<ULSP5O?MN~|!R(s(WM^IL(WhH74LD%v'
            '|DsdnRg57Sw@=NEyD*p#cxr-y~oh;s4Y7OD%tN~du<%>+efwWC3q-bTK_Pqh<tT_mJb;rrLmLL>wU5r19TjAPl2-'
            'NzDLgJM`{Bckpe#@Svk0l8x&CD;ilJ}uq%|D6n%PxE%I3HRP(m-ihE_EXvu(-'
            '38Hus%i^l4YafmN!ovZ@`<OdkPR^A~8R@`+K~t-yBkYQtYSt5IOXt@0B)Z-'
            'c_+_29%6j1vA{uy#C<kh^!NLQ@d@X{f^fCmhV}E`7vl+y{APD4e@IiV6n{@rSP>4rm!`KlpP5;Xpa#QpRK4{aXZcxU{f'
            '<`w<o_j)TJWKZ)~THBkwCfx~A)@vvGxh>ZTzYUoMBQB@5L4SE5FY4_=US8wv{X&9byyMxa~BhffClkA&!6VizYlmBNKp'
            '6ugit}$#xri&=K{`NiD)O(AV9-'
            '4+B*{{&@;XQgRO~cN=PvD7T9{w8Q!@*~6aL0I*u{%5%h6nb;kxnPDQ2GdCf6CD>d^PM<teHF4y{y>1`EWVv92w_&!eG7'
            'jCNqv7@hYu=H)rfIASDX@dc0VS`UvogRX~NwHCR?W&0@GHVa4r3^u$OWwxxR$0jCtQL_!qpb^4%oe2Sd6lR-'
            '1%)lAveaZJ?Z#hd@tpm_@qGs{ki+5bY4ow!4o+34YiKC>K5H*-#=LGukX`q=~_B_l-5Y9)l$RBB%<^n#y)-'
            'FVtSh*|8hg1utX8Aw|=pZ%#Z0It+Y;$QRA^qZ?RvrdhLu06tRb)j*n4B}y)T=js+FJpoaI|dH_^um{p%FsGrl59K@373'
            'zrW?sI1A3t|DgT_EL`fdqhxC>U{0m&!$YF|8P>>Wml^E{B$YYErmUc-5vX~>X&41qocxOW3D^QFHsW-St7lgF2^IK~}='
            'ly&I<vktSj2;zRrGCcg>D2fewfX(9`SZpQ7ZcV#^3x4|I17`t{kF;c^b|VJ+1Yp#CHIn)*kV?9SqVux?l(?bDJbNY&Mb'
            '+d%PIomN;%}k>>)3SpXA#tWCdR(5SBVM_WY~j;??7cq3cL^uMC-jwEa?z{3uJ^Y7W78PE@OzAR>ga^o|xb=4Q)S`LA_l'
            'B8tE;;{n6Is<xC)4n3iHoFN(q3`8}W)w+w$~@i66CqqJ4CkZz1tpwc0>_-'
            'Vc`j%|Mr6ZgyTZ{;X8QhyH`E8gJF#jf=9tOl&MTEKn~7Y=RDEpeXVD0S}-'
            'rBjCb@G|8zYiXT0+fnH)%vj#l{_kZx_ScIu&DKQXKtLQAT)qqMA6wvj&IBxkCOogf$JXcMVjhVwNBz0}vb<}GTowqUZN'
            '_=vuE50eKIu67g3anJo1jipFY%aT8mL8S;7rE|vE%80rZ|6kxGfWwD*c0pQYG*pkONfT2cVPX3UYXr7;`#b3tludpp?L'
            'sa+R!75^=-'
            '<F8Y|mow#N={rzb9lw=y?a!xO(WG_Hz4i@}b?F|nOor1QEyP#UnkC6p#jM@!8pzGyLg;U=^nqDSRm{ew$oymc7DNpcB-'
            '2|M^jAO{J7sK}3L13DElk5~(hmXCpz)$fhUTU4k%y!@c!SYxvvW|xIQyF;w<|vHKq+yQ49<1BR&oWz?1CR15=^igBwu1'
            'dFIG}3@_5ng<flmk&T&x7geG&Mh>puSbu7#l@pF!^IGWI|sHyGXh0@0y9sI#gNzVW?+un*~we{=+-'
            'WLJV+vM6fb%m?Eu9L%%c={Wwi0HhXX<Gl~(@Zy1HTvDe7rsN}C?j1*qtQtTt$qw31{$+7Rd<5o@CivQmFvF_z>61Kuv~'
            '>~zgKt;x3FlnhdlQfT$JRoewlj=%nSpGl2k3+p(6b4gOm~YaTsbI@Y19HT+)shj`w!%ZZ6rRFLio$wMQWe7z+=S}oVYq'
            'gO|A;C)p}Clz}8^KR7WU&PAX!oQcnR{)?=`*6l2f7(TL&T53Po&ATe=+NKDRWo=~ZR@Kr$&xwZ_Al5XNa_(fbX{tv2Ik'
            'Kn7IEvR2A!Jg-@A*22jZWzdcG-(RVKmWmR-U4zfd4`^%T{xYh0y3ZDKtGxvc^#V|-'
            'JOG(zj+%dXQ<$LyLP(EKLs{lcf~S+2Aa-'
            'u1`01L)5B{UVcYgVeC%0@9yaUYZIK6l?$m|DC+B!_Hy?Z8L=h@)NhWIV_5l7+!`wP~G}|16GY-ahRUsKp8$1Jn&0UQD$'
            '$3nb#=|>y0(|G=Lfi60dW*jZlc(3<w_z#RUm%DAl{`40`I(gV^s_E5P+;1I%x5=D&fqD^g8FMABo1=%cU~wAzMG*l9U{'
            '!m&^cZ&;-'
            'cI;pV5sf>zGZt_fUGnEwHQ{r%E$(o+Olu8*hEY+lPkAd2arrQFlhMHY*oHS6kyCuLx}S;bwPDvKdtuPh*8d3pjmAVBPR'
            '-!t$nuE#Kby!awfg)Y3qlU8|7;{?i$BUp)s~xQR>qaJ>w(yzUjWT6lowhCd+x^BWNq6M#c_2c(a@hS-'
            '@;RA|5*Hb`W`aYi<Imp+DETZ`Z!XB_s3JCPSlD{x(Q1sb(HgbhFaS%%*YSz}8i=|qS&B-'
            'oC?ZG~^t^VTw;Nni0&<7;fKZ6kv+0Kao{srcs_u=sWv4yehove(37i=GtI=>R867QDnyF@cbg&%y8!;b8Mlg#zDz0GT>'
            '?9Iqd`kKfpmSQQftW8*eZY2*VtSX|J&QV#eXY(XGO69<ibq2%y9rtaKbQ}*&E>y)nHs@0E)YSA8)nz4Z+Tm0!l=XS{Ke'
            '+H*I>hXQc16=0jh=-'
            ')zX~4_<Sl*ulo79G2$Ds^3n`;NwL#41~J`)s60%+$oK}gE0W!?9>h!*xqpnNSEhS3s)`IfWwtHn^KcoFKx$1=oJSK*V9'
            'dVFObSI)YSK>NNfMUE&DjEk6Kg7h%d308+Y@0HNuvl7TX2m@XpC33;E3e%J1nVbhSz(IN^Ot}hS+ExP?s(h&J5|{zIY)'
            'A0d2QeI+aDle=3DB)Kq5pQV;ZPnoGiv4^uDMqNs{E=D_+kOfn?H*8Z}Bj#QWs%vvGW`c#Sv|@m2mf6Eivf#fnOH0#5ms'
            '$T=>;-L2Vm0M^?jjB{s-'
            '^u7L0*66_qgSP+yC!L;{Y=(>NBbl*xLk1u54cE>Do+20kqP3^&1T^aNWO5ytIPLk^42KRK@N!|n>d-'
            'd&k@XbMhel_DpJ<T1sPgi`7N4zlIk{4|3|G^K(4aifJ2nT!fXiRYy6uD}WgVFieJ;F2R)njPkEd%5Gzk$?D0IHd7MsGn'
            'C{F`VG6XQGK$;fXOchDM4e)beNYB$5Kd&gmEu@Us-HW3zwJD8kk#CVnaQ0aG?DslUg-'
            'T(pSn~0sbcz6l^b8NtiLP5};FG|F;$DpBm1BlD<F#Qgz<H_*>I4bCaS!Sbn)F=>B<U6U%dQJLg*C)(3`v<$+;_+ML3Y-'
            'zVi<J`h@UQ=4`1N2Kws4AK8D}rdAAL_w-}M8jyJ~c{$RCm&i^!^BO*pN&h-'
            'vuyEqbmJVK2S(mY~KNi0XJlYxb;Q=AO@HY<{?a>9)iMFP<z!j|1{l(`|}+zH!Fdz(6QFl8SOGx!G-pj-Wxk9}Qc28H%T'
            'J@xPFppkjChmRejvEq?`EvG_jf8*c~o$z*b&wG+21{KOs2%i#E`6o`mf$GlhbRy$!x16>9#V$_1=Xi?FPHS6!fm*td(@'
            '%)C!Yx^+5p^0v_x(z9eV)S2OgB`Dx@WbB~&{#j`kw41tMAJB4`5Q}Yj{cy1k&B5jQyk5vlvp#7Z{U*ZTWndBPf{=W&pF'
            'pW?2QS4P5ROhQFa;q8&=@0v`M^iZ?0Eel^}ah)uXS_N3<@w2h)j9@Nt&~E}e=*(~EDhHEurn?~NN#D@%lb*)rr+l@6#K'
            'eSl`!w-`sQ*W%rcZ%`;E7B}$iMR#{!*cpABbx8doj_pif-K`D-'
            '$z3cuu%ZiNdo0l*%Z}x1AcI{ZyiC99Np#LKgM;hbkh|U(On*+0cUqJGJAbh1?KOLjB^(_8^ZkFxAN=2Vzk1W2EFcknc<'
            ')vRZi!UF0`Up_eRT+}7ZL3HRYdfzKOu!nR$$-'
            'oCwy~f7jAg|6Eb${fa|^M7<APMq#V~k#~u;3;NA|Hz_+M<Cl^~Mv+(zw88{c^O>KXyq(7rQvGHU!DmWMu(}xxyS^A1pK'
            'Xru3SI7Q${;XwEFZ%x^fAoLj{c4kbPFl7+3A(x*fSGd(rPSTv*~CF2ynG&T?;4=Oa{pn=F$wa(UWOPLMq+8;4yazLj<1'
            'woK>vRqX_K-Z`Qh_h`|85oXfb(|@O^44-'
            '{B?1);82eo^=xRpWkx8EBmzVPMc7TTz{w<I)ZO=j9~KI8)84A`M>u+DcI}h|H=ISZ{%k`e^O!n&lqCE0(Ito%)$Ntl3~'
            '^Te~w|Jp+5QlHH1nJgxOC|A;2^ao=H}KPG$prKKck=PBm*=SMV@5K8S=Ss|Qfej@26U(4n2B`gFfXD~lr_1&n$wqR^f_'
            'l*eV3HK;p}d8w+Hp1a%*?7L6!eu4&U__!MNr7FpXx2^ClGYE3Ty69DZil6f1>HUsFG<`uHvDwW=-'
            'HrnE?N)(EohY<>^$4^~Z{Vq*Nceqw3l3hpiyYZ}?6K83*eLA=RT`%lU*~$QW&H>o$d!kRt9fMjfB-'
            '}N`3+c?c?v()3PZSRDjuCbKv+)8m?qg#m{<A&4_&Cm(@nBibTgP$l<0`h*_mLYejcO#3bSXEKa}e}48^CDDPS@hijC*y'
            'fzrh*WcM|D7__m0$7?#kbo>EfCwS5=4+Sy!S`k>Z<w2mEKkk%mVmV0N$DSj>=n;GX%deZ^zgi!#PriuOww%lhcAB(@E0'
            '5?}O%O)fGMMk=gvqWxXl>a_Cfr)duuURtb=d`d&k?SEKTIVga~a!q_K~GGI>EFk37>feA)m=34Apj}oa-WC_01aSLqQ0'
            'VZ)S<qEk)<}F*;q$3uiBdgJFIvS@~O?Wxq2QMz?pObdnfDbzdU7`K6M0GXXfLdJ3juyXeI^*W9N67|q-'
            '^!9}Nh;QtYaYeq`RDXE39@XC5PBT$0do|fS1x?OlZ`V#JGk|vsUr_gQM2^OD=C$+BzXj5VhW{T>g$&!=gfx#!}KGFh3t'
            '1V&N<rr-'
            'A9WHPBgJfhp72>w~!;t1Z4D@oqkwpUR#kaMQtA`JLhq6f4SUIlE{7Jc!>hY1fCaUHp;i%PR*w|!AQew|xi83#ir&fdLk'
            's|U*&Xr;6Z%vAd!pZ&iwRpuZ4pvCsK$(}>WTR&SX6DJF>Jmk0S$rM0W^$wQ%Byg!!~qRL!(ij9Av6g(4)H}1SiH`QamM'
            'BvopzpEMfZcyb-NL+SY*S#6T)PvUk{8~Sb~ka9iFueLl4e)NPXT8HXF*Ru<Tyc785|xQc)P~>}PPd6@briVYX<?7-'
            'lN#L77_%Y`$8?5_igk>+!m{K}HtZn3=RQcr}T1OT>4@K5*PjjvPGf2SLMGsQI58-k-?B`LD)sX3isY#_F-'
            'j;3P^;o(3WLa?EtJLxnrH7}h)DNjt|F^*<s&xe}S=)2CYC-EkD17nftlR(;A7<_F;xqpY4Qn_!?Z6DFgUfP&=%`0f)=6'
            '(V?umRC4v2WOB;mrxAL&nBsw6du2-'
            'g*9>VSTipZZ$B?243a~V8aKn8Z6{%LLoim$3gDJ}acG__Bq64I$?_~ad~xtQ2`zg_9X1DmH{X9Snvj5{pDWNI?j39CNh'
            'gNLyw42o)?D;x_)Vo=$B}FjhX$3C^f>zni#NR=<J+CkE!P>Xcz)9D4+0_jTW|S}_8b%xjl`hWkr40w3yhVxVXea=aI>}'
            'oS8YW5cbTMQ<p-8p5<jF|E+l;hk>w?(^GSWY{T$=B;Ei^9{1;kCvowonkLU^%l0S-u-'
            '@=iu5(L+s^V#=`1=*X%ig4t3EnU0*C5DJ!M76L}ct6<^d+i2r>7qd3Zxdizm}Y^QYYRHc|3G82F8t&10)MR@WBm7Y30O'
            '^R0UxDUG?Q3N{^n=HnzxTgwb6F0yZ8i;Ca;E?6JJP{QU`L^I#ahUKQ#4g!VIxY5H7h?KBHekZZD64jObf*AGZqI;M`_5'
            'Gqw<gE=qtA$4YYa%Vuzn?IaG5Deirhk8B4m+<Mjt{A6dT)NgSt>5fCU^|1`;U{7>BFiq-mwqVEQSUBN#fK@uwiwo9V!0'
            'GMVV66EpZjRc6=>{V3Jc0w4F|6_Is=tg|_c=(z;ZD@z{EYR#&e9WaWZ{ot94g+_Bgu{JRMM~wJVwMZNarB@`nnSbOeky'
            'L%ux)geu$S<-(VQWd`SL7S$xamaOiplZhRw+nsq)@{mmdf)2xiD58GMNX6d9bbq{{mw4|1G=Yh3Z7uS7=1uo^sjO#C-'
            '<ML@mI{8zK>AUJHijd>f?d~cF<2#Rmv=hHZg`)KuEArd%G-'
            'UmLOB|AXfz{Vf@*gI`?|B03r^{pUPWVxn=P}nyuJXdalxoI@?w@qy!ueRT@D<+L&B-'
            '=sY9k?av7k#Pan+~ixcS8%;v;v1eq!jsikGD*rx>a2P{;<k{2L%g`?VvkXH!v!D^%Eh23@!fP=3H-'
            'u8xVI?>RGSd@zwd^i9La!Yu$D@6n}x49#b2K(6;0mLE@J%@??Xa({EEq=gt9;z}Xw|Bd0Dhzj~>_Yhsa^A@bw>;<nzG?'
            '+g|f>~`=#jyW`2eDk|yQO6tCmvNVgST=_kk;mhe6<~*SGpXkX0+j6A1_9)<N<x*1}e!F1<rG4;GvW#<^5s`%e`F55Z^N'
            'P{<ob(Wwwj{IW^~MTVpZT<1}h9ZlX1xKV6fkOFjSNVkR$5!{_`qxb)==Nh~uUK5Z=^#QFy{+n1rO-y-Hu%af#Wy)ZMSV'
            '*%8EdW2h4?veG{?Rc6)g8lHAAv{$vfb~zuKt^^8zFuLD2c3RlS;2FXd*uTBySxP3b~M4oUCW_zp%#sdy$S{W@kD9L1uL'
            '>+Xl06Y`SX**kYerwp^v5E*{Kl_9IGXFhB?^BZb_oA5f8J-'
            'X%CZmP=LANaUpuT1S99ir^M0Kn_S(^335rAxcKWbICw#f?ejXH%#2>a<oefm{u{*&Uti(<g)^iweJv;*Jp(Ce$8fgP8Q'
            'Es#_)M;vELk=VBU>nRzv{%Eha61UTY(z39hk1sPx5vhg(`gu2vqJNQK2z(lDCDS;kSx87#av$Cc;6bIUYYY2;iPqN%*9'
            'CiVT!Iq{h*YaK*W;P~prETW(w6b0ZTd)cHnjSG<I)$@d}iw+@+3drgWp8>obv0CUqSFL*4b$$T|?41E-'
            'bNX^1w9P;^2vfgFl!LTIQiY)xMyPm4>_+XvOAO;Qj;K5(du`IV8ai<xH@i2w+uF;H#C)2^oP!D3I%t5K@9%Mg#fDZp`N'
            'V#w;xaa<+lf5^gZ}m}@jky8cTP#G_w@+ZYq!vr+(ks$!K(KxOe{f)OA)Fa(p^~Pj@B&$d1w;eF431*MdQXs0+02~W%E{'
            'F97sB_P@9DqS<Mhx9Au6_WJ2;NYL3YS};F+wThIjZFZ$dKx+FMYSr379}gE3nv2t<zgKve@9oonxah2kaLwy+R~f*wM('
            '@h4FF>kQ{NE5L%^yfAN-ChDz-hvOek<8aFja{GoS9dWGywQ>_yYnwT&dLV+Q^PA9N=saxvnGdCnMW7<*1yzze8J*jM;G'
            '}d2x#@18tu~|vIu^Bfr`8V+2p-0A{Seqz%0gu|7M$GTg_FAiY3uC-'
            'l+auUy>(Nl*S!<YJhj8G(YI0Vct}}dj2XFMs>JS7y@2W8^QrM=5#Vd%g@+&i67~M~aJ;z+S$xSv=(G&eGVDI82tL5*tI'
            'qg4(-$8tHNl2EI+&9BhqX2OFHOJd35%|+#GA6&Fq?1&0v$Xs(qjqu#CyYfqYEe}a}LUGC6Se#-!OGy0PVl`kcz4y-'
            'aWDu_xeY`GUgLJ{Wc$09S~*b>+(TDj}EMg*MQk-Rrco1^RU284;I^Zk+qr?*m?3Gwyn7c>Qc+mRJNG%sJtZ2hSp%M5P`'
            'n8c9mT+ImFm^@)nx*+hfJcAzZS(j>`L7!w+wck)^NiBY%My%}!N8j^;%)M=}MuLL70$qX^{75x~EZ-&wA<!({JCVf-'
            ';P3^pNoWX~>6X1`Sku1nZP0?uWj_QDgW7IGGYT<4(xa~H<$$_5#4FWfDafwwnA;Muq5&~D>SD7tWs^ls0_q(VFLa`Ot5'
            '-cM+^Xf(vO29X>8EkpARA-'
            '3^JUgmP}!gc&ClBZEX)2{T;yFAfgkZgoYtgAuut03FCgCCcA$)nlqK8)okW%Uj&fO)%{QBg7*uGM`)IsGTvS3jz-'
            '+g(xt74D+Vt^)Mh?F-L)+{kr92hw=5j=VMxp(`dwSZ15*uy8^gYs@LC*OY-m!DV7QGRF%A48~m-'
            '27C9;8W7Z)4~vel(7Hexmpna=j~aN`{2!7j=coNRP<s-U0@CpN-'
            '<@cBL>d1oRU@n9D!T3Y$dH{^1p!=|=)0sIg~Gz=_vWYQuz3u%GZoq0yuIkK`znaEP0)gS!6=16Wq6;$0`MWy8(InfB12'
            '}e{RfhfSO}AyJ87k?50u0$!1mo+iR}ktcriObm4+Q@UH3)IjXgvXKGw51+}g_3w=9EuvgYJL?|TOKDO=Q;*#O=XPNa(C'
            'B!(wPfOg_}5KnA`yE@$<GUkkbKGYG8)e-Q}G7X;IjwiYGcHp;88dXX-VdH`ryzZI>l7{ox&xhUcyHpMqezRcY6>zZ+yK'
            ';c7YY=jpUWcn)5x8cD5lcV13X7XJgVu{=uzL0tUo6bS@h?))^4^+Ui_QeWrWEop;x5+oFfn|oC9VsKz^daPS=Y-'
            'Fz*|U>d5e)xN^kdIGuZ+*cGd8}xCA<NB;ba9164>5LOgj0@*gSUdpiq;j8PH%I~+gfeL|@9Js<UoIZ4Xe#pHadIMAjQ@'
            'arNUagsWLRU7r`=YBKbdDjRh7D%xU#tty_PV^yn#8phJWRV|oHqf@a0hdGwu!p~l!fT-g*q!tUI#YyT-yOsqSEk6BaVe'
            '&}f-'
            '#sM$;08MSX$b7klZv+#dk%QQSR(wJn%>r>7W)=JygPs844olqWFX_i=Oz#jd6TyV6*HDi5%HL3NIhS{pK&o@`V&y0-'
            'i$Uhf>t(FoTB|?;*#S6+?Y{2rQcR1*Y8{un&!bZBi|)sDN(l@)Kp(dspMFo+-'
            'M@wUH=)`w!<`Uc}CQ^N8g2GpXO#5Xgwn!l&1dQTFDKFk;F;`LDWcu6se~)TfPSP8^37&!Skn&xhiOG8@l^Ttlfo6>?9|'
            '7ccSsq-'
            '{dV(APV~3eiX;_+f%&`Kt+sOG1!ygoTBx2jKLz4ido?fTg;e?4;Zx5Pm&Hn${dgL*1?T)m{qNI&Xk;^=+*B#K&&o4WiP'
            'cb$FQG!%gm~3}%@x+$rLP)ma13ct!!;e}=GrNCn~cJ@4`3_&<7WiU-%_C83??3iiM<S!P~VJX{IR$A(E4IO-UPsn?4cL'
            'Ia7ADe;aba&^*qQ$;B6x)uX6gJAi@3$i%91ZOwL!U{t+#911$M8m|{SJx<^O1PHxAGQH@-'
            '7PPlb(6y@o?UbuSx2>9H8FRI1S}CeM`z9Bi2Q$Xr1HNnu+Y02-'
            't`_KXJenym0tPuJf{rcNH)3~wvy`2!JyV$z^aeu!r|W4<nsPBYJMh*b!cH7^a*(q9Djk=Mw9VbT`8Ho{t+rl<*;GLB~0'
            'IZ3*JQd(5>bV$Y!l#^j0XOa_{!TQ}%qAb*zGjJ)RJou>iN0?PF218#Fdnh%B0&q5CQmP_&)_PxeN@bhsXw&=JM)dH2X&'
            '{Sbzx^FNlpT>!?u`#|KCy+H2WDQe7H2<#wt+_GaAz8s4tjhm`LkTZg`zr9MW3Le4h%-=-r%|H6|o-'
            'FXBhTx`si0+|x!TskAs;3f!A9znevy~>5*j9pnY`Ea9QXu-pe!;_o9e9J+4EgyvVdj1&o~S=ZxAUif(8^NyS)7Z~(!QW'
            'Cs|NTc45JmbVA@^-'
            'Fl#;C`AVGW=dyrlF7}OXSk1@C=TiswKT9D~SClE$+D(Is9+W3lY(?(k=MdUw0g)A|b9*gFK7_H+X7VxdeA33sacpF4UG'
            'NHPR=0v=s3}VFp1_X5VAMqkW?P&TED4y$RCZZ|8TNf3+3rK*_9v8I3B3;+Csnn79n^*))4Pm@>Pk>}HrIawi(&q}O8hu'
            'WwbyMNpxpvz@uCb;u|iFFX4Ot@%n#6sU;Vh-VGx48WdY9;en|B2$A=%vU~0mQe)n<#6@MT4;iMUT(j?0oo>oKC1TU6Eq'
            'B)99ZzU&#RFK>BI27=xVDq{xco(n(TQ)w0M80}BFs(t`I`cu_Bo?*K1b}L1G^Exv;Eu*6l+!ko{CTX1CNB%&$aET!DJp'
            '@XU+3ZV#0~OA%!RD=iX*Xp5xC8D7w%j~;EDNWyk=xf=$IZnT~vUP$Jb)`#d&zcBoo$Da599sLdp7)7if880OWtxV#0?w'
            '%;|N2aY<ghab`C>H@FW!YT{{`VI$f5;XHQb@Z#D1CAiBZkv2Uq#UM9**zMB~-k&Jy**~G4^YTzsKbQIme!-'
            'WkIN2$EcGSdZ1=DyX7uFq(gw4lU^tR4<7?Jo?uH<@(FulAm;mmvJ60wE17Yb-)dpIntt-'
            '@oBQ!IOn#}GCBm}R5(jC7w%(2Djri++AN*s-ev-'
            '2<~xo6`;cs&~=C#wci+wuNB(0&G2&M=YlV;Peh%__GdCcOnoss&mkR*|Us3v0&_R35H{Bm7tU%3kt73(cfxg5Eio<drk'
            '$R$F5q0@6U)0Q<#dV1j5ljopAQAH{2Zmivn>Kj86rpAY$AX#)fXdmw`2S`ots#adQ%%#!qzcaxHSq6vFb&rNHSL34%2*'
            '$X~`nMse0Q)?#TpsJ0%!@Mnpv)awuMjZ-slzFtljJt%{RGajrB57KegsTka(Qt^+#GR9x2Vn|4C1mQQFsA$0tKHg^dUQ'
            '8JJGGkc2*N%XX>=X?NLNHQ#1frFC&@FC+-'
            'g3`TXt<IVoz=%Tt>IYdEQFkg_F?I}2_j>D1?~y7Q>}?qa$?{INoz)^ugS;u;A&X#t`7DDq=4a^HT(V?fI};jV25fZ7W3'
            'Vq{lC;;Yt13;b8o%ygNGxolnVp*>_22n?@gjYoxtax82m}p1g$I~Vtz7$k?$i3jxnp?$;V@8d2%%k`Z_}QnF+yIZ3;ts'
            'T8(MHb0z$7^1xfEQzT7A9E5!Z$#l9D3f-'
            'Tg^IlpYPoNo$Yk5P##p8H!!VmYaEhha><8Vd8E%@Of&6GPHO=<2MQhBlr`wENzcod0}L^^$j1{fVQpES(fi8%Fa_*<Sv'
            '4}^x|OZFD{6vvIXlg)sLKEe~GXYk%0C_h-LPm2#WqH|UT#%dNp^39Xbb1sm&iLS*qzf$<BD2JPxQ&}fYWTVMw5cZj5p)'
            '99RxvI(|++UXr6@o}&dn>`Dc`>^CWKjiaPPD#$5O$uH#ZO^(QJqzbx&P_G7xijJpxbf!kM*5OKhVd>_)hXBxdseki=Zj'
            '297|LGk#!{nwEA-%U740e5|`6*{?TkWX&KKDXE>wzeSb7i6@!GXNGzw>jLM?na><qHxGH-'
            'G48}Xcm$Toun2VW##yoYLIGK%ELk+m*%X{Raz96ajlyWS)i+tV^P#EEkJpsG1L12PdeOONpEstQ;9Cn5Vt0-'
            'VAXG2f17cBHmrd_J#c;TZBT5x$#3%BDKCi03r&{z(4MBZ+p^Yv+pUIJ*G7lkW*z1Z-'
            '(4F*pXkni=MSVwNX!DX7)(ZwPFoHBz*^`BPq^@%v7*Vq#4?o@JZaX3*I9bwoXKTJ){omoO<9PAepTft;uDyTcNQOi0E6'
            'BH3Ea}3Gykw$D%FNI45c645h1Rj607sp?9($PvjXp=6S^Nx5jbL_UZ$nV40y)yxdK3$;8Z7g9z_#W;ENWe`@2e2;Qsl`'
            's-PrA0|L$YEL8FXBUmW;c!qSy#8tNF6tC?{j(L=^s~I1l=~T0x38k?~+Gl-'
            '>zVg|)xaKtMx;Eh^{=?Och()5I24&4S=`DFbX5<$-'
            '*(IKK3nCO<ikG0J|ZBct+dxkrLHIbpDf>J4(UZ@xZ(E<a!5E#n|^!bl$2cV)4nU<!wg{V0BWh_TOv=^8EuPHwx8+^Zr<'
            '-u!yT6*p<vqo>aL9kC8>eslsA_Amj<UTWI#ksfJr!4biSp!<S=srqwl3yH^@0iQ8Fzy~6tZ$hlIF6fF`gYAWIlF!qLIb'
            'Nx#uovLSHbLNNi-RMr>+pU?Y&pr{XQw1ChN#K~xO`C@QV(u&a;Y3X<GT;q_XU`L<m2E}VleX8WwT^&r9q!Y8BNmcf}UN'
            'wFbkKX-'
            '~RRZ`PcyRZB&DyV+`<b^Ct3s50SAh4NXoR#bS?W9FLfSm}jwQJiH2hPx#@5AD1Cr<|g>IZlO75vKX_;gnl0F$Ed^?IOQ'
            'ot)O$pjXVlJ9r;D-JVVZ|E4!mp`gGW$gB!TjO|FK*Qv#C%?GjQc`FlEjfW8~U}?BH!{aqw0LJ!Jso-'
            '!5Oqts4(erZfNpzj3ooyC`jV;E$ZwA7W&;DQd7iA!dIJOuSdd?cWno>-;sGj_k%t4gqGs>h(CXp`N61#6w-'
            'I8b;hYjvu-O@k8ANY$_7Px&y6XGI#cEPbC3<=tbeg1k8Js%~F*1CpEHXDfy3!c_DrS?3ZiDOX5suvyXseV=-(~+k}-#-'
            'r#ULfc$mhVO^phK;`o(JUbOtt~S1m6_6ha7c0(@t1IQf<+>VPzdXPy-Ykxxem~*Hzc(nqLlX>M+`!$QO)Q)GX=-'
            '<AJKJqHFT40p7`n?IU@I6uC(B|jK&7!5`u9{|xWp5@V{XoF^ZtNWQ*$t`j|YWD=Ft()+qC*H1FyPrV{q$J%u&yUulI+^'
            '!=2r9*y}m&X-'
            '!6tm0K~a#Rz}SEXMR6WnB4g7QaXA1>W;1s3_zQzNa}6y%QL!=j0jP4Sl#R<^VqUa0mv?6)^otALEZa5B=koO^Zb>VOv5'
            'ZYOX6Fdsb<%FXY*=dt-pGlD)`vD+yfx+`GI+wS-'
            '=Ow+wW&<jKZEDySiI6bfGEV~NxOc+@t8Tf2wJv6z1Hv?3DQX*2FC6h)q^c95!Y2j6S+uphlx%65Ij&7Lycf?w}%LjJK_'
            'a_o#O^lHSQ>#1@SMHTi{dtr7${16-'
            'q<N{tTWLa1jfqQijeEN19Tf4uKM`@BY(^h~n6u?c)&8NweT^ZoHEe*MLCz5OSZz*`!fUnaL(7$jEbrQ?96OC7}q<8E<+'
            'P#YHne~EZwd7)l#cL=IdP%ZG3}B_1GsHP$pm}#Ju7AA-ImM&lR2e_y-'
            '_!+#vMPA?<31d}#D_~x=Rok;OjLMmgk^)Ha9>0cH4b`#!$Ce)MEPovQE<fG7t^c@Jd1!8)WZ<b%D|xad~8)#5*gErBi|'
            'py0PC+Rw0!i!NS!-'
            'y$utRLW?$hRiJQpEFsCJuNiXggAm{eoLZ5Dmit_W>1*|NPxmtqZ8bNUJMHH6ngyRXdQsktA$n2^mz7fszadHe^^W?{>j'
            '4$*vK8F>up5%eB2X4_`4!Tt_#Hrg4hNL24pMectd>jhhl{&yy=VXV3enrtwvdj|qI(pl+ltikm1!bLMFv_h)&MYp6y|z'
            '2>MA}D`>?wtF!v`><Ee70Qh`{jHD*REw!=B9!U`aYJnB&pk)H>)EcHOv3pWkePrK@7JA2=kzRn<$dL8%Gh<9U4jX#uPh'
            'HwVMg&qVXy0%l~TF{lnEp}txc*fmt4X{<in`nZ7k^@jwxe!>;Q!|&7Y(^b%=R6?VqpR<&O!-'
            '1|XWn{SwFc$w1ME0F*ViDjD%#00Gag{De_a6o8S8b4O<B9!NzPQt2Nc*q+EDG(gMq$x0A{$vi)|C%a*~`aZ-MUM#)Jp^'
            'XUm~ous(`a%A?4+h`S>y>olzC$fCoz#GZ~e-Oj(Wwx`JmOe(blzKU@V^{ZSh1vxS&D&EJt5&)$-'
            'lEA{BzaIE}&Xf|k0Ucv0uUXbm{M_C-zm>V`%*OOfV1wWH*19NlB_(jGw9un){$D!FTx}4K;Gf3@7AiMS~W|cQ3VbO>ad'
            'MX~E?H}#vf_zF_Ttl?0|8;3EseA|V`5DA;$cu2AZlFV(7<gHslZ+JApwfjowwiwml$|o*#D+@XJaY!?&ToWWr?TLvl?;'
            'T^b@-'
            '~s6N6JcU_z7|ijGBtb$ToC{qzI9W201HO9!3~T#CxH4A#s#;y1@e@EliyjZf>hxUar|c4M5{<NeK4$800GuFApp-Iqbh'
            'Js!TwO|u?u{R{a=k>vHAr#Y|lN&Yf<@-'
            'MFfuDaI421S0{#$sU3C10}b+a(;*%E$2EK6ploO_wPG6pc%Qb;~Q5GM511O+O%2QwLv7TI0-'
            '7U*u($VXPo8Q)*V0xi_j1cCfV2t7`{&QnC{ki@b+(*Y~nEM}&i<Qab*YSc7o;AKBm)glC$f;TCT`IAsXZu;xaZ$s3Boo'
            'lC$-'
            ';0}3s&KnZ;`Gc)Bi>AC?3r|)ZgPQ~1u*EJ2+6o3>Sz#M?#EM{QsU==l9wFcP_K{zDg(y3gkCOhHQ2c$EEVfC83!~Rjx<'
            'ehFIJwZB-nXFUSrfG^7N%kwf*~>09XMW}gLAWekH4Phgaogzz<6?>H90B(zn3n7xxx>bjN1?ua|)jn3et1^8IacV2u=G'
            'xktqH&m^Kl{Q=5b7UbFS!(%As!CH7Fu4u);*A4!vHCrMD(q1|$wWM2Ob%#v}$$c~Lr<<kLIzi(r37)HU*u0qDL#bKD-'
            '{Fb%NbtQa?s)eB<ci2~Jj3-Wh0ZRAd!jFrIA)}Ta;@pfXJA)xPEs;j<JO-Ti?oq$U%g|iuMfv+)k&ugb$R~SO=m{ct;f'
            'O1KOYz17KWA8#hZ3M&Xa+>jxPg#U1dfm=a5`}rO-'
            '&PHKJnrsQrm*?@Q5<%$8Eq<+_KDf?`LUsbvCkkFF@?P8OkxuL^BRm_{_fq+LTK{zEuXv<a-'
            'EG=_L&p)QE4oC}a%21hC45qZeW+cS<q_&TL07t`|gp^fN=QwGQkgcHoxRmiR*AV>zh^fog?=)GbvXFD)`5+x(c=FBXXk'
            'RVvuvl8Db<hvIs_P4FE;q48QSEJzDubyWwW>wZc6sAB-'
            '7whpMo2*ao6W!Q2P``~$83ZBo>gTpUup<TieYJEAG`ns8*YX66N|9*tG)v8g-'
            'rV07u4k8b?Gu?ctf=Im`X2miyQ24VqnjEmlx1Wm8rQ$8U=P87L<;y_y_ZloAvLMr!PDd&)qWSesmUY!8SSG(8_-'
            'gA}+<%hDe=QS4`15mO`>++;bKa6lVJ1;r-'
            '^$1c_W%}0I7+#PV`pqMxp(0@hV9{Ci#b|RrMf~ABq+vi`P<1j<PiYfmTIt6#-'
            'Dohu0Xxo4BWt8jI9be<VM0JvS0NczU<3DFRLrKC+-'
            'BQkA>rqr54WrCBSZ#yod`#`Osn4FSz*D5!=StB!c@Y>VAF&;szqj#mBb6>+YAV$8NEh)S8DKXKs)zBYPZdsNPb#HxAu$'
            'hiRu-5teA}$A7m2VB4!KjQM#UPEO6zqQWM^%bkJ-xwg>1?J~BCCu8`@E)dbE1qsb-m}|CxShOaC|7m-'
            'cjqD0MWF$czg(l(_3kSTBBLs(p^gw)3EGi5C2d>de>7KS^yihwxbZ-'
            '{nNMbRnh&D4M8;co1e_7}??T7Y@@6h1CNmN7RGL2C`hW1umxN*)m4o+tyXGbf(%hbiGw-'
            'e~R)fqV%T;L`41gv#y@K^IPV7)j>TIYYnW96~v9Wn|^&Z12BA}KU9&PJ{akwi7E1r)x!GPZwYg4^=njIn2BXc4&tH&~j'
            '$$Z#@haXi5Ez%i&(jD)hU9^mN{5A78V$ehW8MDbR9Fm4F)N1Wh=undhpy%;*0xtRs~zmr!QLS%<;4|T6FLN#VC)UsX@L'
            '5Zzc5FbUdK83=p>1urUQ6F18lrXE$4}#4npfa3=Rz{EE;f8PYhv8e~4DTd*$7M;=>l3hP<$4ga>Y<yvj4|x3IuT!=1&)'
            'h3{#R?~9#v!7{&5{d5s{E|G&(5NPUn4JJ)&U>ql^j-rW4gfQwp8i>98XmP$X2-'
            'IZ+X+eP78qCnlO{C_~d*LPAI;GH*4%ncwzoe(!o`{q|aWJ!|c?KkNEF*ZqC2=f3~A?i*Z#y=(Z#XVj*v6(Y~O5{-'
            '6=<9_qQRJZX2Y;F31-nHeyp%Q-@y{HnkJNAKt?eFkah7P1Cid(I6l(Oo29|&85b`Y~C#k5~47gHafz-iAVVA_oql9-'
            'uG(p6(glT9{`yM6?eCNCu$<(GkfYAi%pCEyj)8se0l2AQQ-'
            'A*I_7<;~84^^yHxo~n(v)zeA0gaP&LT!;<9Ddg74#+vB~C$V0t8Rw=~(IvAq;0p;=G$=R&aVzqPitaYJe)t_e3MnVEvJ'
            'Rq6qAj<m*9Vk-wSc~k4$^vA7x(^>gWIf)tGj<Lg(oLYU{*yOMs0S7r3#6#SW^-1>E=SYngRMH?}4Rd-xDwCO%NJ7pGcN'
            '9q0@z8)UQ8`ZmZKE*>fC7NiD{W(T{6*t$EbtQ56Z{X5hm9qj+Refu;KMXpA$;rEZrF!YuwI=yz&@=G7+!Vb=>S`=|GT>'
            'I*N}l_&#puE~%qa}ic0tb-2!Ofuc)4HP`qp@&^_QDfa>>@CfKjRt(m->ql`jj=Gk=OFi3pc<4Hye0mTPWWE49k<-'
            ')Avd)iFYF{3_gf!!J3gVe)k;BXvpqO)c5#!<4#C;S6pxfek*u@3afy)(zWtJqlj8nG-)gzy=m>GVIj{_q-e2Q}-f5-'
            'XX_j1@+73`(6AQ_<o*+rZAvHxAYiCl#vX!vaxDaanw5>X{61f+D^}-zOF8a(PN}w5*hsPJ>K$El-DLSBoUE5N~YZU>G3'
            'f(~M*ICl$UrphDiWBJO9><>~Yq4}_UAg*zGNzR00NUAt?kYZJL`|SS8y1o&?W=(kx0gEI3dY;JOi{N%6<=+wfzs-'
            '9D6kaJcD;6LRyfrvAvcpw_D(=3^`p<^yTC+wChh0w;4hCxLHchKxeIfAU|!yFD4v{;j+v6!@B5VN8eRpn49iG|^L+Fcc'
            'SqZ0U2wMJ620uFNXqL=AyO&<WUsjrzc1sk<eHS=7>>c>C3_)n@;R>7n?&OOMIH8(OogYfmf$ag(fDw-'
            'CEUn+1%?MBP_jIp!0tS-AJa|R*QrAETxB?TE{ZA_<-k-'
            'EB`PZ&N30zlke&QWP^gH67%@$F*d2}YmR|!at2C%k8UzVW3Vrys8&s`Vqkhv4z-'
            'j$NvaLFSj#g8Kt8JbTn|cgf4bw>{aRFtW9F%xl0As?^NRZBA?5N%gm2ue^)QY5GuPQE9=7CL?36z)~hUgb|s6I9rZHwz'
            'kU86Jp{xBZTNGPFhvKejCr~z5N4@~)GDXf0oCJ<;<5|_Mq2#hCK?z95i_&T695C(mBbwKQO322Nrq1M0Os7lql3!}zg!'
            'xs{p@Xm#GH17v<SmWb_3O{7S;^=HN2)=@MCoV?)pxZFj^EKs!?ZItg*?65Vjse;GK`D(#gO6E4UgsuQ`lcETCO4A3Z;-'
            'fuzLl1^KC&Epx&<yPb;8(~C(u6_i2eLbP)y$pOG_8PnmI0z?Q@>?RmQ?3Y78f6BrO{#;@Y=Nfs_nK(4L`-'
            'S!FXYXt_VAwHM%u4LQ_s+BK*<+)XpSI1T&G7m-'
            '@Op?92V60NtJ0jHNIqgc*H*y^<($2@2N)0#pQPj$nR?(1;a*_>KcE73!x`gFAT4&>2j=sEohc^vl}C27)-'
            'cC(Z|doYWJF71Y@PCd+1Qi6-KD{6L6;0n6^>uKHfJ0NkUMDSv{60};xL$9<ZlxmitLx>qI+-'
            'HlO_fj!T!2$ywtfDWiv*@!yV_1E7idCZA8LIGeC7}5fe0r=0Vt(Rb^z)x!u1ya9x=;&T$_BU_RFgobJsFkqJ#d0VE1Ge'
            '#L9*K!_Il31Kh~CDezPM^-'
            'FF4_IH#b~=s0i;GKprP4E|WM0aMd6AYFAXhTLsI<4ggaU7dulj8bXMg9EtI;|Cg^=7zf8KO@P(d$9fd&^jQIPb<E?LH}'
            'qjvIsjHjN#)Y$dUDGs1+3jlb;#k=D}XUnl}gWa-'
            'A1Z*7L!W+L0hPb_H3`+C{Dq2h^XR57SotM$5l)rtfqMz$P&c!qqCUR%H%az2xGPS*}psY7SGC<I!tp61r;SlCiH2pvU)'
            'L!8H#Tn5fr+@nK1Ls#71*B}<^#BNpE|#DL1VPMV{b2y0#k;`!PWFc2|_Hw$@C-'
            '+r6awp7rH!DS#`RW!6tSq3)?YPmg|vv3t?z~mpip*7<w-YZ-'
            'I^UEUzJ7p)=K3k@NCj9+)yGd5i^4l%i@1Rg~z{nS^xBJlSM>A>Us`(hZ<|5Z@dl$aj5C?Xt|Dw~}O<?~K3)tyeLuj)CL'
            '@y|Xd)b$%8-FRp^{t`q?hUkq*AFpgouOj3nUx+-'
            '8|3msfSEZTws{?=*X!4T+rm4zq(vJmI?hA!K2HKCLx@(#db*{_3S6FBQT5ewwYu&r@T5u-'
            '9d~yRREqmk?+XrKUn)zdkFUfnCnMmk^iis1olBzD%iyUap+v|3G)bE=2+eJ$aDhq?;mE{;r1nw4qAy#?snk~b^`k-'
            'PU9<ori>6`W)ukY2JqVSxEAUv4f|YyBYETJ^#T9kq(c_%5mGQWVMDBB6oN2k7w(v61apEQ5zbMAHF<$63uMs)iS0H=v8'
            '?541)ALGv^nDl)Hk}7hr))Qr*v46W_RvERXM6^1lHNlpsH0!sa+G^{AC9e@4<SM4K{c}&GGDZl4WWwQ=9NOzFU#UNk6R'
            '$q<^o0Wez4%HW}0=TjIQ)hw~BbYgRJ`O9@g2JqIGB!4M;r%(?buDo7>yzCaXcnYgC7vZ3!gM;b-b;e;mL#8-'
            'EW_$Jof3;FS>$PiMZfeEoSNbPT8q^5_%dG9w9JzUS0xUz$L>H_XN9c0TaCt{m8Xvm5W;?%~4Fdq3g1I;2%9A!@|I+SzT'
            'Ee)o4lo7GfYGvGq*wCjPYV<$467cg5=fC_7iaT4z)v_d)hom`9SHOA7uj*ZxnECJi6NMqf^p>^f<KoW9iG!`tFg0<(n0'
            '8TC>!S7mXx~>HwqgaODW%QtCz$CPHUyoVu!eO+c3109Yz;E2f)g~sCVRQO=u*!c%<f1M^oMS8;Yl(+8od*2MG8KBHJ#c'
            'E#Y-'
            'n+EfQG9&GzF4iQivO<ys5%xONycXPzufw%Yfs3#SnJ=7?y5Ig*h85AX3Q`OFyGH`|cIeUNZ$|Yn~%5b?G=!S{ytL+G-'
            'Bg-'
            '@tn(M`21pDB04MN6%#WfO_LzNZ9U<!@URV$=ZznynndJ{@EtO`)6OC5ezez>(Q>dMX+wM5lmWr2ERX@fIBYD$C%E1yjh'
            'Y187;S{xakFQ;e|XH?wbgE%akFA<BwN~TTSo%M5w;tB#=GtMXvRp!HQA$iDg(8lpocEV!2$}>$m}Umla{$vn(`trwckR'
            '%OQJt4sP8Tjs2moNBTE7&XZOD!T#T-'
            '!}@!A6Bv%lHpb)Y7{+c?6r&;+&KS${nHzE2n7I)#OoL$nW8u1svFMLv3=T#yqcDgGw}@coq(v|neL@+H$9{}zX*9#17|'
            'O)HOJ$n>*v{<R6u~HHY-bu|!<jp40vWGaTbLRFk4gA~$E?UmV4m!VVa)q?Fmq)6NA{oHQ|{jN5A>T2>pya}3H?)-'
            'VW63Y;AM2N;OlP<QU8`DO+JeFG<GR>$4WUg)``b=Pv_C?59I|-'
            'g^}3Rah0x7I8P0wlkiQH9ZJoL<~r+S;l&G?f+b(NW3y)*`P}Oi<w?xJ2(fe=ch?Nl%D%wfZ+@g2#tC>d^0{EVbT#*F`0'
            'Ww@=!z|~_y_vUhxOl^%44iwhcW%OK}>vh9OHZ?mU&{$XJlwNqhKG$+<mc~`SDRCvvCrSQEUidifnm|gnSgU@J%G6>&jy'
            'mdj&8zX&BQm5X)rwhB6P<M={^{Ml+3^2qr0hCu8#}oUt<tV$^!VnTYA3jOMCHrtL`-'
            '^LSA_^PnJPB>$KEcCPvd`Ynd_Pn&RryU<n<2U}j!7Va2S&L+6Wu1aw6#9~1&e=gp)>*sDByMrWY4GL^FU8MQBKMF!-'
            'Hr2RxfuJk!B9$qZ#Fww~iJtE+qWF^(*2+5yy8A-~)y}b&^9-'
            'C&tG%2i^*iIu1!rmouIJJk#+!N!Os7qCQQTCw{*nEoU&lVa@!$0?ZS@S;;_bKj|Dk_rp3F!G*^!No;zKdS|4#)fqm2Gm'
            'pg)qrEuI11uD;$Ou7N{p)_Z#_TA=@53gQ?Un;V)LnHw2f7|t^>wXiTU;TW1)7#SN`%riIRa4am$%q`4KIm0Ud2+lCj9@'
            '3#bqEc*F?B6=8`XtvGtBx8q{FM_?E<En~-+k-'
            'Lreon>ic&+@;qRmm$QafCrQL!9J^cbaJOkLc@8}T2DEXlQlpnU4@TXbz!{7#nY&J~nN#|dK{~_BrkHvD>?3kq4GyW?46'
            'F*>6jM=P6;P(?&O8*u3Ls-'
            's}4U0zR+x+zL$Z{Llun`;PnKOTUJaS4p8=l98ljVG$egfXlhK<>9{D582C*T8Y*n|zYO)FFX1f0Q!P1*3`bzy3sfDf`^'
            'Gd65$*?s8~<DAKc&DroA`Mgb^80Q8yY{7=rJ5c<u$RD~>?m7z=8Rn_|Vc&jSwf!KQ6>;b3`~%;8Ty^7IF%~T1&Qp!m4x'
            'fPa*|3Q0+-'
            'UCDPrwFjSj3$x$N0SY1Z>ELMcf%O3qRq`Mh+Vmai`MSu7r<gyU~aZi@0;wz!jN~N49ro!y@h!KE8$pd$3^<cM6~V!jip'
            '@4U4!__}~>*?M!3AA|oYyXbHOru~`v!3ZMAFE<$Wr#GS&&Ww4778y0b=@L3b=BE*J8?pgSt1a=W(!y@h!-sPWFM-'
            'H)J5qAo==(8dpX2T-'
            'x6z*?lMXqJTBJLD!4rjq<*|3N^g}b_0a2*>Kai{RI>_d0%W5XiJ7G4su>TWNK70I>ma)k9%uvw8*3oiveWK&+SSdmN%P'
            's2ZC?O(E3kwgnmfm!$aip7fLS$OiwDj`Db|4f<Lu<)Ex<pX#{pV)tx?b<JqlG6KYZY(kPZ&HtSjgk5D`le6p&vY!28Z$'
            'EW8<9NnKn?viTIx?mZ{+`f{`cP%iv2+'
        ),
    },
    {
        'seed': 20260730,
        'weight': 0.3333333333333333,
        'checkpoint_name': 'final_tcn_2019_2023_seed_20260730.pt',
        'checkpoint_sha256': '1e9300ed7f9200ab75dd9ece1136ed5ba02bba3ea343b12cea8dc0b9f4f90655',
        'raw_bytes': 131294,
        'payload_b85': (
            'c-o}730#cd`~P1mM4>DxO2`)NdvmT^wk(lEb~4qpX`5*(*-EsiR6<B4?RKHioa-'
            '*3ERiKy60&6%vTwhcsl4^^{(QcV|KmRos=3c~KVPqNU1!d6kAt0@yo^le&N6?!ddR5Ca0RZOUco#AH?EMYAF$NNc5FA9'
            '`R!ie9aiyWhR^LHz9CS^74n?jyj+ER*;V`w!{@4qF9rzwy?L%eFMmJ%5T2KZr;sn_D)bj)^__!+UOs_coCQ3WATJ*`XC'
            'coo&|lyjY}|z}KYaMyPEw-3fa}5IE419w#n#84E1cPKu?xTBTv>69A-'
            '~gHIq`4TfFOS7Y4_!Ye8m}dGIrW_!|Y_E`CTNxB!6^W@9OIB<I`d#u*-CT8&AM<8!tB7g|9Tik*_S|cXQ--pWD5?XBRJ'
            'SAisyF*pR1!XN#Z_U&XVtowks#D%FYP_q3CZ<o8<dDZ%s>^7}aQ)#mmV2fFz9yDkmXH`I6a_X{>^>we#MFk`;DXXj}$N'
            '<zLyi)(E=xoCbryCrsV64(7(sG35)mLq?_+@60^r3SUzF-'
            '`abTMQ0rH8@yeP^SeoM93fN$RGB%LE|=q!`s13`6F5l>i%JHWShZJEmS=rU*C~$@V7zZ7K4WEm}Y#V7K6sE22CUeO<Pc'
            'ALcX~p-'
            '=e)iKYxMm?+jYDgPHTKT42_#FpdO9T3{gLGmiYx?R~Ytq`qw0p)B}gT2NzKQR5`2@hzwcLcXmdf8yW1ekX2HJD4SZatm'
            'xqE6h#;o7w`KCgj^Y@~8jpt2JK^?NC<y87(NsR@6)hYE}!%Nywk=$e;74FQfJ&Ztibb2g&d?<j-'
            'rZ==?t_x}dG13tO;@g#5*heCIzM8vQxsmb8Z&@wqJ)U0N-=N-'
            'Vmyuy{hgyCdJ@Z;QWE=h>cS%=c=w=>3PqrEL~{TCl!CzMmuC|8I+}NepNYHR1DHEDBmJ21+anTUbFtey}4yq`k#<qi$J'
            'ynkhfDg|@tvwn9Q%*+N?-<ga$*hqZV1$FK`;&obk$X<<dQversi>snZmLVlDZKl*QHzmpf!o@UNp-'
            '$ILRrNv2T@h!9sLVkiHKk;v8tvO3-'
            '&$8feY+)t0vNlOrn_F00g!~jo{?<R88MhyK+uGAC`P*B|n)*jsceIr?tp%Gd<YzeYcm8S7_|K7-*&b@e-'
            '_>F<tJPw*#9~ehD_6+RbL8j$ZSi;V3fj?(4f(rUE$;cl;@&ok`&zL3h5Q4K{DXg6Y|Y}K_D~~!VT;AXtrm|+EFNuP6$$'
            'ypj{IZoEw&qZ$J^74`6pUvC9O1(ghpFvDCDz_{L=Q${up^>?O7)LlP#?BR@NyA>vRk2jF4a9$glj{+3)0?ZBH}hpKGC='
            'Z>3$3(5hN!7lr&wj{NGsowerda(k8;|4Iw%YAfrSgjLhRx-'
            'R6`I`VJ)>CB}4$h+B|X3oFWTGrctl(nv{tan<l^+Nt#NB+G(Et>o}^6s~XTJRfMEIw$p_)uc;Q48y_kpIMy-}tx1-'
            '^qL0o@U8^)@t$j9~NJ<S$x@oeI?|-cI3bL+hS`Lo7zLI_-'
            '|V*zH7DkUShGih4n$m|LDm7)ZSvdk@vYB&BT!ZrG@sjmG(_S``$wPA>{vb<o{~#?4OY*kP)wB?Quo|S;;jCQP4sBR|`}'
            'iC;n5+708Q!OVI)a@n1qgNAXWbK_~Igza6(FRnYk_6=Q+oZ<Q|psB~>rQTnZ-ELG_yRp~BK>G`*cv?o3O(l8OI{MJzYN'
            '26z}Mz7x*y`>s`q#9}x4Q27|o;<Fbc=>PbLEpdVrULaAx{Xkv(Mq?86!dGQ_y0}Tl+v}N^Z^oj*MI0z=h}a<%>)B~vj_'
            'dd9^A^-'
            '`OO|8We=6IhqZIwHd+M3|Du}<Mo8(?WI784x^0QEvxyRnlnB_^$wmoA{g%*^O6W@^4BFXm9YThG30Me>ekb1epTwKA+B'
            'f~pHj}c=rEH6K_D$RDTmD726j=SHTes3>?X;r=90^}r;-36g0I32aRT$mQy=jYko4@#0f-x<8S)pKT8$VJouGRhc-'
            '|Puewyl&sQT#JVdfxK&cjNguFZJ?s6HJl_N=`vSS3kjI$s48L0pi)g-OEETMSOLYz)t+L<+SGK?9Sr~g9N-'
            'l!BmOhu(^^mpsOd>&yVL5D3~VgqO=oBc>?hz=RmLJJb}IBvfNxn@nttJAFlK;C73S0xJck2L5uU#!A{<;Gf#Sca})Rn_'
            'y-9EGsI03N1xa(d8GzPbKffN?8o&LJ8+Z|W=hTV7q<$yA<hD>-'
            '%{sbf1e;<p0k&)3)hG1=gM>TF%is?^l#vwQT~$CoVcSwe!}*lPLj|8e}-~{d6Ls(>n)N#IeTye1hXZ9gZ~WlF>-'
            'd}1qeMQ`E%v@2?f&5N`vP}g8Tg$EZ|9U5zKQA3=oJ*F1}eH*}2c{+m7h(;VkqQa(!CBf_ah%t$&aBqep`IZIjXOI2W!!'
            'EGf>PpJ0KcQT4AzuAc`_u<)-'
            'NB?2ue6)gJKMy{_^a`C^KcjJkj`33p9@B{*9Nra?SlH5p+(*IhjC2ghp9ra&J#ceCq@6i8JDwnoW{SN#erE+a6)$ibcD'
            'V1AWZ~r0wN2z#irTUkMKTGA_mek*If0W9jtyJwBrKR%x>&8Dy<@K+Pt)=q*SMz_CYN;ebUGj0{`3SlHS|^{j{<Vbs*CP'
            '40RlOzdzf{Oi5~ck+>VK5Szs-D0<iFG=pl!WxW&KBC_-*z5&*nd?B9H|2_#N~|NdhG;-'
            'P^ZF>mh7^%OAxE`n#pI62X64{j&@ql4i-'
            'w8_f0b64&0}f2m+u8!V6)7$`}Ki<sob6@&^x+XjojILCgH1?r!+<>EG;w9$nd$a8K#m9CIZr7NXVsbtLjTe(-'
            'YrTxDadbK2?k2vDLlsHV%t|4y!AH@y-JvIGZ)-'
            '`R3|IdO(v?cN1OS!f!iGLJvo#dKg+cjzVBHP0LC|uN^*IG*!{ilsTixneHv9h$qf2j-'
            'CRXVQ*3f4>3zu|McNmf@cKj+p`N?KxD7EF0*F?nu+IPv=_%ZHiCwo+V|ia(mX0Do7{Kta4Dwv$v?oG>pxFFz0G0P*MC-'
            '%YTgt^dY`Mpk-;M&f@4#)ifwdPc^2;vI8Bo8jO03Aq9fo=}h|X;>f`d|WS|Q0-YGjkLLL-'
            'a&yvo||4vp0p+NxVCt`_i_so+g{Fd(+=?xdTM(~rVUqd^0m1hVrHOFTj<Ht4h(Yf^%9Ej*7o;t({|@dp1=x{+A{6$B7H'
            'zA9vd5_+lplAaFdsIG%!%GNfM_Z&CxP(DmMR<<-kC3n%%|DC8I>JMbcm`d5EiTraRNu-'
            '(ToC)o8d2PsjyBeIvsW+9S0kEyIoU+=G04^u&F36L(q9jT@@%z!N9k-'
            '*1E<MZ)MP$w*5gwzi42^|UQR+x{6kLRU}cAh8F*cFDCtlF8HGH^7G{<Z1t@;VKaO)eCHS6f8)U#HvX%92zJom$QF>(92'
            'h{D{>a6IY6*OQdN2Bh<EoFgm49tks*<i?wr!Zt<z+BNYfX{Ykd$cNS8n@B-'
            '=56KORTBWteD7hl#dy#ET0N5a8n_zUaaeQ+e9r<Z*qp+sZG<kZuR1Pp!EE@dP0J{ZZK|5j+_n-'
            'H3U5Ngkz#3NodW!!Aj+Rtd7CMU#|Cswc>n?mMMblyp&$BLS4iOD_ogLjnc4k|AxQs@js@_Pq}j<VnH}WTkDsJg#5sr&l'
            '^4@}(QVhED@J<)n6O`SeI40;MIF%m)S1fOE}*!+LhvEKQ*_KsuPkxt7*iuv;2*G<bIC{th3d-'
            '!4dlTAN#x_DEx1@y!h1m+X+vaX#W^=_f@X`FeuuE^MpUUTNrb`%2v%L)BWoZ{Ue%F}Fa$K1qws%;}CF#TSEC3HD1e<t@'
            '!m2k+Ljc>m6ZcWckRTYDMvD2z_k^=|D_ta;Zhix+ygcrS_4QM;RW+`Gl)%Y&R%J;JlRTa4IQ%vxgNY~$S`K7Coz<reV+'
            '|A@=|-{O7~EsuCDY7<{J+Lz1Lwu)1wuG7okJ!lhOy*ec=@oua5glQ)f7o?I_aeI%uN-'
            'g3C{}Gq}zs2XB`Ks`8eVe#XXyo~W`mN$fHhFa5&TJE(>zmy0L8et4R;+qoyKY>Yc#WKOi};~`#1;N;@ku}Kdp6B&6X#8'
            'o9r37Vt2k?Y@PnGBahtg19vSiAXcadqm}oonE2mX_TFH;#7V*M=#5?}q;_xjdYo$k<_@X*P$-'
            '7qZx*t~>zMW|k*9o8CaHXMDe9*g7E(dODw24Pw9oiy(_#g33|F^j5Rr`}wOWMTe{qkPAd`hc$sr#*`A06MciTBhw72K%'
            'UDz0Jv;6?v&dadG9H`G6oh_|eIfvy6t0HJfBC)e1_TyR7(OW8=DQn^}K@LWtx&D>oq-Q8Tc#)cMLcMB^cOH(7Bs}ax2-'
            'OQb5#WQzzH!*hQ85;3SE!_-'
            '_U9HT`tXzyN%uEDF+x8W|*RS?UMdEgEX^@U&^ARNU^cQ#uTXqls<(A^#w+#AoWpd^%lPsye;yuZK+;FU|H-'
            '=VjE`}y9#)fWImRuuOF4xe6>tgQ4HM4Nxx*NH=TbR1K8**JN+<B%Z#>OTlrtU_jX0Ddvo>-'
            'Z=8S_j9$J>?!>7?8;7q*i+(I&+;7pK78-O9|w*u~Y=)YX;eZpt&aurRbRa<jB>HQ`#An;9CLS{j-'
            'fa>aU<u0~upGgotSHzS^#p}0X%(za7{<+^(EoW)z`V4k03w<ZutS77>k1(uxp1=#kCL!1S!pPP8=;p;2jlCf=uCF`AdO'
            '?UR?`gytY0)HDWZ8OX>votbycd@cCH+8czwz9P3TJQ|5++AJ8ZY(W~ct#dnS92>fBP$aVb8)DpDc9Z2%*DdY!p(|j#5F'
            'Y%l(p?K|8ejiS|{7ITwL5O-'
            'ApYEjV;_=On63?hHmEWJVSG13paP3iKU?=Q?6Vy7oL$g>tZj)JYz$1D@zMQOB1mx6IWM3`R_&WpY4>tkC$tx;FL5ur=`'
            'h}9uuT%fS+6IrodVHUb?Xk6r7O^q=a_6bm?YRP$9V{-'
            '?~T(DkbkEOPF|F5id^Sg*6~RvJ#(_++NwX`)GOg^3OtluB|mFNZ-'
            'M(Q|Aoj)jv(e$6d*PhyVH=yo!wS@9*GULWMl(R@io|;y+))`-'
            'nGbK0FCA5+15c_Cb>W|DD)PhH3Y$zJa0bSnL1zPXGVQYo0I$L*(|r@`(NraxWAQnbbq*xQ#UK+h&oeOE)Gfz?OMwWq|%'
            '?zKiavY2oDig`k>9!R4+i{Lp@nrHli6wD%({>(hv_bFE;NU7DzShAuRQe}ow%0{WYMq2%dzj4jv>lYQUNQ)!0qJbVB#Z'
            '{nimrTav@rh)NdIaY4uIJlxZfGxhOD4Onemrl@EB_S@4P~+AJ){j#|KP>UZS1#R|;ma+6SG5%vl<#5!cMgP1-'
            '(O_&^sVqF-k%r-^<<XC^kF!&Goe^><cTd?G|+vyE*pH|3Xx48%J}ad$Zj9<9;ZHN!iPgo!OJhxaIBRNtT<H!$-'
            '3c?_wqcPulEMGqLKLX{Vh%mq%h}K_F^xQ6msS}g>CyskQcSNtcO+%8?{J_-Kv-'
            'g+ixF$rL|je;<sX)8mPhsm7Nn6cBmv%4UgiaaxSb|V#03Y4};V}LRj@kojDS-'
            '8&2@MGD*os%<_lt(C$tcDSwd$+txVY!yjYWT*oB3z1Lo1?Q#oOyy4@J1@Sm+MHUXaV9fNK)e$!jIEM0GAuy-'
            'I2Bxg50`tp14o&!$Y+_t5=Hi}8%$cRis@fieVb{jM;-'
            'Zc4eB&O7$+rhDe?@l2tq+`{NDgDB*_Hi%{wruV&xP>?&1he~7xfe4Ncrn*Jk@av9v}4;wMtk>TG0dTjhDl>B3*W-'
            '*I4po=uPaRG6UY1#EZ(-'
            'yF%(XYbrZmk)*5|0acmtc(4Cql1GohrWYpc)Q^|2%tnd%QKZ70cosp=o|=pWkGr!2#waj#w?;4|trlkY?ZZsmZ3f}OK<'
            'qmv9l9S?!p)PLA?b4$_VrLXkn@sbcB-91ORWb`{PQh-'
            'cgVtXXC|_@x@$2BHyOrY>SpXU_9)%?x)hX>2IJi!r@^$b6MJr7XPkcF0nX{HPxY-v!Jh0_#EuH_c-DQ?pYoj(=#z|h`m'
            'MlQ)~n&2r6n8|KEbkUgIT*IeMZILA}B`<A~E*|vabi<#)KQa(Z{PNQiU3Lccm|4ln>Z8^koK=W{W0IEa9A9$Kse_qd`q'
            'e7WcnAXYILKjeR^g9G_*rz{zDd!FkIm=y6(=oonp_@9vFa$FCVhtzE~UcDIo@tM(H%Cp{(C#wNfP^`~&@Yj@1`wZX{cU'
            'Ez7JLKxVz59YbfU@y%s1%s}II47(Ft3EzMv|ZR6rv&w*KRz7CsV`)pQ1v<(%0$7<HTy7<Y=$XK4M=;Z(=qFmspI+p`Y^'
            'jQE>6&A9kR3W#I}jp+x7@tU*m^K-'
            '!19fU3Va3e;rNhtHuawm7(M2yO=uk5bikkl5Tr47RHa0V|(v@1s<jcNOQa&a`!c$r{Dx}Sg6k?IwU~2Qa)!-'
            '_l@wVS63$4y&k*pv>30|2Cy(K6*ld>1LLnuW18Kwfq!fT9^W|~g%1*;mUjsre;vdQJ+hk0Py3ALicaIXF)M-'
            'oTnPL0p2H7kIcDw~eKu!8G#Edqqt}o2Wwxojpi@)t!{S3r&}>;R_N?3$7&1tXndKnP_oE6lTk!=o^gXbCaTn?{yMoq#9'
            'ROy7otS&3+R$uKh+lG@q5I5IoZOU#f-7I4oZWy;YI5XA1&4X|;~;vEs34mQD?|#9N8-'
            '?5quJpV$HD!1EyzLykz4cz{A6ab8$QlqV|E8I_Fq2Y%||KVwnLM>7WfI*<nDwUV=Az&lL=@Xoe345N8yghjMFJyn=M!F'
            '&hC71mCjkIfj4^Uvps9q(d*d?IC`l*+jMXnswee?@#^~IlU*6c^^3;(yW`-{`!7@}ygx)T-'
            'N~KI+4v(c3rf;?xNBx5eOkDakb?u5>+eV5dAak@<IoE<u8m_Gyav#|>9TAepDr-En-'
            '{Hcie*ps$pO37ePN?c431y+5Q?T>5Y4h22v*f|*dc1yFbnU&$|Ji;O|Jsfi#<-'
            '@YaVC8;#h`z=oHLQil*|{N29H49A=yEgKk-'
            '$P{(^A+0hUMBXYO0iLb_Tx^43(YwLCqS(P5lok90tzuOI>eC##aZ10Df%HznL-4%p0-'
            '=6jK;j#8j8lruZ=fHxIU+CSXU2y497EYX&$MaoZ1K-'
            '6T)U7%(!<eu5AoUtPpWBVKzB7`2xIvydQ8Wb>_DR8gKJx51Sq*CNX*EW89YNjNgXoa36onC;nDEe%_+mvSmaFw+yo;x>'
            'UTeo<<8vR5Tk#usyXG#|KYR+0i-'
            '$ArXXn5;ID!j=GuY{~{9vi;1~7Qt5$6uQjDa0)<Fd{V@zlU#I^V90OvJ_bF2)R%CsvhQ`F0#5LJgUB7gg!ABUNbLWXfL'
            'I4sgk?3^JbvVg1I-'
            '^uXI*P_8hO(OqYc(JLpjkJmb2?e_1edff{$&Gr(TMf*YL!VVa(sL4)T{TAa*5~$j*EAZ9g8x$Yh#<||73p<KcVvJ6Vf>'
            'hf~7`@>D{Tdd`+;(!LU*}&3^Cjs}-'
            '>3kSht#0%?)~s8?L2BcXe9g$6lF!+K(#A}iFWb`W`M>Rre=~H#J^FX5rsA&Q>o76tnSLlruJe^*Tmt~sUJyJ_-'
            '?fKcOWlSIx;>BzvAs#9(YJ*HykuHV!q~`fx%rbU~TPC_RNKB&JDQ$(Pwd9uBMlXo<~K~$hg_q>Fz6Z7~GeU%kB(HXXCM'
            'xX*Wi7p&whH{{+n@S+ciYC1AJUsn!#QUInd}P3UUViM>s_lXzcycE8Ieyt(!qiM`Pu6($XZ;$ya$JQ-M*h0&-'
            'YKZtD*jH0`=wqu>&8PO4$Z`fb+4L+?K#B_Xf4`X_kl$dT+XZv1^0^J%TruQf(dTZhY`fg}fRxor6+@A0n9y;}5JXUpMj'
            '@-XTD(hq5%;+D`JHG@*E2p4<n1ahp6?WiI4XWfN%kUL+Sas$Wb?={vb5~b^Npe@Vw67-'
            '4sUOPRfwPpesghIw&4giQtFZiA5$JsRC>ohwgJy%yOh@NK*7mE!_br^v`ZSra^9JsMYlXU?z5gr*Y#+wz7zX0>83}lH^'
            '*gAg*D-'
            'fk85tbC13j;Lvisw<5`*_$*>w3hNUXJIkKAPNhmj6E44wymTYJ*}KPzB=m;)|6R*OepSEK5X@yw0%XHfk06V8y`inDh;'
            'feF?f7=;ou)V51QnSgXKzMM%ro;{0Ix-'
            ';O$s8b}(Z4t51oW^|IDhnTmWswyZdeE2g*Dy_6k6rS*AL|!38B0H?Gpf4pus(4%!pA7wGtLi+ukV3R;h*q|>?p=`VGPy'
            'jm`YvxUqOS)=@73{0lX^=F7><yX<xsPsyT;HrdpRZU6q3KmQ&onSCwYR^TB=d45mYP1Kh2!#a>MsSZUdS%av!sBKQs;i'
            'f`lSnc*nkGmX4_)`JyB=YjIr9x!J}GhE9qg;7SANPI>!SRLpGn+6PEW;cw3)pz9><Jjw9x1*S*MmM3ltQ@<kYY!$XA`O'
            'DWE}*RU7`ms&5s1>>0lTJXu!fgjLV%kZWAP(Sr2F|Uu+0OR4M}!P<9;91tn12}XEksZTP(%*VZYF`PKOz@jK!(iJ(vL<'
            'l-Y;(5hMJg*s48CS(mSmAb9m|!eckV9FrnQ={K91u>B}ra_$4l?$fBeQYbX2QY_tO#-'
            '3lR#yYBf#GPwSVydkIDef49^|RvO`h`v6ea}1E@6cyleb)w!i#Ma2&3Ej#VF6lANrx4C-'
            'eH*fV0K2wbvU};6eieyKF01=Wu~xCMcvXY=*3A{m=Q7!Ms=?R&C!nN`(+w)*HV|spSJ@#RV;(q4|}np)1Sh?2}fa1&lk'
            '|;%`0$s7{tDr6a|Wjzrdl<4k8^Eu$yuO^bH(@bKH@PR>ciC(%A`zjDAHwe^17Pla51GF-ucqteEBXdB80&WgU+khX=A7'
            'QR})o)12r*AbcGAsVM@-=KjEuy=B<pCB^velqE@+Bgb^B&c<_-'
            'u0i1?4K^TLzGTq2Q{X97;VjcG!EbLSv5mLY7)@6dCdTg|G%rj<w{`uPCq}s_nlPS~8G8~=4P{|+_w%s3?^Cphl7T!s3k'
            '+&%1kKNx;CsoF%$*}d!wZJ=i^3T8_~Kry*2zIk{~qgc$0$vBd#<tMon3!6#Xyazel4UwHA~@v*H%Ei0vuHS5uI=KBrEg'
            '-$)Hm#$c$M}ZjP%WSsUM>e{eiJyx5Nk`_9FW*LvWFp62*-?m--65=<QSMdR5VKCbMTz`7sM#akg?;rrPY?DJo*F}8U;?'
            '0WYRVub5xp7~GE?%bV88o{uZUlXB!qXLuT(+H8d#c*i-FgB#)DTsR4mwE7hBD*&E6K$O20_j_9!P21zd+taoT^<-'
            'q=5Ibhf_=`Be8!mVwQ3jUs>X_*>MAn@92*SwScuKrFQJ>+EAjd1By<kYWJ?3Ou{l@N$&f{gY-reNY%HJ2W{g?Iu0H`-'
            'GVnEQ4S5U(Mz7)Rp>C|%_Mz+wr3U=8{XN9zaoFV(2C*gGmV(O}0iJHkq4PsuV#hlI_NZtygy@|@^|>2B?}`pPJi`#a8p'
            '^RtV-%U92?Ew|>;ati;{|q@Wd%`-cVR?|4$&BQ4<Y#l^mFRMrXQ}NCBoZy=Cci~aqPgRIZk1@Q{(7-'
            'xm|eaU=u!n9E=BQli*s@Bv?`!3C4+C*^?e;$w_AgmY#e~p4&acP0mFysc{l`ZydrVtdE51YXcYy-Ya5YP)-'
            's#3TQeKVYjFN?TYHK_u^hy)Tsedcx%9JkUlees(9^kl4W`YeWQzyye@fN<pb%wVzMD&3@l%0#s+l{#FwO0bZgsQs=Krw'
            'GtOubF1R>b%(rG%`-'
            'tbVZF+2Tr5heUxExmW%EbY<rop}M9hl?hU6@`v$sCm*d+A!87|1(r${5`r!lsVgj@O!uaer0;uAGnpS2(F;df8w$<l{j'
            'YE2jiUC*8wsGrKXh>(0Q&5`A#hHD`b9Sq?Vt4KTr9pA9Pu$Jsr1;K(nrIC_yeJJKmlG-'
            '}jgs(<n`%scUcG!>>ncug)*)&7h+k*SasbO<{a#X@pJ7iRM5TnM*TW*o+?$5mZ=vS$K3=<STPqFtv)F`=dvaNv$FJbty'
            'CR%s^V1fdplW6Cu!Th|o~mERJ-ys@~aRDrEh4S-'
            '*}4`F>=H5M1d)6U*Uv8n$cHfxhQ*;MdSlqOpUgElL|jZ07I?Zzlbj=BaakrTnRT#2b0w-'
            'fyn*3!?<^>E)7B}VqiQ`o1o5_G$%vZ=ckW6+>yR6x#PV_6~2_I|-'
            'BA0I(|%Xf47?D~jn3X+)1StlU5Dh}wB)!^P#3h}ZXS-oX<VI?O3m73GRq3j5JhDK_+s)!^UR%gNtBk9{sHf-OWePQcZe'
            'biJVVC_KQ7;8@Z^w(p)kFJF9SsdoO=1<tR;u9zwehvCD3al^`A@*bjMAm7s%@ut`GCuv7FV=<dNM{3v&nm!gVLL#D_mU'
            'P^iRX;N_2kt$cQ7yN!gPPB%PjuflTkR>6YcJufUmu7(DdK}dTRRscAm>&(%W<dd(AZ)3&w8-<-'
            '#2#$oDa9UsM2do^Ww>u?$A9F=FkUqTr!HIUcRnfC1m*sH3{rS5YZ^5Dmw4%N@AD<e|0hegh^|ypC(Q)!?T|`Os8g!kT6'
            'tgyR(p(e`x{4oV%&gbX>1{Q^AD=%G1Bt*0(iu;c(4Jij}z_37C0<yhvpe=G)1pN}WB2Q%?ZEc4;oMO@d=4+B#AGcWzJp'
            'gHq7I1K4Zrry&dhIyu7Z`qkSGcb=FAJPRYt5V@~KXZ&sjfduagP3RoUDVOqhB`wu7=L?3*!BJxgq*B}$mfS3bV(Ng`)='
            'rOS1yXwQD$UiM4YO=y_p{?O0kFP8lrG%J*bZJ#}{k+<FYp=phPPNDyl^!uQmd1d}gp%bd-h-{|>H#JNQO>Ha-'
            '`Og=;$lq15>TPId}_=L)|tjlBXY>26ki)#kI)mZ-4dP4|h;ntj&e=k#Qs6tBiZgSSJKX*o_0FocDoK1@x{d0c56ju+-'
            'QW5q2Sk<LzecFl(f*!V(=>cTtp?Xi_)yTzlUkAN8)G>)~_dx#&GMc@|346U;dS&s=Ci$0FG=`h<ZWI}T{j_Qc9pgOu1?'
            'Rh6*qRK4_Mi(G8M3L2x%tzx(&uQMVBgFPC2h@A(u!V;;h?OU$i=iKr`K<@5xiJFAtgFXitDVSbmEO4LBjCaH#o(ISgLM'
            'd5NcsM&;ku$OM!)OKK3kxPD|;-'
            '$Z@zC!64nHx<jLQ#$|JMGd&<a2{{OEhfAa00{2AGfm60+1z8byHs*&q&Pe9{XZ_pdsf%RN2%lhss6A7K7VavS1?6l1_8'
            '0+~0YK+!XR#|}c^DSYO%N<~A9)h|;6Y{kjp!!)=iG$vL;3d9;Q0tpGPi7Yw+Vo?7D4(NGr&K{$gBqv*#`iFDQ5<H9zJW'
            'bW26v6kQvd7UDW!yq9ftqUCzsu2#{GwXL))>V|9?EXl)hF!ISQ$zitJ7CIiQbr2eS1IMeFohx(NO8;<A0DYd}2Rw8;Z%'
            'PKD!%<hyXZ__%1Hd;}ajI)dsfO2+uI72s%;1iOYVp(7RIIdzi?IrW-HIOFyvfvj5w247Hz>a7dF=vg^Vm^DK5sq{0)Z>'
            'T15vYN`NJ+BJ-'
            '`(s7Z{p{(Om~h;DX(al}_QDOLbRneA0%GuL3F+o@6uO_#fuO|GWa#U1T=cL%I2x}9zgktC@Hmp5Hco&5Z&^6?ekHyMQK'
            'R2eK8o&^Ccve62Qfb~5p*6l(rTlLxM-XX4NFtQ9MN_>lYbiWjc*gnIY7o-zJk3zs^M+c3O<aR0$GZ`sM5o$l-'
            'Ji5c>7hM*YQa36nR1O6h#oN<-kbcF4z#454!@C;TJ~Y-GOOfbWwvYJW65b>2-'
            'AE_(Pm4+qU4?LB5c`xDcb{w4sMt5Snc}0=eH0fKlurSZ&N=2ah+n@R}mXpRT}FtG{Bxn=v@&ld*`%jOUoY^M-'
            '{5w}|R=89Z5%2+6(%P#sf7<86PSSCApD5AB78x8@`Bx<+)@;T{fuzYbKV3GsznAP!fP!`U)FXqWdfbeGp5s7erVX3ON@'
            '=7(KiQ|LJGTJRP!9ge}<%`<6-'
            'H|bdHm<Vx?!#UIBa`1V~a2myohQyCgaL?dA$jzPrgO&GU+zn$CEqg_eS6GXt&Z$JAvJ-?aBG4tU1h-'
            'VwQ$IQxUi7ZQV{c^9nOP50YBX@q!)2U`1Qonv;{+l3;V`o?9xrZg<TToihH2MRU|QBe=ru!@=y%hB1*eu^h;cR=c2Qw_'
            'r3kQO)o?5wyAb-zJ;jgn02dFKf}1bblBHRL!R@d$dgeT{F45fu#kucEr|(M4%IML=O>;DkxNQcnEZ1YS&lU7l90J=qUx'
            '4eEqfy}SnRZuFf~^rJIV<K@!)^5%5$EDioVVT|E^e4bA~^vNGB**1z28(aB77Y*+3$lHwoi$lOcox~3#TiqVyTtmOBh-'
            'l1w)U9A<ugO<ZGJaBI8`xGvFIoJ<~yrXM-'
            '{ALnk<JbPe{ZE+=P|>TvhPYc#Dc7;Wx&VE(T4(1UxGoKvtthxeYaI5LIyWOCu6OAN(M)?~P92B%@T3dDT*3QszHpqfT!'
            '=`>S4l&MIADZ_kWd(soSJYNP@AL623t{z;|nT#(dug5+nIhY=l0>eFb5Fsxa`nWW}#bp~rSz(1_^{*~adHVwPD~W<rJu'
            '0C%J%l{?A_L!7ZGu@*Sr{_)6}jV+ix14Y(|%@gxKm*k%#z*A@e7)aKkG9@rSV0OJhKvp4a>*qt|2f;vTp7j3l-'
            'Pzan{ccpr!Sztk<M0#=|$yfbINL?7uM@Dx1Aw!fpY))yY7|LzQsNc`7dHqJUvvhvM<cAsBLbD}3B>4T?fLl8DE77!|w?'
            'OjOU&8Rosor?=5mfAmZUx9o(kmJEmFVdA><2nW+m$ML1xT^bgYNLE{l_g9%yvAlbd^`N;cA=)B@vg6jF=&=r{aBN|xzB'
            'iU}qVdb|=d}9EWt>36>AZV^Soz^1C-'
            'Ciw6774TWQM(x_^kerW_R(U$4=j)afL_G^8F47W_y6F#co``U6%2=Hym@WKfq6;&fy6sRaA>QgsCO*qPZpCtsRDJM9mq'
            'wFn#7>%yiPi=dW^c&hr7-'
            'XTf{0888;72fZLGb<e=K#ZD098Vj4Q=Y!+bbX;CtMf8r3qZcY~(oxI8VNKl|5`Sf`sOLxnl;>;X6@||nZuUBqEjWv>cB'
            'O#3dOlq5qz=*DPUG1hS?J$)556`tCf_?xLX`v4Ndmb?zEvQ8PPhWLuL(|^`dHMra1X({`A7oG!2h#4O)lhOQoaiu(9Fj'
            'ArbTc=HXWSf*U|6y+#zU9Gkq48Mg}kAFi8^{;pCxxAXs9FOU3JIPW(tPs2ExD!b*<LW4fc!`lB?c_!%94^E5GCavE1WK'
            '1;{K43NpO09~aFdjIx*41Tv5tLDjI;N*F@b#W#x7Js%#`C&{^jKQ6msaSa=5)z+UlX}%MIDY6FZt&>9RPIWl!H4(KQE&'
            'IcgjB$H<5~QYbby#?t|e!Rjd5;qE>8K_i5W9KgeXrKj2N3of7$@99Hax8#X98Rm1;C|*##Dt^F;P$4<STlFNEdJhdl8)'
            '`0PbjIL4L3ksL2f*?*SI>+=H_9#4dc=Nz$C(FC(^zJuN~eNioAJYJYm1bP*3NrkB#^I@$WeehV0QS9}Z3_M*9@=Xm?ZJ'
            'Ht*9w7_E4vX+>0E6cTh-jL%J6$Duj+$pb!QyuWta_`m2|51w7!u)@=L!gT^Z?fjcjC=iD!5liMDDK9Ah#XY<Ea@6uxwU'
            '8xWB14_8FK5>!L?dt!e4F#$h5ZNpPknj~mDynHXYY%b<_{I$AdEHh3?8E9w@y85PfGV%nx_h>Ch7{%|8s%BjZPxew6aT'
            '>*~Y&&Ra>*0>p1usXUIZ;Phj1_d4L?P!EUijwH1uaC(C!r;DBxxg7ou>bJ~xJlg$u4kX)$oAeyMui&U0{L><^r{eU?9I'
            'jcwC7mc_Y-uR7f&N*TB6Dpf;yHKu-Wz==H1>3Z0=3GcEub0Rnj0MAOhE=PRC)41Klq-'
            '1_zzXxAgp#P8xHc<E`mO$j!SOVX}B^f8H5}(<aJ7xrsWyea(`Ob%zl9Y!fl*b)u6-'
            'Hjr34LljZn0dr)c@OWllIB+ePyw?$d?umG4@NfstxQSrVHJaRxmB9`f?@(7JflQko0lSUQ;@8anFl@doXkIx@mbgT~E?'
            'S2%3Mt^JH5J!S^o27^&7jovF&P?m50(W-W7UUXk@gB3)O@;+@}oPW@uO6{lTeHN_ZK-wbh<#dIm@tr?oRYlY=ShV9Nd1'
            's5vh!~g>0rPD%&W6_rL}SH+=+?UabN1>eF<6ZY;#$Idm%DPH$wdMf;#*oZc(?!kI?{L2Y>z9#7~3!xG(z_ik-'
            '0Ke7jdneE`);ivTygEBa@)*i}d<`V7d+vI7&HTZUaDcoGL8DXOftn1l?Y8@ZYhMDU)jhmdnyFLL?-'
            '<M;pUr&6eDPh!gTeRGF6JH6dNJ2mf8vN3Mk%l2q=c9!kJsap%tw>l4S3zNb2Rs<;52tGC@N9Yx?Cn^H<+mr}$5&+#Jh+'
            '6)CoO^#52~S4v;kSI>;)H;@4&C-'
            'yWsz!fbRUfh6KG?0_pL_u;tnah)?c?CZY?tw3DiMO)Q448ll+r=QtSfp$T;98#2Da41e}NOTL^(VD&`M@MR;|3OS;@@N'
            'GC`!BkN2eNMF_6R1bdXiUux1y^xC{Ib2l;pHK6I%)(YPuz}oLUUoV&V738q%&y>sl$4oD3}yti7~rR)3sH5u`cBRRP@l'
            'nRaJGkTt|HVO|e81xe^%Xy`9SJ{y>J!dr2HlpTXf;??|_Gbs#r38t*1Hlj~)LWW?N3B6s>VJ)vPk>inLemGNGb53Z)Qq'
            '!KqVeemSy6{zSdL;D7-fd^HuK>2wVz@A9FC+`i7GUFkotP)07?Z>>FP}=0-'
            '1!azTuqkX4O&Oho1+sHU+yQIsJ#_<;fgR|KM?VR7g#l>p*@u(OGGOz;XwKGmK@d`X4p!%nB0c5igX;yB^!VwFL)XRP&6'
            'uH>Im;M2Zh4B!Zf}RuyrUSNkq)j)4Ken^95BuuDJl*_D405cZt9i-BeLFb68%1t2|{@i)TbMD-'
            'X95z)<tl1WY*)J9a&_9MlF^WedP$0evuuO4PZYv5>|#~!<dcxVQ)~JsN&3cIE*Hkqh5v+W%r_keI79j$fla^S48zIy0b'
            '4Q_oCcK7TAp!g-Smzl0=PT;?Ji%$LD58#w0cvLTf+M#C@Yk(RdX!-'
            'Dg6Lul9z1L+7FAjwbpp@Dk|Q&xiGsjqrg|8J#yS88dx{Lg}x5XyRfB!nj=MI{y~fMTP_IK7vtiv?V`t4g!r>OJ*)Pi>a'
            'O#ShzL}BJB3T+1c^%@}3KLK2XEZ6|T^ech5Q~Oa<1j-UvD$ZPBbMhrVl$gR6D{#LLtMITNyQ-hfD4TVq8x@ay4Vu^DEa'
            'Tn?u9SAu=VL70=VlN{NpjomjlkgryhD)e2BWjV^svrYrx>6dV@>zj%xruQM?3#C@??^0i*N}O{&1C=k02R%7Wh9^7>rd'
            'KjCK_wPW^T%4>d$x~b%Z~x|m&<@zm;_^&{34ex{Dg_$m%^RcNO-'
            '==m8zk5d=_Y9nS2TMFEcHXtshMV&Lh#e|6pQcw*i#0Zj({fKglW%rA=B1B>(+GTxj0`w(mX!cLJ86mbEc7EKWyRo!-'
            'pFnvL{WWhB1em<<(k5nd>K#x3WUq3`8QP`70|*`0p_0u1wTZ$vNn7=8?vi1+NG{7?u^F~peBQ;F>4I7D>|)FK<Od4&#L'
            '^frjDJ)lc>sjtSGy>{5RstG*p^zq#Ead^K<g}zO@MW!C>guW+YNkLC{==yage6&den_EY~Z+<%MD&Di)ZF)jwCPfh6oe'
            'X?bti%Dg)xo*i0gdXb(0XGL_S1NR9?^TOy(~TOpkW0*{2Yl1a+RQ;p@$tfmUyH)7yBeH!p=4?XhKpZUhSGnw!~+X?4J7'
            '2*v){9<Xt2`a%(VYvM#P=)N$LSCQ|hAG@-jo&|~r%POoLoCH(3ISf(wW{qK~)<2%P-^{kWlps5=&-'
            '6%G??!Z|Z^(5}iI*fTg3i=#NfO|)+NXD0NteD@0G!DK`grP=c_N7bI@I@*Vl^4;&8YZwTKo_dV6yr4SDAWqwf2@?;Btj'
            'i;$nf2Q1MhEyi`rrMtg?wtyS4!5T-'
            'yTHb8>O($w2BY<Uqg+P0$Wb#_DIU@z?HNwD0IT?6a!^M#<+xWa$W)?z@&Fnsrk&UT!TKv#;q&lT#2iTtFfkl-WE|Nm6r'
            '{(ZmHB5aT}ux)~n9+}DxR{~CwB|8xX5l4dwOxeG2^vmDm%RHmwPH^K9oY8Y>(PWyJPg>zqKL-'
            ')6GuxZd540=<H_m^g%;nON~{T72;23#Rn=}$X~&w<+PQQ$njNEAmTz~h~M;yqERXvEAP^wx<nWU{ZWNbSu7&PxL!-'
            '08jqVt!sEwbmN&dF3H2eR~so^ps(0cJILKFDr4l;RYx=bcjYPog#A<yI{e<NReRUTF}u*1er&pFx2uM_IkCSawnOI9$l'
            ')#Me-+*b?k<bEA-&<meV-PhT!fKt6?U;6tUBA9HzYkg{N&q;d9b)nErj>-'
            'n;`Z?K83ABZDJ4zXtz(T)4V60TlbIf{yAis>scR)ETO@XofCCuC0L~TIV3}!3<9J%sr?UU<%pimg1##dUWKz{^V0?94g'
            'f|l93wAA%B4$cFpibUQB1K_!0;Fx->E-'
            '@eN$AT?x>rNaQO&i~2f80{u_}+0%6Jt>;MSd*UE&t{#Qyd*;C18~bsA(|4+ML}0!AZaI8zUX0U!<iiH@SGZjC6b>h)qg'
            'vxu%q!}RTIWuJh1V(=K)2$>2Vq#DumfpiB}}>Wi_`@4L%F_ZvEkKPh~0A-'
            'nsYl~qT373tM#?8YP1aYURTJu^J_1tcS#@|@#o6H$`HElyG-vsjm1%-'
            'DKOik3;bGAPn&nmAiFy&WABNFsAbp<&~{G5$&=I}F|`M7owx-*72HMd7veqXt11Y~(L`@&AF{<D4^tbmkhkz9-MDias-'
            '<qA%b#4pN1Z1?|D0~*SKni(sB)k7%aJ2~srzuz>tOit?KTd!ONM#RRB^LG9G-'
            'pK2f{vRqM@8GDy!9__TYRpA%`$C;y7NLorIlwPl2h=<DtS~BWB+Sg14eQFjKGA`f*_btQ5~B_j8A#lj2_M50iJ1*L~K*'
            '!=xF67jXt0419<q6GP}AE=H|XMg2uJxV5w=aP`CJG=4WI9laQ@WMn}}FE3I#y@7O_lml<R@$tHM88#*@r*mS{Vg31HoP'
            '6UToxE)(c6i_c#|FfZ=TZK6`^j94x8DYP%o8y9R{*(K;sOIF?8O2VU9cSxh(#OU<EaUuFg{I$1LhPVE(xT2B6i~1RR`('
            'R(}`F)?*eT~Zp6>>3J^Q^09Jex5YI1qaJ<JgDDG{7F_slDcxe>uh)ATfZ8hlGotw})?Et=LSc_quw^Ize3M(cAqvehQ^'
            '1MC+x^K$IO--'
            '}VH1`!5Z1$RT?!SmLH8K)ro+=?DMstyy@tC@XC4*s7Jq{T@l)Nh}!O*o_)F~Iw+u2??z0Y^Lu_~6Hu?htP!_~Ap;Wo)B'
            'ID*X`Zh^Iz3Q78IjAyMDf@X9*%4BVZ9)oJ2&ZtP#JgJ(#srf2i>yE*4Lwi)&K7b6`GZCMZ-'
            '6Q5ZB4J<Q4pdWL1AQER;@7kTkQJ<f&z{MUh_aj3cMcARn8(q0yHgy-%-'
            '%!$pIQlJFF)XjMfZsOiUS<RcrS;4LK6mbI874gti=O1=i#VOp1IUv7mk-ZfTgqZsbirZo$)adc^&jIS4IwWd+&$Qr$nI'
            'Ylnh0I4^iLxKAAu35ajP1j>@+_K~P8`boP1(rIlykDsvc4j;|$2CW|2X&^CH5?*;UAp@cIp8Qi8g;kb#Hpkns`a;(!!$'
            'e!B^lFH-Z(y@(bYH$tSSXGms7k9#_Tw78*yBl2|-3J3}-'
            'im6{Jh9PyKN_4L4o)v>AnC;}x>v^v>Tg6tb?IB{1@><tBsm4+hH=o_sfc(UJBWIj8BkX!q8h(s*_3(lFg&Xx6ErNFj<9'
            'aVq7?&0&%F~!Df6714!i>qq!)9&{|SugT>=vqz9ae?4~c=&W{`aE=oh;NZX_Lqpus8dao!MWe{?Hm+@YM^8&m1xyWgqP'
            '%mSD*K?Cn(=fI<P?l?facg!vx1i9z)#ryQF<S1tkI_~blJhI57cw-wj3Z3w($3XJ=i5yfeyGw`M_Xa0)#yjIYaA03gdV'
            'b+$dh2vQIJ$5K)F18u(H;?C5O)}jn-8E~Y$%>}Gbb~$BSlW)vxi>BR>=Rl2_5c)!&hbq?Qv-bD#sj0bx|U$H}|F!cdNp'
            'riH^8TWfxAoI|xQ~st4b6Ezq}mLA1zD9L~9g6=&3NP~3Z5d3`(RZB2v=tG9AOU+d%6Q8$36ERP=@UEyK;eDWwh5{A~k0'
            'iPaQz~~KwTW$+stkWf|{@w`tTz$YSCkrFRd$z5U<3RW220XO52)m!(kJX85A!76(ob=Wc``+N=>rTDUH_{oOVJf~*&gJ'
            'ygPXlhvKn(QX#aUkF2m?wAQLf)m6cw3Z%~BmQG1(r@Kf6i|T_=J|<_|=#Fzi^;2^=jr%mTGN*c|MF^)ouLKd0XY>o4Az'
            'pSFoM_udDJbM~9d?#_nwu^F)QdlEc}%R{F*C&|18CTM@=Iq5U%JeqY?fWEtuaMSP-'
            'x|1G;1v@){z#yNDRGkA!oma!9x3!`U&*szP`7X56&2-'
            '#y_biBVu7i_a4jFqVo$Od^P1nU|;`>Dhi59y7*Ce=NV9_p2zVI5l+D*sWVU8qZ<t(TzNCn;g?l5=f1&AI|kDnGsp}GA)'
            '=+|jCT7BA&s*bs!dT5X6;rg}k$onB;(lPLBUPNq%M8FH;DXNQ3gu?1?<h}4BC^>zD39(1XtkX^4y&{*)O}zujWfw?R;7'
            '5*vNd%qU=_1N@F~{QyE9l0lLU_D50(lvQgu7K2v<<BxZOmxmw^oj1^wI_Y<B!R;N4IE=>}@DtAP4WX&Eax77ZZba)9br'
            'BgW`cWQm3s?7hhsg%O;fWyOaoBHb;Wt8V|79BF_dUZ=>Jk;$ih5D_FJkws`tHOQtROMZL_o!U?_g`1rsudh4hGejPc9R'
            '7}mFy*wR}J+vJagfDUZjx6Y~^g3Rd6v=twy%pjt`(b2VF%~A9g45+Z$k?%femT4cuh@5G@=jf$1^GH8dS-'
            '7pc>5?xci)Y%372uIWe)Ti%*W>)oT*XyVUhkwIY|4Ehe|_rab{O_wD-tHxfLhjK$k*ld3ZCtv9`s4d3mttR5o<>+=b&V'
            '*I>Z#ezZ?hHGWgs2K*c8a8a%RIX1gQGw$t%`>T%Nub$=9+_w(T^gTnu>_3rhmyBr~#{_y!I|=gpO(CMx2hXp{MXgcyNa'
            'vdGB0Gahbhxqs>-uje-^xW~!;r(oiGIa-a=N(EwF<Vb`bFm?45RP1b)&|-'
            '!7%r78q9lGL*8W`;jB9{5u?lJfy?tl@bmf*teVx09U<t1X4=P57$pnCp9{r#&cl$`6QTCqI-'
            '0+2v*@f}XR>dEI&$5&;jxGxBF}MopzV}`T?JENhe-|Hbkz!VURQ!fk~(U-'
            'j6rYl88>v#1zfpI5d<YG!Fp*FneCbf+wKp>f^%uODj=5jTay9(eoiGH*hmcb8V9<=7{aUxC%UT7=*@K;&^zqS*?40G%-'
            '7B5T=uFc>1A(4wtk)p0gJn_9l9oi&g{=9wBL>aao_2;%Q2|jbE<XL0tGk{RDlneU7WgDSL^EgdolCPFw8ZN7IpARf!ia'
            '_falh(%)PEzR-'
            '<E{k+q$#QP^{WSpIy0+ObX4{zxurkKZmnCnw`EmnvAfDwWoB+5pOz8|ajG6zkV4#`vhoWXkTRnErAtvHvWGepj^7qi!8'
            'lJb7QFX<LD%r*FXJuo^5Hd=B-_xMG!W3VewYp_{=GSim?yb8lU+kNakw`N4^v1}CV_=i#WFM5roCq#dJr;*ObK=&u@y-'
            'M<#0(Y-=4@#<O3S-u+XjrPLj+4n^&LII-KV_+Dl18G^8vFn~e%s@E>I9K3{CI>Z8_r+1rK2;1051&NuA>-'
            ')iPadM;Z7gb@Ri#%W37sTYj?3fU5$&1t;o4L;;PRuW0?wwLk02e=yaT!OZD{?J>oh;$D#wjRV77Q~dS1N}j;`8_y)xfn'
            'Pi_U7Tk(e8^oXS)VfRVKfpBP4jpM{toCD?gdtuxUJ<N_sMpb(bWR4vw`r<cK<kfv6`EDqrd(E5Rc|{_apE{2h>lVU|>k'
            'TCM^9~w`9WmN|Bd2nCJ}wCkLF1<}SarD=cZToA)9V|^#$&6oRCW@a|4{~(NxiW)&zr;PV2MhGiC`5{f%kO*%<jJ+{RT9'
            'M{O}+-P+pG#$8OLO7Ww$8_5*HOXo!{7%c;DFDlqHcVrk9<9Ja;_<SLY?l|mZXc4j*IFG>YfrC&t1E)7qG8N#9p1H3-'
            'g4oUbq!p4Tc#Zv?D)Ve76w$2Cgv{s-+e<vb4zW_c~ABXz!FEM``Qm-'
            '@H@q<?dt^RJ0rokzg^nESF@0dhP?Mkq@`2sw*Hp4BSmf>EZGHPA9f~LAxa7Dj)z?!Gwof$oFn$l&keP#%s6tsz&dLE5l'
            'S%q4U^6-'
            'UYKUh7|2`?vUf=wS!h*k>+Yo7vWoFzUtZ@Wr!^b2Ux2Nq%{DL|CBAKFaW51V`jV&{}7<hyU+Jk(CZs9PoQcAq}n?LGl>'
            'U7Nu0oCa<gCjjNMOE9+~9-'
            'q9jgt^mVL`|Re(c!vNQQ?&c)}EojeBMO$3RYuK{%vx+<7w*nJs#FfnvD})?ZQjJVCB<$qxGddsc=1LBs8qi$L~`<!TIq'
            '^aov?TTH2&ck7wqQ+d-'
            '+25@QZGeHB@ylbccXqaS?WEWnOW0<d)6XY!u+0xFmH$DVUfk%%YRFzEbsdf@Cj`h69`xMFo|O4yH@&WYgW-AoUUe<RW<'
            '<l!UL64HH!0(*Xz0GWzFSo0+pP8EDAId!%e%H&ER<paWqjwzTn%on_I11|B(!ebx2;MfZdD9MRNZq*n(ytp2W-'
            'LJxk?ceCbZS&zq*>=dv-'
            '9|D^CxVReHBR*9Q24;h1A~$ZxcwxEb70m+T2o_>I}GnYp{*U9OB{$l4jqIygAZ{YCO#$OJv8aJz!NmvOixr^84oJ(5qe'
            'Z9!=w}A;f3!JvdUlpl8-N;uIN0ObzlZQSZ7Oov-IG6r80cEcosB*mt$9NGfcd;4s)EYa_$xmWPUXqgMzTZIOfT5G@tGT'
            '*8@htg4CXP`Pe>uTx10+BM;I*o;_Zl7%LjOC>?IhyGAqm8AF!V9+<;<OXK7D#7gHSbT`_IbC1@;z6EDM$>JmD`2huVIk'
            '|~uEL32?fQ1z+-_RkCUt{Xh%`he79iCV^AI@=yz^velbmY_y?7qH-'
            'Atq9R&3NSpzQH4yp^h71@`qfQXIcR92e)I^jU#a2DWz-jZBe}_0tclkkdY@2z_W{cQC?OHW3OxCRkav6r&Pu{fs=7^`B'
            'QQ`{{j(&>Og}|5q`A{$Hno9@cQgDSXq$`zizEVCz}#bu80N29vS5P^-H+*#dFGeu>dgZD{OiGj2yfD`2Vr;-'
            'v3y?;s1ZgEF(!uG*l=hrO0(2$ZQ%R8j==;2qjcD*?VM@mA%7t9+wq~ga$2@5lWQiONy`e=Rf$qZ|^_gdfu+%d7ii9ali'
            'lYc#^~O-'
            'X<FK7^~MR0Na02%!~C^(2;TnPi6LFvCtP1654>A>R0f``CsIY?JLHY?{kb3_b0%sA{1Bs;zck1NR+;H8}~2C#V4+^=vw'
            '-lZW^(NIbL_%@mCW~Qynm-tQkM36o5eFB-'
            'x=APyB|X;mt!M;QNpWF{XJ$CCLt_@0erJwpL(>#NdPb{P58)1PY|yz<1$j`0`U$)nCh*)bp*wOV@Z*f0t;I57~c6MfL|'
            'Q6%j&Vm(`HhR0p5WYQPb(5eTW|fs89k;B&+WpQ~7b*!<pc({5#=#zvg_BTTk1M(PaThf{^GJYZ7m1?TcBpmF_o!m-'
            'T{=-M~*ry@Hn%lty}ijc{C8-'
            '%@0uRvbpFnHE9;<fjkP*}JMZ&nNAXWM7s7heu9RD)nZ#)X_`TgYli`%WI8Zl>vSNx1H#8~u>{n>KaEpgX@0T-'
            '=#~J1$*=>@Vv`=>2Vw{)Y*BCyv3w^M&-'
            'D!4P(u?qT*GRK;AbKk)Cra2%8MAv<LrkpX26mO0N#=wEOPx1)q=gTov29ISw`n>`R_avD~*Ibya)8uQv0UMRFOf}Icin'
            'Jeu-fv<KBgzJ|u<VM?Ih1pXoo&20JXAuX^*|NAYag44}{(%oYwZZA<3o>k&fStBHEHc+br?e%}Ji!w`e}963>4WGX#>a'
            'X#=mVv17r^U9FCa^Uk7c=aIkk~`25t%So~io?8v_zBjnfq`Z&HNBpQoT&VmrPypM}cdc>3HS9EDjd%-k7?S62%`!TLL-'
            '<?S@oe-4G}Ej^&4-H7e`SHb$1op}8D6XM$No^&1>q|FA1hwP6qz<n*s7Bqsdz-}DwjKInZ@33uUIf^V=u6pFR88Ga-'
            '(J$;3EFx#&(gI%?tT+o#U!+lW^=!21n9r15Cg3P335i#hqFjz0-'
            'ucO=`aCTY^{!o}%iD}FHCrF8PZJDVIR+tbxK)RrZbbJR?=i1Q5jT~-q-'
            'UR3Valm8M#Ot|#JCyMe|weq+U|o_=EE?iwv4%*--'
            '@x`yaJap*jcHE`RId%xA66I1+@3OMSeu_Lw@N=8r<Rn@7KK{Z!XDVre`tQME;=xR?D%C-'
            '3kvrh=cn+MR+VDg33wf5x$UUIG3ePB}}j3rU#o)N9s9q*JKDhKkf&vVw+&k`6qP0?|JlJ{Fh{`bVVVzLUhYlLn#$Ca<B'
            '3fKIHMkXRjRa0gpB+hkb_YJ`TWfWC81su`<l8KM(o$&tk8BIk_Y*K^>nSfrpCsk$?C6Y8_0)YiDXvaq2FPn0+Ctg2(F='
            'cl9CL<5P5~Z#~#8^oLHLJaC>>Kq>vBnDFmEQqMEZEW2$^_yzNkGmR5_TXRWJf&+Xrd_Z)a?}GkEPS$YWJ+%0fjlJ3H&}'
            '{8d^0DM7e%KvN*PbcIKR?glj+?`n9=jHDc6`Czw}UDB?k(W3$_f4(`i`lmL!f42Ifzxepuib|@kNpNGsFhGABtng-fS4'
            '!5I)ZUb$Dlq9GTHeg3Wz*amIa!7*J<?@?-'
            'wqh0YSe4^epQtv|lh5`y*nH35!3##7mL(3CKM)6R%Fo8~$A`CBZJd<ddC&loEt7vV=%8jfbOP$%~;Lfd+>R=o#fMHuM#'
            'Es(6z&4!wo{}}R#Pw@QVn;_BmL*={hBpIwsgFiPjvEjQu?mSjQ4){Ca9C1PS=13gWUks_AgCN2AA~d)yM!#E$sJeU?yw'
            'ORf0`DF}k}U&$<dcDJl*Q~zPH<&;ImYSE`%$YT<oSdXg;hD|{K*Oo%_x4qslwEp3xnl_lPKMrh!aYG>8-'
            'jkk|VPZr{Y5Cmks>T6dH$58XfUl!!BsPQiD&p{YjP@2UfXcgHlN&ZY_vKXL%no?BW9wn-f6)6F-'
            '$R2xdOfsisWfyX2StX55h!2!k^9WGGV~Zp6!j`PzKgcOwjcFLa`w+nT{|?FYD&;R@W|dGpNSihnl;plD+eLq~~?Zd~yk'
            'O%77bZxsNJrCxB*-;&m}+ma85((!sv6s)@a03+;JU?8grd-'
            'M;1qh1_rwf{tQ7X`yt<xk)yRSWh*=~y6b1ibr$@DRh31Y0Dd^F3ao9UFx!EyGd6As_Ud?66*z67GQ#NE|69cS#`RmP*p'
            'VTX*1tX$gLtxQ!P!TxDonkAuj7DLNC91>Wqpp;IRg-Vckb?x^jei6<NA8Oxg>Zg&UV(}(CcfyeYD{{i?~at>$bR?@ijF'
            'F;Bs5LUh)LJh7)B4@7;*{uQez~Vl9Y4w5Z7<!NU=Mqtw#KWVjkMQ(BChqOO2hu!C;G~Zz{x`pKHO9Dbn{f_2u#TjUHzW'
            'evR094`w#8-3*Ue{`FZhihm00dcJAJP}==L+9Hr@gc$ZM$mdm9>$9m18uk3n+P5!~9*KpI5-'
            'QS#zN$k(}rR_8)MBX1isttkg?;soQ9eFstO^TeZ4Jg_R^E}T{op~sH~VdY7H(A_m~=dBLxF`1w<<ri@8cs=;ZKY%LZB&'
            'xkwA4Dx8p=otI%70b?bC@EJ*Vs{QYh^g`F&dY-'
            'H8EGW8Kd{5?__RE0v3l`;u+yL^oW=rgjjJw3CN+ugI~=5_}2nI*Iv3dHy`Fqd5LBACp1HOsOc2P?_!U!Y?B?CK6M*Dbz'
            'h=tANtYMEEk?|Wk8x^DhZn)P*U#$8w31F`<h;u8z_W@;y$=euOG{o`a`$e4y=`ajhlM>$%T~jRCDD{9CYHr-'
            '<@UftFfC5mCmz5j0^5DUk$&d+wscD-wZZ?PwI5}5hzr6;3oxZcs<|(>z7mdgwF<&-IdYq?J($-'
            'rxSsZ_w?_nGE9iHMYb~)=sBCIGQ3wF!i!??cnAlx@n9b=91g}o*?JJQOa>#KCZxr6jHA8bb+b#FF?ahfynjkd<($<8W='
            '#1hJaZxkjlUPbi`A~=wtf$MaFbH8yb=ISN@etWfh@$?hcl(F%A#YSGde95Cm~L+iEXPgb+S5y0k&)Keg0yyV|65KwCh2'
            '}=Vp3HH-J32u@#PAE(eJOJLJiqq?_L_#mEy%q=;Xfsl)jQuj>@zV;2<~+3yWLi<)*9|9g$?JDpJMt0dO-dciB*P=+AeE'
            'T~R6V^-'
            '`O(P^xQAFJ9J)kZC_#dAHr;n+!JKV4<I9oK;+C)CNE_RsLZ{4sX&KEjB+>x3((8~v9&CvM@zFt<kz60O&x(e)Vg`Y}x0'
            '^waQ-<0E{_cNZEt594#EIclRfh<2|s;I8c*d>tBz8RBBB2hH-'
            'V@u9Qu`5!N<q)38HJvss}I*ZBqM{m)aCgCT8XDB9`iberJaQWU1s9>Z)X2CF-wv&f%@g^YnssxO#zaiZTo56Q$(Qf%YU'
            'BJlVg~65}938I3HZ^xfpx<U3S-'
            'k*^eS09ibUXC_dP17o2jFCKGX1M9g9kP`09liWhO%*(pS6>zm+DNejpbo4vz#1w9*v#h_wo2oXWYN30o+q&=-'
            'T}+@WtjC<|PX&T*^8LCdv&M`J@^OroB*Mmp*9j`HtBtyWz2R5DFG~VPwH~D)uXvG`@&uxZdTWl9Fy{e)kg|OE$x7*R2@'
            'u>y9<1BC7u}OOZYI06C$YPJa8Rqu0A@P`s3Z1ES?vs9wToo;5^kvptx$gGHkw74ejlH<pj=0<+20M3?U|-'
            'r&|J=ZhnO?b|6_VtNmSf<NFzg;0|9HmJ68b0x|aW#K?VFnzZ=0zIu=Q2d||uq(VkzT&OGQTmI-'
            'p56w_S2W?3v5jO*d46yB^peA92W{sH2*W6k99=8}oBW(nYmpE<k6_|qO?|)@9dPl#bX+lh5*+)2p?Gl)vL#DWp^lTNm$'
            'U-(3U^WINjJD+F9!ZmkMVWy3Vh|fRF$h%6)h~zfkk!@aE$xG(w24DH@*U2jGE#=co0aNs-XLJb~Jb=h`Z17vxct@Vphv'
            'q7SqWRAAR+K%ItJZKUzqR1wE`=q<6P&%&!1r6|%@#xp*Sc*#{^0aly(ON!)mCIartWkYbPh^Q`%jt|`_c?^|>+?;c^<z'
            'fK^r?w`qI*;yL(>OU}9k^vh{L{-'
            'fK2Pr+Yfc1}uhh_b{k4D~qM!8qXLiD!*Tv@4)i?@a`EnV$Vl+&Lck$6C7mmdIm$3dJt9fxZ!=R%>c1jOwR1?{L=V7a^_'
            'uYIe*B*ztvRxiXyOm%dPtEGm%QDD0)3YmGc(0*A6`7O$6e&k`u8!UxK-'
            '*$j`k~w@mUPy}AW5H|5Y0BTS6^$y5XdgD9kYp_u)*9pQL@$*Wu@O*Jp{1hKI)Q~X@$-'
            '3b0RDM;g7bzW6q57k>s2X4eY_Hzt#{E<2NqLz^ea{H9mIk)QP6w;7UP+ZKiT$zmn3A3<Jr1!hN0{fsylz6v$xK{3;A!%'
            'M&l50`!kJQmqdZC$(#RO9K*!o6C__w08PzW+!)vZn?oDv!Hzv}Yi~I1U#SC!Nh}z0^@6us5UeudVEqd9g1D?`EPJPpk;'
            'js0#_!|I-U?opi9|Tsur)JEHCpMer9R4ro-'
            '9~86GpG*j=+YAYsjUj3@`2p!jlc>;5bhZ8287+iV%O$E!G9@Lo2cSbr^Q^tKgO9UxfdU41D7WCxb1r5P!-SQq-'
            'm9bLzB;^j%N9?vM+W(e(_4W14u1l?GvEqbMUSNVj}wW`wyo($yLicZ@}n2kj;(`@UMmz_p#mm^i|QI%)K}V?~tJ`{?_#'
            '!C3L51|tp-^w@a@XU%l~U;jY)-?Qcg+y3W+K>z>x2MYfO{ey-'
            '4ad<(%1>`$eIBNe2N5#%V++}0*vrWMV?FUgy@HXT~SJ9fpRWzyVAkAs#g+RK2I6hbd>E|*q&HWX_Y-tnC58{M&lZ7z6G'
            'mx=u?{n0u-3ku8-r)PHkH&2FK{-'
            'ibc*3^>l(x#_c)10ACU6hVC9Q)0nLp*@y(9md{>AqHgZXcp7C|ZRCwT1O2{xHb2vn(q9fxe0hlW{Hkvc+gc`@FQtU}(7'
            ')!=$O6Mx#@0p$svx{X<l*qNP&^#d%rxX%)*5<b#4x8)!x6ojQJhe?8lG<I`U!6#J%+@bA5$V?Sf$TKlvYXYb>OCtB)Ac'
            ')C(KxCGPLdv3*$k4kC;bOtq_(}^$X7uo2_+R=l)|U{|SZENMfO`^Au>PVwY!qULP7^OG%c_T}+;aT8#f$NxH4&W*St<s'
            '>7jW+PEJJWm6<=ftpk{Ifb16wB+qyI0^4Ggm$!Hs1*yfCT3r4}}{Vx)Df*+=hdolmTJN$6*ANlX^Dth8rajkvDFcwF(;'
            'bu1;lB@&x>sSL-E9ZfNraAg=`ZH0=i-O%>+hCb@Id~p)fa;MPNE7P;k#kz8C@D$D7TBO>?^#sw4#JFIdJxX#h0-'
            'heSWV7Wxc8<r={Z&o!kbw*CFYMRWwluFcNYu`2T+fY`8=?RMjeZdDAeizoqugn{E#(XuHhh3FK(iSoILXr9~-'
            'm^he3<*1Z`ntQk_9ta7@d_n7^yg-'
            '>r%6zcvZ`*9d|@LIFLldXet=8^L%zejd^fZN!yL*NOdsJJ|Ac3`WEjt9l;1O|Oo>2J=xJ{Mz7!v3WnCR`&><ld7ca)jq'
            '+5K|L`WTS8{{MAND?3CMDoRk?A?1NJ+H&vV~jI`qC3CN92(mHjc$AvJ+7ozBAbMT)S6qX3uL@xcM&j#C;-'
            'pyz2H{rXD|=j<FHVWAg5#4Vf}pP%h=0PF~@rK-'
            'u6<aW(R5cJ+gY{S>7wEejW#$NkiTJbHd8wfz&Qi8l2*jRpgdLS~>3cMn`@W;3nx1GO&8_s(|gMdA}bKXlw))~Odfjg)s'
            'k_ao`Y`||Va-'
            'g{VA*1hp5{y^=W6Z=q$4z&8!TPKn9Ty!X8zu)~e}+9e_uiyixsw?qqWySb@j_OI`%0Ff(;8}4&<q<^7SK1VRiNrvE~MY'
            'Ng=I>%(8&IrbX+e7N&O@=`O`smGDC1|$Q7G3qw!an57W0>n020$lXYxt4ZeOVO?qm+(S!f0=Y7!=7`|IE>+%({e&0)Kd'
            'fS;yk5<3|t5~oWb3@$?1(0eJgu#3T<mdD2WT>qkFLa*4oK#8HNT?UigAI&t`2g-'
            'qd|}xQYv9q7gpXc6@L(_qPoLHVXTxRKm0ykC3U$;}W8Td!1(RJ-g-'
            '|eQhUcm?>E@mw<XgUoWo3N`Cszm4@QkOp>=}|D$N8B>vrf<zw;q#Rm2lbD)7Tx}PZB@aL%`Yu{3^A82!FT3mclW3*vtl'
            'JSNMVSb2gSh@E?o`mO}QC|L{GI;M={SP~@5f(vza3eNPVDR!X4v1<N6nvxjyH#jCJK{==*zCy3p2D1PsV!#1C6FvvJc4'
            'ma*Y?qAtx{UitY9{*5TGb;wF`=UT4N(8osy~Ek{*J`!$RpBQi31yi7LE;u8xZxdvg^!PsU)e73o4&$&rTcWKw3wDY@>N'
            '+|H@Mr!_&RAB6T<0#OI6Q4dIW*1dGN^kU0B>@gF=Ba*fqTbJg$$RE*qjvswNyi5`$}EA{kpxpMz72`)NpL1TAe01s&NV'
            'P>`UBPXoJ2{T`$V_cif@@)@+hoQ{dBb7A?;91{QFI+*@72L%=zZX?n7B|wUf9S)$yr*m*o<u<fcEJyQ|NAMH7C<vrifK'
            '%>HbUu3u^xgfj>V7q<m)Bsr=NQ>@D<7IfLa}Xc0ETJ^!)D<cP$?KhV@?LS@p2j1Y4u?u^kMc+1t^JZfmn?MSiN>P_Ib5'
            '~Ppdw9r}^P;Ut3UG&kO3&#<<qT75Ja!;_usRs;d&BvHuie+LsWOJX2q2&!~sS4Z*N9=nARPUyM^7HPrM-'
            'ICDPhLGjOFSQ)pJ)fCo-'
            'UroKqi8EdxD|H&TNe`fZBL{Qou~4XgR81A#U*OE{I0pZ_6mUx7U|A`9;(FeF_+>T`r@xAV*;*UiDsDg}#K$nVwFTd-ti'
            'idn_rRe+m~Q_uOa5qCVoQBL^&2MOv6YQA%lHqsDsH0YEh@Ow`5UvhCXgsIZxd&y5GXf%OzxUILs7TKxZG|rOT>Bs4)s#'
            'pb&nr*1^#1R{>-'
            'k*3`_!xkrIlt%E&i0h!@s)z^{GbFz*Z`x4D}r>dmMGTb=}Oj~yzX|Fq%>$#h)rO`z#O0ba7V1=YYrsN(v8D_48sm-'
            'G!_bh#Dn%qN-pqp8$LB$|lWWueRcmt=4ak^@U5z&4}`9A$=ZjJu1b9tp#_pSc+Snt^FTlHjT`OQROVB42PNo-MM$GZTf'
            'lM_8lo!S?%De%ug`NBN_{R4}em{YrzBMZo-9F1f|L3Y8Oz%zv$`p}R8)*f#Xj)4Yjj63Ig&j;%q_)LOV!JPOAy-'
            '=*7!O5xJ^P#pTYjQ+g)lU&q}MfdzVm`s0!mK)vaOxz0CUa^Rn3I2olE9V%b;1={h<H1i7n=!sh0zBX-cAzuV%eun7TPe'
            '8ZL@7ka_QQGpOVGQl4YdY~aWG34<ijUm&esk_J}&^ZOUC%XQ&4q_mnUu%@kZBzWB8{%6;8hug|)X%l85=jprP4M)-'
            '5wYKjVeaI%I|C<KKhF)MB{!sSf$-x=G{n)$nm2JM15Cf|NFKOw8&5-qUSNem4Q!{wI)%ue*phme-KO(=vEt_!4PIv&98'
            'v>8N9S6<6#b%-'
            'aX%sKUb<TC)E%U9zH+!C$!pIW4Z?M%il2mOX?5LhjIb@Dw~%4+hT6bX3(VptTb$c$lLP?*`9OsbU5`?B&5tPX~$MKT|N'
            '4ZX$}p4wz%@3Ud3psIHF>7$0ImvuH0&>V-'
            '1{^j&}%$blz#8R&7W0!}|Hq}j1&@SS7>UNchz+5Awn*|#3gN2g)z#sJ(^QcFW#_`_c(HmZNE2sTWs;ztyOrytEQWjGe!'
            'b+5s&>7QifNQ+9#ggmLx;>N;)9@Kdp4|-qT;o2@sR14ezUJ)_4SuGaMIjb=_OO(KF|7kSwQ^6M#Y*=}>h$-'
            '|^8NRU}#?Wp2tfEC*a3%w>s;CpC&VQ$tZ9*VF9?blgsYC0oHG&Pp9WJlXMW4n|=7HT3D6_Dem|kq5nqE`1ed_`gFwudB'
            'n$dOZ#YJ&mTadM)j`lmQzzvbx@b^d{RFV>M_rD*s=K(uYZ1rsj+Uts+3VLbR(N-'
            'w6tihCP?U?hf70$3n;Ac~Una6zK;9MVl5H;_*KeT|UC<l65Me*_aVLTKti8+yh5ZwWU$8#C>X_wI=p#!i@0^n19ChWl!'
            'Xjf~2^H)>xzwceRh4(ca)MjAhtKZ16Ed==04&aXiHH@O*JIHQ(2s$IPal1tvlj}t|P&`O}uDDN&znP-'
            '{C>v&FmxC+MZ_rz~1}`K9(T5!(kYait4YGyE+(UrF`R?c_q>j-'
            '}UTFFC2ON*^#CYAi3<rs=cz2%$Shm^_muIcanY=J^$Xgz+HZ8;EpjqbM>k*j3$bp-'
            'a#b|!Wlx#h6fk>}yr*AK7LVdyvBd0bUaxa@<#^GwZKF)y>r+d)dG{IPKULIUbJ0N~;JMz7H2g2LC=$nP<__Xp8R$G=r^'
            '@dF_wN}5*dT}wFKQKfMpRR%AWGnocm5v9}J8-'
            '{j7A}6f6XZT`V8$AZ!qaQ>Jo%)AUdTCs?@zB|{1#RMT$B#?hkZy<J{QsC2%+s8j?KHBG<4Sc0slEm<aWwJ{>9rtzrzE}'
            'L%QhG%YJCJbCMZUG0z?Tb@=01Exvy90Vl&2BZt^)lsl6KXXm+kws0ei%JsuF!E`tj--B1A8^|PgBstNhO>nI_>{L&M)j'
            'iwb`TV)%r}yBh@AuJMG?unh1%sH{8hSn_1hT>_X?MYwx)CE!JQk%4=?BMYtriz_cBNtQT~B7@<{na%z(%bipEH6cWSK8'
            'ogXw@y3GI7(9U`uzLX>GAJo^2L{xDafBj;pDv5XfUsnwxpvg*jvM<RH$=PlaW%{$6I6)4=)jLO_;AlTA{%hRjL+KUXdX'
            'A6K|!-kNP{Df}*Q${zoUqpq+lA!HaNft#p;ExN5u&>@2JhHpc?a^<%@X&`$-'
            'syw>W+l{}YlW2HTZ};QVyfIHPsbgOzzNxS{3KJz)G<TkJzP%j{1L=Rg%6m!Uj!mwa)R;tSFn9aF}z+RN;oH#QB!u7##Y'
            'N<y`2oaGrLb-E{KKQKg*Eg&R%BR{9bZdaS1Lq?!}LBaX5B&Gw9zNBMO{;n0qh@cm|g+SjC-CXDWv$P8EX7uoL;3RD>fE'
            'BH-'
            '@pjU8!saPz9=B=Kegdh;a`CvhV%UVa#r_DZr6|1@F4U2#m{cu7)zjghhUEtKoy9b}GPL9WtkD7WMi6+LhcpBSmY<87w6'
            'E?y31@H1u|)C1AJDiT>-'
            'g%tt<_;lXYG{>)^*AGLG*wu<W6@T!(qd&yHt%9z@s%Rcvg~m17)Usikd6Qp>y7`H~8HWyPpIAYz>CEqmMMe;LZV9}0G{'
            '$YeM`+)!A?md4EI0^gfKN&nT6}F{Xl`%<`=Bm-`rsNGu(#s&bDJ=N>o-'
            'ZuS%M!n8Q~^{Vi?(V0@pI~XkT~`)!WFAFVmhd#m%x|&Anc_Zg&8-'
            'TP9&v#3Q(It&Z&FtwIwCM;vkSBc*cNfpz!?zAxDVE9NY5*`6TW$XHE|t}dr`t<y|1(_QGdh6yiUJS1DLHe<!Q2wEwT1-'
            '<7KK>C{{`5_h$9#wkeCZhwQ8v}6lmru-JK~gxGt4>}#dyP8vO1M3t2S>a=*2!xGV98To=)D?)qe~j<_+3AdEe^t<b>lF'
            '!%16+1x-n?{=N$4XoW?!AlE8U`jm|h-'
            'guJDzp~fnec|CSqd0c}<$x;DY|0fsP13lsF4L5QaFB0F;0os?;iK&*tDD^{>=#@o5=mi0y$SwijG9%!@qC)ibSO#A&#G'
            'uEtCZ1Wn8xHrFpnNJX^Oz48L(0IM#M_>viQmF%AI&h~zhqlX^0)(T=egmG#ua?Ir5XRs`rt{w7o_5&0WmzYV&0$AU{9b'
            'EByb|QO9o?kRxO!2*@eHH#h}A42VcL|g6D@-aJ}j^JR1{Kd(Unoust~o?Y|@;T{?={ZMXtIedNG*I|`U5-'
            '*}++si#=pkq?P`?$KW+p2(G|18pDb;IPzQ8g#dcEU2r*H9Lw?e4iI4_(r2l&H&`!?gabaeDuK7R_1;MN7^u6L%eQiF}T'
            'aOg8l9&tT`yck}~4Qj?fw~a6AUJQI43XdIArYE&|D@B{*6$O6=d=BVtdyadak$9%gBv#7HH%XY&gOtM-'
            'y2LnmB${wvvWrWDmHwb6y2gIKG7B&*DS?Jju10av(%z`4N|QY-'
            '+jP5MI2^DweA@H=g)^ucKFQFx>h&a|0UhyD8;@YV4dGHvpXhMI`Oq<13eSLa~WN==a?jgidsUNw5oeGl-'
            'i$pE{gG8o>#PoAW+Q2XK}u!I`1BEb)%lKJt?<`690T@2?|>A-2-'
            'b+r3wJ*Jo4hJ2pI=*(k;lA&6_H@6enuX=<29V?^(nIP-@mN+aFfM>z?L3BEnE<Cdiwbv%%)k05F*Ln-SR{TbpU19j!qn'
            '{R>Ou;K6NwB8M1wtNnlVOX?O#WsTxi|Ql+&l6F(&zSr{rV>)Gk6btFy9QX4-'
            '6BBb0egM(}yZ@ykHo(3!pA@7bX}4VN~!M5-wnh`?ra~@Kh0VAV(7P{~6-'
            'c?|x=V)^;MWMpV^SwHbb@#p3a&K4_vF4%^CJ!tP@`$YD<{vg=X~blUC3ybGSpx(7<|XoDj7>sG_e05>(%HNrpLT7++19'
            '*C`rK%dx9lEXU!+DDYI_t!emZLuN0Oj_}e(;Zry+K-'
            '*{cdy&64HjkWgu=`Qm<)K#*r{|MlcMUG1_#Sfy?zx=|4v5TKRbw-'
            'a}55iuz<{$C1572O{7;^(EVIOu)atDSEeok%Z1*=;aU=NE>8rQtImUBH^Mr#wbXrF5t~cKaO%nzy8Ea+#MhjJ?Cy5FU*'
            'pbPozV^KpGD9_tOQKO_JQy&Nqq136F&0$!;_v|2&gK95%%qvuJ8uT%%j0%M>@G_c7-lfdx!zc-{I-'
            'BgSbC=8+?#@O67`*;fMmI+OaX@xaTa{c=HZ;XcWLtBX7uUYJu&gn@Q=C8eAT_8a}blb9d}Nn$}uJ?@A*iX^DfDe+x;-'
            '*$9=Mo``?a=@rWk6aWnvFwVrBCz%j?@jZ;W-viE7GWeET0~7u_g7oVqh-vAk85Y}ci;*oh^_YN%JU4L^d#&<iEDEGFQy'
            'CBP{)5c$C~EZ~o_=n7j|*xwaNSHcuHBxF>~{H>R;>nq>*PS0s}_dr%^<ca9^3g}5WP1%C>E{(E|JnGrOt`BduN$@S98L'
            'prM7sNBLZAQ&qH;T7N+WTg0sF3^}W6Xvi-wA_;v~CxW!W8ONMa3ZaIVF3n8i(c4K$iU8>Zfi_3C{VJn{})IHY$(c?a7?'
            '$$_effFPJ9|Att7%2XXjMq2cKoz?i)Wp4|YGDt-ccn0JIY*+<<pU6vBFbE*=8ckH65*GxHQp@z$FM%N2%Hb6;r>n0;51'
            '@LxWk9Bo+l6^qrFw^E90Oz+X)rdxYN>Y5xD4FJ26-'
            '*441rG;q2XRRM;mDzTH;X9?Z>Ha}VHte;QtpSqSp4I~cLHVd#2s1$>keL-'
            'U(%^mCj&7)1`z0%ko(d)wmIyHDYo<S6V_ilfGF5@4#U2!CC=4<dV`!2D%5j4mr@4!R_g>A*Z{CsP44XQr6a-'
            'B0lHTp|%RjE5yh3z4Va8{3z!18eCBsGjj+Y*5+*?Ge>XZO6UfQy)qmz4M{xb9>1K^IPQmi%O6y7=ozKOh(G-'
            'bQ<vRCt0MzsoIwmhz91<7*=r(Lt?yvqc{`$r5@v+$q%@#pan*?{b=3WLNr|CNZohu2Del5K7C>U#Ap&85P6C2(Gpnku#'
            'U7>#Nn#RPh_a|5!6g;Fm~<gWU5(s;@0MJcw&+O4jcA^`;{mxFfIc@Hx|4o;ARcS$5OsGl9b)D1wRHgz{`O;uwL<r=*o5'
            'D4TB7L;I;+c4a^XQ2s7Myts14%*Fh26N31SUg}6k(c*~`@{)#V6KPHJro<FES*#}s|DFwSElprIy9mh<bW1C?vE|mC+q'
            '88J{Gb;tSmFh7#C>^TG?-'
            'TXrT9kct6gxHVqTzKDoOJNV4DA&BtXWA~)@0)gdqou0Dxe!KZveAr*TAj(0&v|P!Z3qdsA$6pN2S7Gq|%Xb+Fukmr<CG'
            '!U=&#<l?02Ui_mDtag3Z<%Y4AWU}!Ab4F4qDu)~c>)c1K}@n$|U*8Y<8(zEc=JcI6hCkLvR_CmLh1HEsT2%97Zpik5jR'
            '-Baprvn#3-'
            '*6YBZ=DV1UGBk0a#~P1v5@t8_AV5SFM&+qJy>I2fJZI7pjI%9(dX5T)i32$L^p0k@x$i0mnRVy6sG`}bu)8nZVg-rRfq'
            '5Q?h#KPKS<9@Q_*YPinAb%LocV$T;Mrb=AwZARfvQ5V_J9JAh6Ez*JpCz2QX7!n8D@G`JfX?!T;qCmGE6*%*4i3uy9yV'
            'b!ib7;WipjVKCL{bNyT3`9d2_wM)@%FadVY#N+->MXZipfRM}!(;KU)Q>z_wVEsF4=9Py^g-'
            '7A7BNKds^FVyg55~?L!;?5&=)P>tDD!)X=99O{vkR-SAXN?d&qu;USTJhX@erX@QQSWNuHO4Kk%JyS@cVQohVxx#mbI+'
            'ImUt21*w_!;JFh@Z-'
            '!evrrydNgdxHw+V+r4eBPb)_4|e~0ai`fz;2dJ3CjWA9t9b%0)aPbxiOQ(`xGf&Twr+u5u{^kDt_}CvIEm7dPB<B20Cy'
            'Tn;dP`h-'
            'c5@~=dE1ee9#x{v;<%?PcM!<@MgUJuNNcZRzQV78Rj(zGdPr)xUXiAN=%pGex*Fr*sy2*IRng@<)fG<lMSl|WWed02fX'
            '|eL0AxtJ2;<0OM4hbqzJ*|OWUDi$yuB-NQ3$=QBZue0kxx+g0a*D%GYtQy!M`esuk17?{x#o=I<Dh+(a(EOQU~%&q3(t'
            '>tHPGg|eHMlAXW8z|Z~-'
            '1|HlFQyeGJ%RLSz^=IaJ!yh<hUZUCYg)D~KXRL5<qQfE=DEruA)}N_vT&F3Ad81scT;7+|wz(cB8C9eqEgz209z(&`^L'
            'JrwExz9v2^Bdys2t~uxg3|l+9;AT97fPyM3)}fyO|}PAOQ7|lb|5A5C8I9N6Am~T)Nl}8;9>8kG?BRKWWEaUsqW2wGXU'
            'ZeuL$y9<py?IB<UW1isNe*kv_=t=D#-'
            'Tpk}>>v)5MPJFC9p55pb_?%JsGzz(Hch%`vd;(6PT)bCt3M`Kt#>^feme|6pShqhE*K@XDdY~$HwKQPdx(kr}n$q#RTd'
            '{^C14LCS&}3N#xg~xK%ci|SbVn?SDClCSSj9ns*fcyHt-(I<WhO?)gVirlGTd!}Ctq^0hGVi(Q{*k9-'
            '6bEEU%P{D6NfQu@pJshjD+;TIb!==myR9tg}j;DxRh-LYG!yqJ;@+D@}=N_Rt<*7yHlCvb)<NU8s?@M5&8XAkP`l!hz&'
            'JjQPd+0l)4F@S~h{C^LHE^TF7dXe2>9-%0%-'
            '*I_^#11LHi35N>=Em^Lf$X;&`xnYPqLdn*F%&Z0(|m00~#fV6}i0_DqU2p_Lg*V-'
            'vOde)BN<`Iw2B}BomWi=)wI8fJ8Zr0U68Bi(Nh3fjzU_5>sj2f?#T#s6e=VU=mwiv8`_lh1pE=BrQCZf;xXDHv=i%EAD'
            '<87@~a85M>pCvkhPwGv^%8D3rjN`h>MHLfV<@$|b#<>mxo>AKMtr5SD2hok=Q8+BngZq{S;>5h01+3@9(wGq7XwSiCrA'
            'C-@wg>+HO@iy^N*RlOHRG18pFzS5!9zfYa-V!p?DQ|<ZjT(~F#Qit^Q6Nu*Y9Kodr)7wj;!LS!-'
            's1Yu$(!zv4&H8AZe#7xm7BKF7SckVN0^N$O6@M`tV}tUvyu58Qrc6s9FrPf!If1{4iRG;ryk@d>ev7kBjMlQXJTI-'
            'T;>_eosrZ5E!?2Votma9vOIsTFnQ*oa+b-t9#L^sr_K=Y7JjA(n0eYKXMNRz^a-uu-'
            's5i6Oe<IQL2S1mp<YO^B%HOJ{{sxXUMH*4XCDm2OoXQL>AwD6m%Cv?VKZU{kRiYOL)_pZEuNOpeTK6MbPErRuxnG9E{&'
            '8hOr8RSU)t5n%B?5o`GmAJJAbj6P`G$8vsIA7pO)(If26Ci{V*rJpT7yh^3yGh%fzPV3B(nbP8<6+wlx&<F$dYvQQk?e'
            'S!lz=ONOo7~+PjkR#zWdUW4}5ocH2<~a-778Ns8)g7@y@-'
            'dvq?}P4i7r0(#2YZYXFj6Z8ABe3%Ph)Mc%i=}d|HR?8*gq!UCJ(AoVGnfPY4+IWVdIXYXsC9ddE(156#Ub}h}2629{tZ'
            '`P{#)!Y`leHvB{XFp@h;S`n1%V4RJv!KFpv0@9&IZr|?Sb9$JM>D{G<a)(V{7n@Ve{OmR-C1gNSi3W>L3mvjbf-yeiKP'
            'AkxDDqNuR*B+nM9>mA1w}HtkKT?@|8h;MuB1`Ey$e5Iqcgq`z;gzfO_h+E?D+@`8MmAh<;R2bD^5D)=K-'
            'ZXLhJJJn?0Ut7&&M4gGje_}AFcv>nX~xqf&;$EWn)<<AAlmQAmA_*V2wsj&<E+B7}nYVJ)y;*U|J3`-!8)u={Cw&X-'
            'YHmr*QiMFO}YtCm~h&JvtOLz~{445G)=I<$2zaFtdxwgz3Q8g&XA1TOVq+HINzcz!Iey`|zvwDZIx1juee0W8?m#WLoS'
            'd?%bu1=TC7#n-'
            '`%vwzr{@aSZrg+d^bYCMi4VLnl7OLv7wC=5mf74DpI+C~0<sgDeRW&a5ZmqcJ$Mod@wnG}MG!KzV^A*lI}Ot9Er@2HZx'
            '2E74$d=>u>a<x#cv<G~5haO7Kk9%Uwv<6PJiketso!5$9P1U)BOG2qYS&$&-'
            'T;wqp_sGJdbY7fTN=z>|mLOi^&mxRTp5^k@N+6IoB(BAwTYTa*;yyENV{=E@Bezs%Ji^I&7PG2CHc^3tZ*+KY>JnS!uV'
            'BTBRjazy&@Y2{6mAKSG4$28o(cVlX{N8l*OAK@j-'
            'Ge6CC*ZRwnzpFFhb`837zI{J*kXPg8*LteLB9tc>$Rs}jfCOdo3oIa+6_Zl*P!aUCd&R;17(AyC>`%gt_-'
            'l?*0#l}Wt03^<z)ef^aRP}(tk|zr8x{AsZwlxM6nC`h*X#jUJ<y1AFhajVhaVsmd7;pTr_N?(umQ4SaRte>edI61M-'
            '=e&0VSzR@#V3Pc?DDGeK3ot%?{RR}FeLugOA>LaH6rOAWj=C^ZVg;1eA*VOKOf*QueBx?!k$niupUmf=sAVq#P02OZr?'
            '@ELPp6UTl0_QDXR0`9<~FV;B39L7W*Mf^4Km5zL^BeojnQOEijYzxjt(R~leGFv}PNW4nV%*D{&kWBPvYoX51is6~oea'
            'I?SfaQE<IAQ++ek^3ciq0xLX7&rFTTY|%nag-r!He<OmmM<}o+97=t-'
            '_^8nyIaX01;Ns0t@GZ$dw#Lix$2CU+=3Z5N<$K+|rps7q4RY!&s=dJO*iD_pmo&0K-'
            'm5;hFGH#ELzPtcaCD;dAL|_(K|x4{%ZL?n1csa&ujFkuc@G&w>j(FVjOmnuy@i3M6+u$@iIqbR5Ou<cA=n%G;Uo(Kc|t'
            'IhC#n$%P>=LzGxMN#5>>!6Q3vlNhXER6n;sj>a;aj>|*+L|-U86M&<1ge-'
            'L(Cw+zn_$Z<U_swMDk~(f?@%wi)OLsGzXB5L<{bx9L!2%j{I>D%W3qD@Eo`}lnQP+S+%(3`zbj^7T>JEYAl2!<&M07!{'
            '$3JjM`a*@{<Djc7AJ3isLSILHBRootu&uBT-gVc(^qW!`voV3Xq*mOuLLaC0ZbZuk{Iwo?Z;&{lDA*WQPm-'
            '^&Q5gYCWC{9#SHTP|GOvTBmLJGyg*57o|Awa@H^I8GAkgf0fX+fA*uTdQOoChB>XuHNex-oNBfLO_*AjQnT0q#wOf1Pw'
            'z+gXZ6`>qY`fEut<N1G^85bvgK_#n(8a<Z+%_?@)lEnek`XP(qD{6<E;;y1eR3Bni4ZQl44mK&=SUePnW{Xll)J+ZPAx'
            '@@3ha@&@s$jt59n2w{2(X@7hK)at;p?_^V83^t`BK&h``b0a{o_2F9WEoxT74|H=w#f?eT4nT)ab6U?R2?&Eb$hQW-gp'
            '4MrBtfz59oaocJ0JUw*iO<e^k}v2y{8_g0dHlTM)27K;`eUBF1h3@jdOAiZZ*L13*B>3-'
            '8iqo2((8e=r6xmOOkB~k@_Y@KA=;V$w|I1Iw3G(k|j9Nbbb;}$_KdaP<0X&QJ0CP~t;NUV*yn&mamn|b7yMFPlVEP@fI'
            'w~%}37DS(4jPUa_dA3rSX(ru4BMV=H^{#oR|HrP{FtH!KHg|&Zrco501`wa*gQsWa2%KQyrem|r$AV47N=6!O?j+Oj*1'
            'NbzqLnh)q~K}rW)yu8M@s6dVDrD9;Fib<_qlSwe!v0T#kp{4U<|ki8Dd(RCpC;7CbzE^pa+i`@ye=(n;nQd7NvnMQww%'
            'Qg%HPLS#%8&!iA=~P+XHk{MJc8oqYo-'
            'T^6~kL9B)Rb{K#+JKVtU)h9Z8j8!+XSRFt8RYb7^X%JbbNZ!SjK&xOQT)$yXx_T^u*VPCgik35H&A$;VUPXAh&;gR2I-'
            'zya2Od9p3yXXkkx%+4uszH`_U<<1Ji?9rI~*ClgOecRGYxr4T4*e?7S_i`!P7StP`@@5mpnbf@D1#vi#u#;4Gz9gu{`<'
            '&4=S*${#dpL_(o&ED|rJh_pb+*=OYYlj37e2ijXQA2KfOQutL}u0^@zrHXw?oB)kE|n~PK(-'
            'a3G9eJ&JBXpk*S%aCS!;mj%vv=TL9{LagO?>fGqW7NqMm@34e&i}~puI1$IvRv35;sTp|E5JrJAO00BQ1vu*!H9V_Zc&'
            'v2lc3FTFQp%Z1S9c)k~nky&>1?X=SDTg8u9pqANB6=hmf6-'
            '5UnZzqxH9OeNZi4jua>2H~!RdUq~lDBhTTlcPRR_ClZ@a9eD0vJY?%M;tKa0p!NAB-f=jKXU11U-'
            '@0Ksvgb6TXwHTn%F>|k%ua!A%YSgqHyRCHg5g5gX<$+rB5_Cw?T;6siMb*ybK3*&@)hxP@glU*K7qfM7~$*BTENA*2&p'
            '5ds-g7+u8qCUe5zB7H{Ld3#^`t4_pBUGziY>&aXnz1T8KsydQ7=*I%H2*HeTQK6nmsHkV`j!F12Gry?xop-Lnrqv8jVn'
            'P%<dH9mCoc#rU|g2DUb>2j~7tG|2yraIh4mf~ApNqY954=El^#0%T7K10m;g^zxwtV9{-'
            'ci5nKE&an?dp~FQSnGJ&DnJ&1HbA%MIdKg)nmr%R)F=qbqqpe0IXi}ey{h{h~+u~l-'
            'Tv1G|cmtuw_8}U2FF@%98CY#?18ao>P-%-Fd6XKDKRwPc7X0R=3f23m;mImkvN0AWbrhHf*DfS-'
            't`5L?T^JwL@4%rK%b=?#7R}CA!GY`t!0BrWY|V<$VtF5yiG898+CPxJClAxl2%_EbW=uGJ70e9WA>YLW4|}$QDW*}Oz~'
            '995V**xOjl*;OM^RGy37mC}VG4QZL3K_T(|U0v%pLoNY%R-'
            'R%Vl5KWP1TteyIWDxEXp<y9z#+6~O(Yn~^LFLy;6V+`^clske3mpTIKw6!=P|zUv-iRPh&mJE?+gN3*HvtSYn4-'
            'yLdCdLb)K80HlGq47*Bg}g?ZqM8M1{ZrUd+DKHF`D20LHY~~d11)dE!2CuhO!n~Nu_Zfk(*gnP8!m;Al}UKyNg&88f5Z'
            '%fIH-_*Nb;Gsz|fd??;$_z|2YQ5I1S5prqlABf%xQ06S$wArj}a@fRW2f9X0D|o6-'
            'j|zH5q*<B7n%eKU3McuvgavvBRsCG=@nJ){brK(n@kxcgQJbpG;!wJXwr?U)Z7xYG-'
            '!CZS;T$Ce146UGaNb>Q7t6|{Mj)-'
            'C3`jDnY)aH@Nfa0}jH9ClfvYAk*m3aqjq>EuFOJSzuJBSPuX+l%46Wg{$*ki@%BLgBXKPF%jH03J;vXj#}0-'
            'pJd;`(+1aF+MO`jE&H9g%WVY`e4HHYSPB1g<F%XAS$bp(QX%xzjl8hb<$Zd?H7SXlkcHPVH3i%GHi{OqoZxPIB61w$s7'
            'mpz@`PNTX}2op#E`C-'
            'Xnr+dJf<n`jgm7^pjvQPi()wljx*G!jdpc^0C?rmT2ysXYQwX5lW#uHV4_dI+4rZ26e37&#X+1qIaxhV264!F1PtZrGM'
            '|H#crX{c5(x3VNa(2t@VXPEr!U$C`HRB=g2kx9NZ#t5su9BQcAQS{c?R14dvp4H!ig(A*n^?bPmFRbtpcc;H4WiO-'
            'Q?J2xu{_am$SXFpS|<eLV0OUNZdP)}9;Cs9_9GfB0k0+y+uAZOUL22cz(vdbpO8qr!S<h63ZCRJ@ukX#SQEJTa9|Pdaa'
            'hle#M*k~@dUR_SAti!r#do8h%ndzqQCOYqlBEU~|Q6!_BbgU8kF_@{Uau&))t$A3@5n`ft>QYHqj{o#SLx~gzJ;21s3t'
            '%N<jDRAJb{QS8qLDh+3Y#b{A&EJjm>i!UvH*&?bE6UK)iU+yw4ZslJDI(m+11$~JjCA#Re!Y+aJfY=y=t&ulYBxfeD1q'
            '^&<M`J-'
            '0<1c^iKUzzKAPwvy5~h8*CGrxU+5B<UF&c*(hbyv#L#WB5zoX66Yl_b2wdlj3q)Q+sbdKke=Vl#9J6rE++{3Fa%05Qo`'
            'J0RhoEhwLL%fXXocqtoVcKer>g7eJ$XLpmU2cp=@e3tkVcFJmco#)I?XLgVg$uCL0@Mz`aXC{#^dAg&30v6H06zA&wk='
            '|J#ISQaRt81+Q7cX9uW2tKxr=_v`osw0Sv_m!Cpc~12IDJ0oaD?BXCC1jGx|+Uq1>i;sA*o^QeY)G%UV5Ky5hVpmf6{E'
            'L-'
            'J3G!7_W)~Pk1`0+X2!1I}UdDo#_g(xgnY^OzD>k0pAaX9X`ABV=gaQERTl&RbY7V?pFd}kBY=b0i4+Eh_?bul=+qqxIS'
            '943Sa5nFJUDSP4?d=9)pF8Q`&Udc1?zeHhVv;t%&?xFzm5v^&@VsI2sz(BxT2pn5Xm)cs97>-Qlg}<?IMyH<`Xj_xXW-'
            'qiVYQQ%uyx>1uUSiStoWUyY0b|vBbZ_!K&`rogv#c`E9^%3bEjip|brs_b9%7f{17!b+__oRqo0FyC=h01gH+dcAt8vp'
            'a`iVHWe;KT|lYu|(0vI~aRYP4XP+EB-Sv~6q<KLSxPDzmw!*?Edo{V8y&I(*qQ-'
            '@tfef0GDLUN@)372s!pl>hw!<r5?JYxG2)%OOY_a$5WdC`jzweJ=j`nL}(Z!N~6f-'
            '4wkR|dO|X5r68+eo9XC~E%KOE21mFq%pi(izo_m>gXT%besf_^LAexf71Ln?}iMM=RjDPBC@VAM*On(mS4MRJza&WgAw'
            '~_j4!lCf{z_@jD%j4;7=G(Gz+#Ul0APQ{YO|OZ?3Ih>;7zvBaR0erZUD?Gsj1sOUM_>#hjw6*ur!a|3eR_C*E8eEyuRL'
            '0zW^80O+-'
            '2`KUs@17B4icHdxkR&`IA41pplrrT?7vgw&G`3o#Agiko3=^`TSoSomuDVPNUKPQ2b{iBK@F2C~nWXhz1Tn0gro}ha$e'
            'oKTG3-JDESE{e_FdMfZpO>nVLD14o{M2r3GV%W{R^Li+4#TdU+nll=pQIKUxJz?x(F60AP-'
            ';HN!y;FRz|zP<F5x<@iZPf$A8qVwk;>$Pq1N2_I;FG`J6=c-'
            'DGYaa%3<z6%yxFX*5NumU;hM%KUklq2nh>m6CXCI)1w!{NE%}PhtVwyIY|`MwTvbRK~-;qBO5$5zL-az!G~(|7ZTU1?S'
            'oSH~kC6|AYAz72MTM_<N}xHTG1yvdKk_!MdVGtUS~LH+!jdUUyeZsP<J0J>#SH%FI<wuifkaW8FN%a*p4>-'
            ';zR#A|fFQp@CA@=PWeHDk4QzM6yycEA5^3o*G)(`?@~o)!ti6L()=4h^&me|Nnk;Ke~_Ocl;iFkL%g@xZcP4{+{pm>%p'
            'mJI_0Fs)#IebDe9;eO&rv&`Prz29I#b0+GnFyyx@+SrInpp=yL}({+)Jes}8xS@pD<KX_wonbvHaxyB=??c6RnXMI{bu'
            'E^{2zOs%ce{zw1q=<QYiiT)k`1^v>oAB(TdJxRiBa#%kPETx4<o)EneekynD9uenuV;%dx1TM>(Gfo^Rq<de#QBL@-'
            'KzSl$;a0^z=I?1?df-kyYk21YTKVNEtM`*6JtDUQ(y~@k-Q*UcYw?%yQq+XDt_Y`}U-'
            'q*0wESf_w<gdsKQ0=V8AXnd>;CWj&-eVY`cL#L{ulH&Yuc-Akh-IGdyk#kiiwA6%@-'
            'c1H7QxCeLQBZc0AlxEvNp0TI!Ug+C2A%YENbEs42dCsHPxps}{j$t>*O7QY|;YUhTcFtJ)9ITWUXkyQ_)#*sASZYp=FR'
            '`jOg=uKQ~558qcil;Nm$>#Kv>CVxA%pDS(D=0#em=?Yn^@%;L~^WVN-I_E#puk>HgZ*!`c1cr{%`N}NPWqXHSk3Gs-du'
            '#_~Y6jEe<^iNLF`AzGQ^9h5UPO0P4YAU5#8f^_k26$X>#=5HDILt?hYLfuG5b29=XSZEf9FdoV&_X%-'
            '!f!PaUhFGu4ZUXeJ95wIcL4GiLTnmNq46O(~3W<$-'
            'aI5|2zNZZKcQl6a72?3;L_lE5MU8frf}iGu>PAU{#d@>=<Z+X^By~tUZ_XIxE4QymDB}|BV!Z40Qfvpuq_a2n_#0r+7`'
            'F{RJ1$wt9&7Xd}CSaIzSep5V5_{t#G`Lxlfk(T%U(;K$kt@^W7uF3Kpu4`qH}yRjB_>aInnEv~@+#s(1;Gjd$L8yfkO>'
            'GKU^thG=5QRulfp1T%|+CQy;xo$7n+>;Id#)9a!KLwsyMB;|#xvFazTkw8+ByL@J2W($t5%crPs)3IV0l%O=DOkAxM#O'
            'L8@onN*y6z)6_2oIEy3rYRnis*q&1)>B=>*tu>M|q#j2{LnZN%g|I=H`gnQG=%R}|A?qt(j^xF2&5CYla|$NfkM`%({r'
            'ighZ*UVh~5O&&7-'
            'aUocZydw?wqT%a_AF#?i5hK2tQJd)ky3&|m)!`F2+xOgM2+;ZpNeYHA_qYHyUQ?&X4Xkm42%oCW+HfQ~8_}C<BL@G{gN'
            's4RFt?=;$IVsI^u%A%z>|f=N3*DIpAXi!t;Gg`c#Io#MD~OafKv>d$`_^a7A*Rudn;B;d83{3YMf!_<CTO&(EdC?T?hI'
            'xVvvpd&4n;YP?c5rEDf{0gD`Ht9cXYdQ7EX5Jaj3-'
            'va`G4qOmU?RTD;wW`35=fCD^t`i4Ixg;;R60ycbq1na&9!ep%>UE3iJkH56yS=U@#IM7TSrX6vY&S?_xm`TfWz2VUQ3f'
            'yW$sAEe!gU@OXUCMtCw7gjGZD$QS?hisyS0~c&@F^O<i-4`)I&g*#qo?Cb(BGCq@^4;&i-'
            '#R?fdmtB%HvURxf=d><%55MzS4j_nP^7u&|uDkQ0Qf=9I!SIlIN|#!`sX7i8LP?`nRi8l*M7|Fq1AeIfe#eALxxk6lOj'
            'R6PJoeYH~H4&K!)zC`BHK$xp=)19xc7;DsaCoQU@vXZ&E03gNTN`aUQNuTI#b)A9#!v?`gA3wikB?i*HFS|Q{%=3`dH4'
            'h$S=r5@_9Sm*7&Gi}EgK;nXM2s&~d>CGGPrbQ30tyRI#Jiah>FBivSPGGEE2_Dy&VfE?xAa~mrs^?gZ78xOM&1Ef|i+6'
            '?{tfPQ_?~#U9gUIxJST>iF&CNRooWuJt|7Jb@c<hP%is5MB7KjRemNIxO>S^;=Z<6}>EvZ(Hz`AltDAeUcshB)+|AQiF'
            'tvZSCQaNzD_;s95)5s)m4wSt=0SksjaCl!8#IVcJyKaPNfAoaSZ~DRNvK{=9_k>mMZxBv)gVm)2sQ-'
            '(Fru84_p79m<AflTgD4d3d{pabniS@9<x0Y0dgkadq5^&t`o9dr^2ZiR&L^8Sz%A|C`!Qmxs^5au|*BApABk~vn@BHD*'
            '`a#0-'
            'cc)6RM<Kl@4cgQaA${IvxUxVJm(0C|qJkUXPx*Pc{5u*p8%TniW)2K@RKnAcQP%B30dW8BK{os}$Dvkx5@W@MvfcLRcQ'
            '^((<uxc&wWy4UJj9ndGq|TfAIqHjaMLOa?D$-'
            'ZUJgkxvF{}I>o{RQF(#J{?WpTz31Z<;h0D`dK~GIS!=#=;nZ18$M%P>N@cdPLeTx@*Rwoj-'
            'v0G^Jem=h0<_kwVQ(5w7wD7|ZM-u8)1N-'
            '#})^CnSvt^>NQ}HiKmAhappAqVnB*G`@F_?2P8pC!ck?lA3fVo9IiI+{nk=W<6UH$}&>FtNrr^V1#@Cume{33@YYO&Ti'
            '8#%YMFgnRHNH)EN?RRE9yimJXZiNG#_uB#@^RK|jv*)09j-MU=><>wl)&-'
            'Au*FdaKAE|pT9zUCnVATY#!o;DVB$!djaz*(M<?xz)oMrK2oQj`Krsuzl<HNnhEak;R_$5yhZtKs3<m(gou;(d+?|qFH'
            '@uu+3GoOUCp9k@~;jEwGM;KlUJ?Px3Y%=HdN|fYz0`vH#Ax(D)JMBRpPOQI&zV#=uaqdde&*Q|hbPa-jh=ha9DsWV{kY'
            'wamfPL3A+DM)z5BIr|Z_HfizuQMTNGwEXN0aE#MY!Z}4XcE%#wd;wEWWabwI<*aL=A<a*ODycE%Allwe}?I5f?kEy$b)'
            'z2xIz}G-9jiPb1vK@C`E(z0)J$Y5fjZXa0{~+ISd3<T&Z4z>6RyH_JQUCs-'
            'r0!m$6z3gTqJ4S)B&#lhYy@VUnmrY;<URBv1Gh;YHEoN8i|v6^v$oWlN2q{}~clLMw65b@wM3UN+R>+N><JMk1e=0tQK'
            'DZuo;jmUmlhAP>^Xr<>1reXzH3P(V4nGY&@?Sf@SspP@@i(qQ>8!v=kr=nfqXeXzKu_+-'
            'KSN9OE+lMm}JS)*%eKu1Yo3TazDv{$|N;Tp+F-'
            '~HbrDN9$ybyrDignRG*AVaR&cKB$SD*rOn)Q6@BMs~LK$JxifrEM@E3XXXb!BjBTO`a#`+~$Jeq5$D0CU#-'
            'LXKcLjF;yi>qA!4+X~xJSe+9uWC((ELj(GMNu|9}3xFa27)Ljp1c9TTs9&&`#mJZsVQ*~ka)=X^EUkv!o&F$~au1bxPN'
            'Qz)TB3|aaPU?F!+1+SXw{n{tPiCeNzS;atA{Ede9IUijkwjxo~7*a2w(D^B76aDDE-'
            '6~$_#};Xk{6(f0+v}stZ|*RdZm}-~@yvQ`*xJfO`baz}J98h)iCOtItr34<2OH-fbq_t0iFB=1MTRorAs(r=e1j0it$V'
            'u+-'
            'NSyX$4q%E<~7cv3)Qtvz}9QU$4rGjfYvgW>x%*pR3UFqi@t3anTHN%Qc=(oZz6iA99Iu<=4?hstt`n<PEG6`4^daB9qs'
            'k#r}CcG-;Jo|T={)!-1sPfM@|)O>JzObI%AXCr_4L-H;?3Hwqb=xtjGFb=Av=J%aov;Gap>GgpGy<o`V7-'
            'HR(8^Zw6eZW3;0%tY^p_cOH*?s4aOZr$KQ7Hk<9t@OZ=#bFoXE1n}MFKtJi1@TOJ~VfRd-w9-'
            ')eCOM&9WI%tLq0gUHjp?x(-&jj?p;nr|2tq4W&7~z?aUlgMTa$`I3Qhwo%}BaTAov)zTO4mtdws1K8>D<SEZ-'
            'x{PxinomZ7=7Mc-+V%nZRy<>DC{#k_JT`T6DrGdXHzTuTCCtf+gM^3a$hJO$cH(JZzO4zn-bSOLdIWCUB1rr%w$TEq3l'
            'M)o0VaQCLdFgqP<Oe2E5y#@@Vy>XGT4N{39UFse;sa9xr;K}1K{i;cS<y~U}f+CO(^mMjk}s~>&QB2e(Z+lkIB=<Atto'
            'U?k^#Sczkl032xKD*ncVo!dclcJaHQCx1R-'
            'oo49+b72eIblZyrGV0g7MwftlNd1v?H8Q=N1|CuS;#`aR}Kf!QjS0?0nx)W<9armLL8YVxgGs?K_AZl47u+$6CApQ)jH'
            '{6O`k`<5?>I}lGH$u~{Fo^fM0{Kqbknh4zw)e!aL{t*dS|t*`$#YVUzu72Uc?JUX&2eOFA|A=<p;9^0_^aX@NnQ8_owH'
            'MMXhs)5>)0b2sgRxaf3dW{6LeIs;6$__xJd7UcEdwxt{8+_o-'
            'D8(*Cb~mAHmqJPJ9F9FlcoPE&NKUCeIXk7i0o_**^GwcL$hhRg$ddcEBxG22bvK;_rwydNgo?zM87Q#f~+w;=~HF=;L='
            'P=UxEEHPUfgKt6mgF~BPO0Cd=S0HYHsSntQ4QPr_K7<Ta>e&=e2vCAvL{fiKeIab2;+t1+GnoP_-'
            '7EJrKca!$O21x$sK(7`ap<CW|<MnY#8d0(pC!NMXeDhw$dtq5roFk8nEh|a;h!Fdg@&S0a)B-'
            'fE@4>G4ER}CEWw<M&3rZ%U$wg^zypiJoMyaLH+V2NT%N*#z)zu`+LXsGsJ&wcKPw-INGz$F=L@)Jvh~O)xLE{bhdhjK_'
            '-}VeXYb4==A!qPkkw%vUWZ-'
            'A}2jpApIaqnc0~EXWqX}0B{_fv`0+mtZQ&ktVFDXO0&Pd!ZQAKl2_tJssC#3PvZkn`@jo;MES-S!^kwtY2@o)1-'
            '<{P0Xax-5RC4c$ixr;@h7hFerBO=g~>A;+;@($;$v?5>QEs?wMEou*KhTnaEam-'
            ';o=3I;fj(f6L{O&us7@vf(cV08ByF@^(wI4DQEMY?v7pxoN0vl~UynQnQ6)S`2rm}HbzIZ2Yw&KH+dq0s43y(whnIh<V'
            'pMv&n7x8iQFF3I@6Zj8~5(%ehP@T!fAQ?N@X<<Md<~M=NmWTK@q8z90tRZ*QCqN*;2?|qI;q9}-)UQ$v-'
            '24PlBDRL+igbeZvZr)?MLk%l%q2A=PeIK31-Y@c5~j`c84AzUaOn#VNLX1;&tz=GReAPg=RhVHb4Oy-mu6W0Lmt(K>R^'
            '%caR?=sq3KBwD*mZO&e7v=Pq_jnA8bMU^AT7c{}1;^1%v3GTDXyC58D{aXsGlYc;sw{H~)1pnkC#ZaAN@2JJ+M;PH$Lu'
            'C6Jufm&YACZz<217Zlbo$*aYd7<)_)&r2E56Q>Pf^s@lt-'
            'u!m>ra1`aH$Fjn?{gTPQxBt|wZJ#K6cQ!viPjcxa#^knfAy<_;b1<kFxiP!bNpfSLjs8sen|KvJZU-'
            'W4bHP!j)x`!m_8H!u(sF-'
            '59<h19X=(>K~mv?z#iP@>4y;)#qh#8FFa|n1p`(Kk_%r2@D=x3RJq_pIb7$!+~8)$mCSH7PJaLvj|<>*+EYe~Y9(FK$i'
            '>!N#D>DvK{O-&H`a*7P`lh^>}_Ao@i@0S)M#cROwK{;HS;jdd=Xy!=!#3YLm+<TJ(glw4a}@bz#{1?_-'
            'GLip$aQuN!cjzTlbqD%lg5n+-'
            'HVAzMNtBgm1urG9eY6n<s(S#|_?$mQWYX>sZnvig8Zt+0OZaCa<%^bg~Kb)@b7%bHsz^60o(Y0QR&|x*{zORRt4}-'
            '#84H<|9ttc!_Qaja1=pC5`T?F1FG4As3durR!T$(B{4$B`J;6eb%!{@3>V{2d2rU=>?Gcmxq0%YmDqr3q_B?yV!Qfg~s'
            '|iW3c&W#u5l8run~s!Lt$vGgiVHKSSsboum5c)jZYdHfQp1QVE+Cv!TJ(7U!yLK>dj`P}p}2CWLxOFwa%^@@fDqrS4P1'
            'a>}ZkybT!vF~DWBmd>zapw9FOT^o3f9=*F8OfK(6vBVRzJ-80D?`2Z630dfhs|AKw9+Fu<-G9!_-'
            'WSeh?f6hdjV9l+Dt@J~-e;VI-lx~`8Gj>)No|8x-d)hS*%4nUd13V&ChA`vC*;jsyt3v5#LZoZ8m-'
            '%*eOCe)4)~+`g7w&}l|Ae42UKJCE_h{P2s86i>5Jb{u;`N)b_71hyp9dn((Vr{54X@KjAS^uxRZkNZqln;O<0@kP+!0U'
            '(|)R9!%BJL)yW2@)vsXBGhWn{55eHJE({Tu#bXz8skC+ul$cHupR5E7A2$KZf@&(O=*-'
            'H#a|Z8sE1`5&4GkFAX4Y;qg<|a+=pc3m7cOv9aq+9ek4dk=e$Og65+{#lJdM~VkO+yNQ}F)2g#<t!=Pr_F^6U#LQo=59'
            'O|?f0g;nIu&KyQxvpX(VP(+Je5%^C$gPg7!r$-{~;E<mW_Usgewd>|H(^hAI!In0xS{4k7LN55}YAZajY$J~@$-'
            '&%DOHlsbAmDg2g~A??;d;k1OQ`~(wL5Wrw<BJckB0=O9M;U|N93282h=eHvEjf8=-'
            'ayrEH!KKZ`K65ie6<HS0&Ok(OZn*g-'
            'NKj<_GbR;KoDpCu!fXFYvqFqz3a6A!chFJ^MX{7~~e?WVHbm5L}D%Cq(e*&t+_l@%yvBN}(fC8r0cs5gz8f#N=;TgAOe'
            '#@m^jmu&th>z143>uS~)kg)X@5HV!^UvDEwY5s<5F2AhO=*xb(taTQ9`^kpeBi)=8+2H~3EN7}Go2Fi5Ouw&6$xTYizf'
            '0Wzk)#DMc?prif-I0bjtjjSl{4%r7`!nV_i_z5`I-'
            'qcFJN~I2AtE<TNn7*@$X3k6I7h8n{*h;e1+<Z~hpb>sViR`k&%zT&1R#9PcVswS#K_@9yfLnh0?TuV%S<@P)*eRQf>tW'
            '*`v}@S^@0BcH_mrojymf0uq5{q`nn9m0`Wf#v5&2Ax+<BrKsx@M?VZ!*#n37jh^?ie<dwNMHJk_sjSvf1wAKV&lV2gvx'
            '&YjNClY72T*bFbt8mC9j`W6jQ{j!bp(rbk=zSX^d@t@%bc;Ynn<ZFo^Mp6ICIDVIAeW;v^k4SEck>QnzV8UhDcgzrGwt'
            'wX>TbG0<^{GGa;O$Q3B|n!^hk^nfzytgSxbEa$;8fGD4cT~U*&qBL3%UzSVzE~U#Ia%M=6H1{3UE(FUIz8Wn47of$R53'
            ';kS#sQJZ0ojAbFLn#J~DBp6BumPb%dz9#$~u8OBK5Zt#v0P*{y6dwM@yPG)K9JCf*E$YJSLOoc(9AzbU=s}v$04s?2;Q'
            'TdysM-Dzm&vZh-0RgSmUjog?2(60B1Y({OYoldDo_u-'
            '0WP*o{O0};j#}wp)gc8qa#;ktpGt!9a5C(bvB1+~$Kh0eAL6HQJSZW6EyC9+J1QM(`w&>p-'
            'XOd$1ZHHa;dpW*HjT71(xn?vN&5xx*DK)ux;{F@+XUjgu|Vg(B+}bAU<`PX8$9Kx;w^)6?v4<4XM*K??G&n+B~r)Rp+t'
            '@PqSw+GVzpHo^RoTXQe1{GMY32gO+=tAJOX2zmEh#aIxsrt$8eOB25#d{OnO;Leso`EZW&$=0uF8HB*ug3mOYGRPk7n+'
            ';?j_uT?BE~YtTL+pZ@vm1`RKgP|>XjTzl+b@1-'
            '|%S4A?)cx}bUZQ1xlq=u$H^MK{n7I;vwnNG%apl5L`_|0{o{@0TscCI$8IeMG1V*3l^o-'
            '#sQx&vh^_rThFF7RQVDc%a2pqxUQ81lIS!o=PY5ng9>ImH5>`DwVYxE62Rmj%O{T~yS#fauy<z>MKS&^!JCu1XyRj^$5'
            '5@9|AA$tu9%+xEchOT_KwJ-GJ%HVnD%t3q}=Lz;0cY+64}QevKhnW`a`aB+jtYmT($;~=~h4*?fmZ_+BSMa^J2%8m=+V'
            'qH@(SusSVzUAOiM@6#ta4=qbT>za@9$3?dR3}}V?h~=X2&rhW7_7vc1P)wkJOz1+H-'
            'S!ZFvyAILT!K#LqRtJSIR#iCx3^-si$%9cJct@uj@-P-'
            'Bu22LSgV`V;SqHdOxPk_|Y9k2e4OXBOElUfZqo@87bcD>Ei>NaQ?Oc*2y<g48zN(;dh`fa$moRdMg4@tS<~b=09O|@`d'
            'B|vofH|il;^o-jciT57XT(Poe*gK6qr_B?+F@P;AzP*N&YA3r;(D9(4+q-I|Y$Zh0y{kGhln(QJxm0-'
            '^WtDZCt;j90Raz&d3Iy%BZ^sI&l!>wFamy32xw)myT%^gFf9d<;L7+_1zUnxS$k03xrvp!%m4VNcTsvUb4$by$&tH`>d'
            'w(lHtMaxOyPgI)N^DFG(#7n6lgO@aT95k6@92nYHH;kQHvxtA`Bme-flFYg}FvmZs^ZymwY#tM}C&l=d>zl;$wI2W9%-'
            'q2q2Wcctan<^^~u%r$<V24dKL>F7*E0YZn>%I`D>^9@PlRdEfjybC|C<FiUd!X*IL<rjI5986xVDfDcYzy<k<N;n{l4e'
            'V~-'
            'KMGPt;ewNdlD9M#o|!C6LZtn0dizCi>|2N1l`i2P;tu^rmH<6aX%YJJ(q)5aUb6L8qe_I7eIyXdLn+$A9k%9!|Lx2D7U'
            'JFu38kw6o?9fM3Ze8#>a=d11<uuLk+4qxI*4Xd*mpY!xr`M1j8BzL%^Ly*YtEkq2wS5vzZGl;k#fa=Z+R{1wqfLj?pC1'
            'ifX&^P~(O#SO<&4=ZQ`_pfiVc`Pg~Pxv&mhwo8*`I@+`TcgD6T2G#Ks!TPFpbg!TN-p_{<weuUGkK+Jr<>SJecYZK9FO'
            '{N)rXb26P{83kPsnmt5lpJ|g9Q$8*p(CuBWcIbCj2pK>6Fv%E1PNV!y*vX>!RyoxbV_WE?9J?3gt&FSuSRa;hN4}luF%'
            '7H*5DXz7z#OHh&q|y#7ftK0Rb@tIcFhX}N+D9|e604$PH(OT6CSf=lwAgk8Q19&DMUueGvSu?O3zh&G#^@CAHmB7^-'
            'kuW1}B0FI}Gf!E)Bh&S5{MK{yo&AXEbB8m9AZ9i$$ScC7D=n<DC?!@ho0~|PF$<p|>A6)lj<3bJz?2%mo;>(PXFV&mjk'
            '16P*od>d!fjBMW3(jUwVO4|=UMc=UN3Yky!^O_@-bZ~_|LGE{a3K^PvUOnb5h)tmn+%bh-Xydt7aBP36c?nXF&dvZqhh'
            '=YdQL{dbM|c*m2{_yc}*CfP07M|AG{UE3DRsY^psi5YPe;G0&@AdXKg36C3q9=UzzmooHMZFkOuCEo8`<HAI9aj-'
            'SqXVrx5xs5~>-OQ7Kmv?8onev+GuP7db`PhXctLA%wne#G-)&BH%YdThc>V7jAeUlRF+$dK-vA`W4)?FdVJ)9GI&GoZ+'
            '9xOX5+Z4pT!W(6zP|O$#!?_5FHWbEpO99;G;c{$mn#`vCn{t%|EM-'
            '55<~Cvc0{J<z;=k}M1nfUN!?e0H`Lg|^z`=@<^SRF(#7(ET)Xa?e#XNm&nE>c?QCmIpi*aEIMzLty=iby)t$9FLZrfGr'
            '12VcEbLT(qa1h#YzX5AOwIms}dktInX9W;ER{#Dl5MELvu97j5hFaOdSI*c8f%)3<z}v7A>`PTq}Plc<2NYuBUpT?sll'
            'kBOW~Y1G{<lCDwM1@Gb_z)`jlG`_9|TR{hC{kH?QR5IbV$T`aYO9hd$i^ezv;=A@RXxr@z1<N*LbCC+fU-801-'
            '+5HqY6Xn6NWvUnH;i1;Ld1rjqd5OUG=wB@{KgA<+qiM_<*!&yU*Ss~YX~WoW|W%mhcu=R3a`I}=aUM-'
            'D(n?rO#Du+J=6rRlMz7PmSgvzKh>@8q~6a%;a_VK49rb}>EI~r7&L|>=C)|KVHI9V^ntV?7Q{-'
            '(!^hu_;QP{(&htAw+c&cC<*z%SshNXZO&4+XvoNwqdJ6dNW|J%X`{1QqH2spifxNx!f&uG2u|<Rn_0!`}ceNcnVzgjU>'
            '^W#141>&pZ2Y~V7;{$brVIXh;BrV{1m=#@!tO%s<bQ>S#JuV0YZOoLR3Rf+2}PNR_l@^p-'
            'p5CT^}!oW&2GcAq$~FC$%N19>5Q&TnefZf9}<3A0$WXwOw348HH}-~YZ-'
            '#V;i))pYZ(d5AoPqt4LU5G51P8dko$agf0}7R;pSW@63>UT{TmtD?J_876F~+Hyh)wzeP|K1PzhEXAvf2%!`9vY@GvR_'
            '$0xGE<?CW_UtNK(gik@(kR1$VeWDYFb;=1s;TTdRi^}0Ku(!IIcCN~T<5w&(QmY-Bxt-u`xdFLkT1br78q-'
            'S}hODLMI^f~PLL5j>g2~}f@Ydr2U+7kjn){^KP*a-FPa%-2t4g^`Jkdv0g#6v4MO^ET;OZ}7*vFL!s#{FJ_t-)h-'
            '){t$Z)xCn%_3&3;Ve_Q=HbS+S+1N;LC(Mde7XES^Sk*dtvkStH`N2-'
            'sfi{QuAZV23E@z_>&a|>Z^cyIi{zQQ2dPX6g__fabnA!1<h%kuZWiBz%W_&!<XbE@O2whWMkbyj4)BO63gH1~i|?=622'
            'RZi^yjT)(krfq99uTQ(1J-+YN|v3tGm&BWeU{2-'
            'isAi*Q4!EE|iI5qTTFoO0Nig;M79DecFYR2`Tu`bt$c#ZwY+|2hrcL1LKabMbn@Hyn93u%s8@OM0vKC>ZIa#MinD+Fag'
            'XYQqlCG2JF<Dpv;Yb=$KPAd~UmrlW%>oZAB2#ay|%Zqkk9=|1^;e%QE0sxIYaz8wXl)F7!fe3zccIq>1|@KzjEjkgj<Q'
            '9{%=t8B0;}_yozXa7DYBP8>R;!s;mAfo=u)p!J-'
            '9pv)J>;HzH7{pA*@<W~YJS1rh)Kl~`kc#V$}08En(LZxUFEI4u&_ZP^*)jv-'
            '`S}qr!JUxJh=aq0D*D<(6wn7PTg5}N6^n)`G%&9rb%9iV-UpWPF;QBg<j|*1eJyi$`I?ZUXun#z>P149uJg~~fe3mJ9!'
            '`Z_rcs5iURNe<8*IPcqdY?w&-'
            'Z#Q_@Px&pv;Xm6BI&HHWa$XD(e2!e8KZjD$UP$k;`SrNV8JvMxR=e?bF!Rq`Y{K4shB<N{7_D>-'
            'IjsHe_~*neh0as>;|u{E`pw?b5K^sA4;7zVRgV+?5oI!qsL;2WI-'
            'L?oeG271{qi%V1m~l4pOyRHS+dUfy$9rR)~7dbSmdE*|YXGQRlx1PyTYDsi-'
            '}Px~D`QN*=&@T$*6f>k4CA>{ydunqc36DVqIZ(V}b}#+dI5MyjzD6wdZv^D=Y5^K0o3acz{Te1f0$k21UMH{sF3?GV4R'
            '3GK8Nkb$#n;GldTxioK>#P4>)OTR+Uxi1q_BeI}3SOZ<7cfzmx)y%C-'
            'e^SQvLWN5w8>MTd!Py`ZI~AXR;i|>7`cD)+xAz`ildQu^&rWPC_JlIQWPIM##=3H^0)ALCQF5Ulp83LyrCsxh$??S)P^'
            '$~S50~JQM+>R?T0d%}6ouE!{TYfHInd;|8T6Jokm0yt2$L8iM=gtYT5WB@8afv@IW*wk0UgqErJfeu3n%y}3dfeeWW_u'
            'FK#L8^;1Y2gV;(MK-`sT(e{tyJ!v_IyL39E7K5+-'
            '*nrh^@Dh#92D~S|O8gmgN9aJ*~aA$2We!Zeh#Jv#nGxy?R6AtiQw-{wh%i;KTMa*(41({-'
            'A65H$svH6p%<ia|rC_DhQ_WW4&;w&~4yoA2ZNvQDF8H7h>d-LjLL~i^I*-'
            '@29BZms`l3xl{)4z>ABj)(s@EuvvTZ(hGy1@bCMCiBnK^T7qd+SFq!;1^OoQ&yMXb~f{J_+xBa-'
            ')Ky*|=}Oo4nJh#KN4lAn<bm&Ts@`smwE4RFF&+v+eNgNex&enu;8uab!uG76>R6g7BPLI7hNEU%3%7<}_lCd@iWJTm@@'
            '#qrp6(1iY*Zp;9A)CT{0qJAKu~9jV_LOYUc5)TeN8pUwcU?KNomXp(WeR~%e<j-'
            'lx@Ki1k%7tk*pB!^Bf#4S!oVXim>ecMt|)YBV|4aLIqD;0$Is3)paJ|U}bh`_o%PaynYE*7<_;HtSV7<?V)z^dm2>+8A'
            'i)ZESom^ND=l`jEzoiAkgZ;6Lrk8WY&8Xqv%dqQ2VdtyZIN0vaUWU-%-FFv0i0qbutAdkhyL8slgXo)7g+Lj7qMHIO|-'
            'GsxVb{H_*t&SW&h+C%;pysYE-'
            '9EG#2PLE#n$x#1;AjtHhrT#YO1d#z4)7D(BTn>qs4TD?x1zO;4bi5};6(0$aL73v2=c~Eb8#>dvVa@cLm-'
            '0xB+77&kd;sW5vyr;mPJq(S>?K&dB>TT{)D%{ZEnDj=afOCHhU6xV+Tfb`{VHI5@f!7#t=Utg`V}%@KzujUER{?L>~(R'
            'GlKE;;Q``fr2|#(<|6+rTq;&Hvhvb1p!<F}JZX2w?~<ud_fZrLh8plIZw++$`Qb99_bO8#0$EdSE(|rPM66MI4kuPQ&2'
            'q9d$|X3#)4!X@vf!oc%dZkhNOmXMtV?IyiL|CVuN&c0;Vb&{w-o%?cpbB!ii7+UJ$gz|A9qGRfbQ70Wc3p>;8N)(-'
            '_&a0eSHfZ`nCwC(>q9|Tq7Od*MWSW=3)0LK^PVh!^TKAsQ<7T&aS)zwONnJkZUr-'
            '(wIrDUrK@z>VX?m4DK{$z$4>(*vY5|e*Y*m=c|G-y+L|U;{|=o-'
            'VWDF_b|_H%LhA!g;c}j9V_^9ED=py1J~X+U~F`#icNJR9GvNc@{Dd)ZIU;%^4x{_<rhgis~9-'
            'lIv}C?6gDqc2J>sNnC0t49!4f$#j<Gpkz+!%KFlRrvCibv27g#z*haD|IjPzf6Eat;7PepcL7y~7V7BK0cy?BhY1QV0f'
            'fut$;pD8JV_$)oZ8^g{It`fW&2Y$QIo%iZ14mw3g3OPN5dO=MwNlg{+vb<x-'
            'HB=zk9{UnXz~;Ys9t0pZji@02M3|RS%rCYd<~Qy@PeJ*iKtQW9CvMGl9*F-'
            '*eyrI*bkii@yFg^IG@1>DLG82xf%=M{gWWqS_+ZbHz9ER94TG2m;CfR3{H#2Sr_YkFjTn>_Z6|Bdf1kjIRt>s^8?UmI!'
            'Y&50_^9rY}?BbuFN&^o*Y&;hFhXZvwh4F_1=t;Oeqm|pilwT<?lz{vIh{@*h#new8QtAMO3Fy30fT2LPu#GSud<b4~(_'
            'r{fHO%M0-6p?+?NKO24V-'
            '=m>qMAdQQ~72({KEb3(yfj`<EfrolAw3f$Wd0q=q?3>4CF57|LyLB;^_XI3&FvA`l9SmA=1}#sgW7$U!yjo+g{QT)*wE'
            'f6|HXPD8KVK9w1O-$}4HQxKY6=}Lz6e@XCUEd17vt_>H+*^B2V-'
            '59qBTb%uG{nqn6lE)WNx4m6zm2gm(pQULWjcMFj#u20Q219K~2~bnYX=Q9ZLYGz5GD-'
            '=1RD%8^Kt**9kq2kD>k;;F6>UoMW^CZ|klHySrZWnePzOUDXNvTq5C&-d}pw(G`-k=3}F}DOmd-BQKtoU~7#KIsM`|@v'
            'm=3b74`uG@D<XTt;Zi2mr;fF*<$tG0wTp50bIVz})or>>l_)j6i|}9jnCL4emIa_Yhv}vZP(24fLn;G!~6~1O+*9d{{L'
            '|a@EAhu{0MDneBsHKhA*(dsnhTR2u_A7SCq83#|STL5ik4aU;snRqpcmMOqK83KyY&ULJY`bD^W+4RC+%f_pv1@O_yoI'
            'KFtyR4%%M^>-KJ;I;@@=n@WQaZz|=ax-*(I?JHFg(}+7{b)bsin-'
            'Q*XzY#*YW!n4MhqXJ1zQpzSgDh8qc|?u)Qdu?2{5sE9c(<6iq5?}uvldUy?Zkh{XQS0`(%~jsIdh6yy!_U3#X~{n{tAK'
            'xIXyoEd+1THqy|rQZ<a2!Q9MxaG5uPb7l&mFEpNhJEV#tHk)92=~9%EK84F=jvznF3~*{#CDd{XE(yGV`SD>8GW{I*mp'
            '^80{+mpHtcd{S(`N{G+%Mw%^EgJ=^wGzQLTIl|2a=B=WTj~)cBu~1S0#>!&uws&7NMwP8*=42k^_GVN$m47%vvi9N3Y~'
            'yyR|bd?To=D#ddn@l{9?WW{A9JTp=_p7x;S?z??t&SlOL}N2Swo<?X|uy-'
            ')&%{vqki{l)O1b1);7hbo?0KqtIbfv;5<>^$%o*YKP}{?q?(1KShkJ<WmA`zEmK>^;m#ISiG()^v5eGj3x%#al7Ftl=O'
            'A9EggAnYdt5{;C!9m%jns&qwghvvlORE>3yPuEOA!WT<R$qc_bCp~K_Hcrm_;QKzm-7bv8HSkp7$XBB~nLOJM(-'
            'h`ZOH(>PcNBZMvC%E4@1kV$VaC2KA4r}eh*H>l9Vy<FnD03ub5_>W0aTIc%*o|sKX_%26L@><)6|;n4K)VzqmLCMo!y+'
            '(T_y+p<+4R*qUZ@QG0QWzCV4hm^jP|tNL^0b0IK-'
            '8JEsur3u`vUeFZfKO&Wn=<AwImkxQKC~c^4j3T|w9R<d7%@4dggc#AtjH3@<seXw&O3oEB!mx}khXKB5ovy<Lf6V=oag'
            'uZC|%xs0kvKbE)iGkCD0gZwVP3J;t*QKffww{3flp~XyS9ndC985g1Q++CU#Cr!l4`BgU^<WO}iM<}ThN1plXaXN@i23'
            'Rw!n4%t5k$yeuh*eX`c3V_Gvkwp4v&O((PqAB|69&d}U~A5N)kZNtXkBv--e-'
            'w}$jtWQx^xW)b&i4cU+0jnv6T?A$`+1Ef1-'
            'h=Aut%X0ei<27##;rVcg$K(B$TW&pHceGMfwY)3h*uD3VpDv>)2e9maHr86vfBmY--U=(#S2Atgt;;vEm>{)od(n;l5S'
            'mk8uHc7^)C{M0?k6*65<(=puwyjuN|^acCk;$ac0IXzAn{u6}C1x@(Rs1}5qDdD#n#1Vr@+|!neCtlgWV~caZNIL?p%U'
            '=^AqY~K3c@BhLD?{3!#i}dsTjETh7cTmz2UDfn@dNqCa2@cZL7lvCA?GwqhuK5R94B-+bRLa0--FvWHD=L4HhF860%L!'
            'K7?(8qSgGe^aL;vX9B}A>KDno}%(VtJ{^f&9&=115ULD(nci@(Tsi?jr32KwakXw5(gz=mM&lj)ZspK_~sdNJ^o>Dqve'
            'I8P`@#2g1k}N~In*mN9Ftj5L22686UU5EXa%4hkffu^P^Q-oC-AD01x0rhyqA^X+0G2K}#@d>*0y;dkF}nIT3J>Jt3XF'
            'v<#+2xF9ijrxk+_pE@zW7ul)3(!=#%SYzP#vc{%PZf%5`wlIuV$g^U?FNKk!{Vg$0)bNJlpx+AknjW0FI7+k@%hE5q2Y'
            'qYm@R=fIIO4EWoSi|5BB@S*Dzy;>p-TLst94L-^6=9&vutbKx^N<FMG$8r29)<EyJ-'
            '=G?iJ!E7io0g;sqR~_lP5O8QRgKR>%e!>^^283RUJ9W2mX=~q4kuijkqURN&VxTkCy9`(1vRMsOl@PeaQouLI4VFHl|!'
            '>VcFuE5a~fndAMe8tF1#>ot4pTrbm+KZIG(cLppzd)Xk8o^_TA<sQt_H(#lmDdR`3|>ua(mfk`2D^PhscOLhOE$OD9*a'
            'C1MZ0kiP7<^w1;=!nOoM(ULk07*U3vZ$&7f=8WxS7K}|5hvBG09jVd14^?N%Q0hbgO6~~4=4wLTJY&M-'
            '?Ta*|z#XD`U!ZJCI7~YflOLTM$p73Q7~L6*{7>$$l>ZC&2l@@%uzqVIh9sGQaYic|-'
            '{(h}1`Tq59uGOt;>7wr*9&JBMuF4RMVwP<fD=(;G~>1$^fV^0l;3#4tL^mw{>NBaR{<r$S723)9(t(Q;kBFps7%mBd{!'
            'O;=ib!d-{ur}(4&AR0ghnyqY3)eJdjic{_p)olxoka|Ns5PLJpq)SHFVR9uAJHaUQs%8qr%z8-'
            '|1t2lG?`v_CQ~#o6E^KR*1<(*c)?uH%UeM>M<MLYBnwv6Btmnd@7ci0exY&_A-'
            '2QERZ7oOY=~#d<#)Tp5g)i!&f=)E;D_HSp^WH(K(jp0*EXpnya?@XgN8@}m*Ntf~g?{hD}2JqKMYE<nY)9J=UO7_4mG3'
            'zl|lRu#SnfxFhAWDtV|sot>bdkp#E(EtbZZ&5#n%TAlxAgsKu3blJLF}!}(5$kXBR0X4hK<r^9YF<_#XSc@Ep8`v<?Px'
            '3vOhrQq2Lsg2HBpDL5#LvM;#2h|viD*v!~ak*W51LZwkdu=jZ^V>Doh-$w6-EEMg-G~Wbo*}ak~3?IzF*&#FH_Z@N-'
            'in`I219e326gLO08ht@sk;M9N^13kQBWoC-'
            '2h8)=x}EtNh&WztxrjWH|AVa3Q}>^2Icaj$sM?tltB{>&sI$u79hQ;GhH1l+LLfKEx6fXUw@WT{*;Y+sNHzScdc=bV5)'
            'cJe{MTO&-'
            'gvB35WUaB)W1l()Rkbmbo;e@Og@C9z9&w1^L=QbY|_hMmOIFJq<I~CykSxYF5K7~F)i*Y7%4{LBk2JY~yWUg8ImNuo-'
            'k<A%%kmGzV=yZkRnwdC|I1>mPM&vN?_+xzWeFG&PH4sqBg9c1zI27(fZ|ZBox3E|hqou~s;1+^P#y8Q@nw#DHAOQYGUW'
            'LLgX)u^`3+s6!u<+b4c%JhnS7Q9A&l6s@_NXgxoc4iBuDKYtKO2pQ5*W>scj<fXQy8g#1F{xh1k;QEsI8zn#8&KpXOEP'
            '?KX5Kwm62c{eR2sp)`iklyOygi%nrejL0_Uh)rr>Iw}ZFMF2?kRc>3<EE*|;Sjs>qbg5*pUjkc=7CAA^g;iy9w94e$sp'
            'Vb1RDhJmv(pj-hQDkra2jJp6P5MW7V%`s4_P@_#C@W$@E3bz^pX5cdOVJk!t;!+1w*dYwyhGE&J>bC0b*$wM%ix9+8@6'
            'A!qf?PCsAIzb<ukXS=wA(H8=j!5+Uwzw?l@hayoZRKNo9S^l|q}~8dB2D!7dYvqTxq<QU6^sam+f2;aw@1zA~F+$=l;*'
            'Utw%*3WPJoML7RNGS(@wp<^-%rE1zR;A|kiT-^$DKMdmLkx$UCc^l?u|06E!Ey(kHiW+4)qTEUgsQ-'
            'd0r{7w@Gu0QgZNqyEeOwHz)2*zFmYrZ%^c3fPEQC**x8O$fAk33+0l^E)VDE?>DOkqIj(FR`klE=)Jr~Sp_prI4)%X*g'
            'epCi;Zze*Sd?HD%Gs1E0f7JU)D3muu;iGUtH2>>|kE-'
            'j5`LZ*3$7DVz1h}Bu;YjqlS3yEfZ9v=E{IfX{Kik7a@Ys7VqRNmV-'
            '4@RHWn(q@Q<t`rdyOJYsrkbfgBWny5QzQG>G=3j3TQ80f-'
            '9zbV4~|9sTFZYnbfTqu>A_eB`;7lydtc6Wb%aa;kp@>C89ZuRdverO>Yjy=-h+j-'
            'aD~?D8c>uZs`4P1=)KqKuK#jNN>Cj&B2KzX6hBDad|T~diXQnE^mRpwaDn7{D?zqny~b|3}igv#`Y`z=-'
            '3`kyZ5ff7zVGZ$%Z<3&UOICSRTr7)-dv)y5b8-'
            '4T3VKX@^k*jlO=J`qjzeOzKG{heseSj!6MqB7vIQbRqorFvgrqM=4D$xVq&Z>+i{V(7n1I_7{F8V+Gw9Y%YWf1)jM7n<'
            '?mPTY|&SeAGU3o!qomQ=RuT1tctYfXsv~zI-2qb6y?<M@vEW<IOXqkXb}^BI>ZbkPAH*mNI-Do#FjVIS4A*V}Pn1@{QQ'
            'x+R-$XeUZ<gPR0@Uxs*V_eP3u)|3}m|hT_O5KOD*B$32P?SXk7BpD$d-'
            'fRmLZ>Vq(BVSj`${$;8bH61k5M;w}V|0{NBu_LLszeAsr7wcz*0A%hm0{+dB5Fww9fA<>T|J;uKZfPVbT@5fSgs?|tfI'
            '2H(#?f|J6is<SGjzYw-wR9VYYQ$+S<nWzl%nvC+8PY(HZR8XDV*Ty#k-'
            'ErsNdezz*v(+CHFli%lqfUH4QCzyWk=0Dk~rd^wb$6D%R*ID~m<4T_ACIfXX+nCduV9lzYj3I#JeuV!`HMy(1FeEYhd3'
            'e!0M@TScz;tbu~*N4R|<i%jqY!Ej>`I7e=U8+HrH9<v4P%|sZuTYgiEur#!k`p#S>^p2=6?j-'
            'WAY|uTyk5bzu*zBwj{4q3_e$dJT3sq~pwAcgh_5Wg>x3U3FK3V+f<cohVoyCv?6z26PgFgEwy)j;Z)?BGH@XshwicTUL'
            'Evr?7IqF$Hj~vi`t2jDZhT*5zs<i!81z{T3<MojmXqNg&o~iT`-Y<9Ipx_TW&)$?o-B-'
            'Z$bG|tJ<~Nz@8pow07NoLXnfffLr2|gq$fNTqES(+&SkEO7&%UfC^GcVh>f~=h>qCmfkv)jH-YnQ_vu~CIvfwATkm}Ss'
            'JFuPj3$K2AgR0pxvNy3DFZ9_#2;6{y-'
            'Xwg=Vb6+SUIH&WN3=Y^M83I*sy(*&h&bUYp=un|4}!owDtPcA7mRLCq6<9au|%i^%4+q<vb&2_JFDy<c3(7@tWQJv))k'
            'oielDBWuLCy4*<-'
            '8S6l&(Kre`L;Vn>7}HjO5L!^U)w^R=g&k2YcGhe>G2j6?4^<9KGchKgl5u;&TdV^VA!9<Z|l|9K~Iqk})zZWhHK)>SY!'
            'zJz#B8G`>>N1TpqB-'
            '$Kr@t8>jL~IMgT^*Hlzg04>Q}jT2Z#~A@1JAHoaSIAeZh%4B2~`u0kSSx0+a!0OLG4G7tr900!36KAYtewmp|E1u6}IS'
            'Xgz@wCP~i9;&A65#cTx+9)s$xo@EkzisDJP%pI=qGE`Ta8I|xS`g@8Avf?k$-4BOAP(NMDu2re41NcIf~-Kzw-'
            'YnLFt@+{;%3#QIP=g}?p3N~HZ4S6lj<YnbiFuP<AeH&LWn;V7M^ECrt?4TJ96b*&%=azy-Ts@0NWD%`1?8YleOYkhjBa'
            'zI(QwmJ<R&PWY|HyiCVI`oRuL}2CM|jn)0GyWufMduLIfECl%`>L)M_d3LUe`&76h9G@bt$+&UlZm&O9X|E5g7g?KrWV'
            'KP@R)!Nx<RQ*$yX%UNPg?q<<EkCa!1yyq5_#yymE;h+R_su*r-Fo!rJeb$TmENM~cgSS!lwy~TTupD-)gkMU-'
            'v81Bd*I=KPLTus2Qf|(S?QfY|U9uOFFz<BOi7LaUFnaA4!{;~2PDO3%2oD6ZVX$*)u6tJGOgisBg$FTf^B<qkx4?O6Lh'
            'cuAJztOe0bzT9K{xOH-o))5#VFZhP95895l%b_kh}tt^;L7I?TUse3zAXdJF-'
            '6uRgTD+LzTXUwgAMp8KMb7974W;qImqc|;;#HUa)7gdEZ@%{Jq_|yyGI-hR9@1?DjisXK9C)GkWNg_fzdpD#4V-'
            'Zex;cHP(O^Jna0S(O4w$rM+X}a#VREsds=OlXYHW<M!vGp&eITia33s9c+1G})j{BIF4DJjLrc9{8ltFxo20sl^`fUx='
            '^=!7dcCl~^a%8Q?WOIua;VpTh;(juhU&31c%eRuFgCr#k{t%5u0a%EXFWqx?JOK@=^~66X$*Z>38ObsQK)1YY>LrBE_G'
            'kXSZ|6)E8S`QeJwoBEN88-v0*MXnTKzTBC)Kx8|I(L#qZVmxJ}0oK%o<!awy~FLL)Sem7-'
            'gl)3B(<5&sr9(s2DraJ6ow{#Q=Z5|JN_yJ-(0;uQyO^gM{!LK868um-'
            ';iu7tSB`$Wzx95;<+(&1zItj&9Ji8EP875DPO+~kzmM+BKE4Xj?SVOp4V9cv19q0--D*!{%_HO_v8O&(oj*NXROdFwMC'
            '9}I=C;*G$6rW)^7UBL@u-'
            'SkW8H+bCflSY^<A(#Kg;%M<#=>Fmj@M@esj0nfXP<1%?EtRahbq2jQ3gg|r2r8TH!dNx;4Gc&Ls@`0o4P2+T!@#%!cok'
            '<*tCK^l6*uZ3Rn-_r(@&C}`ZL&JXo;V!{oqz~9o{d$iicnKLh-'
            'I6XuK*9uAPm?vcDN<(YPF%9($pEsX12qPLT!+3cD=sk^0k~#JBzw)k*mW-hFH6vWMfuuU-`n9k<8T-'
            '6_Z^6N@VE>OhmvfNpTIBx0W}AotX1To6@{yF0&7*l`Wr*CjH`=6xnL^3@o0&>TjpI8@yOoH3<o1<Ue_CyAK1NcCM%3A{'
            'g9M#3{QX*bt#tQ(VohPL&nFD?saamTQg4C24aVUjkRogYv5V#%sAAUm5wlAE8y{7cFhEw&4dryin1K@cX0t%6JaLH~!f'
            '^M2>Tjrw@WD3Mfzl!Pd&GV{5Q%!o)*GNUacN}^$BZ`pf~tZY8_`3NPHjI4%?qOHB_+tc;@3D5QXcK>>=^FHtMdjD{)8!'
            'L+&nYEpfWZ9=r8c#mHz?>u8tX45Q2ypj;E0a%{!r}I$P~a>){d$LdTlpIetm7ah^EEJD&5|uVS5fhVJRW$@gEw?Jz-PA'
            '&Y-(o*tCvNj890fjeo@m0DK%6vBapD99S-'
            '=jq3O$wD5`KAPN}P))NLtXDGU>%{37IUDg~|oDnM#<FUdD*#N#BDhRUnpz5PHlD_+uq)HZOc^n}Riov1t%KtAP3G3PIq'
            '<4u+@e0;J4MrIv}+=ec?zblCzd+?9@E)J6)+EQd=iY0yyOoC7cEwIpHg2|x}F!)?eyxMJ1Csqa*+c=OUn_7J7*@Yae-'
            '(<?z8Q8@Ug*B4lWfdin9KXnx%#}cP2*dWKR9JSmki>_xX|iqAB4c@GkfFSi?lE@3b&UsceP<_45h<b8){UUE*AO{o-'
            'jVaq8T6V)5l$(Vpi1{@@UWEuUL!H6lrq92;|{p4*B^H<cM^eukBq-'
            'k#VGr<jBeyx!rHSh6&^66;m?*pc<F8l$%jnHbInrv<8>s~TW$sx+Z@=7O+)m;b`ZM30gt@;vCs1`k-'
            'P7Ue<b|yNa`g0zF{2{`u2m}gI+3V|Bd{b93X;Pi^$8Cd@5OfgLHH|LUBU?xm;hP(HXP+A6q}5(WXdBuZ7_J<uv@c@CtD'
            'fti<|{n&<(lSaY8X1*h+TjC2plzhn*TkKH96hP(utD4agzh(A1<(OXyuMx@k1+>aXv*KfklCO6Pqn-'
            'Aq**JF&PBaty{hT&7S)W>ZJD#`Lv@oIDCwo4YcU8)3qtpnhTR3361cuL-WEP_293-'
            'E=>9<23B18cG!^V1%JiPJLHiGV098rlGR&I-'
            'cFM*%WYng{`YNL)mv$g?BUWW}jj;@Vh%5q*wu@A@yWuouNBr#8lUyM3U`5wLxG)(=|7a|d5M<_88(JuaUM1}p9{n32sT'
            '7Ppq7+NS-eu6_@S?XDBLbP*`Mw82{^W9U`|7V39D1k(Zne$Q*s>yIODD)<gYuGa7{c0YD@1feZHfD@i?=|kyclsOcPhF'
            'c!fW(yhm)bBb~)oKCurXcv$XoQ;>lrlDjF#I6~5x3(`D>GKjzDP{EnLvx&*;teHTcF@oG|XR{Wmub@g~1mu!Nxu3pN|m'
            'iXgmoU0)Eq!gymStn+>Z!e8tiXZH9VF3L3Fnp~4=ZPXty#&-'
            'cUh)RCv~Ahw&__@PBuYYM>c^c%*<kbSsLRFoJyPLWU7`mu2=taUa|4|ct(Lr&9`%=4AJnoq4Fq4gyXNCo%7S!YSuv7iy'
            '=JenEBiTt>!o}UUksL+)XFQ`Zf8&yqO$P!YUB@4U4vASgs7S35>pm_y(R{Rm-!kh8OCpQpZ-iaXt)s)NF5T1IRAw467v'
            '|F42_whm^tXU0SZMy-l7_;7{9!BYH*RXt5C4Rg7yk&t-'
            '4P0|efiG9q;9Qp`P7JX@$+6|g$rFcOjy%9tkc;0+J6g|qE5N$EVE7d|j&j|(&|JI(HgpMVUdTf5ysAX=br|^i^K$IBe?'
            '_<M*@m@h+sGSlXRNlJqGBrSEK{?4u<AoJ{#5CMGYxORID4693C9TWGq8e8tppO%{s=?=_~FIs@AUgCUA(l{n=W?s!Fm5'
            'pm{em1hnMUDL(?X@*f>t3QF)%OxZwf9X7%u+Yy-83_(YX#zJr<UC}ghahcjc@$ku-mf8@!7(5`WaXi0*1T@u8R-'
            'x6!B)ToxPB=~9fQQaDLSdnfItxIj;%o<DZI}i&Fh7~A(`2uDQa^t<<DY&*O3%8GIlkmM!|1w+}=trbt)yQp}klqNU3ZD'
            '4uy)fPlki!Eq-1OUdL{BLrc&tB8-'
            '&xzh*g_VLm7WE2g%!B?P6gBzJplKb5xmcS7gLi}@jC~X=KWHAdgtg6S$g*{YB&a=c~T-'
            'Aj5r9peW&2VniDu+B?Heg<u#`lCK6{2We|KkfPO<>xb5~W;2mmWqz1X-'
            'F25C;7xlSelV~F3_x!|z?~}kP>^5p^1Ot6A!jO*ZMa!XM(EdmXI!afdL)`-'
            'GxP*|pHLo?;AP3jTok3rVcZ6r$0{uPiV21EKomzPrZ68ak_n%3J;*3Bn7h&R24o{SH5kp;xUM$@qN{(tyP=&;LWGJy<z'
            'Ti3N+6myjm8rP#iYgeB>-3jSGVb-'
            ';NN+qX#g`f}RL^5KX7Lv?U(Co6r$vi!@FpK@R=bDUU#>yxsr5|Bp;lZKVZaFVRKmShvXEFeP7TJ5QF#T$MK(vk$bSjUX'
            'm7?^F+Ns^(<;nLRHo(^#WiQdV~O<LD&YRni4vnZFlH}>wQNN=_|p?#3wAQCTE2p4!ez2itrycSuGO4+mj|z32I8%n8K@'
            '}wfbqL?>1Uo>Bv-'
            '=*EN=ARs^xEpcwa1W7?Fon4cBQ?LkdoPcteHfo<ZFKb_lWm3G0?}gTmhRtid~;(4)SA=4Gvdk0GI?A=U=p$I3%TT_gIg'
            '7{Pa}-6-U=gi%&7%=AsU1c#dn@nS{}%+K5h8=18%r&Ft-tiGO#<t)WZ)7_Y{@i*D;?MrVTu!reyr7$_r40EC<P$=>x`i'
            '0oTSh^79=MTeEW7Bk2=mbPtb&%U!cbWCG4OG7R78YIOCeOOM!8FDjRvH-LJ~2B~vI>BO&F=W+z*53GZ~@Ev*|D-'
            'tAKnE#r_pcOS%PO0py~Jnye?jWcIBa{z1#vttwu3Tx)Ibge^ArRm#|?^35fLyK_q7mxGJ(ifl@yj$Q5GId?<RauYud!<'
            'DmF&GI9on;+aj^$k%@ueWlr0xo?7TA^Q-vZv8-'
            '|dWzr^|5s+;rFsn8nTMTsYpI+nlRkPIfh742`hV?Wo)d2(;&KVNQ?M9ZC4Pch@I?quHo}i1d8AoC107YxVd~^A$o?U`e'
            'fZ*ayr=V<7)~rlab^+-W=Q~-O*8S_^NmJyCE(Df4EXe+lp$j5i9Gf1DJR2{itTn{)b^-'
            'hVOBL>J98V?CY^?Rg7MhVr3XxFZ@Ax_N6YvsNgKBv96XbSqo*zL&(Ax=y+4InSGyk<Zcf8$1y$1U`ziRDNiiQT3x{^P0'
            'q#HTBg8eA9I$ev&-mrA^WIN}Pf`Fxo{q+2nR;-'
            'bZ8zqH&ckk#NowWb56afwn4LDj^ofztR5?*ZHk@1zYp%9HU;Z?6wbXBTY1zP_elG!>;vvQ?9N&wDBZIP`eWnv`wRldxr'
            'X}P0wEOhyxC>@}U<Z%9CV0wafO|aew1)OhA@j#&EP3UL6ZXyU+*BTv*6M-'
            '>ml5mLRW0~oyo+Q`@Uku(rKn$HK+}a^!CHe*c>Pb4&NoFhx$`rqa34|ur~`#nuGq5u4zjD=p$=}{B=nFbxK-'
            'y;L8DA2cY_b<%T6F3N=b0uFCLZ(%wqSwo1lKS8|KGFHO$T?;kcI+Lm)DOhSe&erQ?1y)8*E@(EUN9$FBq8J;b5h#}`kG'
            'Btaw_(&ZVIsQD)d{_^ew+qrZwZoPrGjME@gY7=mAl!0FhKjC}MP3O~Nup_AnFI2ZeNMIV{!PZfD=4Vb-Qe%jOSvEAT$%'
            'Zc%nowxAobghi42^7-'
            'k{M2a$O<pPHR6})Rx?+^C>$f^2d{%_f;da4MGh}zB%s{bKAg49gr{|DVdU!!Ugul_PGdjFqxUI{<^6WRF?j<;)<3~ByT'
            ')mYa48l`50S8`dTdvXCu8PW@KDnVga7T~x7WjAqqQW}7|h4D&%aWyv3cN74#qpXDoL7N5EZf#gvcLLw0(CZ)G<V9Zg@P'
            'j1#5z%-Y(p=LLdE?B*A8#R18$A0-v{eq--J`%ym8RkqjI4Nm+<Y4<+Jlt%rwtL-C+|8?NZtjdM!_asT9cP3`#{hI-yG8'
            '4GSfy%rA5JH~Es`8tQD%nc6`{bLvpNA}^Q{WxsC9}X*E4=ZtFIlNOkh?ZOZ&?An6^>n*D5d&@<%Q+4D_juuu&vz1gXd0'
            'h)?IxR)?}A(FXU2`qZy8lf1U1{@^g!htMIJ?e!Z^ASWj8uwnhGF8#0?sE>|~7Zn5Q)ny^Ps+M?sI*97^>C;N50#th%U&'
            '78i0g?ym}Eyfv|hij|6Z@#_>hJTy-)iEc+$P!7FUYR*vl6~mPH+X~VAhvC>mJ9v(Yu&yrvyu$Lpcgtcl6ncR}SrJq-'
            'x(>1|xmndtSvdJOgy^Zw5Yydt^jdZlgpW>QyrCGF)YibH_af?ZAsbx(q++Li7}VGABG#izVI<WMm!?=Tz3OG4I4}aQtl'
            't2=9+~)2SPZR{H$r&NE%MQR4xay9iBCS>M5um79wwzT+t&Og{oL`$0h`d*R|Jo855dFm0BUR^2y3mUFe0H2tX>tt0hv{'
            'rxy!F1GfW0^7unJAZChwzYdt8n6k@|M8#?l<fvJ7o8O*&@&{aj8jEwhV;EnAtXebB=*|vkTSO{DaA-KUl22yNq;VI4p9'
            '4{^+9nQPy7p5-'
            'u+&YL1cNJ88JA*FAJ@MVcc;E}U3P!o<P^tZzUUV*k^coN5tW`gAyeu5VTamHXwH20KzX%QwY9Uubhjx^e(m$WracNHyY'
            '?BtlC1Kt;aAg@Q^(1?%IJ{$AR18MRV((VF!kx51`5()KYyv))|8V>F4>0P($K<Tbq8jnj^y0`Gs5)7Lucvr4E&NYXr@y'
            'f{aqcB9l}txzzE|MF#-(xc=pOj;C>n(0nqb|h!<b`MPWuAyV+ovQ2;{s&Mz9H5jy@+pzG#y2FY@%=_a`Xe+C~(oZ-FZJ'
            'Vtm&KXxf~?c&+0Msq;DXSCtL+hChS$LoT?yVKqHs5rYS_x5DLPA|yj?0`-'
            '!RL@co*H}1uNniGenhh`+~G0B2}JKU^aI1Y6mg+O9o5ia=p4p;n^WF$=8gmsJiQ83$wZn0t~6DQt)*0GyZbn^qc^GQ1L'
            '-rGjy-tB@VPq|^`hz;0no@H>Kt0TX+MB$-~amY8i2P^k4(A042!Mm<}RAOKOEBCYu4L$aPdE6`pMR{hh?vM-'
            'I8}0)7I0ySvf*@?F8X7~Jh@txt5ZhWsmK<uuMSag&?f>hhO(NqM*J}q?FD14-ovo&On%-'
            'd0zkIYR;tY<wypHz++92|L9!8tQVl>3U`(;3f*qTv#Umi))QNUZgS0F(?87}`?LvtTIB*SvvR3l^<FC3`?T%QU#)(m)J'
            '#H4QKEzEF_C6MBriR=Dx&vIQuEOk8(pJs9)rIZ0`_sU^eNIZ&~n?gpwYuG>fFVmd2#9v$spl&Q3cFa1`SByDy+!qgi?u'
            '%I8q&5-xdmFJ~FB5MKN1^TQkEkUU0I7>s!-'
            '0pZLF7XO+?(1?gEGy)Wa}i^^+6ANeVU;_?>qK&M1z$?AnfJ#fV6f`hF`n`q&sWCQ3H2ux37kxln6K^C5}gL@5Z>;P^g{'
            'a#Rd1D(0vBJaB$5Eyltb0QBx(P#;l1r%m#v^Z!jq6-=W3=<&f&ugm<59!oK_i@S)ip6%<#Zfld)baoxn|DSy<Iio?|-'
            'A^42<A=4<d3Va7&F$~5<uyL&-'
            'JU?O!`Ffe~fcq966E$VTea?fp_Cx4#co*CcPeC2A4t&?&4O2CnA#=4X%Q~eIRTp!h!JQ2-'
            ';1omdRnJn}Gv_e5)E6b4Kg0d;U>G&(Xl^sUN1sHoNT0<qU_6|GKdoZWF%*l6*OxM;Mw{`Gz76q^d;|{K9PnjLFbeQTU{'
            '~pHniO=BVK<vejIOe%+;kS6L^f6gu|=_hc-'
            'a0gfTRU4)O43WscwJZ1f*3w!Q%A;;Hv(JF4vfZ%q?+bxyl~+;4^|higYob>lt06R6>f_CxKUo8`u842^B3~u-'
            'xPnM)pM!whVJ%Zpo+bs+FN&Q95?2wxipaAn+DtwJLf9(eRW1G1-'
            '(2@u{^q?D!lHzj)YNr?sEs_qSQF>TL&^ep`>TS$vpp!wuD!oXDQk4Dx%WCP+9O$7Stdu-'
            '4rT7n;VP@&;WZAGQj)1O@33$0A%O`w_JpUbR|Qm0**mI3y(5z{3ngJHr5!E%JxI(l>BU%N+`%6p^9Zi#lUA(9;u#(u(8'
            'Ad*BFcQp$zOuxfH>MLisN^_9q33&VlB4|MsZ1TeO0hQXqG<OOpSG5kV>rLV(s+dH60wcwAOJgmGt01J#(<8GM^nxpxP('
            '9z2WkFPn3H_tXwu_6zc*i?r**NMT%R6oqfuYp^+qL5Z^h1K!(m^;7^k#90JI8szGRQC)NJk-P0)$%ARH-'
            'nmPb<m@~8<#N(NoJ)Vqzp6wmun3f$yi5Y(o2cx$&;|ISsR|D@}NoCG|Arf7>@+&fd`8V27fQ4XG9SyVJoy-gyWD$6pXg'
            '6#<x|k(A`u91i0JCV%^)oRE<RQW7+ufN+ejhr=yWh4GOiiqT+Bg<Hiw9Jj-'
            ';4AGvXKlUO^IO?eCo_hzwm{U=(#z!|MP1F7w<<M=J&1bp=PPs7965k~eb0fEONVC{DiM&^<s_n8s=EbIc)`Yv$a<OTc@'
            '%fZv33!}33qi?<hOz+TTT<(a&6a4!i{dz2Z4v$1rwiL9<jl<rEWx&nX0Atq2$s5%b`1i&`V&tR%I|q4L%SHZiQu8y$qA'
            '!miVK9ZP{%Ol7iCxUv)kBF|pFb>kTMU9!9>pI2$BcYG2U;;|__gFay4|V7t6eYPXv|O8QT_;=yHoMVZ7Y1vwS#WocOHf'
            'NcjC%TPswPJ5ZaYV!`I^Vc-'
            '*Fn7EH&1#84W{rhK4BWn)p>Gz@o5D&Qx{C8+<U4SX(_AisM#x>Y`fRb3}B>r*6YHXSC>n{VM_fj}^n)Pl&)8*u(f4zQI'
            '?0Ao)BL`c`rbFaF{Lhb67)EZ3){QQ!BI<=IQy>~aR-20b~ix*JV-'
            'yAsostZHk=7Ohz5Dfc7<A3KiK;un-r1$oC>aHrDNpYeVhI-KbRyXk|y$=df!dO}wgL)bi*?$&N>(yR3-'
            '<(ZG9d8hmWJ|1H^^>8m%}JD&`cUqTaky%F2bgud13dfzES0x`QHDB7{(M6B%+!GX-ATOg`2{7(tN-'
            '!tZsND}0NxU6qXGIQXmjfw+KE>YtN7Lb^dzLo>`#Lli?yh`uaEq6jDj<&kD=>?2ox^7OVk2}X!70vU<qdk2(F0%p7cFn'
            'W86;~L<PaZJp@9_7h_-wJ9<6vgo=t!RK;yK>WXjE<ln5oDlzn7s>%((R`$&>ekc=tgY+o>+yLGUF$Zn`_1MN?fLG3h!j'
            '=gUW*f&Qkp5K!137H)ggpUr=3AiW^Lq#o_k`&`LsX$m6tCTF094V{ke-'
            '+z&i?!0Sne#!5LIkwQpCKp<zOo^h>GhaVCruweX{2|(=z7+36zS$X(?lJJj(#~e8_?BC?4q4_>DO(9H^Bjhc>c8C}iyf'
            'Hw(idsa^`NU*x8<x{@HP985LMw3xOVsxfP(7yhPyK&?$%@w;Oia{99a|F{aeY!lY}kiG~UW-DQeaRqPe7lt?I6XCYbW@'
            '>199R?kQQDE}H_8Ub7tsV|Pn5WMHBUW7se6CC3@$+9%_2UGUS(-'
            'q}qC8MP=T8@V+(M4*ry3>>)zqt=1u_@w@k>uHUEmdsD*J9jXwxu!Qn^U|ZND%KUi5&hvk=^o^TwYcWsto48S?wag663S'
            '{3f*wBzoOQ?TZxZ&?QWk3xxk;Y9?-)k|f=A{CL)-5lt-'
            '8;jYs$?EG7S!fMJ8@Ma#y%|h_tf!$2@*i7;vfJwh?slfLxYl%{22p-'
            'RMg|8eJp#7{5+!&R_r>*zFK3)k|&Q_p8`cgdNr3OPApTJd~Lb_{BDKwANkb_J%R&$>g9_bcm@f8K)*+eTMaYl$G`SUh5'
            'ig7^$<HGiP4M)LFItd;)UBV%~4J2<zB51#l#kV5g$bs3N_<WF8b9}s+wmgZ#-_x9$tgW>5(%08?-7ZTMU!zV-'
            'Te48b3khedfTp5E5J2mCJmYQ(YD*H}h1P8dF<Zh?=eEM)lwxW<UWqYU$!IH9ihrx#qI&ds^iLQ2mkX-Mj<l1YWxonjn>'
            '4}iSP+)Z2(uhV^l-'
            'oB0@mrHw{Yn(1%}$PW!uHW{xasv^vNqDHjsI`01r7lf&y+i<}a0{_|E?~`fGm0;^qG`NX9fQX5GSQ@z2E8&4tVwy#^OS'
            'HO+jT0eJDE8D_8FhM*7ikl%J6d`j8TK++x+!$;}E+Xz>+wV*j}2W;KH1ccojP+Ce0e{TsOzpU2Lo6j=w+sk95Lcf@r$5'
            'h~&sU5hFw+3D%*Mbu))RZ`0L6250)m$$W0m}wXK-an)=EH*w#u1lb@V7`H$M-'
            '0Ju&WC^j_4&?fBTvDM8)t+`zria)Bs)Yhta7x7EMxkK<{TDB=dUVP02>=PF7*EAP7%v>!MpTM9Epl6s+`qO>SP%LL2$p'
            '<XP@Ge0Uv?Uktq=YS}C%#B>9@ehc%$2tT-Ts6$Oo4SsnMj@^$mh^_GmEE?Ge7hjmcF%M5XCV2<*-#BB@iS=;Y@-'
            'SX(h`@o<Da_ldR(N#lBF(!omGJma3@mW}k4|2##d;PeejAL#YkW+|y@v3qjf?y>9l}3JSFq^V8G7$XE;go>W9#cXSYs&'
            'yYY%oY6^|EV@N;gEJHH;6*_1=8TLL=xnnUfNFXiu20;>2GV<yXS`G!z(;ZGfr8T(9=&zE6Gek>ks-'
            '%0zE>d5*<oXlX^k5F~M2yb06f?(k>2>mn;hpiKVr!ok`4hw)wyojdoUfX|l@iXn#X14~|bmErLaxCbz$FuYS*<^H|{4i'
            '=o-Mkn^%ts;o5Ofj35~c7$U;=HCUPCKA^`NZ3jQr&Zf$!-F5IB^D746~hV$9-'
            'StwY2~A{rO!vTO1@PKAbVj!>u<fdBD#&=SR!tVO<Q<ju@J>`&bSKmK{1XW$HpF*t=rpTdw`P6$p17l6ydRtP$L0&;Xjh'
            '*PQ>6lj=Y!cAqI7%D>E23d5ZyJ$qmCQYez2H4K1$C_<U(3UlijPq=)mP-'
            'n(ZR}55cc+KKV&VO;i#gJ|PB9TW_r?L|!p9iz%FPP;_5;niqtIwln6=401I2w3(WRK;p5Z+F!5c%psRHCRo`pl*<!E{>'
            'AJ*;>hRn1f2;6WV_<Dxnh~-'
            'U;+j0o<H#`GyW#COUq$$sCAmeH@P90?8>GRT%5Of8rA7sFkjsjET^#C@vwlLf467Y0QBSZ((f#ewr?AFU7J(kD7OezN-'
            'YhQ+!%i^#_)B|~IlQC07pw(X32lwK3%ykwe6<W)&H^>0*hZ=%RZ6Vsb9R{x4X85!83VFlN3n^LBP}&&=X5mIq)4Lx1nc'
            'DcFxSdLx1T&**guy!DAk6zmLz^d)mJH=%`mwvrNZEEU|C)|EqQ-ddyeb@v;{})T<9Pa)2af0|L%o(R7WuFs|Bf8RcuoA'
            'z_Z$@czJ`|Hqwt!2Dg3n3hNqlH5R(6qj0E)3gOTsae8C|)F&%>Aib_!CF@S|><zT+C2^70?QD86&WM>aU(6UsVF%N>TE'
            '$cz#YzLW;JOi%WDVTlwGkvf74pvHeV*ojZ#(}<AV;g`fd@I4g^C0ZwSPH9VUW2||3H3DOz>ubf#>=f^WQunVM{@qqRN+'
            '`s&)Yz?Rw!X}hbFvs3vP8kCW-YQR8dy24aj9%n$EDsj9@1En%FZJCpTl>nNP&&%?a2aoQN3*-'
            ';wmzg~*+^1Dp*g1iVj!tNTpoh4!mpb1RM7ZqWl7?PK8kR)(p$Edr0fT8DM2<s^99CD3UXhMIws^wGQzIvWdtj$bLnS(m'
            '^Pe|6@frX!H{ybrwuCJETd!L?2vln=fJmXS61Ie3W17Y5;sp*l#pDnRPeN?24}j8}wRnaSRsXp8N*Wk?x^eHzJRr83Aa'
            '3WBsgMfmE=B-<wMFtyIyYT0<InM9bg(=TbUu;F%Ci($SiFju5P?*R@}m;XRs#>nE2R11vbzDrhbegqRXn_#u-'
            'E3&XC9sB}UQGSI+2%NkN!8xV$y!C!6RsbmRUJaiuQG${iBXD`iIqXtMW>i00iH;oV80V0Vhl{tO_f8Sy&iuiU<gO;6Cz'
            '6P4zcn<TssY~DU9eOq9S*Pp(fnf$h^#w;12xqco3H?f=RDv=RT41#Qt@l=Q@s3!9ilULKtve>pG04!`vS9w+iq8guaX9'
            'DqXFi&ZZq8Q`#6*578BKtn(0bmE^uRXk`J-'
            '3sQ<1({I>A`Vf!&ivxWTW!ngLgd3+62+f_iq(HKlMJBZPnIjHev6>9KDnlY2u1Xoj;)UMhBY}n*UaPBF}e{u_St;{8N*'
            '|X4_vx~R}RKw&!4$UVW#?-'
            '`O5Oc#1;Ny^b)KBPPPVMF=HJ1No^TJ=~?r(;_&VKmt{%*8*QcAb^O5vX(Jp^NO7_qlS<>G&ALrRIEg(UjT{2{s79pt<m'
            'AD;Xb!&vW1L1BItv#wRanSWYc@x~D4%AS$_;~fz6_7~;8r%ZT0J|pAJt06agf_f~iq<cGe1E<k8;=gbw{FmlWHos=UwD'
            '(;&FLMvrRqe^^<7!}OpoFv4$LOVhY_)ixM<e_7X~2(8pdrBzd2OP^q9XZ!|F7|}-'
            '=6n>xS!az<^SP+Li^MtNqqDg*o7}s$AK8wTd0Kpv9DxRF@6I#M;urcEYy^8DIphk$%1Kn1nz5&p=rC0(CreIL^0C^-'
            'vwpk?|2reS<^&zwWXu#iAwq@E0Mk|+J_k<79e5uh7A1DXE^;ni>JT+WBr;H*uGUkbDhN@c>YQOe`B<UV(fi-P-'
            'p?GA^9xYg;kMPvE3*<%!i!3aj?>lVusg2eBCTSq>C46PUt2A$809HjYYuBp)r`|-v;ghyvXO)i0$9o;h?-'
            'F9+5Z+kKajXPN!`kQ#oTmUlgN<ct2yy=3cyd)D)&{?qKNgADEUnfn_Ww9P8AF?HVfZyYfD)9i1S1r&g11ab7rZ{Q|b6G'
            '=t6Av#9o34L#D0ao1jPSp68O0Oxn`dH9L}pR>poc0ufR{z_%9)gfd=LW<8~>Z|Ptk<%|B>)mshl|Biao7?EQpttyZpBK'
            'KKQ3UN@7pP9nCdRg}lkj!?Au^WElj_6)2-v%w=t!qSVw^l#(y5K%Wg4I?cwM7XRvE|}27LF-'
            'gacDyBxbn>vrIl4yVF`|_HlO{2`vK#&nPi<RK!nCu^2w91Bd>5LwhqsK-X_A&W^`p$?$qGHp&4}w|XoJ9>$%nl`V%8|1'
            'iD|&cf~CgZMY^4fC`#g4|;X=$u_jr?+*%+6pn0+47onTjqgS+j7hhDPWpfYhbn7BjRCN1jFh{fY+z!-'
            'eCzGD2)Sgu}iS2Vu5C+$2%&ktw-ODl|bex87j|L4EtUv!Be4wFud6e7HsiH?}P}N@XP{tc%Oqe3)mns{~xdANHKS}Kc*'
            '!~+?v|YcVe8WIvTgEfk2t-XtvW4YUJyfp2uH9t8o)}J>X%*$W73in-'
            '#5+`k$CSI<*@1cN0)0^AmdCOoltjB`D+^+fvyQ4<|R8z}<zHa4NSHTvZ=|VdZ_=oVy)LJq$oC*qtsK3LpiGSJGep!9<P'
            '2hnkx&f}z79jC!AMWcL?6a`Z+Dl@+@I!Qzj|Mv*pny_pSy`?{GEim@Qpxe11sS7NVm0K*|~C7IN@PjmH)=x(V@=-'
            'jOjQW38(ppyxgRsWKss|!&`{t@aq2r$2P4`Ju~M+_Z_t$5{40*SNhW#$DoV$qia7$I5+Q^mz(a(6OW#WqGp+wq@nq;Mj'
            '>K_fX&4d*cj$kBXM*2yAfcpLj#Mwv^_Y$Ss3^T4b8Fq8~lBvY@>;u~Qh*s!b<%6dxh*eyMr;R?ZLD@q8{Fb=mn&tiv0D'
            'p{9Wj%QE&>(5aaxc@Q<lRjNT-Nqu?XH^J%|8hskU?H)q&BEHoe?3T#LLD6q7-'
            'ikSAk9ZuyIKx<1SCoM`;$1~l?(M3E)!>NRqVOOK!qnA)R?&)mMDbcK2|pQu_+KjI1?Z|vKXlG2m0vhbMU|223EZ>)Jd&'
            '_ig&Ic<yU0UX=^pE6Jf(o6%J5ebebr<l|X(A1g;%7DL^!}>j(VaT#C;&t%Op=T<TmpP1ClY#n(C&<l}`hT%*4TLth1xH'
            'OiB)ldBXTuT3N&-'
            '+M7CCkff2ZXmCVJ54F)0X8KbQgS>8YzIbZhNBBij9tQ>flu^%mpA4b2QvDXynu{bBe=Km1Rnm`hFg+9lJ+(7$oKR*^Ce'
            'p&6d0Gla*Jr{yUzu0XAPh?S05;SmqP{dSw`J#5tynvQIUs#8Q<n_V4PJCxxP!3#2BaG3DX$Lccu?r-'
            '3#!(?|zKWzXacA??FSg^7g=ejToAn3VmmmVb6(nqVE*I<hxr6$Hmh?-SiUd*X)8f`-'
            '@1))^1`Smr9uj2z|IJZu^9x8z!A=goa=4gn7S-IUkjY*Q3JWz3N-)cq$*FHy6U4n-@TXb041kZ-'
            '!~y6b3K*lZfE9A2>L%6+fhM5!H%#xModhEnhNJor;4iS={LLe3UE-QirF(ga|tIlc__&uqL1s9?uBC(e{JjzFHjwDhAN'
            ';;3l%ewF~V&zo2E}Rv=b<78&OE$nF2i!RX5fY*?@s{~L29&q^1gA!NdMW*$1m>LK$NMd@7*DCv|7liUDTKCY&`E6N$sc'
            '29`+w@^6gdX2Ug1!>$<{!C+jFmPR9IH?>v2j}<Rr-etlP(QexY2cs)(^WDM6cLQFPcFdEWe4HPCROrY#U48>oN=I_3d9'
            'm$Qk}pMFj!s(Yvm(gOGhCVAJ~J<SCV0svMJ8(b4B}WY&4<PhJN5`BG%k}P#a)^@A&VLXp=!wbZ~?mUlQ87d*uz1X#SRb'
            '_cj2I%f@iXQx%)P7elaQ9U29vg6)tD9Cf~iC5wMi**q?+R2PD$Z~jA@3<a<;PNH2)wu9eVdmM?^hv39hXpwXrd|!oNSJ'
            '75x*!U3XZjZsiYk?qbSBE}puYzK92k3q`B8#1tfH*S;YZKzg-p&{@zr~-'
            'acsU>I!?of0xFL2Fromj*S&VuehkqNB!TL)Av_Uegc@mHN8AoBhC=WIVaX|T@VN$A}Ltr+7s$Duj5`H*>hj$cNX|fra@'
            '0?NF`~*P75*+$=wPn}cJ`AoDC)#aU&|e;ldoyN0@<j+U^iL|Bb8`c=GnJs-'
            '7ev|QACaxUbC@TRb<kJV6cvPIAd2P7%)e?6o*xV`T;VbKvOSL}qZR_8XKoYM7pqaacpZJSu9nz5`$K*`8HUYwYGCc7;~'
            '*V7O4J`M00ka3jNj*hEAQmsN?jh@{4J97b>$FOHVx|X^(TBhyMg&5fq<lq8<nbxMLC{qY;d?k+vy3yJ&}W2_hQMPk{0H'
            'Cmlg2*-Gcj=YcO0#6(%=uK$ZF~%5&BVrAstm<XQqoY}ra)%X>3rd5TcqB^ive>Z$hX7`#&zkKSJ1)c%7P2-'
            'g3mhkhQ%?Pfi+O;wt4;hY#m1z8cnu5>7l6r%NF-S|gV7q$#;gp<)va4Tmu?kRo@e38fT=LF(vJzM-HyA|HP4JB*d<>Tj'
            'PE+qPU6%MSeN8vvWBtAQXHr~DpYreduOSVw(y+_c%F$5$Bw!<mbPI_mf0DNZk!na=tX7=fDuA>D*w3E?GWf*ETw?Vhq0'
            'Fm1D6kH~oA*NCgDtt?Ew4)SeI$M$FH$Mq6z6gu#rGPw5M~lo2z&f7AoPXSo31%u#EFTFm7t_ecRZ3{@EQ~c1x|r-lLB-'
            '(;DI<#*7i`MmsSyY5o@Sz9a~O`@>!FEXo-'
            '#9I^68Srb)a}<1Ptxe&|RbpWOkjTV{6p#VDwdHqT*L_;L&+_l`Df9<JtICf1bt!{B14d=_Kh1Wt2Be9X_sbA(jh1gJ;Y'
            'L#SahZ2~jbU5gvoSu8Z+Lk1aTHW`SPpFbes&5-'
            'sTp{07pn&uy08?3W;qgN_k7XHlqoeHgNiB!kPMO#GQEh57cC#L9CJTW{6Euk;z*y7MwztIsAm5y>P}@*koxi|EdE7iji'
            'uDeUmL!u(SE&wo{~YD^8Yqx^p*c+u+~Ilp8MYnR7kk7WsJ<W*{%nXZC@HSA#WW&zyxm}hj4eFq7FI9S?nhTM4;3ND8#p'
            '(f=Zo;-'
            '09UYLtQ+Iucd={JMqN5oYKHQb0Dfl<J<Z5lfdN5YlXL=0M?!`xwH3a4M6M)?>O%>DHKA1f<DBx5;Vzp9359?^`~dsjmg'
            'e>vK%{|3>?Coqt&f%)rb8-'
            'xx;(t<Z88t>;s;IZ=_X6_0vWaBeO9^YbY`!q^Lz6s;$Q#C{@y&5vP2SI$@Bu#6(iB1l(SpJ|0dsIy5x}6KjE3QY#p5cp'
            'u%k;?+%`m9Vc3|?v<e=v*QSfx>KpPW1yrx+|^pt`i^4T`#&A=k&aL@%<_Ohg92{Vh_lDvqrjB*$}#*fm>Ff!MA1I^7G@'
            'TKt!w6k5n`1Ij1>8<Q#h6Y-b%4e-'
            'K`$#mhjheu>)qCjhjUYU7))5R8z3G~PL}twKTSQ>;A>myn1iu%kVC<6%^xg0m>bkRp$hGB@c-'
            '6;bNXQTdf1iXOoA@C4zp>W5(O5b*#6+3Sx9~eTjoP)>fx&@1P`^=x+OKaz<JB&jeD^NQ{{6yO9$^U6S9YL;K@48cY^E4'
            'o2#5HpklN<b$f~tu<<AcKqevY5eriDIjUKrD?VS3~OULNKFE!};YypF-Lj^hhvY67T_TXC^jj{HqaZ=_Q#9XR{0~WRPF'
            'i#U?&e`LGvl@h_{VF!GF2T>tc;d8%pUi~ap!%z(N#dX#gvL4nZ(ArDF36(RU8ArwrUZA!+(6D48I0#}hIZE$csmdUZ(h'
            'j55c>)k`HK+tYYHwU1>r{LQ}B5eKNM8TWAShn=xXcXVdec0{z;Xj%6(<bJYZ*7JL%z}6c2JS_y{DauYeqzM2PagN!Siw'
            'gQe0s^p8LyR{Paa&j0$sLybv17$tQ6!UF80MyL@mhze#b{G${L$InG!#fo!ibUcF`m{12DuOqa|_&)ltbi!^AUu>|MLJ'
            'NTc3?<_9V%1`Zl*<90^E+tk<ap~DBVmjv_)7MEjRZrY%ru%8z#h%@Fr70<756K_PGeF0u(^;<?XV=pU&=wJN*mqR`M|q'
            '%?s%gj8ny@U(!$G!uw5>O+U~H1mkxrI{FMVej6r$B`#6zfM?>cL(0L^vOeZEHN4^m$e-'
            'i+w_BxaFAtN~B5CuK2iWo=V_c0!B=puZLja0=f4HVa$09RfW%(z(u4s$!HxBdWpb=(S~DOupkiiV-'
            'u=~g1&h2Cpi;Mo~Kv-WCwKz=t>3yr6yLf(YqMj)D;mBR;o$4Gx(8Q|p#Fgko2j~orBW<EDb`J4dJDi^}i6DwHNI-kii`'
            'EDFTYqVRI1RM4#!Qox6k!9u!4?4UUE63W<yi5-'
            'yqppDQ_gHLB4S***^T>2q2)xnbhCx;sI!6WrE~Idp_X1gOVTbxvnJDQh1kv*pGOL$jGkHw)#rjFq^-'
            'AWq^aM<Gd_XsBGC&!(-'
            '^}xm#?WJ_GTh!RfiZ~}8P+^XV3XhvYm>vs*VEUT&hay(e||MAi%3Q3CwDQw@)WbNsSQO}d&0Zcbu>31gTyMQ;-'
            'XLb(Cglc&bm#+j%yq2Vcx<unX$CyVK_cDw1K(PvCJ@@0^FMQk+~zXj%Y`#qx3Ry8cJ@EJ3V(%=BY1sKN-L;-'
            'K$_!{v>!BX}}SldCH~vj)G@4nsIet=jZ2CsJ4_8Y!ZPWduMQUj>F4*?$l>XIG+3-'
            'ffY)&Fk;|Lln*CRgT81CT%QjCCgB97Lm<;68#9kI;}K^ki0RB@7|g|j9WwwugEe9JITJMwdf<iJqZpi=2?l>VnI4WivF'
            'FTbn4hde<?bYW7F&&tMv=rmuLYvkr!%wN76HGyANFeN;N7*iiO#QJ`205#f{$%t@Z3$p#!_z_RQpPI_-'
            'w|k#aFTTtqt0-uf-pUws5`B7bUi)VsKqAc4Qrb??x}s;*u6z32DOg@IuB9hCBLP%|PCrNQyT^QKb{1Xn(*3wFO1VOTKo'
            '<Be%eS`2h3QDZmc5Y*=Gmh7;kPSo)@ch-t52E-'
            'DH_t;bSWz!3}ysxFwA%>l;}ONp(Z0r;CHf}0d4JaYa?kz7HOE%BtrGm``>$Ivc|9z19;2<KiUp!(fh@VpUB76%z%_vjR'
            'y+_#Zsxx5!GPrk>V*w^4(noLBaR-(h_Da4s-'
            '<SRJ?=SzLCd#@^doIJ_+UN^<KDzTR~8eRcQnJToCa6yIhUr2v*K1vBUfSX<*y55OL-SfNY5lK-reiKSGR*T^St#VL_G6'
            'b{PQ0&v0C5;AsFtx%Od6`XQZ}$sk_liH{yFVYCSE|Cx&Art0hTe85Oo5!hMuyqA5f<f~!8g2xG-QyIo)H|!TYo#rHCPJ'
            '+i(-i4#$0eV5kP-V6&yPB0E*c+;HS29n6pFV{Ud;$Kdv)2-'
            '%kTI_E=;I*x|+YHlTMF!HRGT6uGR7v(Ep~#H@H^`b)royg4QWD&bA{o1nPo1wB3ag!D^!0mkN%4!u7p=Q@Qs?``N2woC'
            'BEPYQO2M$uU5FgR)7hpl%;a9i0BmJi+`Wla^BIR6Nf*e}9SH#QcdZZGq^!$Mpwnhzmw2Vgx3BYAC`;Q5ILoH3V!(0~fU'
            'rGF6@E#HaJAJ#MCKDUx#+3nD?@F49+Hq2hBjk=dl!N+FAy<VbpWUCVSB$J6d%O{BZ<453lGzo13Rud`Z7{ag!z|fg%ly'
            'zc|nQC1JtiAy(Utoui_Xpx{+ZQ;!L=TG=ae`5199(``j9e~W)RLu!g|12T#B>?*lQ`yzxhNX8lMOpcB8ct#Uh3F)iEIl'
            'nq(}bx!jZN(^5Iqu<OXq~*WC$F8ovTB_}}5n(LHcH<1*P5whASyvT=WO7Tk<I2V3U~QK+{aRoxzv@0VxE1+oXuoHoM;&'
            '%E(>Brk5U4@8}Ff$&hGjjBIA0CHi!>3(J+^}Aeyt33ZBpCZjrN`DSM((8C*Qz&HGM1t@ZWn3Q90wM+RbjbS%ge|$!vf9'
            '-Qg9VMDSK}&1uH28tRgX~aLJ)-'
            'H*@J7?6A*3M|8MW@fJ@K1a0{CY6?D?06M{8l^{zQQU3Lo>Yvp3Iz$EbI<k2N+*Ff0N6t{^OLPgp^7<|GD-'
            'I=c#S^wTYJ;YGs4+r)QR-kj0KSXg~$8sfAkoxk3_KS+)_J9z&w?-'
            'PQ&5KBwPbraCu0W}?iEvwm2P5QakSa!F=fec#<ynOyzMVv>vWj|n&w;yJFtG333?@H?373c<I-'
            'li5hLRpFTA2s?tK^_xdPlSA+S@qdkcI(zj<Cfco-'
            'F*cfI0Je1K6*>N3$dWZmuzfL)F7@p~w>2rn@ou5+8o!IRaNt+klm86&RlGgB3?v=pJH+?$TKp@vV!@DO)fbH#R~}O)+I'
            '1zl^+lDJ=cG4xmXGZvWT?;-mX$l;a#>bf#g)U<I)VEr-mV{!p9hL`vV^WXusCFdAD8_OTmKV5SJx%`8AeuQGB`rW`eNP'
            'h)A$UbK0Bh+f~{)*7-'
            'h1eRa<kM6BL3(XI%qOu2Nr07<{hNllPOg0{ae|6zV??HTGe4g$!u7f9Qhv~PL7hpc@4!)eF!18T{3$oI%rQ$g$Xb*+5h'
            '8Ef!W{X}HY`E_7c`SRh5QOIb(!EVJz#RWfm(L8txJe3FPu>G}wrDsd(Sy&YRItF(jhS*I>Yw*cgMRdX%&k26RMGGWvu+'
            '>-{7$aI78_HVr5Fdzx3$16sDY{<4#MO^f~4hQ2<W+g#yf@##_sklXmYZEsb2R2qSce(^ZlP-'
            '=oSZuO5=z|XCAGt4bV94br*gfSO{v)*<fq93g*JC89(+l(7z_ykU27r5~1zr7gWPI_~b8bit_;j)qL!4xQ1Qsn#g|gHl'
            '#QJ9%|shQj-~a_R$&mH1}RZ_Ph-hZA*Y2vu!Xw<OprjN1*sxC`>%nV&3G*p%KeciEeHL@=S4Kh)V^mu3k?rFB~9jnxRm'
            '$RsdEEZU>9K`m|?VH!!AaagFmMh%~+l-dXeb0(!8s)DI8cawg?mRd8yX2E<EtAoFw`QQUSPb0-'
            'gh(s4l$8~8!`SBD^*N*m^HEI@Tu0X1^$g6cXg=J+>%Vmhh}@3w6OMQ3-oTh|B2SS;9jb&LiH<zRYC1<0P-'
            'gZrX0K>hQ6@V}-C-tw2RTRR5}Wm2$RstgauMM3+zGKNJ=2tLXSrQ&1HNoAKLGB}^WyJwDI{x%RScI8v914-'
            '0E>;&cdn*~o3%E(cTIaE$(M~i(I;N0P5nx=0r!O&baaCN+31o9<8@~3v7LVhT;)e9yzx>I*uCtNB`;nU*Vv?u5~`U@_D'
            'g(fX1aXFIg%U{5<uSo<CJ3qMfyAvwkx8efVMtEaPA*1IZ#EblZ?a>opdg&o;_!S6G-2Zv4k{axWSf;t)CD{LhQ*-'
            ')tKAhn4htri_RL8Rwl}2UZz}8_Dl&wUcYyOxtk%D3xmq^{MU^slVgYwu`!8^x9M*hZSAP*sUEGYoje-'
            '5U{#*5$u?>k8A`O4TYzXW;jr^C^80U((X4#By3m?#y93wLs0`Ook4j7t*e=9w_Q1_yxcLMec{a9nXx8JEmP!Gm~CaJwQ'
            '%lRFoYtJH*?FiZrqMIXo`yFlcd|HKF@s06tjS2Q;XqfZxbV+(5r3Qb?*T>WZv7heG`f1|-'
            'wZWDwHUWGp8rLbDG015?VQ6#W}KAi5s=7$9co)*CSwHeh+d0{fv5>=Ww!7%wb30&_Dk9#f2K-'
            '^VOlIOtFZzaGeDIGSv^TYGLnNZAY2mD=bn5s2K;{{^Tp*xYDS$PR#q&d(&-xsnDwc(?F4y=s%OGllyV#0qXK-hACo=};'
            '^&HzjL>2Wx&Vy}g7nrT?le+a{(nB>D-U36R%f-'
            '>Aiq$=?Z<3Z{zaLN(HD#3Y(bxwvl9>g%Ni*T^}E!{mG4+%T&;gEI|js987R1*2keBl&Mb@y$8D{~XLanT}{quT{2la|8'
            'ZpW*mLIt7J07vdkV22ZXslzk-xGVkNIUw;~hqoUn3i**H}eK*4?<$L76c1b*U!yk?vRz^wX6co|!fexAr8S5^<A*G`rU'
            'K$9lPBl2SZ~?ZRUI$^#3`{yUO69~xf#c!=)fFbigtOp0v<3u&e>-Au(-'
            'HKpio$>^Y%KNO6*%Y1fy+fkY5sO5dU}gdZLVPmp0>gIMFnvD83#EYl#YXobBHd8;n_w#44O>E?58VHRq_b5TsjZm?M;B'
            ')UIkBN%0pRnC0fux^$qXV<5HzGFqq26N@H7mJ>r4K^mS?R;cyT+y6NAtsia-'
            '8Vjyz6o_1|0MQ{20^xT@%f1V?TY=R1Kao0<_x;7VUSPLNKt^o8O?8W{IT2SY^o-'
            'F0+Axa_DB=>M4ZfLrXp$5ZvYUgjTuV&Z$6_HB#M68*;QJ!#-'
            'djv1*3sarDNRvK@f>G6dY9pphOv?_@fCe3OD2f24k~&^@7ismoz7Q@I*AnN{<*ZM3Vf68?TwHAG3KCKqVT1Y}*n90h*6'
            'v)&dU7rdefSIU+ViI<x%4-t484Hn1`Yb{zkKND+YiR?IB_;85TZQ)175v?)_i3P(z(c<96fj%)-@&J-'
            '`(x_B(4tMpU9#MloHYEUKn+XeFTzf8n{$>7k)YX2-'
            'psG;%{RU;7Llu@WiJ?^rr$AYZVaAsCOjn2*CZNZ{hntFPVQE1`@B>G4Amwy|+36%tJ5X^0q`0uyZ-'
            '=j`{&BKBzD_BgWDA^HrGn@{s(TyF%}0X8xP`1?X=ngW<bJ@osM|y?mDoFa50`&UyDSnZYD{y2h||hKbWT;}~~yHEV(9I'
            '(&A?lj^UiMvcWb@Uu1y&t*@6>+hYwzSj=Fy9l9h)GTnym0-'
            'kCHk=d4L#1YQShH;pS>nH+xcqFSMW@E81OI1Qn~?{XoTJG9u=b|kSjOG^KV?WrgA$4)L`s^-'
            '_1>;jL`6w5M1!OWMWHg!^E}He^Bgi<?@a>@qL30IWJu8{iqzNrdHQ?sTkBr;KX9JwIM03T{W>1(wRbcxCF91SM2$Za_|'
            '{RzyEq+`tKNl6w!W$AU}%#qyEa3MS~zaEmj>4yA=n{3NL$WiQ0t4`$R2tXH2uu5+i?Jutz+Tt06V4#v}1D;;_^fH;03='
            '4X6Rm|wz~0HQThOw*LhL+&?a!>T!veZA%+ok*ix-'
            'e(=rNK>W^YzI%hs>xkfA%<gvvKZaJ{|&|!wUcMDx?7z(<2({RY`F)8dQqMEn1k;CsgaObHMDk>&J$1_9WL-'
            'T3ilz2e7mpsGW%~5!uwFW<u>u^3O6BCaZz&llCG^`t-'
            'o{mkZnV|xPumaDq*kG}|DfDMNM*GT8tQmRA`aFDs%r~|~vy%aIX#XqpS*?fM7Y_mD{6_+2^ucqnHVAU(q5bnFaBgmgp9'
            '`kQ7P|-'
            'dx$iqjoT&t5r%&{vO(}MLI0x=AE#&I&A?V=sVcpemr{|h<Vf)!bxFX9M<kKth!mj=JrFbEn($Qn3ebfMpj1bVgeT#K!Q'
            'yyzEWrNh-U1)wsn-#k1Bpg4e4f2P-'
            '(SXFhox)Y|pnYIDW5@4qc=hK2*fC<LTT&vLi)g3_U7w&m+cZ#ZK{4&<5XCRXLGW3xpWZ(`J;!fP;LEp#_~0T3xWq_<9l'
            'Inn>c~KLfFD`mn2+B|hUujfaiE+qH%pz*fJLV#p2eQ(FX!f~2L4cmtE%?k5V?VIMWPfd-'
            '`UYi;xp7IIv?+j6akOV9?<{E1^S1RQQf(f(XZHn$zfmVN!wgJJ!D8lEKKnHqcHMO<Rtuw>m&Zw8E{O#02IAfV$(Vv6h9'
            'M;&3_)@KkpV07PMs8Q%==AV_r}z6b7!W4*2-'
            'A6sk<}2v?yvgPFUc_D|j)hIkea?D*>j&8E_DaV!!1n59^49s=2;iLi?r;*S;VDm&J!$FOWo@?x7Ol)Ya+R|m=1Bi=(5?'
            'sb#Y`+gY8_ky%X`(nkfEUXtHFlZ%!{=BbIM<@~p=4P+R&*P}_ZV2?adqCxv3}!5ipzMM%B&O>C4bHp?zdNkpRpUOGS9u'
            'm2?mDBeu^q~MK7yB?Y=ZAa$5F>AZf<4_GL$#&!}xI<#%hZfVBj5sjTf(A>3bh|xBVKN80sa=UgH{3@whn_3&4L>Rv1_F'
            'nbo&&Ieds$2L;^_NT{m^;Iky6Yl7gZXby^A(1!BfRB%e}!AHaK<f+pt#?%o8=7BCrcJzfi!z~!SPmgZRh=KKT3h-'
            '6^CdhwxMkP^O*fxmdGXG|9N$VndAB!M-'
            'aX9+8X2FWs5UgL!qVG#P;G0Se91FSvQp0RmUb=u9>fK|!OV1$VO~;7NwRjp|o(z#Ks^DsL1*f{IXxG2XP!^#~v2!^*@)'
            '5?0l1!5IE)vF4EEtZS8MM#(GAvQ3q<a)OiRX6<)c%r&+IB{G*3%W7Ze9hy9bp*L7L55%4}!Vsau8_9g9tHYT4maf5vxY'
            '<&Ff+MA}9)+{`{eMrWq<bPJ?*o4ydye0ij!$kyAYow*f`AtEs>)B|)yPmZvwQH{qMEILPFkMHQ2Iut~`lQv9nhcU=c@Y'
            'H0usdsA%Yn8$iG{TDpqwP4dnVfuD^An?joz?$7FVYT5mMs6#UX#Gq^DFFdksvZLOTJv$md?_&b=S6j#Dsb0OD#!?Pp}$'
            'B2-'
            'q^kq*ZnQSN{J+lv7OU%tA;U@mV{4RQt@TE3@rFqL$r3jz^B3`pmnC1<$Yk9)_%JJJq5bNq&F7#9(~5zDSHBSO+7KR_AY'
            'VgX~d7V(Wofef{}AQFx7BBuDMo(0<0d8=-mXLCMiqp=5yM|&aZmhr5tn0e-'
            'm{|q4f7(a1OIZ(Kmf0_4pMyvC9qic;?k?+&2r(!&9)z=02`slmW+z5VFZT6Lm}~AnK?dbz&2SVdp=LKjvo`P^o~j{kFu'
            'gN1t%`?ZtpKqa={~JB?BGg2n0&sQSEZsOEPKH<cQI&TSiX?JR)7jiHz)5`*)i!oX+L0!DbWK;uvuoIb8YZgwk!i`G2VG'
            'e=!OD9V?JK^x6%*aQm9J!t70&)Uf|g3e_bAbwaA-+pfdgPT^kZM`lw%RQsboLtC$q!6?RcQJH@%c0d_8MJvh!-'
            'o&68I#`%Q9>pZ`E?YKt&Icu-@FBV-aPWlCk8CG1!2Z?IM}JLB~^!QF=UA&Jz&@iZf-'
            'f%oP1Say4@4*AN`H%6~_tVY~5VWo+Zu?d8k3*Su~v(#bMi2{4y2+my^$7y`MR1+8IH8h$%kR9)r;$Qw(|OhRgY8>Gh^^'
            '<Wr?s$ht<BTi3%~zAmbg!AH<$7PYev(SxDS@Hks9W`j2fy^KVCi3y@?{gFJ!O>helOn-E;KI4A4lz)#zt-FUgnHA_(@E'
            '!{cMlhIFkM}sYu^jCDNuIJ9-T&_e_UUoc7V(D|&@6-'
            'xK5*3@s}F(lh_l4G!V(<LEQg{dO9)xBA7{7mksqSpVDsUxG$DKtLOS#C>hm<5(v70qZ-qe)mf~^KB>1d916(2;nE1yPQ'
            'f-'
            '5gyFZF9nPh^+RW=k#=|pK?;o9PQ**TxyhfP9rS&BbH#bPWNeFw6D<@N>n!ylnR%UvuR8Nu(a7O=<TJX~?#45xEaRe}q&'
            'kn^br<Os!q@s%06IMjjx?OeoFX#~Anw&7;S5<DjO0rPc^0N*_?us)QAeCY>BgLFQE!DBq_J<j@K+sNXs8A7h>dbr!JiS'
            '9Zx35<tN=u)=@px?`;`u*)m^7HU3%93132K7&n?QC<o;^|M&-6+E-'
            'uPH<$*(I37*FZ)CEJ2v*PE3d2(#Qv!@uk))vdY;8dIU46$LuM_!#qE7f7LO<$%@CG4JmlFP?>S~vJP6&KHU1Mn2a5+L?'
            'feaTqGoge-pOijVIqxenJp)tmf6m<{ZEe2?*h$v6wr<iKYIhU~tkO3ToV7zV86B!6nEvas&H-E|h-'
            '$hbS2>gH$OWFgfo9o!P~}f8GGssZ;}QlY)zywe;%KGJN_5S!{3GQBz9>PTx&~<$U*vX;2`llvJ@U1d0IT)HB@Pa~pPhy'
            'W+I<ICW;Wf|X=99h&Vy*CG}!6Y(W%#Q|_5M;*TW7Kax@-'
            'q<^S7pJuR=4#9X*loSh;qq_tWs5BtSL(rM8;}$p48`PbH%u{YBG!j^vHI5&-'
            '2CD(;Wk)ABP8QN@NYBRGRTA*3k9gW^=HU2@<vznIL4j1-1si1i)=Fa46XgzxcWQ~EK{?AR$C>|vp9gc-'
            '?~Z5En(L4mkU(YiffRA!<|t?Us30@<*3t~L8V50&^Ue>8eJ%)`NkYn=GG(JrX~xhDFZ7mdtp<k8&2%VrGL2`2~+w7vI5'
            '#LSmO-*qkHhBF=gqQJ_ZrdF3|Wp_b%g=WV2i+o>(~wy`QTf@L)Wt-'
            '?a+{E<UD)`K$41P$W)_d4P_zDaL%AqzMkEz^lxgwe4s+s7Qz4z^6{oE0AQp*!PPDZM_J7A_wVoW-'
            '4uNdko=5<uuQJjvenigJM$zq+cx{*EHXP%ExJH%pZ@=(Z;y;iZd)dk^-'
            'UWeB^xRanMh=OKr{Dfott^aQhgmvME;rT3LqJlqHCnE`QPCS~7j~fvwg*u#WKQv9n6tGeAemo#d|XLVNFYczvk|%`6{5'
            'p-'
            '}_*ENlw#i5s!g&4;9E^OE$za^h9N$8Z{PVzsZIRbd*<#~y`RSiv0)W(Q8uSF3i>naVH}(zuGITNB`3)h_s`U&lCbFp}I'
            'f`GyzePoT|xAG-'
            'X~8>%SnPaB6W!{SFQY&0AoB^gh!NjwoJ{(VQ)#7O*j`4@hwh@!7WD{*=<71$a=pyTUK{4OYjE?$l}wr*sOi=INb*-'
            '`kQ9}Vr70`b|Rxypa6j~6z3lk=}98L4s#`0Z^vY&aJU_nfk!`RpaMIg<)ERh=37?SH9^+)a3)6%9N^M_}=3H>e+~N6z?'
            'o!mwS8NqL13@U<R_Cd5$BAQb8wd#J$rTU1L#kQ~(wAX3kk(T3XzSQ=zW;>+VOtN9-t&MAY}-}Ugj4?E5e-'
            '$p+_JBbm4&vDnGKKxdjfve570OsX_XOJAu4oTxoQW4B7W<&KB9Z=lMgO{_!;P#R~RQu8xZ5(kyGEzc=4NI{9))>^;=VS'
            'B(cUZ<Z$1-'
            'e5j9vE|LF%Xl4z{_{0v8wTPG!NDK5G=S4TI+y_i9!L@u)s*&BQGkEO7j}fGW>ljN$oZF!eqaWDm0;hiWj)s8-'
            'O4HX|Cfdn2iLw8YeqYQ`bHo1kK72G4YTP^?3eB))COzB*q#xm*XT*Lc#WmMbCggFlE%F9j3bZjvLI5AhyJm@xX8ydC=h'
            'wll_anPVef_b&wz=U(8QOhy@_Au8SKjv0fJxYC+i^;*wOU}<gwvC$CN?f(k$9$!EM%_YjW3yM(k3j;-'
            '6c^O2w5r${af%7~a+;DCcTuBQACAVydtUCdksb$cZF;o-'
            'B`wONzoJsV@boet_2~Ed3sB~T(>Z<Le*Dv^?<B=p4NtsZ{4|K$1UxXmLVgS8k6|kGBg!R`Sz~fDl=sMGe*IIqBW}J(ba'
            'y=kr@<AZ`JDT$Csf0iSc3Sy)8SZ}C2j|`AxX}|Coog0@xNbCVxtWF<bD4rI`xcxui6sjC@8|>mx4=132D@4wf_&%!x~('
            '`DDi-8{Pqq;rSX~N(zKS&3tiIY#X)&C-'
            'ErR{L{}?5EIZ#SC6R<28Ha~p|r<Pd3ce@D~yuFd3x2lsBvELXAtHvQ*um`qu9>lj-'
            'I$``68#E{6gR)&8mY00RQoC|^Dc(WDEM@TU!B>DYiRkb>9L`@20JSj0g2!s;w%QUq6`V0=_&#n6D}mfNHdy*coA`>GgH'
            'nAY-uSwd2+5y>PuzZJuJMMrvGbsWk^%0%lLJbY)mU^)4>=U=@yK~SnCZ}g3!~kn%D<bm>N-'
            'JDjRJKE5TwKN5nNBTL117eHsp&EpA%Jd)IS{z*M~8FDu{umTrNyVR<a(C?x2M;YOEuZFTf~s2JW0upi6GJ!F%He<h#!?'
            'yco=@y7^%sy?i_y-'
            'n(*Ovy3Lqe*Tdh+Au>cR3~Ubi!hp8@yD*mBQToGhXva8z%S*Gx9mAF=6Ep}{nUc(Ta%%#Rv4p?v%^-'
            'gSlm!F2?|ju<YWC!%y(Uho`tEnUvClQ*6YB=)&V?Hw~FkS)kcL(PrOp-'
            '2?x`hu=Ax7u1GGP>k;9YJCP4N624MSX$G>p8IqTFHt=|r4|;9vz^uPq)G(zV&CZ)pp)KkvDO{WBZQ~upSgw=dD$`2sg}'
            '#zEqOzd;sSfs)aRa{MhSpQL@U?L_7#*C;%*|Y2;TZ)k;qtIvX*)PB%%w-'
            'KOffETECPpH$qZiRI=pmR3{R}S3t~gbc;m_?jMRU}IPGW*_jYUqPN`bhwIc>!4SQqVD>+O(eFqN-iQv*&N$mDZ2mMuwc'
            'v77WD>ueKs#-'
            '|ZS#M9WQLT%0l_aA>%@z#RKTF=R807YnyZGy&HD0(bhK<`P#_UHdIUoiN4{qb<M{h{o>a_pcKj?OusQPdA4;23g`v=!d'
            '$7s`<-6&OCPc!cwfm<I!(P?N0G<m-'
            'upEH9%=Ta?do@!$7HC;yBt$=ISpMbh6Ch$k*72U)P$FgZz&|K+(^<)nOQB%ULYe{#n_&^t}4?%&V7@QLE#t)h?SaF9B='
            'XJ*56*7!`S%PqNj%k*+$o}8`2A``-'
            '4F3P`H~fF~Z}}hS_tf%aw&c4p?WJ6p%SHT{T#k;+N5W1_ku5iwrZq0ihqbQEBr$hp)-n&~RhKzRD{^H{rrlt!3~**L_-'
            '``b=XfwbA8=(F?RH~!pK@nvrMNP~`y82}_U=qUV^8MPA17wy1vh3T%bgkZ-jUgz?#?_g<H7vCZ0^}&Zp{Da*SoxT<iF9'
            's^?#uM2o908WC8A@5@g_zIO|fLDBbhMM`iP;w=`#GC=vNBO@?(#8Ph@%#N^ro)+WAs>M8ZHrhLOLA~5L3=<*37tiKCbo'
            'XR%D?m4B0)!A|KizNM8=1zC+=cS@E3Uq>J0qMF_%{Z7VN^>`^A;PwT;Irru%kH!)<3IXU8_b^lH~P2z5A>(Dc{6q2xiC'
            '-qxH98?9GFWtIx`;*IWfHy+?Yi!?#w{>8%)XR>r4qNcjnsatIV?t+?X!=ZZJg~ZZdNXt}q>(J((g0Y?uu(cFb#b-'
            'poUu4op5H59UHgH>TXkb>?HW>&&uZPv*WS?o5puJEr4NH|8@*C+3ik2h+&!`v3j^*X#d2`fv1a{~zf8`gJ=LMSD}`eLg'
            'H)9Y`;9e5Y(fr-'
            ';_&Yjk<mI^g7Op!4}T@WJ^Z8m`<*8C!(0g>S2haDEXzr@f1^9T5Ug9(MZIay8oiwMC9IYcWJm3@WyqVEmeLpq5#DjAr{'
            'd`j)U^Nyl}nzk8PTXOfrxw)sjF7S;cs|JyX9rT!cJJN^gy`HsBAZbe6It9*exNs*MJ;4ao~mBkm;J&<dv%%Xc0@#f0{2'
            ';s_r+}P##?YRMp-?4;+yRzZDMJ7BpPiHxqZi1)tj^G>lZZeg32y(7p0DCK6$SqsUc=^v4BdqSiLB9yp)9}S-'
            'Tl@*<d=osso`>At9*dS2{^G1(Hs&*Z;E3T|&hF|Yrpo0gA(n{K<%VD^`~!0w>hV=_F=iZVAsU?9fK%ZP$i3JF0ur8({L'
            '7tXmtKgEum&3r+F|(Kn{ZoG2W)E7s58esIK9mSKlp?)UPU_+rT9bmBT$;+Ghdv(HGuaFD`^neG5AO7sUcL*<`=Qh6Il-'
            'qp)W`Q*HyAYg;HO$)!@M%gFAM45xG<S^xGZ|%xXV}Aw?VErGz7)J&&l8zBlp|*HTeN7Ct-'
            'L2Z?PvkZ*l9I3EoIg|S3rcWlAU!M~_r^mUN=c!Mgun#APXIN<a3g^x)Slwk!#SMVWh<*H*D`sJhBni|xu;)XSXf53kEZ'
            'NkdB2xIR9P}156Bm#F4nVTvwR22XM)v;Lf^ePnJze)a>X3+{e33#s%$;eD9XT-~vz~|okklFH-'
            '<elInzRPZ*(t^;P?TknCQHwTw-'
            ';@h}O>%VM!V;XCbi;yo%W%W{y`<DMpR_1;!^N>xu;oSseW{!WtOY(eJx~nh2Vat<bGcwcZxuxNya125-'
            '5|Inm~=|!;Mk!C<*R04nD{6OQ@6Qb!kvXAX`&W*WfQQZ{|>4AtVj>kZ-jIA9dL2q2;)P*QgGf<fZGlWfEu$3JN2&Mgoi'
            '3sP?&^6B?pLaX(gQAMM<|w2E*gYKAe4-'
            '1RgFK<lB!@*7KwGn8EcL+#*^*`4<QHoq3J3t7Yh$cLwm~wh|n$egV@D_T!ryks#sF2>WR)N(Ehm-pV4F$fyF-'
            '$v&u8xlPVHHbAKAZd}FGB46G0iTc;Gkoq7P4$iS)Tzf6pq&x=Uv!7|tuN>6=S^%d4M2N&}54iM{GvJH?s3ggO(Wn3{b7'
            '>}jIMi{8{B3eYv=%1+{=wR;d9^?E{J`QuEOd442TRE!e0qBeUN&)t*rl$xZgert_LrwRVZOleQ4uwo-'
            'H?=Vfy(^jxGPYIW_7cxb~kUQTNf;aM>mo&qr93b=GNop!2u}Dw#U|_8Jc%;l;M7YVo-}eP-Ze(INxIYy=V>x-'
            '@k*LePwt}oEvz~WuU-'
            '{4}`I657^$F2h8XZM*9zQT=Mz`wI7Qhf7{qWD&z_XTy%xpaSy!x#RArIcA>|4ei+bhqw{TJNW!AUSQH#YpBQK0k8T;Xp'
            'YzB>bsZH8TL@2fUngS{N3r%#AP%ZZleA;w<m$C#crCXKRPVLnt1(j|(#wzQOs-JdouwET_nADI@<1z#a!^xWg>Prp;k*'
            '2w$Q&@n0qM6GDinn`7R4Z4pTnxwuO|DR{UclCgODucM1!TJxZ>(>)bzQGE`EAo*;$4T)`OTJirA9wi&J)nu<e8-'
            'ipIv1C+;m6;S)$p_Y^R+x<8Zi7s7GKR0P;|w-S6Aj@qRMpe%ioOuf7V+EG_=Ys*?(weJm8y?Gcat&-'
            'tK?@h>R%Z7*VOJGXAo9dJ&;=i^AEbuTP5%>2JbzymQ{~5^AIT!;OPj{o$o_fq<_kx8*4{1*5eQM^G067cGpiiv}7B;TH'
            'vrdK}*~W`SmQhF=Jz%K%1d7i4i#Pj&(DiTvEU$Y;yO&0hqKPgX9t7YL{lM_Q=?C+K*lQV2RzhD|s)}XyUwWJE8LBzD!x'
            'gSP#-'
            '{coDxbdWfz9p<;P3k&dZ$MntFP+QRWjE>u2>L)qQW55R{%Q0tMLAsV(ea@08OX8P(%4C?s^mmt9Zh|Gx97>?_P@wpSgg'
            'KB_|5Ucj2#vCMevpm25kti|Lh#gh~mZzqTKKfTQ?pUmpJXexD{B`9Z~gaMji=4Z-'
            'O1SBcz)>x><;1^Dn+5^UV!3kx<hqSeYvP{mzC-'
            'W~NI>r{S#=<`MROu`6Nt`=a*<`<a8a~Jrdsu^rc&5(<;14|Wy$n)cQwEtNgWFBgvy9!^RL76XH^S%pwE)k%WXpN)u9Kg'
            ')}B94sIlGz=Fq;2UfOt@4+N86R5urUklUYp@$jwR5c3f4yPJnYh}psSCaVZ8scAHV4L(#M|`)s}U5(xhWcz_qUcm7|Mc'
            '|D-'
            '>pEgnHnc2^i0pis;^A78hNpvJxn=vjFLQ$^~@=<B)sI66YB)(fb<8q@^=>0oLlnTToUWK~5D<zZCyIW)Sv6U>Jx)ChTD'
            'kDEBnn#f0o)8`QmzuP%e8;vV|1i}yH+mPzL8igEfaqvwkM0oC}^;J7CT3`j0U(-Sg)3}NKMz-'
            '4VWFK%o?L+#a*i}~z<|0eD9Y^=q!=MJm^<fW*d7w4wy>cdXR?l&Q-'
            '3Ko?G?HS+F|baTR8@CN#XQS2c+R#33Z{$cps}<n(_|;SHO`_N^-'
            'bWDTpSvF*a)xOFVdR7wUmEn8nzE=;n;W*c{|BY+?n$*c25fIJS;*7Hy44cm@@D>+(XF~5pd5`sCKngBk*fphR5@@;k8E'
            '&eJmP<dw)}W^TeCj1anZUUB39$KpGa=UPoQ0rO@V<%$Vo5pPbIw3;HQuRNFuiMO5BVi^KA;q2Csq#Ij)Z;ZRIIy&Ik+B'
            '!h8aGA^uJ1uJtF*ItWzg7MpvV7cr+*5VI0KxB3e;WE*IqdY~V^*)<wdcGi8FLxTwe6>d*$u=ru#6iT)){+3DUaB_afJe'
            '11!pw>Tc>L~f2-'
            'zAz`3Can?K=;^=EE)G<JSfMRCDmm)p<m<BNsGBa#+U=8tCPBJ|LVLiwhj07>9gqYGezwAb8yy#w%t4vcz?${8mj6ALwR'
            'XF<y!d?UDHPo)nl@)xitxDtNh*6IL`oM?bAkX!Aq?AKc5KImv|(@LL84^ipZo8X1<)HVyo+mj&YwpOCc&DW2LrkL>(sg'
            'Nr^Yz_VYcK+4q`)uy6gDlP(MHMZmANH)|8rm=V~oTDmNvuHN2667<D$pK+4wAzP6Okh7FPbLOy939}uvuf--'
            '=}T0ew9`{}=jtGOH5?f5q-'
            'MS+pt>asMYr0($?y`8C+XPjkVK|uZE=x&CLa8e#M)zM1@^)hz}AYQbWtIAvQCgE?OiZ0=03Uq#(;A3o53pfO0?<q!-'
            'U0xAnLsa>y;0|E~!Y!>-'
            'K_A_XP2)YY3PQyO8y~wV2Mi43>C>;8vSR*tabcmuUT9bol9myTUy@6QaaQXXep>Kb3f*;StO)@<F8<PPC9aM_-'
            '0S6RXunfxEJq^xb_7S?&IS_OUn>#X}Yg1*5))K<(CT*AN_Pkw-Kf&i%NLl7V8dEa*09dPNZnNXF9}POH?Z--6lkMexZW'
            'jmlqhLkHWxth9q~sW5X3O4RA2*|#G2_&cyB{G1b-{A0uDmk)`6!8+){DcE~;Bg?xjnHaKn0Pn$7Q1`lu9{YBi=-'
            'TIF4Zj}TT|5XmMSj3#eS=<Ka|4Eg0~v4M*n{Xnb@F#_A+q0ALgCB;SeHHqwe4QWuf2}^jF*A}#vHq~#gT;@M<K9l1hcp'
            '{LhGsyQu?$X3^!fENvBl!={Eq0|M+npRNw}Mg{-'
            '}*Pixj{%F?7iyUB~DN5I8VK%ccdh0HCt;Yg1sY}x*gTzeP<H?{td7Qr1*^Zh7&akvcQ*#mLvS^#xe_!^>^-'
            'Y8l63Vc0;Fl%Zh$}G&IN?&CWRRalk*I$fT$q%!<rR3;PF^HVI1TKM0ytR+gOU}zNur(h0`*Xmbw_e%4aDXM$5{*<<56G'
            '?OjKjZ=zzsQ9TwOkhCq)n7KH&izoxF&0^F3+ehuf?Z_n%`$ZV_6<`GKR(VY25-'
            '8vF<vCnmF(7>D94&?l)Gx1I7P&sA>VTggn;%#azXSe=7wS6;yl8zpRx{)UsfVHjv?#~9FYhHoX)5IuH?^}8|@SwE9NWn'
            'Tj{B-rDs>uz9G=>q}TyWsDhMtHcv0LRKjSb8)ZyBs}0hQ}MAO$D}1--'
            'RVr2E=07ahj5uMQ@6vk={>U*cKuSa-$*WRI(Rr-yR_w_vNvcoJfW+rVnzfRkPOJ?}YlAOYkst9{u9&hKkePz=)ai$nU@'
            '8x2GXKAC4!rBE_uGnMk~oat<2mGl74pD5+zvXPL+ZK~Rto>hAO=v1;!q!}A?U4~fEhr7yT<10j|LEG)^ZVr-XsK;FL;!'
            'ddSCs5n`J1<&J<d)+cfiNB2U`PnsN?;=rb+gJRQn}Wy2*O37W6P%y14l;DVp+R6O3=Ss4ik`bP)pnZ5f4z^Lx(`^nSEA'
            'tW_XLoeHU&w^FyQr0fs;A*Af0;?loAdSXP0c~e%(X1WoY8<D~mDNs1LM5_<<>)fqQPKsBB-'
            'C0C@#>V2juq2v>QAD|+`p?~*kv5%*1ujQ-tpw|^6^-'
            'RB0PYw{r|z!OBR&XWf+chSrt5F&0U(gn90;Q78RyxqjYjsJvbUUn_Y+^a;qyA;p)=i=-BaWWtj0w+$nfp_C;uq~-'
            'Tv&@U|et8kD>pVb3>?81CjRw5-EW^;DJPhO-'
            '0`@Ck83AJzIPo+MMT_H6scZmRed1`V);c`1w4WL*SB7YFB?zwA2c>MPz@Z+23d?U%-'
            ')MGt^2;5JG$P>Gy*OwZ=Z7#U5t8}yHI+?EL5rKqYh(2G;mo^I{CiOok3IAuH)uYJ@x|a3rKdO*TFz*9)d4%1Xs|wGi3`'
            'FzaM_7X<T-'
            'ng%FaC{_+nWcW+?8$Zsrc~`sI%4E3?rlLkzCACqTZ|cl!3*edG?AA^H++aOcr2NOP%%mHVT}K_d&?XK)gm3Z$Xp@*DiN'
            'v=~NgQ*f<<HPMm3jukBrFzT}xU8GnHc2)C8&P7jT>s>_Eq6>-'
            'h#{+myA`=tj@6h42WiUQ$0Y}nn$P%q{s9U!dbS8WlV@`kZ><(EdV+~O=o<_!R!)lDM93&gv>(Te`1B|qDz^G9+@?}nUW'
            '4k(Xjio@Y;6tnwt)|u+_fr#lGZYJTCwe#3sfP_WvEy2c8>0Qt#v&QBE8<aBVGZ0^R)lLxACbzg#h@{z%UE_j6T>yesil'
            'n(_%p7Optb(cqa=&$gZ^Z8Wiy0r;f5nx?qpBvA$;-'
            '7hL+zB$8haDG?SXIGQ7J6oZ?e(uyP9t%^GJEWwc}Y+$?d`<)PaY73jywR=Se6n%XN1QD#{m!_nLf_Su@!MUmBry%#~`^'
            '(MG@`WMwUEr5*0+$dca27gyglE@V|3I8N7${r8GL)8-'
            'G!I5k{eklUJcI?N6=4asRo@VkU>XnM+by57e$Q;%L6yrU8A9&D`O7*dk9x_@-'
            'tgrKcUi=$)S~WxX3(Dwyfd|lkC?E45$D!@k9i-q%KD-'
            '!SiA(%)V8dyDuyfdi!bLT>SXu^aqkprkudN|nk4|A3_Zn2Yc?>NRQZdj>fLeSL0*y7E=yt&lR?X-'
            '`yUsl7YjPFR)0cqGH(@+_{s%Ri;KhwYpXfn851<?E$i;$k>|7KFQx@|vAnX%0@G*uHwQRL5Kf-ZOEDMF57h!jt2TF;rr'
            '(9k+c<3w>zVNeyXPG$sJh%vUos9+Mw>6l}?~J<^EJM4QF#Oxxh?P4l(7<5{#@(=lTU=2@Gh40}@BPAwwQeXcw*$Y=tfa'
            '%1eyHZWjwIXi)n3!ukB>S|f$n4~&VLv|$G8KKC~c;{M8?s;h8OSEcaariR<M>`9WQMR#RKKQsC*hprsO+`Rj?9r<<-'
            'K}aZhx&vq4SQKGHiKM1AujK;`Xr*0S3HDC2M+<aAe{yha8Q<x2;Sj2_0msJDoF+-V-'
            '?G=>+uL3VZ)_2xNEUZuQc{8I|2j&r<V99vA8?Ik!U@s*t1aT)A=b)a-'
            'j2`fMGDEv78mx{NWqs5B3xsm#tF4Io~kDN9#FtL&x-CF?`X07C#Z#8x#FR5M9-'
            'h<&Q|B$fye+=i`99*_f6rC5JM^?j8+K`e$@4s#*A5Bh!(Xvw1Q{IBVJcaSc&I=%;x42d#U>N&*-'
            '9R$94qa0ZL!DX*7VX-Kh5dK%%GrDPRH}_`W8A_-WpB9Y{ff5z=N{NrX?%ROjKz1j5}VSz$x?|WsIxZ_hTmA={DE>HQ%k'
            '^5LxI%w5s+!wjH#avV2c<pa=mS(lD8MZ+PeIj*{u)g$*5>JbH|2Cn}ox+Lnd_dpe0pYrA}7rGI5Yr;)_>L=)s(AG(%}U'
            '^c~HeV<k5HY-'
            '$P#rZuQG?2UUmI>^bpqR9Pu35K3LKrA>oNY#@%)Qr7`i3?Ma>)3Ai<h&I&r&Qwc$HC}x*dN62Kc^EuCg`Uxi=7iss6=x'
            'Bc1C32I}veoIv<YhajO`oEB)cdt4eHid<@+eZxGNor;Ub*xKeNtKK-yC-lYNzr`qBAmo4Oi<S2Q2P#c6}l2Lf!04bB6t'
            'EDw|*duln-O5Dp%g-pNDBKUlm#pcHh+bS1$XV;q^^J<Z-$IUzzN88ILyV%$zggFw-9`o}L1%$bjQn*6^1Y&P1OF-Xn9R'
            'c2#qY`VcOe*4D5Q;z&T!1)B4B?d`u{iv-<@4BqtJ$BhdYSz%g?Ahl!TWXuj6>@78p6d1NWra(h-'
            'er9O9|R!31^ORk{>|St&q|E{EN-9XR^u8tF{QL`Si5@Kx}kmcbM%cz5F3Efv7Afq|C}UM5-'
            '&N2z>>6Lh{fPeL@fAXobid^jEfUmkiuwdw+>SgS`o^#iGF(Ih!{_6*2f7o@$qve+JTAA*G{S+Yh6bhVQw)qlnfQD#oqa'
            'AYg+g}Q={!#Z4<9SoH%;b^ZF24g!UAy9;q-sx8c&ao)QwNJf7rjHNPW53aFQ?971myOY-h_c>3=rXboIC-PcoJ2DcBP_'
            'uDn<v!F@h98vOu#E^VdtU(Sh}_vyv`kl()wyB=?W*EJNz-s=O2ToWIc>1Z32x79gOGoqed(4qS8bW?)mc&!=-'
            '{iRPZct3b1gpc>~UCFGC&q6g+tJ28f?{q%z**2EujKP<mmuX2VYgOQHWMzCBWf$Mda-'
            '&4dlcjjdxz@0NqJw|t1(uP9XOoR1-{D{*{g1zBlO1Vx)}qp4yAaP3$^wcZyovMoGN$6TXky~Pi5BDR;-'
            '@^N5{r46a@D6M(_tpTLoCc=^vfjBIpM_LN^!WP*OIOY|KY8B7mns);R%)3nfsODn%a6EplctT#~gtM-xl;a=!mn5+vlS'
            'HfSCD%InVRdmGc2zIKTTN=HIv_<$c(~!sU30978zsLSit*yJUqoN^0o*xd4x_Wbi1(*)mHp2d(Bi{HGv0p8%?e;mo%f~'
            '#=QS`rr2)4$pM?=IH*EV8kMfx}AY`VQHi<lgnW}2s%@=?}MXfMiI739O9dYgFN4Q61gutVdaNTu^+FK5h<o&_$IZYA+{'
            'VU1)6+c0KO+T^CG({EJr(|95oChMR@Lhq^oJaN0*622Tcw!4_d_k;H>C14_>m9lWNkgpECUSJ=Q>_2%kLfe-'
            '87=?uxv?=3e*GRd<kvxzLL!QX9LCeNk8!nZ8&)fM0AZ7^Ssdd4D#c6TPXBI<;;w-'
            '2pW~tM$|I<MSw&Z*wvfl+y_l!iNy8+S7;z?gVD{b{6+Us{s@z~W@Nzy5+HD1{=0=#-'
            'Er;o@YMeCJ!^*r+lEx{A&Y_!WeeGMYBcE9JDm`IypB?&Y%+l@Rt1z0Y3~J3(P+ry#M`zZ7bF(74h|IC_`3tPe!RheGY?'
            '$o+v=DCH-T(t%kK@;5FZ>oGhHZh3)SG(=+L+X0ugOo^{XP%ePsD-Z_b?iNu!}V|sYRAwzK)0AD>7nMh?7z8P|(@F3D!>'
            'Mk(KJ&AkS?>9;u$iUpCJ$_3%TO?-'
            'GqRa<|D{(fd$g&SDK4K7#1&Q^ZVS1^j$pOX{C*AYw=Is>Mp5QCZJx<V(dHI{Bpv7aHwhNLCyN>9O~;Y<m$LAD>Tj<tuT'
            '*CIl@dM!;_6Z?MX2gl#$#aM$!S<MVVCY~_hV!+<rc^_#zuD+}kLX5b~*X@}%%n*v6ME+8jfcf+Tzy{OT%4tK>rrn&O2G'
            '%zO$-'
            'fs8D%?<I8#g>3aomRt!%+)Zv`Z=cO+Tq?jS$t+>1Ddw}5H9+b3@hg0xIP0R$cOe^ZepqP2f>w)x$k*08br6efuEm((e#'
            ')b76y33T9yUgT^Iz0N2}pTQz*TVW(LXn)o`o58J;AGqtTanAdw^viQCuUw>%C|I3W+d1?lwVgLtYs&5NlKh+jn4LUH^C'
            '*ykDw9DcH}z~mIY#r2T}_n8pM&$=qn8P5q@Stib#QUdQ^Suo$>IDPPthtf~nAne4ZdiLF0;L+Fs*BOt=>g-'
            'r><Ceso)jQEg7pc4O4cILj12$X-@%i@yaQ3wj{E1FRmG@>a&PasLkr0eJR7{q>=mmo~J(xH+jEnAlC-'
            '<IesmwS8;gXzkve#A_Wwi_$U$(HRiU@ka6_Eg(nDByCjg|1h{R)F)!W~+h*r{FDXR2>?7rr!U1CkIb^s$O?e2K#Z%Ua+'
            '*r;7uu43JDp!ttv^aKSQ^!Rl@!zZ#cd^|1+Zy}p9zZSMoF-'
            'V*S*5(J_1#BslSBB;)*m}Ag1M%s=f7+12U+b?pU`KCd#phN*&wE#q%0e5W6Bp%bzbbOEpwTt#p)?)%125Ar_b_7)T72t'
            'XB15D~q!k^JzuwY>y1WVM=U3Y@uL6|AzR%xKtV?v60W6)}`Cs`u)4!n}GsmD2E@K!p819QDcGeZ!U2frhh27XYgVo6`w'
            '3_`b2B}=ls8=TdCKyvjrSet$yUe8LQ`igz<dw-LP;gK-'
            'l936x`=J`;r=LfS4K9KJB#jzGo3|(^_4re@MNHlvuu5>7#E<B12s?#j)Wo>Z!vI>;fZN$3s52^I{QnD<zlm32?iC4EIL'
            'S5V$&}zNPGLBWmZ^!D;(Xf^jJ6a-mRzX?PcM7lK;KAM~IC<w1c;-'
            'GsVdFDE6&};KIz9MuNQyLl3IMD8YG{r!gXbE1@g928Jw8EH?ebi<w_B_#D8mkWT5bb(x*+Sc2{+ll^bH*Bt0oQ!k&soj'
            '4W$(`P_9)F=dE%Bh8*Db2wkG?6H)W}fe;>D_z`d5UI<P(0yoBOaPNUKd?>j9JIXUD=WGsiHngEnPa-;WoP^zW-'
            '*MxzK>Dwuol3l0L|0aSArpejasJAUpp}w_PZrJAq@JuGpUn=@$YDd=`S}&>RUD>O?6-'
            '0J`8Eg}yoY?!$)KxJOw`29S#lrvP&RZ5<R?e4KjazR@YNF~kL$oJK0@VEf1II$<kZYKdU{%6tf2tx?3REJlecl;<Q?RU'
            '2*YbPw*x6k#H${jEJdEzbRcjcN%_-'
            '?I~MZRuH|1+>!T%CJ1JEG&xZrS|JgFsG_J++u^^Q7Q$k~nOX&DwVXgJ2K>F^GAMT4}SFKr)iZ`7aVZEFxwF^^bX+A52m'
            'AiwevyDA0=RFC9QGpn<{0xZX2*8GXKgRGAV=O-'
            'Rik7L^Age}%_zvq)Ax$rGFZCPt=}17!;R7^0e}Hj?n8PRaM|f?82ji}_)LcH#f(y%IYi5#v;NQ^2^yR;Kq$%zWz3;w`<'
            '!2v_T0V<vH|%&qRXLL3he06Du6=^d+ZWfW{k4NLvhjGh%MC;77Q&s*kKnR&H+~oEU<gVlvZ_YK@%VWyJem9m0_tb!!{u'
            'T4?acu^F=U6_3+kZc<b9NL&mc7t^Kk3_2)t<AiG~SEu;<?kT$4gEZRuwcG`0}d#g<@+ax&JuwgbD%l~B3X4(!D=q3g*U'
            'L!A%9?KMT%l3a&uk2gSbrzggg^g`@46=bwbq4mpF)VO*Li(jbHKktK3&)SdhObbvSyEVwmsSG=l-66gCF$e{uLx^Auo-'
            'SF98?qSWT2U|L2F>+I!90+N@<aKK6x8uQhaOx3;MDn?T4}37#1K1mx?+oLVNdYfYH1i65<(f04j;5q@Xh?KaA7)7#aBT'
            'JKOIQ`t!-s+gr5xy_i5v4c?syT>%mhFme8G4MatU!SP9vsXsvvQEX@kQo$*mH&_77@+xu|=rx94#j-'
            'lUIUu^yM5+WiL;bq+*CJAvt|F&6D&d8@J_ak7#>;=%h#h@I+ZwUYUe0)D7PJ2SGP%W*OH0bzq5OqBRBQ9=G>ytsVCfA{'
            'SkvuwoF~E?m?@6PR3ivtPK(5!)V5$~K`d2b=Bq<3F9<(J#K3LK9WAWe@U;+IS3*bZ589d@44w==Dk!AG+*RLyL{Bya8!'
            'a2J!kj)Oberh0wE(8?dQ?)w#5}k?3&Oh->;9|`&@SbAf(*#Lq?ePQAo5gVTUNV~BsvrV$9JA-'
            'u4B2o_0Q7^V=!e;tH163TuosTND#=L5pU42-'
            'lS+)%rBzTA_LNH2Rbqh~Lg_;e8eg~sUd($)H?cm^rU!gj9$g8aG7P~_paT{~K8Ao22Y8qqPA3(sVA`g1j*nhLmA@Ap=i'
            ';Kp!Td1YQ~~Q#wc)#LHdZ9}Aw0PWQw>%ScESpMUaf)y6XI0mL^|43rNf4|jWFB12S1MeV>Jxf;Edn`NcijmiTAX~`sxC'
            'Db4nM4G##P8FA@W24&Hg6i}Op)p?3c%<P!N!Pfpb{{=VNxy;iOwqe8uKv~)3}AiBS%@UR$uedP_8a)!to<Kv8`y*6NBG'
            '#_{AU4^0T+;~{_Ev#N`4sZJJLEk=mm?~K}mmj%d#kCe-)CbV~i@G31RY-'
            'O&i?Qfd4Mf$g2fmpm46>dNMz>Z2*Qv6a%Ke+EY#%$YjtAm5wbxLS6U0g*@%V0g0s-Mqz+v`oj-LbQqQW5f$1Dbquhm#8'
            'mQEyE4&ZM)C)6<B!xFo`6fQcnQ(dzP$iCx>HEKOdACC=!<mWf=MuD3OI=Mr;@*37I9#wk%3O{J~ogw@3W8uekcPxGLkf'
            'o-42hW%V5WYJ}Fqq4Z4E}J2$lY)B$$0~kBVdQ-LqK%0B|u|e9~7RKA^pPj<ZZ+}^z29g_a97F1YZhCj6B48Blrwf+pT2'
            '@5AVapCeEn3_z33bM4<f3jrcX%i`?0J3OGM9asJbH<lC<#*k>C@T=y#BtKT7H{T5Da;&Q`XitAvzl_Z#_T4PEeKlHrJM'
            'z2zKuwCH@GW+ktt}c13c{ImyDI+ysZ=b`I?JIy|Q7WsjL=fM*i{Mm{Dop%xhr-'
            'q`WRb`{#+MHru+gmzMLWgGg;rhM;U9{tOmk7zy#h?yDnamY4fRU;NZ1Zq<Ju4=@*X>m4lleQj6?u$xDRwVgrd}MPog-'
            'K13`a{u%)Ah{@^KLGz|ZvO?UR;@nk<J()~>3Px;}F*u}uwuR_~(GHF#!9UQWL45P_r@Y{71_TIRKzx>ST@sySHz`Syn3'
            '!5|s<Yl9|O%HjbEdYHjd7$*`F&rM=0e@aSp#m2l!}5x2@ZwJ|ef`}V9TeRm<c<f3b9^JCS_UATkpKc4OJD*22#dGlJi7'
            '4vBfn7|L?~WB;Zt8>v2!FI(n*AHp-SN1y#YR6$UvU8?a+B4gH9ihrjn|iWI>Jw?%VX9h)nKgRj^v%Y$Q9LKO{_TS{LGd'
            'mnyvXF%qX)`Jm2B#5E^o;AG=FYGAXEaa1xGVs&|FU8F1)d`d%$u5z?}D#vPx*8?y2X8i1P0A5&hLBWGw!qtBVuD|EOip'
            '6#qxG@$^m<PjTt~(y>4Mpv+RS?iWTJum*4CSwfLj1;cEOwD;>KA(pK3cECd9&`+Yw9ite7b|`_Q7z$h94K+-'
            'G|@PLhz@f6bvbQ!bNLg*rvD}s(;Oc=cBhVP^Sz}cI1QCI~nNRCPX+EB~Uw#G1kA?Hr%sVong6h9p22;K#c_}L3)i6=pQ'
            '*lvKj)YM5_aso(=;2z5B5!DVP|G7GvGIQ8YC(Az`(RnCZ9@?F;<D;qO1<_G1lZr<*|kKr+3PSWM2`e1vjmIv8&UkIiw@'
            '7`pDi0h4<Bq5P>joby-$Y#)|EXs8Vi?mvlVMEBCjvzu85y-'
            'wlt_r9P~@e=nLjzh4PE&MfGhR;UtA;;;}^nAh_2Jbm@{QKt%)fvu0-bW5BRjx_0#JUv2t~8OPVt(YB_l-oHN``6XW%#{'
            'G1BM2)F?SsyFJ5~<vE~|-'
            'bM3+l(*}4nW`bsF1mpEnw^&?zmeE=V%bK>#FvwJ@!;y!fNJ~;+OJEqi(kKACs#cTH04>V*CjqS-FA>Yt9w=zePIJ^b(K'
            'co&X2!|E_s4uVuku3mhK>qWYe^|%(#D;BJP|;f1N~4vvmQToouX%TY{<H<1@P^I485BeirbAsU=L$0Wj;+JLv|0~kf90'
            ')`}pD-mn{7CX+P_3!+Eehc@C4+RuPqk0OYT;Vp+U-h-cS(u@<qd1x+DK>hZ-'
            '11briMgX+9m^X(h)ruHHj<5WeVUtut=dx8Arnh(1ldc*SLwz#>nj+ES6j&TjGXsxgc9OaT3ij_xEv|kbq<=nz9ho5-'
            '8=`5TxCh$Gw8GU)w8VhD62#-'
            'Yo@tAiD?Rt`sthL2qlSeGIm*RMACKI1cCScswJ`8mVBK9SjxaGMR*jC@7+w{0<ZsZ<?sbdGAd0`kjZghu%s>M)vu?SbU'
            ')IsD=P9h^~NILf{WH{^Xf&Qgkba@vio>9LIR?B!*OytdhL-'
            'PbIQcweJbt$BA4bX1ngK=@gxO8v`J;Ebk@InC5+u4ZQv;@&?O9mduPiL`JKO{53CUnK-Hb&shHZ0xzfoLjQ;cB^XG+Wd'
            'H!(x$;u#BRl!zh*(yoXhrB2n+)S7eX8h?gJCGRU|i7>Nd=nD1j0Dm;z?F1%=5cNy=e7s8_0LKvF!_R$sR;8cd7s+!L-'
            'yz(>)Q=cTEg4Qo`L&6)6tA~=6k-9`?_hk~YC<V4%J&AFLH^Uj>L@<2q1--g8G+lWUx;@!XSw8xx<Guq&qRk-'
            'izBGAQz7r>97s1{80obxx5pUZCgX!x~^vDsRZ!R6gJ^s;{;$#MG^E*_W4s67jK2bc-SA=ubjmIbJ7~lOm@g3JqybN{tv'
            '0y#CIx|en&+wuacMy&gE`X&RT@YZ*K*td?G&su*Z;rFz#LJ7g;y4d=xzUbCw)W!97iuu#@fqJ(@5D_GNuc_D6>N+bM~B'
            '(H=xx}<YCpvR7c;_<KT8sy@*IQcP2UMu(@W^|c?$ynxK%IlkAP3%4otb{0WX59z~1~Ej2jNYI+Z$<o#Wzg_XN@_^@84E'
            'RA7y07U-'
            'S{Br>^`RCDJu*czP$MplRMo@fEurCWj5bsi|oiUs|MO(^xr7e}~U;OdHMG)=9i+PmA3WpNjjIuc>h&=1~udec50C&r7;'
            'g(&+giTqfXMyzz&;TBf|=vo!SpOD8G!ojAxEGQM&>#kw2X$l-'
            'Y*MN=>GU(Y!UX?oy*C4(@91Wr)A$G749P)!`v{4)!A9_xlyaUNc^^15bRvDz{<ySrOsepxTcVYeIR<L*<MC-PsutxGiv'
            'Ap;Ymi6<pqB5*N?rjN6zMz537tX~trA$EoBRDM80BUE%a3HOb`VmI}?+vixR|;$$ehH;Hmr3kaQ8b^p3Y<O;IDPmxRUD'
            'rtmkmQOqqd##HY6U_MpwW}%`o7r_M{i``e{@5PV_R!$ACW+jn6kg=~V$DqcWFe`GhF*_bOskF9>VpE|SF5VS1OTz=-'
            'r*02+U6=4RgiW9_})v5w#Oaan1QGP6p%L<r%2T=(4&X=o{xh7zJhN~G+)v$xEU2xZ-'
            'm>qb^KiKJ4ZtRzxtf4$$IpT0kRj?e3O{Rhu$JkIMpkK=hBrxl#u!62od-hk5rTYT@Zf`~5FA*ouzXjrcg8m|UP@U9wm?'
            '(T3H;?^J^Z4!V^8qw&{vlw7IM7I5|qiR?EaQI*@bmUHu=BOAtWORkvcg8|@ZXG%A|DE_>Du$Ak%5d;aFGeov$Jmu0QF`'
            '-s`1L6nDxbX|Eo&pGZsr$ysnQ+npJ>5(ty+>A5=!E@Z^O@r=BS)&iP9^FNypoKvTxu3<C>Q`<G#2Z)Sf9o^|sSAOfwl<'
            '3fGe^rv&U(;n0;G;V}N86#95d=+qZca*t_<Bjb-?-C-tPsd>Ye(+Om-'
            'c#h%`uU_hX_&kWNcLmw#S@>+o4RTG=QQnb_*8Q4j^5+ds?0*a=w2blG(m)KC7@#_PLvY<_J~dt9gC5Z_^!(Z(8fiw6%j'
            'pDpSXzS`Wogt_E&?=qb>Y>SWO(OWuKn%I4)~q63}jq9$<+rG#=V*;mM_D+z*l6Ig$wKrb0E7FQ)#PODlDh1IOoS3lo(h'
            '<J|#T@&R#FfP%Nd}qXaO~d>Eyk72*r`-!yU8ZB&211M7cZhp?6P;3+l-'
            'OBQS3)yr34di4=V&j<pcpA+PS#0T=2Yd$fbUkvx&jo^Nf&twMlP|Qvc^rS?HJ9ig3SDgUwB4<%<9Zixr7>HGKy0LUn2;'
            '8Q<;6d!u8BLH2RxX`%x!+nk|B@Y@FW3T+2mNSYjVt@Y$|4-'
            'm(_}{pcw_Q;KfE=zlK3%QfGLrR=S5Cn*tQJdjWoiqI_Drt`X}wLdx$$W6@mE5a?I?NhJVAUFqrp*xO@;KN4DM~+zVad!'
            '#*pT;VzDO4N;J1R*YuzMd+JAAy9E=!vlstUS>t3`~qv_isb@|bMe)yF!-'
            '@&9=eG2z}GwH;L|D_97^%Urg(XHS>Xq<i;7@jqdQnHo2CYSdAR#;E+$=F37HH0Veq^zxLAilgFA;_FwcX3cH^W&jtgU$'
            '5~Q_A2fy@3gV_K0!%M8-'
            ')kGnb2)o0f)%BDm#*M3IS;8i(Sp2!Q5PiJXl3Z2|W^FqIMYs<{dyD8;n<+N!|4s}#BCzDrb@=$IAL9Pn0PmqT4DTBw<q'
            'MM-zh>3Ip&(PJ_1Az5bj0r4cksz^gl+3Qh|sy!Q0tcqOWw>wvCK1Uam9NmyVVQgl&hhn&97MgbtF2fK7_wJ1tF(Y7{>R'
            '#V;2?=fz|#TG9VzvU>!380ecUe-'
            '1Hhn#02P}ZiN5wAK30lb^K5KSDXF|{)5|onl$gI8ZPuq$8xPY<n+{lCQmOMv$AIY;;`__8eiOBau)<5;&I1PwV6G%KrQ'
            '{}RC447EL04J-'
            ')|={+vFHIkigI6S;xUPtyI)C^8=IbIoNT;9YdGN!OqF^SYyp5o}1iAugha1E4Kj`FK(mz9%=rs|IGKig!O;oztH?I)UO'
            '~Ijro#Aa98d=$jS7=p9C?c*~MTQ_#q9t_6m>#KKDr4s$jU88cpjvlW_h3o8Af82|oS}gv0X>UZ0<U`B&4SBl{S(F^yqB'
            '^Ds_d&jQa6AApll!SKBq3yPul$x&uMs;pQ6KilltPV0}LD1IaT1^w*w1#9TW<+o9F>LlJe)lI}dh|&6G!;E0AG_r4Y9k'
            'yxuVbB9@Fj{$!>g!#^?-}v1C82?Qx?7`t_DBb*y~Rh`D>Dh}W-'
            '5)>Pl556V|e><AxQ1nT$X=u7#>@+gAA7j9$Z&VzBbQ7V?id0@`ppn@(!w>FjM!BZeYHa#;&<B+DXYJ@bgYAJYOaQ-'
            '#;}_*Kd)q|4jod9oULXU5tp1r8{FaTMM>+-'
            'imq8G~k(DU0KesAr@XzfLhfdRB3Ra+Z=mooFhN8_g5_Bna}LYxG5vny$Flk-ZIQXZbHO~YMj{?dOm7~ACx46)3@cs|BD'
            'cH*<*u`KJ}FycOW42E06B|a)>JSC1R1(XQGqR1hc<qG8&~fgMHN-8oWFhFHTlL<a-'
            'tsu_#5gwv!;Y#TRbrw6LRDM{r{JD$X11M1dnLC>P%f%G_UQPw6t?T0Ms&tKWf3@}8CU%AAF09u*YxY9NIVg~+)PH~bKx'
            'Pg9GM$@k^%$ez;!?|mCd`n<Dn|LPL_{V9z-q!WsBgAT*&jH5U^#RZ<J-'
            'oW#5N)Yl#kP|t7Czea_F^6=f7zV1%+T)wM;gI7Rs{LU(j2A|e6k$8ur1^p5!89079D$W{Z-'
            'Yjd589=9Q?>rhgqJ%B3&{v78!RUI5jD_zu^#5ih0?~6P8gVe0w2~((&*mVgrOS*?xX`tXb_I+e4ry@68Nm1fwo%T$O2{'
            'xQPYgUg`bk~OJ*3Ztr@`JvhV0|O<v0+b^=1ByYc-kH7dzh2qW!>5ULW18DW4st;f&LC179JZG8VS9YD$(v>y6l;r*qse'
            '|a{exx5&&#Tr59MHQx$>H_z@2jKLxjJ)@X$1{vF<dI;)rvwp5FOq}0-vQvCZi&{v>L6)q7kNJU03D7VzzqpHgx-'
            'G5jvLAZ&A>+tdjoN*+>^^b9Apb$r&iz@-$hs<mW~c;3MAT9n;dWpgzro4L&bmsZQ-'
            'f}og_2Z9()lMc3i_e`arw=qA{^&{UlbIVx;nb8#*R=;)--'
            'T9A`J7(r6UvjL%0U?QopkP=xuRB}BgIB{dtlLM<K#VMxLd<6V6wFz+h?a_=So_H*E*$0l0(FB8+LJGS1QqKeN`FgM%`6'
            'AXRPZhZ~I){ukgy$ZA|&k(1c4dCt(Er@C>LvN2=Fd6Zbedd8T)mFR-'
            'e6~o|xLCoMX%ptW&4=epmcz1tbMe>q)3~DHFj_K`@O^S75uQe(%?yFxR`;-'
            'AUn2<Y+lW%j(#cPbC1z;{F^t<n&@Y%lHScYqH|0}E(c(22aF~sC8F~0|Rw7yc_AT64B+mAAV4-'
            'MX9cT%q;v0@9?12qf?VzmP@v8{A9<77BXYbIf1>13xkrLzjhCy=u0OEaXXG~w%&UkaqA7ySkGv3B0Qa=BBNIAHJ>@2uI'
            'Ohuh>_1-+TL{kk(ZB#|q<AtdF(FNbxIfJ#cI&@^o;G3=Ih(e4RR(#%$JBN?54~WOmvw49iwcsHr#*5R0zD-DTifN#JCN'
            '#<Jq?g%=kWx{I;&0!vU$+AGxgDp!Knd4f3&jH`>TvF#D{xd`B|?oLircpk|MfzsVd@Go?x}c2)CgZ2cjLgW5U6tgP0Kg'
            '_A_BU1LEm?nc3u{SSKYph#)dJnRy_(2^sdKj@P-'
            'ZV#%Np3Y;0<B#(iHJ@z@+7sv7UKA_TRO*XJ>~9DNP3CGyna=^}8My$%-|oMmfD&n2xd@8TX&RfdKs7x|Y}1KAz+@IYb-'
            'TDvbK-<V=>LAwDf=QX2gxDaXm?vF{4iNN0S9l38ugUP?saMH~jE=Nh=>=_-'
            'Y^Hvm;=IP?r06koOSQFhh>Vw5S8}JutBo~!(NyJY+rtNqpgf9yNwVC@WXOIWG9o~Sqc_uy1y8z}a2J$bd6EyZzz?SI>b'
            'n1?PytFKwf9W(j1omJaPbgM5o}?Q$?P92Ym_-g=12}a+7o~?ELh<g!$QvAp{QF0-'
            'G2<oGQTHpA`cy}S)UJ^=O|zk&R}n8xRHF~Xkrm075U9e3n{-m4ElHJz{no=T;yr9-'
            '96*~M7kIQk3Qsy|Lix}+@}fzNYEno1u)Y_nr<=h<Cz1@_H)RmJRUmP$tV}<65Q|eEVoEGvXVnY5elP(4&EAD;?&?6ShX'
            '(aHx)C08S+Lt21Mo24bE1Fb14i~<06d!wi*IB?g4#6M{DFln2kJp-w+VdzcnwZO%fg2Fv$4(aD+cx1VbXGaM$-'
            'OP=oz>Svhp0zyZZ@d%Z8%-!&F!x>xL?tRpd0w4;{{Q!NAKH)cxavN@xYwT!vs)-'
            '#!!;yU3o!Q;oT)S1_x8F1A#LFy4Gg!$ng8@Zm}hRC@}eNns}A%9B!ZuI~f37rjB1{f01rY3!eDM;w_I2w(N%+3}G|(Dv'
            'jqJXG6_cX%?uQf)Rv@{s^)Yjyw*%m=4C+_?R01JwAR!B^X3!NWyJ+l_lSc$)>W<;OOIbEyNi9j=Cmr6=K8xCZ`<n%QNM'
            '8WMl|8SGs4js&zQ<ANJ5pjQ%I`ty4`>={^yeo5ch+$ou*ha;m=c6JsfN6Qe2OWdGdyAAy-Q_%O*1exU_h{Z<DBx6MxT-'
            'z%SONNBO<8TUQGiJ17ls`V%{+&8SWg~x^I|<k3W*&YjhZf&|u)lbFAbs))1A=zZrCrtVGWRy}?w=;2zqv@$GZ}y%95}t'
            'G0IREg@Z7K`?s3b+smSk`$Ndil@AbkQn1wHM)!6>QX(&;02WPW|P`A<sKdYQ0&m-?c-'
            'A4te+sJ^?opU&4`h3jMv#C%peV+(PTt^=v2L3!>1nWxrLF0-6h-'
            'eyN)!x;hapnU%ZM77wFFlX<Hf(~;?_a>&UFzs8n+?kE%yHprf0+8jA$8F~IJ)x@Tq=*k`MXb%QI9mJxwrz3CzK(_t_&Q'
            'Hm(d&cMvT_|k??zD3&ec&WOSUGA>an%sLAI}*!w0Z5PSUcU4SEYJe(d&;v)HK|8QsCBh+|bh?@367+lSTU*{)dYjrm&+'
            'C|~evqzw7@&n%6o~Mf2HE}N^kmLmi!koAeG~)_EfsICx9TQYmHYo{??+Jv5ZN{OHmvB?`17)#uP?B_#QR7$WzblIFtMX'
            'u6GsuBU#t$)Z{63vtTMGJ@Hh{sI{b(4k0ec%&%AC{-'
            '*&XxZ!P#pr^AO`OMEG{mQ0e)lhkouMrmY;BAE%9vjtMZvc*h8LL@;;?{e}E}qR@2V5#141h3||ban*b?#<wXxEC>+=%~'
            'VnB%smVho|ANHLL5xw3&7};9xOa1j6SvLbXh?hUfx|rQr%{eXHj?2C@O-{-j_hXMtGyKYCag>8zBaot|;C-'
            '%BWFJ!g4i#Qsuq{n_oteO1lnrymLC}K6Qcd)G$~*EQjV^;o7-'
            ';7I1y=IUOiu!NF5un)0$<&{Inx*dqqbHy9#AJQmXIF46JnP-NU&O1&bN0n6qine6O>V}e_t+P)D?=Embo&EL4IAsjthg'
            '3u|#1AJ%h^^@W)@aK6ZChC`>^cowo-#QApPb=ZE+_^9eStyuZ4lRa~aBzzsToY4;-'
            '5JsFlWhzV(i2eP5K2dRYRUO$m1JlIH@oPq9+mkL06NNwD3$RV20P2xl;0Ia4vL|wejo16dxl$ghrlEGN?KqNOTomSR@c'
            'wG{k=D!Aax-'
            '}{q{4=$?(HFO4^X*I7kl}T%#5>&ZzPz8&%iM=JcMJ%{2HH1U5QJSn}BtjGxVD?~74jZtxzIN+uY`yl<hEks%bV@qxZqp'
            'J0}C0@nNpgCW^&;zA<mBqt4IkCsC2?CWsxSQ0(ys0K2rK~Q4LPwyY<!GJ|^WwFP<py=dj%q%|(@;Py&w=)X*#yv><&~<'
            '{69U$&PF}fYegL_Mfk#m!#SmH0#i{!_5E9%im$OHM60<e5xG=0Hfqq~3z#~{20|I9muJ@P9#`)yU>jiMPCSWZKCF%vI*'
            'S&!mz$?$x`RrW8g5&C{t7U?Xvz^}sBai!=h%yPR29`9D*)?`7#mXd{-eRlA~RS*}oZy`QqBaEdFf^fdUVi-'
            '}}0ckC*D6DW2i+(O)i$odXBW4;D*)N8*t2JrIq7PVlDi1ndts!}%8*tl#AtF@Nqy2{40cY#G!l|eYXn7(8>?;p~BziK2'
            'ZU@1G!Cb65K0(JUmoQ{gc2o5&2iW<XC?c}{G-M9Egph<v2G{gF=z8f)ciWhdbGjU2Q566|#qJ>Pdxv~1+yo}KZ-'
            'Z^uLc$XrL4x)s16NiZR`tw7lCDFOUsuAWS2g6x+3)o7V@tF$PlKQGJhb8aE=HqB20q^>O+N|U0D4slOc`!CNA4=9J1J<'
            'NF<T0!kKTo**N;)@!A&&i)g=pw42c;VN7~2BDKK;<Ylb&~aobI*ko$vle{zG~t<}Wi5}=iG8Jk_mj~w>|hHO$a@CQXfr'
            '>p{=4^ziFZhgEm)vj$NUkv9ILZO(~2I@j|0F7Rdh&^t2i0cXDzfgitEe4EnjV83n-v>ALE+R!Io|ml%aD|-'
            '$LZm#6(9L_(A@z(9>_2r2RIK#y;-a}IFwYbPXfD)+ouxP9SE0#?L=>)RW}ou@MO_c5!^V_qh>y?0Ht`Vb9@_w-'
            'n{D904Bi|%dxtDOp^vRU<v?P#0KQ;q!Ms9cY`=3GSFZaD9bOE0*iit^`bW{{Y6nBQ;42>f;Y<ELK2AhNydg077-'
            '8G21MZEXpePXlMfr6&=AQ*}=fg^mb84vYH&rl|?qo=;jKos{;%H}=Pd;>;p(~F&TrF4#T6N(N5p$Ae^ZUW;rNyYyC`4a'
            'gS`Wsr3PJeoTCixB!Qab2!}V36_`Y%xJTspQy?Y{1dDM&TobHGB+cY4qY%@HI2!g-'
            'MpJ23+2+SUzAW6;%G=6q5>1%mHwO_tsm~>X-)<Z9G-lpXs7*-'
            '0hz=kphZ(Mf24A@+KFw8Sa3%)9V<LO4+JNq84nD>_OKQ^b$e)Z(i*R9yMcN>y*@yMGopW}3X6U@)JLq7Qkg3{<_*hhRo'
            'dQUoBmzqS^iXaA8`V}<1ZwE<&6GTZ&5WK2tAzmqp>@;2pJsnN#EN)x0Q>$aVb_9%fzK5mS`tYsm4VAF_h^<|}N!Ypyux'
            '3u<&tqr7c$PnQ{Pm>?t(!6A{A`eqjiC-szGz&(1cNu5&>q7@U_eV4iXq~#Cc_5e&V|yOysxn7BnKYRNSu}If?D}iY&*j'
            'Zc)!FFS2@VgqraW;bKy8W&Je(s>3PidAC@rc5{7=cv2ercCH9xdL!R|cI#)T21}WSHuUo~?9Jd><#yZiqd8QD$(45ZQe'
            '-vKtSdRTF2B@85i+8KlscP0bT4?y3;lHK?=djk&Lzj<|N@hO8$W9yGGQZNPv)Q;ZB#vBp_YRUDyCTHc<J-'
            'XzT(Xt}rlr9CJDLHXhXe3NL^M5Wnh&=a8`wj;Dv3pn7*-'
            'x%PRV&ccroUS6V7#PJB4AgI$8ll)NX@b*FC68`GA81t1u&;0fm((*mD<g;a?V_WqB$)CGjIoIBf*;gc{jyYR%-'
            '~%u5LuXhZen35fplg}fLYp`t%uqV7py6moq{L?;}`N8T#3=%+B}>Y-q|`whX0wRLcQ-%hgb#7cBvp9db(aNrL&N3}n-'
            '4Azz!I(?KI)>x*OnYTBw_x>`%E(JY+!Akr$>PyP!Ie@(HG2;Hr2j;JGL#g-'
            'q@K1`3s?srFZ+{(<ly>6Rm7%bDjE@=LF94E!lT=ii58m}eVy@UfyqWBZL()dtPma2fD{lO7a72K8c*+-'
            '(Y}dgRn{ZGyX+`ClL6Se^!{+pM!shlc`b*CQdAzxar;0BAut;XyDJj7o*Di9TR3DBitC9K!MC*CVc-YzqYXam!sKyGL5'
            '5yyPTP|eYVqwA2t*D%y2D|2VfZ1Ne#(hW8_Sj=^E15&b#I@0~<TriQ7mxoQw&M$%iy*ZAD;9RVA+i&7@Kj3x4(#0k37Z'
            's&*}`dU`3E5wWfYF1r>hz4M|>dH6$WFIR**jtikF5Opwv@<KIJ(;E^B1qyTE)@4Ys4FlX$T4f+j<3Zv>S}ZP#Aby@tpI'
            '-$0SyUx?o6Kcr<(6Jv78CiZEiaQeEN;I*CEu<x!tl<(S4<f2{JZMSqWm+K66sPHikUWi49-B-'
            'cp@oic^dK@1K3B%m2=D@Y0n2OBzBzgTd<dkO$Hgr73=N=cxgLTccs!^Pt2-'
            '<;vwjDxKP8ekVTn3ucYH*0>ER4N*fF_e^*qHs3K>2>OJ`f0ePQ|!q4pQaci%7|Lagfyfh5QA&7;kU_ZwgDp<c1t=@hK!'
            '%q!Vz;P!TP|Pm;50%1|eh31@da04Wz&t)bgK_+2^+^;i8O?dAF8spWTu&il9YvF%mVsw=?Y(p)N^c%0tNq`1>@J52Wbf'
            'z3q?INehO?|=HD4*wdMV^~dR-L;~Bv+v;1Wx;sp<SP(-st@Zk{D|nC0CY-'
            'P1XEj=lDW^Gp`fQNO2tcpadawdVm<`(gInnE8g-N<9vHWT7i0SsP=v<NrZ8P7AR(Z!))36)7Q^pHYi4+e1-'
            'sOQg%`OiFrYXMx65W>U3VbNZMjAoJ8Vemsd!L~-'
            'h_;W#b7<WiJ{2*6cVIPvJIxnV5eLIL(`!N&g_#0mPr^Li@Xhaoe#l4atqn(@&)>h3hDg(517<t12z~07X`*Bd;fc~^~x'
            'iN_%%f7a#hrt;hc$H>Ud&iuNLje#~)jg$g0yDG3VC|$2gw>A8m3_D4z*F{{~>cA<|zGvJmX5j%(*_ga-Xfa7w}zOY34`'
            '{jDM*@<Im(>W|{_m*1&OyEh7Stw;G%d3r^y7i7FOFm^iwhg*hmY)%1#C+z@Mt~$)v;^hc?%rD@DjRTNAtw_>;kK!wv`&'
            'jHCgcfS;Smb{in!K{;!LbO4U7Sa4TnF*nw<dJ_F@UQbuY#_&3w#_Dz{H3RQ1W{t%zb<l6$|f^!uZ$dvx1M|o6`;*YnGC'
            'f?q+zWQUizE4#9=G89ayW<huDD6khZQWE;Hkv3wsclzEB^rW@(5HD{6U>0Rh{ECn$kHinelBC{R~;|udb+$q$7rTI^Z-'
            'KviasU4jd|7#80Rh>n*{$>%$YD;>0MK|ia@uOPqTVRcM4Crr+h9m_onxJ$MrXB9%D$g_&-EaUt81-'
            'v=Pp9MRQ?1zb=_Ad8AtIA`3Ar1`86RTw;r6Ns*e|Y&A1-=P%blZ;xGw@1-*CgJ?+Fm5@)0M^ve9I~5u(;4Qch9?-'
            'I%5e52AMvkdy+Qw~6$2aS*D12*RVG3ea-)7(S;q;8i9Mfr@?5W-'
            'Z2SxVHm}Plv;;<2TSFF91^JZ$in@UMhRI1Cnm?F*kC}Vb+}TfR^|&nB;GUE6r7?Ynl!JPDJ5-'
            'Y^M85+{yJLyi9e$WK762q`WeY_*Nnpp4+}*99K}GZCVP@`fD}Lc3K4$V#=`6ssz;D3*$?vYz&VJ#Jll^B;sr`{*W6ap7'
            '{h9M>S)~xE!ueV?)!72CsaPj<N3Lxcl%btP`}tmF7R#M?nCu=$FFKZBr0E8%{+s_dvcAA9MPIH5mU@E%VqHi;u4KBU4x'
            '$S4&<a-50OJw-{^awR-'
            '}Y?h8PXhQh_R8ZvHZ0UBl5bfJ|GM2Oy{l{b1|<ybcy>Trj4=VZ|8YofW86ulj?agCWhUH;${`?S=0<gIVUg@#|?Sg;M='
            '6kmZIYZt(Sy@AxtU<qA!wFyUg*|=|{0L-r`K**pOsM;q(-'
            'To&eb?!|fZMq#!yjqWay$lQ;y8{=}qZulR%E<l61DwYyNLle>_&lk{4$@@f_MiXgoqw}%kMnwl{gKs-U9-'
            '&asXzdn+`NSpPuAn4Tot}_(uK^N5WMuTh1_-'
            's1t+}+xNXH2{Hvu)wq$qX4tqIZJ<5TrivGC3B8^_0nbYN>?a<oN3kL^1A$ZcC@{M%?w{;4V;R(i?+ot3}W-'
            'ihGc89TIt^@S)R-'
            '$dl0OYqEo!LPl{JfC^fkk3a>!gn7+E)?<?)?xK;|pt5ZjoKbqEP*q1&;gGL&35G<ddEq*804KlY_yqNADF)_moD9#H(a'
            'zU3ppUx3|PQ!<)2`Kzb@B7MCm+fCjnq$fwH$w<!x;WAK>vI@uA&L{&`bE`U`hpFr)(Uxf8(0lb>Q!v~VKL_6{T`MM+mm'
            'mUa*p6zBhy5K8)G_RIct-k@+n}x7QuO8m;6U;Y?g9~cAAx!B6@-'
            'O%cuf?=*@cnj>eZB~<C*H$F(<N{;vz<m4<$#%02H@}jRVZe{>ZcKS_|tvLv*jZ3SRI1vI|633+H%DG67XX3ITUhf&%D7'
            'U_#jOZv<gm>A&o-#As<NNJkP=j1rh9<T1xhoP0&|G6vj5{qV{qDlvz82-'
            '(81MD>jmJ6B)Qxgs3l~jl6%}l8vjoDE~T9lz#4mC%hApK7E7J$1j1#C0`h;b%V?haq27=iUxi6@#Cx>qL)?-'
            'mI+0;+mHv>o?6H;y!L>6h~~pp-'
            'm&0kS4_5FmSrwv|9}min{fHf<!Gc*O1C!q<IQ;wP}E8d4@$fxx~ldVyyr}riJTWY8S`+cYY_?`5`jO@+VM_eBwI1X1&o'
            'Acv8THoy7}1jV{95bXRjL?^0t$whuB0V&I1ejT#<2QnC6CmA}e34!G!T4`clmkN~H#nZ<L?ye((*BXwSUaE#)x%q6f6+'
            'kKy#)96VQ0i^)c|H2ye;p0Q2AfZd1Tr2Po^F?=Y;fDgZWyRk`(2i4}5V$Am}gIf-LY^%>>jPj*QRBcxd#2(cpc{Y>eB)'
            '<v-'
            'N@;+WYbuo4KOx$}dGyCvA8mKtPT%O(!j!l%F4xq?EW;A=<gX{(xa31(ICGf0w|fFxGzyk{&?e?Hw|wWp{gBUNkF!#1*|'
            'O?9^sMm(s5`nAB%WWS{qzYuoxKc-j(NdOv6tjCM;Fcde5mw~UMf{@MldN7E!9>);rtv(HxI-'
            'EjA1ZawHcoCQP8=|Lz|bKg$7<ja{53k+TIKVlRJiRC-NvHr?Ao2U;|D$#Nfki^YG|DdtA8uG8(MpCc0Mbc#x%y9$`_$z'
            '91f#EZGmITnq5hT?_54>bn^IFaf4PCqT=$3UnfQAvM_p1O=u@obS)F?Ta<Yr8r%bo-8C=d7tC`8C){6?jmH}QK-'
            '<&#>GqXQ8s8V9JDRP+0`MiQH28%BKD{iavsD7uOUwG&CL1%h{_Hl%q=4>!pSh(UKRg*iYLw%4?!<03o8x<!yR}GrV;)i'
            'KID%p@hRCooI_O)Brv2)c2WnS3V6M+g1+KwCo0ljgzG^HG(YMm#m58Tt&K89%L>tF4n6Es{h>JQDq3vA7e=f)jX>q}DJ'
            'XnniIG(tqOLoSSe_3+iI8q0x|pAaZT<+V%eAp&6QJPXdLrttjLo+9aFZcrXliId)8%DkszQ)BhUT-'
            '~*rU{D?lLT}ahuVkuD~^Ih}PFcu%qiGO;JpNK0RGnTji-'
            ';&AJWu+}FYZ^APmfo=Ag!1eD!V`bn=O+$SUETDWh{MbMj`D!YFr6TXSe$6J}<=(%?ld-IRgu>GYnxg7VktY)VL^kgLAX'
            'Ng|+4cBITVyp;DKg=ewg)BCER}<!(G>0nQSQ1+w1{=(*>9ipSCbS*k!`2i~b-aheeq&TyU>-'
            'Fz;0Mq15#(i(4{Tj;1K*3g$pIZpT&ZXQcix-BC5aa_J-'
            'h*hU6x|z!#B{cppIF|t)Q2ikL#o?h%`A&^Oi=F9f3I@y40A4ehk2)=~+}+(iOX?18{635z|ZXAon=F`a6$vd$6D^wB;3'
            'BQl^+O$!f%+xjk6RGY>i~c3|-'
            'LGW_*15U$?Yiz*|3=wN68?g@yXTlnMPQQvzIZ*c=p3mwKRx5BcM=QZ%rk8INM#Sps;3{iN+c`6xHsHIjcf-=8DVZ6m3'
            '--jHA1QACNKe8AfTPh$mQ$-'
            'aQHBvCg3a2F+$+#RB)oDFW7OmoAHs?No#*dxk{+9(H|G5{Nd3@>pEydt&=tpE4E~3W7D>%><O&-oNrTPLTAoH{uI-dk$'
            'UvehCWA4J0%eN6tIaM4o_JhB*zhG7V3Pyi}8<i>MBZHGe<aLB7IRBduJb4CW)){%&-'
            '=2&bzq0^GL~)i~IR0kjqM>gWT)LZ$S-'
            'nqC)om%{q({QaiEm_G?Ka}P`6L=Yq4ec1K9I7mf``m}`1tD$am(o;i5hv>D_#cou0_FVk8#X<fY2Ijh~__)>E!wrFj2X'
            'KN1AGBGhZ?Ui7f<`W-$_ca*EapB!kh=A-'
            'p{Dj_eU@rISmtLG+aaNV{wVS>FJ#6ugE#5z)lsY$axjIl!R7PllPLFXQOCX;LTHMpk|G1K%Pm_!Qv`k=ed*CrlT6Qo50'
            '>5k_}gOY*s%A6l6Ma7O+zKm`l)3l||)Wuf|3R|ccW2OsZx0-fX?#Q)`hoYM|W5p|`luU!$U10i-'
            'S9VJAhnf<yZC|<zBc^)bW5f8S4*Q|Od9}2{uRS#i8v<;kNwqo@`3M#LJ=(RKsb#6Qc7sLFZ_0KXC^vS_X(JcIUBn?MTG'
            '@^9*K8XLv&1{hohVcXSP<a!;w8jvBob_fKef487S9m~@t0L!x)NR<IERA}uf}B^~D^NlCK5)YXnKEkz2SzC_`mlj4&*n'
            'h#v2xtGC=P1}R+G2V3Gl~|i*vQNhxm3YLzb5-SV+xiwLVXVmVX73w8fzMM-'
            '@|bC9x+}h*_qs3nk7>%GFRp&i_uvSC>ZV!m}5sjMY*?|GlH<^?XSy&uWOZGzGn}K!$Z`IbL)00t?4VGTHh8A6|{aw@M<'
            'YIK$nZa__-'
            '1q6lt9lfeD93KngT#q`IAP$O@cYN0hPu**WPt$svxIRkZydT}048onvkz?H#Q$#^|4r&P%ePK2mowcsC|wP}K}*Y!N!G'
            'kr~((o^A=uL`Q0$ClN}`Dr!!*Wt;AgSeFYC)x4R4wj0S(rWW#m|`<TPkQm9-'
            '1%(mkj@7Gt!E%=LnC94f(SEP`Z5k)PlUK7Olr3I3U&t6u}>}FKxIuSqrk$2G$_0#pAX+4y`2L1yQ`R?H18$od8i<B;1*'
            's_NvFljYEbrVH=9f3D7*_>3x_{xYpLHjLsvR9po#xJc$b+7Y9((lRA>XLZ}NcV!YS%kuodcBkyuyU!XHOYL%{+MVxnJ7'
            'og$)Os`@p3KXRLnl~tg{<!Q2eLmY0o^&BLNW7sZ=McCPz0CSVx5&Z>`<d*A7y0oT|u09w5c3dnh)Gx!fj8Qtax|RI0n}'
            'hehJ|!i2DJbE4oW=@CFuITI2dCmIaNngIkGO^)qgMu^$IPiTe+v#s6ru^tVbmr?z|@Ul`uV^en7Wun?G=OJdv`y2?t6w'
            '6jzYk!P)7rqhuB{gj5P-n2=}*E!W!^~GkL)nIZ#C|Mv20+>?-'
            '8Z)r7JjL2@E57&^{sL3&RRwycZ;m%Ku%P`i@czTQf$CK93j-Cg?5ZUsDj&O!GN>rpBHIXn#MqR0FOsoR`-'
            'IPf(E{yyqPkM{xOH8UAEW(v~+Rsl-&TI2GOEO@Qyh@Qt3VYE;S_HY95QeG|bep5rwyhx^Us-'
            '5sacq=A_CD9+*l$!i&h3$WniM8Y}nz$(fnp!Q0wdiV0w>gX#j_ty<q+OsI?aO|8yqa1E`=XY`8MIKp3~WfHc<&l<eR`c'
            '?{oNOXMZ!@;LJ)pA1>w4xJ6f#S0BI^gprlp+8KaW$Qfq+jObNsI?nsQio=OsR?gBH%6(waaLWzPjjy&&zfHR&%Lt7I(j'
            '=AGLzai3G7!T6!_SDnmDr`=zhBP`KLcWLKmh3`E5|^R7R4RbmI}i;|^CENCL!x4&jO#pnP)014*z?_ooQ0sB?!OT;9*A'
            'RF$^o=GdlwFTG{zS<QW?fn51(v1f_A$XL-'
            'B|<xiv+|{Ma0jH9txVJDQ*`IFaZzzDL!2GE_n2G_a0)YL~CN4Q7R>NzAzpFe)ymn_m0D@90YMwC^6qBo&d+LO=RWQWu}'
            'MJ_2C@SFoJz$mn=)2bOi`Fg<EQd)8<%x{n%R>ue!pE0n^mlb!fvw-M5h#gJtcK>LnNl6E6Gbe-OVG~E!=zMO*u{$X-'
            'HOdlh94}!d$5?+tE2On~p$Z$>$ls2q~b;Ul67pZ;>exXQOAp3?<lfDnuT&p49yw&Mzu2cwHUWfJnlF(=TH^yG#=2(7L#'
            'HNZVVv(NlKmTIkawA_5J|6>Fq6?vJN!Sdsn?hof8y(AVhd++G7+oEXCk~7e4_9Z9Sf7mF<qV;3)?n$<FBizp{c%vM-'
            'H)d4C1K>yDd6R8BU>9Y@sQNa-h4@hw3Hg$5$}j^E=SU{{R?s4oC{zg)I(*tuY%z7?U14LjovF(#3P66F?Uua_Ft-lTgy'
            '+Q>ij*p);<fTtr;kvUri3}EnsXsnuikVW1y%hg@*S7*_&%@fax`(k%SMy&-o>=|C%r^n$d+Jj#WtREzsutYh~=-'
            'ydGujPmxZrrK3vact<LM>|g%|>(BqDDF@1N+s$QQotRAB+$G^*voeN!ipDcjo7pY;%i#FyFgzMnTQ*V_1TQwLlimF?jE'
            'O~O(RR`TYcCfP>%|7J^1d{z6u*b9ZD*kPKm)%1&7%LZM$zl>7C4=CA1*#M!6!$`G5^e6)LmeLf3-'
            'tl%?3|cUvLYihYN}O)q2pk3c<AY{S0T%;4;6LOq>(f1SZx9i>`44D=`xMOeMiia~o_>yN%C}dE>0Zyo}?M-'
            '4O4oK~fEu<C?s!X!ys9Dn3t#&mS)_cuMNw>aTR*mE$74pDb~-'
            'Rt78;=1~{Et?VUAb=rPcBk;@0GFUsq2TO!Z=?iyNhE?#0me8lC7;|-'
            ')EI0T{Ekt$ULD?aiKOKrKx6|=ZZzy`dD}bHl0%YSy9eC1w0%ooFR_3?rF^c~jqgKw_QS#{m5YoDbp&Ko+&VPt1T{cAzM'
            'Ps}a90+?EM=_V*2B)<<+1F1g1O9$Z{Zj&9(>NP51V+d!PAMp<22=G(SMXD+rTpi`;nj~QbPA0H1J7`DmrA6+tp(7w=PS'
            '%IkH@wgHT3DwLK|ZXy5fBlSg2P5+a?rmm^NTk-6W$*NeZnK-@zKQ!x$2?483Roj!1bzlUY89m@x6gGe@*GzKkvF{J^-'
            '`2rD(zQOYtMWA=N4c7O}^Zxleuph|Y_!df)a8zohJ?YQ7VBu)lB$1`G)aJoAdL-yVU+gNVW@^Axo4BkdP)AhJMWINRwG'
            'lqHM^O<XRP$+cS4vM$ZY5QVc^1go?JNJSH64yamR&yNv*2I$=-xW#U?L^$Iq76>jIrvP%75^-'
            '`1J<&Q5cRW>eq7>9=424$_4)`=UoK+&xFOUDkHOx;Ix2cE1V_1^;sf<ukUXnN&g_oCF1Z=5ZRHEAO1F@l)!ESPT>!&_g'
            '>ZS!ZVaCNkC^M;gLk`=V61$MUYMg>Rx&#dPrq=b|I(k5jG=W@?(RbRb(;?smA`@PpbGdLai64#{-'
            'JvI#>g3J1Lm|RjP~%syv|72SbhktxdX}smBgS+@*=q`mjZ|T3^A!YkqxG4*jU!bu*<Kb5r00@`i^Mg^=AcH_)`KUVpHI'
            'xgB7kPTc9&YhaKjZ53XTJD4yv_&v2YzPeUA{@Mk;`{DWR^en6H#?<Hr-<;c22>2x-'
            'J!u+{&FhL>#Myrb1SyDpa$fJr!%r%iWfUq~-d<Ld(0h629aP+w>2JD!^vqctK@fA@Z^T-'
            '>{hD%{x+ifCh6bD!5WI)%dxmb9%nA{CZ1h45r2pe(3#P9Ej$b47$Jw8I)t2ctzKm=U>1Xx=7k3Dd_65F$H;f?A0VD~zd'
            'qPYS*d6vj-'
            '+&oA*!;#unH+4v=dmB{qn828fI8?T8LEo4>INkD&4z*r@^`~~AnMo>n6QY2Yqywhs8RPBUbMUNI3^w0+gthbXar|!&Rl'
            'RorCgL98z6)Wf?4OTmJ0{ty+e6u1k}fp<iXa@E9RmC=$?X1scwErxh>xCkLO^m6>3q-'
            '%gA?YU`zrvS*oVMiLj&dqSYhg$QkXuH2<jL6p*}pBh8S&z-'
            'OM$(>I=@`b1=9ryhhX4|D$FL)1maU0tpE|jfuDstxcU~usD*mk0*ec+ys>si-GT!hLCh2fgO>d4a(tr!ExnVdQpkN7OD'
            '%O?Fq@`=nEF!Vkcmd<Q>v1+Y0qJMM30z8f+`?girrIu|*!Ags2&wF6v!_mv=-'
            '#L&y!79sHR5kqRMQJ3TOcss^kx5<rwE4jUI%;MyP6)PKhwjK4O)-'
            'oLAqe)RhQ!F_j0m|i61hvV4a=8H`WoPm4eO(_0sfFj3S(Cw%yW4dUJPMDrU9qE5G9?#&aK11MhHUdFyBX}#_ij5h1aB8'
            'X)=JAAs=Hm_EuVaX#(NAFbNB}%JkWaQ|rxN3s4R~tLX1d1M6I>6JpzYaQP;0)-uxfutg6U`Ke8>l9!6-'
            'FZ#bz`(CIjdD8N9UX70N|&Fe+aKRmc_WH}%6{wIn>i`w)wUqCs5o1m5XL#FzIk;Be$BR6iHb`0DTlY)LD9l3|V))ZUYU'
            '!)HkFqz$sF-ZCt%1cKXC1$*kzTo~hYP(AV*_X{6|?Tl(1H!y;!itS`hZZ&*pi7QhxiGdRj!?Et>4tAEdEFQg<LRYm2!k'
            '(^2=-ls*&wkxya87Z<;q5WdIBzp#I&sskSV?HH&p{CtF3$h>4^BPp^#4!%7h3-'
            'X|G_e*GA<K&h!qA;aDyqu>=TRN{GFq4EtQ+|-'
            'ti;!yI+8FxwCQOngn=uX@CT+{D}7|dBJHxJ}$>dws3hEJU7k5tvmJ5*P;ylPkQ0zfUh&R))L9zA*_`)fa%xCB>bfk9=A'
            'M)sq<HZ%c3mE+|Uh!q2lcSssDA+zw`fh|Ai12@BhF4jk;U8xc;mbLjIMDU|2N`OhisVIrA|N`eoxo{RX<fAP-'
            'I$Zim;W67k*UhfrgbOtg%|u<dXhG`Q*F(6BGt)_VZgU94a<eaoV}?iI*?i3y%t5VG#3!RELv<jO@6TqQqqch)|`K-'
            '(4Yx84>82fmQ)cZD!6vF(5TGkocNvHyX8WTWzbVSY)QZm_fi-'
            'B{;^U0A!omt~;m$(rhMXI(XNWu4jN$|`hsW_g@&Vr|&!#d3*qWLeMlWK|~KU{$s|vg9|ouuQf(v0VI}SSjhQEHQl-'
            'R<?*2OGd(pmA=b`b!){<mUz7rE3oSp>rR3r>wTviYu0R6)($0S7Vj!ImeRccli#y0)br4PQvZJ=ziDW?lN}YFM+*PUFB'
            '|LKgn>ysAiq}t<iBT8<2{C?Wh$I1xh$sDyz6k}bOBo;UL7MWwdiV&1%0ibLRC~+X`xvi<J|G(Abj7OW?JsXw=LDBHx%s'
            '|dlis9>{3Z8nGz&pfERz*xKO2f9>Tith#j#{78$n9|C`^l8e0YalluP~`AreE?ySw{Z?G~I9a*I+H&}1%Zn73JX6701!'
            'rJlAiFItZ3#+%tj#cLA$oegMnPpF{SxfZnSvvhTtcmkZtO{cf*0vmb7JunY)<MZztc0BFth{+{tmXTiS@KaXtUV?6ETs'
            'o_tP}-jRvE*QwQ#dN%O}!-HOJnC)v)~k%wOe&-'
            'qe3m|9>OD>4c&Z>`A{sve1%FrkAnp)4SR0mbnvF=y59V^OQ(A&ByD0C$Tb)8&9usrAFJ581=<m+V^j~XW!ADjUy~c=sL'
            '8RUb^E!^|y$kL*6%fI#&n`lXkNu-s)ra(>si9>Xvlsc?bO?v!ASQ>ZW_7WGJ6l6}c!?`M>!+oM4CN|4IG-'
            'jr^wX1=g&<3P%=a^9`0oj2#O?Tv(n}w^+QHS6H=G*I4##4y=`;cC56P8!X1eHCEX-CsxszEvw|JE6dI98f(3j9c$C?8!'
            'Y917uIJZH`eq=C)VuaS6RX`*I9oSy0G+OU0Cy7oLNe@Z?IxN-e6sqbY%^-U1zcG*|0>fxv{*wFS3Li-'
            'C0{rU0B2XR;&OUM;8CKi!8$(S6FkDtXTTXoLSb48!XX8Hx_%I7wf_d#*Ck`WgV|^Wl1)Au-'
            'a8Fvi4r|V7YC&$ufWF#M;z-ll5Kn3`@o68mo!hf%WUQD@$(OHP)?qSC%)^g|%(29jkSG=KCbhv%DXi`rrLq-'
            '1Vl8Yt~GQXZ^qR|G$wRRa0$4M|73hY4IZDC10}k<-'
            'S=AVOu2mx87^NxU`Zz@aYkWJ$#RmHQG#W<u8JV>#`WfkIf=x0`@fY(+t=6$Bhp6xMy^uALH-vIYyh0D{a?#N0lcZ((es'
            '%Y^HM@@tOCCeB8m0DlQtOC=y4ul_3%SK1$<{yR!M+x$u3ZAtPSxEme@!B#9oWB;}JM`!h^1B7ZBRafc@{S?54?*vDyU('
            '?;x($RgVcSCjX~TC{q06P<haDNXA&q3aEwvQ@vxlH-3H*rfv%R9|!+EjUw7w))PYeH;(^=->q1-k-)8*XM@8yB-'
            'XwWx;eZKb-te|LrFlM*oxk|2Oih4%kVN1ce;j!VzHh>^p)*PA}2-aSlCFluMEVv~k+Cf*zK-'
            'L+cG5(0i;MSRDQnqHeT+@%4D3@7~2owmhcg^&^`e77fC(h)ytVdq5SZ%RqdsIc4_k1Bb`A;KxC4c*IiyPn+E6n%p4tZ|'
            '0?2=KJB&y>GDALIjnQ9HGZ&5jb{wqI#|d<=gNFewu0EBgaZu|MfZ5B0q^tZ3hTGy9Pmxr_e904A+kiLwmC)*fws#J^XA'
            '4k9x$&*u}%Fx9TG0N;xooVHhJiw6VBcka>2)2#CwP#qjT6p>|g;IlN;#t>H<A6Fu`eA?FUmdhQZvy5$6Qoec=Hs^RD9$'
            'M|H2Dvr$;CZ!6m80#<mq;sxG!}gl(V5aj7l0xU>i(DPNVSSM(EY!m2u?!5hXot98C7`?^5W=7IXur$71jBj9z|Vg@to?'
            'Hh6|ZzN>XXA6XD;0UGr>nR=fghC`q)h;Um#`p9fu@O3p5yxg`tpqnA3e0-4x}SKh}N6=}dm+o>TnHBBu-tSKI>gw*SD?'
            'd#)%XwwY|u{s=+8M^N2o2S(}3GM7!Ig8Y6X60>0lLj;UbLh&6@>olU1W^b|M>l##Dmk*UjP53pk8`XF5f`z0XFgxD@rz'
            'sl$y%q$$D-x)6ISJB!XW&)d7C5xo0=jlLknw;|=s%Q?m$-'
            'bP;?*yj<5+?_;>y7E$SCBc*+bkbXBddBrhm5O!)UTKeD6!bn>*jaj&J^G`u73KJC1_+?Jo=s9}jTKh=eV=Aqb%r$j3^+'
            '590!yKl`MZq0ho0OMESkx{AUjw+bx1Ym0je?Wmyec1YuUk52uwIF<KvKxIKVJLgvtOvi480ImnP_-'
            '8x0Qx`#>%AWyWoln?(vj<Cm3zGd+vQ+=yV|ey%0W<I&KhxYe67-'
            '*}q~AuS;i@+;vsWb(wiO+tC4&X<K4Ce|m5&3?YhfZU(?I&gkKskJJ(xaO34c!vLU(`-t-ha4KWi-'
            '_!V8|^=hAFC+V~PYZ4QtntyZM^<a>D8JAq~=+~Et>;e4}t8tv1Ld?p33qt>61wr(DtDD5Uc^g~hqLjgHnwvhB}c|&$z;'
            '^u7C+C-'
            'ZLm6^_Ld#J5Ggyl=@@nh+7l(ngYx((MsTjUtd_mkta?2ToIoNB>M7u%`w0dDwkb|2jR+D(j?&cd9O1+Zlf10EbxVQR@g'
            'qGE?5@VLt$E+5UMBUzg<tH>W8CC~JzC<Z>Bh=)rb+(`P7BUH#jgmWSJDp{t>2LHqaMl#P25D*%MY5QuN@^_#+4f#1<3f'
            '`Cymdki6KLMsQn!ah3ICIvkwRB3kjqb=SBlq?G(as`$!dfBD+~3tk)7q9n)z5m&<E^K$*Y1Pz@&W3lnE~!~A8}b>7?da'
            'qF>~yHfszRqN8n!$J*eh~p1lpg*|dmhE$0i{ecr;?pQ{-?k4@p2m?d?MuL3t-'
            'f1D#V!ZzN<!{igp0$Vp5baq|D6das{me(qrt%o---v_+Fug;Rpxu1Sv<BcHP<{CiOe-pqrRRJKZ$;&JoIu9X@?EsqZ;Q'
            'pe=5G#Bbw2xJ>kDcR?Lpi&#cfKNQop?y@>Tw~-_=Or>%Wz@$LUQ+M4-'
            '~F`56UV6Oox^`kYmV%Y@ZSOH)9Iqs&2#hu@^X3G8%5YQ>S_lt3W<)1KeR}v0az>;FpK%nP%&s5SN}35MC6Ha<2!VRz(%'
            'JM2FH-1+JvqLJe#h(m)xa;qAjLknais`(u9aG~WleSaj2E&9_0Yx&=1=?jxyOo@Ae`280&bVtbteBkV#C7-'
            ';7~ZO9GGy%I`nt~A3AjUJd@yaxE%rO{HmmCYIF$71VfJmhf<v!36B*s>zVzN)t*Gj<*LS6m}+-'
            '(QBsk($&17eT_80vh|zik+*w9kN${C1+$~Q7hDz_D>hmxhJl{)g8Vxreqv^12}LzJQ6J`{lH|yLvZTd2Ne|#SS#6#+xC'
            '8;3v5K;+|E=QZWsuM#%pMU08lX(Ramishgp1g5LdSTpw~<uqKvUWN(xWHE-6*AdZ9k9de;QM(-'
            '%<ZuK}fEH9^Ru5l0)VykNJHBj^lQGuD|8fu&^({C}){X;@C-_iic;ng^m8g_5aA^*-'
            'yBc`B)d216*3c_<ML(x^f6JZVxi(7d1ZRvJhuDUFgOX&?$Aa{B$goloaF=lnnH&wK5))^p$M-q(Gt=NM-'
            '`lq@Yr<5M1RuuBq6#*0w@s2ImA?HnN~A$XB34x5E?;rOf<<W<;ZdS`tKJhz>RJC?iSiYR${;!-'
            'T_S#?X#arg^djE#nCTW8{mgGrz{;*IU&JVeGf8%8EWbdUQzMCT=+aqL+Yz6gz`=jY#q&-'
            'T7l{X;5<t!SrjZrq2Z6_Vg^%@+0vr9ps%EY3Y$gU?QsK!cYEtP#D!TA9Vm{`b2D9c<p<c0LidSXLb{(#}-'
            'At(nN5J4ku{Ff0Y?2#uH5g7Yu~B67E}etRCowbwI;4(vsqRR_Stc@GkmED*}i1@TvxVEy%e;+!DN9IHKt`mT<!r?8IPE'
            'k;(}Zx{N(<`1$Cs-wu|1lU}54XwiCQAS}U_B7o9>*Na9U^q#G--V&uP!Pyfj1xGy3x$f!am%nbwml4{zZXa7U8_jNGb~'
            'Y1y>y7M+a%%enNtw_?uu@<@B}6W@k84(XMDekhj81wgPpoDOc~CFpo{8QAK!=v`)0BYldM=dL<)Dmm_E~73E(FliJ6y1'
            'U^ibN33X(!<l#7+^STHj^TbfTry6IZN3fPG8-'
            'tCCgK$lp3+^_`z=xkJP$#t&PM+$ZbNvQTQ%IcSe(V&f53WX+?^5K7)I;Q+m4|Co1vr{3nn_DwEqI-'
            'h#>Mq9s8QOF$$5+L(i$y%<#&{rT8u)Ozz_H^wi)i{M1b^mzz37DFxYVlReA!UL_Y;}bpzntxHzeEdWHcjb4bVWpO}7E3'
            '|^f23*XcVQ02TE{8<r<Kh$I(elb59MsEZk9}BSH?<X(pwvi+DwfJPSJ9GF6hxPce1iW}2N^RCf;QVH3Jge-'
            '1T3SdYc0PgL_V37J-b@PhFJR}V&x``BByL9PdLGWPSg6fK*0XX!<=q1qbbp9nFWVB&=5hQ~;ZHUeJ^-'
            'spDb&cB0WEWmWBX_;au?3V30^>X-'
            '$Hm~u^b+Mmtx=Br2$W+^I%cVTbv&nilJ)t<b%q8?)+X30@X3l@+O@|Z|lVS!FOo%djlvtF^YYQW|K&3HG1w>5=Q*f1pi'
            '`jNEiDCpRJ=H@9qk4K5s#|3bL_cOUE>C1ww1jUwF9I2|SW+VET4(&QSXwrgB9C7zg~p6NmGlYO0%2uaZDfnM-'
            ')?`z)|xiL-OxKEn7*G8}<tTR}zgHgm7I6>DVz;fa_U$6B`vKYjfL>$;0z-'
            'fU6!%cvxp$Lo)y7AH|Gaz764osTc~sBn&zZw4<9ZqA*tBh3D!P{_VeNgul};aKhdN8CsJz;muWrX2L4HA*J1UCtBDe7Z'
            '@4*&-'
            'Tq=?z$jXoKAy3!Lq$fg}1~Ay#+@$kYn3So;bq9gpClbG6WttB>w}E$F+y104h6u`Xi_QgRxJiMczofyKh|ErPgHhmvxA'
            '4{X~01g({#naw|HFu^ASnHi3-'
            '_VRV;4;9DEtVzHZmmuDzoUW^kL(f!idcyJr&g8_=^{>xkVO0SNKiL62Q^mBc<qXXTjDyNg6?lc`5X?ECM6J*JLBGfgTz'
            '(~*Rx1Zg&prZ}{W<_z4}66Qt5wrA|BY=+V{zBMDKr&-'
            '2mU5~WL<4Au2HEg{W*;hRZ$G6rM)K;F?Z21OaWdmw#GlN?l9eEWyDf<gm7%$z|WmkFuNfYtnWHuPT??)?s@}X)N8RZ>m'
            'FpDRt7oKS)4~oyWs7O4`@5@9?YESMK;KZVM|OCv5$%)RvQLsczqDG8g2uh7opg5I|!ZacTrLI6tuK%gN2`a@UU+WM(p8'
            '%yPs!3nA99jM8-_Gzi}P*&KQSJd?jF<I70VamSLaKjwh{?({bEbgei|I(D%$cxINj6qKbRr@K_Xl65?UU-OD2evI6XiC'
            'SEo^=!VS?V!_Tckh;C$Whea&#(T=aq{vB@ZDsHnjSXc$ko6ru7-'
            'xWr{zXiXjE9&LXVJM%hkeo}6Zno4!)v){B4R9z4%b$oot`e%IHZ!$$`kZK)?;v;utt}x&FJ}eD_$bU>ETtTV9dV_I&yZ'
            'CUPhjpY|n%p?~CB7aW-7|tq+USn?X>*lUc7clWky|MmV37;Ox7swY(4b*dNwRqC9U8dLBB2FKq+Jjd3J<2i0-'
            '2uNB>~Ru;uwU(kB$M=h<M<BL8)xE%L@e*EM~EMrGm$`O5Vc-$733D}^M{XUp`B^$Wv2=>PO#fn|-'
            '5c<0pR{oY}yL^&{o!{<3_?a2(E@1^+RQ42tzKpO=ygQ8Rz-yXUbkL*sHXi%@6bs^3;>&<xJgcOQcO&j&huI|<TCkoYap'
            ')NIZTSF$WuNJe2rD2Ky~rAqWuH5+n8OMzhm}9~>zS4w#K2#dVCU9l5aMzcwU=~*_xnefpeM<Zb9oD~r~X0p$3{5slLZ@'
            '^Hq%XOhM8|+Zgi^Q3#Hi^Q2!!{=F3K6c#}Qt81aUMA@#UZs~PRrsAE4*2A=Qv0SCn%f<~tg&AV-YC;zpuiUkR<PJU<jP'
            'G{=vaVvxrB<#dQ7Fvhqq0YXw;9VljCYRjcpmr8L(lHEa>uab}+7s~Fa1z(7{>x;3ZUpz+3vg5E2{?G<6&x1S$JMKa*?Z'
            'KaFq0>UZo6s;;X4IzXS5#}llSy>{%u(A_#LM*mV)C4FPzW!ovtzKg%{S8Xll>ktT^e3HCg`5;Z0?DAnpx3R}Ldn54)(`'
            '&35w2b0Ny;Wzf0N>7Y9Cy3}&q5bE7?A-I+o_}!Y|vt~RhALC{_HFlCP-'
            'IpLH+D~NTJs^R70<Qju0j_`6jK#BJY&5$Ajd$)cyH14TyTcJoY`{-ED`X4#cZ-mnT0z$9zW|?Uyb8D0MHmp`tWt~xWyT'
            'rqm2AZm_w(Rzuq9TFK4nGhy@Lxjt%2auZBXAYhZ@J9q2?wbcIAgUT=MfNYi5WftafuHm-'
            '23dN3#krS|zkjXSA3V9tUq;uBS4KL}2NKa{T!@1(VPCqL;Hc=PAz`w5ypU`j<=Tmbhd*`{6D$>ZLGijZ1O#(O<lGBLnt'
            '!D#O&=l_;;=L65|Lh0iCWv1}@UaPLk6dF37?iKZkquLQK82*SE~9nkT}h^pHsQfsyZ3Jo{F>neYYiPge{ZWZ9@94C5et'
            'D&Vjnkck9pi7jaVXImizLtt$o$<R$mW(}z#S_Qy^Oy{j#ku3^v{zJdkr{q&@TKn-'
            '93}T(=}?*6Fo^#8kKCEg**nXn7@2(!-'
            '7c)a^9R%znH&8S#P?&8nLqA!l!Tjara3}A9o~d%f}qP~!n<fDXQyrxNH&L%;F(!uX-'
            '6tZ@qNW^<2;sDayUqR>87hpzoKPlFUiOeMXrjQARo~U1sen4e6uKSvUmfUU)M1`%ATZ4L>8aiN~1N;_A<w2TjS7M1_iu'
            'LshK1XRTRvC=VH$x*JA??WGi8r#W~2IWrTG!6yzSxMU$!fkmFShrO!(-T+x-e*-'
            ';H^wcnGIlG&uiR37)4w?f0h)$m<rE~xC};auOj1-L`Sfs`e&Hrq|-Q(Flqy5-??mk4`Z)@8T~p3u?v5We~5fQFAAP#-'
            '1MOLa;1v&%=o;!Fj&75n3dPw!#iU>S;e){+XlVZ6Sl8Lc;@!IUaDTW>D1)+9#4<&jJHZnZXO=tMzDNFcd2|1V~Z@UfHU'
            '2H_vv1HXNEFkWplNOm=%)|w>7*fI(Fj-'
            '=w#j975&9K$BLVKDNuBwu~FIoB_ykdTjvVS=|Y%;Px}hs;2&yDGSd_Z2KEYsVNXS*o1<4~n%{vog58({1Y<>G-'
            'e*3Va`jd`TXTYy=OMjvs-'
            '^&?JzFj)q+`SAqS7SQIT6=cFwELUpeC!0$>syfOBG76<dPofW2YHgOf(SA5osKH^TlPtUBA#}1Lw;x{0;n~P&zHH~HN?'
            'xptyTygEJc1ZE8!4IqNqQ?V&lC`)R?k$#szSVNnOh<(Zmj7Yp{C!Es9+|)mll}DL!U<}U>j{rnK7y-'
            '_pCB>E9|cDDg6Wl>>Aw5He07{6_b$!hP}Op1<L|~G7a#aJUmPXh)sTsQ9H_Mt<O~j*!awCoJo9-'
            'O&aZBR9oHve?c`~wtiFQ<O9DZPUmIqG@1}oE`8b7zq7agJmR0$s6xO&3vfErbkoV(mSn@p>+8ku@e#3oWU*cs~uAYHMA'
            '5QmWFd^H9UZSjVEEbsZaBiNq2G!ZE)Fe{`_pDZ67o`3{8Rr@_RT1Ib@0DZQzup0_#CE}jg;gMumx<Xc6tU^J9D5{c6!r'
            '`{p#9(^bQ{ek0zr?#T=oSk#&#5|;_L{UR1?i_jo>hRpRD{83Ic5*<h*)3RQ#;Lu(PJH%HSoYEQ|%Qg|o>dUktc%&4YwT'
            'cBm0GAEG4vL4DqH6m>5khs;#)n(qkN8RH02<D0<7m7nc>N0c4+OaylKFXo6aBJ6=ft3jc}3tMy=Nm61m`Zi~Rn%yrbe7'
            '}Niw)7rR6!}PZ&sSmGi$cN6Clkh-pJ9XfARg%60U0-'
            'rgSMn7yIlW8>CJ=zGI#42*5be{h#hzXpEE+qotdUE91)C8=aYe~sb&U^W2Vo!6pl%>gM@|{C*^%0oh_IEX{E^!7@vi!E'
            'h4D&hiC9$X9q|(G^5PdR2*2c2rM_>g!h6an5fZ?_q1KW=*U4F@8MzfUCgIPR_6g*n2WQbzyo>@t)TJD2RK<$3M*3mp!i'
            '2T;gCo$cr3~8($YYM${b{;M8i*$0C@Ig8B0Ad8BV_5kBhZRaN(sAEN<0>kj1fZLR^Ubew-'
            'g3`l<qJVHEk{BuDSOX~chjqUli!1-KG$7(V;wPzl4C7<nTeEtnac6Ar<otNaGqj}hR$+<-0-'
            'p<w^)EU^jU#TG{;IR0=o_9PC{_>hPA?eGCmRgmE1be>_wbi0F`OEs+0Ujx-'
            'n#(0gi5_kPf0^NEr<Xej`M7h|E+dT;N%K|Bu9xnfIid0<>hl<k;DDCwMZS8p3&9haI*JgUQozNiR&DW=Sw}vdg`40Bie'
            'xZk&(m=0&20O;=5pEkgfn3Ta_>$ihj%)VgqP{9vXDP-mC@La`%W}!kxEmgRG!K)F6T!}Z6<(ax2FVRApyl<Hh{!!Bu^%'
            'eXD?}Q9+wO!rmy_wc<2e|6DjOe;2%^(IA^dfynD)0H#_ZNG+}aR_>tFH$YtkB`<c<<Sw_@D=#2N&-'
            '50T9UQkdA4$}CKD1N$Gbcqo&XJ$K;(<`Pcoo^;Uz@#vMPpZ<}q?P`b7YJT7seFi)ySAea(8uGsIg<$J`Y_#+PF726Ym('
            'SVEnO%;6;&ZXib{7_^E8qvit3XpMa46IZj<^Nm(B>q3+3yA)nN2Y9Y6-cJs|nIwOX2)$Znn>EH}Fb~f+lZ$SX-'
            '#Y+~FH$UF@ub_6=(=Vtqb2vrvrlttc3rb05NM_budoNFMrcD1|HE7)TJ|;+&Zgf;;T@FfXspgsJ{@7;?)RE!RZgsGR^E'
            'DAk8I8D6+3;Ru<S9SRXsJIMB@H>T&uRb*(u5w{v}aasj5(N-'
            'pi&dmD)&ri&l#*=bf96lS`#rQbOx9_KqPjI7c$^go(Ov3uFO|Yi<3U){|fJCnjEq!za7k*Pifzm^zri+F^&7q%h%DD>>'
            'XM>rWUAN)WGj%$|PJ|8X8u7Gq4vLs-'
            '<1^8xs8YWVFU#1YH}^5Dm>C7b9dY#f)N)`H6`7T?qr}RxgWj*@An%M=`t93R{FKzd8Wk@G*5ZvsF|Lf{KWZXg>HVxN<I'
            'z<0-EF*h?KjP-'
            'Pk;-W132K82Tj>Ea7ZL*T0hQVvn3wGzWjqYFU*;!9QVf*?__4xJWCuBj=>)jH^Ej&5Sp`8AYy|s6kN)Jc({fM-wwf2sa'
            'W!A{1?7eI)Xg%zA*Q9DQ*5yj%h{ZM9rxRGK?e1DT&SaU3fF}Q!cz`Sw&XNPlLEdS7cB05r^;2U}Gjk|DB4aGhg0<p$<('
            '<A8sTK8wEM)-'
            'REFl?+Rv~V+?h<G)C7|AH#cgW56Ep#s1it@aDl&%;JhBo?wjaWgDS%>o=BJ<_?l!wi%zggrJ4<K8W9}j$(fnK<&E#(A('
            'RBO4-'
            '8fK20_{t>IxG&)fvb2l?2ZP7E$sUrd|QJ8{eI6v&K>B7z`@{u=G%tav4E3!A}y^LrNbw28wdvybHD<}c*%0uHfG;)BT9'
            'MYJn<9G2IAf+P7(kacSVIGhi``xleo;B-w4wmiX7v)>@5d=rUkGA>U~K;<++j{o<6I6NKzS7fqLV0Z!M-'
            '&q5`KY7_zJq6^oT@DKOnM3BoESPP-'
            '47YXVff(k~7|)|{!!r?oEL#lM!U3wsmXW$J7kbPypA}H!4)=`ClCQNskXi18ZT?k+>fOU=k1UY$HUh8x=OJFk9mcuRQN'
            'V6HX|xi;QQsPzWy!&Ra~r_i^ChV-'
            '>&ELhD2hHQD4u`t165vR0M}mYLQ~pBNEfb#IDdCgepXC1ua;obibW_fnZdGevBgDu+d!+hpIrG1xbn?e#3OyE@N+l3bS'
            '|031$R`M#?GRS_n`mg6^Jfl>(#zniIXKlXck4W>F+%nJJ}C~Z+}Am(P=ynEXJ^hYz&MGN8g-'
            'S=vo>8GwcURw9PZ{^fRM-'
            '{bg|1(GYl@><KfXTIf9fQB=0d!8uO)cs4v29UIk1O5p}*Qp>?Ixht^p+a2;kX*!<p3|RG5kgOFU_)H#2VZ$H%CgwzQ2D'
            'S0zV?`!8^CgyCzX?8<y@`Q~GI%_6fWl*Yp)$#bcqb0S)q!w4eMcMAXWygzuIcc3T`;)6j%Q6aKZS2!XMshU07{W!_;KG'
            'D<y*Me3QO<Ob4i!s{JMvr+c29wFd7b$(uIU`-'
            '47<dZ6QiGY~kl%FbV0uO>OE#I2*S<g!%ux;Q1jV7)>)lhkQQv=jTeC{D!q?V|)>=)w&@G3xaj#?}*};WpH!-'
            'Zn9vPJZhY`g{H1BP*;*gM)NJWuStRt<{dcoB{TDl3P_8tI7gIwlxcEpgrVbN7|@k~wW6+Qp)3NM3^w6wUK`Np*@GT&!+'
            '7ZC8$1)Nhx;eJP_@PtT-V3b3Mo%ka+V-m+r!QN+7(6$20qc#C+Birg&5*`^wV?MJ`c}+j01tAK6tfy8C`Zm6pJ>^!#%-'
            'iRDka?eEcwzZdvjZ{Pbe5X8RF1y2%WG+B>0Uzb`J~b_LEw4(PiG!-'
            'x4uul?j_7g&9V&VNg=$oL`5*qsM8`3dmj$ry~!e+n~=a-rIKKBUMypk~-'
            'y2#^(KoBUn_GtQUe9QZ|6O5~zT(Ic|Tx&_>o@~7(?iH%tlF5Piutd7?~`tT#VNU#i?#YH$fX6=F*F;}ppq6qFiGsG<8d'
            '>Gl>4`wbyU|^CCuNuoBJ5&Nf_wRvqECckgv0)k8ErOV{^NB{=2#%JAP%WN^U^se~PHg9aFV0^vUfYZ+d&|N6*bxk=Jpp'
            'qvdgy`$?XdN25^jG#t;=S!;EID3&ePApwND&L@$Khm<gte=yx&a{yYykNpDeg-'
            'u%t`VgxQYrYar=}FJ8HxgC{S{V)u$QQ+0(UILt1DPH_)f)7s7?F1!u*`JTd$u5dj3w2;ivoyJ&YC;a((E2umS$K>p<pg'
            'Wj^-'
            'rw%SjZt14wW$WOy#%|nvmqkS9wWnlLz{>i$Sdx}gYFrieAN&KBLeCAsUavhZ~}({p1|xL4kKtY0<v~az&V>n)B>d0D~6'
            'k(C8z*-LaJdo|8rOn=?tyRHMm=$gO!Qf39D`ol^c8v0TwNgWf6>*-'
            '(7?9OHaV<!4}xQe+pDz?*XIBB5aA<p77z_cd9M^5B^ldVoU62$SxjX^d8=a*l8{)AKV7I`<&s=R2w{%6ks2=kmf{bhN6'
            'a_J5G!(rNfC{R7B|xM)IBkR_X`bY4Doma+P2~Vh6JCs^bm&sQ>P7z8?Bph)bD^>p#!`jsM_(Vt=(&f(&dc>Bc^G6%4o9'
            '(RS{gVEBleQ(-=WReg2%^n5!*z5Kx9cO5Y6WvGs@2}Um34T=Zb@I|#AKCu?S6M0)e@|Y@#Ow9xN@em?y>j!O7S-'
            '3Jj1T=50Kn<6Cc;3$d%uNCQ@4TF#NCVog^#&{F&HS%_ba<`8|C9gfe`J64NoCCXv5Vo&)ua8BU3An_4Gd$uN;})uLeA@'
            '8`1INXdjz&Z|9b%xwN?dFffRf@%N`tlTi{%IH>~?7q`UN0scvgxD>1&UK`+kZ0*{6x<o;9#b$qo7nZK)G<zzaX>p!Pwu'
            '=^2t8@d7yua4F;^nY4<?Tqh#{U^bWvEBbK-~T`1pB3s4`L73T$*H!A=@)bH{vSN7`u}?jtG8?*|6d`f-5zChT-'
            '>oNTn85TOb~0+1dMz90R4A`Q_aeL=IfSAu<^w%)YhM1taj<)*wICl^GAw(JTafy)?R}Wq57cou8V%jF2bV&`$@h=2(E6'
            'tkKd{q$-IbcG<4d6+e22<5$#8qUiud8e-xtOx1R(IMX*A=8}lO-'
            '@tENpVsfpYbdH&!yhk=Utnd`xeRYE1C)rS55P%Oe+F9FlDRz0!ChubcFyl}n=sh*WFXwBR+ZFD3>UtTRs|W*G^*n6+Sp'
            'g<X<XB&X>RA_?f9Pplbt{FOI51amVjeG}P}^Dxy=@LC<rV<ZB}a+q!$y_?Q~;;^8oYmBMm02l5!dT$Sw3m8uw#saEwz='
            'PSf&b9Lua7;Ho<`TztPrdHsnUw<Fh0!^po|1#kWT>r`Ln3#K+;X7m}P&HGX<#Z7ZBR{ZY@<rIiZYXdvg$M8USPU(DC43'
            'lMLl0Jlcwur^m-g;7TrymTW5y{xssG1mlA+-*>&A_oMtGH{lKH*VHi1ijq;bncVe$YtY7o7z^xz8!~g(kcr!o#Q7SYz-'
            'Kz$^#X)6*vqp1ofFb?8uWf@bSzRW|u`89F&ltyh`t=foK45dc?qruav&tw-QWx^&shJE70f~Xz@4=lY&o?qr<@u?cNyv'
            ';0hEBMMKGWExl_wAD$}-'
            'px9d<jEoh<kIWU!&GZD}nki~a<3J%lljWczgKf)JKvjN+UgW1wL{#fDaoDpEJnB74e^1wE_0wPYKtmmt*t@faOuke19g'
            'C@x*(@--{u#?}TY#oj6bL$=z#|2oq``J^X_KlN2pJ5Li=q3`Rn&oquX~L=yfZm-FASl-'
            'S0k=ZzK8oB7c%i8^H9)t19e<WP;}gYDl9$?!h8DAWo09_gE;VCG>50Hs_@1*cv{;ngC|`7P$GyIhmRYQx`wyt!MTqXmm'
            'ia8zdE?=-~<DE@6&vxm#n6F2{2=u6S$uEME8ZXFvtJ1)AgkWM$eSzc;X~{{gw!?1*@>0|0sTqH-'
            ')>i?U2tol1}!{Ws627K<!@+=2jGvEAEZBbY=zo;a&(WhYP9Cz5$AJOh9~|Du>@a7mi)|!Ti%d2Wtc`!>*DblGzuGdRac'
            '?yi67@{WFTEp2y*NiILKko7>T;f*-f<ivZo>yNDb2;Z$%M^6s1kPqsFc{2jcBIln#;GtF}l5l{ntce-KW9yvTgL@<4>F'
            '1w051B|{8;2Ezd`X^($H=Y+Wd@nXZk8mjX1@aK#6{{eqiPEF_F(iXgz@mlqU~<$GvUQiD*3m?qwdpLhKR-'
            'tQcCW>4AOv@8+^9>BF`WF*J#&Yo*cXJp!j!NdnjE<YIr>$|wa1?^X*&tyuIbqGwVjAx`Aal)Gtg-NQ`{I*Lb{h7C4T#_'
            ')2)i;SO<0R;qZNUu*v~^{FC8zr76Bz@eI5d?j`{!hyGjh(e0uK%H2GVtDe@tmDB<#IT{4<>esOCOCLz@UjgsA5=e1cBQ'
            'zho1*PMKxaZ{<yfW@2awX?6ZMOg{k#@s|*O4fG_!hQbY%dim<iYJ-'
            '>5!$j8tiBoh)V|H#7;3im}&(+GjSRoycMoIx`)E-'
            'FF0Z7$6C_!0^e`(rrr*&WZ<U(<cdCkhH0JLd0;2Ri5rr@t0CYgvL1aJ>_J3Ro_=KJpvu^J?3Z$bwq!{-'
            'arY7!B&E`Ly)b;@a~0i&mSb=BF4m$$5k!GE11eiq5{Z#O+#1<}OCs;1Lj5xAtS|z_fp995IYKAwN^$w2OxSI?nI_C{qX'
            '(XPp=s6{y5TTFnxP}wZdC>D2A}Dk;KS@uN$l&m0s;G3xcYG%wCzxYGh7Lv^JW$n(z}$eSs$ba&*PX)EOA^E$+DDeh1xS'
            'ij6u)>R>zP#=JjRc>?7)6&h3GQWkT3@A%&$`t%P=>_vkjiFL*X=9|?N36_;d3=miQ2aByiR)&>vIVzYB(?w)##`Z+=-'
            'zN=t!<qAkkx26pTK9+6>bRpsef%JKw1A4dWp|4O3w5O$@ELSk7(in)Z-GjG|`=XY|9A<-6F-'
            'Uj^qh@&*=3GmroA``jl1CA>PB@^3aV<og^T*cnH=x-'
            '>fT}ufAc<v>;NtlJv{k>OS;%!7DffiRoK*$!>iwADpa#5Wh1pLMZDH`99$Y)yLHAtohBk3^vgy$*nmav1?s)Wyym9|hI'
            '`w=pGL<Rds5p)B^%N#X7U91-9uCLm0Ap#h4j*o+gVXDd<Hvt|C^EhaM^F4@GFOM;<x|o0VB}BeSbGa%UoRqC_--'
            '@Gib3$pL7t;uq|B1}{+sGN_C||)u|!3d7o;bg>ArMnT;Wkg){Gp*ppC=yOJhEff2dBge@K%T0co(pc#`>WdkOk}--'
            'Z&wo|q(5L1%e9Leq!qsKmVpxL&so&(-'
            '_kkW(b29xZ{t%j2QLI2g8l{0rG8w^1d<r<BaPfSlnBDvk*0M#f|yj*pYn`~~EUX0okz=HQp!Dx7)84TO3nF|qw7I%vg#'
            '?d}zzlfRlxUBZd*91%LXKY$b@%tSurY!unEg)DMUWR`FF#_Ik|A>KRyG8-'
            '~UuuKtbIYwct&k)W&U;tMaAA!_@VleNN!DSIaa7nrxrmlKJ*Jc^WyY7u`G49y^B8C`T>A`$A4e+a1A-'
            'Bh8u?OpWP}ckpuK)d)4tx<{zxis0zs%K$*+da-kx!#0V%sUxy991sa3@{GYr!CEBRnk;WhaX_kTrTqsGwa+t}eWao-'
            ';d1Sk*q*t#g-'
            'js=WeNo*OuL!WsRRhEPkXNSsr~p&HujsCPv?#@Zah<8Mx5<1!T($h5(x>3pB#D}tt%o8gUp0et?W3GMnJs2Jo5cdoP$_'
            'tR12J6|!iS~eTXu^DwO)lqgo(jmJqlvVf)E#7LuzGHlx3~p}BJ<1KqYuBQ>=~==`@`jd@jri)Q1~z<&0Y2ByBsumD`K{'
            'Z+x+Jv$HyU{1qt+h0DdvDX5}LtSCI~-$pM`cZ$sjTzh^D6k@McpGW-Yz}Ru7_}yk47WiV34uH5n}LRvqB7*#!{#hY?#b'
            'm$@MR1#bFRlg4Rmvex=T^Im$w{Wt!o(0Uq@t@~haRs#gqwUceLQ^@h3DL9Xm)0I=M@NX&=#~xe4wqJf=vNR9ubZVgR)l'
            'E_{;|6M9Y0^_$6hnqbb|TldHL(4{YMht8ALeZLAwL#0l1r|2@a4>DR^svtu&_Rm@=lz^s%tvPEo_N<Mef1tHc8Oia~Sh'
            'azQP+7dJvElMVOCv^l06EviH0nG|??^>`N0!c;(|^cQf>pECkPO(Rk6a44!Uq1}ptiV&YLqXu%t(KcRx!k{NVS%_gwBR'
            '7-bih;VpnxbgMmeH`{pVr}$FM^?=nQhv-8dX-'
            '$^!>pTde?*YxjP<~g_0P#FWquHkPQtOxVocvRDY(16gGh=Ruu>Rbaw}Pj%=JzsFAVa~L;Mcv?*0z8)e2BXCm3lRdnl;$'
            'V+PO90O@8+tevq3))(<WA72rz{OOOCa_8vPvT{6n(-'
            '4hkPhhNO45~j*fvZBPkiqp3B@<FXM{fwuTP*}@``=LK5=i+zJ%Kt=btW_RBQ6|2h>hmSSpSuu;W~Q(#vi7T7g0B1L!%`'
            '7?(ac}&{BsZbt2I1w1|1%nFcT4S0MNL5cGd2#A&ue{Gz`cqpnqg_MX%DedH$ey5vIC^-'
            'RX?jSMDBXin?jCV0TL7`BKW!{J_kc(*7Dmh%SFZ|!oNcVQo3$M9F0_ca0YcDUnU$4Qh6IEkZ@2Y}OFMf;pX@jzAt>b3>'
            'qjYqQ~XR#yEKH~w)3}mR<o)%iWzKV7&nZY^R{udj|^U3kcS)gU*51NVdfqPdLtGY;#+K%+m`Dtn3c*z8^47>1M`3#ste'
            '8FzA2Dcuwf?GR0@#F46-'
            'S4r{xW4KJ?9ZR13pNKqaI!wrdD8&b^ZCG@kRtTC!N#&H!st+W2xlxSMxWlT5V^LDKL3<Q5B)Jm>SPP^V@$CpdnbP1au='
            'e4_}LnB4Qci+Av}Fb60eILhi2ECn6-'
            '46EIfOO96u6{kDfR|_GAjCdn%Li17|RG$`lNyx$J|_LXNH89V+0Lg^O=U;8qa_{5fA94)Uzi+odQAnsF~s=UO;XcDoNr'
            '`YIsQl0sK`%HZL6S1_JX<a}7w2cqc_a9cqWj$AsAG79zJkl_XT($$Q-&@sX~u$2t^`r=-'
            '06Y?laf?T|o3jSi_#8~SitsP>>gxGc1dm$1h-~1&5S6-'
            'mP;su<ptC6Jt*9Ic~Hi3#ZCNj}w{HR$qirbfpa!TCRaGb4o!LRB7%5S3vj>Va*9qNIUj9kX`A*z_S-Uy|}qfzT;D=I}F'
            'gyKFQ#%k&+s%=dn+JlRr@yH0O)*Zr}dwgIl_nfp1>;QgFD9-G=0Q(-@XVyA45N76WD7#z(yBrZ63o@v}j=z-'
            'moGG?H<>s_jhu}nuIrxvHgIAIPI<(9K9*Z`p36BD8?jYRPm&f|4>WON<o8hsn2rL_)1D)ax(5mo`<#5209OLpPmA|sVU'
            'GNlcd+ow3P*@G77he+9^)aX~ol7$+13=9@j`iYi8|$q8J6v4KO%4`?;PD>>O@EvO#oTNhS>cb`^?kVO^#Fbu421=`Ct#'
            '_uDBP*A#-'
            '}whIIG|?%JrQB(cnceFEknUI=fQy6d6`(Vm>Tr%%q+s5x}onU)nZjls?Rnq&>bapzU&!HgvQT?rvo|a>WCjS90N=o!fC'
            'tp3p_C0_LOh^qj;MjH(wb^a8Egp!Dim)+<+EY!h%NL%DI7Q=*BVoqp52xI{d+Z7(yPa0l$GAK@E;Rp2!!jVnL*!Y#S|%'
            'uJW%#NR%bA#-'
            'KWWLYcpipAjh$$GrTJ4P*dm#~clxY(1_75u>)c270IrkR$w<>nEL+*Cl7)XGS?+EXfaH3~YvdNW<05%&&jV6ve(Ml@Z5'
            '>ciX6{oWXDX_yI%_rAtCdi<PC|M?}PYzU8T*uv>-'
            'Pu%^u6s;e*LZ6@jMoRV(&Z|_Ymrf#kHa!GanHrSgZU+U4H1OeBfx&Jo(7GT9xSZ=@Kx!OXc8su$pTC2GO-'
            '*oob}4u&^+8Z$9()pag;eEGn4NBdX8R6<Ywkmcx>JrC)+=x>9L1YAO7z;hf3V!%1QW~0X(<2Gh-Js#58FGU;n0RTFz@j'
            '#dUWs^9xpbc8*TX5Kk8SreKwC%@wH=c_26L?7R$vMcepsSrW$Co%q%v4rXseyFM<yJAXXS2W421`zzz2!G&NNdCsn(#?'
            'N}!blNVxt;4jCg#*YY(kulV6@y8#NBjjoRa<XSjJWP3rv3$CBp^mL2)4W2Ob9A2?Rq<6}-}q5P%kIjuaqn-UbWDQH9xo'
            'yrFEnA0)GOxn)nDXD;2DfMf0C${&gPuBX9K75f^lZ-J)B#(2Cko6MUTyx0i7G?V-'
            'yB4r(7=J@fs;M%WONiuC@k+)@_8RQxL;767a*8xop#GC*gv)BjddP9e7vDV#1n08a-'
            'M8%Io#XpY8>C+|7?37qEd`EmvS!ucNKqU&(~i7pgSaMvl4F08eEzTpJAo(~KedFY^KH`z}tmODe;_`ERVxdxg>5#2A}i'
            'KEgTWZgfe18qDK<3@h(029eCS(6sphoLssBNLL<ier?aVEpP=9o*L@wcpZhOYGJ8fFDxyx0XNCHxG!9SeZ6-'
            '+`})OlICNk(JHOy2sH^V4OLODl$TAtYvo`=fL=3V5oQ`99>0L;hKZk9<;4LZi_(+<vm!Mq4L&DEJ%-'
            'otPg`pG1u<q#+(A^S5`jVvBa_4zK!?2l#=?Abz#!tfIfID=Tel^igQi84VW280o4fQ%SK>7NF*exZ`!E9eKEnV0_-'
            '#w5bavdV@=9(egB3IE!*B6X<<WbY`J1*ySWsRHpAa79uw1()j%#S@GU4K3>@6WFXjq(Q=Vc>?_r@X0hn7SUkoCBjXE<p'
            'FAa<Vz75WV$xqVa=EcwpQfR|l`eY^7EVKXeLaJod$TC&Hm<hc869N8|Fky|8<10j}SyfOa8wu&dsWs*Z_~xUsLaR&-'
            'jwIz%ujZ(rezDjyPAs6n5wD?z90DD)lO1H1y4LF`r>4tv~V6+GUCf2yj9bDAA$**ieaeG!%vzc`0)U<e(uuF}6jY(_g+'
            'o(P^^L`V0p$2DL0VH~sZl%FRsY-up&TY{q(p3^ML{itqe4(BgUQM15o)Cj!@dfAU))~Nu9_FhOn?b`?}5=FKgNTLUdM#'
            '<mhtBL9kA^PX>ZIm@VgRM@wXk>X1cLwuPaohWh{Fzm7WSZ}?6Go}YCO#r;zq_=y)gKghaj}hN_#$7b2QFiU!~4QdRLZ{'
            '%SsG{IhV)7l+HHZI3LEHt$87rTx-'
            '4W=nSr%K4LOkN1vTndLG0Bg+*04q2*GmbGTMq~3$LT%om(jWF+q=Gq(hxvd4sxd7^c2{O`)C_&U62y#5Mvp89jl*bY(0'
            '&FN%KGEunqZFjd!l06RHOc>H4r{kP)`UFH?cNW1yd`RCK&9G@NC{45{p`fZ?jJPM8<y@N9x1UOa4jX<S;8M&?XmMY6%1'
            '84Rq%l@4JCqCsp(R*e9YMz0pv`UKoDJ+EK>eZu6OgJ7>P-'
            'H*#@WQG`Zg81<GpzZM2|n3#VADVe6Hw5JZt=w+xPX_VnYbOl#$~{)y2q&TIRO>8U*al(80@<jhEMncu;suk_W0=p2xpE'
            '1*Ht~Jlu^Q6JMW->lQ%Rq-ok2bArw(9hhtI)Vf%MSm}{v;?bDt09`Qy%bFMQU77oL~$H(A8yA&OtdITHZ&cHA0%E`&eP'
            'LllT60EZN%pCQYjiSU1RRkBqcgqmW8f>E`mlN?NQ;9|E)LC!M-QkF0JcvK3!;ZTvK~avOy4Gwuv7-'
            'Rw=C8tqCC*^4_JAr}_)6<n@4=yFXV~7W%^q1_N49MmC9C!+!UvgE7#;SL%&VVY%I_BlPpl>&zf=ZhT)hN^-'
            '^9Q?&YbW#$Kn<nE>3Hm4|o+Jt`cX6>XjJs`awHgr&tP|TYO1mmp%QcDa_fIBZcq(Hqt}MqomyTE@{=Z#T8)@*lDPV6Dx'
            'ah^xz#(x!hV>mF7jg1uN*Dp%4`P%FU6xZ42G(i!c!$fZvonFw0no5xC6(al)b>Pv+trDRr!elMsGP9A?RE23G%mm{{})'
            'QilBCz!eP;elmjmzB@_I>NLVuI7U9o&Bqk(Lpbl%HL8A@51M5v@srpL&c4Z|DEW*JRDT`BiML&t(yj$`yE7WxOu-'
            'b>9x#u*3KF~zh>c1Xnx7`HV6X%)<hS6R#wg%K#Uh6eLXM3vjytYo4!+w4J5E;Oad$4d&Rm!pvM5f}Siw-'
            ')OIjTqhC96c=_wr(oZGSwAE$?*=={e}efx`E$?wHz($foB8+~D+zAJHV$i;spotPM`2=yLQL~(&4xlj~9(^zvrv8*4bm'
            'fgiA+oQln@fDn%Z3{;)7z4bN$A|l}$#0Y8XrXTdn>1?hK4^hXJ|EFE%El7`wL~+jfv!qlLQKC{Bl9noS}Q+-'
            'Z~K9@?Q|&WWk(@n#gEztWs$y$1NX)8)Xc*jAEf*y;T!w#t9vmVy&W^H>7+4Yl$X9JUWrSd^WlTHf|z91O)7%I$dy;FXf'
            '_!S_Z{p}?&c1>#lFSNQhATL-^XyhggAQrc}3KM($W0ba+LfQgIO;Ep-'
            '@X0=l+_3lhbu~%2guWB|eO_$u=xsf0PiPOIZG68LYe}MU3>a@$AJ?6hvXpZ(e_x=^jcp?Y|0E-'
            'a$;nul>v&hfl1Lr;#{#;TZTQ6+`IWlT<S`7!%#O*m`U}PQfiRaP+98cDHX~(yvNPn|BJ5mz{zq6&oQ$YdM}Wt%La=>!4'
            'ZrF5SjdAuqEX)V6x#xYaChc5p^x(*V-5a){3Ua)o@ZQ^GO9W!T0)tqaX4jH=mU=L>#%qVf(-'
            '2y~K#{bHQn(YF}KTSQM3ZKl8Xh7*p(XC^Xx7*D<bfldz-'
            'k*SPB>s6VI;gA#za$SJrHJLPhJObYi#1dV1L;SHedzzEFS!NFD^rqVYrlqNIW?ij@kflK+^1(is6@G$*hBh-djbqR<t&'
            'te+*JdAZl7i#vdqBvW0gY82<hD#K<ge%FY<5p0VoCGh`RhW^{PY;2)0VSi9YxudT-#x9WGpVIk*3jh@|@i-'
            'dY~*Ilf_sFvKMURhwe9dOkv;z^CLhIX8aRkx5?dNS&8joHO>#h@^7hlWxXjgzR8j}{QL|>;0m|%r&#A?2%J$qjs}Th)G'
            '|$)qr}_?OWWtzwE7kDcOAi}N<D;sju*H;o}$<Fbl48+&oO+l7G0972%ilVF@28=J{V788E?>{O^a_pe}+FuJXFRT8+ve'
            'uh&fte7zxgPK+hywL2`s9S;`hhzxE`^Gdh5LG3Mx58NsTXr$lN(%R%9+6R>BV#vL#9LDxr&h@b*TJh}$E{4?;K-'
            'dbdz@}``RJ3z+SiV;1hPuCpUMBj&%BHunX$O}_a8}|;syY!>Hz)i+hx`3#N)IjhqQCREOiQ_7^sHN%+$3r=IQqqb#E*P'
            'bq-`qj?kpx}pxEJh1?&8)DJJBtp7bI?W;=oB>_L*ottXzEs&XioBA3ymrD|q5qCkBt=f;s<iV^J=)n|onJYa12q-'
            'vInEfhaS}6<lsf!iU#vX8*63RP*9Z81ho3ydf!cEHNCmF_Ad@U^h`wDaLcRYpKhcOgx=(A0&<qvJCg#p<9;L;@Z?OJgF'
            'm323N`w+Xqf$#GOqo_NVHJ=GWn^a8Yy#4Z*W%M@X5;v@UARhc)*y@Gh@3Y`l<4pIaXVmpD}{=FvgrQ-QE0Q;r@~m%)Ns'
            'q|(DyI8Qnm1twSG>3w_f%&(Hta~s0IWn?9ttMVNrPPemmTh69VS`Ik2pb&f1B+({Tj2(Z|jV^0bg{YSWaNJ@W{PW{xdq'
            'xSf_oj!@iAM^+qum8s>$ySGV34XPiV)?3bf)ZA7kyQhNpjm><Br^4G<HoSJ#kAGo{bOa6)~Y;s@g<*)e=GP<1yXCB_;4'
            '<M-kKM@ew3vYEq-d&*W%*Ic^{POs<QsMgQ{qDBbRW3-'
            '^k%p3LqbI%gRuzdwV$EOaTm`+5cGC`!R4>l9Hkh7GIConcAjQD{8K&k6R7#tp&BoSPx*psaH}^zwZw)&AB4%-'
            'DW9k~JHgbatb=U;_SrXOA!0lxQZiOOuZCLT*PdR#z3_{)-9Z!YL^b{$T(IC+?x@5gs<*w;m$9NP#`La0iTaufRD=y`X-'
            'PD%mUj6BfyGVfe$vV0eg|&}(OKlY168J<Mdizn4NTFRg}m@^%0b+c81^E|F<hBmx~)G;5bA3dK}Yh1AE)y32xCqv8awP'
            'el_Q+RNxSWkcG55p*~m3>roWSo?khMtX8VPuT;8Z|DJrX>xF=R2}ywCF7WZBlz#S4bCe(;McR4aO;;YJ><Cyzijx*y8h'
            'Y`V{S7<<nw#hu%Q5$3kiVa&?AgqsftE-'
            '`8jI+i=h3ADcFvA;iYdUQ0BsJkdJ;ueL2&d_c2D#%kmWnEYD=#Y>|dV77YFIID!hK-o;f0i}BJuN3?jRhR-'
            '}_vW?j1Ve0}J`p38q%6G^z_sd-2>c=MNnAi@9KK77oVF<FL8L(4H8$K;MLw`)Y0VUz3p!Ih)++Lx91=^qK%jFL-'
            'cgbItk@YbmeoC9=qgsGdS;tUUw-kTASA&R{({v;y1K%!9gnaWQC=)+Lth|Ic%Fphhg3Nr5QM)M|4)DWm=L}HeR~ZO#ht'
            'WB!J@Mqg4X}`xqm5G)m>hNqqBcGyZ>^HZ$s;F9hx&S`;n8g5>n@_)=e|SZjcDqp!^55%AkMyE<pnR!sNmyW42+!+#k2G'
            'c$=r>U-'
            '7sBB=@Yp35kKb(ONg=A_nIiB#6Z!2Eq=P<3@6sR<3bM==%&pO8MqMb5AOnp#rue5$$M7P<UZ)nk*C4=p7<WDz;1TBZvU'
            'S^rb}23`<!For@cAY#H53RRVON(m*V`<9>CkICVb&Jho$@D4s5!u1Lu!OQOWmhXxRLX7(QfC<)6vW^)iJ#zE{oMe3*w}'
            '?gv5o&|9kTM~Q9Dc|}xZW!OvnS3{n`61bSa&6(UJgJEmzSQjS5V3p=x))CGCJ)I!~i)GshF5HZ9Ukl;gY6<4Uv}V|~^f'
            'g=x&c}qDThnLqJ}ND{idVh<k$Pf7j_s1>)YYcaKhfQI<)<j#cx-'
            '{?AH5)h9gH33H{oz;HvKx728Y$GpoHa(uO$*l`;$&smn4UEv!&po&uR3LF^8=mb5V`&C|U4h39dA9$KN^jz+SuyV!;Tl'
            'Kbw%n^b)=Z45!3C7Ed*1Vdsu}%tuFMT${EJL}OM%RYe?*)`){0`w)7zW)ktfyR097d>|)Xm~(5D4p{Lf)0<x1OyA-'
            '4)BV+lGM(p%?;m|M(AFUp*2%bK(iC51#K9oX7U&OMiFJ|jtgViLxN~YbrgBY^IrDGRK+`8wMqwsLsLcygo&6z;X9^~T*'
            'MRg9ezLY|4tCjxV|$_`QB!#eS-i10r0q@`tv!iGK{xI;J%px>gznzlL%V-%!Xk-'
            '}#PZxPa$%zn@pX!UdEs6(_oF2`j0vM&P%s)uu7z9ak3m#k32FtF6H%{76fgV*QBnWkL1RAM-8D)FuY^PErbc|YU6kF-'
            ')l62(-emN;O6aNiY49p?9@X!^3LU#=v72Z2fk<5rJdP6pgZ1Sgq2@)F%v}c`Y>y*fW-wmq4#A7bi4bn%R!VXvnR_L&q;'
            'T6tP;@;5<4G>4C1M8q7Wv@M;=fSxNtHdE9FBGmeo?aF19Ulg!Xk|#c=-oW*DDMfv^TTv1zBNL`Zwef(7-KAdU0ykLA-'
            't;2tDNw5bo4OT)yNGW@l!g-'
            'Lpp!{5BM?q((tU|1mny&<b;Y8eo}dExB}IDVDEE!e_qCkcMU?_}^)=L1zKkbw(D>EvlkIoxk8#z63i!yabE_w_|VT8YG'
            '4J^j&f(hz#VzeqBC-T`!Q6JBzc09RV#{qw!n34;*>_mVTL4h8qt@!)_}!r1kzG)${H3IO`iwSMdSaG{0mTJF4N-'
            'tGR@(In2Ck*o3xk^%?Q0ThzrV7Sd*ag_$3H=^Bf8<o*;0C&uic&G<ZduC)w%cV+|c#{eL5?r2jhf;@9B<3!mP>V4<|W3'
            'Mz$qWhH4oWBxuY`$QuY#=^&&n0|g&9HF%Eu^ZxqgmqiSmjzp)wvI|3{ROuW>*^>DEbXX8&81bV6C3uo~=+QyO=Hb<}It'
            'Yb`a9X1fWNI2WXs@Wh?KogZD*y^-OfH!yMgAnmFdivU%nUeF_Ue=VB^o+HQc@*J7x#rjlHe&4bHdd$7_X2*7-'
            'TrOh`5C%17RKT4L4OPIln-8ZpscMbBKT?;xlKatHB;#p;jJ-~KF1Ps%e?4b4iz-KxDN-Dzae>3}uZgC-'
            '=Ju6N6rc}ZFbS&-`*w32silg^meW9nXN+JK|U-Z3r6rS0v47zV<X(c-r7Jo6qf2-'
            '!PtXca&u%5y10#+EucORR|j>5ZcZw$;qba%Lm;}H~iM62<!;~{*y>nrs7WkEI=!FtuT=+dYK%T1QhzZHJ?xnzr8kH$&X'
            '*N9EziDrV{`OfE<@G%*bE$gV>-Q{%Gqa+N}^JM7^#i6=RB-YjzfvV+PIB_QeX!Un6j5cLS1`5#X3q#aSQ5#XJ4;`jzVH'
            'RZ412s{!W+`BI(P4NOISjl@T3|!PC&*m(9%h}3fnk-'
            '!$Q|iI|JF&sPQf0=EkIh&XU0m{p;t<I?&Z+poIGfFCXU?mx^Wh(05+TV|F720JgUaE{o_hVQIR1>%1}~oX|CPg_q88mh'
            'HxU1xgr!orCCZTB}#LskZ6#kiS~W%q+@76<`Pas89S&<sq<Fn=lr&3^Ly8O)^D%1*R$4M&u3lVecj*ZdanDA`@Zqzg<j'
            '}edH_a$A~gP2d(gOANdD!uAI4?(L#x+jFf7O$r|4E-'
            'N5yk!UHE~>K9%KlxDCYIJz1!Frimmh8;pO}$nfVX3(4<thtMZ86l<P+g?r7F<PG;Bu~T%#y@^8HYViRx2e^WJv^I~FuE'
            'Pmk{dhrA7&(~kj1@1ppwX-Qf`Xu~ngAJS?w0|^p#@;mG#FeI{NYU4eE2#kh6c;%fm+8&5(Wo&Z#UV4-'
            't#_u)if0{yP~U~v5h13qZdK$7Ikp&mgGAa)YD6q7t!fxG`?Lez>)jEg7Ftm<Vs$G(*v9!vFp7hNruy+tCR7w=2_nH=)0'
            's;aRuIVJxe1#AK?}6kwNRZsW>w427X+*9NPC?qpey+Ah&E4`ueRWlkDqhY1=_8<_-'
            '{MDz%}|tC?ypF`!Lfn?Y*U8T=BN361hH{Gj(+!NH;r(YoUdR*@WDu3<9d&XorRhb&yS!G~z<t;0>h1z^7JDOudrlmAiI'
            'ijO(|So@_OlD!eHbFPyhvp1yNeg<saQb-T-U8wh^BeeeLZOB{wk$4&&K*!M*!ZL4-'
            'u6{fR^2b*~oAN`_AuR!$T|?=|<<=-'
            ';@`#$w+Xq!9V?Z&a2%P56C^k7P&(D5;oZBxU6cQsYLGXr)sC>u}d=qlu_n<E{Al3#t&1}K>?qW<hZ3|lG`oZ++5zwKgE'
            '_ik<gVxBTVtZ~OHVpSiE4A&!%<U-|(X<;cSC7C^dWCRmsV#2Ydmng5Y~V@~gD;)hQO<e<ozXc5M-'
            'EK~qjS9=;lKf0cwZF_))e9css6ZCF%7nC^9TKw?PO5i4SFf)K24rcz?0m20FK?73wL`L;gi-'
            '6FibWLH15PujhbFK<lq>7#>$5@Mz$7o3*`7GcN_%gal=SPh%$MdqC^gbMxv#l1D?6-'
            '(~b#~AjNzN<lYPebvXr|`pPIWT&^CUFSv#CvYuew{6>1X@GY!c7)(bTHKk_~0`TehRdC^73c?L#i8yc00O8(6t!V7E3p'
            '2)KL2M9#*-8qKcDW76YXvm<^advvWZ~YFTY^W1Z*i#WSM*vuP)Or$V`)naE^AssD>qF-_x9e@%<2lB?G;EzT`HzdzK0='
            '-TTUGo<<JjfM_@vk5qRV{!K|w?D6_K)<J;tfA<I`oU4RS<tY1;v#X-'
            '17^C8+4NTEZ@e$4S3$*YN7z|XV~$CwH}mIuxxX@^tsfaWB;IBO<;Rz4<Zo_rhobf#lLxB;3Nufd_G!eLtT2=G2S3s*1o'
            '5U3>tfN@D7w8}Qa#(9rOU#CQ<GPa{u^4Ub^(;!|&*JC@RNtv&vdK1nrd&O0+9*yNwnu(8uG=FGwBe5(iB!lX95>>7_QT'
            '7^3!c#s#^y@_!XJdrg8QUP;Q69_Q#E{8LGvH`_3_Mlnnk()sLqDMw>W4oj5{^~K(awZx?XN(>Pan7{F}!w#Js95icO02'
            'tPLAyy2v4^8<MNbM<ZMSCjF~?hp)i=YDdj0lTXGvNIS-`{UGKDycgI3(o-'
            '}`E|A##D3s;blT7hvUa`<ZWC&6#1hW1YSXwYvTSo#&Bci}#8?BzzXT(w|STP^wIY)8{SpNFIcfXmzyU}^gbB!dd!%QX|'
            '&;~xXM^OY!n%qgVdcK}pgf}x`nzhSNf)}kr3NY4g~qlr+tNs@0Fc@~O2l(4Pj6gXbgz)@~mkkV&4eUY>qwl1opvp#eRM'
            'sD%KW(@~eRWlVUE>FU2&Ttsw6bVT>CGh;fJe2%%5Clo4^Q1<dK;JW-kT>=vO?w+prmbHA&bE2*s(%bgLpi?g`S&p2Mnz'
            'zJ(iN<ZHjt&7j|6_Zv$-'
            'h^Pl!?F1{^jif^Hsvk+|kG(37jL6aW2oL`iQDQ4Y>VcaVV5zht1Y`XoFN=7=pBNzl7EfaX7b2HG`k#7Wr~d&}*?b@qMX'
            'Uf1{(j#vi+^GmTdZw2>OKs7~;!eZylA^eki?hv8<k^50+2Ms@Q3~I}!pw6;1aDA2n4>bA+CEZRGM@JJui03fma%R%Ly_'
            'ZtAxA|ZnG6v0x-od@LIM5*v@ts-'
            '(?44ds3W}p}wtPKhk|dD#NrtRSD8kdd$ANSF0?2#32FHXA0GG;Ec=GuJjE#DYw}ZlPgw`VZZkR2uejWv?DRO*2)PS7C5'
            'D18g#^3JWJ6B<S4vs{p@J>!JM(5}2V8Wkj(3v9*_B&P3z<nG)*I5AJvUc=Fg$-'
            '8E$ivrt+Ca9%2)KV#fucq(4bjshX0g4&BwC-mJN_FwRJ)Seozg<3axI~KlpVHSQYXYA5Pdnp#3QE-gj03sh3(tnV5|b~'
            '{mKe@Z|*ic`KK(9BdPfQg9Mos>IA<&oCoPf(ez`{I<%j*1FtcSC_gs{Hz@2zsg6c?Ymkfs$3>CGghU!#kpSywJpgmL$r'
            '!t54@63263>QY>NOkjWk4qux{t$8%vgGNu02{mU%;EPg%457W6-2%GOf!gMhzuxzHq=qsF-'
            'L5t>>%3&2A|F;ImRXB;5yORSHSTj%y%w*9FPEt8nL{EbyL=$Lyig!QuX4@C?05syD`U^=iTNMMx!A^U7h2aFrLP{bG+w'
            '*&XDG_IbfF#d1{B>d!ZzkpewmMSkzn2A*xz8NA%&i}-jS*?#Lco|Q{5{MsQ6>aA&L`uPq-{IU@;q{BeIB>)@F-'
            'KLL{0%>KH6$WPrcsaiuqEdS2;E^p4iY6<8>&kfCcr+aHTdD*P-'
            'xuKX74M2IMvoFM&mJXIE3pyy1dhW+)A|cL9Ph(J>m1_sNR3R0DF6!>MVPC3k?u%!fo+o0aC%5KnL=7fi_B`c^>_&|p$='
            'GXnoO3A4I;BYEdf*akJy;vf$pJQorvrecq3InTo+D&4$DKJw{;EN3U$IkO$dV<Tgb~|Ni<nF6+a(&T^!JVD5|7=q#Yl1'
            'vF64ws_lLU&4;HFd!IRQ!iqzOZQIO6onVybUx)U(1h}z9nS{ixpd_afRM#q@^z~WDncf0_C}}{6`2cha5fH_ht0C5MD*'
            'ieu9u4<LK;G<hawN_I`!!9+v^XI)hgqRz3xgYV8i};B0J3;tm@5B*mMUMPCFz==dhZjd;TD1H(EzB})yd6?)Z*paC85W'
            'zX=qiluxIVT$a>%U|Ga;=*#6nZ-TP-BzgGu><IU(d`V99?P7^t;vksJ#b)i;15XzVb5Te+|yU}M3uGEP~ohvu7By1GAd'
            'cTGf=NfQ*;$d95_Z)S7V2_WoZ6Nb&7+iN<g)0x1lj1wlxQurTUK;5@z{$<U9#Vtp*B$DhJmfUQ%Gbf_`de^Td0Wr=4aL'
            '+dy??O(w@J7DL_<#|**kza9qhuqObB2O-V9`-'
            'HpDP;gLgBNi`|$Fwc8n)a!)4nln3Jz>cv?31~U3hKFpLmkxa9?KQkb87gN61hcVpa$7o!RX7qcxF&D0TFqwW^nUXH{$g'
            'u!M>zxlX@Juk%zG(+DCVV%exhaI1SGt3#ckt@jfA*M~P5nR6Z`!TD(9#GMUcaCgQMMRnvY4))x~F*P=@1OaBJ|zq{Wz>'
            'c1LHMKFo{`&Gj{^=`f1{mJBGNjwH*61NqAyL6>$<u;ISt!sMf#+eAi%$)>C3|x<Vv=*;q=0=O<GYP6}1cm_zCBQ&G2F$'
            'epNq6uqh<1Zx(g^z6TWq99u8AL!?F>));C%P4sGGCG_<#@^JQQSsc$3^w;>3QWD3_&^^fWBxWqZTb!-'
            ';g<lWc2+QB)8N5ux#`J7EC^<D-'
            '?%VqEp{=f2f~>{+txDLHQSloNN>jZhBxzOttV5ayo1S4^k>?}`!d;|{h4s|XOvF)Fejb{Fa?%7nW{J2d-'
            '89i+j{mN=r`-uZ<p0UTMv)N)R8sC_r1>2M&JF#(Za4;=e&uYk=CT6jPhyLSQq+UH-jd7<Wt>y?KHl#@LbvLacDSXEzgc'
            '9;}t7PliJE@s?jBScj#DrH{d;YxZ`!fm+Sp#ll47%!akkKDoNlWcS}Lpk-5d7*vYlYH6ne&&-'
            'dh?zN*On@76E5JGi(yZg&2ESifW_)2-k}^14NmT`?s8PX#Ky4E|Oyu_uMD4la&%8y&sv+`81Pa$I9RW#WG+NZ-JSqi1T'
            'sF)%XIGc-'
            '0aGcz#O*XI~;^f>w)V>43|GeeHPo{5P`x5_+?eESJqIwtg}l;{@wx6b5&ew{|Dy?S+j6~vT_j{C$d8#c1(UC=P1Z`ZZ^'
            'JLo$yNcDeddoMSK%`R&kT-dnA<veSD#jXKV?6#Tc=dSAa!FBc6Z0P#|ao-02d$!t;#p<(JPRH%CZ-rm}J2ujY&58w1dQ'
            'oKkE%5iSf&&{CiySy74SyV2VJ#arV8f5H)=K_3a%3DEHe|zY$wn7{0>`sqBQ|VT`(@it-~={o%!Z9-'
            'Yn=HBoXCbv*znC8=NJA2KEQ@e+3<4r#zQ|D=YwpRBR0<bT9u!Sa}^slW5Zr6&5nGF{JlFBuCidUVXpIQJ^N$TmK|(X%$'
            '>omgBJf-'
            'b+wKJ3l?*y)#hnUKY=H*VKLeKi}sa2fpyuim^<%I=)Ck3SdR^hx%1}CSoNR4`fOOtovt%eeSVznY6CVb=FZR!!?*l6vg'
            'K+vEapzp<7-&(8a6EEPSLYpSh8c-u$VhV4_;x_&R7;KHd3O8mavNun-'
            'z1X=!qZfBE*Kp+$nlo2D=EcVKH}#o;ATPLTp&<KZ_oez%D{;Sj?TGyZp22$RRc?=1$Qi`mD%FY*@^lqU+mPk%eqn%$=f'
            '(!&z_%8y0h?=&EiOe4Y)9xl?pn_Psk}*sxf#MW;lpy8DjBisf2#I>LG?*sNHpMW+JavytyvtXQT++wkvM%N7<ZmT1uyn'
            'Du{KS*%!|MVnVv2@zrcr)6ftqCKU`ckqHXiT}{;T3Yn&tNpDv?yd7Tsm;0u$^3Qw)F$y)IxPD3>%RK-'
            'NbY%{x_;}^_b*1f=l;L${Wtw5U!e'
        ),
    },
]

_MODEL_CACHE = None

@dataclass(frozen=True)
class ShardArrays:
    """One compact monthly bar store plus daily sample indices."""

    bar_features: np.ndarray
    valid_price: np.ndarray
    valid_book: np.ndarray
    time_day: np.ndarray
    time_bar_index: np.ndarray
    sample_stock: np.ndarray
    sample_end_index: np.ndarray
    target: np.ndarray
    residual_target: np.ndarray
    old_factor: np.ndarray
    date: np.ndarray
    target_date: np.ndarray

def _safe_ratio(
    numerator: np.ndarray,
    denominator: np.ndarray,
    *,
    clip: float,
) -> np.ndarray:
    result = np.zeros_like(numerator, dtype=np.float64)
    valid = (
        np.isfinite(numerator)
        & np.isfinite(denominator)
        & (np.abs(denominator) > 1e-12)
    )
    np.divide(numerator, denominator, out=result, where=valid)
    return np.clip(result, -clip, clip)

def canonicalize_bar_frame(frame: pd.DataFrame) -> pd.DataFrame:
    """Validate timestamps and calculate simple per-bar microstructure states."""

    missing = sorted(set(CANONICAL_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(
            "The DAI table or field map is missing required fields: "
            f"{missing}"
        )

    data = frame.loc[:, CANONICAL_COLUMNS].copy()
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data["instrument"] = data["instrument"].astype(str)
    data = data.dropna(subset=["date", "instrument"])
    minute = data["date"].dt.hour * 60 + data["date"].dt.minute
    data["bar_index"] = minute.map(BAR_END_MINUTES)
    if data["bar_index"].isna().any():
        bad = sorted(minute.loc[data["bar_index"].isna()].unique().tolist())
        raise ValueError(
            "Unexpected 30-minute bar timestamps. Expected endpoints "
            f"{sorted(BAR_END_MINUTES)}, found minute-of-day values {bad[:20]}"
        )
    data["bar_index"] = data["bar_index"].astype(np.int8)
    data["trading_date"] = data["date"].dt.normalize()

    numeric_columns = CANONICAL_COLUMNS[2:]
    for column in numeric_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    data = (
        data.sort_values(["date", "instrument"])
        .drop_duplicates(["date", "instrument"], keep="last")
        .reset_index(drop=True)
    )

    ask_volume_raw = data[
        [f"ask_volume{level}" for level in (1, 2, 3)]
    ].to_numpy(dtype=np.float64)
    bid_volume_raw = data[
        [f"bid_volume{level}" for level in (1, 2, 3)]
    ].to_numpy(dtype=np.float64)
    old_volume_valid = (
        np.isfinite(ask_volume_raw).all(axis=1)
        & np.isfinite(bid_volume_raw).all(axis=1)
        & (ask_volume_raw >= 0).all(axis=1)
        & (bid_volume_raw >= 0).all(axis=1)
    )
    old_ask_volume = np.sum(ask_volume_raw, axis=1)
    old_bid_volume = np.sum(bid_volume_raw, axis=1)
    old_volume_imbalance = _safe_ratio(
        old_bid_volume - old_ask_volume,
        old_bid_volume + old_ask_volume,
        clip=1.0,
    )
    old_volume_imbalance[~old_volume_valid] = np.nan

    ask_volume = sum(
        data[f"ask_volume{level}"].clip(lower=0).fillna(0.0)
        for level in (1, 2, 3)
    ).to_numpy(dtype=np.float64)
    bid_volume = sum(
        data[f"bid_volume{level}"].clip(lower=0).fillna(0.0)
        for level in (1, 2, 3)
    ).to_numpy(dtype=np.float64)
    ask_orders = sum(
        data[f"ask_num_orders{level}"].clip(lower=0).fillna(0.0)
        for level in (1, 2, 3)
    ).to_numpy(dtype=np.float64)
    bid_orders = sum(
        data[f"bid_num_orders{level}"].clip(lower=0).fillna(0.0)
        for level in (1, 2, 3)
    ).to_numpy(dtype=np.float64)

    ask_volume1 = (
        data["ask_volume1"].clip(lower=0).fillna(0.0).to_numpy(dtype=np.float64)
    )
    bid_volume1 = (
        data["bid_volume1"].clip(lower=0).fillna(0.0).to_numpy(dtype=np.float64)
    )
    ask_price1 = data["ask_price1"].to_numpy(dtype=np.float64)
    bid_price1 = data["bid_price1"].to_numpy(dtype=np.float64)
    midpoint = (ask_price1 + bid_price1) / 2.0
    old_spread_valid = (
        np.isfinite(ask_price1)
        & np.isfinite(bid_price1)
        & (ask_price1 > 0)
        & (bid_price1 > 0)
        & (ask_price1 >= bid_price1)
    )
    old_relative_spread = _safe_ratio(
        ask_price1 - bid_price1,
        midpoint,
        clip=0.20,
    )
    old_relative_spread[~old_spread_valid] = np.nan
    open_price = data["open"].to_numpy(dtype=np.float64)
    high_price = data["high"].to_numpy(dtype=np.float64)
    low_price = data["low"].to_numpy(dtype=np.float64)
    close_price = data["close"].to_numpy(dtype=np.float64)
    book_columns = [
        *(f"ask_price{level}" for level in (1, 2, 3)),
        *(f"bid_price{level}" for level in (1, 2, 3)),
        *(f"ask_volume{level}" for level in (1, 2, 3)),
        *(f"bid_volume{level}" for level in (1, 2, 3)),
        *(f"ask_num_orders{level}" for level in (1, 2, 3)),
        *(f"bid_num_orders{level}" for level in (1, 2, 3)),
    ]
    raw_book = data[book_columns].to_numpy(dtype=np.float64)
    valid_book = np.isfinite(raw_book).all(axis=1)
    valid_book &= (data[[f"ask_price{level}" for level in (1, 2, 3)]].to_numpy() > 0).all(axis=1)
    valid_book &= (data[[f"bid_price{level}" for level in (1, 2, 3)]].to_numpy() > 0).all(axis=1)
    valid_book &= (data[[f"ask_volume{level}" for level in (1, 2, 3)]].to_numpy() >= 0).all(axis=1)
    valid_book &= (data[[f"bid_volume{level}" for level in (1, 2, 3)]].to_numpy() >= 0).all(axis=1)
    valid_book &= (data[[f"ask_num_orders{level}" for level in (1, 2, 3)]].to_numpy() >= 0).all(axis=1)
    valid_book &= (data[[f"bid_num_orders{level}" for level in (1, 2, 3)]].to_numpy() >= 0).all(axis=1)
    valid_book &= ask_price1 >= bid_price1

    data["volume_imbalance_l3"] = _safe_ratio(
        bid_volume - ask_volume,
        bid_volume + ask_volume,
        clip=1.0,
    )
    data["volume_imbalance_l3_raw"] = old_volume_imbalance
    data["order_count_imbalance_l3"] = _safe_ratio(
        bid_orders - ask_orders,
        bid_orders + ask_orders,
        clip=1.0,
    )
    bid_mean_size = bid_volume / np.maximum(bid_orders, 1.0)
    ask_mean_size = ask_volume / np.maximum(ask_orders, 1.0)
    data["average_order_size_gap"] = np.clip(
        np.log1p(bid_mean_size) - np.log1p(ask_mean_size),
        -10.0,
        10.0,
    )
    data["l1_depth_concentration_gap"] = (
        _safe_ratio(bid_volume1, bid_volume, clip=1.0)
        - _safe_ratio(ask_volume1, ask_volume, clip=1.0)
    )
    data["relative_spread_l1"] = _safe_ratio(
        ask_price1 - bid_price1,
        midpoint,
        clip=0.20,
    )
    data["relative_spread_l1_raw"] = old_relative_spread
    data["log_total_depth"] = np.log1p(
        np.clip(bid_volume + ask_volume, 0.0, None)
    )
    data["log_total_order_count"] = np.log1p(
        np.clip(bid_orders + ask_orders, 0.0, None)
    )
    data["bar_return"] = _safe_ratio(
        close_price - open_price,
        open_price,
        clip=0.20,
    )
    data["bar_range"] = _safe_ratio(
        high_price - low_price,
        open_price,
        clip=0.20,
    )
    data["log_volume"] = np.log1p(
        data["volume"].clip(lower=0).fillna(0.0).to_numpy(dtype=np.float64)
    )
    data["log_amount"] = np.log1p(
        data["amount"].clip(lower=0).fillna(0.0).to_numpy(dtype=np.float64)
    )
    data["log_deal_number"] = np.log1p(
        data["deal_number"].clip(lower=0).fillna(0.0).to_numpy(dtype=np.float64)
    )

    valid_ohlc = (
        np.isfinite(open_price)
        & np.isfinite(high_price)
        & np.isfinite(low_price)
        & np.isfinite(close_price)
        & (open_price > 0)
        & (high_price > 0)
        & (low_price > 0)
        & (close_price > 0)
    )
    data["valid_bar"] = valid_ohlc
    data["valid_book"] = valid_book
    feature_values = data[BASE_FEATURE_NAMES].replace(
        [np.inf, -np.inf],
        np.nan,
    )
    feature_values = feature_values.where(
        pd.Series(valid_book, index=data.index),
        np.nan,
    )
    # Timestamp-level ranks remove most absolute size and liquidity exposure.
    ranked = (
        feature_values.groupby(data["date"], observed=True)
        .rank(method="average", pct=True)
        .sub(0.5)
    )
    data[BASE_FEATURE_NAMES] = ranked.fillna(0.0).astype(np.float32)
    return data

def build_old_factor(
    bars: pd.DataFrame,
    pool: pd.DataFrame,
    exposure: pd.DataFrame,
    core_start: pd.Timestamp,
    core_end: pd.Timestamp,
) -> pd.DataFrame:
    """Recreate the submitted six-feature industry-ranked factor."""

    grouped = bars.sort_values(["trading_date", "instrument", "date"]).groupby(
        ["trading_date", "instrument"],
        observed=True,
        sort=True,
    )
    daily = grouped.agg(
        book_imbalance_mean=("volume_imbalance_l3_raw", "mean"),
        book_imbalance_last=("volume_imbalance_l3_raw", "last"),
        relative_spread_mean=("relative_spread_l1_raw", "mean"),
        relative_spread_last=("relative_spread_l1_raw", "last"),
        close=("close", "last"),
        adjust_factor=("adjust_factor", "last"),
    ).reset_index()
    daily = daily.rename(columns={"trading_date": "date"})
    daily = daily.loc[daily["date"].between(core_start, core_end)].copy()
    daily = daily.merge(
        pool,
        on=["date", "instrument"],
        how="inner",
        validate="one_to_one",
    )
    daily["pressure_persistence"] = (
        daily["book_imbalance_mean"] * daily["book_imbalance_last"]
    )
    daily["imbalance_change"] = (
        daily["book_imbalance_last"] - daily["book_imbalance_mean"]
    )
    feature_names = list(OLD_FACTOR_WEIGHTS)
    ranked = (
        daily.groupby("date", observed=True)[feature_names]
        .rank(method="average", pct=True)
        .sub(0.5)
        .fillna(0.0)
    )
    daily["old_raw"] = OLD_FACTOR_INTERCEPT
    for name, weight in OLD_FACTOR_WEIGHTS.items():
        daily["old_raw"] += ranked[name] * weight

    daily = daily.merge(
        exposure,
        on=["date", "instrument"],
        how="left",
        validate="one_to_one",
    )
    daily["industry_code"] = (
        daily["industry_code"].astype("string").fillna("UNKNOWN")
    )
    global_rank = (
        daily.groupby("date", observed=True)["old_raw"]
        .rank(method="average", pct=True)
        .sub(0.5)
    )
    industry_group = daily.groupby(
        ["date", "industry_code"],
        observed=True,
    )["old_raw"]
    industry_rank = (
        industry_group.rank(method="average", pct=True).sub(0.5)
    )
    industry_count = industry_group.transform("count")
    daily["old_factor"] = industry_rank.where(
        industry_count >= MIN_INDUSTRY_STOCKS,
        global_rank,
    )
    daily["adjusted_close"] = daily["close"] * daily["adjust_factor"]
    daily.loc[
        (~np.isfinite(daily["adjusted_close"]))
        | (daily["adjusted_close"] <= 0),
        "adjusted_close",
    ] = np.nan
    return daily[
        ["date", "instrument", "adjusted_close", "old_factor"]
    ]

class CompactWindowDataset:
    """Dynamically materialize 40-bar model windows from a compact shard."""

    def __init__(
        self,
        shard: ShardArrays,
        *,
        lookback_bars: int = LOOKBACK_BARS,
    ) -> None:
        self.shard = shard
        self.lookback_bars = lookback_bars

    def __len__(self) -> int:
        return len(self.shard.target)

    def __getitem__(self, index: int) -> tuple[np.ndarray, np.ndarray]:
        x, mask = self.batch(np.asarray([index], dtype=np.int64))
        return x[0], mask[0]

    def batch(
        self,
        sample_indices: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray]:
        sample_indices = np.asarray(sample_indices, dtype=np.int64)
        stock = self.shard.sample_stock[sample_indices].astype(
            np.int64,
            copy=False,
        )
        end = self.shard.sample_end_index[sample_indices].astype(
            np.int64,
            copy=False,
        )
        offsets = np.arange(
            self.lookback_bars - 1,
            -1,
            -1,
            dtype=np.int64,
        )
        window_index = end[:, None] - offsets[None, :]
        in_range = window_index >= 0
        safe_index = np.maximum(window_index, 0)
        raw = self.shard.bar_features[
            stock[:, None],
            safe_index,
        ].astype(np.float32)
        valid_price = (
            self.shard.valid_price[stock[:, None], safe_index].astype(bool)
            & in_range
        )
        valid_book = (
            self.shard.valid_book[stock[:, None], safe_index].astype(bool)
            & in_range
        )
        valid = valid_price & valid_book
        mask = valid.astype(np.float32)
        raw *= mask[:, :, None]

        denominator = np.maximum(mask.sum(axis=1, keepdims=True), 1.0)
        center = (
            (raw * mask[:, :, None]).sum(axis=1, keepdims=True)
            / denominator[:, :, None]
        )
        centered = (raw - center) * mask[:, :, None]

        day = self.shard.time_day[safe_index]
        bar_index = self.shard.time_bar_index[safe_index]
        overnight = np.zeros_like(valid, dtype=bool)
        overnight[:, 1:] = day[:, 1:] != day[:, :-1]
        lunch = np.zeros_like(valid, dtype=bool)
        lunch[:, 1:] = (
            (day[:, 1:] == day[:, :-1])
            & (bar_index[:, :-1] == 3)
            & (bar_index[:, 1:] == 4)
        )
        intraday_gap = np.zeros_like(valid, dtype=bool)
        intraday_gap[:, 1:] = (
            (day[:, 1:] == day[:, :-1])
            & (bar_index[:, 1:] != bar_index[:, :-1] + 1)
        )
        session_boundary = overnight | lunch | intraday_gap
        session_boundary[:, 0] = True

        pair_valid = np.zeros_like(valid, dtype=bool)
        pair_valid[:, 1:] = (
            valid[:, 1:]
            & valid[:, :-1]
            & ~session_boundary[:, 1:]
        )
        delta = np.zeros_like(raw, dtype=np.float32)
        delta[:, 1:] = (
            raw[:, 1:] - raw[:, :-1]
        ) * pair_valid[:, 1:, None]

        book_marker = valid_book.astype(np.float32)[:, :, None]
        boundary_marker = (
            session_boundary.astype(np.float32) * mask
        )[:, :, None]
        overnight_marker = (
            overnight.astype(np.float32) * mask
        )[:, :, None]
        x = np.concatenate(
            [
                raw,
                centered,
                delta,
                book_marker,
                boundary_marker,
                overnight_marker,
            ],
            axis=2,
        ).astype(np.float32, copy=False)
        if x.shape[2] != len(MODEL_FEATURE_NAMES):
            raise RuntimeError(
                f"Dynamic feature width {x.shape[2]} does not match "
                f"manifest width {len(MODEL_FEATURE_NAMES)}."
            )
        return x, mask

def build_tcn_model(torch_module: Any, nn: Any, config: dict[str, Any]) -> Any:
    class CausalConv1d(nn.Conv1d):
        def __init__(
            self,
            in_channels: int,
            out_channels: int,
            kernel_size: int,
            dilation: int,
        ) -> None:
            self.left_padding = (kernel_size - 1) * dilation
            super().__init__(
                in_channels,
                out_channels,
                kernel_size,
                dilation=dilation,
                padding=self.left_padding,
            )

        def forward(self, x: Any) -> Any:
            output = super().forward(x)
            if self.left_padding:
                output = output[:, :, : -self.left_padding]
            return output

    class ResidualBlock(nn.Module):
        def __init__(
            self,
            channels: int,
            kernel_size: int,
            dilation: int,
            dropout: float,
        ) -> None:
            super().__init__()
            self.conv1 = CausalConv1d(
                channels,
                channels,
                kernel_size,
                dilation,
            )
            self.conv2 = CausalConv1d(
                channels,
                channels,
                kernel_size,
                dilation,
            )
            self.norm1 = nn.GroupNorm(1, channels)
            self.norm2 = nn.GroupNorm(1, channels)
            self.activation = nn.GELU()
            self.dropout = nn.Dropout(dropout)

        def forward(self, x: Any, mask: Any) -> Any:
            residual = x
            output = self.conv1(x)
            output = self.dropout(self.activation(self.norm1(output))) * mask
            output = self.conv2(output)
            output = self.dropout(self.activation(self.norm2(output)))
            return (residual + output) * mask

    class OrderFlowPathTCN(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            channels = int(config["channels"])
            self.projection = nn.Conv1d(
                int(config["n_features"]),
                channels,
                kernel_size=1,
            )
            self.blocks = nn.ModuleList(
                [
                    ResidualBlock(
                        channels,
                        int(config["kernel_size"]),
                        int(dilation),
                        float(config["dropout"]),
                    )
                    for dilation in config["dilations"]
                ]
            )
            self.head = nn.Sequential(
                nn.LayerNorm(channels * 2),
                nn.Linear(channels * 2, channels),
                nn.GELU(),
                nn.Dropout(float(config["dropout"])),
                nn.Linear(channels, 1),
            )

        def forward(self, x: Any, valid_mask: Any) -> Any:
            temporal_mask = valid_mask[:, None, :]
            hidden = self.projection(x.transpose(1, 2)) * temporal_mask
            for block in self.blocks:
                hidden = block(hidden, temporal_mask)
            denominator = temporal_mask.sum(dim=2).clamp_min(1.0)
            mean_pool = (hidden * temporal_mask).sum(dim=2) / denominator
            positions = (
                torch_module.arange(
                    valid_mask.shape[1],
                    device=valid_mask.device,
                )[None, :]
                .expand_as(valid_mask)
            )
            last_index = torch_module.where(
                valid_mask > 0,
                positions,
                torch_module.zeros_like(positions),
            ).max(dim=1).values
            last_pool = hidden.gather(
                2,
                last_index[:, None, None].expand(-1, hidden.shape[1], 1),
            ).squeeze(2)
            pooled = torch_module.cat([mean_pool, last_pool], dim=1)
            return self.head(pooled).squeeze(-1)

    return OrderFlowPathTCN()

def orthogonalize_predictions(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["orthogonal_prediction"] = np.nan
    for _, index in result.groupby("date", observed=True).groups.items():
        idx = np.asarray(index)
        prediction = (
            result.loc[idx, "prediction"]
            .rank(method="average", pct=True)
            .sub(0.5)
            .to_numpy(dtype=np.float64)
        )
        old = (
            result.loc[idx, "old_factor"]
            .rank(method="average", pct=True)
            .sub(0.5)
            .to_numpy(dtype=np.float64)
        )
        valid = np.isfinite(prediction) & np.isfinite(old)
        if int(valid.sum()) < 20:
            continue
        prediction_centered = (
            prediction[valid] - float(np.mean(prediction[valid]))
        )
        old_centered = old[valid] - float(np.mean(old[valid]))
        denominator = float(np.dot(old_centered, old_centered))
        beta = (
            float(
                np.dot(old_centered, prediction_centered) / denominator
            )
            if denominator > 1e-12
            else 0.0
        )
        intercept = float(
            np.mean(prediction[valid]) - beta * np.mean(old[valid])
        )
        residual = np.full(len(idx), np.nan, dtype=np.float64)
        residual[valid] = prediction[valid] - (
            intercept + beta * old[valid]
        )
        result.loc[idx, "orthogonal_prediction"] = residual
    return result


def _validate_identifier(value: str, *, table: bool = False) -> str:
    pattern = TABLE_IDENTIFIER if table else IDENTIFIER
    if not isinstance(value, str) or not pattern.fullmatch(value):
        kind = "table" if table else "column"
        raise ValueError(f"Unsafe {kind} identifier: {value!r}")
    return value


def _date_filter_end(value: pd.Timestamp) -> pd.Timestamp:
    return value.normalize() + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1)


def _query_pool(start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    import dai

    frame = dai.query(
        f"SELECT date, instrument FROM {POOL_TABLE}",
        filters={"date": [str(start), str(_date_filter_end(end))]},
        compression=True,
    ).df()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
    frame["instrument"] = frame["instrument"].astype(str)
    return (
        frame.dropna(subset=["date", "instrument"])
        .drop_duplicates(["date", "instrument"])
        .reset_index(drop=True)
    )


def _resolve_padding_start(start: pd.Timestamp) -> pd.Timestamp:
    import dai

    probe_start = start - pd.Timedelta(days=PADDING_PROBE_CALENDAR_DAYS)
    calendar = dai.query(
        f"SELECT DISTINCT date FROM {POOL_TABLE}",
        filters={
            "date": [str(probe_start), str(_date_filter_end(start))]
        },
        compression=True,
    ).df()
    calendar["date"] = pd.to_datetime(
        calendar["date"],
        errors="coerce",
    ).dt.normalize()
    previous = np.sort(
        calendar.loc[calendar["date"] < start, "date"]
        .dropna()
        .unique()
        .astype("datetime64[ns]")
    )
    if len(previous) >= PREVIOUS_TRADING_DAYS:
        return pd.Timestamp(previous[-PREVIOUS_TRADING_DAYS]).normalize()
    raise RuntimeError(
        "The runtime datasource does not expose five trading days before "
        f"{start.date()}; found only {len(previous)} within the preceding "
        f"{PADDING_PROBE_CALENDAR_DAYS} calendar days."
    )


def _query_exposure(start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    import dai

    frame = dai.query(
        (
            "SELECT date, instrument, "
            f"{INDUSTRY_FIELD} AS industry_code FROM {EXPOSURE_TABLE}"
        ),
        filters={"date": [str(start), str(_date_filter_end(end))]},
        compression=True,
    ).df()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
    frame["instrument"] = frame["instrument"].astype(str)
    frame["industry_code"] = (
        frame["industry_code"].astype("string").fillna("UNKNOWN")
    )
    return (
        frame.dropna(subset=["date", "instrument"])
        .drop_duplicates(["date", "instrument"], keep="last")
        .reset_index(drop=True)
    )


def _query_and_rebuild_30m(
    table_name: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> pd.DataFrame:
    import dai

    _validate_identifier(table_name, table=True)
    sql = REBUILD_SQL_TEMPLATE.replace("__TABLE_NAME__", table_name)
    frame = dai.query(
        sql,
        filters={"date": [str(start), str(_date_filter_end(end))]},
        compression=True,
    ).df()
    if frame.empty:
        return pd.DataFrame(columns=CANONICAL_COLUMNS)
    frame["trading_day"] = pd.to_datetime(
        frame["trading_day"],
        errors="coerce",
    ).dt.normalize()
    frame["bar_index"] = pd.to_numeric(
        frame["bar_index"],
        errors="coerce",
    )
    frame = frame.dropna(subset=["trading_day", "instrument", "bar_index"])
    frame["bar_index"] = frame["bar_index"].astype(np.int8)
    if not frame["bar_index"].between(1, N_BARS_PER_DAY).all():
        raise RuntimeError("30-minute reconstruction produced an invalid bucket.")
    endpoint_minutes = np.asarray(
        [600, 630, 660, 690, 810, 840, 870, 900],
        dtype=np.int16,
    )
    frame["date"] = frame["trading_day"] + pd.to_timedelta(
        endpoint_minutes[frame["bar_index"].to_numpy() - 1],
        unit="m",
    )
    duplicates = frame.duplicated(["date", "instrument"], keep=False)
    if duplicates.any():
        raise RuntimeError("30-minute reconstruction produced duplicate bars.")
    return frame.loc[:, CANONICAL_COLUMNS]


def _dense_inference_store(
    bars: pd.DataFrame,
    daily: pd.DataFrame,
) -> tuple[ShardArrays, pd.DataFrame]:
    timestamps = np.sort(bars["date"].unique().astype("datetime64[ns]"))
    timestamp_to_index = {
        pd.Timestamp(value): position for position, value in enumerate(timestamps)
    }
    instruments = np.sort(bars["instrument"].astype(str).unique())
    instrument_to_index = {
        value: position for position, value in enumerate(instruments)
    }
    dense = np.zeros(
        (len(instruments), len(timestamps), len(BASE_FEATURE_NAMES)),
        dtype=np.float16,
    )
    valid_price = np.zeros((len(instruments), len(timestamps)), dtype=np.uint8)
    valid_book = np.zeros((len(instruments), len(timestamps)), dtype=np.uint8)
    stock_index = bars["instrument"].astype(str).map(instrument_to_index).to_numpy()
    time_index = bars["date"].map(timestamp_to_index).to_numpy()
    # This assignment deliberately reproduces the training cache round trip.
    dense[stock_index, time_index] = bars[BASE_FEATURE_NAMES].to_numpy(
        dtype=np.float16
    )
    valid_price[stock_index, time_index] = bars["valid_bar"].to_numpy(
        dtype=np.uint8
    )
    valid_book[stock_index, time_index] = bars["valid_book"].to_numpy(
        dtype=np.uint8
    )

    last_timestamp = (
        bars.groupby("trading_date", observed=True)["date"].max().to_dict()
    )
    sample_stock = daily["instrument"].astype(str).map(instrument_to_index)
    sample_end = daily["date"].map(last_timestamp).map(timestamp_to_index)
    structural = sample_stock.notna() & sample_end.notna()
    samples = daily.loc[structural].reset_index(drop=True)
    sample_stock_array = sample_stock.loc[structural].to_numpy(dtype=np.int32)
    sample_end_array = sample_end.loc[structural].to_numpy(dtype=np.int32)

    offsets = np.arange(LOOKBACK_BARS - 1, -1, -1, dtype=np.int64)
    window_index = sample_end_array[:, None] - offsets[None, :]
    in_range = window_index >= 0
    safe_index = np.maximum(window_index, 0)
    compact_mask = (
        valid_price[sample_stock_array[:, None], safe_index]
        * valid_book[sample_stock_array[:, None], safe_index]
        * in_range
    )
    keep = compact_mask.sum(axis=1) >= MIN_VALID_BARS
    samples = samples.loc[keep].reset_index(drop=True)
    sample_stock_array = sample_stock_array[keep]
    sample_end_array = sample_end_array[keep]

    datetime_index = pd.to_datetime(timestamps)
    time_day = datetime_index.strftime("%Y%m%d").astype(int).to_numpy(
        dtype=np.int32
    )
    time_bar_index = np.asarray(
        [
            BAR_END_MINUTES[value.hour * 60 + value.minute]
            for value in datetime_index
        ],
        dtype=np.int8,
    )
    n_samples = len(samples)
    shard = ShardArrays(
        bar_features=dense,
        valid_price=valid_price,
        valid_book=valid_book,
        time_day=time_day,
        time_bar_index=time_bar_index,
        sample_stock=sample_stock_array,
        sample_end_index=sample_end_array.astype(np.int32, copy=False),
        target=np.zeros(n_samples, dtype=np.float32),
        residual_target=np.zeros(n_samples, dtype=np.float32),
        old_factor=samples["old_factor"].to_numpy(dtype=np.float32),
        date=samples["date"].dt.strftime("%Y%m%d").astype(int).to_numpy(
            dtype=np.int32
        ),
        target_date=np.zeros(n_samples, dtype=np.int32),
    )
    return shard, samples


def _decode_checkpoint(record: dict[str, Any]) -> bytes:
    compressed = base64.b85decode(record["payload_b85"].encode("ascii"))
    raw = zlib.decompress(compressed)
    if len(raw) != int(record["raw_bytes"]):
        raise RuntimeError(
            f"Embedded checkpoint length mismatch for seed={record['seed']}."
        )
    digest = hashlib.sha256(raw).hexdigest()
    if digest != record["checkpoint_sha256"]:
        raise RuntimeError(
            f"Embedded checkpoint hash mismatch for seed={record['seed']}."
        )
    return raw


def _load_models() -> tuple[Any, list[tuple[Any, float]]]:
    global _MODEL_CACHE
    if _MODEL_CACHE is not None:
        return _MODEL_CACHE
    if STRUCTURE_ONLY:
        raise RuntimeError(
            "This notebook was built with dry-run placeholder weights and "
            "cannot be submitted or executed."
        )
    import torch
    import torch.nn as nn

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    models: list[tuple[Any, float]] = []
    for record in MODEL_PAYLOADS:
        raw = _decode_checkpoint(record)
        try:
            checkpoint = torch.load(
                io.BytesIO(raw),
                map_location="cpu",
                weights_only=False,
            )
        except TypeError:
            checkpoint = torch.load(io.BytesIO(raw), map_location="cpu")
        if not isinstance(checkpoint, dict):
            raise TypeError("Embedded checkpoint must contain a dictionary.")
        if checkpoint.get("model_kind") != "tcn":
            raise ValueError("Embedded checkpoint is not an OrderFlowPath TCN.")
        if checkpoint.get("feature_names") != MODEL_FEATURE_NAMES:
            raise ValueError("Checkpoint model feature order does not match.")
        if checkpoint.get("base_feature_names") != BASE_FEATURE_NAMES:
            raise ValueError("Checkpoint base feature order does not match.")
        if int(checkpoint.get("lookback_bars", -1)) != LOOKBACK_BARS:
            raise ValueError("Checkpoint lookback does not match.")
        if int(checkpoint.get("min_valid_bars", -1)) != MIN_VALID_BARS:
            raise ValueError("Checkpoint minimum-valid-bars does not match.")
        if int(checkpoint.get("seed", -1)) != int(record["seed"]):
            raise ValueError("Checkpoint seed does not match ensemble manifest.")
        if checkpoint.get("training_period") != "2019-01-01/2023-12-31":
            raise ValueError(
                "Checkpoint is not the locked 2019-2023 retrained model."
            )
        if int(checkpoint.get("locked_epochs", 0)) < 1:
            raise ValueError("Checkpoint is missing a positive locked_epochs.")
        declared = {
            str(checkpoint[key]).lower()
            for key in (
                "script_sha256",
                "training_script_sha256",
                "source_sha256",
                "training_source_sha256",
            )
            if key in checkpoint
        }
        if declared and declared != {EXPECTED_TRAINING_SCRIPT_SHA256.lower()}:
            raise RuntimeError(
                "Checkpoint/training-script hash mismatch: "
                f"checkpoint={sorted(declared)}, "
                f"expected={EXPECTED_TRAINING_SCRIPT_SHA256}"
            )
        config = checkpoint.get("model_config")
        if not isinstance(config, dict):
            raise TypeError("Checkpoint is missing model_config.")
        model = build_tcn_model(torch, nn, config).to(device)
        model.load_state_dict(checkpoint["state_dict"], strict=True)
        model.eval()
        models.append((model, float(record["weight"])))
    _MODEL_CACHE = (device, models)
    return _MODEL_CACHE


def _predict_seed_matrix(shard: ShardArrays) -> np.ndarray:
    import torch

    device, models = _load_models()
    dataset = CompactWindowDataset(shard)
    prediction = np.zeros(
        (len(shard.target), len(models)),
        dtype=np.float32,
    )
    with torch.no_grad():
        for start in range(0, len(shard.target), INFERENCE_BATCH_SIZE):
            index = np.arange(
                start,
                min(start + INFERENCE_BATCH_SIZE, len(shard.target)),
                dtype=np.int64,
            )
            x_batch, mask_batch = dataset.batch(index)
            xb = torch.from_numpy(x_batch).to(device)
            mb = torch.from_numpy(mask_batch).to(device)
            for model_index, (model, _) in enumerate(models):
                values = model(xb, mb).detach().cpu().numpy()
                prediction[index, model_index] = values.astype(
                    np.float32,
                    copy=False,
                )
    return prediction


def _combine_seed_predictions(
    samples: pd.DataFrame,
    seed_prediction: np.ndarray,
) -> pd.DataFrame:
    if seed_prediction.shape != (len(samples), len(MODEL_PAYLOADS)):
        raise RuntimeError(
            "Seed prediction matrix does not match sample/seed dimensions."
        )
    work = samples[["date", "instrument", "old_factor"]].copy()
    ranked_columns = []
    for seed_index, record in enumerate(MODEL_PAYLOADS):
        raw_column = f"seed_{seed_index}_raw"
        rank_column = f"seed_{seed_index}_rank"
        work[raw_column] = seed_prediction[:, seed_index]
        work[rank_column] = (
            work.groupby("date", observed=True)[raw_column]
            .rank(method="average", pct=True)
            .sub(0.5)
        )
        ranked_columns.append((rank_column, float(record["weight"])))
    work["prediction"] = 0.0
    for column, weight in ranked_columns:
        work["prediction"] += weight * work[column]
    work = orthogonalize_predictions(
        work[["date", "instrument", "old_factor", "prediction"]]
    )
    # The public submission protocol uses only the residual's ordering.
    work["factor"] = (
        work.groupby("date", observed=True)["orthogonal_prediction"]
        .rank(method="average", pct=True)
        .sub(0.5)
    )
    return work[["date", "instrument", "factor"]]


def _run_core_month(
    table_name: str,
    core_start: pd.Timestamp,
    core_end: pd.Timestamp,
) -> pd.DataFrame:
    padded_start = _resolve_padding_start(core_start)
    pool = _query_pool(padded_start, core_end)
    raw_bars = _query_and_rebuild_30m(
        table_name,
        padded_start,
        core_end,
    )
    if raw_bars.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    raw_bars["date"] = pd.to_datetime(raw_bars["date"], errors="coerce")
    raw_bars["instrument"] = raw_bars["instrument"].astype(str)
    raw_bars["trading_date"] = raw_bars["date"].dt.normalize()
    padded_pool = pool.rename(columns={"date": "trading_date"})
    raw_bars = raw_bars.merge(
        padded_pool[["trading_date", "instrument"]].drop_duplicates(),
        on=["trading_date", "instrument"],
        how="inner",
    ).drop(columns=["trading_date"])
    bars = canonicalize_bar_frame(raw_bars)

    exposure = _query_exposure(core_start, core_end)
    daily = build_old_factor(
        bars,
        pool,
        exposure,
        core_start,
        core_end,
    )
    daily = daily.sort_values(["date", "instrument"]).reset_index(drop=True)
    if daily.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    shard, samples = _dense_inference_store(bars, daily)
    if len(samples) == 0:
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    seed_prediction = _predict_seed_matrix(shard)
    result = _combine_seed_predictions(samples, seed_prediction)
    result["date"] = pd.to_datetime(
        result["date"],
        errors="coerce",
    ).dt.normalize()
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    return result.loc[
        result["date"].between(core_start, core_end)
        & np.isfinite(result["factor"].to_numpy(dtype=np.float64)),
        ["date", "instrument", "factor"],
    ].reset_index(drop=True)


def _release_month_memory() -> None:
    gc.collect()
    if _MODEL_CACHE is None:
        return
    device, _ = _MODEL_CACHE
    if getattr(device, "type", "") == "cuda":
        import torch

        torch.cuda.empty_cache()


def main(datasources, start_date, end_date):
    if not isinstance(datasources, dict) or "bar1m" not in datasources:
        raise KeyError("datasources must contain 'bar1m'.")
    start = pd.Timestamp(start_date).normalize()
    end = pd.Timestamp(end_date).normalize()
    if start > end:
        raise ValueError("start_date must not be after end_date.")
    table_name = _validate_identifier(str(datasources["bar1m"]), table=True)
    parts = []
    for period in pd.period_range(start=start, end=end, freq="M"):
        core_start = max(start, period.start_time.normalize())
        core_end = min(end, period.end_time.normalize())
        part = _run_core_month(table_name, core_start, core_end)
        if not part.empty:
            parts.append(part)
        _release_month_memory()
    if not parts:
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    result = pd.concat(parts, ignore_index=True)
    return (
        result.drop_duplicates(["date", "instrument"], keep="last")
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
        .loc[:, ["date", "instrument", "factor"]]
    )
